In [3]:
!pip install gurobipy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


##  DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON + GAP PAR RAPPORT AU BKS  (10 épisodes) ==>12.90%

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON    ║
║                          + GAP PAR RAPPORT AU BKS                           ║
║                                                                              ║
║  Version corrigée : reprend toutes les visualisations de la variante        ║
║  précédente (courbes de convergence, tracés des routes avant/après,        ║
║  graphique dédié gap % vs BKS, tableau récapitulatif véhicules/distance)   ║
║  MAIS applique désormais un vrai protocole ENTRAÎNEMENT / TEST DISJOINT,   ║
║  en un seul lancement, sans aucune intervention manuelle :                 ║
║                                                                              ║
║   ÉTAPE 1 (ENTRAÎNEMENT) — génère des instances SYNTHÉTIQUES (reprises de   ║
║      generate_instance() de main_vrptw_comparaison_gurobi.py / "Code 1")    ║
║      et entraîne le DQN dessus (poids mis à jour via store/learn/decay).    ║
║                                                                              ║
║   ÉTAPE 2 (TEST) — charge les 56 instances Solomon OFFICIELLES et évalue    ║
║      le même agent, GELÉ (aucun store/learn/decay_epsilon, uniquement      ║
║      select_action en pur mode inférence), sur ce jeu jamais vu à          ║
║      l'entraînement. Toutes les courbes/tracés/tableaux ci-dessous portent  ║
║      désormais sur ce TEST gelé, pas sur les résultats d'entraînement.     ║
║                                                                              ║
║  Situation 1 = LNS + DQN(entraîné sur synthétique) + Gurobi, ÉVALUÉ sur     ║
║                Solomon (test)                                → gap vs BKS   ║
║  (La "Situation 2 = Gurobi seul sur l'instance complète" reste retirée,     ║
║   à la demande, car sans limite de temps elle est beaucoup trop longue      ║
║   sur des instances à 100 clients avec fenêtres de temps serrées.)          ║
║                                                                              ║
║  Visualisations produites (toutes sur le TEST gelé, sauf la 1ère) :        ║
║    [A] Convergence de l'ENTRAÎNEMENT (synthétique) : reward global +       ║
║        amélioration % par instance synthétique                            ║
║    [B] Convergence détaillée (instances Solomon exemples, TEST) : coût     ║
║        final/meilleur par épisode local + trajectoire de coût             ║
║    [C] Tracés des routes AVANT (LNS) / APRÈS (DQN gelé) pour des           ║
║        instances Solomon exemples                                         ║
║    [D] Tableau récapitulatif : Instance / Nb véhicules / Distance / Gap %  ║
║        (TEST Solomon vs BKS)                                              ║
║    [E] Graphique dédié gap % vs BKS par instance + gap moyen              ║
║    [F] Diagnostic automatique de licence Gurobi (taille limitée ou non)   ║
║                                                                              ║
║  Aucun rappel manuel de "Code 1" n'est nécessaire : la génération +        ║
║  l'entraînement synthétique sont réintégrés ici (MODULE 1bis) et           ║
║  s'exécutent automatiquement avant le chargement de Solomon. Voir le flag  ║
║  optionnel CFG['REUSE_EXISTING_SYNTH_MODEL'] plus bas si vous préférez     ║
║  recharger un .pth déjà entraîné au lieu de ré-entraîner à chaque run.     ║
║                                                                              ║
║  Par défaut, INSTANCE_FILES contient toutes les instances *.txt trouvées   ║
║  dans outputs/solomon56/. Pour se limiter à 6 instances représentatives    ║
║  (une par groupe C1/C2/R1/R2/RC1/RC2), décommentez le bloc plus bas.       ║
║                                                                              ║
║  Exécution :  python main_vrptw_synth_train_solomon_convergence.py         ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import os, sys, json, time, copy, random, csv, glob
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, deque
import torch
import torch.nn as nn
import torch.optim as optim
import gurobipy as gp
from gurobipy import GRB
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG GLOBALE
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    # ── ÉTAPE 1 : ENTRAÎNEMENT sur instances SYNTHÉTIQUES (jamais Solomon) ──
    N_SYNTH_INSTANCES          = 80,     # comme les 80 instances de Code 1
    SYNTH_N_CUSTOMERS          = 100,
    SYNTH_N_VEHICLES           = None,   # None → calculé depuis la demande (cf. generate_instance)
    SYNTH_VEHICLE_SAFETY_FACTOR= 1.5,
    SYNTH_CAPACITY             = 200,
    SYNTH_HORIZON              = 1000,
    EPISODES_PER_INSTANCE      = 10,     # épisodes d'ENTRAÎNEMENT par instance synthétique
    # Optionnel : True → recharge un .pth déjà entraîné sur du synthétique
    # (voir MODEL_FILE) au lieu de régénérer + ré-entraîner. False (par
    # défaut) → tout se fait automatiquement dans CE run, sans intervention
    # manuelle, aucun besoin de relancer "Code 1" séparément.
    REUSE_EXISTING_SYNTH_MODEL = False,
    # ── ÉTAPE 2 : TEST gelé sur les 56 instances Solomon officielles ───────
    EPISODES_PER_INSTANCE_TEST = 3,      # épisodes d'ÉVALUATION (aucun apprentissage) par instance Solomon
    # VEHICLE_SAFETY_FACTOR = None : on utilise la flotte OFFICIELLE de chaque
    # instance, pas une borne calculée sur la demande. Sur R101/RC101
    # (fenêtres de temps serrées), le BKS utilise 19 et 14 véhicules alors
    # que la demande seule suggérerait ~12-14 : une borne basée uniquement
    # sur la demande sous-estime le besoin réel et peut rendre le MILP
    # infaisable. La flotte officielle est plus grande (MILP plus lourd)
    # mais garantit qu'une solution proche du BKS reste atteignable.
    VEHICLE_SAFETY_FACTOR  = None,
    LNS_ITER               = 30,
    LNS_K_RATIO            = 0.20,
    K_DESTROY_RATIO        = 0.24,
    MAX_STEPS              = 20,
    # repair par bouts (Situation 1, appelé à CHAQUE step du LNS/DQN — voir
    # check_gurobi_license() plus bas pour l'impact d'une licence à taille
    # limitée). None = aucune limite de temps (supporté par le code).
    GUROBI_TLIMIT           = 2.0,
    GUROBI_GAP              = 0.05,
    HIDDEN                  = 64,
    LR                      = 1e-3,
    GAMMA                   = 0.95,
    EPS_START                = 1.0,
    EPS_END                  = 0.05,
    EPS_DECAY                = 0.95,  # decay plus rapide : peu d'instances/épisodes ici
    BUFFER_SIZE               = 5000,
    BATCH_SIZE                = 32,
    TARGET_UPDATE              = 10,
    # ── Instances exemples (pour courbes détaillées + routes, TEST/Solomon) ─
    # Noms recherchés en priorité (représentatifs des 6 groupes Solomon) ;
    # si un nom n'est pas présent dans INSTANCE_FILES, il est ignoré et
    # remplacé automatiquement par le 1er type disponible.
    EXAMPLE_NAMES              = ['C101', 'R101', 'RC101'],
    OUTDIR                        = os.path.join(os.getcwd(), 'outputs'),
    MODELDIR                       = os.path.join(os.getcwd(), 'outputs', 'model'),
    MODEL_FILE                      = 'dqn_vrptw_synth_trained.pth',
)
CFG['LNS_K'] = 20   # ~20% de 100 clients, cohérent avec LNS_K_RATIO
CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))  # 24, cohérent avec K_DESTROY_RATIO
os.makedirs(CFG['OUTDIR'],   exist_ok=True)
os.makedirs(CFG['MODELDIR'], exist_ok=True)
ACTION_NAMES = {
    0: "Destruction spatiale",
    1: "Destruction temporelle",
    2: "Destruction par coût élevé",
    3: "Destruction aléatoire",
}
# ── Dossier contenant les instances Solomon officielles (extraites du zip) ──
SOLOMON_DIR = os.path.join(CFG['OUTDIR'], 'Solomon')
# Toutes les instances *.txt trouvées (décommentez le bloc ci-dessous pour
# revenir à seulement 6 instances représentatives, une par groupe
# C1/C2/R1/R2/RC1/RC2, afin de garder un temps d'exécution raisonnable).
INSTANCE_FILES = sorted(glob.glob(os.path.join(SOLOMON_DIR, '*.txt')))
# INSTANCE_FILES = [
#     os.path.join(SOLOMON_DIR, 'c101.txt'),
#     os.path.join(SOLOMON_DIR, 'c201.txt'),
#     os.path.join(SOLOMON_DIR, 'r101.txt'),
#     os.path.join(SOLOMON_DIR, 'r201.txt'),
#     os.path.join(SOLOMON_DIR, 'rc101.txt'),
#     os.path.join(SOLOMON_DIR, 'rc201.txt'),
# ]
# ── BKS officiel complet, 56 instances (source : SINTEF / Solomon 1987) ──────
BKS_REFERENCE = {
    'C101': {'vehicles': 10, 'distance': 828.94}, 'C102': {'vehicles': 10, 'distance': 828.94},
    'C103': {'vehicles': 10, 'distance': 828.06}, 'C104': {'vehicles': 10, 'distance': 824.78},
    'C105': {'vehicles': 10, 'distance': 828.94}, 'C106': {'vehicles': 10, 'distance': 828.94},
    'C107': {'vehicles': 10, 'distance': 828.94}, 'C108': {'vehicles': 10, 'distance': 828.94},
    'C109': {'vehicles': 10, 'distance': 828.94},
    'C201': {'vehicles': 3, 'distance': 591.56}, 'C202': {'vehicles': 3, 'distance': 591.56},
    'C203': {'vehicles': 3, 'distance': 591.17}, 'C204': {'vehicles': 3, 'distance': 590.60},
    'C205': {'vehicles': 3, 'distance': 588.88}, 'C206': {'vehicles': 3, 'distance': 588.49},
    'C207': {'vehicles': 3, 'distance': 588.29}, 'C208': {'vehicles': 3, 'distance': 588.32},
    'R101': {'vehicles': 19, 'distance': 1650.80}, 'R102': {'vehicles': 17, 'distance': 1486.12},
    'R103': {'vehicles': 13, 'distance': 1292.68}, 'R104': {'vehicles': 9,  'distance': 1007.31},
    'R105': {'vehicles': 14, 'distance': 1377.11}, 'R106': {'vehicles': 12, 'distance': 1252.03},
    'R107': {'vehicles': 10, 'distance': 1104.66}, 'R108': {'vehicles': 9,  'distance': 960.88},
    'R109': {'vehicles': 11, 'distance': 1194.73}, 'R110': {'vehicles': 10, 'distance': 1118.84},
    'R111': {'vehicles': 10, 'distance': 1096.72}, 'R112': {'vehicles': 9,  'distance': 982.14},
    'R201': {'vehicles': 4, 'distance': 1252.37}, 'R202': {'vehicles': 3, 'distance': 1191.70},
    'R203': {'vehicles': 3, 'distance': 939.50},  'R204': {'vehicles': 2, 'distance': 825.52},
    'R205': {'vehicles': 3, 'distance': 994.43},  'R206': {'vehicles': 3, 'distance': 906.14},
    'R207': {'vehicles': 2, 'distance': 890.61},  'R208': {'vehicles': 2, 'distance': 726.82},
    'R209': {'vehicles': 3, 'distance': 909.16},  'R210': {'vehicles': 3, 'distance': 939.37},
    'R211': {'vehicles': 2, 'distance': 885.71},
    'RC101': {'vehicles': 14, 'distance': 1696.95}, 'RC102': {'vehicles': 12, 'distance': 1554.75},
    'RC103': {'vehicles': 11, 'distance': 1261.67}, 'RC104': {'vehicles': 10, 'distance': 1135.48},
    'RC105': {'vehicles': 13, 'distance': 1629.44}, 'RC106': {'vehicles': 11, 'distance': 1424.73},
    'RC107': {'vehicles': 11, 'distance': 1230.48}, 'RC108': {'vehicles': 10, 'distance': 1139.82},
    'RC201': {'vehicles': 4, 'distance': 1406.94}, 'RC202': {'vehicles': 3, 'distance': 1365.65},
    'RC203': {'vehicles': 3, 'distance': 1049.62}, 'RC204': {'vehicles': 3, 'distance': 798.46},
    'RC205': {'vehicles': 4, 'distance': 1297.65}, 'RC206': {'vehicles': 3, 'distance': 1146.32},
    'RC207': {'vehicles': 3, 'distance': 1061.14}, 'RC208': {'vehicles': 3, 'distance': 828.14},
}
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1  —  LECTURE DES INSTANCES SOLOMON OFFICIELLES  (jeu de TEST)
# ══════════════════════════════════════════════════════════════════════════════
def euclidean(a, b):
    return float(np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2))
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1bis  —  GÉNÉRATEUR D'INSTANCES SYNTHÉTIQUES  (jeu d'ENTRAÎNEMENT)
#  Repris tel quel de main_vrptw_comparaison_gurobi.py ("Code 1") : mêmes
#  fonctions _coords_C/_coords_R/_coords_RC et generate_instance(), pour que
#  les instances d'entraînement soient générées exactement comme dans Code 1.
# ══════════════════════════════════════════════════════════════════════════════
def _coords_C(n, rng):
    n_cl = max(3, n // 10)
    centers = rng.uniform(15, 85, (n_cl, 2))
    pts = []
    for i in range(n):
        p = rng.normal(centers[i % n_cl], 8)
        pts.append(np.clip(p, 1, 99))
    return np.array(pts)
def _coords_R(n, rng):
    return rng.uniform(1, 99, (n, 2))
def _coords_RC(n, rng):
    h = n // 2
    return np.vstack([_coords_C(h, rng), _coords_R(n - h, rng)])
def generate_instance(n_customers=50, n_vehicles=None, capacity=200,
                      type_='C', seed=42, horizon=1000,
                      vehicle_safety_factor=1.5):
    rng     = np.random.RandomState(seed)
    depot   = np.array([50.0, 50.0])
    gen_fn  = {'C': _coords_C, 'R': _coords_R, 'RC': _coords_RC}[type_]
    coords  = gen_fn(n_customers, rng)
    max_dem = max(1, capacity // 5)
    demands = rng.randint(1, max_dem + 1, n_customers)
    svc     = rng.randint(10, 31, n_customers)
    if n_vehicles is None:
        min_vehicles = int(np.ceil(demands.sum() / capacity))
        n_vehicles   = max(2, int(np.ceil(min_vehicles * vehicle_safety_factor)))
    customers = []
    for i in range(n_customers):
        d        = euclidean(depot, coords[i])
        earliest = max(0.0, round(d + rng.uniform(-15, 15), 1))
        delta    = rng.uniform(30, 80)
        latest   = min(float(horizon), round(earliest + delta, 1))
        latest   = max(latest, earliest + svc[i] + 5)
        latest   = min(latest, float(horizon))
        if d > latest:
            earliest = round(d, 1)
            latest   = min(float(horizon), round(earliest + delta, 1))
            latest   = max(latest, earliest + svc[i] + 5)
        if d + svc[i] + d > horizon:
            svc[i] = max(5, int(horizon - 2 * d - 5))
        customers.append({
            'id':           i + 1,
            'x':            float(round(coords[i][0], 2)),
            'y':            float(round(coords[i][1], 2)),
            'demand':       int(demands[i]),
            'earliest':     float(earliest),
            'latest':       float(latest),
            'service_time': int(svc[i]),
        })
    return {
        'depot': {
            'id': 0, 'x': 50.0, 'y': 50.0, 'demand': 0,
            'earliest': 0.0, 'latest': float(horizon), 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    capacity,
        'horizon':     horizon,
        'type':        type_,
        'seed':        seed,
        'n_customers': n_customers,
    }
def generate_synthetic_training_set(cfg, verbose=True):
    """
    Génère les instances synthétiques d'ENTRAÎNEMENT en cyclant les 3 types
    Solomon C / R / RC (même logique que train() dans Code 1 : type =
    ['C','R','RC'][i % 3], seed = i*7+1). Ces instances ne recouvrent JAMAIS
    les 56 instances Solomon officielles utilisées plus bas comme jeu de
    TEST : le DQN ne les voit donc jamais avant l'évaluation gelée.
    Chaque instance reçoit en plus une clé 'name' (ex. 'SYN-C001') pour
    rester compatible avec les fonctions d'entraînement/affichage qui
    utilisent inst['name'].
    """
    types = ['C', 'R', 'RC']
    instances = []
    if verbose:
        _sec(f"ÉTAPE 1 — GÉNÉRATION DE {cfg['N_SYNTH_INSTANCES']} INSTANCES "
             f"SYNTHÉTIQUES (ENTRAÎNEMENT)")
    for i in range(cfg['N_SYNTH_INSTANCES']):
        t    = types[i % 3]
        seed = i * 7 + 1
        inst = generate_instance(
            n_customers = cfg['SYNTH_N_CUSTOMERS'],
            n_vehicles  = cfg['SYNTH_N_VEHICLES'],
            capacity    = cfg['SYNTH_CAPACITY'],
            horizon     = cfg['SYNTH_HORIZON'],
            type_       = t,
            seed        = seed,
            vehicle_safety_factor = cfg['SYNTH_VEHICLE_SAFETY_FACTOR'],
        )
        inst['name'] = f"SYN-{t}{i+1:03d}"
        instances.append(inst)
        if verbose:
            print(f"  Instance {i+1:03d}/{cfg['N_SYNTH_INSTANCES']} | {inst['name']:<10} "
                  f"type={t:<3} | seed={seed} | véhicules={inst['n_vehicles']}")
    return instances
def parse_solomon_instance(filepath, vehicle_safety_factor=None):
    """
    Parse un fichier d'instance Solomon au format texte officiel :
        <NOM>
        VEHICLE
        NUMBER  CAPACITY
        <n>     <cap>
        CUSTOMER
        CUST NO.  XCOORD.  YCOORD.  DEMAND  READY TIME  DUE DATE  SERVICE TIME
        0  ...   (dépôt)
        1  ...
        ...
    Retourne un dict compatible avec le reste du pipeline.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = [l.rstrip('\n') for l in f]
    name = lines[0].strip()
    idx_veh = next(i for i, l in enumerate(lines) if l.strip() == 'VEHICLE')
    veh_number, veh_capacity = map(int, lines[idx_veh + 2].split())
    idx_cust = next(i for i, l in enumerate(lines) if l.strip() == 'CUSTOMER')
    data_lines = [l for l in lines[idx_cust + 2:] if l.strip()]
    rows = [list(map(float, l.split())) for l in data_lines]
    depot_row = rows[0]
    customers = []
    for r in rows[1:]:
        cid, x, y, dem, ready, due, svc = r
        customers.append({
            'id':           int(cid),
            'x':            x,
            'y':            y,
            'demand':       int(dem),
            'earliest':     ready,
            'latest':       due,
            'service_time': int(svc),
        })
    name_upper = name.upper()
    type_ = ''.join(ch for ch in name_upper if ch.isalpha())   # 'C101' -> 'C', 'RC101' -> 'RC'
    n_vehicles = veh_number
    if vehicle_safety_factor is not None:
        total_demand = sum(c['demand'] for c in customers)
        min_veh      = int(np.ceil(total_demand / veh_capacity))
        n_vehicles   = min(veh_number, max(2, int(np.ceil(min_veh * vehicle_safety_factor))))
    return {
        'depot': {
            'id': 0, 'x': depot_row[1], 'y': depot_row[2], 'demand': 0,
            'earliest': depot_row[4], 'latest': depot_row[5], 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    veh_capacity,
        'horizon':     depot_row[5],
        'type':        type_,
        'seed':        name_upper,
        'n_customers': len(customers),
        'name':        name_upper,          # nom officiel normalisé (ex. 'C101')
    }
def load_all_instances(files, vehicle_safety_factor=None):
    if not files:
        raise FileNotFoundError(
            f"Aucun fichier d'instance trouvé dans {SOLOMON_DIR}. "
            f"Vérifiez que le dossier existe et contient les .txt Solomon."
        )
    instances = []
    for fp in files:
        inst = parse_solomon_instance(fp, vehicle_safety_factor=vehicle_safety_factor)
        instances.append(inst)
        print(f"  Chargé : {inst['name']:<8} type={inst['type']:<3} "
              f"clients={inst['n_customers']} véhicules={inst['n_vehicles']} "
              f"capacité={inst['capacity']} horizon={inst['horizon']}")
    return instances
def pick_example_indices(instances, preferred_names=None, max_examples=3):
    """
    Sélectionne des instances "exemples" pour les courbes détaillées et les
    tracés de routes. Cherche d'abord les noms préférés (ex. C101/R101/RC101).
    Si absents, retombe sur la 1ère instance de chaque type détecté, puis sur
    les toutes premières instances de la liste.
    """
    preferred_names = preferred_names or []
    by_name = {inst['name']: i for i, inst in enumerate(instances)}
    idxs = [by_name[nm] for nm in preferred_names if nm in by_name]
    if not idxs:
        seen_types = {}
        for i, inst in enumerate(instances):
            if inst['type'] not in seen_types:
                seen_types[inst['type']] = i
            if len(seen_types) >= max_examples:
                break
        idxs = list(seen_types.values())
    if not idxs:
        idxs = list(range(min(max_examples, len(instances))))
    return idxs[:max_examples]
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 2  —  SOLUTION INITIALE PAR LNS  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
def _node_xy(nid, instance):
    if nid == 0:
        return instance['depot']['x'], instance['depot']['y']
    c = instance['customers'][nid - 1]
    return c['x'], c['y']
def _node_info(nid, instance):
    if nid == 0:
        return instance['depot']
    return instance['customers'][nid - 1]
def compute_cost(solution, instance):
    depot = (instance['depot']['x'], instance['depot']['y'])
    total = 0.0
    for route in solution:
        if not route:
            continue
        prev = depot
        for cid in route:
            cur   = _node_xy(cid, instance)
            total += euclidean(prev, cur)
            prev  = cur
        total += euclidean(prev, depot)
    return total
def _route_feasible(route, instance):
    if not route:
        return True
    cap  = instance['capacity']
    load = sum(_node_info(c, instance)['demand'] for c in route)
    if load > cap:
        return False
    depot = instance['depot']
    t     = 0.0
    px, py = depot['x'], depot['y']
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        if t > info['latest']:
            return False
        t      = max(t, info['earliest']) + info['service_time']
        px, py = cx, cy
    t += euclidean((px, py), (depot['x'], depot['y']))
    return t <= depot['latest']
def is_feasible(solution, instance):
    return all(_route_feasible(r, instance) for r in solution)
def solution_stats(solution, instance):
    cap        = instance['capacity']
    total_cost = compute_cost(solution, instance)
    n_vehicles = sum(1 for r in solution if r)
    loads      = []
    total_wait = 0.0
    for route in solution:
        if not route:
            continue
        load = sum(_node_info(c, instance)['demand'] for c in route)
        loads.append(load / cap)
        depot  = instance['depot']
        t      = 0.0
        px, py = depot['x'], depot['y']
        for cid in route:
            cx, cy = _node_xy(cid, instance)
            t     += euclidean((px, py), (cx, cy))
            info   = _node_info(cid, instance)
            total_wait += max(0.0, info['earliest'] - t)
            t      = max(t, info['earliest']) + info['service_time']
            px, py = cx, cy
    return {
        'total_cost':         round(total_cost, 2),
        'n_vehicles':         n_vehicles,
        'avg_load_ratio':     round(np.mean(loads) if loads else 0.0, 3),
        'total_wait_time':    round(total_wait, 2),
        'n_customers_served': sum(len(r) for r in solution),
    }
def _insertion_cost(cid, pos, route, instance):
    depot   = (instance['depot']['x'], instance['depot']['y'])
    nodes   = [0] + route + [0]
    prev_id = nodes[pos]
    next_id = nodes[pos + 1]
    px, py  = _node_xy(prev_id, instance) if prev_id != 0 else depot
    nx, ny  = _node_xy(next_id, instance) if next_id != 0 else depot
    cx, cy  = _node_xy(cid, instance)
    return (euclidean((px, py), (cx, cy)) + euclidean((cx, cy), (nx, ny))
            - euclidean((px, py), (nx, ny)))
def _try_insert(cid, route, instance):
    best_cost, best_pos = float('inf'), -1
    for pos in range(len(route) + 1):
        candidate = route[:pos] + [cid] + route[pos:]
        if _route_feasible(candidate, instance):
            cost = _insertion_cost(cid, pos, route, instance)
            if cost < best_cost:
                best_cost, best_pos = cost, pos
    return best_cost, best_pos
def _repair_greedy(removed, partial, instance):
    routes = [list(r) for r in partial]
    for cid in removed:
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _build_initial(instance):
    customers = sorted(instance['customers'], key=lambda c: c['earliest'])
    routes    = []
    for c in customers:
        cid = c['id']
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _lns_improve(solution, instance, n_iter=50, k=10, seed=0):
    rng       = np.random.RandomState(seed)
    current   = [list(r) for r in solution]
    best      = copy.deepcopy(current)
    best_cost = compute_cost(best, instance)
    for _ in range(n_iter):
        all_c   = [c for r in current for c in r]
        k_      = min(k, max(1, len(all_c) - 1))
        removed = list(rng.choice(all_c, k_, replace=False))
        removed_set = set(removed)
        partial = [[c for c in r if c not in removed_set] for r in current]
        partial = [r for r in partial if r]
        candidate = _repair_greedy(removed, partial, instance)
        c_cost    = compute_cost(candidate, instance)
        if c_cost < best_cost and is_feasible(candidate, instance):
            best, best_cost = copy.deepcopy(candidate), c_cost
        current = candidate
    return best
def get_initial_solution(instance, lns_iter=50, lns_k=10):
    sol = _build_initial(instance)
    sol = _lns_improve(sol, instance, n_iter=lns_iter, k=lns_k)
    sol = [r for r in sol if r]
    return sol, solution_stats(sol, instance)
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 3  —  ENVIRONNEMENT VRPTW  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class VRPTWEnv:
    def __init__(self, instance, initial_solution,
                 K=15, max_steps=20,
                 gurobi_tlimit=2.0, gurobi_gap=0.05,
                 verbose=False):
        self.instance    = instance
        self.initial_sol = copy.deepcopy(initial_solution)
        self.K           = K
        self.max_steps   = max_steps
        self.tlimit      = gurobi_tlimit
        self.gap         = gurobi_gap
        self.verbose     = verbose
        self.obs_dim     = 7
        self.n_actions   = 4
        self._dist = {}
        depot = (instance['depot']['x'], instance['depot']['y'])
        for c in instance['customers']:
            d = euclidean(depot, (c['x'], c['y']))
            self._dist[(0, c['id'])] = d
            self._dist[(c['id'], 0)] = d
        for ci in instance['customers']:
            for cj in instance['customers']:
                self._dist[(ci['id'], cj['id'])] = euclidean(
                    (ci['x'], ci['y']), (cj['x'], cj['y']))
        self.reset()
    def _d(self, a, b):
        return self._dist.get((a, b), 0.0)
    def _info(self, nid):
        return _node_info(nid, self.instance)
    def reset(self, start_solution=None):
        if start_solution is not None:
            self.solution = copy.deepcopy(start_solution)
        else:
            self.solution = copy.deepcopy(self.initial_sol)
        self.current_cost     = compute_cost(self.solution, self.instance)
        self.best_cost        = self.current_cost
        self.initial_cost     = compute_cost(self.initial_sol, self.instance)
        self.step_count       = 0
        self.steps_no_improve = 0
        self._action_history  = []
        return self._obs()
    def _obs(self):
        stats = solution_stats(self.solution, self.instance)
        ic    = max(self.initial_cost, 1.0)
        return np.array([
            self.current_cost / ic,
            stats['n_vehicles'] / max(self.instance['n_vehicles'], 1),
            stats['avg_load_ratio'],
            stats['total_wait_time'] / max(ic * 0.5, 1.0),
            self.steps_no_improve / self.max_steps,
            self.best_cost / ic,
            1.0,
        ], dtype=np.float32)
    def _destroy(self, action):
        all_c = [c for r in self.solution for c in r]
        if not all_c:
            return [], copy.deepcopy(self.solution)
        K   = min(self.K, len(all_c))
        rng = np.random.RandomState(self.step_count + action * 100)
        if action == 0:
            pivot   = int(rng.choice(all_c))
            px, py  = self._info(pivot)['x'], self._info(pivot)['y']
            dists   = {c: euclidean((px, py),
                                    (self._info(c)['x'], self._info(c)['y']))
                       for c in all_c}
            destroyed = sorted(all_c, key=lambda c: dists[c])[:K]
        elif action == 1:
            pivot    = int(rng.choice(all_c))
            e_pivot  = self._info(pivot)['earliest']
            tdiffs   = {c: abs(self._info(c)['earliest'] - e_pivot) for c in all_c}
            destroyed = sorted(all_c, key=lambda c: tdiffs[c])[:K]
        elif action == 2:
            savings = {}
            for r in self.solution:
                if not r:
                    continue
                nodes = [0] + r + [0]
                for i, cid in enumerate(r):
                    prev, nxt     = nodes[i], nodes[i + 2]
                    savings[cid]  = (self._d(prev, cid) + self._d(cid, nxt)
                                     - self._d(prev, nxt))
            destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
        else:
            idx       = rng.choice(len(all_c), K, replace=False)
            destroyed = [all_c[i] for i in idx]
        destroyed_set = set(destroyed)
        partial = [[c for c in r if c not in destroyed_set]
                   for r in self.solution]
        partial = [r for r in partial if r]
        return list(destroyed), partial
    def _repair_gurobi(self, destroyed, partial):
        if not destroyed:
            return copy.deepcopy(self.solution)
        inst    = self.instance
        depot   = inst['depot']
        horizon = inst['horizon']
        cap     = inst['capacity']
        nodes   = [0] + destroyed
        N       = len(nodes)
        V       = min(len(destroyed), inst['n_vehicles'])
        env, m = None, None
        try:
            env = gp.Env(empty=True)
            env.setParam('OutputFlag', 0)
            env.start()
            m = gp.Model(env=env)
            if self.tlimit is not None:
                m.setParam('TimeLimit', self.tlimit)
            m.setParam('MIPGap',     self.gap)
            m.setParam('OutputFlag', 0)
            x = m.addVars(N, N, V, vtype=GRB.BINARY, name='x')
            t = m.addVars(N, V, lb=0, ub=horizon,    name='t')
            m.setObjective(
                gp.quicksum(
                    self._d(nodes[i], nodes[j]) * x[i, j, v]
                    for i in range(N) for j in range(N) for v in range(V)
                    if i != j
                ),
                GRB.MINIMIZE
            )
            for i in range(1, N):
                m.addConstr(
                    gp.quicksum(x[i, j, v]
                                for j in range(N) for v in range(V)
                                if i != j) == 1
                )
            for v in range(V):
                for i in range(1, N):
                    m.addConstr(
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j) ==
                        gp.quicksum(x[j, i, v] for j in range(N) if i != j)
                    )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(x[0, j, v] for j in range(1, N)) <= 1
                )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(
                        inst['customers'][nodes[i] - 1]['demand'] *
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j)
                        for i in range(1, N)
                    ) <= cap
                )
            M = horizon + 1000
            for v in range(V):
                m.addConstr(t[0, v] == 0)
                for i in range(N):
                    ni  = nodes[i]
                    ei  = depot['earliest'] if ni == 0 else inst['customers'][ni-1]['earliest']
                    li  = depot['latest']   if ni == 0 else inst['customers'][ni-1]['latest']
                    sti = depot['service_time'] if ni == 0 else inst['customers'][ni-1]['service_time']
                    m.addConstr(t[i, v] >= ei)
                    m.addConstr(t[i, v] <= li)
                    for j in range(N):
                        if i == j:
                            continue
                        m.addConstr(
                            t[j, v] >= t[i, v] + sti + self._d(ni, nodes[j])
                            - M * (1 - x[i, j, v])
                        )
            m.optimize()
            if m.SolCount > 0:
                eps        = 0.5
                new_routes = []
                for v in range(V):
                    if sum(x[0, j, v].X for j in range(1, N)) < eps:
                        continue
                    route_v, cur = [], 0
                    for _ in range(N):
                        nxt = next((j for j in range(N)
                                    if j != cur and x[cur, j, v].X > eps), None)
                        if nxt is None or nxt == 0:
                            break
                        route_v.append(nodes[nxt])
                        cur = nxt
                    if route_v:
                        new_routes.append(route_v)
                full = [r for r in list(partial) + new_routes if r]
                return full
            else:
                raise ValueError("Gurobi : aucune solution trouvée")
        except Exception as e:
            if self.verbose:
                print(f"    [Gurobi→Greedy fallback] {e}")
            return _repair_greedy(destroyed, partial, self.instance)
        finally:
            if m is not None:
                m.dispose()
            if env is not None:
                env.dispose()
    def step(self, action):
        old_cost            = self.current_cost
        destroyed, partial  = self._destroy(action)
        new_sol             = self._repair_gurobi(destroyed, partial)
        new_cost            = compute_cost(new_sol, self.instance)
        feasible            = is_feasible(new_sol, self.instance)
        if feasible:
            self.solution     = new_sol
            self.current_cost = new_cost
        else:
            new_cost = old_cost * 1.001
        improvement = old_cost - self.current_cost
        if improvement > 0:
            self.best_cost        = min(self.best_cost, self.current_cost)
            self.steps_no_improve = 0
        else:
            self.steps_no_improve += 1
        reward = improvement / max(self.initial_cost, 1.0)
        self.step_count += 1
        self._action_history.append(action)
        done = (self.step_count >= self.max_steps)
        info = {
            'cost':          self.current_cost,
            'best_cost':     self.best_cost,
            'action_taken':  action,
            'action_name':   ACTION_NAMES[action],
            'improvement':   improvement,
            'destroyed_k':   len(destroyed),
            'destroyed_ids': [int(d) for d in destroyed[:5]],
            'feasible':      feasible,
        }
        return self._obs(), float(reward), done, info
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 4  —  AGENT DQN  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((
            np.array(s,  dtype=np.float32), int(a), float(r),
            np.array(s2, dtype=np.float32), float(done),
        ))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.FloatTensor(np.stack(s)), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(np.stack(s2)),
                torch.FloatTensor(d))
    def __len__(self):
        return len(self.buf)
class DQNNet(nn.Module):
    def __init__(self, obs_dim=7, n_actions=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim,  hidden),  nn.ReLU(),
            nn.Linear(hidden,   hidden),  nn.ReLU(),
            nn.Linear(hidden,   n_actions),
        )
    def forward(self, x):
        return self.net(x)
class DQNAgent:
    def __init__(self, obs_dim=7, n_actions=4, hidden=64,
                 lr=1e-3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_size=2000, batch_size=32,
                 target_update_freq=10, seed=0):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        self.n_actions          = n_actions
        self.gamma              = gamma
        self.epsilon            = epsilon_start
        self.epsilon_end        = epsilon_end
        self.epsilon_decay      = epsilon_decay
        self.batch_size         = batch_size
        self.target_update_freq = target_update_freq
        self.step_count         = 0
        self.q_net      = DQNNet(obs_dim, n_actions, hidden)
        self.target_net = DQNNet(obs_dim, n_actions, hidden)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)
        self.loss_fn   = nn.MSELoss()
    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state).unsqueeze(0))
            return int(q.argmax().item())
    def store(self, s, a, r, s2, done):
        self.buffer.push(s, a, r, s2, done)
    def learn(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        s, a, r, s2, done = self.buffer.sample(self.batch_size)
        q_vals = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q  = self.target_net(s2).max(1)[0]
            targets = r + self.gamma * next_q * (1 - done)
        loss = self.loss_fn(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.optimizer.step()
        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.update_target()
        return float(loss.item())
    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    def save(self, path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        torch.save({'q_net': self.q_net.state_dict(),
                    'target_net': self.target_net.state_dict(),
                    'epsilon': self.epsilon, 'step_count': self.step_count}, path)
        print(f"    [DQN] Modèle sauvegardé → {path}")
    def load(self, path):
        """
        Recharge un checkpoint sauvegardé par save() (mêmes clés). Utilisé
        uniquement si CFG['REUSE_EXISTING_SYNTH_MODEL']=True, pour réutiliser
        un modèle déjà entraîné sur du synthétique au lieu de ré-entraîner.
        """
        ckpt = torch.load(path, map_location='cpu')
        self.q_net.load_state_dict(ckpt['q_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.epsilon    = ckpt.get('epsilon', self.epsilon)
        self.step_count = ckpt.get('step_count', self.step_count)
        print(f"    [DQN] Modèle rechargé ← {path}")
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5  —  ENTRAÎNEMENT DU DQN  (ÉTAPE 1, sur les instances SYNTHÉTIQUES)
#  Stocke cost_trace (trajectoire de coût step par step) dans le log, utilisé
#  ensuite pour les courbes de convergence détaillées. Retourne aussi
#  initial_sols pour les tracés de routes avant/après.
# ══════════════════════════════════════════════════════════════════════════════
BANNER = "═" * 72
def _sec(title):
    print(f"\n{BANNER}\n  {title}\n{BANNER}")
def train_on_instances(instances, episodes_per_instance=10):
    """
    ENTRAÎNEMENT (les poids du DQN sont mis à jour ici via store()/learn()/
    decay_epsilon()). Générique : fonctionne sur n'importe quelle liste
    d'instances au format standard — dans ce script, appelé avec les
    instances SYNTHÉTIQUES produites par generate_synthetic_training_set(),
    JAMAIS avec les instances Solomon (réservées à evaluate_frozen_
    on_instances(), plus bas, pour le TEST).
    Retourne : agent, training_log, initial_costs, best_costs, best_solutions,
               instance_times, initial_sols
    """
    n = len(instances)
    _sec(f"ÉTAPE 1 — LNS initial pour les {n} instances d'entraînement (synthétiques)")
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    _sec("ÉTAPE 1 — Entraînement DQN sur le synthétique")
    agent = DQNAgent(
        obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
        gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
        epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
        buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
        target_update_freq=CFG['TARGET_UPDATE'],
    )
    training_log   = []
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    episode_global = 0
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TRAIN] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)
                next_obs, reward, done, info = env.step(action)
                agent.store(obs, action, reward, next_obs, float(done))
                agent.learn()
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            agent.decay_epsilon()
            training_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TRAIN] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    agent.save(model_path)
    return agent, training_log, initial_costs, best_costs, best_solutions, instance_times, initial_sols
def evaluate_frozen_on_instances(instances, agent, episodes_per_instance=3):
    """
    TEST — ÉVALUATION GELÉE : l'agent DQN déjà entraîné (ÉTAPE 1, sur le
    synthétique) est ici appliqué en pur mode INFÉRENCE sur les instances
    passées en argument (les 56 Solomon officielles). Différences avec
    train_on_instances() :
      - AUCUN appel à agent.store() / agent.learn() / agent.decay_epsilon() :
        les poids du réseau ne sont JAMAIS mis à jour ici.
      - epsilon est temporairement forcé à 0.0 (politique gloutonne, pure
        exploitation du Q-réseau appris), puis restauré à sa valeur
        d'origine à la fin.
      - Le modèle n'est PAS sauvegardé ici (rien n'a changé dans ses poids).
    Le log retourné (test_log) a exactement la même forme que training_log
    (y compris 'cost_trace'), pour pouvoir réutiliser telles quelles les
    fonctions de tracé (plot_instance_convergence_examples, etc.) sur le
    TEST plutôt que sur l'entraînement.
    Retourne : initial_costs, best_costs, best_solutions, instance_times,
               initial_sols, test_log
    """
    n = len(instances)
    _sec(f"ÉTAPE 2 — ÉVALUATION GELÉE (TEST) sur {n} instances Solomon "
         f"— aucun apprentissage, epsilon=0")
    saved_epsilon = agent.epsilon
    agent.epsilon = 0.0
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    test_log       = []
    episode_global = 0
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TEST] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)          # inférence pure (epsilon=0)
                next_obs, reward, done, info = env.step(action)
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            test_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TEST] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    agent.epsilon = saved_epsilon
    return initial_costs, best_costs, best_solutions, instance_times, initial_sols, test_log
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6  —  TABLEAU FINAL : SITUATION 1 vs BKS
#  (la Situation 2 "Gurobi seul sur l'instance complète" a été retirée)
# ══════════════════════════════════════════════════════════════════════════════
def build_bks_comparison_table(instances, initial_costs, best_costs, best_solutions,
                                instance_times, csv_path=None, latex_path=None):
    """
    Tableau final : Situation 1 (LNS+DQN+Gurobi) vs BKS, avec gap % et le
    nombre de véhicules utilisés pour chaque instance.
    """
    _sec("TABLEAU FINAL — SITUATION 1 vs BKS")
    rows = []
    for i, inst in enumerate(instances):
        name     = inst['name']
        bks      = BKS_REFERENCE.get(name)
        bks_dist = bks['distance'] if bks else None
        n_veh_s1 = sum(1 for r in best_solutions[i] if r)
        row = {
            'instance':             name,
            'type':                 inst['type'],
            'bks_vehicules':        bks['vehicles'] if bks else None,
            'bks_distance':         bks_dist,
            'n_vehicules_situation1': n_veh_s1,
            'distance_situation1':  round(best_costs[i], 2),
            'temps_s_situation1':   round(instance_times[i], 2),
            'gap_%_situation1_bks': round((best_costs[i] - bks_dist) / bks_dist * 100, 2) if bks_dist else None,
        }
        rows.append(row)
    print(f"\n  {'Instance':<9} {'BKS Vhc':>8} {'BKS Dist':>10} {'Vhc S1':>7} "
          f"{'Dist. S1':>10} {'Temps S1':>10} {'Gap %':>8}")
    print("  " + "─" * 70)
    for r in rows:
        bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "—"
        bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "—"
        g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {bks_v:>8} {bks_d:>10} "
              f"{r['n_vehicules_situation1']:>7} {r['distance_situation1']:>10.2f} "
              f"{r['temps_s_situation1']:>10.1f} {g1:>8}")
    # ── Gap moyen vs BKS (Situation 1) ──────────────────────────────────
    valid_gaps = [r['gap_%_situation1_bks'] for r in rows if r['gap_%_situation1_bks'] is not None]
    gap_moyen  = float(np.mean(valid_gaps)) if valid_gaps else None
    if gap_moyen is not None:
        print(f"\n  Gap moyen vs BKS (Situation 1) : {gap_moyen:+.2f}%  "
              f"({len(valid_gaps)}/{len(rows)} instances avec BKS connu)")
        print(f"  Gap min : {min(valid_gaps):+.2f}%  |  Gap max : {max(valid_gaps):+.2f}%")
    else:
        print("\n  Gap moyen vs BKS : indisponible (aucun BKS connu pour ces instances)")
    # ── Tableau récapitulatif simple : Instance / Nb véhicules / Distance / Gap % ──
    # (une ligne "Situation 1" + une ligne "BKS" par instance, format demandé)
    print(f"\n  Récapitulatif Instance / Nb véhicules / Distance / Gap % vs BKS")
    print(f"  {'Instance':<9} {'Situation':<26} {'Nb véhicules':>13} {'Distance':>10} {'Gap %':>8}")
    print("  " + "─" * 71)
    recap_rows = []
    for r in rows:
        g1 = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {'Situation 1 (DQN+LNS+Gurobi)':<26} "
              f"{r['n_vehicules_situation1']:>13} {r['distance_situation1']:>10.2f} {g1:>8}")
        recap_rows.append({
            'instance':    r['instance'],
            'situation':   'Situation 1 (DQN+LNS+Gurobi)',
            'n_vehicules': r['n_vehicules_situation1'],
            'distance':    r['distance_situation1'],
            'gap_%_vs_bks': r['gap_%_situation1_bks'],
        })
        if r['bks_distance'] is not None:
            print(f"  {r['instance']:<9} {'BKS (référence)':<26} "
                  f"{(r['bks_vehicules'] if r['bks_vehicules'] is not None else '—'):>13} "
                  f"{r['bks_distance']:>10.2f} {'0.00':>8}")
            recap_rows.append({
                'instance':    r['instance'],
                'situation':   'BKS (référence)',
                'n_vehicules': r['bks_vehicules'],
                'distance':    r['bks_distance'],
                'gap_%_vs_bks': 0.0,
            })
    if gap_moyen is not None:
        print("  " + "─" * 71)
        print(f"  {'':<9} {'MOYENNE (Situation 1)':<26} {'':>13} {'':>10} {gap_moyen:>+8.2f}")
    if csv_path:
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
        print(f"\n  Tableau final exporté (CSV) → {csv_path}")
        recap_csv_path = csv_path.replace('.csv', '_recap_vehicules_distance.csv')
        if recap_csv_path == csv_path:
            recap_csv_path = csv_path + '.recap_vehicules_distance.csv'
        with open(recap_csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=['instance', 'situation', 'n_vehicules', 'distance', 'gap_%_vs_bks'])
            w.writeheader(); w.writerows(recap_rows)
            if gap_moyen is not None:
                w.writerow({'instance': '', 'situation': 'MOYENNE (Situation 1)',
                            'n_vehicules': '', 'distance': '', 'gap_%_vs_bks': round(gap_moyen, 2)})
        print(f"  Récapitulatif Instance/Véhicules/Distance/Gap exporté (CSV) → {recap_csv_path}")
    if latex_path:
        lines = [
            "% Tableau généré automatiquement — Situation 1 vs BKS",
            "\\begin{table}[htbp]", "  \\centering",
            "  \\caption{Comparaison par instance Solomon officielle entre la méthode "
            "hybride (LNS + DQN + Gurobi repair) et le Best Known Solution (BKS) : "
            "nombre de véhicules, distance totale et écart au BKS.}",
            "  \\label{tab:comparaison-bks}",
            "  \\begin{tabular}{lrrrrrr}", "    \\toprule",
            "    Instance & BKS Vhc & BKS Dist. & Vhc. S1 & Dist. S1 & "
            "Temps S1 (s) & Gap S1 (\\%) \\\\",
            "    \\midrule",
        ]
        for r in rows:
            bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "--"
            bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "--"
            g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "--"
            lines.append(f"    {r['instance']} & {bks_v} & {bks_d} & "
                        f"{r['n_vehicules_situation1']} & {r['distance_situation1']:.2f} & "
                        f"{r['temps_s_situation1']:.1f} & {g1} \\\\")
        lines += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
        with open(latex_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        print(f"  Tableau final exporté (LaTeX) → {latex_path}")
    return rows
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6bis  —  DIAGNOSTIC : LICENCE GUROBI À TAILLE LIMITÉE ?
# ══════════════════════════════════════════════════════════════════════════════
def check_gurobi_license(min_vars_needed=2500):
    """
    Test rapide, effectué AVANT tout calcul, pour savoir si la licence
    Gurobi active est une licence "size-limited" (Community/gratuite,
    plafonnée à ~2000 variables ET ~2000 contraintes) plutôt qu'une licence
    complète (académique ou commerciale, sans limite de taille).

    Pourquoi c'est important ici : le repair Gurobi de la Situation 1
    (_repair_gurobi, appelé à CHAQUE step du LNS) crée jusqu'à
    (K_DESTROY+1)² * V variables binaires — selon K_DESTROY (cf. CFG) et le
    nombre de véhicules de l'instance, cela peut dépasser 2000. Un fallback
    vers le repair glouton est déjà en place dans _repair_gurobi (try/
    except), donc l'exécution ne plantera pas, mais peut silencieusement
    retomber sur le glouton plus souvent que prévu si la licence est limitée.

    Ce test ne bloque JAMAIS l'exécution : il informe seulement, une seule
    fois, en tête de run.
    """
    env, m = None, None
    try:
        env = gp.Env(empty=True)
        env.setParam('OutputFlag', 0)
        env.start()
        m = gp.Model(env=env)
        m.setParam('OutputFlag', 0)
        x = m.addVars(min_vars_needed, vtype=GRB.BINARY)
        m.setObjective(gp.quicksum(x[i] for i in range(min_vars_needed)), GRB.MAXIMIZE)
        m.optimize()
        print(f"  ✓ Licence Gurobi : aucune restriction de taille détectée "
              f"(modèle-test à {min_vars_needed} variables résolu sans erreur).")
        return True
    except gp.GurobiError as e:
        errno = getattr(e, 'errno', None)
        is_size_limit = (errno == getattr(GRB.Error, 'SIZE_LIMIT_EXCEEDED', object()))
        if is_size_limit:
            k_repair_vars = (CFG['K_DESTROY'] + 1) ** 2 * CFG['K_DESTROY']
            print(f"\n  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée "
                  f"(erreur {errno} : {e})\n"
                  f"    → Licence Community/gratuite, plafonnée à ~2000 variables "
                  f"et ~2000 contraintes.\n"
                  f"    → Le repair Gurobi de la Situation 1 (jusqu'à "
                  f"~{k_repair_vars} variables selon K_DESTROY={CFG['K_DESTROY']} et "
                  f"le nombre de véhicules) peut dépasser cette limite sur certaines "
                  f"instances — un repli automatique vers le repair glouton est déjà "
                  f"en place, donc pas de plantage, mais un usage de Gurobi moins "
                  f"fréquent que prévu.\n"
                  f"    → Pour lever cette limite : une licence académique Gurobi "
                  f"est gratuite et sans limite de taille pour les universités "
                  f"(https://www.gurobi.com/academia/) — utile si vous êtes "
                  f"affiliée à ENSI-UMA.\n")
        else:
            print(f"  ⚠ Erreur Gurobi lors du test de licence (non liée à la taille) : {e}")
        return False
    except Exception as e:
        print(f"  ⚠ Impossible de tester la licence Gurobi ({type(e).__name__}: {e}) "
              f"— test ignoré, l'exécution continue normalement.")
        return None
    finally:
        if m is not None:
            m.dispose()
        if env is not None:
            env.dispose()
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 7  —  COURBES DE CONVERGENCE
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_mean(vals, window=20):
    out = []
    for i in range(len(vals)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(vals[lo:i + 1])))
    return out
def plot_global_convergence(training_log, instances, initial_costs, best_costs, outdir):
    """
    Figure globale (2 panneaux) :
      (a) reward par épisode global + moyenne mobile → convergence de
          l'apprentissage DQN
      (b) amélioration % par instance (une barre par instance Solomon) vs
          coût LNS initial
    """
    episodes = [e['episode_global'] for e in training_log]
    rewards  = [e['total_reward']   for e in training_log]
    reward_smooth = _rolling_mean(rewards, window=20)
    improvements  = [(initial_costs[i] - best_costs[i]) / initial_costs[i] * 100
                      for i in range(len(initial_costs))]
    names = [inst['name'] for inst in instances]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    axes[0].plot(episodes, rewards, color='lightgray', linewidth=0.6, label='reward brut / épisode')
    axes[0].plot(episodes, reward_smooth, color='crimson', linewidth=2,
                 label='moyenne mobile (20 épisodes)')
    axes[0].set_xlabel('Épisode global')
    axes[0].set_ylabel('Reward de l\'épisode')
    axes[0].set_title("Convergence de l'apprentissage DQN")
    axes[0].legend(loc='best', fontsize=9)
    axes[0].grid(alpha=0.3)
    colors = ['#2ca02c' if v >= 0 else '#d62728' for v in improvements]
    axes[1].bar(range(len(improvements)), improvements, color=colors)
    axes[1].axhline(np.mean(improvements), color='navy', linestyle='--', linewidth=1.5,
                     label=f"Moyenne = {np.mean(improvements):.2f}%")
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names, rotation=90, fontsize=7)
    axes[1].set_xlabel('Instance')
    axes[1].set_ylabel('Amélioration vs solution LNS initiale (%)')
    axes[1].set_title("Amélioration du coût par instance après DQN+Gurobi")
    axes[1].legend(loc='best', fontsize=9)
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_globale.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence globale → {path}")
    return path
def plot_gap_vs_bks(bks_rows, outdir):
    """
    Barres du gap % vs BKS (Situation 1) pour chaque instance, avec la
    moyenne indiquée par une ligne horizontale. N'utilise que les instances
    pour lesquelles un BKS est connu (gap_%_situation1_bks non None).
    `bks_rows` = la liste `rows` retournée par build_bks_comparison_table().
    """
    data = [(r['instance'], r['gap_%_situation1_bks']) for r in bks_rows
            if r['gap_%_situation1_bks'] is not None]
    if not data:
        print("  [plot_gap_vs_bks] Aucune instance avec BKS connu — graphique ignoré.")
        return None
    names = [d[0] for d in data]
    gaps  = [d[1] for d in data]
    gap_moyen = float(np.mean(gaps))
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(names)), 5.5))
    colors = ['#2ca02c' if g <= 0 else '#d62728' for g in gaps]
    ax.bar(range(len(names)), gaps, color=colors)
    ax.axhline(gap_moyen, color='navy', linestyle='--', linewidth=1.5,
               label=f"Gap moyen = {gap_moyen:+.2f}%")
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_xlabel('Instance')
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_title(f"Écart au Best Known Solution par instance "
                 f"({len(names)} instances avec BKS connu)")
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique gap vs BKS → {path}")
    return path
def plot_instance_convergence_examples(training_log, instances, example_idxs, outdir):
    """
    Pour chaque instance exemple :
      - gauche : coût final / meilleur coût cumulé au fil des épisodes locaux
      - droite : trajectoire de coût step par step du dernier épisode local
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(13, 4.2 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        entries = [e for e in training_log if e['instance_id'] == idx]
        entries.sort(key=lambda e: e['ep_local'])
        ep_locals      = [e['ep_local']   for e in entries]
        best_costs_ep  = [e['best_cost']  for e in entries]
        final_costs_ep = [e['final_cost'] for e in entries]
        name = instances[idx]['name']
        axes[row, 0].plot(ep_locals, final_costs_ep, '-o', color='steelblue',
                           label='coût final de l\'épisode')
        axes[row, 0].plot(ep_locals, best_costs_ep, '-s', color='crimson',
                           label='meilleur coût cumulé')
        axes[row, 0].set_title(f"Instance {name} — convergence par épisode local")
        axes[row, 0].set_xlabel('Épisode local')
        axes[row, 0].set_ylabel('Coût de la tournée')
        axes[row, 0].legend(loc='best', fontsize=9)
        axes[row, 0].grid(alpha=0.3)
        last = entries[-1] if entries else None
        trace = last.get('cost_trace', []) if last else []
        if trace:
            axes[row, 1].plot(range(len(trace)), trace, '-o', color='darkorange')
            axes[row, 1].set_title(f"Instance {name} — trajectoire de coût "
                                    f"(dernier épisode local, {len(trace)-1} steps)")
            axes[row, 1].set_xlabel('Step')
            axes[row, 1].set_ylabel('Coût courant')
            axes[row, 1].grid(alpha=0.3)
        else:
            axes[row, 1].axis('off')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_instances_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence instances exemples → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 8  —  TRACÉS DES ROUTES (AVANT LNS / APRÈS DQN)
# ══════════════════════════════════════════════════════════════════════════════
def _plot_routes_on_ax(ax, instance, solution, title):
    depot = instance['depot']
    ax.scatter([depot['x']], [depot['y']], c='black', marker='s', s=110,
               zorder=5, label='Dépôt')
    n_routes = max(len([r for r in solution if r]), 1)
    try:
        cmap = matplotlib.colormaps.get_cmap('tab20').resampled(max(n_routes, 1))
    except AttributeError:
        cmap = plt.cm.get_cmap('tab20', max(n_routes, 1))
    ridx = 0
    for route in solution:
        if not route:
            continue
        xs = [depot['x']] + [instance['customers'][cid - 1]['x'] for cid in route] + [depot['x']]
        ys = [depot['y']] + [instance['customers'][cid - 1]['y'] for cid in route] + [depot['y']]
        ax.plot(xs, ys, '-o', color=cmap(ridx), markersize=4, linewidth=1.3)
        ridx += 1
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.3)
def plot_routes_examples(instances, initial_sols, best_solutions, example_idxs, outdir):
    """
    Pour chaque instance exemple : routes AVANT (solution LNS initiale, avant
    DQN) à gauche, routes APRÈS (meilleure solution trouvée par DQN+Gurobi)
    à droite. Le BKS (véhicules/distance) est indiqué en sous-titre s'il
    est connu pour l'instance.
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.8 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        inst = instances[idx]
        name = inst['name']
        bks  = BKS_REFERENCE.get(name)
        bks_str = (f" | BKS: dist={bks['distance']:.1f}, vhc={bks['vehicles']}"
                   if bks else "")
        cost_before = compute_cost(initial_sols[idx], inst)
        cost_after  = compute_cost(best_solutions[idx], inst)
        nveh_before = sum(1 for r in initial_sols[idx] if r)
        nveh_after  = sum(1 for r in best_solutions[idx] if r)
        _plot_routes_on_ax(
            axes[row, 0], inst, initial_sols[idx],
            f"Instance {name} — AVANT (LNS)\n"
            f"dist={cost_before:.1f}  véhicules={nveh_before}{bks_str}"
        )
        _plot_routes_on_ax(
            axes[row, 1], inst, best_solutions[idx],
            f"Instance {name} — APRÈS (DQN+Gurobi)\n"
            f"dist={cost_after:.1f}  véhicules={nveh_after}{bks_str}"
        )
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_routes_avant_apres_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique routes avant/après → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  EXÉCUTION AUTOMATIQUE
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    t0 = time.time()
    print(f"\n{'═'*72}\n  MODE ENTRAÎNEMENT/TEST DISJOINT\n"
          f"  ENTRAÎNEMENT = {CFG['N_SYNTH_INSTANCES']} instances SYNTHÉTIQUES générées "
          f"| TEST = 56 instances Solomon officielles (gelé)\n{'═'*72}")

    # ── DIAGNOSTIC PRÉALABLE : licence Gurobi à taille limitée ? ────────────
    # Informatif uniquement — n'interrompt jamais l'exécution.
    _sec("DIAGNOSTIC — LICENCE GUROBI")
    check_gurobi_license()

    # ── ÉTAPE 1a : GÉNÉRATION DES INSTANCES SYNTHÉTIQUES D'ENTRAÎNEMENT ─────
    # (reprend generate_instance() de Code 1 — cf. MODULE 1bis ci-dessus).
    # Ces instances SYN-* ne recoupent jamais les noms Solomon (C101, R101,
    # ...) chargés à l'ÉTAPE 2 : aucune fuite entre train et test.
    synth_instances = generate_synthetic_training_set(CFG)

    # ── ÉTAPE 1b : ENTRAÎNEMENT DU DQN — uniquement sur le synthétique ──────
    # Valeurs par défaut : si le modèle est rechargé (branche ci-dessous), il
    # n'y a pas de training_log/coûts synthétiques dans CE run — les
    # graphiques de convergence d'entraînement seront alors simplement omis.
    training_log, synth_initial_costs, synth_best_costs = [], None, None
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    if CFG.get('REUSE_EXISTING_SYNTH_MODEL', False) and os.path.exists(model_path):
        _sec(f"ÉTAPE 1b — RÉUTILISATION D'UN MODÈLE DÉJÀ ENTRAÎNÉ SUR LE SYNTHÉTIQUE "
             f"({model_path})")
        agent = DQNAgent(
            obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
            gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
            epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
            buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
            target_update_freq=CFG['TARGET_UPDATE'],
        )
        agent.load(model_path)
    else:
        agent, training_log, synth_initial_costs, synth_best_costs, \
            synth_best_solutions, synth_instance_times, synth_initial_sols = train_on_instances(
                synth_instances, episodes_per_instance=CFG['EPISODES_PER_INSTANCE'])

    # ── ÉTAPE 2a : CHARGEMENT DES 56 INSTANCES SOLOMON (TEST — jamais vues) ─
    _sec("ÉTAPE 2 — CHARGEMENT DES INSTANCES SOLOMON (TEST)")
    test_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
    example_idxs = pick_example_indices(test_instances, preferred_names=CFG['EXAMPLE_NAMES'])

    # ── ÉTAPE 2b : ÉVALUATION GELÉE — le même agent en mémoire, sans réentraîner ──
    initial_costs, best_costs, best_solutions, instance_times, test_initial_sols, test_log = \
        evaluate_frozen_on_instances(
            test_instances, agent, episodes_per_instance=CFG['EPISODES_PER_INSTANCE_TEST'])

    # ── [A] Convergence de l'ENTRAÎNEMENT (synthétique uniquement) ──────────
    _sec("GRAPHIQUE — CONVERGENCE DE L'ENTRAÎNEMENT (SYNTHÉTIQUE)")
    if training_log:
        plot_global_convergence(training_log, synth_instances,
                                 synth_initial_costs, synth_best_costs, CFG['OUTDIR'])
    else:
        print("  (pas de courbe de convergence : modèle rechargé, aucun "
              "entraînement effectué dans ce run)")

    # ── [B] Convergence détaillée + [C] Routes avant/après — SUR LE TEST ────
    _sec("GRAPHIQUES — DÉTAIL DU TEST (INSTANCES SOLOMON EXEMPLES)")
    plot_instance_convergence_examples(test_log, test_instances, example_idxs, CFG['OUTDIR'])
    plot_routes_examples(test_instances, test_initial_sols, best_solutions, example_idxs, CFG['OUTDIR'])

    # ── [D] TABLEAU FINAL : SITUATION 1 (test gelé, entraîné sur synthétique) ─
    # vs BKS (gap % par instance + gap moyen) ──────────────────────────────
    bks_rows = build_bks_comparison_table(
        test_instances, initial_costs, best_costs, best_solutions, instance_times,
        csv_path=os.path.join(CFG['OUTDIR'], 'solomon_comparaison_bks.csv'),
        latex_path=os.path.join(CFG['OUTDIR'], 'solomon_tableau_bks.tex'),
    )
    # ── [E] Graphique dédié : gap % vs BKS par instance + gap moyen (TEST) ──
    _sec("GRAPHIQUE GAP % vs BKS (TEST)")
    plot_gap_vs_bks(bks_rows, CFG['OUTDIR'])

    print(f"\n  ✓ Terminé en {time.time()-t0:.1f}s ({(time.time()-t0)/3600:.2f}h)\n")

##  DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON + GAP PAR RAPPORT AU BKS  (30 épisodes) ==>+12.31% (Part1)

In [4]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON    ║
║                          + GAP PAR RAPPORT AU BKS                           ║
║                                                                              ║
║  Version corrigée : reprend toutes les visualisations de la variante        ║
║  précédente (courbes de convergence, tracés des routes avant/après,        ║
║  graphique dédié gap % vs BKS, tableau récapitulatif véhicules/distance)   ║
║  MAIS applique désormais un vrai protocole ENTRAÎNEMENT / TEST DISJOINT,   ║
║  en un seul lancement, sans aucune intervention manuelle :                 ║
║                                                                              ║
║   ÉTAPE 1 (ENTRAÎNEMENT) — génère des instances SYNTHÉTIQUES (reprises de   ║
║      generate_instance() de main_vrptw_comparaison_gurobi.py / "Code 1")    ║
║      et entraîne le DQN dessus (poids mis à jour via store/learn/decay).    ║
║                                                                              ║
║   ÉTAPE 2 (TEST) — charge les 56 instances Solomon OFFICIELLES et évalue    ║
║      le même agent, GELÉ (aucun store/learn/decay_epsilon, uniquement      ║
║      select_action en pur mode inférence), sur ce jeu jamais vu à          ║
║      l'entraînement. Toutes les courbes/tracés/tableaux ci-dessous portent  ║
║      désormais sur ce TEST gelé, pas sur les résultats d'entraînement.     ║
║                                                                              ║
║  Situation 1 = LNS + DQN(entraîné sur synthétique) + Gurobi, ÉVALUÉ sur     ║
║                Solomon (test)                                → gap vs BKS   ║
║  (La "Situation 2 = Gurobi seul sur l'instance complète" reste retirée,     ║
║   à la demande, car sans limite de temps elle est beaucoup trop longue      ║
║   sur des instances à 100 clients avec fenêtres de temps serrées.)          ║
║                                                                              ║
║  Visualisations produites (toutes sur le TEST gelé, sauf la 1ère) :        ║
║    [A] Convergence de l'ENTRAÎNEMENT (synthétique) : reward global +       ║
║        amélioration % par instance synthétique                            ║
║    [B] Convergence détaillée (instances Solomon exemples, TEST) : coût     ║
║        final/meilleur par épisode local + trajectoire de coût             ║
║    [C] Tracés des routes AVANT (LNS) / APRÈS (DQN gelé) pour des           ║
║        instances Solomon exemples                                         ║
║    [D] Tableau récapitulatif : Instance / Nb véhicules / Distance / Gap %  ║
║        (TEST Solomon vs BKS)                                              ║
║    [E] Graphique dédié gap % vs BKS par instance + gap moyen              ║
║    [F] Diagnostic automatique de licence Gurobi (taille limitée ou non)   ║
║                                                                              ║
║  Aucun rappel manuel de "Code 1" n'est nécessaire : la génération +        ║
║  l'entraînement synthétique sont réintégrés ici (MODULE 1bis) et           ║
║  s'exécutent automatiquement avant le chargement de Solomon. Voir le flag  ║
║  optionnel CFG['REUSE_EXISTING_SYNTH_MODEL'] plus bas si vous préférez     ║
║  recharger un .pth déjà entraîné au lieu de ré-entraîner à chaque run.     ║
║                                                                              ║
║  Par défaut, INSTANCE_FILES contient toutes les instances *.txt trouvées   ║
║  dans outputs/solomon56/. Pour se limiter à 6 instances représentatives    ║
║  (une par groupe C1/C2/R1/R2/RC1/RC2), décommentez le bloc plus bas.       ║
║                                                                              ║
║  Exécution :  python main_vrptw_synth_train_solomon_convergence.py         ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import os, sys, json, time, copy, random, csv, glob
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, deque
import torch
import torch.nn as nn
import torch.optim as optim
import gurobipy as gp
from gurobipy import GRB
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG GLOBALE
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    # ── ÉTAPE 1 : ENTRAÎNEMENT sur instances SYNTHÉTIQUES (jamais Solomon) ──
    N_SYNTH_INSTANCES          = 80,     # comme les 80 instances de Code 1
    SYNTH_N_CUSTOMERS          = 100,
    SYNTH_N_VEHICLES           = None,   # None → calculé depuis la demande (cf. generate_instance)
    SYNTH_VEHICLE_SAFETY_FACTOR= 1.5,
    SYNTH_CAPACITY             = 200,
    SYNTH_HORIZON              = 1000,
    EPISODES_PER_INSTANCE      = 30,     # épisodes d'ENTRAÎNEMENT par instance synthétique
    # Optionnel : True → recharge un .pth déjà entraîné sur du synthétique
    # (voir MODEL_FILE) au lieu de régénérer + ré-entraîner. False (par
    # défaut) → tout se fait automatiquement dans CE run, sans intervention
    # manuelle, aucun besoin de relancer "Code 1" séparément.
    REUSE_EXISTING_SYNTH_MODEL = False,
    # ── ÉTAPE 2 : TEST gelé sur les 56 instances Solomon officielles ───────
    EPISODES_PER_INSTANCE_TEST = 3,      # épisodes d'ÉVALUATION (aucun apprentissage) par instance Solomon
    # VEHICLE_SAFETY_FACTOR = None : on utilise la flotte OFFICIELLE de chaque
    # instance, pas une borne calculée sur la demande. Sur R101/RC101
    # (fenêtres de temps serrées), le BKS utilise 19 et 14 véhicules alors
    # que la demande seule suggérerait ~12-14 : une borne basée uniquement
    # sur la demande sous-estime le besoin réel et peut rendre le MILP
    # infaisable. La flotte officielle est plus grande (MILP plus lourd)
    # mais garantit qu'une solution proche du BKS reste atteignable.
    VEHICLE_SAFETY_FACTOR  = None,
    LNS_ITER               = 30,
    LNS_K_RATIO            = 0.20,
    K_DESTROY_RATIO        = 0.24,
    MAX_STEPS              = 20,
    # repair par bouts (Situation 1, appelé à CHAQUE step du LNS/DQN — voir
    # check_gurobi_license() plus bas pour l'impact d'une licence à taille
    # limitée). None = aucune limite de temps (supporté par le code).
    GUROBI_TLIMIT           = 2.0,
    GUROBI_GAP              = 0.05,
    HIDDEN                  = 64,
    LR                      = 1e-3,
    GAMMA                   = 0.95,
    EPS_START                = 1.0,
    EPS_END                  = 0.05,
    EPS_DECAY                = 0.95,  # ⚠ ignoré — recalculé automatiquement plus bas (cf. commentaire)
    BUFFER_SIZE               = 5000,
    BATCH_SIZE                = 32,
    TARGET_UPDATE              = 10,
    # ── Instances exemples (pour courbes détaillées + routes, TEST/Solomon) ─
    # Noms recherchés en priorité (représentatifs des 6 groupes Solomon) ;
    # si un nom n'est pas présent dans INSTANCE_FILES, il est ignoré et
    # remplacé automatiquement par le 1er type disponible.
    EXAMPLE_NAMES              = ['C101', 'R101', 'RC101'],
    OUTDIR                        = os.path.join(os.getcwd(), 'outputs'),
    MODELDIR                       = os.path.join(os.getcwd(), 'outputs', 'model'),
    MODEL_FILE                      = 'dqn_vrptw_synth_trained.pth',
)
CFG['LNS_K'] = 20   # ~20% de 100 clients, cohérent avec LNS_K_RATIO
CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))  # 24, cohérent avec K_DESTROY_RATIO
# ── Recalibrage automatique de EPS_DECAY ────────────────────────────────
# La valeur fixe 0.95 était calibrée pour un tout petit nombre d'épisodes
# (contexte d'origine : entraînement direct sur quelques dizaines
# d'instances Solomon). Avec EPS_DECAY=0.95, epsilon atteint son plancher
# (EPS_END) après seulement ~58 décroissances — soit, avec les valeurs par
# défaut (N_SYNTH_INSTANCES=80, EPISODES_PER_INSTANCE=10), après à peine
# ~6 des 80 instances synthétiques : plus de 90% de l'entraînement se
# ferait alors avec epsilon au plancher (quasi aucune exploration), quel
# que soit EPISODES_PER_INSTANCE — et AUGMENTER EPISODES_PER_INSTANCE
# aggrave le problème (le plancher est atteint en encore moins d'instances
# couvertes). On recalcule donc EPS_DECAY ici pour que epsilon décroisse
# de EPS_START à EPS_END sur TOUTE la durée prévue de l'entraînement
# (N_SYNTH_INSTANCES × EPISODES_PER_INSTANCE décroissances), quel que soit
# le nombre d'épisodes par instance choisi.
_n_decays_total = max(1, CFG['N_SYNTH_INSTANCES'] * CFG['EPISODES_PER_INSTANCE'])
CFG['EPS_DECAY'] = (CFG['EPS_END'] / CFG['EPS_START']) ** (1.0 / _n_decays_total)
os.makedirs(CFG['OUTDIR'],   exist_ok=True)
os.makedirs(CFG['MODELDIR'], exist_ok=True)
ACTION_NAMES = {
    0: "Destruction spatiale",
    1: "Destruction temporelle",
    2: "Destruction par coût élevé",
    3: "Destruction aléatoire",
}
# ── Dossier contenant les instances Solomon officielles (extraites du zip) ──
SOLOMON_DIR = os.path.join(CFG['OUTDIR'], 'Solomon')
# Toutes les instances *.txt trouvées (décommentez le bloc ci-dessous pour
# revenir à seulement 6 instances représentatives, une par groupe
# C1/C2/R1/R2/RC1/RC2, afin de garder un temps d'exécution raisonnable).
INSTANCE_FILES = sorted(glob.glob(os.path.join(SOLOMON_DIR, '*.txt')))
# INSTANCE_FILES = [
#     os.path.join(SOLOMON_DIR, 'c101.txt'),
#     os.path.join(SOLOMON_DIR, 'c201.txt'),
#     os.path.join(SOLOMON_DIR, 'r101.txt'),
#     os.path.join(SOLOMON_DIR, 'r201.txt'),
#     os.path.join(SOLOMON_DIR, 'rc101.txt'),
#     os.path.join(SOLOMON_DIR, 'rc201.txt'),
# ]
# ── BKS officiel complet, 56 instances (source : SINTEF / Solomon 1987) ──────
BKS_REFERENCE = {
    'C101': {'vehicles': 10, 'distance': 828.94}, 'C102': {'vehicles': 10, 'distance': 828.94},
    'C103': {'vehicles': 10, 'distance': 828.06}, 'C104': {'vehicles': 10, 'distance': 824.78},
    'C105': {'vehicles': 10, 'distance': 828.94}, 'C106': {'vehicles': 10, 'distance': 828.94},
    'C107': {'vehicles': 10, 'distance': 828.94}, 'C108': {'vehicles': 10, 'distance': 828.94},
    'C109': {'vehicles': 10, 'distance': 828.94},
    'C201': {'vehicles': 3, 'distance': 591.56}, 'C202': {'vehicles': 3, 'distance': 591.56},
    'C203': {'vehicles': 3, 'distance': 591.17}, 'C204': {'vehicles': 3, 'distance': 590.60},
    'C205': {'vehicles': 3, 'distance': 588.88}, 'C206': {'vehicles': 3, 'distance': 588.49},
    'C207': {'vehicles': 3, 'distance': 588.29}, 'C208': {'vehicles': 3, 'distance': 588.32},
    'R101': {'vehicles': 19, 'distance': 1650.80}, 'R102': {'vehicles': 17, 'distance': 1486.12},
    'R103': {'vehicles': 13, 'distance': 1292.68}, 'R104': {'vehicles': 9,  'distance': 1007.31},
    'R105': {'vehicles': 14, 'distance': 1377.11}, 'R106': {'vehicles': 12, 'distance': 1252.03},
    'R107': {'vehicles': 10, 'distance': 1104.66}, 'R108': {'vehicles': 9,  'distance': 960.88},
    'R109': {'vehicles': 11, 'distance': 1194.73}, 'R110': {'vehicles': 10, 'distance': 1118.84},
    'R111': {'vehicles': 10, 'distance': 1096.72}, 'R112': {'vehicles': 9,  'distance': 982.14},
    'R201': {'vehicles': 4, 'distance': 1252.37}, 'R202': {'vehicles': 3, 'distance': 1191.70},
    'R203': {'vehicles': 3, 'distance': 939.50},  'R204': {'vehicles': 2, 'distance': 825.52},
    'R205': {'vehicles': 3, 'distance': 994.43},  'R206': {'vehicles': 3, 'distance': 906.14},
    'R207': {'vehicles': 2, 'distance': 890.61},  'R208': {'vehicles': 2, 'distance': 726.82},
    'R209': {'vehicles': 3, 'distance': 909.16},  'R210': {'vehicles': 3, 'distance': 939.37},
    'R211': {'vehicles': 2, 'distance': 885.71},
    'RC101': {'vehicles': 14, 'distance': 1696.95}, 'RC102': {'vehicles': 12, 'distance': 1554.75},
    'RC103': {'vehicles': 11, 'distance': 1261.67}, 'RC104': {'vehicles': 10, 'distance': 1135.48},
    'RC105': {'vehicles': 13, 'distance': 1629.44}, 'RC106': {'vehicles': 11, 'distance': 1424.73},
    'RC107': {'vehicles': 11, 'distance': 1230.48}, 'RC108': {'vehicles': 10, 'distance': 1139.82},
    'RC201': {'vehicles': 4, 'distance': 1406.94}, 'RC202': {'vehicles': 3, 'distance': 1365.65},
    'RC203': {'vehicles': 3, 'distance': 1049.62}, 'RC204': {'vehicles': 3, 'distance': 798.46},
    'RC205': {'vehicles': 4, 'distance': 1297.65}, 'RC206': {'vehicles': 3, 'distance': 1146.32},
    'RC207': {'vehicles': 3, 'distance': 1061.14}, 'RC208': {'vehicles': 3, 'distance': 828.14},
}
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1  —  LECTURE DES INSTANCES SOLOMON OFFICIELLES  (jeu de TEST)
# ══════════════════════════════════════════════════════════════════════════════
def euclidean(a, b):
    return float(np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2))
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1bis  —  GÉNÉRATEUR D'INSTANCES SYNTHÉTIQUES  (jeu d'ENTRAÎNEMENT)
#  Repris tel quel de main_vrptw_comparaison_gurobi.py ("Code 1") : mêmes
#  fonctions _coords_C/_coords_R/_coords_RC et generate_instance(), pour que
#  les instances d'entraînement soient générées exactement comme dans Code 1.
# ══════════════════════════════════════════════════════════════════════════════
def _coords_C(n, rng):
    n_cl = max(3, n // 10)
    centers = rng.uniform(15, 85, (n_cl, 2))
    pts = []
    for i in range(n):
        p = rng.normal(centers[i % n_cl], 8)
        pts.append(np.clip(p, 1, 99))
    return np.array(pts)
def _coords_R(n, rng):
    return rng.uniform(1, 99, (n, 2))
def _coords_RC(n, rng):
    h = n // 2
    return np.vstack([_coords_C(h, rng), _coords_R(n - h, rng)])
def generate_instance(n_customers=50, n_vehicles=None, capacity=200,
                      type_='C', seed=42, horizon=1000,
                      vehicle_safety_factor=1.5):
    rng     = np.random.RandomState(seed)
    depot   = np.array([50.0, 50.0])
    gen_fn  = {'C': _coords_C, 'R': _coords_R, 'RC': _coords_RC}[type_]
    coords  = gen_fn(n_customers, rng)
    max_dem = max(1, capacity // 5)
    demands = rng.randint(1, max_dem + 1, n_customers)
    svc     = rng.randint(10, 31, n_customers)
    if n_vehicles is None:
        min_vehicles = int(np.ceil(demands.sum() / capacity))
        n_vehicles   = max(2, int(np.ceil(min_vehicles * vehicle_safety_factor)))
    customers = []
    for i in range(n_customers):
        d        = euclidean(depot, coords[i])
        earliest = max(0.0, round(d + rng.uniform(-15, 15), 1))
        delta    = rng.uniform(30, 80)
        latest   = min(float(horizon), round(earliest + delta, 1))
        latest   = max(latest, earliest + svc[i] + 5)
        latest   = min(latest, float(horizon))
        if d > latest:
            earliest = round(d, 1)
            latest   = min(float(horizon), round(earliest + delta, 1))
            latest   = max(latest, earliest + svc[i] + 5)
        if d + svc[i] + d > horizon:
            svc[i] = max(5, int(horizon - 2 * d - 5))
        customers.append({
            'id':           i + 1,
            'x':            float(round(coords[i][0], 2)),
            'y':            float(round(coords[i][1], 2)),
            'demand':       int(demands[i]),
            'earliest':     float(earliest),
            'latest':       float(latest),
            'service_time': int(svc[i]),
        })
    return {
        'depot': {
            'id': 0, 'x': 50.0, 'y': 50.0, 'demand': 0,
            'earliest': 0.0, 'latest': float(horizon), 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    capacity,
        'horizon':     horizon,
        'type':        type_,
        'seed':        seed,
        'n_customers': n_customers,
    }
def generate_synthetic_training_set(cfg, verbose=True):
    """
    Génère les instances synthétiques d'ENTRAÎNEMENT en cyclant les 3 types
    Solomon C / R / RC (même logique que train() dans Code 1 : type =
    ['C','R','RC'][i % 3], seed = i*7+1). Ces instances ne recouvrent JAMAIS
    les 56 instances Solomon officielles utilisées plus bas comme jeu de
    TEST : le DQN ne les voit donc jamais avant l'évaluation gelée.
    Chaque instance reçoit en plus une clé 'name' (ex. 'SYN-C001') pour
    rester compatible avec les fonctions d'entraînement/affichage qui
    utilisent inst['name'].
    """
    types = ['C', 'R', 'RC']
    instances = []
    if verbose:
        _sec(f"ÉTAPE 1 — GÉNÉRATION DE {cfg['N_SYNTH_INSTANCES']} INSTANCES "
             f"SYNTHÉTIQUES (ENTRAÎNEMENT)")
    for i in range(cfg['N_SYNTH_INSTANCES']):
        t    = types[i % 3]
        seed = i * 7 + 1
        inst = generate_instance(
            n_customers = cfg['SYNTH_N_CUSTOMERS'],
            n_vehicles  = cfg['SYNTH_N_VEHICLES'],
            capacity    = cfg['SYNTH_CAPACITY'],
            horizon     = cfg['SYNTH_HORIZON'],
            type_       = t,
            seed        = seed,
            vehicle_safety_factor = cfg['SYNTH_VEHICLE_SAFETY_FACTOR'],
        )
        inst['name'] = f"SYN-{t}{i+1:03d}"
        instances.append(inst)
        if verbose:
            print(f"  Instance {i+1:03d}/{cfg['N_SYNTH_INSTANCES']} | {inst['name']:<10} "
                  f"type={t:<3} | seed={seed} | véhicules={inst['n_vehicles']}")
    return instances
def parse_solomon_instance(filepath, vehicle_safety_factor=None):
    """
    Parse un fichier d'instance Solomon au format texte officiel :
        <NOM>
        VEHICLE
        NUMBER  CAPACITY
        <n>     <cap>
        CUSTOMER
        CUST NO.  XCOORD.  YCOORD.  DEMAND  READY TIME  DUE DATE  SERVICE TIME
        0  ...   (dépôt)
        1  ...
        ...
    Retourne un dict compatible avec le reste du pipeline.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = [l.rstrip('\n') for l in f]
    name = lines[0].strip()
    idx_veh = next(i for i, l in enumerate(lines) if l.strip() == 'VEHICLE')
    veh_number, veh_capacity = map(int, lines[idx_veh + 2].split())
    idx_cust = next(i for i, l in enumerate(lines) if l.strip() == 'CUSTOMER')
    data_lines = [l for l in lines[idx_cust + 2:] if l.strip()]
    rows = [list(map(float, l.split())) for l in data_lines]
    depot_row = rows[0]
    customers = []
    for r in rows[1:]:
        cid, x, y, dem, ready, due, svc = r
        customers.append({
            'id':           int(cid),
            'x':            x,
            'y':            y,
            'demand':       int(dem),
            'earliest':     ready,
            'latest':       due,
            'service_time': int(svc),
        })
    name_upper = name.upper()
    type_ = ''.join(ch for ch in name_upper if ch.isalpha())   # 'C101' -> 'C', 'RC101' -> 'RC'
    n_vehicles = veh_number
    if vehicle_safety_factor is not None:
        total_demand = sum(c['demand'] for c in customers)
        min_veh      = int(np.ceil(total_demand / veh_capacity))
        n_vehicles   = min(veh_number, max(2, int(np.ceil(min_veh * vehicle_safety_factor))))
    return {
        'depot': {
            'id': 0, 'x': depot_row[1], 'y': depot_row[2], 'demand': 0,
            'earliest': depot_row[4], 'latest': depot_row[5], 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    veh_capacity,
        'horizon':     depot_row[5],
        'type':        type_,
        'seed':        name_upper,
        'n_customers': len(customers),
        'name':        name_upper,          # nom officiel normalisé (ex. 'C101')
    }
def load_all_instances(files, vehicle_safety_factor=None):
    if not files:
        raise FileNotFoundError(
            f"Aucun fichier d'instance trouvé dans {SOLOMON_DIR}. "
            f"Vérifiez que le dossier existe et contient les .txt Solomon."
        )
    instances = []
    for fp in files:
        inst = parse_solomon_instance(fp, vehicle_safety_factor=vehicle_safety_factor)
        instances.append(inst)
        print(f"  Chargé : {inst['name']:<8} type={inst['type']:<3} "
              f"clients={inst['n_customers']} véhicules={inst['n_vehicles']} "
              f"capacité={inst['capacity']} horizon={inst['horizon']}")
    return instances
def pick_example_indices(instances, preferred_names=None, max_examples=3):
    """
    Sélectionne des instances "exemples" pour les courbes détaillées et les
    tracés de routes. Cherche d'abord les noms préférés (ex. C101/R101/RC101).
    Si absents, retombe sur la 1ère instance de chaque type détecté, puis sur
    les toutes premières instances de la liste.
    """
    preferred_names = preferred_names or []
    by_name = {inst['name']: i for i, inst in enumerate(instances)}
    idxs = [by_name[nm] for nm in preferred_names if nm in by_name]
    if not idxs:
        seen_types = {}
        for i, inst in enumerate(instances):
            if inst['type'] not in seen_types:
                seen_types[inst['type']] = i
            if len(seen_types) >= max_examples:
                break
        idxs = list(seen_types.values())
    if not idxs:
        idxs = list(range(min(max_examples, len(instances))))
    return idxs[:max_examples]
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 2  —  SOLUTION INITIALE PAR LNS  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
def _node_xy(nid, instance):
    if nid == 0:
        return instance['depot']['x'], instance['depot']['y']
    c = instance['customers'][nid - 1]
    return c['x'], c['y']
def _node_info(nid, instance):
    if nid == 0:
        return instance['depot']
    return instance['customers'][nid - 1]
def compute_cost(solution, instance):
    depot = (instance['depot']['x'], instance['depot']['y'])
    total = 0.0
    for route in solution:
        if not route:
            continue
        prev = depot
        for cid in route:
            cur   = _node_xy(cid, instance)
            total += euclidean(prev, cur)
            prev  = cur
        total += euclidean(prev, depot)
    return total
def _route_feasible(route, instance):
    if not route:
        return True
    cap  = instance['capacity']
    load = sum(_node_info(c, instance)['demand'] for c in route)
    if load > cap:
        return False
    depot = instance['depot']
    t     = 0.0
    px, py = depot['x'], depot['y']
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        if t > info['latest']:
            return False
        t      = max(t, info['earliest']) + info['service_time']
        px, py = cx, cy
    t += euclidean((px, py), (depot['x'], depot['y']))
    return t <= depot['latest']
def is_feasible(solution, instance):
    return all(_route_feasible(r, instance) for r in solution)
def solution_stats(solution, instance):
    cap        = instance['capacity']
    total_cost = compute_cost(solution, instance)
    n_vehicles = sum(1 for r in solution if r)
    loads      = []
    total_wait = 0.0
    for route in solution:
        if not route:
            continue
        load = sum(_node_info(c, instance)['demand'] for c in route)
        loads.append(load / cap)
        depot  = instance['depot']
        t      = 0.0
        px, py = depot['x'], depot['y']
        for cid in route:
            cx, cy = _node_xy(cid, instance)
            t     += euclidean((px, py), (cx, cy))
            info   = _node_info(cid, instance)
            total_wait += max(0.0, info['earliest'] - t)
            t      = max(t, info['earliest']) + info['service_time']
            px, py = cx, cy
    return {
        'total_cost':         round(total_cost, 2),
        'n_vehicles':         n_vehicles,
        'avg_load_ratio':     round(np.mean(loads) if loads else 0.0, 3),
        'total_wait_time':    round(total_wait, 2),
        'n_customers_served': sum(len(r) for r in solution),
    }
def _insertion_cost(cid, pos, route, instance):
    depot   = (instance['depot']['x'], instance['depot']['y'])
    nodes   = [0] + route + [0]
    prev_id = nodes[pos]
    next_id = nodes[pos + 1]
    px, py  = _node_xy(prev_id, instance) if prev_id != 0 else depot
    nx, ny  = _node_xy(next_id, instance) if next_id != 0 else depot
    cx, cy  = _node_xy(cid, instance)
    return (euclidean((px, py), (cx, cy)) + euclidean((cx, cy), (nx, ny))
            - euclidean((px, py), (nx, ny)))
def _try_insert(cid, route, instance):
    best_cost, best_pos = float('inf'), -1
    for pos in range(len(route) + 1):
        candidate = route[:pos] + [cid] + route[pos:]
        if _route_feasible(candidate, instance):
            cost = _insertion_cost(cid, pos, route, instance)
            if cost < best_cost:
                best_cost, best_pos = cost, pos
    return best_cost, best_pos
def _repair_greedy(removed, partial, instance):
    routes = [list(r) for r in partial]
    for cid in removed:
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _build_initial(instance):
    customers = sorted(instance['customers'], key=lambda c: c['earliest'])
    routes    = []
    for c in customers:
        cid = c['id']
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _lns_improve(solution, instance, n_iter=50, k=10, seed=0):
    rng       = np.random.RandomState(seed)
    current   = [list(r) for r in solution]
    best      = copy.deepcopy(current)
    best_cost = compute_cost(best, instance)
    for _ in range(n_iter):
        all_c   = [c for r in current for c in r]
        k_      = min(k, max(1, len(all_c) - 1))
        removed = list(rng.choice(all_c, k_, replace=False))
        removed_set = set(removed)
        partial = [[c for c in r if c not in removed_set] for r in current]
        partial = [r for r in partial if r]
        candidate = _repair_greedy(removed, partial, instance)
        c_cost    = compute_cost(candidate, instance)
        if c_cost < best_cost and is_feasible(candidate, instance):
            best, best_cost = copy.deepcopy(candidate), c_cost
        current = candidate
    return best
def get_initial_solution(instance, lns_iter=50, lns_k=10):
    sol = _build_initial(instance)
    sol = _lns_improve(sol, instance, n_iter=lns_iter, k=lns_k)
    sol = [r for r in sol if r]
    return sol, solution_stats(sol, instance)
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 3  —  ENVIRONNEMENT VRPTW  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class VRPTWEnv:
    def __init__(self, instance, initial_solution,
                 K=15, max_steps=20,
                 gurobi_tlimit=2.0, gurobi_gap=0.05,
                 verbose=False):
        self.instance    = instance
        self.initial_sol = copy.deepcopy(initial_solution)
        self.K           = K
        self.max_steps   = max_steps
        self.tlimit      = gurobi_tlimit
        self.gap         = gurobi_gap
        self.verbose     = verbose
        self.obs_dim     = 7
        self.n_actions   = 4
        self._dist = {}
        depot = (instance['depot']['x'], instance['depot']['y'])
        for c in instance['customers']:
            d = euclidean(depot, (c['x'], c['y']))
            self._dist[(0, c['id'])] = d
            self._dist[(c['id'], 0)] = d
        for ci in instance['customers']:
            for cj in instance['customers']:
                self._dist[(ci['id'], cj['id'])] = euclidean(
                    (ci['x'], ci['y']), (cj['x'], cj['y']))
        self.reset()
    def _d(self, a, b):
        return self._dist.get((a, b), 0.0)
    def _info(self, nid):
        return _node_info(nid, self.instance)
    def reset(self, start_solution=None):
        if start_solution is not None:
            self.solution = copy.deepcopy(start_solution)
        else:
            self.solution = copy.deepcopy(self.initial_sol)
        self.current_cost     = compute_cost(self.solution, self.instance)
        self.best_cost        = self.current_cost
        self.initial_cost     = compute_cost(self.initial_sol, self.instance)
        self.step_count       = 0
        self.steps_no_improve = 0
        self._action_history  = []
        return self._obs()
    def _obs(self):
        stats = solution_stats(self.solution, self.instance)
        ic    = max(self.initial_cost, 1.0)
        return np.array([
            self.current_cost / ic,
            stats['n_vehicles'] / max(self.instance['n_vehicles'], 1),
            stats['avg_load_ratio'],
            stats['total_wait_time'] / max(ic * 0.5, 1.0),
            self.steps_no_improve / self.max_steps,
            self.best_cost / ic,
            1.0,
        ], dtype=np.float32)
    def _destroy(self, action):
        all_c = [c for r in self.solution for c in r]
        if not all_c:
            return [], copy.deepcopy(self.solution)
        K   = min(self.K, len(all_c))
        rng = np.random.RandomState(self.step_count + action * 100)
        if action == 0:
            pivot   = int(rng.choice(all_c))
            px, py  = self._info(pivot)['x'], self._info(pivot)['y']
            dists   = {c: euclidean((px, py),
                                    (self._info(c)['x'], self._info(c)['y']))
                       for c in all_c}
            destroyed = sorted(all_c, key=lambda c: dists[c])[:K]
        elif action == 1:
            pivot    = int(rng.choice(all_c))
            e_pivot  = self._info(pivot)['earliest']
            tdiffs   = {c: abs(self._info(c)['earliest'] - e_pivot) for c in all_c}
            destroyed = sorted(all_c, key=lambda c: tdiffs[c])[:K]
        elif action == 2:
            savings = {}
            for r in self.solution:
                if not r:
                    continue
                nodes = [0] + r + [0]
                for i, cid in enumerate(r):
                    prev, nxt     = nodes[i], nodes[i + 2]
                    savings[cid]  = (self._d(prev, cid) + self._d(cid, nxt)
                                     - self._d(prev, nxt))
            destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
        else:
            idx       = rng.choice(len(all_c), K, replace=False)
            destroyed = [all_c[i] for i in idx]
        destroyed_set = set(destroyed)
        partial = [[c for c in r if c not in destroyed_set]
                   for r in self.solution]
        partial = [r for r in partial if r]
        return list(destroyed), partial
    def _repair_gurobi(self, destroyed, partial):
        if not destroyed:
            return copy.deepcopy(self.solution)
        inst    = self.instance
        depot   = inst['depot']
        horizon = inst['horizon']
        cap     = inst['capacity']
        nodes   = [0] + destroyed
        N       = len(nodes)
        V       = min(len(destroyed), inst['n_vehicles'])
        env, m = None, None
        try:
            env = gp.Env(empty=True)
            env.setParam('OutputFlag', 0)
            env.start()
            m = gp.Model(env=env)
            if self.tlimit is not None:
                m.setParam('TimeLimit', self.tlimit)
            m.setParam('MIPGap',     self.gap)
            m.setParam('OutputFlag', 0)
            x = m.addVars(N, N, V, vtype=GRB.BINARY, name='x')
            t = m.addVars(N, V, lb=0, ub=horizon,    name='t')
            m.setObjective(
                gp.quicksum(
                    self._d(nodes[i], nodes[j]) * x[i, j, v]
                    for i in range(N) for j in range(N) for v in range(V)
                    if i != j
                ),
                GRB.MINIMIZE
            )
            for i in range(1, N):
                m.addConstr(
                    gp.quicksum(x[i, j, v]
                                for j in range(N) for v in range(V)
                                if i != j) == 1
                )
            for v in range(V):
                for i in range(1, N):
                    m.addConstr(
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j) ==
                        gp.quicksum(x[j, i, v] for j in range(N) if i != j)
                    )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(x[0, j, v] for j in range(1, N)) <= 1
                )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(
                        inst['customers'][nodes[i] - 1]['demand'] *
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j)
                        for i in range(1, N)
                    ) <= cap
                )
            M = horizon + 1000
            for v in range(V):
                m.addConstr(t[0, v] == 0)
                for i in range(N):
                    ni  = nodes[i]
                    ei  = depot['earliest'] if ni == 0 else inst['customers'][ni-1]['earliest']
                    li  = depot['latest']   if ni == 0 else inst['customers'][ni-1]['latest']
                    sti = depot['service_time'] if ni == 0 else inst['customers'][ni-1]['service_time']
                    m.addConstr(t[i, v] >= ei)
                    m.addConstr(t[i, v] <= li)
                    for j in range(N):
                        if i == j:
                            continue
                        m.addConstr(
                            t[j, v] >= t[i, v] + sti + self._d(ni, nodes[j])
                            - M * (1 - x[i, j, v])
                        )
            m.optimize()
            if m.SolCount > 0:
                eps        = 0.5
                new_routes = []
                for v in range(V):
                    if sum(x[0, j, v].X for j in range(1, N)) < eps:
                        continue
                    route_v, cur = [], 0
                    for _ in range(N):
                        nxt = next((j for j in range(N)
                                    if j != cur and x[cur, j, v].X > eps), None)
                        if nxt is None or nxt == 0:
                            break
                        route_v.append(nodes[nxt])
                        cur = nxt
                    if route_v:
                        new_routes.append(route_v)
                full = [r for r in list(partial) + new_routes if r]
                return full
            else:
                raise ValueError("Gurobi : aucune solution trouvée")
        except Exception as e:
            if self.verbose:
                print(f"    [Gurobi→Greedy fallback] {e}")
            return _repair_greedy(destroyed, partial, self.instance)
        finally:
            if m is not None:
                m.dispose()
            if env is not None:
                env.dispose()
    def step(self, action):
        old_cost            = self.current_cost
        destroyed, partial  = self._destroy(action)
        new_sol             = self._repair_gurobi(destroyed, partial)
        new_cost            = compute_cost(new_sol, self.instance)
        feasible            = is_feasible(new_sol, self.instance)
        if feasible:
            self.solution     = new_sol
            self.current_cost = new_cost
        else:
            new_cost = old_cost * 1.001
        improvement = old_cost - self.current_cost
        if improvement > 0:
            self.best_cost        = min(self.best_cost, self.current_cost)
            self.steps_no_improve = 0
        else:
            self.steps_no_improve += 1
        reward = improvement / max(self.initial_cost, 1.0)
        self.step_count += 1
        self._action_history.append(action)
        done = (self.step_count >= self.max_steps)
        info = {
            'cost':          self.current_cost,
            'best_cost':     self.best_cost,
            'action_taken':  action,
            'action_name':   ACTION_NAMES[action],
            'improvement':   improvement,
            'destroyed_k':   len(destroyed),
            'destroyed_ids': [int(d) for d in destroyed[:5]],
            'feasible':      feasible,
        }
        return self._obs(), float(reward), done, info
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 4  —  AGENT DQN  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((
            np.array(s,  dtype=np.float32), int(a), float(r),
            np.array(s2, dtype=np.float32), float(done),
        ))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.FloatTensor(np.stack(s)), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(np.stack(s2)),
                torch.FloatTensor(d))
    def __len__(self):
        return len(self.buf)
class DQNNet(nn.Module):
    def __init__(self, obs_dim=7, n_actions=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim,  hidden),  nn.ReLU(),
            nn.Linear(hidden,   hidden),  nn.ReLU(),
            nn.Linear(hidden,   n_actions),
        )
    def forward(self, x):
        return self.net(x)
class DQNAgent:
    def __init__(self, obs_dim=7, n_actions=4, hidden=64,
                 lr=1e-3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_size=2000, batch_size=32,
                 target_update_freq=10, seed=0):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        self.n_actions          = n_actions
        self.gamma              = gamma
        self.epsilon            = epsilon_start
        self.epsilon_end        = epsilon_end
        self.epsilon_decay      = epsilon_decay
        self.batch_size         = batch_size
        self.target_update_freq = target_update_freq
        self.step_count         = 0
        self.q_net      = DQNNet(obs_dim, n_actions, hidden)
        self.target_net = DQNNet(obs_dim, n_actions, hidden)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)
        self.loss_fn   = nn.MSELoss()
    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state).unsqueeze(0))
            return int(q.argmax().item())
    def store(self, s, a, r, s2, done):
        self.buffer.push(s, a, r, s2, done)
    def learn(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        s, a, r, s2, done = self.buffer.sample(self.batch_size)
        q_vals = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q  = self.target_net(s2).max(1)[0]
            targets = r + self.gamma * next_q * (1 - done)
        loss = self.loss_fn(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.optimizer.step()
        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.update_target()
        return float(loss.item())
    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    def save(self, path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        torch.save({'q_net': self.q_net.state_dict(),
                    'target_net': self.target_net.state_dict(),
                    'epsilon': self.epsilon, 'step_count': self.step_count}, path)
        print(f"    [DQN] Modèle sauvegardé → {path}")
    def load(self, path):
        """
        Recharge un checkpoint sauvegardé par save() (mêmes clés). Utilisé
        uniquement si CFG['REUSE_EXISTING_SYNTH_MODEL']=True, pour réutiliser
        un modèle déjà entraîné sur du synthétique au lieu de ré-entraîner.
        """
        ckpt = torch.load(path, map_location='cpu')
        self.q_net.load_state_dict(ckpt['q_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.epsilon    = ckpt.get('epsilon', self.epsilon)
        self.step_count = ckpt.get('step_count', self.step_count)
        print(f"    [DQN] Modèle rechargé ← {path}")
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5  —  ENTRAÎNEMENT DU DQN  (ÉTAPE 1, sur les instances SYNTHÉTIQUES)
#  Stocke cost_trace (trajectoire de coût step par step) dans le log, utilisé
#  ensuite pour les courbes de convergence détaillées. Retourne aussi
#  initial_sols pour les tracés de routes avant/après.
# ══════════════════════════════════════════════════════════════════════════════
BANNER = "═" * 72
def _sec(title):
    print(f"\n{BANNER}\n  {title}\n{BANNER}")
def train_on_instances(instances, episodes_per_instance=30):
    """
    ENTRAÎNEMENT (les poids du DQN sont mis à jour ici via store()/learn()/
    decay_epsilon()). Générique : fonctionne sur n'importe quelle liste
    d'instances au format standard — dans ce script, appelé avec les
    instances SYNTHÉTIQUES produites par generate_synthetic_training_set(),
    JAMAIS avec les instances Solomon (réservées à evaluate_frozen_
    on_instances(), plus bas, pour le TEST).
    Retourne : agent, training_log, initial_costs, best_costs, best_solutions,
               instance_times, initial_sols
    """
    n = len(instances)
    _sec(f"ÉTAPE 1 — LNS initial pour les {n} instances d'entraînement (synthétiques)")
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    _sec("ÉTAPE 1 — Entraînement DQN sur le synthétique")
    agent = DQNAgent(
        obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
        gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
        epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
        buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
        target_update_freq=CFG['TARGET_UPDATE'],
    )
    training_log   = []
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    episode_global = 0
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TRAIN] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)
                next_obs, reward, done, info = env.step(action)
                agent.store(obs, action, reward, next_obs, float(done))
                agent.learn()
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            agent.decay_epsilon()
            training_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TRAIN] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    agent.save(model_path)
    return agent, training_log, initial_costs, best_costs, best_solutions, instance_times, initial_sols
def evaluate_frozen_on_instances(instances, agent, episodes_per_instance=3):
    """
    TEST — ÉVALUATION GELÉE : l'agent DQN déjà entraîné (ÉTAPE 1, sur le
    synthétique) est ici appliqué en pur mode INFÉRENCE sur les instances
    passées en argument (les 56 Solomon officielles). Différences avec
    train_on_instances() :
      - AUCUN appel à agent.store() / agent.learn() / agent.decay_epsilon() :
        les poids du réseau ne sont JAMAIS mis à jour ici.
      - epsilon est temporairement forcé à 0.0 (politique gloutonne, pure
        exploitation du Q-réseau appris), puis restauré à sa valeur
        d'origine à la fin.
      - Le modèle n'est PAS sauvegardé ici (rien n'a changé dans ses poids).
    Le log retourné (test_log) a exactement la même forme que training_log
    (y compris 'cost_trace'), pour pouvoir réutiliser telles quelles les
    fonctions de tracé (plot_instance_convergence_examples, etc.) sur le
    TEST plutôt que sur l'entraînement.
    Retourne : initial_costs, best_costs, best_solutions, instance_times,
               initial_sols, test_log
    """
    n = len(instances)
    _sec(f"ÉTAPE 2 — ÉVALUATION GELÉE (TEST) sur {n} instances Solomon "
         f"— aucun apprentissage, epsilon=0")
    saved_epsilon = agent.epsilon
    agent.epsilon = 0.0
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    test_log       = []
    episode_global = 0
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TEST] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)          # inférence pure (epsilon=0)
                next_obs, reward, done, info = env.step(action)
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            test_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TEST] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    agent.epsilon = saved_epsilon
    return initial_costs, best_costs, best_solutions, instance_times, initial_sols, test_log
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6  —  TABLEAU FINAL : SITUATION 1 vs BKS
#  (la Situation 2 "Gurobi seul sur l'instance complète" a été retirée)
# ══════════════════════════════════════════════════════════════════════════════
def build_bks_comparison_table(instances, initial_costs, best_costs, best_solutions,
                                instance_times, csv_path=None, latex_path=None):
    """
    Tableau final : Situation 1 (LNS+DQN+Gurobi) vs BKS, avec gap % et le
    nombre de véhicules utilisés pour chaque instance.
    """
    _sec("TABLEAU FINAL — SITUATION 1 vs BKS")
    rows = []
    for i, inst in enumerate(instances):
        name     = inst['name']
        bks      = BKS_REFERENCE.get(name)
        bks_dist = bks['distance'] if bks else None
        n_veh_s1 = sum(1 for r in best_solutions[i] if r)
        row = {
            'instance':             name,
            'type':                 inst['type'],
            'bks_vehicules':        bks['vehicles'] if bks else None,
            'bks_distance':         bks_dist,
            'n_vehicules_situation1': n_veh_s1,
            'distance_situation1':  round(best_costs[i], 2),
            'temps_s_situation1':   round(instance_times[i], 2),
            'gap_%_situation1_bks': round((best_costs[i] - bks_dist) / bks_dist * 100, 2) if bks_dist else None,
        }
        rows.append(row)
    print(f"\n  {'Instance':<9} {'BKS Vhc':>8} {'BKS Dist':>10} {'Vhc S1':>7} "
          f"{'Dist. S1':>10} {'Temps S1':>10} {'Gap %':>8}")
    print("  " + "─" * 70)
    for r in rows:
        bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "—"
        bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "—"
        g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {bks_v:>8} {bks_d:>10} "
              f"{r['n_vehicules_situation1']:>7} {r['distance_situation1']:>10.2f} "
              f"{r['temps_s_situation1']:>10.1f} {g1:>8}")
    # ── Gap moyen vs BKS (Situation 1) ──────────────────────────────────
    valid_gaps = [r['gap_%_situation1_bks'] for r in rows if r['gap_%_situation1_bks'] is not None]
    gap_moyen  = float(np.mean(valid_gaps)) if valid_gaps else None
    if gap_moyen is not None:
        print(f"\n  Gap moyen vs BKS (Situation 1) : {gap_moyen:+.2f}%  "
              f"({len(valid_gaps)}/{len(rows)} instances avec BKS connu)")
        print(f"  Gap min : {min(valid_gaps):+.2f}%  |  Gap max : {max(valid_gaps):+.2f}%")
    else:
        print("\n  Gap moyen vs BKS : indisponible (aucun BKS connu pour ces instances)")
    # ── Tableau récapitulatif simple : Instance / Nb véhicules / Distance / Gap % ──
    # (une ligne "Situation 1" + une ligne "BKS" par instance, format demandé)
    print(f"\n  Récapitulatif Instance / Nb véhicules / Distance / Gap % vs BKS")
    print(f"  {'Instance':<9} {'Situation':<26} {'Nb véhicules':>13} {'Distance':>10} {'Gap %':>8}")
    print("  " + "─" * 71)
    recap_rows = []
    for r in rows:
        g1 = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {'Situation 1 (DQN+LNS+Gurobi)':<26} "
              f"{r['n_vehicules_situation1']:>13} {r['distance_situation1']:>10.2f} {g1:>8}")
        recap_rows.append({
            'instance':    r['instance'],
            'situation':   'Situation 1 (DQN+LNS+Gurobi)',
            'n_vehicules': r['n_vehicules_situation1'],
            'distance':    r['distance_situation1'],
            'gap_%_vs_bks': r['gap_%_situation1_bks'],
        })
        if r['bks_distance'] is not None:
            print(f"  {r['instance']:<9} {'BKS (référence)':<26} "
                  f"{(r['bks_vehicules'] if r['bks_vehicules'] is not None else '—'):>13} "
                  f"{r['bks_distance']:>10.2f} {'0.00':>8}")
            recap_rows.append({
                'instance':    r['instance'],
                'situation':   'BKS (référence)',
                'n_vehicules': r['bks_vehicules'],
                'distance':    r['bks_distance'],
                'gap_%_vs_bks': 0.0,
            })
    if gap_moyen is not None:
        print("  " + "─" * 71)
        print(f"  {'':<9} {'MOYENNE (Situation 1)':<26} {'':>13} {'':>10} {gap_moyen:>+8.2f}")
    if csv_path:
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
        print(f"\n  Tableau final exporté (CSV) → {csv_path}")
        recap_csv_path = csv_path.replace('.csv', '_recap_vehicules_distance.csv')
        if recap_csv_path == csv_path:
            recap_csv_path = csv_path + '.recap_vehicules_distance.csv'
        with open(recap_csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=['instance', 'situation', 'n_vehicules', 'distance', 'gap_%_vs_bks'])
            w.writeheader(); w.writerows(recap_rows)
            if gap_moyen is not None:
                w.writerow({'instance': '', 'situation': 'MOYENNE (Situation 1)',
                            'n_vehicules': '', 'distance': '', 'gap_%_vs_bks': round(gap_moyen, 2)})
        print(f"  Récapitulatif Instance/Véhicules/Distance/Gap exporté (CSV) → {recap_csv_path}")
    if latex_path:
        lines = [
            "% Tableau généré automatiquement — Situation 1 vs BKS",
            "\\begin{table}[htbp]", "  \\centering",
            "  \\caption{Comparaison par instance Solomon officielle entre la méthode "
            "hybride (LNS + DQN + Gurobi repair) et le Best Known Solution (BKS) : "
            "nombre de véhicules, distance totale et écart au BKS.}",
            "  \\label{tab:comparaison-bks}",
            "  \\begin{tabular}{lrrrrrr}", "    \\toprule",
            "    Instance & BKS Vhc & BKS Dist. & Vhc. S1 & Dist. S1 & "
            "Temps S1 (s) & Gap S1 (\\%) \\\\",
            "    \\midrule",
        ]
        for r in rows:
            bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "--"
            bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "--"
            g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "--"
            lines.append(f"    {r['instance']} & {bks_v} & {bks_d} & "
                        f"{r['n_vehicules_situation1']} & {r['distance_situation1']:.2f} & "
                        f"{r['temps_s_situation1']:.1f} & {g1} \\\\")
        lines += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
        with open(latex_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        print(f"  Tableau final exporté (LaTeX) → {latex_path}")
    return rows
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6bis  —  DIAGNOSTIC : LICENCE GUROBI À TAILLE LIMITÉE ?
# ══════════════════════════════════════════════════════════════════════════════
def check_gurobi_license(min_vars_needed=2500):
    """
    Test rapide, effectué AVANT tout calcul, pour savoir si la licence
    Gurobi active est une licence "size-limited" (Community/gratuite,
    plafonnée à ~2000 variables ET ~2000 contraintes) plutôt qu'une licence
    complète (académique ou commerciale, sans limite de taille).

    Pourquoi c'est important ici : le repair Gurobi de la Situation 1
    (_repair_gurobi, appelé à CHAQUE step du LNS) crée jusqu'à
    (K_DESTROY+1)² * V variables binaires — selon K_DESTROY (cf. CFG) et le
    nombre de véhicules de l'instance, cela peut dépasser 2000. Un fallback
    vers le repair glouton est déjà en place dans _repair_gurobi (try/
    except), donc l'exécution ne plantera pas, mais peut silencieusement
    retomber sur le glouton plus souvent que prévu si la licence est limitée.

    Ce test ne bloque JAMAIS l'exécution : il informe seulement, une seule
    fois, en tête de run.
    """
    env, m = None, None
    try:
        env = gp.Env(empty=True)
        env.setParam('OutputFlag', 0)
        env.start()
        m = gp.Model(env=env)
        m.setParam('OutputFlag', 0)
        x = m.addVars(min_vars_needed, vtype=GRB.BINARY)
        m.setObjective(gp.quicksum(x[i] for i in range(min_vars_needed)), GRB.MAXIMIZE)
        m.optimize()
        print(f"  ✓ Licence Gurobi : aucune restriction de taille détectée "
              f"(modèle-test à {min_vars_needed} variables résolu sans erreur).")
        return True
    except gp.GurobiError as e:
        errno = getattr(e, 'errno', None)
        is_size_limit = (errno == getattr(GRB.Error, 'SIZE_LIMIT_EXCEEDED', object()))
        if is_size_limit:
            k_repair_vars = (CFG['K_DESTROY'] + 1) ** 2 * CFG['K_DESTROY']
            print(f"\n  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée "
                  f"(erreur {errno} : {e})\n"
                  f"    → Licence Community/gratuite, plafonnée à ~2000 variables "
                  f"et ~2000 contraintes.\n"
                  f"    → Le repair Gurobi de la Situation 1 (jusqu'à "
                  f"~{k_repair_vars} variables selon K_DESTROY={CFG['K_DESTROY']} et "
                  f"le nombre de véhicules) peut dépasser cette limite sur certaines "
                  f"instances — un repli automatique vers le repair glouton est déjà "
                  f"en place, donc pas de plantage, mais un usage de Gurobi moins "
                  f"fréquent que prévu.\n"
                  f"    → Pour lever cette limite : une licence académique Gurobi "
                  f"est gratuite et sans limite de taille pour les universités "
                  f"(https://www.gurobi.com/academia/) — utile si vous êtes "
                  f"affiliée à ENSI-UMA.\n")
        else:
            print(f"  ⚠ Erreur Gurobi lors du test de licence (non liée à la taille) : {e}")
        return False
    except Exception as e:
        print(f"  ⚠ Impossible de tester la licence Gurobi ({type(e).__name__}: {e}) "
              f"— test ignoré, l'exécution continue normalement.")
        return None
    finally:
        if m is not None:
            m.dispose()
        if env is not None:
            env.dispose()
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 7  —  COURBES DE CONVERGENCE
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_mean(vals, window=20):
    out = []
    for i in range(len(vals)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(vals[lo:i + 1])))
    return out
def plot_global_convergence(training_log, instances, initial_costs, best_costs, outdir):
    """
    Figure globale (2 panneaux) :
      (a) reward par épisode global + moyenne mobile → convergence de
          l'apprentissage DQN
      (b) amélioration % par instance (une barre par instance Solomon) vs
          coût LNS initial
    """
    episodes = [e['episode_global'] for e in training_log]
    rewards  = [e['total_reward']   for e in training_log]
    reward_smooth = _rolling_mean(rewards, window=20)
    improvements  = [(initial_costs[i] - best_costs[i]) / initial_costs[i] * 100
                      for i in range(len(initial_costs))]
    names = [inst['name'] for inst in instances]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    axes[0].plot(episodes, rewards, color='lightgray', linewidth=0.6, label='reward brut / épisode')
    axes[0].plot(episodes, reward_smooth, color='crimson', linewidth=2,
                 label='moyenne mobile (20 épisodes)')
    axes[0].set_xlabel('Épisode global')
    axes[0].set_ylabel('Reward de l\'épisode')
    axes[0].set_title("Convergence de l'apprentissage DQN")
    axes[0].legend(loc='best', fontsize=9)
    axes[0].grid(alpha=0.3)
    colors = ['#2ca02c' if v >= 0 else '#d62728' for v in improvements]
    axes[1].bar(range(len(improvements)), improvements, color=colors)
    axes[1].axhline(np.mean(improvements), color='navy', linestyle='--', linewidth=1.5,
                     label=f"Moyenne = {np.mean(improvements):.2f}%")
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names, rotation=90, fontsize=7)
    axes[1].set_xlabel('Instance')
    axes[1].set_ylabel('Amélioration vs solution LNS initiale (%)')
    axes[1].set_title("Amélioration du coût par instance après DQN+Gurobi")
    axes[1].legend(loc='best', fontsize=9)
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_globale.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence globale → {path}")
    return path
def plot_gap_vs_bks(bks_rows, outdir):
    """
    Barres du gap % vs BKS (Situation 1) pour chaque instance, avec la
    moyenne indiquée par une ligne horizontale. N'utilise que les instances
    pour lesquelles un BKS est connu (gap_%_situation1_bks non None).
    `bks_rows` = la liste `rows` retournée par build_bks_comparison_table().
    """
    data = [(r['instance'], r['gap_%_situation1_bks']) for r in bks_rows
            if r['gap_%_situation1_bks'] is not None]
    if not data:
        print("  [plot_gap_vs_bks] Aucune instance avec BKS connu — graphique ignoré.")
        return None
    names = [d[0] for d in data]
    gaps  = [d[1] for d in data]
    gap_moyen = float(np.mean(gaps))
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(names)), 5.5))
    colors = ['#2ca02c' if g <= 0 else '#d62728' for g in gaps]
    ax.bar(range(len(names)), gaps, color=colors)
    ax.axhline(gap_moyen, color='navy', linestyle='--', linewidth=1.5,
               label=f"Gap moyen = {gap_moyen:+.2f}%")
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_xlabel('Instance')
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_title(f"Écart au Best Known Solution par instance "
                 f"({len(names)} instances avec BKS connu)")
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique gap vs BKS → {path}")
    return path
def plot_instance_convergence_examples(training_log, instances, example_idxs, outdir):
    """
    Pour chaque instance exemple :
      - gauche : coût final / meilleur coût cumulé au fil des épisodes locaux
      - droite : trajectoire de coût step par step du dernier épisode local
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(13, 4.2 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        entries = [e for e in training_log if e['instance_id'] == idx]
        entries.sort(key=lambda e: e['ep_local'])
        ep_locals      = [e['ep_local']   for e in entries]
        best_costs_ep  = [e['best_cost']  for e in entries]
        final_costs_ep = [e['final_cost'] for e in entries]
        name = instances[idx]['name']
        axes[row, 0].plot(ep_locals, final_costs_ep, '-o', color='steelblue',
                           label='coût final de l\'épisode')
        axes[row, 0].plot(ep_locals, best_costs_ep, '-s', color='crimson',
                           label='meilleur coût cumulé')
        axes[row, 0].set_title(f"Instance {name} — convergence par épisode local")
        axes[row, 0].set_xlabel('Épisode local')
        axes[row, 0].set_ylabel('Coût de la tournée')
        axes[row, 0].legend(loc='best', fontsize=9)
        axes[row, 0].grid(alpha=0.3)
        last = entries[-1] if entries else None
        trace = last.get('cost_trace', []) if last else []
        if trace:
            axes[row, 1].plot(range(len(trace)), trace, '-o', color='darkorange')
            axes[row, 1].set_title(f"Instance {name} — trajectoire de coût "
                                    f"(dernier épisode local, {len(trace)-1} steps)")
            axes[row, 1].set_xlabel('Step')
            axes[row, 1].set_ylabel('Coût courant')
            axes[row, 1].grid(alpha=0.3)
        else:
            axes[row, 1].axis('off')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_instances_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence instances exemples → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 8  —  TRACÉS DES ROUTES (AVANT LNS / APRÈS DQN)
# ══════════════════════════════════════════════════════════════════════════════
def _plot_routes_on_ax(ax, instance, solution, title):
    depot = instance['depot']
    ax.scatter([depot['x']], [depot['y']], c='black', marker='s', s=110,
               zorder=5, label='Dépôt')
    n_routes = max(len([r for r in solution if r]), 1)
    try:
        cmap = matplotlib.colormaps.get_cmap('tab20').resampled(max(n_routes, 1))
    except AttributeError:
        cmap = plt.cm.get_cmap('tab20', max(n_routes, 1))
    ridx = 0
    for route in solution:
        if not route:
            continue
        xs = [depot['x']] + [instance['customers'][cid - 1]['x'] for cid in route] + [depot['x']]
        ys = [depot['y']] + [instance['customers'][cid - 1]['y'] for cid in route] + [depot['y']]
        ax.plot(xs, ys, '-o', color=cmap(ridx), markersize=4, linewidth=1.3)
        ridx += 1
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.3)
def plot_routes_examples(instances, initial_sols, best_solutions, example_idxs, outdir):
    """
    Pour chaque instance exemple : routes AVANT (solution LNS initiale, avant
    DQN) à gauche, routes APRÈS (meilleure solution trouvée par DQN+Gurobi)
    à droite. Le BKS (véhicules/distance) est indiqué en sous-titre s'il
    est connu pour l'instance.
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.8 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        inst = instances[idx]
        name = inst['name']
        bks  = BKS_REFERENCE.get(name)
        bks_str = (f" | BKS: dist={bks['distance']:.1f}, vhc={bks['vehicles']}"
                   if bks else "")
        cost_before = compute_cost(initial_sols[idx], inst)
        cost_after  = compute_cost(best_solutions[idx], inst)
        nveh_before = sum(1 for r in initial_sols[idx] if r)
        nveh_after  = sum(1 for r in best_solutions[idx] if r)
        _plot_routes_on_ax(
            axes[row, 0], inst, initial_sols[idx],
            f"Instance {name} — AVANT (LNS)\n"
            f"dist={cost_before:.1f}  véhicules={nveh_before}{bks_str}"
        )
        _plot_routes_on_ax(
            axes[row, 1], inst, best_solutions[idx],
            f"Instance {name} — APRÈS (DQN+Gurobi)\n"
            f"dist={cost_after:.1f}  véhicules={nveh_after}{bks_str}"
        )
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_routes_avant_apres_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique routes avant/après → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  EXÉCUTION AUTOMATIQUE
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    t0 = time.time()
    print(f"\n{'═'*72}\n  MODE ENTRAÎNEMENT/TEST DISJOINT\n"
          f"  ENTRAÎNEMENT = {CFG['N_SYNTH_INSTANCES']} instances SYNTHÉTIQUES générées "
          f"| TEST = 56 instances Solomon officielles (gelé)\n{'═'*72}")

    # ── DIAGNOSTIC PRÉALABLE : licence Gurobi à taille limitée ? ────────────
    # Informatif uniquement — n'interrompt jamais l'exécution.
    _sec("DIAGNOSTIC — LICENCE GUROBI")
    check_gurobi_license()

    # ── ÉTAPE 1a : GÉNÉRATION DES INSTANCES SYNTHÉTIQUES D'ENTRAÎNEMENT ─────
    # (reprend generate_instance() de Code 1 — cf. MODULE 1bis ci-dessus).
    # Ces instances SYN-* ne recoupent jamais les noms Solomon (C101, R101,
    # ...) chargés à l'ÉTAPE 2 : aucune fuite entre train et test.
    synth_instances = generate_synthetic_training_set(CFG)

    # ── ÉTAPE 1b : ENTRAÎNEMENT DU DQN — uniquement sur le synthétique ──────
    # Valeurs par défaut : si le modèle est rechargé (branche ci-dessous), il
    # n'y a pas de training_log/coûts synthétiques dans CE run — les
    # graphiques de convergence d'entraînement seront alors simplement omis.
    training_log, synth_initial_costs, synth_best_costs = [], None, None
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    if CFG.get('REUSE_EXISTING_SYNTH_MODEL', False) and os.path.exists(model_path):
        _sec(f"ÉTAPE 1b — RÉUTILISATION D'UN MODÈLE DÉJÀ ENTRAÎNÉ SUR LE SYNTHÉTIQUE "
             f"({model_path})")
        agent = DQNAgent(
            obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
            gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
            epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
            buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
            target_update_freq=CFG['TARGET_UPDATE'],
        )
        agent.load(model_path)
    else:
        agent, training_log, synth_initial_costs, synth_best_costs, \
            synth_best_solutions, synth_instance_times, synth_initial_sols = train_on_instances(
                synth_instances, episodes_per_instance=CFG['EPISODES_PER_INSTANCE'])

    # ── ÉTAPE 2a : CHARGEMENT DES 56 INSTANCES SOLOMON (TEST — jamais vues) ─
    _sec("ÉTAPE 2 — CHARGEMENT DES INSTANCES SOLOMON (TEST)")
    test_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
    example_idxs = pick_example_indices(test_instances, preferred_names=CFG['EXAMPLE_NAMES'])

    # ── ÉTAPE 2b : ÉVALUATION GELÉE — le même agent en mémoire, sans réentraîner ──
    initial_costs, best_costs, best_solutions, instance_times, test_initial_sols, test_log = \
        evaluate_frozen_on_instances(
            test_instances, agent, episodes_per_instance=CFG['EPISODES_PER_INSTANCE_TEST'])

    # ── [A] Convergence de l'ENTRAÎNEMENT (synthétique uniquement) ──────────
    _sec("GRAPHIQUE — CONVERGENCE DE L'ENTRAÎNEMENT (SYNTHÉTIQUE)")
    if training_log:
        plot_global_convergence(training_log, synth_instances,
                                 synth_initial_costs, synth_best_costs, CFG['OUTDIR'])
    else:
        print("  (pas de courbe de convergence : modèle rechargé, aucun "
              "entraînement effectué dans ce run)")

    # ── [B] Convergence détaillée + [C] Routes avant/après — SUR LE TEST ────
    _sec("GRAPHIQUES — DÉTAIL DU TEST (INSTANCES SOLOMON EXEMPLES)")
    plot_instance_convergence_examples(test_log, test_instances, example_idxs, CFG['OUTDIR'])
    plot_routes_examples(test_instances, test_initial_sols, best_solutions, example_idxs, CFG['OUTDIR'])

    # ── [D] TABLEAU FINAL : SITUATION 1 (test gelé, entraîné sur synthétique) ─
    # vs BKS (gap % par instance + gap moyen) ──────────────────────────────
    bks_rows = build_bks_comparison_table(
        test_instances, initial_costs, best_costs, best_solutions, instance_times,
        csv_path=os.path.join(CFG['OUTDIR'], 'solomon_comparaison_bks.csv'),
        latex_path=os.path.join(CFG['OUTDIR'], 'solomon_tableau_bks.tex'),
    )
    # ── [E] Graphique dédié : gap % vs BKS par instance + gap moyen (TEST) ──
    _sec("GRAPHIQUE GAP % vs BKS (TEST)")
    plot_gap_vs_bks(bks_rows, CFG['OUTDIR'])

    print(f"\n  ✓ Terminé en {time.time()-t0:.1f}s ({(time.time()-t0)/3600:.2f}h)\n")


════════════════════════════════════════════════════════════════════════
  MODE ENTRAÎNEMENT/TEST DISJOINT
  ENTRAÎNEMENT = 80 instances SYNTHÉTIQUES générées | TEST = 56 instances Solomon officielles (gelé)
════════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════════
  DIAGNOSTIC — LICENCE GUROBI
════════════════════════════════════════════════════════════════════════

  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée (erreur 10010 : Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information)
    → Licence Community/gratuite, plafonnée à ~2000 variables et ~2000 contraintes.
    → Le repair Gurobi de la Situation 1 (jusqu'à ~15000 variables selon K_DESTROY=24 et le nombre de véhicules) peut dépasser cette limite sur certaines instances — un repli automatique vers le repair glouton est déjà en place, donc pas de plantage, mais un usage de Gurobi moins fréquent que

FileNotFoundError: Aucun fichier d'instance trouvé dans C:\Users\amelb\PhD Thesis\outputs\Solomon. Vérifiez que le dossier existe et contient les .txt Solomon.

##  DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON + GAP PAR RAPPORT AU BKS  (30 épisodes) ==>+12.31% (Part2)

In [5]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON    ║
║                          + GAP PAR RAPPORT AU BKS                           ║
║                                                                              ║
║  Version corrigée : reprend toutes les visualisations de la variante        ║
║  précédente (courbes de convergence, tracés des routes avant/après,        ║
║  graphique dédié gap % vs BKS, tableau récapitulatif véhicules/distance)   ║
║  MAIS applique désormais un vrai protocole ENTRAÎNEMENT / TEST DISJOINT,   ║
║  en un seul lancement, sans aucune intervention manuelle :                 ║
║                                                                              ║
║   ÉTAPE 1 (ENTRAÎNEMENT) — génère des instances SYNTHÉTIQUES (reprises de   ║
║      generate_instance() de main_vrptw_comparaison_gurobi.py / "Code 1")    ║
║      et entraîne le DQN dessus (poids mis à jour via store/learn/decay).    ║
║                                                                              ║
║   ÉTAPE 2 (TEST) — charge les 56 instances Solomon OFFICIELLES et évalue    ║
║      le même agent, GELÉ (aucun store/learn/decay_epsilon, uniquement      ║
║      select_action en pur mode inférence), sur ce jeu jamais vu à          ║
║      l'entraînement. Toutes les courbes/tracés/tableaux ci-dessous portent  ║
║      désormais sur ce TEST gelé, pas sur les résultats d'entraînement.     ║
║                                                                              ║
║  Situation 1 = LNS + DQN(entraîné sur synthétique) + Gurobi, ÉVALUÉ sur     ║
║                Solomon (test)                                → gap vs BKS   ║
║  (La "Situation 2 = Gurobi seul sur l'instance complète" reste retirée,     ║
║   à la demande, car sans limite de temps elle est beaucoup trop longue      ║
║   sur des instances à 100 clients avec fenêtres de temps serrées.)          ║
║                                                                              ║
║  Visualisations produites (toutes sur le TEST gelé, sauf la 1ère) :        ║
║    [A] Convergence de l'ENTRAÎNEMENT (synthétique) : reward global +       ║
║        amélioration % par instance synthétique                            ║
║    [B] Convergence détaillée (instances Solomon exemples, TEST) : coût     ║
║        final/meilleur par épisode local + trajectoire de coût             ║
║    [C] Tracés des routes AVANT (LNS) / APRÈS (DQN gelé) pour des           ║
║        instances Solomon exemples                                         ║
║    [D] Tableau récapitulatif : Instance / Nb véhicules / Distance / Gap %  ║
║        (TEST Solomon vs BKS)                                              ║
║    [E] Graphique dédié gap % vs BKS par instance + gap moyen              ║
║    [F] Diagnostic automatique de licence Gurobi (taille limitée ou non)   ║
║                                                                              ║
║  Aucun rappel manuel de "Code 1" n'est nécessaire : la génération +        ║
║  l'entraînement synthétique sont réintégrés ici (MODULE 1bis) et           ║
║  s'exécutent automatiquement avant le chargement de Solomon. Voir le flag  ║
║  optionnel CFG['REUSE_EXISTING_SYNTH_MODEL'] plus bas si vous préférez     ║
║  recharger un .pth déjà entraîné au lieu de ré-entraîner à chaque run.     ║
║                                                                              ║
║  Par défaut, INSTANCE_FILES contient toutes les instances *.txt trouvées   ║
║  dans outputs/solomon56/. Pour se limiter à 6 instances représentatives    ║
║  (une par groupe C1/C2/R1/R2/RC1/RC2), décommentez le bloc plus bas.       ║
║                                                                              ║
║  Exécution :  python main_vrptw_synth_train_solomon_convergence.py         ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import os, sys, json, time, copy, random, csv, glob
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, deque
import torch
import torch.nn as nn
import torch.optim as optim
import gurobipy as gp
from gurobipy import GRB
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG GLOBALE
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    # ── ÉTAPE 1 : ENTRAÎNEMENT sur instances SYNTHÉTIQUES (jamais Solomon) ──
    N_SYNTH_INSTANCES          = 80,     # comme les 80 instances de Code 1
    SYNTH_N_CUSTOMERS          = 100,
    SYNTH_N_VEHICLES           = None,   # None → calculé depuis la demande (cf. generate_instance)
    SYNTH_VEHICLE_SAFETY_FACTOR= 1.5,
    SYNTH_CAPACITY             = 200,
    SYNTH_HORIZON              = 1000,
    EPISODES_PER_INSTANCE      = 10,     # épisodes d'ENTRAÎNEMENT par instance synthétique
    # Optionnel : True → recharge un .pth déjà entraîné sur du synthétique
    # (voir MODEL_FILE) au lieu de régénérer + ré-entraîner. False (par
    # défaut) → tout se fait automatiquement dans CE run, sans intervention
    # manuelle, aucun besoin de relancer "Code 1" séparément.
    REUSE_EXISTING_SYNTH_MODEL = True,
    # ── ÉTAPE 2 : TEST gelé sur les 56 instances Solomon officielles ───────
    EPISODES_PER_INSTANCE_TEST = 3,      # épisodes d'ÉVALUATION (aucun apprentissage) par instance Solomon
    # VEHICLE_SAFETY_FACTOR = None : on utilise la flotte OFFICIELLE de chaque
    # instance, pas une borne calculée sur la demande. Sur R101/RC101
    # (fenêtres de temps serrées), le BKS utilise 19 et 14 véhicules alors
    # que la demande seule suggérerait ~12-14 : une borne basée uniquement
    # sur la demande sous-estime le besoin réel et peut rendre le MILP
    # infaisable. La flotte officielle est plus grande (MILP plus lourd)
    # mais garantit qu'une solution proche du BKS reste atteignable.
    VEHICLE_SAFETY_FACTOR  = None,
    LNS_ITER               = 30,
    LNS_K_RATIO            = 0.20,
    K_DESTROY_RATIO        = 0.24,
    MAX_STEPS              = 20,
    # repair par bouts (Situation 1, appelé à CHAQUE step du LNS/DQN — voir
    # check_gurobi_license() plus bas pour l'impact d'une licence à taille
    # limitée). None = aucune limite de temps (supporté par le code).
    GUROBI_TLIMIT           = 2.0,
    GUROBI_GAP              = 0.05,
    HIDDEN                  = 64,
    LR                      = 1e-3,
    GAMMA                   = 0.95,
    EPS_START                = 1.0,
    EPS_END                  = 0.05,
    EPS_DECAY                = 0.95,  # ⚠ ignoré — recalculé automatiquement plus bas (cf. commentaire)
    BUFFER_SIZE               = 5000,
    BATCH_SIZE                = 32,
    TARGET_UPDATE              = 10,
    # ── Instances exemples (pour courbes détaillées + routes, TEST/Solomon) ─
    # Noms recherchés en priorité (représentatifs des 6 groupes Solomon) ;
    # si un nom n'est pas présent dans INSTANCE_FILES, il est ignoré et
    # remplacé automatiquement par le 1er type disponible.
    EXAMPLE_NAMES              = ['C101', 'R101', 'RC101'],
    OUTDIR                        = os.path.join(os.getcwd(), 'outputs'),
    MODELDIR                       = os.path.join(os.getcwd(), 'outputs', 'model'),
    MODEL_FILE                      = 'dqn_vrptw_synth_trained.pth',
)
CFG['LNS_K'] = 20   # ~20% de 100 clients, cohérent avec LNS_K_RATIO
CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))  # 24, cohérent avec K_DESTROY_RATIO
# ── Recalibrage automatique de EPS_DECAY ────────────────────────────────
# La valeur fixe 0.95 était calibrée pour un tout petit nombre d'épisodes
# (contexte d'origine : entraînement direct sur quelques dizaines
# d'instances Solomon). Avec EPS_DECAY=0.95, epsilon atteint son plancher
# (EPS_END) après seulement ~58 décroissances — soit, avec les valeurs par
# défaut (N_SYNTH_INSTANCES=80, EPISODES_PER_INSTANCE=10), après à peine
# ~6 des 80 instances synthétiques : plus de 90% de l'entraînement se
# ferait alors avec epsilon au plancher (quasi aucune exploration), quel
# que soit EPISODES_PER_INSTANCE — et AUGMENTER EPISODES_PER_INSTANCE
# aggrave le problème (le plancher est atteint en encore moins d'instances
# couvertes). On recalcule donc EPS_DECAY ici pour que epsilon décroisse
# de EPS_START à EPS_END sur TOUTE la durée prévue de l'entraînement
# (N_SYNTH_INSTANCES × EPISODES_PER_INSTANCE décroissances), quel que soit
# le nombre d'épisodes par instance choisi.
_n_decays_total = max(1, CFG['N_SYNTH_INSTANCES'] * CFG['EPISODES_PER_INSTANCE'])
CFG['EPS_DECAY'] = (CFG['EPS_END'] / CFG['EPS_START']) ** (1.0 / _n_decays_total)
os.makedirs(CFG['OUTDIR'],   exist_ok=True)
os.makedirs(CFG['MODELDIR'], exist_ok=True)
ACTION_NAMES = {
    0: "Destruction spatiale",
    1: "Destruction temporelle",
    2: "Destruction par coût élevé",
    3: "Destruction aléatoire",
}
# ── Dossier contenant les instances Solomon officielles (extraites du zip) ──
SOLOMON_DIR = os.path.join(CFG['OUTDIR'], 'Solomon')
# Toutes les instances *.txt trouvées (décommentez le bloc ci-dessous pour
# revenir à seulement 6 instances représentatives, une par groupe
# C1/C2/R1/R2/RC1/RC2, afin de garder un temps d'exécution raisonnable).
INSTANCE_FILES = sorted(glob.glob(os.path.join(SOLOMON_DIR, '*.txt')))
# INSTANCE_FILES = [
#     os.path.join(SOLOMON_DIR, 'c101.txt'),
#     os.path.join(SOLOMON_DIR, 'c201.txt'),
#     os.path.join(SOLOMON_DIR, 'r101.txt'),
#     os.path.join(SOLOMON_DIR, 'r201.txt'),
#     os.path.join(SOLOMON_DIR, 'rc101.txt'),
#     os.path.join(SOLOMON_DIR, 'rc201.txt'),
# ]
# ── BKS officiel complet, 56 instances (source : SINTEF / Solomon 1987) ──────
BKS_REFERENCE = {
    'C101': {'vehicles': 10, 'distance': 828.94}, 'C102': {'vehicles': 10, 'distance': 828.94},
    'C103': {'vehicles': 10, 'distance': 828.06}, 'C104': {'vehicles': 10, 'distance': 824.78},
    'C105': {'vehicles': 10, 'distance': 828.94}, 'C106': {'vehicles': 10, 'distance': 828.94},
    'C107': {'vehicles': 10, 'distance': 828.94}, 'C108': {'vehicles': 10, 'distance': 828.94},
    'C109': {'vehicles': 10, 'distance': 828.94},
    'C201': {'vehicles': 3, 'distance': 591.56}, 'C202': {'vehicles': 3, 'distance': 591.56},
    'C203': {'vehicles': 3, 'distance': 591.17}, 'C204': {'vehicles': 3, 'distance': 590.60},
    'C205': {'vehicles': 3, 'distance': 588.88}, 'C206': {'vehicles': 3, 'distance': 588.49},
    'C207': {'vehicles': 3, 'distance': 588.29}, 'C208': {'vehicles': 3, 'distance': 588.32},
    'R101': {'vehicles': 19, 'distance': 1650.80}, 'R102': {'vehicles': 17, 'distance': 1486.12},
    'R103': {'vehicles': 13, 'distance': 1292.68}, 'R104': {'vehicles': 9,  'distance': 1007.31},
    'R105': {'vehicles': 14, 'distance': 1377.11}, 'R106': {'vehicles': 12, 'distance': 1252.03},
    'R107': {'vehicles': 10, 'distance': 1104.66}, 'R108': {'vehicles': 9,  'distance': 960.88},
    'R109': {'vehicles': 11, 'distance': 1194.73}, 'R110': {'vehicles': 10, 'distance': 1118.84},
    'R111': {'vehicles': 10, 'distance': 1096.72}, 'R112': {'vehicles': 9,  'distance': 982.14},
    'R201': {'vehicles': 4, 'distance': 1252.37}, 'R202': {'vehicles': 3, 'distance': 1191.70},
    'R203': {'vehicles': 3, 'distance': 939.50},  'R204': {'vehicles': 2, 'distance': 825.52},
    'R205': {'vehicles': 3, 'distance': 994.43},  'R206': {'vehicles': 3, 'distance': 906.14},
    'R207': {'vehicles': 2, 'distance': 890.61},  'R208': {'vehicles': 2, 'distance': 726.82},
    'R209': {'vehicles': 3, 'distance': 909.16},  'R210': {'vehicles': 3, 'distance': 939.37},
    'R211': {'vehicles': 2, 'distance': 885.71},
    'RC101': {'vehicles': 14, 'distance': 1696.95}, 'RC102': {'vehicles': 12, 'distance': 1554.75},
    'RC103': {'vehicles': 11, 'distance': 1261.67}, 'RC104': {'vehicles': 10, 'distance': 1135.48},
    'RC105': {'vehicles': 13, 'distance': 1629.44}, 'RC106': {'vehicles': 11, 'distance': 1424.73},
    'RC107': {'vehicles': 11, 'distance': 1230.48}, 'RC108': {'vehicles': 10, 'distance': 1139.82},
    'RC201': {'vehicles': 4, 'distance': 1406.94}, 'RC202': {'vehicles': 3, 'distance': 1365.65},
    'RC203': {'vehicles': 3, 'distance': 1049.62}, 'RC204': {'vehicles': 3, 'distance': 798.46},
    'RC205': {'vehicles': 4, 'distance': 1297.65}, 'RC206': {'vehicles': 3, 'distance': 1146.32},
    'RC207': {'vehicles': 3, 'distance': 1061.14}, 'RC208': {'vehicles': 3, 'distance': 828.14},
}
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1  —  LECTURE DES INSTANCES SOLOMON OFFICIELLES  (jeu de TEST)
# ══════════════════════════════════════════════════════════════════════════════
def euclidean(a, b):
    return float(np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2))
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1bis  —  GÉNÉRATEUR D'INSTANCES SYNTHÉTIQUES  (jeu d'ENTRAÎNEMENT)
#  Repris tel quel de main_vrptw_comparaison_gurobi.py ("Code 1") : mêmes
#  fonctions _coords_C/_coords_R/_coords_RC et generate_instance(), pour que
#  les instances d'entraînement soient générées exactement comme dans Code 1.
# ══════════════════════════════════════════════════════════════════════════════
def _coords_C(n, rng):
    n_cl = max(3, n // 10)
    centers = rng.uniform(15, 85, (n_cl, 2))
    pts = []
    for i in range(n):
        p = rng.normal(centers[i % n_cl], 8)
        pts.append(np.clip(p, 1, 99))
    return np.array(pts)
def _coords_R(n, rng):
    return rng.uniform(1, 99, (n, 2))
def _coords_RC(n, rng):
    h = n // 2
    return np.vstack([_coords_C(h, rng), _coords_R(n - h, rng)])
def generate_instance(n_customers=50, n_vehicles=None, capacity=200,
                      type_='C', seed=42, horizon=1000,
                      vehicle_safety_factor=1.5):
    rng     = np.random.RandomState(seed)
    depot   = np.array([50.0, 50.0])
    gen_fn  = {'C': _coords_C, 'R': _coords_R, 'RC': _coords_RC}[type_]
    coords  = gen_fn(n_customers, rng)
    max_dem = max(1, capacity // 5)
    demands = rng.randint(1, max_dem + 1, n_customers)
    svc     = rng.randint(10, 31, n_customers)
    if n_vehicles is None:
        min_vehicles = int(np.ceil(demands.sum() / capacity))
        n_vehicles   = max(2, int(np.ceil(min_vehicles * vehicle_safety_factor)))
    customers = []
    for i in range(n_customers):
        d        = euclidean(depot, coords[i])
        earliest = max(0.0, round(d + rng.uniform(-15, 15), 1))
        delta    = rng.uniform(30, 80)
        latest   = min(float(horizon), round(earliest + delta, 1))
        latest   = max(latest, earliest + svc[i] + 5)
        latest   = min(latest, float(horizon))
        if d > latest:
            earliest = round(d, 1)
            latest   = min(float(horizon), round(earliest + delta, 1))
            latest   = max(latest, earliest + svc[i] + 5)
        if d + svc[i] + d > horizon:
            svc[i] = max(5, int(horizon - 2 * d - 5))
        customers.append({
            'id':           i + 1,
            'x':            float(round(coords[i][0], 2)),
            'y':            float(round(coords[i][1], 2)),
            'demand':       int(demands[i]),
            'earliest':     float(earliest),
            'latest':       float(latest),
            'service_time': int(svc[i]),
        })
    return {
        'depot': {
            'id': 0, 'x': 50.0, 'y': 50.0, 'demand': 0,
            'earliest': 0.0, 'latest': float(horizon), 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    capacity,
        'horizon':     horizon,
        'type':        type_,
        'seed':        seed,
        'n_customers': n_customers,
    }
def generate_synthetic_training_set(cfg, verbose=True):
    """
    Génère les instances synthétiques d'ENTRAÎNEMENT en cyclant les 3 types
    Solomon C / R / RC (même logique que train() dans Code 1 : type =
    ['C','R','RC'][i % 3], seed = i*7+1). Ces instances ne recouvrent JAMAIS
    les 56 instances Solomon officielles utilisées plus bas comme jeu de
    TEST : le DQN ne les voit donc jamais avant l'évaluation gelée.
    Chaque instance reçoit en plus une clé 'name' (ex. 'SYN-C001') pour
    rester compatible avec les fonctions d'entraînement/affichage qui
    utilisent inst['name'].
    """
    types = ['C', 'R', 'RC']
    instances = []
    if verbose:
        _sec(f"ÉTAPE 1 — GÉNÉRATION DE {cfg['N_SYNTH_INSTANCES']} INSTANCES "
             f"SYNTHÉTIQUES (ENTRAÎNEMENT)")
    for i in range(cfg['N_SYNTH_INSTANCES']):
        t    = types[i % 3]
        seed = i * 7 + 1
        inst = generate_instance(
            n_customers = cfg['SYNTH_N_CUSTOMERS'],
            n_vehicles  = cfg['SYNTH_N_VEHICLES'],
            capacity    = cfg['SYNTH_CAPACITY'],
            horizon     = cfg['SYNTH_HORIZON'],
            type_       = t,
            seed        = seed,
            vehicle_safety_factor = cfg['SYNTH_VEHICLE_SAFETY_FACTOR'],
        )
        inst['name'] = f"SYN-{t}{i+1:03d}"
        instances.append(inst)
        if verbose:
            print(f"  Instance {i+1:03d}/{cfg['N_SYNTH_INSTANCES']} | {inst['name']:<10} "
                  f"type={t:<3} | seed={seed} | véhicules={inst['n_vehicles']}")
    return instances
def parse_solomon_instance(filepath, vehicle_safety_factor=None):
    """
    Parse un fichier d'instance Solomon au format texte officiel :
        <NOM>
        VEHICLE
        NUMBER  CAPACITY
        <n>     <cap>
        CUSTOMER
        CUST NO.  XCOORD.  YCOORD.  DEMAND  READY TIME  DUE DATE  SERVICE TIME
        0  ...   (dépôt)
        1  ...
        ...
    Retourne un dict compatible avec le reste du pipeline.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = [l.rstrip('\n') for l in f]
    name = lines[0].strip()
    idx_veh = next(i for i, l in enumerate(lines) if l.strip() == 'VEHICLE')
    veh_number, veh_capacity = map(int, lines[idx_veh + 2].split())
    idx_cust = next(i for i, l in enumerate(lines) if l.strip() == 'CUSTOMER')
    data_lines = [l for l in lines[idx_cust + 2:] if l.strip()]
    rows = [list(map(float, l.split())) for l in data_lines]
    depot_row = rows[0]
    customers = []
    for r in rows[1:]:
        cid, x, y, dem, ready, due, svc = r
        customers.append({
            'id':           int(cid),
            'x':            x,
            'y':            y,
            'demand':       int(dem),
            'earliest':     ready,
            'latest':       due,
            'service_time': int(svc),
        })
    name_upper = name.upper()
    type_ = ''.join(ch for ch in name_upper if ch.isalpha())   # 'C101' -> 'C', 'RC101' -> 'RC'
    n_vehicles = veh_number
    if vehicle_safety_factor is not None:
        total_demand = sum(c['demand'] for c in customers)
        min_veh      = int(np.ceil(total_demand / veh_capacity))
        n_vehicles   = min(veh_number, max(2, int(np.ceil(min_veh * vehicle_safety_factor))))
    return {
        'depot': {
            'id': 0, 'x': depot_row[1], 'y': depot_row[2], 'demand': 0,
            'earliest': depot_row[4], 'latest': depot_row[5], 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    veh_capacity,
        'horizon':     depot_row[5],
        'type':        type_,
        'seed':        name_upper,
        'n_customers': len(customers),
        'name':        name_upper,          # nom officiel normalisé (ex. 'C101')
    }
def load_all_instances(files, vehicle_safety_factor=None):
    if not files:
        raise FileNotFoundError(
            f"Aucun fichier d'instance trouvé dans {SOLOMON_DIR}. "
            f"Vérifiez que le dossier existe et contient les .txt Solomon."
        )
    instances = []
    for fp in files:
        inst = parse_solomon_instance(fp, vehicle_safety_factor=vehicle_safety_factor)
        instances.append(inst)
        print(f"  Chargé : {inst['name']:<8} type={inst['type']:<3} "
              f"clients={inst['n_customers']} véhicules={inst['n_vehicles']} "
              f"capacité={inst['capacity']} horizon={inst['horizon']}")
    return instances
def pick_example_indices(instances, preferred_names=None, max_examples=3):
    """
    Sélectionne des instances "exemples" pour les courbes détaillées et les
    tracés de routes. Cherche d'abord les noms préférés (ex. C101/R101/RC101).
    Si absents, retombe sur la 1ère instance de chaque type détecté, puis sur
    les toutes premières instances de la liste.
    """
    preferred_names = preferred_names or []
    by_name = {inst['name']: i for i, inst in enumerate(instances)}
    idxs = [by_name[nm] for nm in preferred_names if nm in by_name]
    if not idxs:
        seen_types = {}
        for i, inst in enumerate(instances):
            if inst['type'] not in seen_types:
                seen_types[inst['type']] = i
            if len(seen_types) >= max_examples:
                break
        idxs = list(seen_types.values())
    if not idxs:
        idxs = list(range(min(max_examples, len(instances))))
    return idxs[:max_examples]
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 2  —  SOLUTION INITIALE PAR LNS  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
def _node_xy(nid, instance):
    if nid == 0:
        return instance['depot']['x'], instance['depot']['y']
    c = instance['customers'][nid - 1]
    return c['x'], c['y']
def _node_info(nid, instance):
    if nid == 0:
        return instance['depot']
    return instance['customers'][nid - 1]
def compute_cost(solution, instance):
    depot = (instance['depot']['x'], instance['depot']['y'])
    total = 0.0
    for route in solution:
        if not route:
            continue
        prev = depot
        for cid in route:
            cur   = _node_xy(cid, instance)
            total += euclidean(prev, cur)
            prev  = cur
        total += euclidean(prev, depot)
    return total
def _route_feasible(route, instance):
    if not route:
        return True
    cap  = instance['capacity']
    load = sum(_node_info(c, instance)['demand'] for c in route)
    if load > cap:
        return False
    depot = instance['depot']
    t     = 0.0
    px, py = depot['x'], depot['y']
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        if t > info['latest']:
            return False
        t      = max(t, info['earliest']) + info['service_time']
        px, py = cx, cy
    t += euclidean((px, py), (depot['x'], depot['y']))
    return t <= depot['latest']
def is_feasible(solution, instance):
    return all(_route_feasible(r, instance) for r in solution)
def solution_stats(solution, instance):
    cap        = instance['capacity']
    total_cost = compute_cost(solution, instance)
    n_vehicles = sum(1 for r in solution if r)
    loads      = []
    total_wait = 0.0
    for route in solution:
        if not route:
            continue
        load = sum(_node_info(c, instance)['demand'] for c in route)
        loads.append(load / cap)
        depot  = instance['depot']
        t      = 0.0
        px, py = depot['x'], depot['y']
        for cid in route:
            cx, cy = _node_xy(cid, instance)
            t     += euclidean((px, py), (cx, cy))
            info   = _node_info(cid, instance)
            total_wait += max(0.0, info['earliest'] - t)
            t      = max(t, info['earliest']) + info['service_time']
            px, py = cx, cy
    return {
        'total_cost':         round(total_cost, 2),
        'n_vehicles':         n_vehicles,
        'avg_load_ratio':     round(np.mean(loads) if loads else 0.0, 3),
        'total_wait_time':    round(total_wait, 2),
        'n_customers_served': sum(len(r) for r in solution),
    }
def _insertion_cost(cid, pos, route, instance):
    depot   = (instance['depot']['x'], instance['depot']['y'])
    nodes   = [0] + route + [0]
    prev_id = nodes[pos]
    next_id = nodes[pos + 1]
    px, py  = _node_xy(prev_id, instance) if prev_id != 0 else depot
    nx, ny  = _node_xy(next_id, instance) if next_id != 0 else depot
    cx, cy  = _node_xy(cid, instance)
    return (euclidean((px, py), (cx, cy)) + euclidean((cx, cy), (nx, ny))
            - euclidean((px, py), (nx, ny)))
def _try_insert(cid, route, instance):
    best_cost, best_pos = float('inf'), -1
    for pos in range(len(route) + 1):
        candidate = route[:pos] + [cid] + route[pos:]
        if _route_feasible(candidate, instance):
            cost = _insertion_cost(cid, pos, route, instance)
            if cost < best_cost:
                best_cost, best_pos = cost, pos
    return best_cost, best_pos
def _repair_greedy(removed, partial, instance):
    routes = [list(r) for r in partial]
    for cid in removed:
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _build_initial(instance):
    customers = sorted(instance['customers'], key=lambda c: c['earliest'])
    routes    = []
    for c in customers:
        cid = c['id']
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _lns_improve(solution, instance, n_iter=50, k=10, seed=0):
    rng       = np.random.RandomState(seed)
    current   = [list(r) for r in solution]
    best      = copy.deepcopy(current)
    best_cost = compute_cost(best, instance)
    for _ in range(n_iter):
        all_c   = [c for r in current for c in r]
        k_      = min(k, max(1, len(all_c) - 1))
        removed = list(rng.choice(all_c, k_, replace=False))
        removed_set = set(removed)
        partial = [[c for c in r if c not in removed_set] for r in current]
        partial = [r for r in partial if r]
        candidate = _repair_greedy(removed, partial, instance)
        c_cost    = compute_cost(candidate, instance)
        if c_cost < best_cost and is_feasible(candidate, instance):
            best, best_cost = copy.deepcopy(candidate), c_cost
        current = candidate
    return best
def get_initial_solution(instance, lns_iter=50, lns_k=10):
    sol = _build_initial(instance)
    sol = _lns_improve(sol, instance, n_iter=lns_iter, k=lns_k)
    sol = [r for r in sol if r]
    return sol, solution_stats(sol, instance)
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 3  —  ENVIRONNEMENT VRPTW  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class VRPTWEnv:
    def __init__(self, instance, initial_solution,
                 K=15, max_steps=20,
                 gurobi_tlimit=2.0, gurobi_gap=0.05,
                 verbose=False):
        self.instance    = instance
        self.initial_sol = copy.deepcopy(initial_solution)
        self.K           = K
        self.max_steps   = max_steps
        self.tlimit      = gurobi_tlimit
        self.gap         = gurobi_gap
        self.verbose     = verbose
        self.obs_dim     = 7
        self.n_actions   = 4
        self._dist = {}
        depot = (instance['depot']['x'], instance['depot']['y'])
        for c in instance['customers']:
            d = euclidean(depot, (c['x'], c['y']))
            self._dist[(0, c['id'])] = d
            self._dist[(c['id'], 0)] = d
        for ci in instance['customers']:
            for cj in instance['customers']:
                self._dist[(ci['id'], cj['id'])] = euclidean(
                    (ci['x'], ci['y']), (cj['x'], cj['y']))
        self.reset()
    def _d(self, a, b):
        return self._dist.get((a, b), 0.0)
    def _info(self, nid):
        return _node_info(nid, self.instance)
    def reset(self, start_solution=None):
        if start_solution is not None:
            self.solution = copy.deepcopy(start_solution)
        else:
            self.solution = copy.deepcopy(self.initial_sol)
        self.current_cost     = compute_cost(self.solution, self.instance)
        self.best_cost        = self.current_cost
        self.initial_cost     = compute_cost(self.initial_sol, self.instance)
        self.step_count       = 0
        self.steps_no_improve = 0
        self._action_history  = []
        return self._obs()
    def _obs(self):
        stats = solution_stats(self.solution, self.instance)
        ic    = max(self.initial_cost, 1.0)
        return np.array([
            self.current_cost / ic,
            stats['n_vehicles'] / max(self.instance['n_vehicles'], 1),
            stats['avg_load_ratio'],
            stats['total_wait_time'] / max(ic * 0.5, 1.0),
            self.steps_no_improve / self.max_steps,
            self.best_cost / ic,
            1.0,
        ], dtype=np.float32)
    def _destroy(self, action):
        all_c = [c for r in self.solution for c in r]
        if not all_c:
            return [], copy.deepcopy(self.solution)
        K   = min(self.K, len(all_c))
        rng = np.random.RandomState(self.step_count + action * 100)
        if action == 0:
            pivot   = int(rng.choice(all_c))
            px, py  = self._info(pivot)['x'], self._info(pivot)['y']
            dists   = {c: euclidean((px, py),
                                    (self._info(c)['x'], self._info(c)['y']))
                       for c in all_c}
            destroyed = sorted(all_c, key=lambda c: dists[c])[:K]
        elif action == 1:
            pivot    = int(rng.choice(all_c))
            e_pivot  = self._info(pivot)['earliest']
            tdiffs   = {c: abs(self._info(c)['earliest'] - e_pivot) for c in all_c}
            destroyed = sorted(all_c, key=lambda c: tdiffs[c])[:K]
        elif action == 2:
            savings = {}
            for r in self.solution:
                if not r:
                    continue
                nodes = [0] + r + [0]
                for i, cid in enumerate(r):
                    prev, nxt     = nodes[i], nodes[i + 2]
                    savings[cid]  = (self._d(prev, cid) + self._d(cid, nxt)
                                     - self._d(prev, nxt))
            destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
        else:
            idx       = rng.choice(len(all_c), K, replace=False)
            destroyed = [all_c[i] for i in idx]
        destroyed_set = set(destroyed)
        partial = [[c for c in r if c not in destroyed_set]
                   for r in self.solution]
        partial = [r for r in partial if r]
        return list(destroyed), partial
    def _repair_gurobi(self, destroyed, partial):
        if not destroyed:
            return copy.deepcopy(self.solution)
        inst    = self.instance
        depot   = inst['depot']
        horizon = inst['horizon']
        cap     = inst['capacity']
        nodes   = [0] + destroyed
        N       = len(nodes)
        V       = min(len(destroyed), inst['n_vehicles'])
        env, m = None, None
        try:
            env = gp.Env(empty=True)
            env.setParam('OutputFlag', 0)
            env.start()
            m = gp.Model(env=env)
            if self.tlimit is not None:
                m.setParam('TimeLimit', self.tlimit)
            m.setParam('MIPGap',     self.gap)
            m.setParam('OutputFlag', 0)
            x = m.addVars(N, N, V, vtype=GRB.BINARY, name='x')
            t = m.addVars(N, V, lb=0, ub=horizon,    name='t')
            m.setObjective(
                gp.quicksum(
                    self._d(nodes[i], nodes[j]) * x[i, j, v]
                    for i in range(N) for j in range(N) for v in range(V)
                    if i != j
                ),
                GRB.MINIMIZE
            )
            for i in range(1, N):
                m.addConstr(
                    gp.quicksum(x[i, j, v]
                                for j in range(N) for v in range(V)
                                if i != j) == 1
                )
            for v in range(V):
                for i in range(1, N):
                    m.addConstr(
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j) ==
                        gp.quicksum(x[j, i, v] for j in range(N) if i != j)
                    )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(x[0, j, v] for j in range(1, N)) <= 1
                )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(
                        inst['customers'][nodes[i] - 1]['demand'] *
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j)
                        for i in range(1, N)
                    ) <= cap
                )
            M = horizon + 1000
            for v in range(V):
                m.addConstr(t[0, v] == 0)
                for i in range(N):
                    ni  = nodes[i]
                    ei  = depot['earliest'] if ni == 0 else inst['customers'][ni-1]['earliest']
                    li  = depot['latest']   if ni == 0 else inst['customers'][ni-1]['latest']
                    sti = depot['service_time'] if ni == 0 else inst['customers'][ni-1]['service_time']
                    m.addConstr(t[i, v] >= ei)
                    m.addConstr(t[i, v] <= li)
                    for j in range(N):
                        if i == j:
                            continue
                        m.addConstr(
                            t[j, v] >= t[i, v] + sti + self._d(ni, nodes[j])
                            - M * (1 - x[i, j, v])
                        )
            m.optimize()
            if m.SolCount > 0:
                eps        = 0.5
                new_routes = []
                for v in range(V):
                    if sum(x[0, j, v].X for j in range(1, N)) < eps:
                        continue
                    route_v, cur = [], 0
                    for _ in range(N):
                        nxt = next((j for j in range(N)
                                    if j != cur and x[cur, j, v].X > eps), None)
                        if nxt is None or nxt == 0:
                            break
                        route_v.append(nodes[nxt])
                        cur = nxt
                    if route_v:
                        new_routes.append(route_v)
                full = [r for r in list(partial) + new_routes if r]
                return full
            else:
                raise ValueError("Gurobi : aucune solution trouvée")
        except Exception as e:
            if self.verbose:
                print(f"    [Gurobi→Greedy fallback] {e}")
            return _repair_greedy(destroyed, partial, self.instance)
        finally:
            if m is not None:
                m.dispose()
            if env is not None:
                env.dispose()
    def step(self, action):
        old_cost            = self.current_cost
        destroyed, partial  = self._destroy(action)
        new_sol             = self._repair_gurobi(destroyed, partial)
        new_cost            = compute_cost(new_sol, self.instance)
        feasible            = is_feasible(new_sol, self.instance)
        if feasible:
            self.solution     = new_sol
            self.current_cost = new_cost
        else:
            new_cost = old_cost * 1.001
        improvement = old_cost - self.current_cost
        if improvement > 0:
            self.best_cost        = min(self.best_cost, self.current_cost)
            self.steps_no_improve = 0
        else:
            self.steps_no_improve += 1
        reward = improvement / max(self.initial_cost, 1.0)
        self.step_count += 1
        self._action_history.append(action)
        done = (self.step_count >= self.max_steps)
        info = {
            'cost':          self.current_cost,
            'best_cost':     self.best_cost,
            'action_taken':  action,
            'action_name':   ACTION_NAMES[action],
            'improvement':   improvement,
            'destroyed_k':   len(destroyed),
            'destroyed_ids': [int(d) for d in destroyed[:5]],
            'feasible':      feasible,
        }
        return self._obs(), float(reward), done, info
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 4  —  AGENT DQN  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((
            np.array(s,  dtype=np.float32), int(a), float(r),
            np.array(s2, dtype=np.float32), float(done),
        ))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.FloatTensor(np.stack(s)), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(np.stack(s2)),
                torch.FloatTensor(d))
    def __len__(self):
        return len(self.buf)
class DQNNet(nn.Module):
    def __init__(self, obs_dim=7, n_actions=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim,  hidden),  nn.ReLU(),
            nn.Linear(hidden,   hidden),  nn.ReLU(),
            nn.Linear(hidden,   n_actions),
        )
    def forward(self, x):
        return self.net(x)
class DQNAgent:
    def __init__(self, obs_dim=7, n_actions=4, hidden=64,
                 lr=1e-3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_size=2000, batch_size=32,
                 target_update_freq=10, seed=0):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        self.n_actions          = n_actions
        self.gamma              = gamma
        self.epsilon            = epsilon_start
        self.epsilon_end        = epsilon_end
        self.epsilon_decay      = epsilon_decay
        self.batch_size         = batch_size
        self.target_update_freq = target_update_freq
        self.step_count         = 0
        self.q_net      = DQNNet(obs_dim, n_actions, hidden)
        self.target_net = DQNNet(obs_dim, n_actions, hidden)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)
        self.loss_fn   = nn.MSELoss()
    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state).unsqueeze(0))
            return int(q.argmax().item())
    def store(self, s, a, r, s2, done):
        self.buffer.push(s, a, r, s2, done)
    def learn(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        s, a, r, s2, done = self.buffer.sample(self.batch_size)
        q_vals = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q  = self.target_net(s2).max(1)[0]
            targets = r + self.gamma * next_q * (1 - done)
        loss = self.loss_fn(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.optimizer.step()
        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.update_target()
        return float(loss.item())
    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    def save(self, path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        torch.save({'q_net': self.q_net.state_dict(),
                    'target_net': self.target_net.state_dict(),
                    'epsilon': self.epsilon, 'step_count': self.step_count}, path)
        print(f"    [DQN] Modèle sauvegardé → {path}")
    def load(self, path):
        """
        Recharge un checkpoint sauvegardé par save() (mêmes clés). Utilisé
        uniquement si CFG['REUSE_EXISTING_SYNTH_MODEL']=True, pour réutiliser
        un modèle déjà entraîné sur du synthétique au lieu de ré-entraîner.
        """
        ckpt = torch.load(path, map_location='cpu')
        self.q_net.load_state_dict(ckpt['q_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.epsilon    = ckpt.get('epsilon', self.epsilon)
        self.step_count = ckpt.get('step_count', self.step_count)
        print(f"    [DQN] Modèle rechargé ← {path}")
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5  —  ENTRAÎNEMENT DU DQN  (ÉTAPE 1, sur les instances SYNTHÉTIQUES)
#  Stocke cost_trace (trajectoire de coût step par step) dans le log, utilisé
#  ensuite pour les courbes de convergence détaillées. Retourne aussi
#  initial_sols pour les tracés de routes avant/après.
# ══════════════════════════════════════════════════════════════════════════════
BANNER = "═" * 72
def _sec(title):
    print(f"\n{BANNER}\n  {title}\n{BANNER}")
def train_on_instances(instances, episodes_per_instance=30):
    """
    ENTRAÎNEMENT (les poids du DQN sont mis à jour ici via store()/learn()/
    decay_epsilon()). Générique : fonctionne sur n'importe quelle liste
    d'instances au format standard — dans ce script, appelé avec les
    instances SYNTHÉTIQUES produites par generate_synthetic_training_set(),
    JAMAIS avec les instances Solomon (réservées à evaluate_frozen_
    on_instances(), plus bas, pour le TEST).
    Retourne : agent, training_log, initial_costs, best_costs, best_solutions,
               instance_times, initial_sols
    """
    n = len(instances)
    _sec(f"ÉTAPE 1 — LNS initial pour les {n} instances d'entraînement (synthétiques)")
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    _sec("ÉTAPE 1 — Entraînement DQN sur le synthétique")
    agent = DQNAgent(
        obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
        gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
        epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
        buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
        target_update_freq=CFG['TARGET_UPDATE'],
    )
    training_log   = []
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    episode_global = 0
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TRAIN] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)
                next_obs, reward, done, info = env.step(action)
                agent.store(obs, action, reward, next_obs, float(done))
                agent.learn()
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            agent.decay_epsilon()
            training_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TRAIN] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    agent.save(model_path)
    return agent, training_log, initial_costs, best_costs, best_solutions, instance_times, initial_sols
def evaluate_frozen_on_instances(instances, agent, episodes_per_instance=3):
    """
    TEST — ÉVALUATION GELÉE : l'agent DQN déjà entraîné (ÉTAPE 1, sur le
    synthétique) est ici appliqué en pur mode INFÉRENCE sur les instances
    passées en argument (les 56 Solomon officielles). Différences avec
    train_on_instances() :
      - AUCUN appel à agent.store() / agent.learn() / agent.decay_epsilon() :
        les poids du réseau ne sont JAMAIS mis à jour ici.
      - epsilon est temporairement forcé à 0.0 (politique gloutonne, pure
        exploitation du Q-réseau appris), puis restauré à sa valeur
        d'origine à la fin.
      - Le modèle n'est PAS sauvegardé ici (rien n'a changé dans ses poids).
    Le log retourné (test_log) a exactement la même forme que training_log
    (y compris 'cost_trace'), pour pouvoir réutiliser telles quelles les
    fonctions de tracé (plot_instance_convergence_examples, etc.) sur le
    TEST plutôt que sur l'entraînement.
    Retourne : initial_costs, best_costs, best_solutions, instance_times,
               initial_sols, test_log
    """
    n = len(instances)
    _sec(f"ÉTAPE 2 — ÉVALUATION GELÉE (TEST) sur {n} instances Solomon "
         f"— aucun apprentissage, epsilon=0")
    saved_epsilon = agent.epsilon
    agent.epsilon = 0.0
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    test_log       = []
    episode_global = 0
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TEST] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)          # inférence pure (epsilon=0)
                next_obs, reward, done, info = env.step(action)
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            test_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TEST] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    agent.epsilon = saved_epsilon
    return initial_costs, best_costs, best_solutions, instance_times, initial_sols, test_log
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6  —  TABLEAU FINAL : SITUATION 1 vs BKS
#  (la Situation 2 "Gurobi seul sur l'instance complète" a été retirée)
# ══════════════════════════════════════════════════════════════════════════════
def build_bks_comparison_table(instances, initial_costs, best_costs, best_solutions,
                                instance_times, csv_path=None, latex_path=None):
    """
    Tableau final : Situation 1 (LNS+DQN+Gurobi) vs BKS, avec gap % et le
    nombre de véhicules utilisés pour chaque instance.
    """
    _sec("TABLEAU FINAL — SITUATION 1 vs BKS")
    rows = []
    for i, inst in enumerate(instances):
        name     = inst['name']
        bks      = BKS_REFERENCE.get(name)
        bks_dist = bks['distance'] if bks else None
        n_veh_s1 = sum(1 for r in best_solutions[i] if r)
        row = {
            'instance':             name,
            'type':                 inst['type'],
            'bks_vehicules':        bks['vehicles'] if bks else None,
            'bks_distance':         bks_dist,
            'n_vehicules_situation1': n_veh_s1,
            'distance_situation1':  round(best_costs[i], 2),
            'temps_s_situation1':   round(instance_times[i], 2),
            'gap_%_situation1_bks': round((best_costs[i] - bks_dist) / bks_dist * 100, 2) if bks_dist else None,
        }
        rows.append(row)
    print(f"\n  {'Instance':<9} {'BKS Vhc':>8} {'BKS Dist':>10} {'Vhc S1':>7} "
          f"{'Dist. S1':>10} {'Temps S1':>10} {'Gap %':>8}")
    print("  " + "─" * 70)
    for r in rows:
        bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "—"
        bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "—"
        g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {bks_v:>8} {bks_d:>10} "
              f"{r['n_vehicules_situation1']:>7} {r['distance_situation1']:>10.2f} "
              f"{r['temps_s_situation1']:>10.1f} {g1:>8}")
    # ── Gap moyen vs BKS (Situation 1) ──────────────────────────────────
    valid_gaps = [r['gap_%_situation1_bks'] for r in rows if r['gap_%_situation1_bks'] is not None]
    gap_moyen  = float(np.mean(valid_gaps)) if valid_gaps else None
    if gap_moyen is not None:
        print(f"\n  Gap moyen vs BKS (Situation 1) : {gap_moyen:+.2f}%  "
              f"({len(valid_gaps)}/{len(rows)} instances avec BKS connu)")
        print(f"  Gap min : {min(valid_gaps):+.2f}%  |  Gap max : {max(valid_gaps):+.2f}%")
    else:
        print("\n  Gap moyen vs BKS : indisponible (aucun BKS connu pour ces instances)")
    # ── Tableau récapitulatif simple : Instance / Nb véhicules / Distance / Gap % ──
    # (une ligne "Situation 1" + une ligne "BKS" par instance, format demandé)
    print(f"\n  Récapitulatif Instance / Nb véhicules / Distance / Gap % vs BKS")
    print(f"  {'Instance':<9} {'Situation':<26} {'Nb véhicules':>13} {'Distance':>10} {'Gap %':>8}")
    print("  " + "─" * 71)
    recap_rows = []
    for r in rows:
        g1 = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {'Situation 1 (DQN+LNS+Gurobi)':<26} "
              f"{r['n_vehicules_situation1']:>13} {r['distance_situation1']:>10.2f} {g1:>8}")
        recap_rows.append({
            'instance':    r['instance'],
            'situation':   'Situation 1 (DQN+LNS+Gurobi)',
            'n_vehicules': r['n_vehicules_situation1'],
            'distance':    r['distance_situation1'],
            'gap_%_vs_bks': r['gap_%_situation1_bks'],
        })
        if r['bks_distance'] is not None:
            print(f"  {r['instance']:<9} {'BKS (référence)':<26} "
                  f"{(r['bks_vehicules'] if r['bks_vehicules'] is not None else '—'):>13} "
                  f"{r['bks_distance']:>10.2f} {'0.00':>8}")
            recap_rows.append({
                'instance':    r['instance'],
                'situation':   'BKS (référence)',
                'n_vehicules': r['bks_vehicules'],
                'distance':    r['bks_distance'],
                'gap_%_vs_bks': 0.0,
            })
    if gap_moyen is not None:
        print("  " + "─" * 71)
        print(f"  {'':<9} {'MOYENNE (Situation 1)':<26} {'':>13} {'':>10} {gap_moyen:>+8.2f}")
    if csv_path:
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
        print(f"\n  Tableau final exporté (CSV) → {csv_path}")
        recap_csv_path = csv_path.replace('.csv', '_recap_vehicules_distance.csv')
        if recap_csv_path == csv_path:
            recap_csv_path = csv_path + '.recap_vehicules_distance.csv'
        with open(recap_csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=['instance', 'situation', 'n_vehicules', 'distance', 'gap_%_vs_bks'])
            w.writeheader(); w.writerows(recap_rows)
            if gap_moyen is not None:
                w.writerow({'instance': '', 'situation': 'MOYENNE (Situation 1)',
                            'n_vehicules': '', 'distance': '', 'gap_%_vs_bks': round(gap_moyen, 2)})
        print(f"  Récapitulatif Instance/Véhicules/Distance/Gap exporté (CSV) → {recap_csv_path}")
    if latex_path:
        lines = [
            "% Tableau généré automatiquement — Situation 1 vs BKS",
            "\\begin{table}[htbp]", "  \\centering",
            "  \\caption{Comparaison par instance Solomon officielle entre la méthode "
            "hybride (LNS + DQN + Gurobi repair) et le Best Known Solution (BKS) : "
            "nombre de véhicules, distance totale et écart au BKS.}",
            "  \\label{tab:comparaison-bks}",
            "  \\begin{tabular}{lrrrrrr}", "    \\toprule",
            "    Instance & BKS Vhc & BKS Dist. & Vhc. S1 & Dist. S1 & "
            "Temps S1 (s) & Gap S1 (\\%) \\\\",
            "    \\midrule",
        ]
        for r in rows:
            bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "--"
            bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "--"
            g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "--"
            lines.append(f"    {r['instance']} & {bks_v} & {bks_d} & "
                        f"{r['n_vehicules_situation1']} & {r['distance_situation1']:.2f} & "
                        f"{r['temps_s_situation1']:.1f} & {g1} \\\\")
        lines += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
        with open(latex_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        print(f"  Tableau final exporté (LaTeX) → {latex_path}")
    return rows
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6bis  —  DIAGNOSTIC : LICENCE GUROBI À TAILLE LIMITÉE ?
# ══════════════════════════════════════════════════════════════════════════════
def check_gurobi_license(min_vars_needed=2500):
    """
    Test rapide, effectué AVANT tout calcul, pour savoir si la licence
    Gurobi active est une licence "size-limited" (Community/gratuite,
    plafonnée à ~2000 variables ET ~2000 contraintes) plutôt qu'une licence
    complète (académique ou commerciale, sans limite de taille).

    Pourquoi c'est important ici : le repair Gurobi de la Situation 1
    (_repair_gurobi, appelé à CHAQUE step du LNS) crée jusqu'à
    (K_DESTROY+1)² * V variables binaires — selon K_DESTROY (cf. CFG) et le
    nombre de véhicules de l'instance, cela peut dépasser 2000. Un fallback
    vers le repair glouton est déjà en place dans _repair_gurobi (try/
    except), donc l'exécution ne plantera pas, mais peut silencieusement
    retomber sur le glouton plus souvent que prévu si la licence est limitée.

    Ce test ne bloque JAMAIS l'exécution : il informe seulement, une seule
    fois, en tête de run.
    """
    env, m = None, None
    try:
        env = gp.Env(empty=True)
        env.setParam('OutputFlag', 0)
        env.start()
        m = gp.Model(env=env)
        m.setParam('OutputFlag', 0)
        x = m.addVars(min_vars_needed, vtype=GRB.BINARY)
        m.setObjective(gp.quicksum(x[i] for i in range(min_vars_needed)), GRB.MAXIMIZE)
        m.optimize()
        print(f"  ✓ Licence Gurobi : aucune restriction de taille détectée "
              f"(modèle-test à {min_vars_needed} variables résolu sans erreur).")
        return True
    except gp.GurobiError as e:
        errno = getattr(e, 'errno', None)
        is_size_limit = (errno == getattr(GRB.Error, 'SIZE_LIMIT_EXCEEDED', object()))
        if is_size_limit:
            k_repair_vars = (CFG['K_DESTROY'] + 1) ** 2 * CFG['K_DESTROY']
            print(f"\n  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée "
                  f"(erreur {errno} : {e})\n"
                  f"    → Licence Community/gratuite, plafonnée à ~2000 variables "
                  f"et ~2000 contraintes.\n"
                  f"    → Le repair Gurobi de la Situation 1 (jusqu'à "
                  f"~{k_repair_vars} variables selon K_DESTROY={CFG['K_DESTROY']} et "
                  f"le nombre de véhicules) peut dépasser cette limite sur certaines "
                  f"instances — un repli automatique vers le repair glouton est déjà "
                  f"en place, donc pas de plantage, mais un usage de Gurobi moins "
                  f"fréquent que prévu.\n"
                  f"    → Pour lever cette limite : une licence académique Gurobi "
                  f"est gratuite et sans limite de taille pour les universités "
                  f"(https://www.gurobi.com/academia/) — utile si vous êtes "
                  f"affiliée à ENSI-UMA.\n")
        else:
            print(f"  ⚠ Erreur Gurobi lors du test de licence (non liée à la taille) : {e}")
        return False
    except Exception as e:
        print(f"  ⚠ Impossible de tester la licence Gurobi ({type(e).__name__}: {e}) "
              f"— test ignoré, l'exécution continue normalement.")
        return None
    finally:
        if m is not None:
            m.dispose()
        if env is not None:
            env.dispose()
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 7  —  COURBES DE CONVERGENCE
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_mean(vals, window=20):
    out = []
    for i in range(len(vals)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(vals[lo:i + 1])))
    return out
def plot_global_convergence(training_log, instances, initial_costs, best_costs, outdir):
    """
    Figure globale (2 panneaux) :
      (a) reward par épisode global + moyenne mobile → convergence de
          l'apprentissage DQN
      (b) amélioration % par instance (une barre par instance Solomon) vs
          coût LNS initial
    """
    episodes = [e['episode_global'] for e in training_log]
    rewards  = [e['total_reward']   for e in training_log]
    reward_smooth = _rolling_mean(rewards, window=20)
    improvements  = [(initial_costs[i] - best_costs[i]) / initial_costs[i] * 100
                      for i in range(len(initial_costs))]
    names = [inst['name'] for inst in instances]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    axes[0].plot(episodes, rewards, color='lightgray', linewidth=0.6, label='reward brut / épisode')
    axes[0].plot(episodes, reward_smooth, color='crimson', linewidth=2,
                 label='moyenne mobile (20 épisodes)')
    axes[0].set_xlabel('Épisode global')
    axes[0].set_ylabel('Reward de l\'épisode')
    axes[0].set_title("Convergence de l'apprentissage DQN")
    axes[0].legend(loc='best', fontsize=9)
    axes[0].grid(alpha=0.3)
    colors = ['#2ca02c' if v >= 0 else '#d62728' for v in improvements]
    axes[1].bar(range(len(improvements)), improvements, color=colors)
    axes[1].axhline(np.mean(improvements), color='navy', linestyle='--', linewidth=1.5,
                     label=f"Moyenne = {np.mean(improvements):.2f}%")
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names, rotation=90, fontsize=7)
    axes[1].set_xlabel('Instance')
    axes[1].set_ylabel('Amélioration vs solution LNS initiale (%)')
    axes[1].set_title("Amélioration du coût par instance après DQN+Gurobi")
    axes[1].legend(loc='best', fontsize=9)
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_globale.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence globale → {path}")
    return path
def plot_gap_vs_bks(bks_rows, outdir):
    """
    Barres du gap % vs BKS (Situation 1) pour chaque instance, avec la
    moyenne indiquée par une ligne horizontale. N'utilise que les instances
    pour lesquelles un BKS est connu (gap_%_situation1_bks non None).
    `bks_rows` = la liste `rows` retournée par build_bks_comparison_table().
    """
    data = [(r['instance'], r['gap_%_situation1_bks']) for r in bks_rows
            if r['gap_%_situation1_bks'] is not None]
    if not data:
        print("  [plot_gap_vs_bks] Aucune instance avec BKS connu — graphique ignoré.")
        return None
    names = [d[0] for d in data]
    gaps  = [d[1] for d in data]
    gap_moyen = float(np.mean(gaps))
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(names)), 5.5))
    colors = ['#2ca02c' if g <= 0 else '#d62728' for g in gaps]
    ax.bar(range(len(names)), gaps, color=colors)
    ax.axhline(gap_moyen, color='navy', linestyle='--', linewidth=1.5,
               label=f"Gap moyen = {gap_moyen:+.2f}%")
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_xlabel('Instance')
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_title(f"Écart au Best Known Solution par instance "
                 f"({len(names)} instances avec BKS connu)")
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique gap vs BKS → {path}")
    return path
def plot_instance_convergence_examples(training_log, instances, example_idxs, outdir):
    """
    Pour chaque instance exemple :
      - gauche : coût final / meilleur coût cumulé au fil des épisodes locaux
      - droite : trajectoire de coût step par step du dernier épisode local
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(13, 4.2 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        entries = [e for e in training_log if e['instance_id'] == idx]
        entries.sort(key=lambda e: e['ep_local'])
        ep_locals      = [e['ep_local']   for e in entries]
        best_costs_ep  = [e['best_cost']  for e in entries]
        final_costs_ep = [e['final_cost'] for e in entries]
        name = instances[idx]['name']
        axes[row, 0].plot(ep_locals, final_costs_ep, '-o', color='steelblue',
                           label='coût final de l\'épisode')
        axes[row, 0].plot(ep_locals, best_costs_ep, '-s', color='crimson',
                           label='meilleur coût cumulé')
        axes[row, 0].set_title(f"Instance {name} — convergence par épisode local")
        axes[row, 0].set_xlabel('Épisode local')
        axes[row, 0].set_ylabel('Coût de la tournée')
        axes[row, 0].legend(loc='best', fontsize=9)
        axes[row, 0].grid(alpha=0.3)
        last = entries[-1] if entries else None
        trace = last.get('cost_trace', []) if last else []
        if trace:
            axes[row, 1].plot(range(len(trace)), trace, '-o', color='darkorange')
            axes[row, 1].set_title(f"Instance {name} — trajectoire de coût "
                                    f"(dernier épisode local, {len(trace)-1} steps)")
            axes[row, 1].set_xlabel('Step')
            axes[row, 1].set_ylabel('Coût courant')
            axes[row, 1].grid(alpha=0.3)
        else:
            axes[row, 1].axis('off')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_instances_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence instances exemples → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 8  —  TRACÉS DES ROUTES (AVANT LNS / APRÈS DQN)
# ══════════════════════════════════════════════════════════════════════════════
def _plot_routes_on_ax(ax, instance, solution, title):
    depot = instance['depot']
    ax.scatter([depot['x']], [depot['y']], c='black', marker='s', s=110,
               zorder=5, label='Dépôt')
    n_routes = max(len([r for r in solution if r]), 1)
    try:
        cmap = matplotlib.colormaps.get_cmap('tab20').resampled(max(n_routes, 1))
    except AttributeError:
        cmap = plt.cm.get_cmap('tab20', max(n_routes, 1))
    ridx = 0
    for route in solution:
        if not route:
            continue
        xs = [depot['x']] + [instance['customers'][cid - 1]['x'] for cid in route] + [depot['x']]
        ys = [depot['y']] + [instance['customers'][cid - 1]['y'] for cid in route] + [depot['y']]
        ax.plot(xs, ys, '-o', color=cmap(ridx), markersize=4, linewidth=1.3)
        ridx += 1
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.3)
def plot_routes_examples(instances, initial_sols, best_solutions, example_idxs, outdir):
    """
    Pour chaque instance exemple : routes AVANT (solution LNS initiale, avant
    DQN) à gauche, routes APRÈS (meilleure solution trouvée par DQN+Gurobi)
    à droite. Le BKS (véhicules/distance) est indiqué en sous-titre s'il
    est connu pour l'instance.
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.8 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        inst = instances[idx]
        name = inst['name']
        bks  = BKS_REFERENCE.get(name)
        bks_str = (f" | BKS: dist={bks['distance']:.1f}, vhc={bks['vehicles']}"
                   if bks else "")
        cost_before = compute_cost(initial_sols[idx], inst)
        cost_after  = compute_cost(best_solutions[idx], inst)
        nveh_before = sum(1 for r in initial_sols[idx] if r)
        nveh_after  = sum(1 for r in best_solutions[idx] if r)
        _plot_routes_on_ax(
            axes[row, 0], inst, initial_sols[idx],
            f"Instance {name} — AVANT (LNS)\n"
            f"dist={cost_before:.1f}  véhicules={nveh_before}{bks_str}"
        )
        _plot_routes_on_ax(
            axes[row, 1], inst, best_solutions[idx],
            f"Instance {name} — APRÈS (DQN+Gurobi)\n"
            f"dist={cost_after:.1f}  véhicules={nveh_after}{bks_str}"
        )
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_routes_avant_apres_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique routes avant/après → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  EXÉCUTION AUTOMATIQUE
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    t0 = time.time()
    print(f"\n{'═'*72}\n  MODE ENTRAÎNEMENT/TEST DISJOINT\n"
          f"  ENTRAÎNEMENT = {CFG['N_SYNTH_INSTANCES']} instances SYNTHÉTIQUES générées "
          f"| TEST = 56 instances Solomon officielles (gelé)\n{'═'*72}")

    # ── DIAGNOSTIC PRÉALABLE : licence Gurobi à taille limitée ? ────────────
    # Informatif uniquement — n'interrompt jamais l'exécution.
    _sec("DIAGNOSTIC — LICENCE GUROBI")
    check_gurobi_license()

    # ── ÉTAPE 1a : GÉNÉRATION DES INSTANCES SYNTHÉTIQUES D'ENTRAÎNEMENT ─────
    # (reprend generate_instance() de Code 1 — cf. MODULE 1bis ci-dessus).
    # Ces instances SYN-* ne recoupent jamais les noms Solomon (C101, R101,
    # ...) chargés à l'ÉTAPE 2 : aucune fuite entre train et test.
    synth_instances = generate_synthetic_training_set(CFG)

    # ── ÉTAPE 1b : ENTRAÎNEMENT DU DQN — uniquement sur le synthétique ──────
    # Valeurs par défaut : si le modèle est rechargé (branche ci-dessous), il
    # n'y a pas de training_log/coûts synthétiques dans CE run — les
    # graphiques de convergence d'entraînement seront alors simplement omis.
    training_log, synth_initial_costs, synth_best_costs = [], None, None
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    if CFG.get('REUSE_EXISTING_SYNTH_MODEL', True) and os.path.exists(model_path):
        _sec(f"ÉTAPE 1b — RÉUTILISATION D'UN MODÈLE DÉJÀ ENTRAÎNÉ SUR LE SYNTHÉTIQUE "
             f"({model_path})")
        agent = DQNAgent(
            obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
            gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
            epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
            buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
            target_update_freq=CFG['TARGET_UPDATE'],
        )
        agent.load(model_path)
    else:
        agent, training_log, synth_initial_costs, synth_best_costs, \
            synth_best_solutions, synth_instance_times, synth_initial_sols = train_on_instances(
                synth_instances, episodes_per_instance=CFG['EPISODES_PER_INSTANCE'])

    # ── ÉTAPE 2a : CHARGEMENT DES 56 INSTANCES SOLOMON (TEST — jamais vues) ─
    _sec("ÉTAPE 2 — CHARGEMENT DES INSTANCES SOLOMON (TEST)")
    test_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
    example_idxs = pick_example_indices(test_instances, preferred_names=CFG['EXAMPLE_NAMES'])

    # ── ÉTAPE 2b : ÉVALUATION GELÉE — le même agent en mémoire, sans réentraîner ──
    initial_costs, best_costs, best_solutions, instance_times, test_initial_sols, test_log = \
        evaluate_frozen_on_instances(
            test_instances, agent, episodes_per_instance=CFG['EPISODES_PER_INSTANCE_TEST'])

    # ── [A] Convergence de l'ENTRAÎNEMENT (synthétique uniquement) ──────────
    _sec("GRAPHIQUE — CONVERGENCE DE L'ENTRAÎNEMENT (SYNTHÉTIQUE)")
    if training_log:
        plot_global_convergence(training_log, synth_instances,
                                 synth_initial_costs, synth_best_costs, CFG['OUTDIR'])
    else:
        print("  (pas de courbe de convergence : modèle rechargé, aucun "
              "entraînement effectué dans ce run)")

    # ── [B] Convergence détaillée + [C] Routes avant/après — SUR LE TEST ────
    _sec("GRAPHIQUES — DÉTAIL DU TEST (INSTANCES SOLOMON EXEMPLES)")
    plot_instance_convergence_examples(test_log, test_instances, example_idxs, CFG['OUTDIR'])
    plot_routes_examples(test_instances, test_initial_sols, best_solutions, example_idxs, CFG['OUTDIR'])

    # ── [D] TABLEAU FINAL : SITUATION 1 (test gelé, entraîné sur synthétique) ─
    # vs BKS (gap % par instance + gap moyen) ──────────────────────────────
    bks_rows = build_bks_comparison_table(
        test_instances, initial_costs, best_costs, best_solutions, instance_times,
        csv_path=os.path.join(CFG['OUTDIR'], 'solomon_comparaison_bks.csv'),
        latex_path=os.path.join(CFG['OUTDIR'], 'solomon_tableau_bks.tex'),
    )
    # ── [E] Graphique dédié : gap % vs BKS par instance + gap moyen (TEST) ──
    _sec("GRAPHIQUE GAP % vs BKS (TEST)")
    plot_gap_vs_bks(bks_rows, CFG['OUTDIR'])

    print(f"\n  ✓ Terminé en {time.time()-t0:.1f}s ({(time.time()-t0)/3600:.2f}h)\n")


════════════════════════════════════════════════════════════════════════
  MODE ENTRAÎNEMENT/TEST DISJOINT
  ENTRAÎNEMENT = 80 instances SYNTHÉTIQUES générées | TEST = 56 instances Solomon officielles (gelé)
════════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════════
  DIAGNOSTIC — LICENCE GUROBI
════════════════════════════════════════════════════════════════════════

  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée (erreur 10010 : Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information)
    → Licence Community/gratuite, plafonnée à ~2000 variables et ~2000 contraintes.
    → Le repair Gurobi de la Situation 1 (jusqu'à ~15000 variables selon K_DESTROY=24 et le nombre de véhicules) peut dépasser cette limite sur certaines instances — un repli automatique vers le repair glouton est déjà en place, donc pas de plantage, mais un usage de Gurobi moins fréquent que

C:\Users\amelb\AppData\Local\Temp\ipykernel_18372\2796669767.py:851: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location='cpu')


    [DQN] Modèle rechargé ← C:\Users\amelb\PhD Thesis\outputs\model\dqn_vrptw_synth_trained.pth

════════════════════════════════════════════════════════════════════════
  ÉTAPE 2 — CHARGEMENT DES INSTANCES SOLOMON (TEST)
════════════════════════════════════════════════════════════════════════
  Chargé : C101     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C102     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C103     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C104     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C105     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C106     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C107     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C108     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C109     type=C   clients=100 véhicule

## Plan Taguchi 

In [8]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON    ║
║                          + GAP PAR RAPPORT AU BKS                           ║
║                                                                              ║
║  Version corrigée : reprend toutes les visualisations de la variante        ║
║  précédente (courbes de convergence, tracés des routes avant/après,        ║
║  graphique dédié gap % vs BKS, tableau récapitulatif véhicules/distance)   ║
║  MAIS applique désormais un vrai protocole ENTRAÎNEMENT / TEST DISJOINT,   ║
║  en un seul lancement, sans aucune intervention manuelle :                 ║
║                                                                              ║
║   ÉTAPE 1 (ENTRAÎNEMENT) — génère des instances SYNTHÉTIQUES (reprises de   ║
║      generate_instance() de main_vrptw_comparaison_gurobi.py / "Code 1")    ║
║      et entraîne le DQN dessus (poids mis à jour via store/learn/decay).    ║
║                                                                              ║
║   ÉTAPE 2 (TEST) — charge les 56 instances Solomon OFFICIELLES et évalue    ║
║      le même agent, GELÉ (aucun store/learn/decay_epsilon, uniquement      ║
║      select_action en pur mode inférence), sur ce jeu jamais vu à          ║
║      l'entraînement. Toutes les courbes/tracés/tableaux ci-dessous portent  ║
║      désormais sur ce TEST gelé, pas sur les résultats d'entraînement.     ║
║                                                                              ║
║  Situation 1 = LNS + DQN(entraîné sur synthétique) + Gurobi, ÉVALUÉ sur     ║
║                Solomon (test)                                → gap vs BKS   ║
║  (La "Situation 2 = Gurobi seul sur l'instance complète" reste retirée,     ║
║   à la demande, car sans limite de temps elle est beaucoup trop longue      ║
║   sur des instances à 100 clients avec fenêtres de temps serrées.)          ║
║                                                                              ║
║  Visualisations produites (toutes sur le TEST gelé, sauf la 1ère) :        ║
║    [A] Convergence de l'ENTRAÎNEMENT (synthétique) : reward global +       ║
║        amélioration % par instance synthétique                            ║
║    [B] Convergence détaillée (instances Solomon exemples, TEST) : coût     ║
║        final/meilleur par épisode local + trajectoire de coût             ║
║    [C] Tracés des routes AVANT (LNS) / APRÈS (DQN gelé) pour des           ║
║        instances Solomon exemples                                         ║
║    [D] Tableau récapitulatif : Instance / Nb véhicules / Distance / Gap %  ║
║        (TEST Solomon vs BKS)                                              ║
║    [E] Graphique dédié gap % vs BKS par instance + gap moyen              ║
║    [F] Diagnostic automatique de licence Gurobi (taille limitée ou non)   ║
║                                                                              ║
║  Aucun rappel manuel de "Code 1" n'est nécessaire : la génération +        ║
║  l'entraînement synthétique sont réintégrés ici (MODULE 1bis) et           ║
║  s'exécutent automatiquement avant le chargement de Solomon. Voir le flag  ║
║  optionnel CFG['REUSE_EXISTING_SYNTH_MODEL'] plus bas si vous préférez     ║
║  recharger un .pth déjà entraîné au lieu de ré-entraîner à chaque run.     ║
║                                                                              ║
║  Par défaut, INSTANCE_FILES contient toutes les instances *.txt trouvées   ║
║  dans outputs/Solomon/. Pour se limiter à 6 instances représentatives      ║
║  (une par groupe C1/C2/R1/R2/RC1/RC2), décommentez le bloc plus bas.       ║
║                                                                              ║
║  Exécution :  python main_vrptw_synth_train_solomon_convergence.py         ║
║                                                                              ║
║  CORRECTIONS APPORTÉES SUR CETTE VERSION :                                 ║
║   1. SOLOMON_DIR pointait vers un chemin Google Colab                      ║
║      ('/content/sample_data/Solomon') — corrigé pour pointer vers          ║
║      outputs/Solomon (dossier local Windows).                              ║
║   2. Le mode CFG['TAGUCHI_SYNTH']['ENABLE']=True s'arrêtait après la       ║
║      calibration et exigeait de relancer le script à la main pour la      ║
║      validation finale. Il enchaîne désormais AUTOMATIQUEMENT, dans le    ║
║      même lancement, sur l'entraînement complet (80 instances               ║
║      synthétiques) + le test gelé sur les 56 Solomon, en appliquant       ║
║      automatiquement la meilleure combinaison d'hyperparamètres trouvée.  ║
║      Voir run_final_validation() et le bloc __main__ tout en bas.         ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import os, sys, json, time, copy, random, csv, glob, itertools
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from contextlib import contextmanager
import torch
import torch.nn as nn
import torch.optim as optim
import gurobipy as gp
from gurobipy import GRB
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG GLOBALE
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    # ── ÉTAPE 1 : ENTRAÎNEMENT sur instances SYNTHÉTIQUES (jamais Solomon) ──
    N_SYNTH_INSTANCES          = 80,     # comme les 80 instances de Code 1
    SYNTH_N_CUSTOMERS          = 100,
    SYNTH_N_VEHICLES           = None,   # None → calculé depuis la demande (cf. generate_instance)
    SYNTH_VEHICLE_SAFETY_FACTOR= 1.5,
    SYNTH_CAPACITY             = 200,
    SYNTH_HORIZON              = 1000,
    EPISODES_PER_INSTANCE      = 10,     # épisodes d'ENTRAÎNEMENT par instance synthétique
    # Optionnel : True → recharge un .pth déjà entraîné sur du synthétique
    # (voir MODEL_FILE) au lieu de régénérer + ré-entraîner. False (par
    # défaut) → tout se fait automatiquement dans CE run, sans intervention
    # manuelle, aucun besoin de relancer "Code 1" séparément.
    REUSE_EXISTING_SYNTH_MODEL = False,
    # ── ÉTAPE 2 : TEST gelé sur les 56 instances Solomon officielles ───────
    EPISODES_PER_INSTANCE_TEST = 3,      # épisodes d'ÉVALUATION (aucun apprentissage) par instance Solomon
    # VEHICLE_SAFETY_FACTOR = None : on utilise la flotte OFFICIELLE de chaque
    # instance, pas une borne calculée sur la demande. Sur R101/RC101
    # (fenêtres de temps serrées), le BKS utilise 19 et 14 véhicules alors
    # que la demande seule suggérerait ~12-14 : une borne basée uniquement
    # sur la demande sous-estime le besoin réel et peut rendre le MILP
    # infaisable. La flotte officielle est plus grande (MILP plus lourd)
    # mais garantit qu'une solution proche du BKS reste atteignable.
    VEHICLE_SAFETY_FACTOR  = None,
    LNS_ITER               = 30,
    LNS_K_RATIO            = 0.20,
    K_DESTROY_RATIO        = 0.24,
    MAX_STEPS               = 20,
    # repair par bouts (Situation 1, appelé à CHAQUE step du LNS/DQN — voir
    # check_gurobi_license() plus bas pour l'impact d'une licence à taille
    # limitée). None = aucune limite de temps (supporté par le code).
    GUROBI_TLIMIT            = 2.0,
    GUROBI_GAP               = 0.05,
    HIDDEN                   = 64,
    LR                       = 1e-3,
    GAMMA                    = 0.95,
    EPS_START                = 1.0,
    EPS_END                  = 0.05,
    EPS_DECAY                = 0.95,  # ⚠ ignoré — recalculé automatiquement plus bas (cf. commentaire)
    BUFFER_SIZE               = 5000,
    BATCH_SIZE                = 32,
    TARGET_UPDATE              = 10,
    # ── Instances exemples (pour courbes détaillées + routes, TEST/Solomon) ─
    # Noms recherchés en priorité (représentatifs des 6 groupes Solomon) ;
    # si un nom n'est pas présent dans INSTANCE_FILES, il est ignoré et
    # remplacé automatiquement par le 1er type disponible.
    EXAMPLE_NAMES              = ['C101', 'R101', 'RC101'],
    # ── Pénalité par véhicule (repair MILP) ────────────────────────
    # 0.0 = comportement d'origine (aucune pénalité, Gurobi peut ouvrir
    # librement des véhicules supplémentaires à chaque repair). Une valeur
    # > 0 ajoute un coût fixe par véhicule utilisé dans l'objectif du MILP
    # de repair, pour limiter l'inflation du nombre de véhicules observée
    # dans la validation vs BKS. C'est aussi l'un des facteurs calibrés par
    # le plan d'expérience Taguchi (MODULE 10 plus bas) — calibré
    # UNIQUEMENT sur du synthétique (CFG['TAGUCHI_SYNTH']), jamais sur
    # Solomon, pour ne pas faire fuiter d'information du test set vers le
    # choix des hyperparamètres.
    VEHICLE_PENALTY            = 0.0,
    OUTDIR                        = os.path.join(os.getcwd(), 'outputs'),
    MODELDIR                       = os.path.join(os.getcwd(), 'outputs', 'model'),
    MODEL_FILE                      = 'dqn_vrptw_synth_trained.pth',
)
CFG['LNS_K'] = 20   # ~20% de 100 clients, cohérent avec LNS_K_RATIO
CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))  # 24, cohérent avec K_DESTROY_RATIO
# ── Recalibrage automatique de EPS_DECAY ────────────────────────────────
# La valeur fixe 0.95 était calibrée pour un tout petit nombre d'épisodes
# (contexte d'origine : entraînement direct sur quelques dizaines
# d'instances Solomon). Avec EPS_DECAY=0.95, epsilon atteint son plancher
# (EPS_END) après seulement ~58 décroissances — soit, avec les valeurs par
# défaut (N_SYNTH_INSTANCES=80, EPISODES_PER_INSTANCE=10), après à peine
# ~6 des 80 instances synthétiques : plus de 90% de l'entraînement se
# ferait alors avec epsilon au plancher (quasi aucune exploration), quel
# que soit EPISODES_PER_INSTANCE — et AUGMENTER EPISODES_PER_INSTANCE
# aggrave le problème (le plancher est atteint en encore moins d'instances
# couvertes). On recalcule donc EPS_DECAY ici pour que epsilon décroisse
# de EPS_START à EPS_END sur TOUTE la durée prévue de l'entraînement
# (N_SYNTH_INSTANCES × EPISODES_PER_INSTANCE décroissances), quel que soit
# le nombre d'épisodes par instance choisi.
_n_decays_total = max(1, CFG['N_SYNTH_INSTANCES'] * CFG['EPISODES_PER_INSTANCE'])
CFG['EPS_DECAY'] = (CFG['EPS_END'] / CFG['EPS_START']) ** (1.0 / _n_decays_total)
os.makedirs(CFG['OUTDIR'],   exist_ok=True)
os.makedirs(CFG['MODELDIR'], exist_ok=True)
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG DU PLAN D'EXPÉRIENCE TAGUCHI (calibration des hyperparamètres)
# ══════════════════════════════════════════════════════════════════════════════
# Deux modes de calibration disponibles (voir plus bas pour le mode
# TAGUCHI_SYNTH, recommandé, qui enchaîne désormais automatiquement sur la
# validation finale). Activez explicitement l'UN DES DEUX (pas les deux en
# même temps) :
#
#  • CFG['TAGUCHI_SYNTH']['ENABLE'] = True   (RECOMMANDÉ)
#    Calibre en s'entraînant UNIQUEMENT sur des instances SYNTHÉTIQUES
#    générées à la volée (jamais Solomon). Métrique de classement :
#    amélioration % vs coût LNS initial (pas de BKS requis). La meilleure
#    combinaison trouvée n'a jamais "vu" les 56 instances Solomon → aucune
#    fuite d'hyperparamètres vers le jeu de test, contrairement au mode
#    historique ci-dessous. Une fois la calibration terminée, le script
#    enchaîne AUTOMATIQUEMENT sur la validation finale (entraînement 80
#    instances + test gelé sur les 56 Solomon) avec ces hyperparamètres.
#
#  • CFG['TAGUCHI']['ENABLE'] = True   (⚠ HISTORIQUE — NON RECOMMANDÉ)
#    Calibre en s'entraînant DIRECTEMENT sur les 56 instances Solomon (la
#    matrice externe = le jeu de test lui-même) et classe les combinaisons
#    par leur gap % vs BKS sur ces mêmes instances. C'est exactement le
#    protocole "Run 16" identifié comme fuite d'hyperparamètres (on choisit
#    les hyperparamètres en fonction de la performance sur le test) —
#    conservé uniquement pour comparaison / compatibilité avec d'anciens
#    résultats, PAS pour produire les hyperparamètres d'une validation
#    finale train/test disjointe. Ce mode n'enchaîne PAS automatiquement
#    sur la validation finale (voir le bloc __main__ tout en bas).
CFG['TAGUCHI'] = dict(
    ENABLE                     = False,  # ⚠ historique — fuite vers Solomon (test set), voir ci-dessus
    EPISODES_PER_INSTANCE_DOE  = 5,
    OUTER_INSTANCE_NAMES       = 'ALL',  # 'ALL' = les 56 instances Solomon comme matrice externe
)
CFG['TAGUCHI_SYNTH'] = dict(
    ENABLE                     = True,  # True = lance le plan Taguchi calibré sur synthétique (recommandé)
    # Volume d'instances synthétiques utilisé comme matrice externe pendant
    # CE plan de calibration (indépendant de N_SYNTH_INSTANCES du protocole
    # d'entraînement final — un sous-ensemble suffit ici, l'objectif est de
    # classer les 18 combinaisons, pas d'entraîner le modèle définitif).
    N_INSTANCES_DOE            = 24,   # 8 par type C/R/RC
    EPISODES_PER_INSTANCE_DOE  = 5,
)
# Facteurs de contrôle et leurs niveaux (plan Taguchi L18 : 7 facteurs à 3
# niveaux, 18 combinaisons au lieu des 3^7 = 2187 combinaisons possibles).
TAGUCHI_FACTORS = [
    ('LNS_ITER',         [30, 60, 100]),
    ('K_DESTROY_RATIO',  [0.15, 0.20, 0.24]),
    ('GUROBI_TLIMIT',    [1.0, 2.0, 3.0]),
    ('GUROBI_GAP',       [0.05, 0.03, 0.01]),
    ('MAX_STEPS',        [10, 15, 20]),
    ('EPS_DECAY',        [0.90, 0.95, 0.99]),
    ('VEHICLE_PENALTY',  [0.0, 300.0, 800.0]),
]
GROUP_ORDER = ['C1', 'C2', 'R1', 'R2', 'RC1', 'RC2']
ACTION_NAMES = {
    0: "Destruction spatiale",
    1: "Destruction temporelle",
    2: "Destruction par coût élevé",
    3: "Destruction aléatoire",
}
# ── Dossier contenant les instances Solomon officielles (extraites du zip) ──
# CORRIGÉ : pointait vers '/content/sample_data/Solomon' (chemin Google
# Colab, inexistant sur une machine locale) — repointé vers outputs/Solomon.
SOLOMON_DIR = os.path.join(CFG['OUTDIR'], 'Solomon')
# Toutes les instances *.txt trouvées (décommentez le bloc ci-dessous pour
# revenir à seulement 6 instances représentatives, une par groupe
# C1/C2/R1/R2/RC1/RC2, afin de garder un temps d'exécution raisonnable).
INSTANCE_FILES = sorted(glob.glob(os.path.join(SOLOMON_DIR, '*.txt')))
# INSTANCE_FILES = [
#     os.path.join(SOLOMON_DIR, 'c101.txt'),
#     os.path.join(SOLOMON_DIR, 'c201.txt'),
#     os.path.join(SOLOMON_DIR, 'r101.txt'),
#     os.path.join(SOLOMON_DIR, 'r201.txt'),
#     os.path.join(SOLOMON_DIR, 'rc101.txt'),
#     os.path.join(SOLOMON_DIR, 'rc201.txt'),
# ]
# ── BKS officiel complet, 56 instances (source : SINTEF / Solomon 1987) ──────
BKS_REFERENCE = {
    'C101': {'vehicles': 10, 'distance': 828.94}, 'C102': {'vehicles': 10, 'distance': 828.94},
    'C103': {'vehicles': 10, 'distance': 828.06}, 'C104': {'vehicles': 10, 'distance': 824.78},
    'C105': {'vehicles': 10, 'distance': 828.94}, 'C106': {'vehicles': 10, 'distance': 828.94},
    'C107': {'vehicles': 10, 'distance': 828.94}, 'C108': {'vehicles': 10, 'distance': 828.94},
    'C109': {'vehicles': 10, 'distance': 828.94},
    'C201': {'vehicles': 3, 'distance': 591.56}, 'C202': {'vehicles': 3, 'distance': 591.56},
    'C203': {'vehicles': 3, 'distance': 591.17}, 'C204': {'vehicles': 3, 'distance': 590.60},
    'C205': {'vehicles': 3, 'distance': 588.88}, 'C206': {'vehicles': 3, 'distance': 588.49},
    'C207': {'vehicles': 3, 'distance': 588.29}, 'C208': {'vehicles': 3, 'distance': 588.32},
    'R101': {'vehicles': 19, 'distance': 1650.80}, 'R102': {'vehicles': 17, 'distance': 1486.12},
    'R103': {'vehicles': 13, 'distance': 1292.68}, 'R104': {'vehicles': 9,  'distance': 1007.31},
    'R105': {'vehicles': 14, 'distance': 1377.11}, 'R106': {'vehicles': 12, 'distance': 1252.03},
    'R107': {'vehicles': 10, 'distance': 1104.66}, 'R108': {'vehicles': 9,  'distance': 960.88},
    'R109': {'vehicles': 11, 'distance': 1194.73}, 'R110': {'vehicles': 10, 'distance': 1118.84},
    'R111': {'vehicles': 10, 'distance': 1096.72}, 'R112': {'vehicles': 9,  'distance': 982.14},
    'R201': {'vehicles': 4, 'distance': 1252.37}, 'R202': {'vehicles': 3, 'distance': 1191.70},
    'R203': {'vehicles': 3, 'distance': 939.50},  'R204': {'vehicles': 2, 'distance': 825.52},
    'R205': {'vehicles': 3, 'distance': 994.43},  'R206': {'vehicles': 3, 'distance': 906.14},
    'R207': {'vehicles': 2, 'distance': 890.61},  'R208': {'vehicles': 2, 'distance': 726.82},
    'R209': {'vehicles': 3, 'distance': 909.16},  'R210': {'vehicles': 3, 'distance': 939.37},
    'R211': {'vehicles': 2, 'distance': 885.71},
    'RC101': {'vehicles': 14, 'distance': 1696.95}, 'RC102': {'vehicles': 12, 'distance': 1554.75},
    'RC103': {'vehicles': 11, 'distance': 1261.67}, 'RC104': {'vehicles': 10, 'distance': 1135.48},
    'RC105': {'vehicles': 13, 'distance': 1629.44}, 'RC106': {'vehicles': 11, 'distance': 1424.73},
    'RC107': {'vehicles': 11, 'distance': 1230.48}, 'RC108': {'vehicles': 10, 'distance': 1139.82},
    'RC201': {'vehicles': 4, 'distance': 1406.94}, 'RC202': {'vehicles': 3, 'distance': 1365.65},
    'RC203': {'vehicles': 3, 'distance': 1049.62}, 'RC204': {'vehicles': 3, 'distance': 798.46},
    'RC205': {'vehicles': 4, 'distance': 1297.65}, 'RC206': {'vehicles': 3, 'distance': 1146.32},
    'RC207': {'vehicles': 3, 'distance': 1061.14}, 'RC208': {'vehicles': 3, 'distance': 828.14},
}
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1  —  LECTURE DES INSTANCES SOLOMON OFFICIELLES  (jeu de TEST)
# ══════════════════════════════════════════════════════════════════════════════
def euclidean(a, b):
    return float(np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2))
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1bis  —  GÉNÉRATEUR D'INSTANCES SYNTHÉTIQUES  (jeu d'ENTRAÎNEMENT)
#  Repris tel quel de main_vrptw_comparaison_gurobi.py ("Code 1") : mêmes
#  fonctions _coords_C/_coords_R/_coords_RC et generate_instance(), pour que
#  les instances d'entraînement soient générées exactement comme dans Code 1.
# ══════════════════════════════════════════════════════════════════════════════
def _coords_C(n, rng):
    n_cl = max(3, n // 10)
    centers = rng.uniform(15, 85, (n_cl, 2))
    pts = []
    for i in range(n):
        p = rng.normal(centers[i % n_cl], 8)
        pts.append(np.clip(p, 1, 99))
    return np.array(pts)
def _coords_R(n, rng):
    return rng.uniform(1, 99, (n, 2))
def _coords_RC(n, rng):
    h = n // 2
    return np.vstack([_coords_C(h, rng), _coords_R(n - h, rng)])
def generate_instance(n_customers=50, n_vehicles=None, capacity=200,
                      type_='C', seed=42, horizon=1000,
                      vehicle_safety_factor=1.5):
    rng     = np.random.RandomState(seed)
    depot   = np.array([50.0, 50.0])
    gen_fn  = {'C': _coords_C, 'R': _coords_R, 'RC': _coords_RC}[type_]
    coords  = gen_fn(n_customers, rng)
    max_dem = max(1, capacity // 5)
    demands = rng.randint(1, max_dem + 1, n_customers)
    svc     = rng.randint(10, 31, n_customers)
    if n_vehicles is None:
        min_vehicles = int(np.ceil(demands.sum() / capacity))
        n_vehicles   = max(2, int(np.ceil(min_vehicles * vehicle_safety_factor)))
    customers = []
    for i in range(n_customers):
        d        = euclidean(depot, coords[i])
        earliest = max(0.0, round(d + rng.uniform(-15, 15), 1))
        delta    = rng.uniform(30, 80)
        latest   = min(float(horizon), round(earliest + delta, 1))
        latest   = max(latest, earliest + svc[i] + 5)
        latest   = min(latest, float(horizon))
        if d > latest:
            earliest = round(d, 1)
            latest   = min(float(horizon), round(earliest + delta, 1))
            latest   = max(latest, earliest + svc[i] + 5)
        if d + svc[i] + d > horizon:
            svc[i] = max(5, int(horizon - 2 * d - 5))
        customers.append({
            'id':           i + 1,
            'x':            float(round(coords[i][0], 2)),
            'y':            float(round(coords[i][1], 2)),
            'demand':       int(demands[i]),
            'earliest':     float(earliest),
            'latest':       float(latest),
            'service_time': int(svc[i]),
        })
    return {
        'depot': {
            'id': 0, 'x': 50.0, 'y': 50.0, 'demand': 0,
            'earliest': 0.0, 'latest': float(horizon), 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    capacity,
        'horizon':     horizon,
        'type':        type_,
        'seed':        seed,
        'n_customers': n_customers,
    }
def generate_synthetic_training_set(cfg, verbose=True):
    """
    Génère les instances synthétiques d'ENTRAÎNEMENT en cyclant les 3 types
    Solomon C / R / RC (même logique que train() dans Code 1 : type =
    ['C','R','RC'][i % 3], seed = i*7+1). Ces instances ne recouvrent JAMAIS
    les 56 instances Solomon officielles utilisées plus bas comme jeu de
    TEST : le DQN ne les voit donc jamais avant l'évaluation gelée.
    Chaque instance reçoit en plus une clé 'name' (ex. 'SYN-C001') pour
    rester compatible avec les fonctions d'entraînement/affichage qui
    utilisent inst['name'].
    """
    types = ['C', 'R', 'RC']
    instances = []
    if verbose:
        _sec(f"ÉTAPE 1 — GÉNÉRATION DE {cfg['N_SYNTH_INSTANCES']} INSTANCES "
             f"SYNTHÉTIQUES (ENTRAÎNEMENT)")
    for i in range(cfg['N_SYNTH_INSTANCES']):
        t    = types[i % 3]
        seed = i * 7 + 1
        inst = generate_instance(
            n_customers = cfg['SYNTH_N_CUSTOMERS'],
            n_vehicles  = cfg['SYNTH_N_VEHICLES'],
            capacity    = cfg['SYNTH_CAPACITY'],
            horizon     = cfg['SYNTH_HORIZON'],
            type_       = t,
            seed        = seed,
            vehicle_safety_factor = cfg['SYNTH_VEHICLE_SAFETY_FACTOR'],
        )
        inst['name'] = f"SYN-{t}{i+1:03d}"
        instances.append(inst)
        if verbose:
            print(f"  Instance {i+1:03d}/{cfg['N_SYNTH_INSTANCES']} | {inst['name']:<10} "
                  f"type={t:<3} | seed={seed} | véhicules={inst['n_vehicles']}")
    return instances
def parse_solomon_instance(filepath, vehicle_safety_factor=None):
    """
    Parse un fichier d'instance Solomon au format texte officiel :
        <NOM>
        VEHICLE
        NUMBER  CAPACITY
        <n>     <cap>
        CUSTOMER
        CUST NO.  XCOORD.  YCOORD.  DEMAND  READY TIME  DUE DATE  SERVICE TIME
        0  ...   (dépôt)
        1  ...
        ...
    Retourne un dict compatible avec le reste du pipeline.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = [l.rstrip('\n') for l in f]
    name = lines[0].strip()
    idx_veh = next(i for i, l in enumerate(lines) if l.strip() == 'VEHICLE')
    veh_number, veh_capacity = map(int, lines[idx_veh + 2].split())
    idx_cust = next(i for i, l in enumerate(lines) if l.strip() == 'CUSTOMER')
    data_lines = [l for l in lines[idx_cust + 2:] if l.strip()]
    rows = [list(map(float, l.split())) for l in data_lines]
    depot_row = rows[0]
    customers = []
    for r in rows[1:]:
        cid, x, y, dem, ready, due, svc = r
        customers.append({
            'id':           int(cid),
            'x':            x,
            'y':            y,
            'demand':       int(dem),
            'earliest':     ready,
            'latest':       due,
            'service_time': int(svc),
        })
    name_upper = name.upper()
    type_ = ''.join(ch for ch in name_upper if ch.isalpha())   # 'C101' -> 'C', 'RC101' -> 'RC'
    n_vehicles = veh_number
    if vehicle_safety_factor is not None:
        total_demand = sum(c['demand'] for c in customers)
        min_veh      = int(np.ceil(total_demand / veh_capacity))
        n_vehicles   = min(veh_number, max(2, int(np.ceil(min_veh * vehicle_safety_factor))))
    return {
        'depot': {
            'id': 0, 'x': depot_row[1], 'y': depot_row[2], 'demand': 0,
            'earliest': depot_row[4], 'latest': depot_row[5], 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    veh_capacity,
        'horizon':     depot_row[5],
        'type':        type_,
        'seed':        name_upper,
        'n_customers': len(customers),
        'name':        name_upper,          # nom officiel normalisé (ex. 'C101')
    }
def load_all_instances(files, vehicle_safety_factor=None):
    if not files:
        raise FileNotFoundError(
            f"Aucun fichier d'instance trouvé dans {SOLOMON_DIR}. "
            f"Vérifiez que le dossier existe et contient les .txt Solomon."
        )
    instances = []
    for fp in files:
        inst = parse_solomon_instance(fp, vehicle_safety_factor=vehicle_safety_factor)
        instances.append(inst)
        print(f"  Chargé : {inst['name']:<8} type={inst['type']:<3} "
              f"clients={inst['n_customers']} véhicules={inst['n_vehicles']} "
              f"capacité={inst['capacity']} horizon={inst['horizon']}")
    return instances
def pick_example_indices(instances, preferred_names=None, max_examples=3):
    """
    Sélectionne des instances "exemples" pour les courbes détaillées et les
    tracés de routes. Cherche d'abord les noms préférés (ex. C101/R101/RC101).
    Si absents, retombe sur la 1ère instance de chaque type détecté, puis sur
    les toutes premières instances de la liste.
    """
    preferred_names = preferred_names or []
    by_name = {inst['name']: i for i, inst in enumerate(instances)}
    idxs = [by_name[nm] for nm in preferred_names if nm in by_name]
    if not idxs:
        seen_types = {}
        for i, inst in enumerate(instances):
            if inst['type'] not in seen_types:
                seen_types[inst['type']] = i
            if len(seen_types) >= max_examples:
                break
        idxs = list(seen_types.values())
    if not idxs:
        idxs = list(range(min(max_examples, len(instances))))
    return idxs[:max_examples]
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 2  —  SOLUTION INITIALE PAR LNS  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
def _node_xy(nid, instance):
    if nid == 0:
        return instance['depot']['x'], instance['depot']['y']
    c = instance['customers'][nid - 1]
    return c['x'], c['y']
def _node_info(nid, instance):
    if nid == 0:
        return instance['depot']
    return instance['customers'][nid - 1]
def compute_cost(solution, instance):
    depot = (instance['depot']['x'], instance['depot']['y'])
    total = 0.0
    for route in solution:
        if not route:
            continue
        prev = depot
        for cid in route:
            cur   = _node_xy(cid, instance)
            total += euclidean(prev, cur)
            prev  = cur
        total += euclidean(prev, depot)
    return total
def _route_feasible(route, instance):
    if not route:
        return True
    cap  = instance['capacity']
    load = sum(_node_info(c, instance)['demand'] for c in route)
    if load > cap:
        return False
    depot = instance['depot']
    t     = 0.0
    px, py = depot['x'], depot['y']
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        if t > info['latest']:
            return False
        t      = max(t, info['earliest']) + info['service_time']
        px, py = cx, cy
    t += euclidean((px, py), (depot['x'], depot['y']))
    return t <= depot['latest']
def is_feasible(solution, instance):
    return all(_route_feasible(r, instance) for r in solution)
def solution_stats(solution, instance):
    cap        = instance['capacity']
    total_cost = compute_cost(solution, instance)
    n_vehicles = sum(1 for r in solution if r)
    loads      = []
    total_wait = 0.0
    for route in solution:
        if not route:
            continue
        load = sum(_node_info(c, instance)['demand'] for c in route)
        loads.append(load / cap)
        depot  = instance['depot']
        t      = 0.0
        px, py = depot['x'], depot['y']
        for cid in route:
            cx, cy = _node_xy(cid, instance)
            t     += euclidean((px, py), (cx, cy))
            info   = _node_info(cid, instance)
            total_wait += max(0.0, info['earliest'] - t)
            t      = max(t, info['earliest']) + info['service_time']
            px, py = cx, cy
    return {
        'total_cost':         round(total_cost, 2),
        'n_vehicles':         n_vehicles,
        'avg_load_ratio':     round(np.mean(loads) if loads else 0.0, 3),
        'total_wait_time':    round(total_wait, 2),
        'n_customers_served': sum(len(r) for r in solution),
    }
def _insertion_cost(cid, pos, route, instance):
    depot   = (instance['depot']['x'], instance['depot']['y'])
    nodes   = [0] + route + [0]
    prev_id = nodes[pos]
    next_id = nodes[pos + 1]
    px, py  = _node_xy(prev_id, instance) if prev_id != 0 else depot
    nx, ny  = _node_xy(next_id, instance) if next_id != 0 else depot
    cx, cy  = _node_xy(cid, instance)
    return (euclidean((px, py), (cx, cy)) + euclidean((cx, cy), (nx, ny))
            - euclidean((px, py), (nx, ny)))
def _try_insert(cid, route, instance):
    best_cost, best_pos = float('inf'), -1
    for pos in range(len(route) + 1):
        candidate = route[:pos] + [cid] + route[pos:]
        if _route_feasible(candidate, instance):
            cost = _insertion_cost(cid, pos, route, instance)
            if cost < best_cost:
                best_cost, best_pos = cost, pos
    return best_cost, best_pos
def _repair_greedy(removed, partial, instance):
    routes = [list(r) for r in partial]
    for cid in removed:
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _build_initial(instance):
    customers = sorted(instance['customers'], key=lambda c: c['earliest'])
    routes    = []
    for c in customers:
        cid = c['id']
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _lns_improve(solution, instance, n_iter=50, k=10, seed=0):
    rng       = np.random.RandomState(seed)
    current   = [list(r) for r in solution]
    best      = copy.deepcopy(current)
    best_cost = compute_cost(best, instance)
    for _ in range(n_iter):
        all_c   = [c for r in current for c in r]
        k_      = min(k, max(1, len(all_c) - 1))
        removed = list(rng.choice(all_c, k_, replace=False))
        removed_set = set(removed)
        partial = [[c for c in r if c not in removed_set] for r in current]
        partial = [r for r in partial if r]
        candidate = _repair_greedy(removed, partial, instance)
        c_cost    = compute_cost(candidate, instance)
        if c_cost < best_cost and is_feasible(candidate, instance):
            best, best_cost = copy.deepcopy(candidate), c_cost
        current = candidate
    return best
def get_initial_solution(instance, lns_iter=50, lns_k=10):
    sol = _build_initial(instance)
    sol = _lns_improve(sol, instance, n_iter=lns_iter, k=lns_k)
    sol = [r for r in sol if r]
    return sol, solution_stats(sol, instance)
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 3  —  ENVIRONNEMENT VRPTW  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class VRPTWEnv:
    def __init__(self, instance, initial_solution,
                 K=15, max_steps=20,
                 gurobi_tlimit=2.0, gurobi_gap=0.05,
                 vehicle_penalty=0.0,
                 verbose=False):
        self.instance    = instance
        self.initial_sol = copy.deepcopy(initial_solution)
        self.K           = K
        self.max_steps   = max_steps
        self.tlimit      = gurobi_tlimit
        self.gap         = gurobi_gap
        self.vehicle_penalty = vehicle_penalty
        self.verbose     = verbose
        self.obs_dim     = 7
        self.n_actions   = 4
        self._dist = {}
        depot = (instance['depot']['x'], instance['depot']['y'])
        for c in instance['customers']:
            d = euclidean(depot, (c['x'], c['y']))
            self._dist[(0, c['id'])] = d
            self._dist[(c['id'], 0)] = d
        for ci in instance['customers']:
            for cj in instance['customers']:
                self._dist[(ci['id'], cj['id'])] = euclidean(
                    (ci['x'], ci['y']), (cj['x'], cj['y']))
        self.reset()
    def _d(self, a, b):
        return self._dist.get((a, b), 0.0)
    def _info(self, nid):
        return _node_info(nid, self.instance)
    def reset(self, start_solution=None):
        if start_solution is not None:
            self.solution = copy.deepcopy(start_solution)
        else:
            self.solution = copy.deepcopy(self.initial_sol)
        self.current_cost     = compute_cost(self.solution, self.instance)
        self.best_cost        = self.current_cost
        self.initial_cost     = compute_cost(self.initial_sol, self.instance)
        self.step_count       = 0
        self.steps_no_improve = 0
        self._action_history  = []
        return self._obs()
    def _obs(self):
        stats = solution_stats(self.solution, self.instance)
        ic    = max(self.initial_cost, 1.0)
        return np.array([
            self.current_cost / ic,
            stats['n_vehicles'] / max(self.instance['n_vehicles'], 1),
            stats['avg_load_ratio'],
            stats['total_wait_time'] / max(ic * 0.5, 1.0),
            self.steps_no_improve / self.max_steps,
            self.best_cost / ic,
            1.0,
        ], dtype=np.float32)
    def _destroy(self, action):
        all_c = [c for r in self.solution for c in r]
        if not all_c:
            return [], copy.deepcopy(self.solution)
        K   = min(self.K, len(all_c))
        rng = np.random.RandomState(self.step_count + action * 100)
        if action == 0:
            pivot   = int(rng.choice(all_c))
            px, py  = self._info(pivot)['x'], self._info(pivot)['y']
            dists   = {c: euclidean((px, py),
                                    (self._info(c)['x'], self._info(c)['y']))
                       for c in all_c}
            destroyed = sorted(all_c, key=lambda c: dists[c])[:K]
        elif action == 1:
            pivot    = int(rng.choice(all_c))
            e_pivot  = self._info(pivot)['earliest']
            tdiffs   = {c: abs(self._info(c)['earliest'] - e_pivot) for c in all_c}
            destroyed = sorted(all_c, key=lambda c: tdiffs[c])[:K]
        elif action == 2:
            savings = {}
            for r in self.solution:
                if not r:
                    continue
                nodes = [0] + r + [0]
                for i, cid in enumerate(r):
                    prev, nxt     = nodes[i], nodes[i + 2]
                    savings[cid]  = (self._d(prev, cid) + self._d(cid, nxt)
                                     - self._d(prev, nxt))
            destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
        else:
            idx       = rng.choice(len(all_c), K, replace=False)
            destroyed = [all_c[i] for i in idx]
        destroyed_set = set(destroyed)
        partial = [[c for c in r if c not in destroyed_set]
                   for r in self.solution]
        partial = [r for r in partial if r]
        return list(destroyed), partial
    def _repair_gurobi(self, destroyed, partial):
        if not destroyed:
            return copy.deepcopy(self.solution)
        inst    = self.instance
        depot   = inst['depot']
        horizon = inst['horizon']
        cap     = inst['capacity']
        nodes   = [0] + destroyed
        N       = len(nodes)
        V       = min(len(destroyed), inst['n_vehicles'])
        env, m = None, None
        try:
            env = gp.Env(empty=True)
            env.setParam('OutputFlag', 0)
            env.start()
            m = gp.Model(env=env)
            if self.tlimit is not None:
                m.setParam('TimeLimit', self.tlimit)
            m.setParam('MIPGap',     self.gap)
            m.setParam('OutputFlag', 0)
            x = m.addVars(N, N, V, vtype=GRB.BINARY, name='x')
            t = m.addVars(N, V, lb=0, ub=horizon,    name='t')
            m.setObjective(
                gp.quicksum(
                    self._d(nodes[i], nodes[j]) * x[i, j, v]
                    for i in range(N) for j in range(N) for v in range(V)
                    if i != j
                )
                # Pénalité par véhicule ouvert (0.0 = comportement d'origine,
                # inchangé). Un coefficient > 0 décourage Gurobi d'ouvrir un
                # véhicule de plus que nécessaire lors du repair, ce qui
                # limite l'inflation du nombre de véhicules par rapport au
                # BKS observée dans la validation.
                + self.vehicle_penalty * gp.quicksum(
                    x[0, j, v] for j in range(1, N) for v in range(V)
                ),
                GRB.MINIMIZE
            )
            for i in range(1, N):
                m.addConstr(
                    gp.quicksum(x[i, j, v]
                                for j in range(N) for v in range(V)
                                if i != j) == 1
                )
            for v in range(V):
                for i in range(1, N):
                    m.addConstr(
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j) ==
                        gp.quicksum(x[j, i, v] for j in range(N) if i != j)
                    )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(x[0, j, v] for j in range(1, N)) <= 1
                )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(
                        inst['customers'][nodes[i] - 1]['demand'] *
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j)
                        for i in range(1, N)
                    ) <= cap
                )
            M = horizon + 1000
            for v in range(V):
                m.addConstr(t[0, v] == 0)
                for i in range(N):
                    ni  = nodes[i]
                    ei  = depot['earliest'] if ni == 0 else inst['customers'][ni-1]['earliest']
                    li  = depot['latest']   if ni == 0 else inst['customers'][ni-1]['latest']
                    sti = depot['service_time'] if ni == 0 else inst['customers'][ni-1]['service_time']
                    m.addConstr(t[i, v] >= ei)
                    m.addConstr(t[i, v] <= li)
                    for j in range(N):
                        if i == j:
                            continue
                        m.addConstr(
                            t[j, v] >= t[i, v] + sti + self._d(ni, nodes[j])
                            - M * (1 - x[i, j, v])
                        )
            m.optimize()
            if m.SolCount > 0:
                eps        = 0.5
                new_routes = []
                for v in range(V):
                    if sum(x[0, j, v].X for j in range(1, N)) < eps:
                        continue
                    route_v, cur = [], 0
                    for _ in range(N):
                        nxt = next((j for j in range(N)
                                    if j != cur and x[cur, j, v].X > eps), None)
                        if nxt is None or nxt == 0:
                            break
                        route_v.append(nodes[nxt])
                        cur = nxt
                    if route_v:
                        new_routes.append(route_v)
                full = [r for r in list(partial) + new_routes if r]
                return full
            else:
                raise ValueError("Gurobi : aucune solution trouvée")
        except Exception as e:
            if self.verbose:
                print(f"    [Gurobi→Greedy fallback] {e}")
            return _repair_greedy(destroyed, partial, self.instance)
        finally:
            if m is not None:
                m.dispose()
            if env is not None:
                env.dispose()
    def step(self, action):
        old_cost            = self.current_cost
        destroyed, partial  = self._destroy(action)
        new_sol             = self._repair_gurobi(destroyed, partial)
        new_cost            = compute_cost(new_sol, self.instance)
        feasible            = is_feasible(new_sol, self.instance)
        if feasible:
            self.solution     = new_sol
            self.current_cost = new_cost
        else:
            new_cost = old_cost * 1.001
        improvement = old_cost - self.current_cost
        if improvement > 0:
            self.best_cost        = min(self.best_cost, self.current_cost)
            self.steps_no_improve = 0
        else:
            self.steps_no_improve += 1
        reward = improvement / max(self.initial_cost, 1.0)
        self.step_count += 1
        self._action_history.append(action)
        done = (self.step_count >= self.max_steps)
        info = {
            'cost':          self.current_cost,
            'best_cost':     self.best_cost,
            'action_taken':  action,
            'action_name':   ACTION_NAMES[action],
            'improvement':   improvement,
            'destroyed_k':   len(destroyed),
            'destroyed_ids': [int(d) for d in destroyed[:5]],
            'feasible':      feasible,
        }
        return self._obs(), float(reward), done, info
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 4  —  AGENT DQN  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((
            np.array(s,  dtype=np.float32), int(a), float(r),
            np.array(s2, dtype=np.float32), float(done),
        ))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.FloatTensor(np.stack(s)), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(np.stack(s2)),
                torch.FloatTensor(d))
    def __len__(self):
        return len(self.buf)
class DQNNet(nn.Module):
    def __init__(self, obs_dim=7, n_actions=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim,  hidden),  nn.ReLU(),
            nn.Linear(hidden,   hidden),  nn.ReLU(),
            nn.Linear(hidden,   n_actions),
        )
    def forward(self, x):
        return self.net(x)
class DQNAgent:
    def __init__(self, obs_dim=7, n_actions=4, hidden=64,
                 lr=1e-3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_size=2000, batch_size=32,
                 target_update_freq=10, seed=0):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        self.n_actions          = n_actions
        self.gamma              = gamma
        self.epsilon            = epsilon_start
        self.epsilon_end        = epsilon_end
        self.epsilon_decay      = epsilon_decay
        self.batch_size         = batch_size
        self.target_update_freq = target_update_freq
        self.step_count         = 0
        self.q_net      = DQNNet(obs_dim, n_actions, hidden)
        self.target_net = DQNNet(obs_dim, n_actions, hidden)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)
        self.loss_fn   = nn.MSELoss()
    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state).unsqueeze(0))
            return int(q.argmax().item())
    def store(self, s, a, r, s2, done):
        self.buffer.push(s, a, r, s2, done)
    def learn(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        s, a, r, s2, done = self.buffer.sample(self.batch_size)
        q_vals = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q  = self.target_net(s2).max(1)[0]
            targets = r + self.gamma * next_q * (1 - done)
        loss = self.loss_fn(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.optimizer.step()
        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.update_target()
        return float(loss.item())
    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    def save(self, path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        torch.save({'q_net': self.q_net.state_dict(),
                    'target_net': self.target_net.state_dict(),
                    'epsilon': self.epsilon, 'step_count': self.step_count}, path)
        print(f"    [DQN] Modèle sauvegardé → {path}")
    def load(self, path):
        """
        Recharge un checkpoint sauvegardé par save() (mêmes clés). Utilisé
        uniquement si CFG['REUSE_EXISTING_SYNTH_MODEL']=True, pour réutiliser
        un modèle déjà entraîné sur du synthétique au lieu de ré-entraîner.
        """
        ckpt = torch.load(path, map_location='cpu')
        self.q_net.load_state_dict(ckpt['q_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.epsilon    = ckpt.get('epsilon', self.epsilon)
        self.step_count = ckpt.get('step_count', self.step_count)
        print(f"    [DQN] Modèle rechargé ← {path}")
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5  —  ENTRAÎNEMENT DU DQN  (ÉTAPE 1, sur les instances SYNTHÉTIQUES)
#  Stocke cost_trace (trajectoire de coût step par step) dans le log, utilisé
#  ensuite pour les courbes de convergence détaillées. Retourne aussi
#  initial_sols pour les tracés de routes avant/après.
# ══════════════════════════════════════════════════════════════════════════════
BANNER = "═" * 72
def _sec(title):
    print(f"\n{BANNER}\n  {title}\n{BANNER}")
def train_on_instances(instances, episodes_per_instance=10, save_model=True, verbose=True):
    """
    ENTRAÎNEMENT (les poids du DQN sont mis à jour ici via store()/learn()/
    decay_epsilon()). Générique : fonctionne sur n'importe quelle liste
    d'instances au format standard — dans ce script, appelé avec les
    instances SYNTHÉTIQUES produites par generate_synthetic_training_set(),
    JAMAIS avec les instances Solomon (réservées à evaluate_frozen_
    on_instances(), plus bas, pour le TEST).
    save_model=False et verbose=False sont utilisés par le plan Taguchi
    (MODULE 10bis, run interne allégé : pas besoin d'écraser le modèle
    final ni d'un log console détaillé à chaque combinaison testée).
    Retourne : agent, training_log, initial_costs, best_costs, best_solutions,
               instance_times, initial_sols
    """
    n = len(instances)
    if verbose:
        _sec(f"ÉTAPE 1 — LNS initial pour les {n} instances d'entraînement (synthétiques)")
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        if verbose:
            print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
                  f"| véhicules={stats['n_vehicles']}")
    if verbose:
        _sec("ÉTAPE 1 — Entraînement DQN sur le synthétique")
    agent = DQNAgent(
        obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
        gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
        epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
        buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
        target_update_freq=CFG['TARGET_UPDATE'],
    )
    training_log   = []
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        if verbose:
            print(f"\n  {'═'*50}\n  [TRAIN] INSTANCE {inst['name']}  type={inst['type']}"
                  f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)
                next_obs, reward, done, info = env.step(action)
                agent.store(obs, action, reward, next_obs, float(done))
                agent.learn()
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            agent.decay_epsilon()
            training_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            if verbose:
                imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
                print(f"  [TRAIN] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                      f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        if verbose:
            print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    if save_model:
        model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
        agent.save(model_path)
    return agent, training_log, initial_costs, best_costs, best_solutions, instance_times, initial_sols
def evaluate_frozen_on_instances(instances, agent, episodes_per_instance=3):
    """
    TEST — ÉVALUATION GELÉE : l'agent DQN déjà entraîné (ÉTAPE 1, sur le
    synthétique) est ici appliqué en pur mode INFÉRENCE sur les instances
    passées en argument (les 56 Solomon officielles). Différences avec
    train_on_instances() :
      - AUCUN appel à agent.store() / agent.learn() / agent.decay_epsilon() :
        les poids du réseau ne sont JAMAIS mis à jour ici.
      - epsilon est temporairement forcé à 0.0 (politique gloutonne, pure
        exploitation du Q-réseau appris), puis restauré à sa valeur
        d'origine à la fin.
      - Le modèle n'est PAS sauvegardé ici (rien n'a changé dans ses poids).
    Le log retourné (test_log) a exactement la même forme que training_log
    (y compris 'cost_trace'), pour pouvoir réutiliser telles quelles les
    fonctions de tracé (plot_instance_convergence_examples, etc.) sur le
    TEST plutôt que sur l'entraînement.
    Retourne : initial_costs, best_costs, best_solutions, instance_times,
               initial_sols, test_log
    """
    n = len(instances)
    _sec(f"ÉTAPE 2 — ÉVALUATION GELÉE (TEST) sur {n} instances Solomon "
         f"— aucun apprentissage, epsilon=0")
    saved_epsilon = agent.epsilon
    agent.epsilon = 0.0
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    test_log       = []
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TEST] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)          # inférence pure (epsilon=0)
                next_obs, reward, done, info = env.step(action)
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            test_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TEST] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    agent.epsilon = saved_epsilon
    return initial_costs, best_costs, best_solutions, instance_times, initial_sols, test_log
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6  —  TABLEAU FINAL : SITUATION 1 vs BKS
#  (la Situation 2 "Gurobi seul sur l'instance complète" a été retirée)
# ══════════════════════════════════════════════════════════════════════════════
def build_bks_comparison_table(instances, initial_costs, best_costs, best_solutions,
                                instance_times, csv_path=None, latex_path=None):
    """
    Tableau final : Situation 1 (LNS+DQN+Gurobi) vs BKS, avec gap % et le
    nombre de véhicules utilisés pour chaque instance.
    """
    _sec("TABLEAU FINAL — SITUATION 1 vs BKS")
    rows = []
    for i, inst in enumerate(instances):
        name     = inst['name']
        bks      = BKS_REFERENCE.get(name)
        bks_dist = bks['distance'] if bks else None
        n_veh_s1 = sum(1 for r in best_solutions[i] if r)
        row = {
            'instance':             name,
            'type':                 inst['type'],
            'bks_vehicules':        bks['vehicles'] if bks else None,
            'bks_distance':         bks_dist,
            'n_vehicules_situation1': n_veh_s1,
            'distance_situation1':  round(best_costs[i], 2),
            'temps_s_situation1':   round(instance_times[i], 2),
            'gap_%_situation1_bks': round((best_costs[i] - bks_dist) / bks_dist * 100, 2) if bks_dist else None,
        }
        rows.append(row)
    print(f"\n  {'Instance':<9} {'BKS Vhc':>8} {'BKS Dist':>10} {'Vhc S1':>7} "
          f"{'Dist. S1':>10} {'Temps S1':>10} {'Gap %':>8}")
    print("  " + "─" * 70)
    for r in rows:
        bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "—"
        bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "—"
        g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {bks_v:>8} {bks_d:>10} "
              f"{r['n_vehicules_situation1']:>7} {r['distance_situation1']:>10.2f} "
              f"{r['temps_s_situation1']:>10.1f} {g1:>8}")
    # ── Gap moyen vs BKS (Situation 1) ──────────────────────────────────
    valid_gaps = [r['gap_%_situation1_bks'] for r in rows if r['gap_%_situation1_bks'] is not None]
    gap_moyen  = float(np.mean(valid_gaps)) if valid_gaps else None
    if gap_moyen is not None:
        print(f"\n  Gap moyen vs BKS (Situation 1) : {gap_moyen:+.2f}%  "
              f"({len(valid_gaps)}/{len(rows)} instances avec BKS connu)")
        print(f"  Gap min : {min(valid_gaps):+.2f}%  |  Gap max : {max(valid_gaps):+.2f}%")
    else:
        print("\n  Gap moyen vs BKS : indisponible (aucun BKS connu pour ces instances)")
    # ── Tableau récapitulatif simple : Instance / Nb véhicules / Distance / Gap % ──
    # (une ligne "Situation 1" + une ligne "BKS" par instance, format demandé)
    print(f"\n  Récapitulatif Instance / Nb véhicules / Distance / Gap % vs BKS")
    print(f"  {'Instance':<9} {'Situation':<26} {'Nb véhicules':>13} {'Distance':>10} {'Gap %':>8}")
    print("  " + "─" * 71)
    recap_rows = []
    for r in rows:
        g1 = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {'Situation 1 (DQN+LNS+Gurobi)':<26} "
              f"{r['n_vehicules_situation1']:>13} {r['distance_situation1']:>10.2f} {g1:>8}")
        recap_rows.append({
            'instance':    r['instance'],
            'situation':   'Situation 1 (DQN+LNS+Gurobi)',
            'n_vehicules': r['n_vehicules_situation1'],
            'distance':    r['distance_situation1'],
            'gap_%_vs_bks': r['gap_%_situation1_bks'],
        })
        if r['bks_distance'] is not None:
            print(f"  {r['instance']:<9} {'BKS (référence)':<26} "
                  f"{(r['bks_vehicules'] if r['bks_vehicules'] is not None else '—'):>13} "
                  f"{r['bks_distance']:>10.2f} {'0.00':>8}")
            recap_rows.append({
                'instance':    r['instance'],
                'situation':   'BKS (référence)',
                'n_vehicules': r['bks_vehicules'],
                'distance':    r['bks_distance'],
                'gap_%_vs_bks': 0.0,
            })
    if gap_moyen is not None:
        print("  " + "─" * 71)
        print(f"  {'':<9} {'MOYENNE (Situation 1)':<26} {'':>13} {'':>10} {gap_moyen:>+8.2f}")
    if csv_path:
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
        print(f"\n  Tableau final exporté (CSV) → {csv_path}")
        recap_csv_path = csv_path.replace('.csv', '_recap_vehicules_distance.csv')
        if recap_csv_path == csv_path:
            recap_csv_path = csv_path + '.recap_vehicules_distance.csv'
        with open(recap_csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=['instance', 'situation', 'n_vehicules', 'distance', 'gap_%_vs_bks'])
            w.writeheader(); w.writerows(recap_rows)
            if gap_moyen is not None:
                w.writerow({'instance': '', 'situation': 'MOYENNE (Situation 1)',
                            'n_vehicules': '', 'distance': '', 'gap_%_vs_bks': round(gap_moyen, 2)})
        print(f"  Récapitulatif Instance/Véhicules/Distance/Gap exporté (CSV) → {recap_csv_path}")
    if latex_path:
        lines = [
            "% Tableau généré automatiquement — Situation 1 vs BKS",
            "\\begin{table}[htbp]", "  \\centering",
            "  \\caption{Comparaison par instance Solomon officielle entre la méthode "
            "hybride (LNS + DQN + Gurobi repair) et le Best Known Solution (BKS) : "
            "nombre de véhicules, distance totale et écart au BKS.}",
            "  \\label{tab:comparaison-bks}",
            "  \\begin{tabular}{lrrrrrr}", "    \\toprule",
            "    Instance & BKS Vhc & BKS Dist. & Vhc. S1 & Dist. S1 & "
            "Temps S1 (s) & Gap S1 (\\%) \\\\",
            "    \\midrule",
        ]
        for r in rows:
            bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "--"
            bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "--"
            g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "--"
            lines.append(f"    {r['instance']} & {bks_v} & {bks_d} & "
                        f"{r['n_vehicules_situation1']} & {r['distance_situation1']:.2f} & "
                        f"{r['temps_s_situation1']:.1f} & {g1} \\\\")
        lines += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
        with open(latex_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        print(f"  Tableau final exporté (LaTeX) → {latex_path}")
    return rows
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6bis  —  DIAGNOSTIC : LICENCE GUROBI À TAILLE LIMITÉE ?
# ══════════════════════════════════════════════════════════════════════════════
def check_gurobi_license(min_vars_needed=2500):
    """
    Test rapide, effectué AVANT tout calcul, pour savoir si la licence
    Gurobi active est une licence "size-limited" (Community/gratuite,
    plafonnée à ~2000 variables ET ~2000 contraintes) plutôt qu'une licence
    complète (académique ou commerciale, sans limite de taille).

    Pourquoi c'est important ici : le repair Gurobi de la Situation 1
    (_repair_gurobi, appelé à CHAQUE step du LNS) crée jusqu'à
    (K_DESTROY+1)² * V variables binaires — selon K_DESTROY (cf. CFG) et le
    nombre de véhicules de l'instance, cela peut dépasser 2000. Un fallback
    vers le repair glouton est déjà en place dans _repair_gurobi (try/
    except), donc l'exécution ne plantera pas, mais peut silencieusement
    retomber sur le glouton plus souvent que prévu si la licence est limitée.

    Ce test ne bloque JAMAIS l'exécution : il informe seulement, une seule
    fois, en tête de run.
    """
    env, m = None, None
    try:
        env = gp.Env(empty=True)
        env.setParam('OutputFlag', 0)
        env.start()
        m = gp.Model(env=env)
        m.setParam('OutputFlag', 0)
        x = m.addVars(min_vars_needed, vtype=GRB.BINARY)
        m.setObjective(gp.quicksum(x[i] for i in range(min_vars_needed)), GRB.MAXIMIZE)
        m.optimize()
        print(f"  ✓ Licence Gurobi : aucune restriction de taille détectée "
              f"(modèle-test à {min_vars_needed} variables résolu sans erreur).")
        return True
    except gp.GurobiError as e:
        errno = getattr(e, 'errno', None)
        is_size_limit = (errno == getattr(GRB.Error, 'SIZE_LIMIT_EXCEEDED', object()))
        if is_size_limit:
            k_repair_vars = (CFG['K_DESTROY'] + 1) ** 2 * CFG['K_DESTROY']
            print(f"\n  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée "
                  f"(erreur {errno} : {e})\n"
                  f"    → Licence Community/gratuite, plafonnée à ~2000 variables "
                  f"et ~2000 contraintes.\n"
                  f"    → Le repair Gurobi de la Situation 1 (jusqu'à "
                  f"~{k_repair_vars} variables selon K_DESTROY={CFG['K_DESTROY']} et "
                  f"le nombre de véhicules) peut dépasser cette limite sur certaines "
                  f"instances — un repli automatique vers le repair glouton est déjà "
                  f"en place, donc pas de plantage, mais un usage de Gurobi moins "
                  f"fréquent que prévu.\n"
                  f"    → Pour lever cette limite : une licence académique Gurobi "
                  f"est gratuite et sans limite de taille pour les universités "
                  f"(https://www.gurobi.com/academia/) — utile si vous êtes "
                  f"affiliée à ENSI-UMA.\n")
        else:
            print(f"  ⚠ Erreur Gurobi lors du test de licence (non liée à la taille) : {e}")
        return False
    except Exception as e:
        print(f"  ⚠ Impossible de tester la licence Gurobi ({type(e).__name__}: {e}) "
              f"— test ignoré, l'exécution continue normalement.")
        return None
    finally:
        if m is not None:
            m.dispose()
        if env is not None:
            env.dispose()
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 7  —  COURBES DE CONVERGENCE
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_mean(vals, window=20):
    out = []
    for i in range(len(vals)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(vals[lo:i + 1])))
    return out
def plot_global_convergence(training_log, instances, initial_costs, best_costs, outdir):
    """
    Figure globale (2 panneaux) :
      (a) reward par épisode global + moyenne mobile → convergence de
          l'apprentissage DQN
      (b) amélioration % par instance (une barre par instance Solomon) vs
          coût LNS initial
    """
    episodes = [e['episode_global'] for e in training_log]
    rewards  = [e['total_reward']   for e in training_log]
    reward_smooth = _rolling_mean(rewards, window=20)
    improvements  = [(initial_costs[i] - best_costs[i]) / initial_costs[i] * 100
                      for i in range(len(initial_costs))]
    names = [inst['name'] for inst in instances]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    axes[0].plot(episodes, rewards, color='lightgray', linewidth=0.6, label='reward brut / épisode')
    axes[0].plot(episodes, reward_smooth, color='crimson', linewidth=2,
                 label='moyenne mobile (20 épisodes)')
    axes[0].set_xlabel('Épisode global')
    axes[0].set_ylabel('Reward de l\'épisode')
    axes[0].set_title("Convergence de l'apprentissage DQN")
    axes[0].legend(loc='best', fontsize=9)
    axes[0].grid(alpha=0.3)
    colors = ['#2ca02c' if v >= 0 else '#d62728' for v in improvements]
    axes[1].bar(range(len(improvements)), improvements, color=colors)
    axes[1].axhline(np.mean(improvements), color='navy', linestyle='--', linewidth=1.5,
                     label=f"Moyenne = {np.mean(improvements):.2f}%")
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names, rotation=90, fontsize=7)
    axes[1].set_xlabel('Instance')
    axes[1].set_ylabel('Amélioration vs solution LNS initiale (%)')
    axes[1].set_title("Amélioration du coût par instance après DQN+Gurobi")
    axes[1].legend(loc='best', fontsize=9)
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_globale.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence globale → {path}")
    return path
def plot_gap_vs_bks(bks_rows, outdir):
    """
    Barres du gap % vs BKS (Situation 1) pour chaque instance, avec la
    moyenne indiquée par une ligne horizontale. N'utilise que les instances
    pour lesquelles un BKS est connu (gap_%_situation1_bks non None).
    `bks_rows` = la liste `rows` retournée par build_bks_comparison_table().
    """
    data = [(r['instance'], r['gap_%_situation1_bks']) for r in bks_rows
            if r['gap_%_situation1_bks'] is not None]
    if not data:
        print("  [plot_gap_vs_bks] Aucune instance avec BKS connu — graphique ignoré.")
        return None
    names = [d[0] for d in data]
    gaps  = [d[1] for d in data]
    gap_moyen = float(np.mean(gaps))
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(names)), 5.5))
    colors = ['#2ca02c' if g <= 0 else '#d62728' for g in gaps]
    ax.bar(range(len(names)), gaps, color=colors)
    ax.axhline(gap_moyen, color='navy', linestyle='--', linewidth=1.5,
               label=f"Gap moyen = {gap_moyen:+.2f}%")
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_xlabel('Instance')
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_title(f"Écart au Best Known Solution par instance "
                 f"({len(names)} instances avec BKS connu)")
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique gap vs BKS → {path}")
    return path
def plot_instance_convergence_examples(training_log, instances, example_idxs, outdir):
    """
    Pour chaque instance exemple :
      - gauche : coût final / meilleur coût cumulé au fil des épisodes locaux
      - droite : trajectoire de coût step par step du dernier épisode local
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(13, 4.2 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        entries = [e for e in training_log if e['instance_id'] == idx]
        entries.sort(key=lambda e: e['ep_local'])
        ep_locals      = [e['ep_local']   for e in entries]
        best_costs_ep  = [e['best_cost']  for e in entries]
        final_costs_ep = [e['final_cost'] for e in entries]
        name = instances[idx]['name']
        axes[row, 0].plot(ep_locals, final_costs_ep, '-o', color='steelblue',
                           label='coût final de l\'épisode')
        axes[row, 0].plot(ep_locals, best_costs_ep, '-s', color='crimson',
                           label='meilleur coût cumulé')
        axes[row, 0].set_title(f"Instance {name} — convergence par épisode local")
        axes[row, 0].set_xlabel('Épisode local')
        axes[row, 0].set_ylabel('Coût de la tournée')
        axes[row, 0].legend(loc='best', fontsize=9)
        axes[row, 0].grid(alpha=0.3)
        last = entries[-1] if entries else None
        trace = last.get('cost_trace', []) if last else []
        if trace:
            axes[row, 1].plot(range(len(trace)), trace, '-o', color='darkorange')
            axes[row, 1].set_title(f"Instance {name} — trajectoire de coût "
                                    f"(dernier épisode local, {len(trace)-1} steps)")
            axes[row, 1].set_xlabel('Step')
            axes[row, 1].set_ylabel('Coût courant')
            axes[row, 1].grid(alpha=0.3)
        else:
            axes[row, 1].axis('off')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_instances_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence instances exemples → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 8  —  TRACÉS DES ROUTES (AVANT LNS / APRÈS DQN)
# ══════════════════════════════════════════════════════════════════════════════
def _plot_routes_on_ax(ax, instance, solution, title):
    depot = instance['depot']
    ax.scatter([depot['x']], [depot['y']], c='black', marker='s', s=110,
               zorder=5, label='Dépôt')
    n_routes = max(len([r for r in solution if r]), 1)
    try:
        cmap = matplotlib.colormaps.get_cmap('tab20').resampled(max(n_routes, 1))
    except AttributeError:
        cmap = plt.cm.get_cmap('tab20', max(n_routes, 1))
    ridx = 0
    for route in solution:
        if not route:
            continue
        xs = [depot['x']] + [instance['customers'][cid - 1]['x'] for cid in route] + [depot['x']]
        ys = [depot['y']] + [instance['customers'][cid - 1]['y'] for cid in route] + [depot['y']]
        ax.plot(xs, ys, '-o', color=cmap(ridx), markersize=4, linewidth=1.3)
        ridx += 1
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.3)
def plot_routes_examples(instances, initial_sols, best_solutions, example_idxs, outdir):
    """
    Pour chaque instance exemple : routes AVANT (solution LNS initiale, avant
    DQN) à gauche, routes APRÈS (meilleure solution trouvée par DQN+Gurobi)
    à droite. Le BKS (véhicules/distance) est indiqué en sous-titre s'il
    est connu pour l'instance.
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.8 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        inst = instances[idx]
        name = inst['name']
        bks  = BKS_REFERENCE.get(name)
        bks_str = (f" | BKS: dist={bks['distance']:.1f}, vhc={bks['vehicles']}"
                   if bks else "")
        cost_before = compute_cost(initial_sols[idx], inst)
        cost_after  = compute_cost(best_solutions[idx], inst)
        nveh_before = sum(1 for r in initial_sols[idx] if r)
        nveh_after  = sum(1 for r in best_solutions[idx] if r)
        _plot_routes_on_ax(
            axes[row, 0], inst, initial_sols[idx],
            f"Instance {name} — AVANT (LNS)\n"
            f"dist={cost_before:.1f}  véhicules={nveh_before}{bks_str}"
        )
        _plot_routes_on_ax(
            axes[row, 1], inst, best_solutions[idx],
            f"Instance {name} — APRÈS (DQN+Gurobi)\n"
            f"dist={cost_after:.1f}  véhicules={nveh_after}{bks_str}"
        )
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_routes_avant_apres_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique routes avant/après → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 9  —  ANALYSES PAR GROUPE D'INSTANCES (C1/C2/R1/R2/RC1/RC2)
#  Appliqué uniquement aux résultats du TEST (gap vs BKS) — ne joue aucun
#  rôle dans le choix des hyperparamètres, purement diagnostique a posteriori.
# ══════════════════════════════════════════════════════════════════════════════
def get_instance_group(inst):
    """
    Déduit le groupe Solomon (C1/C2/R1/R2/RC1/RC2) à partir du type et du nom
    de l'instance. Ex. : type='C', name='C101' -> 'C1' ; type='RC', name=
    'RC201' -> 'RC2'. Le chiffre juste après le préfixe alphabétique indique
    le sous-groupe (1 = fenêtres serrées / faible capacité, 2 = fenêtres
    larges / grande capacité).
    """
    t = inst['type']
    name = inst['name']
    digit = name[len(t)] if len(name) > len(t) else ''
    return f"{t}{digit}"
def plot_gap_boxplot_by_group(bks_rows, instances, outdir):
    """
    Boxplot du gap % vs BKS regroupé par C1/C2/R1/R2/RC1/RC2, pour visualiser
    quels groupes d'instances posent le plus de difficulté à la méthode.
    """
    by_name = {inst['name']: inst for inst in instances}
    groups = defaultdict(list)
    for r in bks_rows:
        if r['gap_%_situation1_bks'] is None:
            continue
        inst = by_name.get(r['instance'])
        if inst is None:
            continue
        groups[get_instance_group(inst)].append(r['gap_%_situation1_bks'])
    labels = [g for g in GROUP_ORDER if g in groups]
    data   = [groups[g] for g in labels]
    if not data:
        print("  [plot_gap_boxplot_by_group] Aucune donnée — graphique ignoré.")
        return None
    fig, ax = plt.subplots(figsize=(8, 5.5))
    try:
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, showmeans=True)
    except TypeError:
        # matplotlib < 3.9 : le paramètre s'appelait encore 'labels'.
        bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#a6c8ff')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_xlabel('Groupe d\'instances Solomon')
    ax.set_title("Distribution du gap par groupe d'instances")
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_boxplot_par_groupe.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique boxplot gap par groupe → {path}")
    return path
def plot_vehicles_vs_bks(bks_rows, instances, outdir):
    """
    Scatter plot : nombre de véhicules utilisés (Situation 1) vs nombre de
    véhicules du BKS, par instance, coloré par groupe. La diagonale y=x
    marque l'égalité ; les points au-dessus utilisent plus de véhicules que
    le BKS — c'est le phénomène d'inflation identifié dans la validation.
    """
    by_name = {inst['name']: inst for inst in instances}
    cmap_colors = plt.cm.tab10.colors
    color_map = {g: cmap_colors[i % len(cmap_colors)] for i, g in enumerate(GROUP_ORDER)}
    fig, ax = plt.subplots(figsize=(7, 7))
    seen_groups = set()
    max_v = 1
    for r in bks_rows:
        if r['bks_vehicules'] is None or r['n_vehicules_situation1'] is None:
            continue
        inst = by_name.get(r['instance'])
        grp  = get_instance_group(inst) if inst else '?'
        c    = color_map.get(grp, 'gray')
        label = grp if grp not in seen_groups else None
        seen_groups.add(grp)
        ax.scatter(r['bks_vehicules'], r['n_vehicules_situation1'], color=c,
                   label=label, s=45, alpha=0.85, edgecolor='black', linewidth=0.4)
        max_v = max(max_v, r['bks_vehicules'], r['n_vehicules_situation1'])
    ax.plot([0, max_v + 1], [0, max_v + 1], '--', color='gray', linewidth=1, label='y = x (égalité)')
    ax.set_xlabel('Véhicules BKS')
    ax.set_ylabel('Véhicules Situation 1 (DQN+LNS+Gurobi)')
    ax.set_title('Nombre de véhicules utilisés vs BKS, par instance')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_vehicules_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique véhicules vs BKS → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 10  —  PLAN D'EXPÉRIENCE TAGUCHI (calibration des hyperparamètres)
#
#  Objectif : identifier quels hyperparamètres (LNS_ITER, K_DESTROY_RATIO,
#  GUROBI_TLIMIT, GUROBI_GAP, MAX_STEPS, EPS_DECAY, VEHICLE_PENALTY) ont le
#  plus d'effet sur la performance, et quelle combinaison la maximise, sans
#  tester les 3^7 = 2187 combinaisons possibles.
#
#  ⚠ POINT CRITIQUE (train/test disjoint) : ce module supporte DEUX modes,
#  sélectionnés par le paramètre `instances_override`/`metric` de
#  run_taguchi_doe() :
#    - mode SYNTHÉTIQUE (recommandé, piloté par CFG['TAGUCHI_SYNTH']) :
#      `instances_override` = des instances synthétiques générées à la
#      volée ; `metric='lns_improvement'` = % d'amélioration vs le coût LNS
#      initial (aucun BKS requis). Les 56 Solomon ne sont JAMAIS chargées
#      dans ce mode → zéro fuite d'hyperparamètres vers le jeu de test.
#    - mode SOLOMON historique (⚠ non recommandé, piloté par CFG['TAGUCHI']) :
#      `instances_override=None` → charge INSTANCE_FILES (les 56 Solomon)
#      comme matrice externe ; `metric='bks_gap'` = gap % vs BKS_REFERENCE.
#      Ce mode entraîne et évalue directement sur le jeu de test : la
#      combinaison qu'il retourne a "vu" Solomon et ne doit pas être
#      utilisée comme hyperparamètres d'une validation train/test disjointe
#      (elle resterait valable seulement comme référence de comparaison).
#
#  Principe commun (matrice orthogonale L9/L18) :
#    - Matrice interne  : la matrice orthogonale L18 ci-dessous (18 lignes,
#      7 facteurs, 3 niveaux) — les combinaisons de facteurs à tester.
#    - Matrice externe (bruit) : les instances sur lesquelles chaque
#      combinaison de la matrice interne est évaluée ; le ratio signal/bruit
#      (S/N) résume à la fois la performance moyenne ET la stabilité face à
#      la variabilité entre instances/groupes.
#    - Après le plan, reportez la meilleure combinaison dans CFG, repassez
#      ENABLE=False, et relancez le script pour la validation ENTRAÎNEMENT/
#      TEST DISJOINT normale (avec ces hyperparamètres).
# ══════════════════════════════════════════════════════════════════════════════
# Matrice orthogonale L9 (3^4) — 9 runs, jusqu'à 4 facteurs à 3 niveaux.
L9_ARRAY = [
    [1, 1, 1, 1], [1, 2, 2, 2], [1, 3, 3, 3],
    [2, 1, 2, 3], [2, 2, 3, 1], [2, 3, 1, 2],
    [3, 1, 3, 2], [3, 2, 1, 3], [3, 3, 2, 1],
]
# Matrice orthogonale L18 (2^1 x 3^7) — 18 runs. La 1ère colonne (2 niveaux)
# n'est pas utilisée ici (tous nos facteurs sont à 3 niveaux) ; seules les
# colonnes 2 à 8 (7 facteurs à 3 niveaux) servent au plan.
L18_ARRAY_FULL = [
    [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 2, 2, 2, 2, 2, 2], [1, 1, 3, 3, 3, 3, 3, 3],
    [1, 2, 1, 1, 2, 2, 3, 3], [1, 2, 2, 2, 3, 3, 1, 1], [1, 2, 3, 3, 1, 1, 2, 2],
    [1, 3, 1, 2, 1, 3, 2, 3], [1, 3, 2, 3, 2, 1, 3, 1], [1, 3, 3, 1, 3, 2, 1, 2],
    [2, 1, 1, 3, 3, 2, 2, 1], [2, 1, 2, 1, 1, 3, 3, 2], [2, 1, 3, 2, 2, 1, 1, 3],
    [2, 2, 1, 2, 3, 1, 3, 2], [2, 2, 2, 3, 1, 2, 1, 3], [2, 2, 3, 1, 2, 3, 2, 1],
    [2, 3, 1, 3, 2, 3, 1, 2], [2, 3, 2, 1, 3, 1, 2, 3], [2, 3, 3, 2, 1, 2, 3, 1],
]
L18_ARRAY = [row[1:] for row in L18_ARRAY_FULL]   # ne garder que les 7 colonnes à 3 niveaux
def check_orthogonal_array(array):
    """
    Vérifie la propriété d'orthogonalité : pour chaque paire de colonnes,
    chaque combinaison de niveaux doit apparaître le même nombre de fois.
    Retourne True/False — sert de garde-fou avant de lancer un plan
    potentiellement long sur une matrice mal formée.
    """
    arr = np.array(array)
    n_runs, n_cols = arr.shape
    ok = True
    for c1, c2 in itertools.combinations(range(n_cols), 2):
        counts = defaultdict(int)
        for row in arr:
            counts[(row[c1], row[c2])] += 1
        if len(set(counts.values())) > 1:
            ok = False
    return ok
def select_orthogonal_array(n_factors):
    if n_factors <= 4:
        return 'L9', [row[:n_factors] for row in L9_ARRAY]
    elif n_factors <= 7:
        return 'L18', [row[:n_factors] for row in L18_ARRAY]
    else:
        raise ValueError(
            f"{n_factors} facteurs demandés, mais seules les matrices L9 (4 "
            f"facteurs max) et L18 (7 facteurs max) sont disponibles. "
            f"Réduisez TAGUCHI_FACTORS ou étendez ce module avec une L27."
        )
@contextmanager
def _cfg_override(**kwargs):
    """Applique temporairement des valeurs de CFG (pour un run du plan
    Taguchi), puis restaure les valeurs d'origine à la sortie du bloc."""
    old = {k: CFG.get(k) for k in kwargs}
    CFG.update(kwargs)
    try:
        yield
    finally:
        CFG.update(old)
def taguchi_sn_ratio_stb(values):
    """
    Ratio signal/bruit 'plus petit est meilleur' (Smaller-The-Better),
    calculé sur une liste de valeurs (le gap %/l'amélioration % peut être
    négatif — la formule standard SN = -10*log10(mean(y_i^2)) reste valide,
    elle pénalise tout écart à zéro, positif ou négatif, ainsi que la
    variance entre les instances de la matrice externe).
    """
    y = np.array(values, dtype=float)
    msd = max(float(np.mean(y ** 2)), 1e-9)
    return -10.0 * np.log10(msd)
def taguchi_main_effects(results, factor_names, response_key):
    """Moyenne de la réponse (gap/amélioration ou S/N) par niveau, pour
    chaque facteur."""
    effects = {}
    for fn in factor_names:
        by_level = defaultdict(list)
        for r in results:
            if r.get(response_key) is not None:
                by_level[r[fn]].append(r[response_key])
        effects[fn] = {lvl: float(np.mean(vals)) for lvl, vals in sorted(by_level.items(), key=lambda kv: kv[0])}
    return effects
def taguchi_anova_contributions(results, factor_names, response_key):
    """
    ANOVA simplifiée (méthode des sommes de carrés à partir des moyennes par
    niveau) : contribution en % de chaque facteur à la variance totale de la
    réponse. Ne remplace pas une ANOVA complète avec terme d'erreur pur,
    mais donne une hiérarchie fiable de l'importance relative des facteurs.
    """
    valid = [r for r in results if r.get(response_key) is not None]
    if not valid:
        return {fn: 0.0 for fn in factor_names}
    y = np.array([r[response_key] for r in valid])
    grand_mean = y.mean()
    ss_total = float(np.sum((y - grand_mean) ** 2))
    contributions = {}
    for fn in factor_names:
        by_level = defaultdict(list)
        for r in valid:
            by_level[r[fn]].append(r[response_key])
        ss_factor = sum(len(vals) * (np.mean(vals) - grand_mean) ** 2 for vals in by_level.values())
        contributions[fn] = (ss_factor / ss_total * 100.0) if ss_total > 0 else 0.0
    return contributions
def plot_taguchi_main_effects(effects, factor_names, response_label, outdir, filename):
    """Graphique classique des effets principaux Taguchi : un panneau par
    facteur, moyenne de la réponse pour chaque niveau, reliée par une ligne."""
    n = len(factor_names)
    ncols = min(4, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, fn in enumerate(factor_names):
        levels = list(effects[fn].keys())
        means  = list(effects[fn].values())
        axes[i].plot(range(len(levels)), means, '-o', color='steelblue')
        axes[i].set_xticks(range(len(levels)))
        axes[i].set_xticklabels([str(l) for l in levels], fontsize=8)
        axes[i].set_title(fn, fontsize=10)
        axes[i].set_ylabel(response_label, fontsize=8)
        axes[i].grid(alpha=0.3)
    for j in range(len(factor_names), len(axes)):
        axes[j].axis('off')
    fig.suptitle(f"Effets principaux — {response_label}", fontsize=13)
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique effets principaux ({response_label}) → {path}")
    return path
def plot_taguchi_sn_bars(results, outdir, filename='taguchi_sn_ratios.png'):
    """Ratio S/N obtenu pour chaque run du plan, run avec le meilleur S/N
    mis en évidence en vert."""
    valid = [r for r in results if r.get('sn_ratio') is not None]
    if not valid:
        print("  [plot_taguchi_sn_bars] Aucun résultat — graphique ignoré.")
        return None
    runs = [r['run'] for r in valid]
    sns_ = [r['sn_ratio'] for r in valid]
    best_idx = int(np.argmax(sns_))
    colors = ['#2ca02c' if i == best_idx else '#4c72b0' for i in range(len(sns_))]
    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(runs)), 5))
    ax.bar([str(r) for r in runs], sns_, color=colors)
    ax.set_xlabel('Run (combinaison de la matrice L9/L18)')
    ax.set_ylabel('Ratio S/N (plus grand = meilleur)')
    ax.set_title('Ratio signal/bruit par combinaison — plan Taguchi')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique S/N par run → {path}")
    return path
def plot_taguchi_pareto(anova_contributions, outdir, filename='taguchi_pareto_effets.png'):
    """Diagramme en barres horizontales de la contribution (%) de chaque
    facteur à la variance de la réponse — équivalent simplifié d'un Pareto ANOVA."""
    items = sorted(anova_contributions.items(), key=lambda kv: -kv[1])
    names = [k for k, v in items][::-1]
    vals  = [v for k, v in items][::-1]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(range(len(names)), vals, color='#4c72b0')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('Contribution à la variance de la réponse (%)')
    ax.set_title("Importance relative des facteurs (ANOVA simplifiée Taguchi)")
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique Pareto des effets → {path}")
    return path
def run_taguchi_doe(factors, outer_instance_names, episodes_per_instance_doe, outdir,
                     instances_override=None, metric='bks_gap', json_suffix=''):
    """
    Exécute le plan d'expérience Taguchi complet :
      1. Construit la matrice orthogonale adaptée au nombre de facteurs
      2. Pour chaque run, applique les niveaux de facteurs (via _cfg_override)
         et entraîne le pipeline complet (LNS+DQN+Gurobi) sur les instances
         de la matrice externe, avec un nombre d'épisodes réduit
      3. Calcule la métrique de classement par instance (voir `metric`), la
         moyenne, et le ratio S/N
      4. Analyse les effets principaux + ANOVA simplifiée, génère les graphes
      5. Rapporte la meilleure combinaison trouvée
    instances_override : si fourni (liste d'instances déjà construites),
        utilisé DIRECTEMENT comme matrice externe, SANS jamais charger les
        fichiers Solomon — c'est ce qui permet de calibrer sur des instances
        SYNTHÉTIQUES (voir CFG['TAGUCHI_SYNTH']) sans aucune fuite vers le
        jeu de test. Si None (mode historique Solomon), charge les instances
        via INSTANCE_FILES et `outer_instance_names`.
    metric : 'bks_gap' (gap % vs BKS_REFERENCE — ne fonctionne que pour des
        instances Solomon officielles connues dans BKS_REFERENCE) ou
        'lns_improvement' (% d'amélioration vs le coût LNS initial — aucun
        BKS requis, utilisable sur des instances synthétiques). Convention
        "plus petit est meilleur" identique dans les deux cas (négatif =
        amélioration), donc toute la mécanique Taguchi (S/N, effets
        principaux, ANOVA, combinaison prédite) fonctionne sans modification.
    json_suffix : suffixe inséré dans les noms des fichiers de sortie (CSV,
        JSON, PNG) — ex. '_synthetique' — pour ne jamais écraser les
        résultats d'un plan calibré sur d'autres instances/métrique.
    Retourne : results (liste de dicts, un par run), effects_gap, effects_sn,
               anova_contributions, best_run
    """
    factor_names  = [f[0] for f in factors]
    factor_levels = [f[1] for f in factors]
    n_factors     = len(factors)
    array_name, array = select_orthogonal_array(n_factors)
    ortho_ok = check_orthogonal_array(array)
    if instances_override is not None:
        outer_instances = list(instances_override)
    else:
        all_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
        if outer_instance_names == 'ALL':
            # Sentinel spécial : utiliser TOUTES les instances Solomon
            # chargées comme matrice externe. ⚠ Mode historique — voir
            # avertissement CFG['TAGUCHI'] plus haut : fuite d'hyperparamètres
            # vers le jeu de test. Multiplie aussi fortement le temps total
            # du plan (voir avertissement de durée ci-dessous).
            outer_instances = list(all_instances)
        else:
            by_name = {inst['name']: inst for inst in all_instances}
            outer_instances = [by_name[nm] for nm in outer_instance_names if nm in by_name]
    if not outer_instances:
        raise ValueError("Aucune instance de la matrice externe trouvée "
                          "(instances_override vide, ou outer_instance_names "
                          "absent de INSTANCE_FILES).")
    gurobi_tlimit_max = max(TAGUCHI_FACTORS[[f[0] for f in TAGUCHI_FACTORS].index('GUROBI_TLIMIT')][1]) \
        if 'GUROBI_TLIMIT' in factor_names else CFG['GUROBI_TLIMIT']
    max_steps_max = max(TAGUCHI_FACTORS[[f[0] for f in TAGUCHI_FACTORS].index('MAX_STEPS')][1]) \
        if 'MAX_STEPS' in factor_names else CFG['MAX_STEPS']
    worst_case_h = (len(array) * len(outer_instances) * gurobi_tlimit_max
                     * max_steps_max * episodes_per_instance_doe) / 3600.0
    _sec(f"PLAN D'EXPÉRIENCE TAGUCHI — {array_name} ({len(array)} runs, "
         f"{n_factors} facteurs, métrique={metric}, orthogonalité="
         f"{'OK' if ortho_ok else 'A VERIFIER !'})")
    if len(outer_instances) <= 12:
        print(f"  Matrice externe (bruit) : {[i['name'] for i in outer_instances]}")
    else:
        print(f"  Matrice externe (bruit) : {len(outer_instances)} instances "
              f"({outer_instances[0]['name']} ... {outer_instances[-1]['name']})")
    print(f"  Épisodes/instance pendant le plan : {episodes_per_instance_doe}")
    print(f"  ⚠ Temps pire cas estimé (si Gurobi atteint systématiquement sa limite de "
          f"temps) : ~{worst_case_h:.1f}h ({worst_case_h/24:.1f} jours). "
          f"Le temps réel est généralement inférieur, mais reste substantiel à cette échelle.")
    results = []
    for run_idx, level_idx_row in enumerate(array, start=1):
        levels = {factor_names[i]: factor_levels[i][level_idx_row[i] - 1] for i in range(n_factors)}
        print(f"\n  Run {run_idx:02d}/{len(array)} : {levels}")
        overrides = dict(levels)
        if 'K_DESTROY_RATIO' in overrides:
            overrides['K_DESTROY'] = max(3, round(100 * overrides['K_DESTROY_RATIO']))
        t_run = time.time()
        with _cfg_override(**overrides):
            _, _, initial_costs_o, best_costs_o, _, _, _ = train_on_instances(
                outer_instances, episodes_per_instance=episodes_per_instance_doe,
                save_model=False, verbose=False,
            )
        values = []
        for i, inst in enumerate(outer_instances):
            if metric == 'bks_gap':
                bks = BKS_REFERENCE.get(inst['name'])
                if bks:
                    values.append((best_costs_o[i] - bks['distance']) / bks['distance'] * 100.0)
            elif metric == 'lns_improvement':
                ic = initial_costs_o[i]
                if ic:
                    values.append((best_costs_o[i] - ic) / ic * 100.0)
            else:
                raise ValueError(f"metric inconnue : {metric!r} (attendu 'bks_gap' ou 'lns_improvement')")
        mean_val = float(np.mean(values)) if values else None
        sn       = taguchi_sn_ratio_stb(values) if values else None
        row_result = dict(levels)
        row_result.update({
            'run': run_idx,
            'mean_gap_%': round(mean_val, 3) if mean_val is not None else None,
            'sn_ratio':   round(sn, 3) if sn is not None else None,
        })
        results.append(row_result)
        elapsed = time.time() - t_run
        if mean_val is not None:
            print(f"    → {'gap' if metric=='bks_gap' else 'amélioration'} moyen(ne) = "
                  f"{mean_val:+.2f}%  |  S/N = {sn:.2f}  |  {elapsed:.1f}s")
        else:
            print(f"    → indisponible  |  {elapsed:.1f}s")
    csv_path = os.path.join(outdir, f'taguchi_resultats{json_suffix}.csv')
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = factor_names + ['run', 'mean_gap_%', 'sn_ratio']
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        w.writeheader()
        for r in results:
            w.writerow(r)
    print(f"\n  Résultats Taguchi exportés (CSV) → {csv_path}")
    effects_gap = taguchi_main_effects(results, factor_names, 'mean_gap_%')
    effects_sn  = taguchi_main_effects(results, factor_names, 'sn_ratio')
    anova       = taguchi_anova_contributions(results, factor_names, 'mean_gap_%')
    response_label = 'Gap moyen vs BKS (%)' if metric == 'bks_gap' else 'Amélioration moyenne vs LNS (%)'
    _sec("GRAPHIQUES TAGUCHI")
    plot_taguchi_main_effects(effects_gap, factor_names, response_label, outdir,
                              f'taguchi_effets_principaux_gap{json_suffix}.png')
    plot_taguchi_main_effects(effects_sn, factor_names, 'Ratio S/N', outdir,
                              f'taguchi_effets_principaux_sn{json_suffix}.png')
    plot_taguchi_sn_bars(results, outdir, filename=f'taguchi_sn_ratios{json_suffix}.png')
    plot_taguchi_pareto(anova, outdir, filename=f'taguchi_pareto_effets{json_suffix}.png')
    valid_runs = [r for r in results if r['mean_gap_%'] is not None]
    best_run = min(valid_runs, key=lambda r: abs(r['mean_gap_%'])) if valid_runs else None
    print(f"\n{'█'*72}")
    print(f"  █  MEILLEURE COMBINAISON TROUVÉE — PLAN TAGUCHI{' (' + json_suffix + ')' if json_suffix else ''}")
    print(f"{'█'*72}")
    if best_run:
        print(f"  Run {best_run['run']} : {'gap' if metric=='bks_gap' else 'amélioration'} moyen(ne) = "
              f"{best_run['mean_gap_%']:+.2f}%  |  S/N = {best_run['sn_ratio']:.2f}")
        print(f"  ── Hyperparamètres recommandés ──")
        for fn in factor_names:
            print(f"    {fn:<18s} = {best_run[fn]}")
        print(f"\n  ── Contribution des facteurs à la variance de la réponse (ANOVA) ──")
        for fn, pct in sorted(anova.items(), key=lambda kv: -kv[1]):
            print(f"    {fn:<18s} : {pct:5.1f}%")
        print(f"\n  → Reportez ces valeurs dans CFG (et TAGUCHI_FACTORS si besoin), "
              f"repassez ENABLE=False, et relancez le script pour la validation "
              f"ENTRAÎNEMENT/TEST DISJOINT sur les 56 instances Solomon.")
        best_json_path = os.path.join(outdir, f'taguchi_meilleure_combinaison{json_suffix}.json')
        with open(best_json_path, 'w', encoding='utf-8') as f:
            json.dump({
                'run':               best_run['run'],
                'metric':            metric,
                'mean_gap_%':        best_run['mean_gap_%'],
                'sn_ratio':          best_run['sn_ratio'],
                'hyperparametres':   {fn: best_run[fn] for fn in factor_names},
                'contribution_anova_%': anova,
                'array':             array_name,
                'n_runs':            len(array),
                'n_outer_instances': len(outer_instances),
                'episodes_per_instance_doe': episodes_per_instance_doe,
            }, f, indent=2, ensure_ascii=False)
        print(f"\n  Meilleure combinaison sauvegardée (JSON) → {best_json_path}")
    print(f"{'█'*72}\n")
    return results, effects_gap, effects_sn, anova, best_run
def load_and_apply_taguchi_best(outdir, json_suffix=''):
    """
    Étape de confirmation : cherche taguchi_meilleure_combinaison<suffix>.json
    (créé par run_taguchi_doe) et applique ces hyperparamètres à CFG, pour
    que la validation ENTRAÎNEMENT/TEST DISJOINT utilise automatiquement la
    meilleure combinaison trouvée — sans avoir à la reporter à la main.
    Passez json_suffix='_synthetique' pour charger la combinaison calibrée
    UNIQUEMENT sur du synthétique (recommandé). Ne modifie rien (et affiche
    un avertissement) si le fichier n'existe pas encore.
    Retourne True si des hyperparamètres ont été appliqués, False sinon.
    """
    path = os.path.join(outdir, f'taguchi_meilleure_combinaison{json_suffix}.json')
    if not os.path.exists(path):
        cfg_key_hint = "CFG['TAGUCHI_SYNTH']" if json_suffix else "CFG['TAGUCHI']"
        print(f"  ⚠ {path} introuvable — lancez d'abord le plan Taguchi correspondant "
              f"({cfg_key_hint}['ENABLE']=True). Hyperparamètres CFG actuels inchangés.")
        return False
    with open(path, 'r', encoding='utf-8') as f:
        best = json.load(f)
    hp = best.get('hyperparametres', {})
    for fn, val in hp.items():
        CFG[fn] = val
    if 'K_DESTROY_RATIO' in hp:
        CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))
    print(f"  ✓ Hyperparamètres Taguchi appliqués depuis {path} :")
    for fn, val in hp.items():
        print(f"    {fn:<18s} = {val}")
    return True
# ══════════════════════════════════════════════════════════════════════════════
#  EXÉCUTION AUTOMATIQUE
# ══════════════════════════════════════════════════════════════════════════════
def run_final_validation():
    """
    MODE VALIDATION : ENTRAÎNEMENT/TEST DISJOINT.
    Entraîne le DQN sur CFG['N_SYNTH_INSTANCES'] instances synthétiques puis
    l'évalue gelé sur les 56 instances Solomon officielles. Si
    CFG['USE_TAGUCHI_BEST']=True, applique d'abord automatiquement la
    meilleure combinaison d'hyperparamètres trouvée par le plan Taguchi
    synthétique (fichier taguchi_meilleure_combinaison_synthetique.json).

    Cette fonction regroupe exactement ce que faisait auparavant la branche
    `else` du bloc __main__ — extraite dans une fonction pour pouvoir être
    appelée aussi bien seule (validation directe, sans calibration) que
    depuis la branche Taguchi synthétique, en enchaînement automatique.
    """
    t0 = time.time()
    print(f"\n{'═'*72}\n  MODE ENTRAÎNEMENT/TEST DISJOINT\n"
          f"  ENTRAÎNEMENT = {CFG['N_SYNTH_INSTANCES']} instances SYNTHÉTIQUES générées "
          f"| TEST = 56 instances Solomon officielles (gelé)\n{'═'*72}")

    # ── DIAGNOSTIC PRÉALABLE : licence Gurobi à taille limitée ? ────────
    # Informatif uniquement — n'interrompt jamais l'exécution.
    _sec("DIAGNOSTIC — LICENCE GUROBI")
    check_gurobi_license()

    # ── Optionnel : appliquer la meilleure combinaison Taguchi calibrée
    # sur synthétique (si le plan CFG['TAGUCHI_SYNTH'] a déjà été lancé
    # au moins une fois et a produit le JSON correspondant) ───────────
    if CFG.get('USE_TAGUCHI_BEST', False):
        _sec("APPLICATION DE LA MEILLEURE COMBINAISON TAGUCHI (SYNTHÉTIQUE)")
        load_and_apply_taguchi_best(CFG['OUTDIR'], json_suffix='_synthetique')

    # ── ÉTAPE 1a : GÉNÉRATION DES INSTANCES SYNTHÉTIQUES D'ENTRAÎNEMENT ─
    # (reprend generate_instance() de Code 1 — cf. MODULE 1bis ci-dessus).
    # Ces instances SYN-* ne recoupent jamais les noms Solomon (C101, R101,
    # ...) chargés à l'ÉTAPE 2 : aucune fuite entre train et test.
    synth_instances = generate_synthetic_training_set(CFG)

    # ── ÉTAPE 1b : ENTRAÎNEMENT DU DQN — uniquement sur le synthétique ──
    # Valeurs par défaut : si le modèle est rechargé (branche ci-dessous),
    # il n'y a pas de training_log/coûts synthétiques dans CE run — les
    # graphiques de convergence d'entraînement seront alors simplement omis.
    training_log, synth_initial_costs, synth_best_costs = [], None, None
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    if CFG.get('REUSE_EXISTING_SYNTH_MODEL', False) and os.path.exists(model_path):
        _sec(f"ÉTAPE 1b — RÉUTILISATION D'UN MODÈLE DÉJÀ ENTRAÎNÉ SUR LE SYNTHÉTIQUE "
             f"({model_path})")
        agent = DQNAgent(
            obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
            gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
            epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
            buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
            target_update_freq=CFG['TARGET_UPDATE'],
        )
        agent.load(model_path)
    else:
        agent, training_log, synth_initial_costs, synth_best_costs, \
            synth_best_solutions, synth_instance_times, synth_initial_sols = train_on_instances(
                synth_instances, episodes_per_instance=CFG['EPISODES_PER_INSTANCE'])

    # ── ÉTAPE 2a : CHARGEMENT DES 56 INSTANCES SOLOMON (TEST — jamais vues) ─
    _sec("ÉTAPE 2 — CHARGEMENT DES INSTANCES SOLOMON (TEST)")
    test_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
    example_idxs = pick_example_indices(test_instances, preferred_names=CFG['EXAMPLE_NAMES'])

    # ── ÉTAPE 2b : ÉVALUATION GELÉE — le même agent en mémoire, sans réentraîner ──
    initial_costs, best_costs, best_solutions, instance_times, test_initial_sols, test_log = \
        evaluate_frozen_on_instances(
            test_instances, agent, episodes_per_instance=CFG['EPISODES_PER_INSTANCE_TEST'])

    # ── [A] Convergence de l'ENTRAÎNEMENT (synthétique uniquement) ──────
    _sec("GRAPHIQUE — CONVERGENCE DE L'ENTRAÎNEMENT (SYNTHÉTIQUE)")
    if training_log:
        plot_global_convergence(training_log, synth_instances,
                                 synth_initial_costs, synth_best_costs, CFG['OUTDIR'])
    else:
        print("  (pas de courbe de convergence : modèle rechargé, aucun "
              "entraînement effectué dans ce run)")

    # ── [B] Convergence détaillée + [C] Routes avant/après — SUR LE TEST ─
    _sec("GRAPHIQUES — DÉTAIL DU TEST (INSTANCES SOLOMON EXEMPLES)")
    plot_instance_convergence_examples(test_log, test_instances, example_idxs, CFG['OUTDIR'])
    plot_routes_examples(test_instances, test_initial_sols, best_solutions, example_idxs, CFG['OUTDIR'])

    # ── [D] TABLEAU FINAL : SITUATION 1 (test gelé, entraîné sur synthétique) ─
    # vs BKS (gap % par instance + gap moyen) ──────────────────────────
    bks_rows = build_bks_comparison_table(
        test_instances, initial_costs, best_costs, best_solutions, instance_times,
        csv_path=os.path.join(CFG['OUTDIR'], 'solomon_comparaison_bks.csv'),
        latex_path=os.path.join(CFG['OUTDIR'], 'solomon_tableau_bks.tex'),
    )
    # ── [E] Graphique dédié : gap % vs BKS par instance + gap moyen (TEST) ─
    _sec("GRAPHIQUE GAP % vs BKS (TEST)")
    plot_gap_vs_bks(bks_rows, CFG['OUTDIR'])

    # ── [F] Analyses par groupe (C1/C2/R1/R2/RC1/RC2) — véhicules & gap ──
    _sec("GRAPHIQUES — ANALYSES PAR GROUPE D'INSTANCES (TEST)")
    plot_gap_boxplot_by_group(bks_rows, test_instances, CFG['OUTDIR'])
    plot_vehicles_vs_bks(bks_rows, test_instances, CFG['OUTDIR'])

    print(f"\n  ✓ Validation finale terminée en {time.time()-t0:.1f}s "
          f"({(time.time()-t0)/3600:.2f}h)\n")
    return bks_rows


if __name__ == '__main__':
    t_global0 = time.time()

    if CFG['TAGUCHI_SYNTH']['ENABLE'] and CFG['TAGUCHI']['ENABLE']:
        raise ValueError(
            "CFG['TAGUCHI_SYNTH']['ENABLE'] et CFG['TAGUCHI']['ENABLE'] sont tous les "
            "deux True — activez UN SEUL des deux modes de calibration à la fois "
            "(TAGUCHI_SYNTH est recommandé, voir les commentaires plus haut)."
        )

    if CFG['TAGUCHI_SYNTH']['ENABLE']:
        # ── MODE CALIBRATION TAGUCHI — SYNTHÉTIQUE UNIQUEMENT (RECOMMANDÉ) ──
        # Aucune instance Solomon n'est chargée dans cette branche : zéro
        # fuite d'hyperparamètres vers le jeu de test.
        ts = CFG['TAGUCHI_SYNTH']
        print(f"\n{'═'*72}\n  MODE PLAN D'EXPÉRIENCE TAGUCHI — SYNTHÉTIQUE UNIQUEMENT "
              f"({ts['N_INSTANCES_DOE']} instances, métrique = amélioration vs LNS, "
              f"pas de BKS, zéro fuite vers Solomon)\n{'═'*72}")
        synth_doe_instances = generate_synthetic_training_set(
            dict(N_SYNTH_INSTANCES=ts['N_INSTANCES_DOE'],
                 SYNTH_N_CUSTOMERS=CFG['SYNTH_N_CUSTOMERS'],
                 SYNTH_N_VEHICLES=CFG['SYNTH_N_VEHICLES'],
                 SYNTH_VEHICLE_SAFETY_FACTOR=CFG['SYNTH_VEHICLE_SAFETY_FACTOR'],
                 SYNTH_CAPACITY=CFG['SYNTH_CAPACITY'], SYNTH_HORIZON=CFG['SYNTH_HORIZON']),
            verbose=True,
        )
        (ts_results, ts_effects_gap, ts_effects_sn,
         ts_anova, ts_best_run) = run_taguchi_doe(
            factors=TAGUCHI_FACTORS,
            outer_instance_names=None,
            episodes_per_instance_doe=ts['EPISODES_PER_INSTANCE_DOE'],
            outdir=CFG['OUTDIR'],
            instances_override=synth_doe_instances,
            metric='lns_improvement',
            json_suffix='_synthetique',
        )
        _sec("RÉCAPITULATIF FINAL DU PLAN TAGUCHI (SYNTHÉTIQUE)")
        if ts_best_run:
            print("  Meilleurs hyperparamètres trouvés (calibrés sur synthétique) :")
            for fn, _ in TAGUCHI_FACTORS:
                print(f"    {fn:<18s} = {ts_best_run[fn]}")
            print(f"  Amélioration moyenne vs LNS obtenue : {-ts_best_run['mean_gap_%']:.2f}%")
            print(f"  Ratio S/N                           : {ts_best_run['sn_ratio']:.2f}")
        print(f"\n  Meilleure combinaison (JSON) : "
              f"{os.path.join(CFG['OUTDIR'], 'taguchi_meilleure_combinaison_synthetique.json')}")
        print(f"\n  ✓ Plan Taguchi (synthétique) terminé en {time.time()-t_global0:.1f}s "
              f"({(time.time()-t_global0)/3600:.2f}h).")

        # ── ENCHAÎNEMENT AUTOMATIQUE → VALIDATION FINALE AVEC LES BEST HYPERPARAMÈTRES ──
        # Entraînement 80 instances synthétiques + test gelé sur les 56 Solomon,
        # en un seul lancement, sans intervention manuelle.
        if ts_best_run:
            _sec("ENCHAÎNEMENT AUTOMATIQUE → VALIDATION FINALE")
            CFG['USE_TAGUCHI_BEST'] = True
            # Repart d'un modèle neuf avec les NOUVEAUX hyperparamètres —
            # ne jamais réutiliser un .pth calibré avec d'anciens réglages ici.
            CFG['REUSE_EXISTING_SYNTH_MODEL'] = False
            run_final_validation()
        else:
            print("  ⚠ Aucune combinaison valide trouvée par le plan Taguchi — "
                  "validation finale non lancée.")

    elif CFG['TAGUCHI']['ENABLE']:
        # ── MODE CALIBRATION TAGUCHI — SOLOMON (⚠ HISTORIQUE, NON RECOMMANDÉ) ──
        # Volontairement PAS d'enchaînement automatique ici : ce mode calibre
        # directement sur le jeu de test (fuite train/test), la combinaison
        # trouvée ne doit pas servir à une validation train/test disjointe.
        print(f"\n{'═'*72}\n  ⚠ MODE PLAN D'EXPÉRIENCE TAGUCHI — SOLOMON (mode historique)\n"
              f"  Ce mode entraîne ET évalue directement sur les 56 instances Solomon :\n"
              f"  la combinaison retournée aura VU le jeu de test et ne doit PAS être\n"
              f"  utilisée comme hyperparamètres d'une validation train/test disjointe.\n"
              f"  Préférez CFG['TAGUCHI_SYNTH']['ENABLE']=True (calibration synthétique).\n"
              f"{'═'*72}")
        tg = CFG['TAGUCHI']
        (tg_results, tg_effects_gap, tg_effects_sn,
         tg_anova, tg_best_run) = run_taguchi_doe(
            factors=TAGUCHI_FACTORS,
            outer_instance_names=tg['OUTER_INSTANCE_NAMES'],
            episodes_per_instance_doe=tg['EPISODES_PER_INSTANCE_DOE'],
            outdir=CFG['OUTDIR'],
            instances_override=None,
            metric='bks_gap',
            json_suffix='',
        )
        _sec("RÉCAPITULATIF FINAL DU PLAN TAGUCHI (SOLOMON — HISTORIQUE)")
        if tg_best_run:
            print("  Meilleurs hyperparamètres trouvés (⚠ calibrés sur Solomon — fuite) :")
            for fn, _ in TAGUCHI_FACTORS:
                print(f"    {fn:<18s} = {tg_best_run[fn]}")
            print(f"  Gap moyen obtenu : {tg_best_run['mean_gap_%']:+.2f}%")
            print(f"  Ratio S/N        : {tg_best_run['sn_ratio']:.2f}")
        print(f"\n  ✓ Plan Taguchi (Solomon) terminé en {time.time()-t_global0:.1f}s "
              f"({(time.time()-t_global0)/3600:.2f}h). Pas d'enchaînement automatique "
              f"(mode avec fuite) — repassez CFG['TAGUCHI']['ENABLE']=False et relancez "
              f"manuellement si vous souhaitez tout de même l'utiliser, à titre de "
              f"comparaison uniquement.\n")

    else:
        # ── MODE VALIDATION SEULE (comportement normal, sans calibration) ──
        run_final_validation()


════════════════════════════════════════════════════════════════════════
  MODE PLAN D'EXPÉRIENCE TAGUCHI — SYNTHÉTIQUE UNIQUEMENT (24 instances, métrique = amélioration vs LNS, pas de BKS, zéro fuite vers Solomon)
════════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════════
  ÉTAPE 1 — GÉNÉRATION DE 24 INSTANCES SYNTHÉTIQUES (ENTRAÎNEMENT)
════════════════════════════════════════════════════════════════════════
  Instance 001/24 | SYN-C001   type=C   | seed=1 | véhicules=17
  Instance 002/24 | SYN-R002   type=R   | seed=8 | véhicules=17
  Instance 003/24 | SYN-RC003  type=RC  | seed=15 | véhicules=18
  Instance 004/24 | SYN-C004   type=C   | seed=22 | véhicules=17
  Instance 005/24 | SYN-R005   type=R   | seed=29 | véhicules=17
  Instance 006/24 | SYN-RC006  type=RC  | seed=36 | véhicules=15
  Instance 007/24 | SYN-C007   type=C   | seed=43 | véhicules=15
  Instance 008/24 | SYN-R008   type=R   | se

## Plan Taguchi (calibration) : EPISODES_PER_INSTANCE_DOE = 5 / Entraînement : EPISODES_PER_INSTANCE = 20 /Test : EPISODES_PER_INSTANCE_TEST = 10

In [10]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON    ║
║                          + GAP PAR RAPPORT AU BKS                           ║
║                                                                              ║
║  Version corrigée : reprend toutes les visualisations de la variante        ║
║  précédente (courbes de convergence, tracés des routes avant/après,        ║
║  graphique dédié gap % vs BKS, tableau récapitulatif véhicules/distance)   ║
║  MAIS applique désormais un vrai protocole ENTRAÎNEMENT / TEST DISJOINT,   ║
║  en un seul lancement, sans aucune intervention manuelle :                 ║
║                                                                              ║
║   ÉTAPE 1 (ENTRAÎNEMENT) — génère des instances SYNTHÉTIQUES (reprises de   ║
║      generate_instance() de main_vrptw_comparaison_gurobi.py / "Code 1")    ║
║      et entraîne le DQN dessus (poids mis à jour via store/learn/decay).    ║
║                                                                              ║
║   ÉTAPE 2 (TEST) — charge les 56 instances Solomon OFFICIELLES et évalue    ║
║      le même agent, GELÉ (aucun store/learn/decay_epsilon, uniquement      ║
║      select_action en pur mode inférence), sur ce jeu jamais vu à          ║
║      l'entraînement. Toutes les courbes/tracés/tableaux ci-dessous portent  ║
║      désormais sur ce TEST gelé, pas sur les résultats d'entraînement.     ║
║                                                                              ║
║  Situation 1 = LNS + DQN(entraîné sur synthétique) + Gurobi, ÉVALUÉ sur     ║
║                Solomon (test)                                → gap vs BKS   ║
║  (La "Situation 2 = Gurobi seul sur l'instance complète" reste retirée,     ║
║   à la demande, car sans limite de temps elle est beaucoup trop longue      ║
║   sur des instances à 100 clients avec fenêtres de temps serrées.)          ║
║                                                                              ║
║  Visualisations produites (toutes sur le TEST gelé, sauf la 1ère) :        ║
║    [A] Convergence de l'ENTRAÎNEMENT (synthétique) : reward global +       ║
║        amélioration % par instance synthétique                            ║
║    [B] Convergence détaillée (instances Solomon exemples, TEST) : coût     ║
║        final/meilleur par épisode local + trajectoire de coût             ║
║    [C] Tracés des routes AVANT (LNS) / APRÈS (DQN gelé) pour des           ║
║        instances Solomon exemples                                         ║
║    [D] Tableau récapitulatif : Instance / Nb véhicules / Distance / Gap %  ║
║        (TEST Solomon vs BKS)                                              ║
║    [E] Graphique dédié gap % vs BKS par instance + gap moyen              ║
║    [F] Diagnostic automatique de licence Gurobi (taille limitée ou non)   ║
║                                                                              ║
║  Aucun rappel manuel de "Code 1" n'est nécessaire : la génération +        ║
║  l'entraînement synthétique sont réintégrés ici (MODULE 1bis) et           ║
║  s'exécutent automatiquement avant le chargement de Solomon. Voir le flag  ║
║  optionnel CFG['REUSE_EXISTING_SYNTH_MODEL'] plus bas si vous préférez     ║
║  recharger un .pth déjà entraîné au lieu de ré-entraîner à chaque run.     ║
║                                                                              ║
║  Par défaut, INSTANCE_FILES contient toutes les instances *.txt trouvées   ║
║  dans outputs/Solomon/. Pour se limiter à 6 instances représentatives      ║
║  (une par groupe C1/C2/R1/R2/RC1/RC2), décommentez le bloc plus bas.       ║
║                                                                              ║
║  Exécution :  python main_vrptw_synth_train_solomon_convergence.py         ║
║                                                                              ║
║  CORRECTIONS APPORTÉES SUR CETTE VERSION :                                 ║
║   1. SOLOMON_DIR pointait vers un chemin Google Colab                      ║
║      ('/content/sample_data/Solomon') — corrigé pour pointer vers          ║
║      outputs/Solomon (dossier local Windows).                              ║
║   2. Le mode CFG['TAGUCHI_SYNTH']['ENABLE']=True s'arrêtait après la       ║
║      calibration et exigeait de relancer le script à la main pour la      ║
║      validation finale. Il enchaîne désormais AUTOMATIQUEMENT, dans le    ║
║      même lancement, sur l'entraînement complet (80 instances               ║
║      synthétiques) + le test gelé sur les 56 Solomon, en appliquant       ║
║      automatiquement la meilleure combinaison d'hyperparamètres trouvée.  ║
║      Voir run_final_validation() et le bloc __main__ tout en bas.         ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import os, sys, json, time, copy, random, csv, glob, itertools
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from contextlib import contextmanager
import torch
import torch.nn as nn
import torch.optim as optim
import gurobipy as gp
from gurobipy import GRB
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG GLOBALE
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    # ── ÉTAPE 1 : ENTRAÎNEMENT sur instances SYNTHÉTIQUES (jamais Solomon) ──
    N_SYNTH_INSTANCES          = 80,     # comme les 80 instances de Code 1
    SYNTH_N_CUSTOMERS          = 100,
    SYNTH_N_VEHICLES           = None,   # None → calculé depuis la demande (cf. generate_instance)
    SYNTH_VEHICLE_SAFETY_FACTOR= 1.5,
    SYNTH_CAPACITY             = 200,
    SYNTH_HORIZON              = 1000,
    EPISODES_PER_INSTANCE      = 20,     # épisodes d'ENTRAÎNEMENT par instance synthétique
    # Optionnel : True → recharge un .pth déjà entraîné sur du synthétique
    # (voir MODEL_FILE) au lieu de régénérer + ré-entraîner. False (par
    # défaut) → tout se fait automatiquement dans CE run, sans intervention
    # manuelle, aucun besoin de relancer "Code 1" séparément.
    # ⚠ Mis à True ici : on veut RÉUTILISER le modèle déjà entraîné (avec
    # les meilleurs hyperparamètres Taguchi) au lieu de relancer tout
    # l'entraînement — seul le TEST (plus d'épisodes) doit être rejoué.
    REUSE_EXISTING_SYNTH_MODEL = True,
    # True → applique automatiquement, au début de run_final_validation(),
    # la meilleure combinaison d'hyperparamètres déjà trouvée par le plan
    # Taguchi synthétique (fichier taguchi_meilleure_combinaison_synthetique.json,
    # généré par un run PRÉCÉDENT avec CFG['TAGUCHI_SYNTH']['ENABLE']=True).
    # Nécessaire ici car CFG['TAGUCHI_SYNTH']['ENABLE'] est repassé à False
    # plus bas (pas de nouvelle calibration) : sans ce flag, CFG utiliserait
    # les valeurs par défaut du haut du script, pas les valeurs calibrées.
    USE_TAGUCHI_BEST            = True,
    # ── ÉTAPE 2 : TEST gelé sur les 56 instances Solomon officielles ───────
    EPISODES_PER_INSTANCE_TEST = 10,     # épisodes d'ÉVALUATION (aucun apprentissage) par instance Solomon
    # VEHICLE_SAFETY_FACTOR = None : on utilise la flotte OFFICIELLE de chaque
    # instance, pas une borne calculée sur la demande. Sur R101/RC101
    # (fenêtres de temps serrées), le BKS utilise 19 et 14 véhicules alors
    # que la demande seule suggérerait ~12-14 : une borne basée uniquement
    # sur la demande sous-estime le besoin réel et peut rendre le MILP
    # infaisable. La flotte officielle est plus grande (MILP plus lourd)
    # mais garantit qu'une solution proche du BKS reste atteignable.
    VEHICLE_SAFETY_FACTOR  = None,
    LNS_ITER               = 30,
    LNS_K_RATIO            = 0.20,
    K_DESTROY_RATIO        = 0.24,
    MAX_STEPS               = 20,
    # repair par bouts (Situation 1, appelé à CHAQUE step du LNS/DQN — voir
    # check_gurobi_license() plus bas pour l'impact d'une licence à taille
    # limitée). None = aucune limite de temps (supporté par le code).
    GUROBI_TLIMIT            = 2.0,
    GUROBI_GAP               = 0.05,
    HIDDEN                   = 64,
    LR                       = 1e-3,
    GAMMA                    = 0.95,
    EPS_START                = 1.0,
    EPS_END                  = 0.05,
    EPS_DECAY                = 0.95,  # ⚠ ignoré — recalculé automatiquement plus bas (cf. commentaire)
    BUFFER_SIZE               = 5000,
    BATCH_SIZE                = 32,
    TARGET_UPDATE              = 10,
    # ── Instances exemples (pour courbes détaillées + routes, TEST/Solomon) ─
    # Noms recherchés en priorité (représentatifs des 6 groupes Solomon) ;
    # si un nom n'est pas présent dans INSTANCE_FILES, il est ignoré et
    # remplacé automatiquement par le 1er type disponible.
    EXAMPLE_NAMES              = ['C101', 'R101', 'RC101'],
    # ── Pénalité par véhicule (repair MILP) ────────────────────────
    # 0.0 = comportement d'origine (aucune pénalité, Gurobi peut ouvrir
    # librement des véhicules supplémentaires à chaque repair). Une valeur
    # > 0 ajoute un coût fixe par véhicule utilisé dans l'objectif du MILP
    # de repair, pour limiter l'inflation du nombre de véhicules observée
    # dans la validation vs BKS. C'est aussi l'un des facteurs calibrés par
    # le plan d'expérience Taguchi (MODULE 10 plus bas) — calibré
    # UNIQUEMENT sur du synthétique (CFG['TAGUCHI_SYNTH']), jamais sur
    # Solomon, pour ne pas faire fuiter d'information du test set vers le
    # choix des hyperparamètres.
    VEHICLE_PENALTY            = 0.0,
    OUTDIR                        = os.path.join(os.getcwd(), 'outputs'),
    MODELDIR                       = os.path.join(os.getcwd(), 'outputs', 'model'),
    MODEL_FILE                      = 'dqn_vrptw_synth_trained.pth',
)
CFG['LNS_K'] = 20   # ~20% de 100 clients, cohérent avec LNS_K_RATIO
CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))  # 24, cohérent avec K_DESTROY_RATIO
# ── Recalibrage automatique de EPS_DECAY ────────────────────────────────
# La valeur fixe 0.95 était calibrée pour un tout petit nombre d'épisodes
# (contexte d'origine : entraînement direct sur quelques dizaines
# d'instances Solomon). Avec EPS_DECAY=0.95, epsilon atteint son plancher
# (EPS_END) après seulement ~58 décroissances — soit, avec les valeurs par
# défaut (N_SYNTH_INSTANCES=80, EPISODES_PER_INSTANCE=10), après à peine
# ~6 des 80 instances synthétiques : plus de 90% de l'entraînement se
# ferait alors avec epsilon au plancher (quasi aucune exploration), quel
# que soit EPISODES_PER_INSTANCE — et AUGMENTER EPISODES_PER_INSTANCE
# aggrave le problème (le plancher est atteint en encore moins d'instances
# couvertes). On recalcule donc EPS_DECAY ici pour que epsilon décroisse
# de EPS_START à EPS_END sur TOUTE la durée prévue de l'entraînement
# (N_SYNTH_INSTANCES × EPISODES_PER_INSTANCE décroissances), quel que soit
# le nombre d'épisodes par instance choisi.
_n_decays_total = max(1, CFG['N_SYNTH_INSTANCES'] * CFG['EPISODES_PER_INSTANCE'])
CFG['EPS_DECAY'] = (CFG['EPS_END'] / CFG['EPS_START']) ** (1.0 / _n_decays_total)
os.makedirs(CFG['OUTDIR'],   exist_ok=True)
os.makedirs(CFG['MODELDIR'], exist_ok=True)
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG DU PLAN D'EXPÉRIENCE TAGUCHI (calibration des hyperparamètres)
# ══════════════════════════════════════════════════════════════════════════════
# Deux modes de calibration disponibles (voir plus bas pour le mode
# TAGUCHI_SYNTH, recommandé, qui enchaîne désormais automatiquement sur la
# validation finale). Activez explicitement l'UN DES DEUX (pas les deux en
# même temps) :
#
#  • CFG['TAGUCHI_SYNTH']['ENABLE'] = True   (RECOMMANDÉ)
#    Calibre en s'entraînant UNIQUEMENT sur des instances SYNTHÉTIQUES
#    générées à la volée (jamais Solomon). Métrique de classement :
#    amélioration % vs coût LNS initial (pas de BKS requis). La meilleure
#    combinaison trouvée n'a jamais "vu" les 56 instances Solomon → aucune
#    fuite d'hyperparamètres vers le jeu de test, contrairement au mode
#    historique ci-dessous. Une fois la calibration terminée, le script
#    enchaîne AUTOMATIQUEMENT sur la validation finale (entraînement 80
#    instances + test gelé sur les 56 Solomon) avec ces hyperparamètres.
#
#  • CFG['TAGUCHI']['ENABLE'] = True   (⚠ HISTORIQUE — NON RECOMMANDÉ)
#    Calibre en s'entraînant DIRECTEMENT sur les 56 instances Solomon (la
#    matrice externe = le jeu de test lui-même) et classe les combinaisons
#    par leur gap % vs BKS sur ces mêmes instances. C'est exactement le
#    protocole "Run 16" identifié comme fuite d'hyperparamètres (on choisit
#    les hyperparamètres en fonction de la performance sur le test) —
#    conservé uniquement pour comparaison / compatibilité avec d'anciens
#    résultats, PAS pour produire les hyperparamètres d'une validation
#    finale train/test disjointe. Ce mode n'enchaîne PAS automatiquement
#    sur la validation finale (voir le bloc __main__ tout en bas).
CFG['TAGUCHI'] = dict(
    ENABLE                     = False,  # ⚠ historique — fuite vers Solomon (test set), voir ci-dessus
    EPISODES_PER_INSTANCE_DOE  = 5,
    OUTER_INSTANCE_NAMES       = 'ALL',  # 'ALL' = les 56 instances Solomon comme matrice externe
)
CFG['TAGUCHI_SYNTH'] = dict(
    # False ici : la calibration a déjà été faite lors d'un run précédent
    # (le JSON taguchi_meilleure_combinaison_synthetique.json existe déjà).
    # Repasser à True relancerait tout le plan Taguchi (18 runs) pour rien —
    # CFG['USE_TAGUCHI_BEST']=True ci-dessus suffit à réutiliser ses résultats.
    ENABLE                     = False,  # True = lance le plan Taguchi calibré sur synthétique (recommandé)
    # Volume d'instances synthétiques utilisé comme matrice externe pendant
    # CE plan de calibration (indépendant de N_SYNTH_INSTANCES du protocole
    # d'entraînement final — un sous-ensemble suffit ici, l'objectif est de
    # classer les 18 combinaisons, pas d'entraîner le modèle définitif).
    N_INSTANCES_DOE            = 24,   # 8 par type C/R/RC
    EPISODES_PER_INSTANCE_DOE  = 5,
)
# Facteurs de contrôle et leurs niveaux (plan Taguchi L18 : 7 facteurs à 3
# niveaux, 18 combinaisons au lieu des 3^7 = 2187 combinaisons possibles).
TAGUCHI_FACTORS = [
    ('LNS_ITER',         [30, 60, 100]),
    ('K_DESTROY_RATIO',  [0.15, 0.20, 0.24]),
    ('GUROBI_TLIMIT',    [1.0, 2.0, 3.0]),
    ('GUROBI_GAP',       [0.05, 0.03, 0.01]),
    ('MAX_STEPS',        [10, 15, 20]),
    ('EPS_DECAY',        [0.90, 0.95, 0.99]),
    ('VEHICLE_PENALTY',  [0.0, 300.0, 800.0]),
]
GROUP_ORDER = ['C1', 'C2', 'R1', 'R2', 'RC1', 'RC2']
ACTION_NAMES = {
    0: "Destruction spatiale",
    1: "Destruction temporelle",
    2: "Destruction par coût élevé",
    3: "Destruction aléatoire",
}
# ── Dossier contenant les instances Solomon officielles (extraites du zip) ──
# CORRIGÉ : pointait vers '/content/sample_data/Solomon' (chemin Google
# Colab, inexistant sur une machine locale) — repointé vers outputs/Solomon.
SOLOMON_DIR = os.path.join(CFG['OUTDIR'], 'Solomon')
# Toutes les instances *.txt trouvées (décommentez le bloc ci-dessous pour
# revenir à seulement 6 instances représentatives, une par groupe
# C1/C2/R1/R2/RC1/RC2, afin de garder un temps d'exécution raisonnable).
INSTANCE_FILES = sorted(glob.glob(os.path.join(SOLOMON_DIR, '*.txt')))
# INSTANCE_FILES = [
#     os.path.join(SOLOMON_DIR, 'c101.txt'),
#     os.path.join(SOLOMON_DIR, 'c201.txt'),
#     os.path.join(SOLOMON_DIR, 'r101.txt'),
#     os.path.join(SOLOMON_DIR, 'r201.txt'),
#     os.path.join(SOLOMON_DIR, 'rc101.txt'),
#     os.path.join(SOLOMON_DIR, 'rc201.txt'),
# ]
# ── BKS officiel complet, 56 instances (source : SINTEF / Solomon 1987) ──────
BKS_REFERENCE = {
    'C101': {'vehicles': 10, 'distance': 828.94}, 'C102': {'vehicles': 10, 'distance': 828.94},
    'C103': {'vehicles': 10, 'distance': 828.06}, 'C104': {'vehicles': 10, 'distance': 824.78},
    'C105': {'vehicles': 10, 'distance': 828.94}, 'C106': {'vehicles': 10, 'distance': 828.94},
    'C107': {'vehicles': 10, 'distance': 828.94}, 'C108': {'vehicles': 10, 'distance': 828.94},
    'C109': {'vehicles': 10, 'distance': 828.94},
    'C201': {'vehicles': 3, 'distance': 591.56}, 'C202': {'vehicles': 3, 'distance': 591.56},
    'C203': {'vehicles': 3, 'distance': 591.17}, 'C204': {'vehicles': 3, 'distance': 590.60},
    'C205': {'vehicles': 3, 'distance': 588.88}, 'C206': {'vehicles': 3, 'distance': 588.49},
    'C207': {'vehicles': 3, 'distance': 588.29}, 'C208': {'vehicles': 3, 'distance': 588.32},
    'R101': {'vehicles': 19, 'distance': 1650.80}, 'R102': {'vehicles': 17, 'distance': 1486.12},
    'R103': {'vehicles': 13, 'distance': 1292.68}, 'R104': {'vehicles': 9,  'distance': 1007.31},
    'R105': {'vehicles': 14, 'distance': 1377.11}, 'R106': {'vehicles': 12, 'distance': 1252.03},
    'R107': {'vehicles': 10, 'distance': 1104.66}, 'R108': {'vehicles': 9,  'distance': 960.88},
    'R109': {'vehicles': 11, 'distance': 1194.73}, 'R110': {'vehicles': 10, 'distance': 1118.84},
    'R111': {'vehicles': 10, 'distance': 1096.72}, 'R112': {'vehicles': 9,  'distance': 982.14},
    'R201': {'vehicles': 4, 'distance': 1252.37}, 'R202': {'vehicles': 3, 'distance': 1191.70},
    'R203': {'vehicles': 3, 'distance': 939.50},  'R204': {'vehicles': 2, 'distance': 825.52},
    'R205': {'vehicles': 3, 'distance': 994.43},  'R206': {'vehicles': 3, 'distance': 906.14},
    'R207': {'vehicles': 2, 'distance': 890.61},  'R208': {'vehicles': 2, 'distance': 726.82},
    'R209': {'vehicles': 3, 'distance': 909.16},  'R210': {'vehicles': 3, 'distance': 939.37},
    'R211': {'vehicles': 2, 'distance': 885.71},
    'RC101': {'vehicles': 14, 'distance': 1696.95}, 'RC102': {'vehicles': 12, 'distance': 1554.75},
    'RC103': {'vehicles': 11, 'distance': 1261.67}, 'RC104': {'vehicles': 10, 'distance': 1135.48},
    'RC105': {'vehicles': 13, 'distance': 1629.44}, 'RC106': {'vehicles': 11, 'distance': 1424.73},
    'RC107': {'vehicles': 11, 'distance': 1230.48}, 'RC108': {'vehicles': 10, 'distance': 1139.82},
    'RC201': {'vehicles': 4, 'distance': 1406.94}, 'RC202': {'vehicles': 3, 'distance': 1365.65},
    'RC203': {'vehicles': 3, 'distance': 1049.62}, 'RC204': {'vehicles': 3, 'distance': 798.46},
    'RC205': {'vehicles': 4, 'distance': 1297.65}, 'RC206': {'vehicles': 3, 'distance': 1146.32},
    'RC207': {'vehicles': 3, 'distance': 1061.14}, 'RC208': {'vehicles': 3, 'distance': 828.14},
}
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1  —  LECTURE DES INSTANCES SOLOMON OFFICIELLES  (jeu de TEST)
# ══════════════════════════════════════════════════════════════════════════════
def euclidean(a, b):
    return float(np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2))
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1bis  —  GÉNÉRATEUR D'INSTANCES SYNTHÉTIQUES  (jeu d'ENTRAÎNEMENT)
#  Repris tel quel de main_vrptw_comparaison_gurobi.py ("Code 1") : mêmes
#  fonctions _coords_C/_coords_R/_coords_RC et generate_instance(), pour que
#  les instances d'entraînement soient générées exactement comme dans Code 1.
# ══════════════════════════════════════════════════════════════════════════════
def _coords_C(n, rng):
    n_cl = max(3, n // 10)
    centers = rng.uniform(15, 85, (n_cl, 2))
    pts = []
    for i in range(n):
        p = rng.normal(centers[i % n_cl], 8)
        pts.append(np.clip(p, 1, 99))
    return np.array(pts)
def _coords_R(n, rng):
    return rng.uniform(1, 99, (n, 2))
def _coords_RC(n, rng):
    h = n // 2
    return np.vstack([_coords_C(h, rng), _coords_R(n - h, rng)])
def generate_instance(n_customers=50, n_vehicles=None, capacity=200,
                      type_='C', seed=42, horizon=1000,
                      vehicle_safety_factor=1.5):
    rng     = np.random.RandomState(seed)
    depot   = np.array([50.0, 50.0])
    gen_fn  = {'C': _coords_C, 'R': _coords_R, 'RC': _coords_RC}[type_]
    coords  = gen_fn(n_customers, rng)
    max_dem = max(1, capacity // 5)
    demands = rng.randint(1, max_dem + 1, n_customers)
    svc     = rng.randint(10, 31, n_customers)
    if n_vehicles is None:
        min_vehicles = int(np.ceil(demands.sum() / capacity))
        n_vehicles   = max(2, int(np.ceil(min_vehicles * vehicle_safety_factor)))
    customers = []
    for i in range(n_customers):
        d        = euclidean(depot, coords[i])
        earliest = max(0.0, round(d + rng.uniform(-15, 15), 1))
        delta    = rng.uniform(30, 80)
        latest   = min(float(horizon), round(earliest + delta, 1))
        latest   = max(latest, earliest + svc[i] + 5)
        latest   = min(latest, float(horizon))
        if d > latest:
            earliest = round(d, 1)
            latest   = min(float(horizon), round(earliest + delta, 1))
            latest   = max(latest, earliest + svc[i] + 5)
        if d + svc[i] + d > horizon:
            svc[i] = max(5, int(horizon - 2 * d - 5))
        customers.append({
            'id':           i + 1,
            'x':            float(round(coords[i][0], 2)),
            'y':            float(round(coords[i][1], 2)),
            'demand':       int(demands[i]),
            'earliest':     float(earliest),
            'latest':       float(latest),
            'service_time': int(svc[i]),
        })
    return {
        'depot': {
            'id': 0, 'x': 50.0, 'y': 50.0, 'demand': 0,
            'earliest': 0.0, 'latest': float(horizon), 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    capacity,
        'horizon':     horizon,
        'type':        type_,
        'seed':        seed,
        'n_customers': n_customers,
    }
def generate_synthetic_training_set(cfg, verbose=True):
    """
    Génère les instances synthétiques d'ENTRAÎNEMENT en cyclant les 3 types
    Solomon C / R / RC (même logique que train() dans Code 1 : type =
    ['C','R','RC'][i % 3], seed = i*7+1). Ces instances ne recouvrent JAMAIS
    les 56 instances Solomon officielles utilisées plus bas comme jeu de
    TEST : le DQN ne les voit donc jamais avant l'évaluation gelée.
    Chaque instance reçoit en plus une clé 'name' (ex. 'SYN-C001') pour
    rester compatible avec les fonctions d'entraînement/affichage qui
    utilisent inst['name'].
    """
    types = ['C', 'R', 'RC']
    instances = []
    if verbose:
        _sec(f"ÉTAPE 1 — GÉNÉRATION DE {cfg['N_SYNTH_INSTANCES']} INSTANCES "
             f"SYNTHÉTIQUES (ENTRAÎNEMENT)")
    for i in range(cfg['N_SYNTH_INSTANCES']):
        t    = types[i % 3]
        seed = i * 7 + 1
        inst = generate_instance(
            n_customers = cfg['SYNTH_N_CUSTOMERS'],
            n_vehicles  = cfg['SYNTH_N_VEHICLES'],
            capacity    = cfg['SYNTH_CAPACITY'],
            horizon     = cfg['SYNTH_HORIZON'],
            type_       = t,
            seed        = seed,
            vehicle_safety_factor = cfg['SYNTH_VEHICLE_SAFETY_FACTOR'],
        )
        inst['name'] = f"SYN-{t}{i+1:03d}"
        instances.append(inst)
        if verbose:
            print(f"  Instance {i+1:03d}/{cfg['N_SYNTH_INSTANCES']} | {inst['name']:<10} "
                  f"type={t:<3} | seed={seed} | véhicules={inst['n_vehicles']}")
    return instances
def parse_solomon_instance(filepath, vehicle_safety_factor=None):
    """
    Parse un fichier d'instance Solomon au format texte officiel :
        <NOM>
        VEHICLE
        NUMBER  CAPACITY
        <n>     <cap>
        CUSTOMER
        CUST NO.  XCOORD.  YCOORD.  DEMAND  READY TIME  DUE DATE  SERVICE TIME
        0  ...   (dépôt)
        1  ...
        ...
    Retourne un dict compatible avec le reste du pipeline.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = [l.rstrip('\n') for l in f]
    name = lines[0].strip()
    idx_veh = next(i for i, l in enumerate(lines) if l.strip() == 'VEHICLE')
    veh_number, veh_capacity = map(int, lines[idx_veh + 2].split())
    idx_cust = next(i for i, l in enumerate(lines) if l.strip() == 'CUSTOMER')
    data_lines = [l for l in lines[idx_cust + 2:] if l.strip()]
    rows = [list(map(float, l.split())) for l in data_lines]
    depot_row = rows[0]
    customers = []
    for r in rows[1:]:
        cid, x, y, dem, ready, due, svc = r
        customers.append({
            'id':           int(cid),
            'x':            x,
            'y':            y,
            'demand':       int(dem),
            'earliest':     ready,
            'latest':       due,
            'service_time': int(svc),
        })
    name_upper = name.upper()
    type_ = ''.join(ch for ch in name_upper if ch.isalpha())   # 'C101' -> 'C', 'RC101' -> 'RC'
    n_vehicles = veh_number
    if vehicle_safety_factor is not None:
        total_demand = sum(c['demand'] for c in customers)
        min_veh      = int(np.ceil(total_demand / veh_capacity))
        n_vehicles   = min(veh_number, max(2, int(np.ceil(min_veh * vehicle_safety_factor))))
    return {
        'depot': {
            'id': 0, 'x': depot_row[1], 'y': depot_row[2], 'demand': 0,
            'earliest': depot_row[4], 'latest': depot_row[5], 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    veh_capacity,
        'horizon':     depot_row[5],
        'type':        type_,
        'seed':        name_upper,
        'n_customers': len(customers),
        'name':        name_upper,          # nom officiel normalisé (ex. 'C101')
    }
def load_all_instances(files, vehicle_safety_factor=None):
    if not files:
        raise FileNotFoundError(
            f"Aucun fichier d'instance trouvé dans {SOLOMON_DIR}. "
            f"Vérifiez que le dossier existe et contient les .txt Solomon."
        )
    instances = []
    for fp in files:
        inst = parse_solomon_instance(fp, vehicle_safety_factor=vehicle_safety_factor)
        instances.append(inst)
        print(f"  Chargé : {inst['name']:<8} type={inst['type']:<3} "
              f"clients={inst['n_customers']} véhicules={inst['n_vehicles']} "
              f"capacité={inst['capacity']} horizon={inst['horizon']}")
    return instances
def pick_example_indices(instances, preferred_names=None, max_examples=3):
    """
    Sélectionne des instances "exemples" pour les courbes détaillées et les
    tracés de routes. Cherche d'abord les noms préférés (ex. C101/R101/RC101).
    Si absents, retombe sur la 1ère instance de chaque type détecté, puis sur
    les toutes premières instances de la liste.
    """
    preferred_names = preferred_names or []
    by_name = {inst['name']: i for i, inst in enumerate(instances)}
    idxs = [by_name[nm] for nm in preferred_names if nm in by_name]
    if not idxs:
        seen_types = {}
        for i, inst in enumerate(instances):
            if inst['type'] not in seen_types:
                seen_types[inst['type']] = i
            if len(seen_types) >= max_examples:
                break
        idxs = list(seen_types.values())
    if not idxs:
        idxs = list(range(min(max_examples, len(instances))))
    return idxs[:max_examples]
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 2  —  SOLUTION INITIALE PAR LNS  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
def _node_xy(nid, instance):
    if nid == 0:
        return instance['depot']['x'], instance['depot']['y']
    c = instance['customers'][nid - 1]
    return c['x'], c['y']
def _node_info(nid, instance):
    if nid == 0:
        return instance['depot']
    return instance['customers'][nid - 1]
def compute_cost(solution, instance):
    depot = (instance['depot']['x'], instance['depot']['y'])
    total = 0.0
    for route in solution:
        if not route:
            continue
        prev = depot
        for cid in route:
            cur   = _node_xy(cid, instance)
            total += euclidean(prev, cur)
            prev  = cur
        total += euclidean(prev, depot)
    return total
def _route_feasible(route, instance):
    if not route:
        return True
    cap  = instance['capacity']
    load = sum(_node_info(c, instance)['demand'] for c in route)
    if load > cap:
        return False
    depot = instance['depot']
    t     = 0.0
    px, py = depot['x'], depot['y']
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        if t > info['latest']:
            return False
        t      = max(t, info['earliest']) + info['service_time']
        px, py = cx, cy
    t += euclidean((px, py), (depot['x'], depot['y']))
    return t <= depot['latest']
def is_feasible(solution, instance):
    return all(_route_feasible(r, instance) for r in solution)
def solution_stats(solution, instance):
    cap        = instance['capacity']
    total_cost = compute_cost(solution, instance)
    n_vehicles = sum(1 for r in solution if r)
    loads      = []
    total_wait = 0.0
    for route in solution:
        if not route:
            continue
        load = sum(_node_info(c, instance)['demand'] for c in route)
        loads.append(load / cap)
        depot  = instance['depot']
        t      = 0.0
        px, py = depot['x'], depot['y']
        for cid in route:
            cx, cy = _node_xy(cid, instance)
            t     += euclidean((px, py), (cx, cy))
            info   = _node_info(cid, instance)
            total_wait += max(0.0, info['earliest'] - t)
            t      = max(t, info['earliest']) + info['service_time']
            px, py = cx, cy
    return {
        'total_cost':         round(total_cost, 2),
        'n_vehicles':         n_vehicles,
        'avg_load_ratio':     round(np.mean(loads) if loads else 0.0, 3),
        'total_wait_time':    round(total_wait, 2),
        'n_customers_served': sum(len(r) for r in solution),
    }
def _insertion_cost(cid, pos, route, instance):
    depot   = (instance['depot']['x'], instance['depot']['y'])
    nodes   = [0] + route + [0]
    prev_id = nodes[pos]
    next_id = nodes[pos + 1]
    px, py  = _node_xy(prev_id, instance) if prev_id != 0 else depot
    nx, ny  = _node_xy(next_id, instance) if next_id != 0 else depot
    cx, cy  = _node_xy(cid, instance)
    return (euclidean((px, py), (cx, cy)) + euclidean((cx, cy), (nx, ny))
            - euclidean((px, py), (nx, ny)))
def _try_insert(cid, route, instance):
    best_cost, best_pos = float('inf'), -1
    for pos in range(len(route) + 1):
        candidate = route[:pos] + [cid] + route[pos:]
        if _route_feasible(candidate, instance):
            cost = _insertion_cost(cid, pos, route, instance)
            if cost < best_cost:
                best_cost, best_pos = cost, pos
    return best_cost, best_pos
def _repair_greedy(removed, partial, instance):
    routes = [list(r) for r in partial]
    for cid in removed:
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _build_initial(instance):
    customers = sorted(instance['customers'], key=lambda c: c['earliest'])
    routes    = []
    for c in customers:
        cid = c['id']
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _lns_improve(solution, instance, n_iter=50, k=10, seed=0):
    rng       = np.random.RandomState(seed)
    current   = [list(r) for r in solution]
    best      = copy.deepcopy(current)
    best_cost = compute_cost(best, instance)
    for _ in range(n_iter):
        all_c   = [c for r in current for c in r]
        k_      = min(k, max(1, len(all_c) - 1))
        removed = list(rng.choice(all_c, k_, replace=False))
        removed_set = set(removed)
        partial = [[c for c in r if c not in removed_set] for r in current]
        partial = [r for r in partial if r]
        candidate = _repair_greedy(removed, partial, instance)
        c_cost    = compute_cost(candidate, instance)
        if c_cost < best_cost and is_feasible(candidate, instance):
            best, best_cost = copy.deepcopy(candidate), c_cost
        current = candidate
    return best
def get_initial_solution(instance, lns_iter=50, lns_k=10):
    sol = _build_initial(instance)
    sol = _lns_improve(sol, instance, n_iter=lns_iter, k=lns_k)
    sol = [r for r in sol if r]
    return sol, solution_stats(sol, instance)
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 3  —  ENVIRONNEMENT VRPTW  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class VRPTWEnv:
    def __init__(self, instance, initial_solution,
                 K=15, max_steps=20,
                 gurobi_tlimit=2.0, gurobi_gap=0.05,
                 vehicle_penalty=0.0,
                 verbose=False):
        self.instance    = instance
        self.initial_sol = copy.deepcopy(initial_solution)
        self.K           = K
        self.max_steps   = max_steps
        self.tlimit      = gurobi_tlimit
        self.gap         = gurobi_gap
        self.vehicle_penalty = vehicle_penalty
        self.verbose     = verbose
        self.obs_dim     = 7
        self.n_actions   = 4
        self._dist = {}
        depot = (instance['depot']['x'], instance['depot']['y'])
        for c in instance['customers']:
            d = euclidean(depot, (c['x'], c['y']))
            self._dist[(0, c['id'])] = d
            self._dist[(c['id'], 0)] = d
        for ci in instance['customers']:
            for cj in instance['customers']:
                self._dist[(ci['id'], cj['id'])] = euclidean(
                    (ci['x'], ci['y']), (cj['x'], cj['y']))
        self.reset()
    def _d(self, a, b):
        return self._dist.get((a, b), 0.0)
    def _info(self, nid):
        return _node_info(nid, self.instance)
    def reset(self, start_solution=None):
        if start_solution is not None:
            self.solution = copy.deepcopy(start_solution)
        else:
            self.solution = copy.deepcopy(self.initial_sol)
        self.current_cost     = compute_cost(self.solution, self.instance)
        self.best_cost        = self.current_cost
        self.initial_cost     = compute_cost(self.initial_sol, self.instance)
        self.step_count       = 0
        self.steps_no_improve = 0
        self._action_history  = []
        return self._obs()
    def _obs(self):
        stats = solution_stats(self.solution, self.instance)
        ic    = max(self.initial_cost, 1.0)
        return np.array([
            self.current_cost / ic,
            stats['n_vehicles'] / max(self.instance['n_vehicles'], 1),
            stats['avg_load_ratio'],
            stats['total_wait_time'] / max(ic * 0.5, 1.0),
            self.steps_no_improve / self.max_steps,
            self.best_cost / ic,
            1.0,
        ], dtype=np.float32)
    def _destroy(self, action):
        all_c = [c for r in self.solution for c in r]
        if not all_c:
            return [], copy.deepcopy(self.solution)
        K   = min(self.K, len(all_c))
        rng = np.random.RandomState(self.step_count + action * 100)
        if action == 0:
            pivot   = int(rng.choice(all_c))
            px, py  = self._info(pivot)['x'], self._info(pivot)['y']
            dists   = {c: euclidean((px, py),
                                    (self._info(c)['x'], self._info(c)['y']))
                       for c in all_c}
            destroyed = sorted(all_c, key=lambda c: dists[c])[:K]
        elif action == 1:
            pivot    = int(rng.choice(all_c))
            e_pivot  = self._info(pivot)['earliest']
            tdiffs   = {c: abs(self._info(c)['earliest'] - e_pivot) for c in all_c}
            destroyed = sorted(all_c, key=lambda c: tdiffs[c])[:K]
        elif action == 2:
            savings = {}
            for r in self.solution:
                if not r:
                    continue
                nodes = [0] + r + [0]
                for i, cid in enumerate(r):
                    prev, nxt     = nodes[i], nodes[i + 2]
                    savings[cid]  = (self._d(prev, cid) + self._d(cid, nxt)
                                     - self._d(prev, nxt))
            destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
        else:
            idx       = rng.choice(len(all_c), K, replace=False)
            destroyed = [all_c[i] for i in idx]
        destroyed_set = set(destroyed)
        partial = [[c for c in r if c not in destroyed_set]
                   for r in self.solution]
        partial = [r for r in partial if r]
        return list(destroyed), partial
    def _repair_gurobi(self, destroyed, partial):
        if not destroyed:
            return copy.deepcopy(self.solution)
        inst    = self.instance
        depot   = inst['depot']
        horizon = inst['horizon']
        cap     = inst['capacity']
        nodes   = [0] + destroyed
        N       = len(nodes)
        V       = min(len(destroyed), inst['n_vehicles'])
        env, m = None, None
        try:
            env = gp.Env(empty=True)
            env.setParam('OutputFlag', 0)
            env.start()
            m = gp.Model(env=env)
            if self.tlimit is not None:
                m.setParam('TimeLimit', self.tlimit)
            m.setParam('MIPGap',     self.gap)
            m.setParam('OutputFlag', 0)
            x = m.addVars(N, N, V, vtype=GRB.BINARY, name='x')
            t = m.addVars(N, V, lb=0, ub=horizon,    name='t')
            m.setObjective(
                gp.quicksum(
                    self._d(nodes[i], nodes[j]) * x[i, j, v]
                    for i in range(N) for j in range(N) for v in range(V)
                    if i != j
                )
                # Pénalité par véhicule ouvert (0.0 = comportement d'origine,
                # inchangé). Un coefficient > 0 décourage Gurobi d'ouvrir un
                # véhicule de plus que nécessaire lors du repair, ce qui
                # limite l'inflation du nombre de véhicules par rapport au
                # BKS observée dans la validation.
                + self.vehicle_penalty * gp.quicksum(
                    x[0, j, v] for j in range(1, N) for v in range(V)
                ),
                GRB.MINIMIZE
            )
            for i in range(1, N):
                m.addConstr(
                    gp.quicksum(x[i, j, v]
                                for j in range(N) for v in range(V)
                                if i != j) == 1
                )
            for v in range(V):
                for i in range(1, N):
                    m.addConstr(
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j) ==
                        gp.quicksum(x[j, i, v] for j in range(N) if i != j)
                    )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(x[0, j, v] for j in range(1, N)) <= 1
                )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(
                        inst['customers'][nodes[i] - 1]['demand'] *
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j)
                        for i in range(1, N)
                    ) <= cap
                )
            M = horizon + 1000
            for v in range(V):
                m.addConstr(t[0, v] == 0)
                for i in range(N):
                    ni  = nodes[i]
                    ei  = depot['earliest'] if ni == 0 else inst['customers'][ni-1]['earliest']
                    li  = depot['latest']   if ni == 0 else inst['customers'][ni-1]['latest']
                    sti = depot['service_time'] if ni == 0 else inst['customers'][ni-1]['service_time']
                    m.addConstr(t[i, v] >= ei)
                    m.addConstr(t[i, v] <= li)
                    for j in range(N):
                        if i == j:
                            continue
                        m.addConstr(
                            t[j, v] >= t[i, v] + sti + self._d(ni, nodes[j])
                            - M * (1 - x[i, j, v])
                        )
            m.optimize()
            if m.SolCount > 0:
                eps        = 0.5
                new_routes = []
                for v in range(V):
                    if sum(x[0, j, v].X for j in range(1, N)) < eps:
                        continue
                    route_v, cur = [], 0
                    for _ in range(N):
                        nxt = next((j for j in range(N)
                                    if j != cur and x[cur, j, v].X > eps), None)
                        if nxt is None or nxt == 0:
                            break
                        route_v.append(nodes[nxt])
                        cur = nxt
                    if route_v:
                        new_routes.append(route_v)
                full = [r for r in list(partial) + new_routes if r]
                return full
            else:
                raise ValueError("Gurobi : aucune solution trouvée")
        except Exception as e:
            if self.verbose:
                print(f"    [Gurobi→Greedy fallback] {e}")
            return _repair_greedy(destroyed, partial, self.instance)
        finally:
            if m is not None:
                m.dispose()
            if env is not None:
                env.dispose()
    def step(self, action):
        old_cost            = self.current_cost
        destroyed, partial  = self._destroy(action)
        new_sol             = self._repair_gurobi(destroyed, partial)
        new_cost            = compute_cost(new_sol, self.instance)
        feasible            = is_feasible(new_sol, self.instance)
        if feasible:
            self.solution     = new_sol
            self.current_cost = new_cost
        else:
            new_cost = old_cost * 1.001
        improvement = old_cost - self.current_cost
        if improvement > 0:
            self.best_cost        = min(self.best_cost, self.current_cost)
            self.steps_no_improve = 0
        else:
            self.steps_no_improve += 1
        reward = improvement / max(self.initial_cost, 1.0)
        self.step_count += 1
        self._action_history.append(action)
        done = (self.step_count >= self.max_steps)
        info = {
            'cost':          self.current_cost,
            'best_cost':     self.best_cost,
            'action_taken':  action,
            'action_name':   ACTION_NAMES[action],
            'improvement':   improvement,
            'destroyed_k':   len(destroyed),
            'destroyed_ids': [int(d) for d in destroyed[:5]],
            'feasible':      feasible,
        }
        return self._obs(), float(reward), done, info
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 4  —  AGENT DQN  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((
            np.array(s,  dtype=np.float32), int(a), float(r),
            np.array(s2, dtype=np.float32), float(done),
        ))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.FloatTensor(np.stack(s)), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(np.stack(s2)),
                torch.FloatTensor(d))
    def __len__(self):
        return len(self.buf)
class DQNNet(nn.Module):
    def __init__(self, obs_dim=7, n_actions=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim,  hidden),  nn.ReLU(),
            nn.Linear(hidden,   hidden),  nn.ReLU(),
            nn.Linear(hidden,   n_actions),
        )
    def forward(self, x):
        return self.net(x)
class DQNAgent:
    def __init__(self, obs_dim=7, n_actions=4, hidden=64,
                 lr=1e-3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_size=2000, batch_size=32,
                 target_update_freq=10, seed=0):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        self.n_actions          = n_actions
        self.gamma              = gamma
        self.epsilon            = epsilon_start
        self.epsilon_end        = epsilon_end
        self.epsilon_decay      = epsilon_decay
        self.batch_size         = batch_size
        self.target_update_freq = target_update_freq
        self.step_count         = 0
        self.q_net      = DQNNet(obs_dim, n_actions, hidden)
        self.target_net = DQNNet(obs_dim, n_actions, hidden)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)
        self.loss_fn   = nn.MSELoss()
    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state).unsqueeze(0))
            return int(q.argmax().item())
    def store(self, s, a, r, s2, done):
        self.buffer.push(s, a, r, s2, done)
    def learn(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        s, a, r, s2, done = self.buffer.sample(self.batch_size)
        q_vals = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q  = self.target_net(s2).max(1)[0]
            targets = r + self.gamma * next_q * (1 - done)
        loss = self.loss_fn(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.optimizer.step()
        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.update_target()
        return float(loss.item())
    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    def save(self, path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        torch.save({'q_net': self.q_net.state_dict(),
                    'target_net': self.target_net.state_dict(),
                    'epsilon': self.epsilon, 'step_count': self.step_count}, path)
        print(f"    [DQN] Modèle sauvegardé → {path}")
    def load(self, path):
        """
        Recharge un checkpoint sauvegardé par save() (mêmes clés). Utilisé
        uniquement si CFG['REUSE_EXISTING_SYNTH_MODEL']=True, pour réutiliser
        un modèle déjà entraîné sur du synthétique au lieu de ré-entraîner.
        """
        ckpt = torch.load(path, map_location='cpu')
        self.q_net.load_state_dict(ckpt['q_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.epsilon    = ckpt.get('epsilon', self.epsilon)
        self.step_count = ckpt.get('step_count', self.step_count)
        print(f"    [DQN] Modèle rechargé ← {path}")
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5  —  ENTRAÎNEMENT DU DQN  (ÉTAPE 1, sur les instances SYNTHÉTIQUES)
#  Stocke cost_trace (trajectoire de coût step par step) dans le log, utilisé
#  ensuite pour les courbes de convergence détaillées. Retourne aussi
#  initial_sols pour les tracés de routes avant/après.
# ══════════════════════════════════════════════════════════════════════════════
BANNER = "═" * 72
def _sec(title):
    print(f"\n{BANNER}\n  {title}\n{BANNER}")
def train_on_instances(instances, episodes_per_instance=10, save_model=True, verbose=True):
    """
    ENTRAÎNEMENT (les poids du DQN sont mis à jour ici via store()/learn()/
    decay_epsilon()). Générique : fonctionne sur n'importe quelle liste
    d'instances au format standard — dans ce script, appelé avec les
    instances SYNTHÉTIQUES produites par generate_synthetic_training_set(),
    JAMAIS avec les instances Solomon (réservées à evaluate_frozen_
    on_instances(), plus bas, pour le TEST).
    save_model=False et verbose=False sont utilisés par le plan Taguchi
    (MODULE 10bis, run interne allégé : pas besoin d'écraser le modèle
    final ni d'un log console détaillé à chaque combinaison testée).
    Retourne : agent, training_log, initial_costs, best_costs, best_solutions,
               instance_times, initial_sols
    """
    n = len(instances)
    if verbose:
        _sec(f"ÉTAPE 1 — LNS initial pour les {n} instances d'entraînement (synthétiques)")
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        if verbose:
            print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
                  f"| véhicules={stats['n_vehicles']}")
    if verbose:
        _sec("ÉTAPE 1 — Entraînement DQN sur le synthétique")
    agent = DQNAgent(
        obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
        gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
        epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
        buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
        target_update_freq=CFG['TARGET_UPDATE'],
    )
    training_log   = []
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        if verbose:
            print(f"\n  {'═'*50}\n  [TRAIN] INSTANCE {inst['name']}  type={inst['type']}"
                  f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)
                next_obs, reward, done, info = env.step(action)
                agent.store(obs, action, reward, next_obs, float(done))
                agent.learn()
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            agent.decay_epsilon()
            training_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            if verbose:
                imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
                print(f"  [TRAIN] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                      f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        if verbose:
            print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    if save_model:
        model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
        agent.save(model_path)
    return agent, training_log, initial_costs, best_costs, best_solutions, instance_times, initial_sols
def evaluate_frozen_on_instances(instances, agent, episodes_per_instance=3):
    """
    TEST — ÉVALUATION GELÉE : l'agent DQN déjà entraîné (ÉTAPE 1, sur le
    synthétique) est ici appliqué en pur mode INFÉRENCE sur les instances
    passées en argument (les 56 Solomon officielles). Différences avec
    train_on_instances() :
      - AUCUN appel à agent.store() / agent.learn() / agent.decay_epsilon() :
        les poids du réseau ne sont JAMAIS mis à jour ici.
      - epsilon est temporairement forcé à 0.0 (politique gloutonne, pure
        exploitation du Q-réseau appris), puis restauré à sa valeur
        d'origine à la fin.
      - Le modèle n'est PAS sauvegardé ici (rien n'a changé dans ses poids).
    Le log retourné (test_log) a exactement la même forme que training_log
    (y compris 'cost_trace'), pour pouvoir réutiliser telles quelles les
    fonctions de tracé (plot_instance_convergence_examples, etc.) sur le
    TEST plutôt que sur l'entraînement.
    Retourne : initial_costs, best_costs, best_solutions, instance_times,
               initial_sols, test_log
    """
    n = len(instances)
    _sec(f"ÉTAPE 2 — ÉVALUATION GELÉE (TEST) sur {n} instances Solomon "
         f"— aucun apprentissage, epsilon=0")
    saved_epsilon = agent.epsilon
    agent.epsilon = 0.0
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    test_log       = []
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TEST] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)          # inférence pure (epsilon=0)
                next_obs, reward, done, info = env.step(action)
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            test_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TEST] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    agent.epsilon = saved_epsilon
    return initial_costs, best_costs, best_solutions, instance_times, initial_sols, test_log
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6  —  TABLEAU FINAL : SITUATION 1 vs BKS
#  (la Situation 2 "Gurobi seul sur l'instance complète" a été retirée)
# ══════════════════════════════════════════════════════════════════════════════
def build_bks_comparison_table(instances, initial_costs, best_costs, best_solutions,
                                instance_times, csv_path=None, latex_path=None):
    """
    Tableau final : Situation 1 (LNS+DQN+Gurobi) vs BKS, avec gap % et le
    nombre de véhicules utilisés pour chaque instance.
    """
    _sec("TABLEAU FINAL — SITUATION 1 vs BKS")
    rows = []
    for i, inst in enumerate(instances):
        name     = inst['name']
        bks      = BKS_REFERENCE.get(name)
        bks_dist = bks['distance'] if bks else None
        n_veh_s1 = sum(1 for r in best_solutions[i] if r)
        row = {
            'instance':             name,
            'type':                 inst['type'],
            'bks_vehicules':        bks['vehicles'] if bks else None,
            'bks_distance':         bks_dist,
            'n_vehicules_situation1': n_veh_s1,
            'distance_situation1':  round(best_costs[i], 2),
            'temps_s_situation1':   round(instance_times[i], 2),
            'gap_%_situation1_bks': round((best_costs[i] - bks_dist) / bks_dist * 100, 2) if bks_dist else None,
        }
        rows.append(row)
    print(f"\n  {'Instance':<9} {'BKS Vhc':>8} {'BKS Dist':>10} {'Vhc S1':>7} "
          f"{'Dist. S1':>10} {'Temps S1':>10} {'Gap %':>8}")
    print("  " + "─" * 70)
    for r in rows:
        bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "—"
        bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "—"
        g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {bks_v:>8} {bks_d:>10} "
              f"{r['n_vehicules_situation1']:>7} {r['distance_situation1']:>10.2f} "
              f"{r['temps_s_situation1']:>10.1f} {g1:>8}")
    # ── Gap moyen vs BKS (Situation 1) ──────────────────────────────────
    valid_gaps = [r['gap_%_situation1_bks'] for r in rows if r['gap_%_situation1_bks'] is not None]
    gap_moyen  = float(np.mean(valid_gaps)) if valid_gaps else None
    if gap_moyen is not None:
        print(f"\n  Gap moyen vs BKS (Situation 1) : {gap_moyen:+.2f}%  "
              f"({len(valid_gaps)}/{len(rows)} instances avec BKS connu)")
        print(f"  Gap min : {min(valid_gaps):+.2f}%  |  Gap max : {max(valid_gaps):+.2f}%")
    else:
        print("\n  Gap moyen vs BKS : indisponible (aucun BKS connu pour ces instances)")
    # ── Tableau récapitulatif simple : Instance / Nb véhicules / Distance / Gap % ──
    # (une ligne "Situation 1" + une ligne "BKS" par instance, format demandé)
    print(f"\n  Récapitulatif Instance / Nb véhicules / Distance / Gap % vs BKS")
    print(f"  {'Instance':<9} {'Situation':<26} {'Nb véhicules':>13} {'Distance':>10} {'Gap %':>8}")
    print("  " + "─" * 71)
    recap_rows = []
    for r in rows:
        g1 = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {'Situation 1 (DQN+LNS+Gurobi)':<26} "
              f"{r['n_vehicules_situation1']:>13} {r['distance_situation1']:>10.2f} {g1:>8}")
        recap_rows.append({
            'instance':    r['instance'],
            'situation':   'Situation 1 (DQN+LNS+Gurobi)',
            'n_vehicules': r['n_vehicules_situation1'],
            'distance':    r['distance_situation1'],
            'gap_%_vs_bks': r['gap_%_situation1_bks'],
        })
        if r['bks_distance'] is not None:
            print(f"  {r['instance']:<9} {'BKS (référence)':<26} "
                  f"{(r['bks_vehicules'] if r['bks_vehicules'] is not None else '—'):>13} "
                  f"{r['bks_distance']:>10.2f} {'0.00':>8}")
            recap_rows.append({
                'instance':    r['instance'],
                'situation':   'BKS (référence)',
                'n_vehicules': r['bks_vehicules'],
                'distance':    r['bks_distance'],
                'gap_%_vs_bks': 0.0,
            })
    if gap_moyen is not None:
        print("  " + "─" * 71)
        print(f"  {'':<9} {'MOYENNE (Situation 1)':<26} {'':>13} {'':>10} {gap_moyen:>+8.2f}")
    if csv_path:
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
        print(f"\n  Tableau final exporté (CSV) → {csv_path}")
        recap_csv_path = csv_path.replace('.csv', '_recap_vehicules_distance.csv')
        if recap_csv_path == csv_path:
            recap_csv_path = csv_path + '.recap_vehicules_distance.csv'
        with open(recap_csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=['instance', 'situation', 'n_vehicules', 'distance', 'gap_%_vs_bks'])
            w.writeheader(); w.writerows(recap_rows)
            if gap_moyen is not None:
                w.writerow({'instance': '', 'situation': 'MOYENNE (Situation 1)',
                            'n_vehicules': '', 'distance': '', 'gap_%_vs_bks': round(gap_moyen, 2)})
        print(f"  Récapitulatif Instance/Véhicules/Distance/Gap exporté (CSV) → {recap_csv_path}")
    if latex_path:
        lines = [
            "% Tableau généré automatiquement — Situation 1 vs BKS",
            "\\begin{table}[htbp]", "  \\centering",
            "  \\caption{Comparaison par instance Solomon officielle entre la méthode "
            "hybride (LNS + DQN + Gurobi repair) et le Best Known Solution (BKS) : "
            "nombre de véhicules, distance totale et écart au BKS.}",
            "  \\label{tab:comparaison-bks}",
            "  \\begin{tabular}{lrrrrrr}", "    \\toprule",
            "    Instance & BKS Vhc & BKS Dist. & Vhc. S1 & Dist. S1 & "
            "Temps S1 (s) & Gap S1 (\\%) \\\\",
            "    \\midrule",
        ]
        for r in rows:
            bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "--"
            bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "--"
            g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "--"
            lines.append(f"    {r['instance']} & {bks_v} & {bks_d} & "
                        f"{r['n_vehicules_situation1']} & {r['distance_situation1']:.2f} & "
                        f"{r['temps_s_situation1']:.1f} & {g1} \\\\")
        lines += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
        with open(latex_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        print(f"  Tableau final exporté (LaTeX) → {latex_path}")
    return rows
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6bis  —  DIAGNOSTIC : LICENCE GUROBI À TAILLE LIMITÉE ?
# ══════════════════════════════════════════════════════════════════════════════
def check_gurobi_license(min_vars_needed=2500):
    """
    Test rapide, effectué AVANT tout calcul, pour savoir si la licence
    Gurobi active est une licence "size-limited" (Community/gratuite,
    plafonnée à ~2000 variables ET ~2000 contraintes) plutôt qu'une licence
    complète (académique ou commerciale, sans limite de taille).

    Pourquoi c'est important ici : le repair Gurobi de la Situation 1
    (_repair_gurobi, appelé à CHAQUE step du LNS) crée jusqu'à
    (K_DESTROY+1)² * V variables binaires — selon K_DESTROY (cf. CFG) et le
    nombre de véhicules de l'instance, cela peut dépasser 2000. Un fallback
    vers le repair glouton est déjà en place dans _repair_gurobi (try/
    except), donc l'exécution ne plantera pas, mais peut silencieusement
    retomber sur le glouton plus souvent que prévu si la licence est limitée.

    Ce test ne bloque JAMAIS l'exécution : il informe seulement, une seule
    fois, en tête de run.
    """
    env, m = None, None
    try:
        env = gp.Env(empty=True)
        env.setParam('OutputFlag', 0)
        env.start()
        m = gp.Model(env=env)
        m.setParam('OutputFlag', 0)
        x = m.addVars(min_vars_needed, vtype=GRB.BINARY)
        m.setObjective(gp.quicksum(x[i] for i in range(min_vars_needed)), GRB.MAXIMIZE)
        m.optimize()
        print(f"  ✓ Licence Gurobi : aucune restriction de taille détectée "
              f"(modèle-test à {min_vars_needed} variables résolu sans erreur).")
        return True
    except gp.GurobiError as e:
        errno = getattr(e, 'errno', None)
        is_size_limit = (errno == getattr(GRB.Error, 'SIZE_LIMIT_EXCEEDED', object()))
        if is_size_limit:
            k_repair_vars = (CFG['K_DESTROY'] + 1) ** 2 * CFG['K_DESTROY']
            print(f"\n  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée "
                  f"(erreur {errno} : {e})\n"
                  f"    → Licence Community/gratuite, plafonnée à ~2000 variables "
                  f"et ~2000 contraintes.\n"
                  f"    → Le repair Gurobi de la Situation 1 (jusqu'à "
                  f"~{k_repair_vars} variables selon K_DESTROY={CFG['K_DESTROY']} et "
                  f"le nombre de véhicules) peut dépasser cette limite sur certaines "
                  f"instances — un repli automatique vers le repair glouton est déjà "
                  f"en place, donc pas de plantage, mais un usage de Gurobi moins "
                  f"fréquent que prévu.\n"
                  f"    → Pour lever cette limite : une licence académique Gurobi "
                  f"est gratuite et sans limite de taille pour les universités "
                  f"(https://www.gurobi.com/academia/) — utile si vous êtes "
                  f"affiliée à ENSI-UMA.\n")
        else:
            print(f"  ⚠ Erreur Gurobi lors du test de licence (non liée à la taille) : {e}")
        return False
    except Exception as e:
        print(f"  ⚠ Impossible de tester la licence Gurobi ({type(e).__name__}: {e}) "
              f"— test ignoré, l'exécution continue normalement.")
        return None
    finally:
        if m is not None:
            m.dispose()
        if env is not None:
            env.dispose()
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 7  —  COURBES DE CONVERGENCE
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_mean(vals, window=20):
    out = []
    for i in range(len(vals)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(vals[lo:i + 1])))
    return out
def plot_global_convergence(training_log, instances, initial_costs, best_costs, outdir):
    """
    Figure globale (2 panneaux) :
      (a) reward par épisode global + moyenne mobile → convergence de
          l'apprentissage DQN
      (b) amélioration % par instance (une barre par instance Solomon) vs
          coût LNS initial
    """
    episodes = [e['episode_global'] for e in training_log]
    rewards  = [e['total_reward']   for e in training_log]
    reward_smooth = _rolling_mean(rewards, window=20)
    improvements  = [(initial_costs[i] - best_costs[i]) / initial_costs[i] * 100
                      for i in range(len(initial_costs))]
    names = [inst['name'] for inst in instances]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    axes[0].plot(episodes, rewards, color='lightgray', linewidth=0.6, label='reward brut / épisode')
    axes[0].plot(episodes, reward_smooth, color='crimson', linewidth=2,
                 label='moyenne mobile (20 épisodes)')
    axes[0].set_xlabel('Épisode global')
    axes[0].set_ylabel('Reward de l\'épisode')
    axes[0].set_title("Convergence de l'apprentissage DQN")
    axes[0].legend(loc='best', fontsize=9)
    axes[0].grid(alpha=0.3)
    colors = ['#2ca02c' if v >= 0 else '#d62728' for v in improvements]
    axes[1].bar(range(len(improvements)), improvements, color=colors)
    axes[1].axhline(np.mean(improvements), color='navy', linestyle='--', linewidth=1.5,
                     label=f"Moyenne = {np.mean(improvements):.2f}%")
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names, rotation=90, fontsize=7)
    axes[1].set_xlabel('Instance')
    axes[1].set_ylabel('Amélioration vs solution LNS initiale (%)')
    axes[1].set_title("Amélioration du coût par instance après DQN+Gurobi")
    axes[1].legend(loc='best', fontsize=9)
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_globale.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence globale → {path}")
    return path
def plot_gap_vs_bks(bks_rows, outdir):
    """
    Barres du gap % vs BKS (Situation 1) pour chaque instance, avec la
    moyenne indiquée par une ligne horizontale. N'utilise que les instances
    pour lesquelles un BKS est connu (gap_%_situation1_bks non None).
    `bks_rows` = la liste `rows` retournée par build_bks_comparison_table().
    """
    data = [(r['instance'], r['gap_%_situation1_bks']) for r in bks_rows
            if r['gap_%_situation1_bks'] is not None]
    if not data:
        print("  [plot_gap_vs_bks] Aucune instance avec BKS connu — graphique ignoré.")
        return None
    names = [d[0] for d in data]
    gaps  = [d[1] for d in data]
    gap_moyen = float(np.mean(gaps))
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(names)), 5.5))
    colors = ['#2ca02c' if g <= 0 else '#d62728' for g in gaps]
    ax.bar(range(len(names)), gaps, color=colors)
    ax.axhline(gap_moyen, color='navy', linestyle='--', linewidth=1.5,
               label=f"Gap moyen = {gap_moyen:+.2f}%")
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_xlabel('Instance')
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_title(f"Écart au Best Known Solution par instance "
                 f"({len(names)} instances avec BKS connu)")
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique gap vs BKS → {path}")
    return path
def plot_instance_convergence_examples(training_log, instances, example_idxs, outdir):
    """
    Pour chaque instance exemple :
      - gauche : coût final / meilleur coût cumulé au fil des épisodes locaux
      - droite : trajectoire de coût step par step du dernier épisode local
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(13, 4.2 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        entries = [e for e in training_log if e['instance_id'] == idx]
        entries.sort(key=lambda e: e['ep_local'])
        ep_locals      = [e['ep_local']   for e in entries]
        best_costs_ep  = [e['best_cost']  for e in entries]
        final_costs_ep = [e['final_cost'] for e in entries]
        name = instances[idx]['name']
        axes[row, 0].plot(ep_locals, final_costs_ep, '-o', color='steelblue',
                           label='coût final de l\'épisode')
        axes[row, 0].plot(ep_locals, best_costs_ep, '-s', color='crimson',
                           label='meilleur coût cumulé')
        axes[row, 0].set_title(f"Instance {name} — convergence par épisode local")
        axes[row, 0].set_xlabel('Épisode local')
        axes[row, 0].set_ylabel('Coût de la tournée')
        axes[row, 0].legend(loc='best', fontsize=9)
        axes[row, 0].grid(alpha=0.3)
        last = entries[-1] if entries else None
        trace = last.get('cost_trace', []) if last else []
        if trace:
            axes[row, 1].plot(range(len(trace)), trace, '-o', color='darkorange')
            axes[row, 1].set_title(f"Instance {name} — trajectoire de coût "
                                    f"(dernier épisode local, {len(trace)-1} steps)")
            axes[row, 1].set_xlabel('Step')
            axes[row, 1].set_ylabel('Coût courant')
            axes[row, 1].grid(alpha=0.3)
        else:
            axes[row, 1].axis('off')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_instances_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence instances exemples → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 8  —  TRACÉS DES ROUTES (AVANT LNS / APRÈS DQN)
# ══════════════════════════════════════════════════════════════════════════════
def _plot_routes_on_ax(ax, instance, solution, title):
    depot = instance['depot']
    ax.scatter([depot['x']], [depot['y']], c='black', marker='s', s=110,
               zorder=5, label='Dépôt')
    n_routes = max(len([r for r in solution if r]), 1)
    try:
        cmap = matplotlib.colormaps.get_cmap('tab20').resampled(max(n_routes, 1))
    except AttributeError:
        cmap = plt.cm.get_cmap('tab20', max(n_routes, 1))
    ridx = 0
    for route in solution:
        if not route:
            continue
        xs = [depot['x']] + [instance['customers'][cid - 1]['x'] for cid in route] + [depot['x']]
        ys = [depot['y']] + [instance['customers'][cid - 1]['y'] for cid in route] + [depot['y']]
        ax.plot(xs, ys, '-o', color=cmap(ridx), markersize=4, linewidth=1.3)
        ridx += 1
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.3)
def plot_routes_examples(instances, initial_sols, best_solutions, example_idxs, outdir):
    """
    Pour chaque instance exemple : routes AVANT (solution LNS initiale, avant
    DQN) à gauche, routes APRÈS (meilleure solution trouvée par DQN+Gurobi)
    à droite. Le BKS (véhicules/distance) est indiqué en sous-titre s'il
    est connu pour l'instance.
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.8 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        inst = instances[idx]
        name = inst['name']
        bks  = BKS_REFERENCE.get(name)
        bks_str = (f" | BKS: dist={bks['distance']:.1f}, vhc={bks['vehicles']}"
                   if bks else "")
        cost_before = compute_cost(initial_sols[idx], inst)
        cost_after  = compute_cost(best_solutions[idx], inst)
        nveh_before = sum(1 for r in initial_sols[idx] if r)
        nveh_after  = sum(1 for r in best_solutions[idx] if r)
        _plot_routes_on_ax(
            axes[row, 0], inst, initial_sols[idx],
            f"Instance {name} — AVANT (LNS)\n"
            f"dist={cost_before:.1f}  véhicules={nveh_before}{bks_str}"
        )
        _plot_routes_on_ax(
            axes[row, 1], inst, best_solutions[idx],
            f"Instance {name} — APRÈS (DQN+Gurobi)\n"
            f"dist={cost_after:.1f}  véhicules={nveh_after}{bks_str}"
        )
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_routes_avant_apres_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique routes avant/après → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 9  —  ANALYSES PAR GROUPE D'INSTANCES (C1/C2/R1/R2/RC1/RC2)
#  Appliqué uniquement aux résultats du TEST (gap vs BKS) — ne joue aucun
#  rôle dans le choix des hyperparamètres, purement diagnostique a posteriori.
# ══════════════════════════════════════════════════════════════════════════════
def get_instance_group(inst):
    """
    Déduit le groupe Solomon (C1/C2/R1/R2/RC1/RC2) à partir du type et du nom
    de l'instance. Ex. : type='C', name='C101' -> 'C1' ; type='RC', name=
    'RC201' -> 'RC2'. Le chiffre juste après le préfixe alphabétique indique
    le sous-groupe (1 = fenêtres serrées / faible capacité, 2 = fenêtres
    larges / grande capacité).
    """
    t = inst['type']
    name = inst['name']
    digit = name[len(t)] if len(name) > len(t) else ''
    return f"{t}{digit}"
def plot_gap_boxplot_by_group(bks_rows, instances, outdir):
    """
    Boxplot du gap % vs BKS regroupé par C1/C2/R1/R2/RC1/RC2, pour visualiser
    quels groupes d'instances posent le plus de difficulté à la méthode.
    """
    by_name = {inst['name']: inst for inst in instances}
    groups = defaultdict(list)
    for r in bks_rows:
        if r['gap_%_situation1_bks'] is None:
            continue
        inst = by_name.get(r['instance'])
        if inst is None:
            continue
        groups[get_instance_group(inst)].append(r['gap_%_situation1_bks'])
    labels = [g for g in GROUP_ORDER if g in groups]
    data   = [groups[g] for g in labels]
    if not data:
        print("  [plot_gap_boxplot_by_group] Aucune donnée — graphique ignoré.")
        return None
    fig, ax = plt.subplots(figsize=(8, 5.5))
    try:
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, showmeans=True)
    except TypeError:
        # matplotlib < 3.9 : le paramètre s'appelait encore 'labels'.
        bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#a6c8ff')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_xlabel('Groupe d\'instances Solomon')
    ax.set_title("Distribution du gap par groupe d'instances")
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_boxplot_par_groupe.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique boxplot gap par groupe → {path}")
    return path
def plot_vehicles_vs_bks(bks_rows, instances, outdir):
    """
    Scatter plot : nombre de véhicules utilisés (Situation 1) vs nombre de
    véhicules du BKS, par instance, coloré par groupe. La diagonale y=x
    marque l'égalité ; les points au-dessus utilisent plus de véhicules que
    le BKS — c'est le phénomène d'inflation identifié dans la validation.
    """
    by_name = {inst['name']: inst for inst in instances}
    cmap_colors = plt.cm.tab10.colors
    color_map = {g: cmap_colors[i % len(cmap_colors)] for i, g in enumerate(GROUP_ORDER)}
    fig, ax = plt.subplots(figsize=(7, 7))
    seen_groups = set()
    max_v = 1
    for r in bks_rows:
        if r['bks_vehicules'] is None or r['n_vehicules_situation1'] is None:
            continue
        inst = by_name.get(r['instance'])
        grp  = get_instance_group(inst) if inst else '?'
        c    = color_map.get(grp, 'gray')
        label = grp if grp not in seen_groups else None
        seen_groups.add(grp)
        ax.scatter(r['bks_vehicules'], r['n_vehicules_situation1'], color=c,
                   label=label, s=45, alpha=0.85, edgecolor='black', linewidth=0.4)
        max_v = max(max_v, r['bks_vehicules'], r['n_vehicules_situation1'])
    ax.plot([0, max_v + 1], [0, max_v + 1], '--', color='gray', linewidth=1, label='y = x (égalité)')
    ax.set_xlabel('Véhicules BKS')
    ax.set_ylabel('Véhicules Situation 1 (DQN+LNS+Gurobi)')
    ax.set_title('Nombre de véhicules utilisés vs BKS, par instance')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_vehicules_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique véhicules vs BKS → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 10  —  PLAN D'EXPÉRIENCE TAGUCHI (calibration des hyperparamètres)
#
#  Objectif : identifier quels hyperparamètres (LNS_ITER, K_DESTROY_RATIO,
#  GUROBI_TLIMIT, GUROBI_GAP, MAX_STEPS, EPS_DECAY, VEHICLE_PENALTY) ont le
#  plus d'effet sur la performance, et quelle combinaison la maximise, sans
#  tester les 3^7 = 2187 combinaisons possibles.
#
#  ⚠ POINT CRITIQUE (train/test disjoint) : ce module supporte DEUX modes,
#  sélectionnés par le paramètre `instances_override`/`metric` de
#  run_taguchi_doe() :
#    - mode SYNTHÉTIQUE (recommandé, piloté par CFG['TAGUCHI_SYNTH']) :
#      `instances_override` = des instances synthétiques générées à la
#      volée ; `metric='lns_improvement'` = % d'amélioration vs le coût LNS
#      initial (aucun BKS requis). Les 56 Solomon ne sont JAMAIS chargées
#      dans ce mode → zéro fuite d'hyperparamètres vers le jeu de test.
#    - mode SOLOMON historique (⚠ non recommandé, piloté par CFG['TAGUCHI']) :
#      `instances_override=None` → charge INSTANCE_FILES (les 56 Solomon)
#      comme matrice externe ; `metric='bks_gap'` = gap % vs BKS_REFERENCE.
#      Ce mode entraîne et évalue directement sur le jeu de test : la
#      combinaison qu'il retourne a "vu" Solomon et ne doit pas être
#      utilisée comme hyperparamètres d'une validation train/test disjointe
#      (elle resterait valable seulement comme référence de comparaison).
#
#  Principe commun (matrice orthogonale L9/L18) :
#    - Matrice interne  : la matrice orthogonale L18 ci-dessous (18 lignes,
#      7 facteurs, 3 niveaux) — les combinaisons de facteurs à tester.
#    - Matrice externe (bruit) : les instances sur lesquelles chaque
#      combinaison de la matrice interne est évaluée ; le ratio signal/bruit
#      (S/N) résume à la fois la performance moyenne ET la stabilité face à
#      la variabilité entre instances/groupes.
#    - Après le plan, reportez la meilleure combinaison dans CFG, repassez
#      ENABLE=False, et relancez le script pour la validation ENTRAÎNEMENT/
#      TEST DISJOINT normale (avec ces hyperparamètres).
# ══════════════════════════════════════════════════════════════════════════════
# Matrice orthogonale L9 (3^4) — 9 runs, jusqu'à 4 facteurs à 3 niveaux.
L9_ARRAY = [
    [1, 1, 1, 1], [1, 2, 2, 2], [1, 3, 3, 3],
    [2, 1, 2, 3], [2, 2, 3, 1], [2, 3, 1, 2],
    [3, 1, 3, 2], [3, 2, 1, 3], [3, 3, 2, 1],
]
# Matrice orthogonale L18 (2^1 x 3^7) — 18 runs. La 1ère colonne (2 niveaux)
# n'est pas utilisée ici (tous nos facteurs sont à 3 niveaux) ; seules les
# colonnes 2 à 8 (7 facteurs à 3 niveaux) servent au plan.
L18_ARRAY_FULL = [
    [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 2, 2, 2, 2, 2, 2], [1, 1, 3, 3, 3, 3, 3, 3],
    [1, 2, 1, 1, 2, 2, 3, 3], [1, 2, 2, 2, 3, 3, 1, 1], [1, 2, 3, 3, 1, 1, 2, 2],
    [1, 3, 1, 2, 1, 3, 2, 3], [1, 3, 2, 3, 2, 1, 3, 1], [1, 3, 3, 1, 3, 2, 1, 2],
    [2, 1, 1, 3, 3, 2, 2, 1], [2, 1, 2, 1, 1, 3, 3, 2], [2, 1, 3, 2, 2, 1, 1, 3],
    [2, 2, 1, 2, 3, 1, 3, 2], [2, 2, 2, 3, 1, 2, 1, 3], [2, 2, 3, 1, 2, 3, 2, 1],
    [2, 3, 1, 3, 2, 3, 1, 2], [2, 3, 2, 1, 3, 1, 2, 3], [2, 3, 3, 2, 1, 2, 3, 1],
]
L18_ARRAY = [row[1:] for row in L18_ARRAY_FULL]   # ne garder que les 7 colonnes à 3 niveaux
def check_orthogonal_array(array):
    """
    Vérifie la propriété d'orthogonalité : pour chaque paire de colonnes,
    chaque combinaison de niveaux doit apparaître le même nombre de fois.
    Retourne True/False — sert de garde-fou avant de lancer un plan
    potentiellement long sur une matrice mal formée.
    """
    arr = np.array(array)
    n_runs, n_cols = arr.shape
    ok = True
    for c1, c2 in itertools.combinations(range(n_cols), 2):
        counts = defaultdict(int)
        for row in arr:
            counts[(row[c1], row[c2])] += 1
        if len(set(counts.values())) > 1:
            ok = False
    return ok
def select_orthogonal_array(n_factors):
    if n_factors <= 4:
        return 'L9', [row[:n_factors] for row in L9_ARRAY]
    elif n_factors <= 7:
        return 'L18', [row[:n_factors] for row in L18_ARRAY]
    else:
        raise ValueError(
            f"{n_factors} facteurs demandés, mais seules les matrices L9 (4 "
            f"facteurs max) et L18 (7 facteurs max) sont disponibles. "
            f"Réduisez TAGUCHI_FACTORS ou étendez ce module avec une L27."
        )
@contextmanager
def _cfg_override(**kwargs):
    """Applique temporairement des valeurs de CFG (pour un run du plan
    Taguchi), puis restaure les valeurs d'origine à la sortie du bloc."""
    old = {k: CFG.get(k) for k in kwargs}
    CFG.update(kwargs)
    try:
        yield
    finally:
        CFG.update(old)
def taguchi_sn_ratio_stb(values):
    """
    Ratio signal/bruit 'plus petit est meilleur' (Smaller-The-Better),
    calculé sur une liste de valeurs (le gap %/l'amélioration % peut être
    négatif — la formule standard SN = -10*log10(mean(y_i^2)) reste valide,
    elle pénalise tout écart à zéro, positif ou négatif, ainsi que la
    variance entre les instances de la matrice externe).
    """
    y = np.array(values, dtype=float)
    msd = max(float(np.mean(y ** 2)), 1e-9)
    return -10.0 * np.log10(msd)
def taguchi_main_effects(results, factor_names, response_key):
    """Moyenne de la réponse (gap/amélioration ou S/N) par niveau, pour
    chaque facteur."""
    effects = {}
    for fn in factor_names:
        by_level = defaultdict(list)
        for r in results:
            if r.get(response_key) is not None:
                by_level[r[fn]].append(r[response_key])
        effects[fn] = {lvl: float(np.mean(vals)) for lvl, vals in sorted(by_level.items(), key=lambda kv: kv[0])}
    return effects
def taguchi_anova_contributions(results, factor_names, response_key):
    """
    ANOVA simplifiée (méthode des sommes de carrés à partir des moyennes par
    niveau) : contribution en % de chaque facteur à la variance totale de la
    réponse. Ne remplace pas une ANOVA complète avec terme d'erreur pur,
    mais donne une hiérarchie fiable de l'importance relative des facteurs.
    """
    valid = [r for r in results if r.get(response_key) is not None]
    if not valid:
        return {fn: 0.0 for fn in factor_names}
    y = np.array([r[response_key] for r in valid])
    grand_mean = y.mean()
    ss_total = float(np.sum((y - grand_mean) ** 2))
    contributions = {}
    for fn in factor_names:
        by_level = defaultdict(list)
        for r in valid:
            by_level[r[fn]].append(r[response_key])
        ss_factor = sum(len(vals) * (np.mean(vals) - grand_mean) ** 2 for vals in by_level.values())
        contributions[fn] = (ss_factor / ss_total * 100.0) if ss_total > 0 else 0.0
    return contributions
def plot_taguchi_main_effects(effects, factor_names, response_label, outdir, filename):
    """Graphique classique des effets principaux Taguchi : un panneau par
    facteur, moyenne de la réponse pour chaque niveau, reliée par une ligne."""
    n = len(factor_names)
    ncols = min(4, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, fn in enumerate(factor_names):
        levels = list(effects[fn].keys())
        means  = list(effects[fn].values())
        axes[i].plot(range(len(levels)), means, '-o', color='steelblue')
        axes[i].set_xticks(range(len(levels)))
        axes[i].set_xticklabels([str(l) for l in levels], fontsize=8)
        axes[i].set_title(fn, fontsize=10)
        axes[i].set_ylabel(response_label, fontsize=8)
        axes[i].grid(alpha=0.3)
    for j in range(len(factor_names), len(axes)):
        axes[j].axis('off')
    fig.suptitle(f"Effets principaux — {response_label}", fontsize=13)
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique effets principaux ({response_label}) → {path}")
    return path
def plot_taguchi_sn_bars(results, outdir, filename='taguchi_sn_ratios.png'):
    """Ratio S/N obtenu pour chaque run du plan, run avec le meilleur S/N
    mis en évidence en vert."""
    valid = [r for r in results if r.get('sn_ratio') is not None]
    if not valid:
        print("  [plot_taguchi_sn_bars] Aucun résultat — graphique ignoré.")
        return None
    runs = [r['run'] for r in valid]
    sns_ = [r['sn_ratio'] for r in valid]
    best_idx = int(np.argmax(sns_))
    colors = ['#2ca02c' if i == best_idx else '#4c72b0' for i in range(len(sns_))]
    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(runs)), 5))
    ax.bar([str(r) for r in runs], sns_, color=colors)
    ax.set_xlabel('Run (combinaison de la matrice L9/L18)')
    ax.set_ylabel('Ratio S/N (plus grand = meilleur)')
    ax.set_title('Ratio signal/bruit par combinaison — plan Taguchi')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique S/N par run → {path}")
    return path
def plot_taguchi_pareto(anova_contributions, outdir, filename='taguchi_pareto_effets.png'):
    """Diagramme en barres horizontales de la contribution (%) de chaque
    facteur à la variance de la réponse — équivalent simplifié d'un Pareto ANOVA."""
    items = sorted(anova_contributions.items(), key=lambda kv: -kv[1])
    names = [k for k, v in items][::-1]
    vals  = [v for k, v in items][::-1]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(range(len(names)), vals, color='#4c72b0')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('Contribution à la variance de la réponse (%)')
    ax.set_title("Importance relative des facteurs (ANOVA simplifiée Taguchi)")
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique Pareto des effets → {path}")
    return path
def run_taguchi_doe(factors, outer_instance_names, episodes_per_instance_doe, outdir,
                     instances_override=None, metric='bks_gap', json_suffix=''):
    """
    Exécute le plan d'expérience Taguchi complet :
      1. Construit la matrice orthogonale adaptée au nombre de facteurs
      2. Pour chaque run, applique les niveaux de facteurs (via _cfg_override)
         et entraîne le pipeline complet (LNS+DQN+Gurobi) sur les instances
         de la matrice externe, avec un nombre d'épisodes réduit
      3. Calcule la métrique de classement par instance (voir `metric`), la
         moyenne, et le ratio S/N
      4. Analyse les effets principaux + ANOVA simplifiée, génère les graphes
      5. Rapporte la meilleure combinaison trouvée
    instances_override : si fourni (liste d'instances déjà construites),
        utilisé DIRECTEMENT comme matrice externe, SANS jamais charger les
        fichiers Solomon — c'est ce qui permet de calibrer sur des instances
        SYNTHÉTIQUES (voir CFG['TAGUCHI_SYNTH']) sans aucune fuite vers le
        jeu de test. Si None (mode historique Solomon), charge les instances
        via INSTANCE_FILES et `outer_instance_names`.
    metric : 'bks_gap' (gap % vs BKS_REFERENCE — ne fonctionne que pour des
        instances Solomon officielles connues dans BKS_REFERENCE) ou
        'lns_improvement' (% d'amélioration vs le coût LNS initial — aucun
        BKS requis, utilisable sur des instances synthétiques). Convention
        "plus petit est meilleur" identique dans les deux cas (négatif =
        amélioration), donc toute la mécanique Taguchi (S/N, effets
        principaux, ANOVA, combinaison prédite) fonctionne sans modification.
    json_suffix : suffixe inséré dans les noms des fichiers de sortie (CSV,
        JSON, PNG) — ex. '_synthetique' — pour ne jamais écraser les
        résultats d'un plan calibré sur d'autres instances/métrique.
    Retourne : results (liste de dicts, un par run), effects_gap, effects_sn,
               anova_contributions, best_run
    """
    factor_names  = [f[0] for f in factors]
    factor_levels = [f[1] for f in factors]
    n_factors     = len(factors)
    array_name, array = select_orthogonal_array(n_factors)
    ortho_ok = check_orthogonal_array(array)
    if instances_override is not None:
        outer_instances = list(instances_override)
    else:
        all_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
        if outer_instance_names == 'ALL':
            # Sentinel spécial : utiliser TOUTES les instances Solomon
            # chargées comme matrice externe. ⚠ Mode historique — voir
            # avertissement CFG['TAGUCHI'] plus haut : fuite d'hyperparamètres
            # vers le jeu de test. Multiplie aussi fortement le temps total
            # du plan (voir avertissement de durée ci-dessous).
            outer_instances = list(all_instances)
        else:
            by_name = {inst['name']: inst for inst in all_instances}
            outer_instances = [by_name[nm] for nm in outer_instance_names if nm in by_name]
    if not outer_instances:
        raise ValueError("Aucune instance de la matrice externe trouvée "
                          "(instances_override vide, ou outer_instance_names "
                          "absent de INSTANCE_FILES).")
    gurobi_tlimit_max = max(TAGUCHI_FACTORS[[f[0] for f in TAGUCHI_FACTORS].index('GUROBI_TLIMIT')][1]) \
        if 'GUROBI_TLIMIT' in factor_names else CFG['GUROBI_TLIMIT']
    max_steps_max = max(TAGUCHI_FACTORS[[f[0] for f in TAGUCHI_FACTORS].index('MAX_STEPS')][1]) \
        if 'MAX_STEPS' in factor_names else CFG['MAX_STEPS']
    worst_case_h = (len(array) * len(outer_instances) * gurobi_tlimit_max
                     * max_steps_max * episodes_per_instance_doe) / 3600.0
    _sec(f"PLAN D'EXPÉRIENCE TAGUCHI — {array_name} ({len(array)} runs, "
         f"{n_factors} facteurs, métrique={metric}, orthogonalité="
         f"{'OK' if ortho_ok else 'A VERIFIER !'})")
    if len(outer_instances) <= 12:
        print(f"  Matrice externe (bruit) : {[i['name'] for i in outer_instances]}")
    else:
        print(f"  Matrice externe (bruit) : {len(outer_instances)} instances "
              f"({outer_instances[0]['name']} ... {outer_instances[-1]['name']})")
    print(f"  Épisodes/instance pendant le plan : {episodes_per_instance_doe}")
    print(f"  ⚠ Temps pire cas estimé (si Gurobi atteint systématiquement sa limite de "
          f"temps) : ~{worst_case_h:.1f}h ({worst_case_h/24:.1f} jours). "
          f"Le temps réel est généralement inférieur, mais reste substantiel à cette échelle.")
    results = []
    for run_idx, level_idx_row in enumerate(array, start=1):
        levels = {factor_names[i]: factor_levels[i][level_idx_row[i] - 1] for i in range(n_factors)}
        print(f"\n  Run {run_idx:02d}/{len(array)} : {levels}")
        overrides = dict(levels)
        if 'K_DESTROY_RATIO' in overrides:
            overrides['K_DESTROY'] = max(3, round(100 * overrides['K_DESTROY_RATIO']))
        t_run = time.time()
        with _cfg_override(**overrides):
            _, _, initial_costs_o, best_costs_o, _, _, _ = train_on_instances(
                outer_instances, episodes_per_instance=episodes_per_instance_doe,
                save_model=False, verbose=False,
            )
        values = []
        for i, inst in enumerate(outer_instances):
            if metric == 'bks_gap':
                bks = BKS_REFERENCE.get(inst['name'])
                if bks:
                    values.append((best_costs_o[i] - bks['distance']) / bks['distance'] * 100.0)
            elif metric == 'lns_improvement':
                ic = initial_costs_o[i]
                if ic:
                    values.append((best_costs_o[i] - ic) / ic * 100.0)
            else:
                raise ValueError(f"metric inconnue : {metric!r} (attendu 'bks_gap' ou 'lns_improvement')")
        mean_val = float(np.mean(values)) if values else None
        sn       = taguchi_sn_ratio_stb(values) if values else None
        row_result = dict(levels)
        row_result.update({
            'run': run_idx,
            'mean_gap_%': round(mean_val, 3) if mean_val is not None else None,
            'sn_ratio':   round(sn, 3) if sn is not None else None,
        })
        results.append(row_result)
        elapsed = time.time() - t_run
        if mean_val is not None:
            print(f"    → {'gap' if metric=='bks_gap' else 'amélioration'} moyen(ne) = "
                  f"{mean_val:+.2f}%  |  S/N = {sn:.2f}  |  {elapsed:.1f}s")
        else:
            print(f"    → indisponible  |  {elapsed:.1f}s")
    csv_path = os.path.join(outdir, f'taguchi_resultats{json_suffix}.csv')
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = factor_names + ['run', 'mean_gap_%', 'sn_ratio']
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        w.writeheader()
        for r in results:
            w.writerow(r)
    print(f"\n  Résultats Taguchi exportés (CSV) → {csv_path}")
    effects_gap = taguchi_main_effects(results, factor_names, 'mean_gap_%')
    effects_sn  = taguchi_main_effects(results, factor_names, 'sn_ratio')
    anova       = taguchi_anova_contributions(results, factor_names, 'mean_gap_%')
    response_label = 'Gap moyen vs BKS (%)' if metric == 'bks_gap' else 'Amélioration moyenne vs LNS (%)'
    _sec("GRAPHIQUES TAGUCHI")
    plot_taguchi_main_effects(effects_gap, factor_names, response_label, outdir,
                              f'taguchi_effets_principaux_gap{json_suffix}.png')
    plot_taguchi_main_effects(effects_sn, factor_names, 'Ratio S/N', outdir,
                              f'taguchi_effets_principaux_sn{json_suffix}.png')
    plot_taguchi_sn_bars(results, outdir, filename=f'taguchi_sn_ratios{json_suffix}.png')
    plot_taguchi_pareto(anova, outdir, filename=f'taguchi_pareto_effets{json_suffix}.png')
    valid_runs = [r for r in results if r['mean_gap_%'] is not None]
    best_run = min(valid_runs, key=lambda r: abs(r['mean_gap_%'])) if valid_runs else None
    print(f"\n{'█'*72}")
    print(f"  █  MEILLEURE COMBINAISON TROUVÉE — PLAN TAGUCHI{' (' + json_suffix + ')' if json_suffix else ''}")
    print(f"{'█'*72}")
    if best_run:
        print(f"  Run {best_run['run']} : {'gap' if metric=='bks_gap' else 'amélioration'} moyen(ne) = "
              f"{best_run['mean_gap_%']:+.2f}%  |  S/N = {best_run['sn_ratio']:.2f}")
        print(f"  ── Hyperparamètres recommandés ──")
        for fn in factor_names:
            print(f"    {fn:<18s} = {best_run[fn]}")
        print(f"\n  ── Contribution des facteurs à la variance de la réponse (ANOVA) ──")
        for fn, pct in sorted(anova.items(), key=lambda kv: -kv[1]):
            print(f"    {fn:<18s} : {pct:5.1f}%")
        print(f"\n  → Reportez ces valeurs dans CFG (et TAGUCHI_FACTORS si besoin), "
              f"repassez ENABLE=False, et relancez le script pour la validation "
              f"ENTRAÎNEMENT/TEST DISJOINT sur les 56 instances Solomon.")
        best_json_path = os.path.join(outdir, f'taguchi_meilleure_combinaison{json_suffix}.json')
        with open(best_json_path, 'w', encoding='utf-8') as f:
            json.dump({
                'run':               best_run['run'],
                'metric':            metric,
                'mean_gap_%':        best_run['mean_gap_%'],
                'sn_ratio':          best_run['sn_ratio'],
                'hyperparametres':   {fn: best_run[fn] for fn in factor_names},
                'contribution_anova_%': anova,
                'array':             array_name,
                'n_runs':            len(array),
                'n_outer_instances': len(outer_instances),
                'episodes_per_instance_doe': episodes_per_instance_doe,
            }, f, indent=2, ensure_ascii=False)
        print(f"\n  Meilleure combinaison sauvegardée (JSON) → {best_json_path}")
    print(f"{'█'*72}\n")
    return results, effects_gap, effects_sn, anova, best_run
def load_and_apply_taguchi_best(outdir, json_suffix=''):
    """
    Étape de confirmation : cherche taguchi_meilleure_combinaison<suffix>.json
    (créé par run_taguchi_doe) et applique ces hyperparamètres à CFG, pour
    que la validation ENTRAÎNEMENT/TEST DISJOINT utilise automatiquement la
    meilleure combinaison trouvée — sans avoir à la reporter à la main.
    Passez json_suffix='_synthetique' pour charger la combinaison calibrée
    UNIQUEMENT sur du synthétique (recommandé). Ne modifie rien (et affiche
    un avertissement) si le fichier n'existe pas encore.
    Retourne True si des hyperparamètres ont été appliqués, False sinon.
    """
    path = os.path.join(outdir, f'taguchi_meilleure_combinaison{json_suffix}.json')
    if not os.path.exists(path):
        cfg_key_hint = "CFG['TAGUCHI_SYNTH']" if json_suffix else "CFG['TAGUCHI']"
        print(f"  ⚠ {path} introuvable — lancez d'abord le plan Taguchi correspondant "
              f"({cfg_key_hint}['ENABLE']=True). Hyperparamètres CFG actuels inchangés.")
        return False
    with open(path, 'r', encoding='utf-8') as f:
        best = json.load(f)
    hp = best.get('hyperparametres', {})
    for fn, val in hp.items():
        CFG[fn] = val
    if 'K_DESTROY_RATIO' in hp:
        CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))
    print(f"  ✓ Hyperparamètres Taguchi appliqués depuis {path} :")
    for fn, val in hp.items():
        print(f"    {fn:<18s} = {val}")
    return True
# ══════════════════════════════════════════════════════════════════════════════
#  EXÉCUTION AUTOMATIQUE
# ══════════════════════════════════════════════════════════════════════════════
def run_final_validation():
    """
    MODE VALIDATION : ENTRAÎNEMENT/TEST DISJOINT.
    Entraîne le DQN sur CFG['N_SYNTH_INSTANCES'] instances synthétiques puis
    l'évalue gelé sur les 56 instances Solomon officielles. Si
    CFG['USE_TAGUCHI_BEST']=True, applique d'abord automatiquement la
    meilleure combinaison d'hyperparamètres trouvée par le plan Taguchi
    synthétique (fichier taguchi_meilleure_combinaison_synthetique.json).

    Cette fonction regroupe exactement ce que faisait auparavant la branche
    `else` du bloc __main__ — extraite dans une fonction pour pouvoir être
    appelée aussi bien seule (validation directe, sans calibration) que
    depuis la branche Taguchi synthétique, en enchaînement automatique.
    """
    t0 = time.time()
    print(f"\n{'═'*72}\n  MODE ENTRAÎNEMENT/TEST DISJOINT\n"
          f"  ENTRAÎNEMENT = {CFG['N_SYNTH_INSTANCES']} instances SYNTHÉTIQUES générées "
          f"| TEST = 56 instances Solomon officielles (gelé)\n{'═'*72}")

    # ── DIAGNOSTIC PRÉALABLE : licence Gurobi à taille limitée ? ────────
    # Informatif uniquement — n'interrompt jamais l'exécution.
    _sec("DIAGNOSTIC — LICENCE GUROBI")
    check_gurobi_license()

    # ── Optionnel : appliquer la meilleure combinaison Taguchi calibrée
    # sur synthétique (si le plan CFG['TAGUCHI_SYNTH'] a déjà été lancé
    # au moins une fois et a produit le JSON correspondant) ───────────
    if CFG.get('USE_TAGUCHI_BEST', False):
        _sec("APPLICATION DE LA MEILLEURE COMBINAISON TAGUCHI (SYNTHÉTIQUE)")
        load_and_apply_taguchi_best(CFG['OUTDIR'], json_suffix='_synthetique')

    # ── ÉTAPE 1a : GÉNÉRATION DES INSTANCES SYNTHÉTIQUES D'ENTRAÎNEMENT ─
    # (reprend generate_instance() de Code 1 — cf. MODULE 1bis ci-dessus).
    # Ces instances SYN-* ne recoupent jamais les noms Solomon (C101, R101,
    # ...) chargés à l'ÉTAPE 2 : aucune fuite entre train et test.
    synth_instances = generate_synthetic_training_set(CFG)

    # ── ÉTAPE 1b : ENTRAÎNEMENT DU DQN — uniquement sur le synthétique ──
    # Valeurs par défaut : si le modèle est rechargé (branche ci-dessous),
    # il n'y a pas de training_log/coûts synthétiques dans CE run — les
    # graphiques de convergence d'entraînement seront alors simplement omis.
    training_log, synth_initial_costs, synth_best_costs = [], None, None
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    if CFG.get('REUSE_EXISTING_SYNTH_MODEL', False) and os.path.exists(model_path):
        _sec(f"ÉTAPE 1b — RÉUTILISATION D'UN MODÈLE DÉJÀ ENTRAÎNÉ SUR LE SYNTHÉTIQUE "
             f"({model_path})")
        agent = DQNAgent(
            obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
            gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
            epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
            buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
            target_update_freq=CFG['TARGET_UPDATE'],
        )
        agent.load(model_path)
    else:
        agent, training_log, synth_initial_costs, synth_best_costs, \
            synth_best_solutions, synth_instance_times, synth_initial_sols = train_on_instances(
                synth_instances, episodes_per_instance=CFG['EPISODES_PER_INSTANCE'])

    # ── ÉTAPE 2a : CHARGEMENT DES 56 INSTANCES SOLOMON (TEST — jamais vues) ─
    _sec("ÉTAPE 2 — CHARGEMENT DES INSTANCES SOLOMON (TEST)")
    test_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
    example_idxs = pick_example_indices(test_instances, preferred_names=CFG['EXAMPLE_NAMES'])

    # ── ÉTAPE 2b : ÉVALUATION GELÉE — le même agent en mémoire, sans réentraîner ──
    initial_costs, best_costs, best_solutions, instance_times, test_initial_sols, test_log = \
        evaluate_frozen_on_instances(
            test_instances, agent, episodes_per_instance=CFG['EPISODES_PER_INSTANCE_TEST'])

    # ── [A] Convergence de l'ENTRAÎNEMENT (synthétique uniquement) ──────
    _sec("GRAPHIQUE — CONVERGENCE DE L'ENTRAÎNEMENT (SYNTHÉTIQUE)")
    if training_log:
        plot_global_convergence(training_log, synth_instances,
                                 synth_initial_costs, synth_best_costs, CFG['OUTDIR'])
    else:
        print("  (pas de courbe de convergence : modèle rechargé, aucun "
              "entraînement effectué dans ce run)")

    # ── [B] Convergence détaillée + [C] Routes avant/après — SUR LE TEST ─
    _sec("GRAPHIQUES — DÉTAIL DU TEST (INSTANCES SOLOMON EXEMPLES)")
    plot_instance_convergence_examples(test_log, test_instances, example_idxs, CFG['OUTDIR'])
    plot_routes_examples(test_instances, test_initial_sols, best_solutions, example_idxs, CFG['OUTDIR'])

    # ── [D] TABLEAU FINAL : SITUATION 1 (test gelé, entraîné sur synthétique) ─
    # vs BKS (gap % par instance + gap moyen) ──────────────────────────
    bks_rows = build_bks_comparison_table(
        test_instances, initial_costs, best_costs, best_solutions, instance_times,
        csv_path=os.path.join(CFG['OUTDIR'], 'solomon_comparaison_bks.csv'),
        latex_path=os.path.join(CFG['OUTDIR'], 'solomon_tableau_bks.tex'),
    )
    # ── [E] Graphique dédié : gap % vs BKS par instance + gap moyen (TEST) ─
    _sec("GRAPHIQUE GAP % vs BKS (TEST)")
    plot_gap_vs_bks(bks_rows, CFG['OUTDIR'])

    # ── [F] Analyses par groupe (C1/C2/R1/R2/RC1/RC2) — véhicules & gap ──
    _sec("GRAPHIQUES — ANALYSES PAR GROUPE D'INSTANCES (TEST)")
    plot_gap_boxplot_by_group(bks_rows, test_instances, CFG['OUTDIR'])
    plot_vehicles_vs_bks(bks_rows, test_instances, CFG['OUTDIR'])

    print(f"\n  ✓ Validation finale terminée en {time.time()-t0:.1f}s "
          f"({(time.time()-t0)/3600:.2f}h)\n")
    return bks_rows


if __name__ == '__main__':
    t_global0 = time.time()

    if CFG['TAGUCHI_SYNTH']['ENABLE'] and CFG['TAGUCHI']['ENABLE']:
        raise ValueError(
            "CFG['TAGUCHI_SYNTH']['ENABLE'] et CFG['TAGUCHI']['ENABLE'] sont tous les "
            "deux True — activez UN SEUL des deux modes de calibration à la fois "
            "(TAGUCHI_SYNTH est recommandé, voir les commentaires plus haut)."
        )

    if CFG['TAGUCHI_SYNTH']['ENABLE']:
        # ── MODE CALIBRATION TAGUCHI — SYNTHÉTIQUE UNIQUEMENT (RECOMMANDÉ) ──
        # Aucune instance Solomon n'est chargée dans cette branche : zéro
        # fuite d'hyperparamètres vers le jeu de test.
        ts = CFG['TAGUCHI_SYNTH']
        print(f"\n{'═'*72}\n  MODE PLAN D'EXPÉRIENCE TAGUCHI — SYNTHÉTIQUE UNIQUEMENT "
              f"({ts['N_INSTANCES_DOE']} instances, métrique = amélioration vs LNS, "
              f"pas de BKS, zéro fuite vers Solomon)\n{'═'*72}")
        synth_doe_instances = generate_synthetic_training_set(
            dict(N_SYNTH_INSTANCES=ts['N_INSTANCES_DOE'],
                 SYNTH_N_CUSTOMERS=CFG['SYNTH_N_CUSTOMERS'],
                 SYNTH_N_VEHICLES=CFG['SYNTH_N_VEHICLES'],
                 SYNTH_VEHICLE_SAFETY_FACTOR=CFG['SYNTH_VEHICLE_SAFETY_FACTOR'],
                 SYNTH_CAPACITY=CFG['SYNTH_CAPACITY'], SYNTH_HORIZON=CFG['SYNTH_HORIZON']),
            verbose=True,
        )
        (ts_results, ts_effects_gap, ts_effects_sn,
         ts_anova, ts_best_run) = run_taguchi_doe(
            factors=TAGUCHI_FACTORS,
            outer_instance_names=None,
            episodes_per_instance_doe=ts['EPISODES_PER_INSTANCE_DOE'],
            outdir=CFG['OUTDIR'],
            instances_override=synth_doe_instances,
            metric='lns_improvement',
            json_suffix='_synthetique',
        )
        _sec("RÉCAPITULATIF FINAL DU PLAN TAGUCHI (SYNTHÉTIQUE)")
        if ts_best_run:
            print("  Meilleurs hyperparamètres trouvés (calibrés sur synthétique) :")
            for fn, _ in TAGUCHI_FACTORS:
                print(f"    {fn:<18s} = {ts_best_run[fn]}")
            print(f"  Amélioration moyenne vs LNS obtenue : {-ts_best_run['mean_gap_%']:.2f}%")
            print(f"  Ratio S/N                           : {ts_best_run['sn_ratio']:.2f}")
        print(f"\n  Meilleure combinaison (JSON) : "
              f"{os.path.join(CFG['OUTDIR'], 'taguchi_meilleure_combinaison_synthetique.json')}")
        print(f"\n  ✓ Plan Taguchi (synthétique) terminé en {time.time()-t_global0:.1f}s "
              f"({(time.time()-t_global0)/3600:.2f}h).")

        # ── ENCHAÎNEMENT AUTOMATIQUE → VALIDATION FINALE AVEC LES BEST HYPERPARAMÈTRES ──
        # Entraînement 80 instances synthétiques + test gelé sur les 56 Solomon,
        # en un seul lancement, sans intervention manuelle.
        if ts_best_run:
            _sec("ENCHAÎNEMENT AUTOMATIQUE → VALIDATION FINALE")
            CFG['USE_TAGUCHI_BEST'] = True
            # Repart d'un modèle neuf avec les NOUVEAUX hyperparamètres —
            # ne jamais réutiliser un .pth calibré avec d'anciens réglages ici.
            CFG['REUSE_EXISTING_SYNTH_MODEL'] = False
            run_final_validation()
        else:
            print("  ⚠ Aucune combinaison valide trouvée par le plan Taguchi — "
                  "validation finale non lancée.")

    elif CFG['TAGUCHI']['ENABLE']:
        # ── MODE CALIBRATION TAGUCHI — SOLOMON (⚠ HISTORIQUE, NON RECOMMANDÉ) ──
        # Volontairement PAS d'enchaînement automatique ici : ce mode calibre
        # directement sur le jeu de test (fuite train/test), la combinaison
        # trouvée ne doit pas servir à une validation train/test disjointe.
        print(f"\n{'═'*72}\n  ⚠ MODE PLAN D'EXPÉRIENCE TAGUCHI — SOLOMON (mode historique)\n"
              f"  Ce mode entraîne ET évalue directement sur les 56 instances Solomon :\n"
              f"  la combinaison retournée aura VU le jeu de test et ne doit PAS être\n"
              f"  utilisée comme hyperparamètres d'une validation train/test disjointe.\n"
              f"  Préférez CFG['TAGUCHI_SYNTH']['ENABLE']=True (calibration synthétique).\n"
              f"{'═'*72}")
        tg = CFG['TAGUCHI']
        (tg_results, tg_effects_gap, tg_effects_sn,
         tg_anova, tg_best_run) = run_taguchi_doe(
            factors=TAGUCHI_FACTORS,
            outer_instance_names=tg['OUTER_INSTANCE_NAMES'],
            episodes_per_instance_doe=tg['EPISODES_PER_INSTANCE_DOE'],
            outdir=CFG['OUTDIR'],
            instances_override=None,
            metric='bks_gap',
            json_suffix='',
        )
        _sec("RÉCAPITULATIF FINAL DU PLAN TAGUCHI (SOLOMON — HISTORIQUE)")
        if tg_best_run:
            print("  Meilleurs hyperparamètres trouvés (⚠ calibrés sur Solomon — fuite) :")
            for fn, _ in TAGUCHI_FACTORS:
                print(f"    {fn:<18s} = {tg_best_run[fn]}")
            print(f"  Gap moyen obtenu : {tg_best_run['mean_gap_%']:+.2f}%")
            print(f"  Ratio S/N        : {tg_best_run['sn_ratio']:.2f}")
        print(f"\n  ✓ Plan Taguchi (Solomon) terminé en {time.time()-t_global0:.1f}s "
              f"({(time.time()-t_global0)/3600:.2f}h). Pas d'enchaînement automatique "
              f"(mode avec fuite) — repassez CFG['TAGUCHI']['ENABLE']=False et relancez "
              f"manuellement si vous souhaitez tout de même l'utiliser, à titre de "
              f"comparaison uniquement.\n")

    else:
        # ── MODE VALIDATION SEULE (comportement normal, sans calibration) ──
        run_final_validation()


════════════════════════════════════════════════════════════════════════
  MODE ENTRAÎNEMENT/TEST DISJOINT
  ENTRAÎNEMENT = 80 instances SYNTHÉTIQUES générées | TEST = 56 instances Solomon officielles (gelé)
════════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════════
  DIAGNOSTIC — LICENCE GUROBI
════════════════════════════════════════════════════════════════════════

  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée (erreur 10010 : Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information)
    → Licence Community/gratuite, plafonnée à ~2000 variables et ~2000 contraintes.
    → Le repair Gurobi de la Situation 1 (jusqu'à ~15000 variables selon K_DESTROY=24 et le nombre de véhicules) peut dépasser cette limite sur certaines instances — un repli automatique vers le repair glouton est déjà en place, donc pas de plantage, mais un usage de Gurobi moins fréquent que

C:\Users\amelb\AppData\Local\Temp\ipykernel_18372\3280002955.py:956: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location='cpu')


    [DQN] Modèle rechargé ← C:\Users\amelb\PhD Thesis\outputs\model\dqn_vrptw_synth_trained.pth

════════════════════════════════════════════════════════════════════════
  ÉTAPE 2 — CHARGEMENT DES INSTANCES SOLOMON (TEST)
════════════════════════════════════════════════════════════════════════
  Chargé : C101     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C102     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C103     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C104     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C105     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C106     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C107     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C108     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C109     type=C   clients=100 véhicule

## DQN + LNS + GUROBI + I1 / Entraînement : EPISODES_PER_INSTANCE = 20 / Test : EPISODES_PER_INSTANCE_TEST = 10

In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON    ║
║                          + GAP PAR RAPPORT AU BKS                           ║
║                                                                              ║
║  Version corrigée : reprend toutes les visualisations de la variante        ║
║  précédente (courbes de convergence, tracés des routes avant/après,        ║
║  graphique dédié gap % vs BKS, tableau récapitulatif véhicules/distance)   ║
║  MAIS applique désormais un vrai protocole ENTRAÎNEMENT / TEST DISJOINT,   ║
║  en un seul lancement, sans aucune intervention manuelle :                 ║
║                                                                              ║
║   ÉTAPE 1 (ENTRAÎNEMENT) — génère des instances SYNTHÉTIQUES (reprises de   ║
║      generate_instance() de main_vrptw_comparaison_gurobi.py / "Code 1")    ║
║      et entraîne le DQN dessus (poids mis à jour via store/learn/decay).    ║
║                                                                              ║
║   ÉTAPE 2 (TEST) — charge les 56 instances Solomon OFFICIELLES et évalue    ║
║      le même agent, GELÉ (aucun store/learn/decay_epsilon, uniquement      ║
║      select_action en pur mode inférence), sur ce jeu jamais vu à          ║
║      l'entraînement. Toutes les courbes/tracés/tableaux ci-dessous portent  ║
║      désormais sur ce TEST gelé, pas sur les résultats d'entraînement.     ║
║                                                                              ║
║  Situation 1 = LNS + DQN(entraîné sur synthétique) + Gurobi, ÉVALUÉ sur     ║
║                Solomon (test)                                → gap vs BKS   ║
║  (La "Situation 2 = Gurobi seul sur l'instance complète" reste retirée,     ║
║   à la demande, car sans limite de temps elle est beaucoup trop longue      ║
║   sur des instances à 100 clients avec fenêtres de temps serrées.)          ║
║                                                                              ║
║  Visualisations produites (toutes sur le TEST gelé, sauf la 1ère) :        ║
║    [A] Convergence de l'ENTRAÎNEMENT (synthétique) : reward global +       ║
║        amélioration % par instance synthétique                            ║
║    [B] Convergence détaillée (instances Solomon exemples, TEST) : coût     ║
║        final/meilleur par épisode local + trajectoire de coût             ║
║    [C] Tracés des routes AVANT (LNS) / APRÈS (DQN gelé) pour des           ║
║        instances Solomon exemples                                         ║
║    [D] Tableau récapitulatif : Instance / Nb véhicules / Distance / Gap %  ║
║        (TEST Solomon vs BKS)                                              ║
║    [E] Graphique dédié gap % vs BKS par instance + gap moyen              ║
║    [F] Diagnostic automatique de licence Gurobi (taille limitée ou non)   ║
║                                                                              ║
║  Aucun rappel manuel de "Code 1" n'est nécessaire : la génération +        ║
║  l'entraînement synthétique sont réintégrés ici (MODULE 1bis) et           ║
║  s'exécutent automatiquement avant le chargement de Solomon. Voir le flag  ║
║  optionnel CFG['REUSE_EXISTING_SYNTH_MODEL'] plus bas si vous préférez     ║
║  recharger un .pth déjà entraîné au lieu de ré-entraîner à chaque run.     ║
║                                                                              ║
║  Par défaut, INSTANCE_FILES contient toutes les instances *.txt trouvées   ║
║  dans outputs/Solomon/. Pour se limiter à 6 instances représentatives      ║
║  (une par groupe C1/C2/R1/R2/RC1/RC2), décommentez le bloc plus bas.       ║
║                                                                              ║
║  Exécution :  python main_vrptw_synth_train_solomon_convergence.py         ║
║                                                                              ║
║  CORRECTIONS APPORTÉES SUR CETTE VERSION :                                 ║
║   1. SOLOMON_DIR pointait vers un chemin Google Colab                      ║
║      ('/content/sample_data/Solomon') — corrigé pour pointer vers          ║
║      outputs/Solomon (dossier local Windows).                              ║
║   2. Le mode CFG['TAGUCHI_SYNTH']['ENABLE']=True s'arrêtait après la       ║
║      calibration et exigeait de relancer le script à la main pour la      ║
║      validation finale. Il enchaîne désormais AUTOMATIQUEMENT, dans le    ║
║      même lancement, sur l'entraînement complet (80 instances               ║
║      synthétiques) + le test gelé sur les 56 Solomon, en appliquant       ║
║      automatiquement la meilleure combinaison d'hyperparamètres trouvée.  ║
║      Voir run_final_validation() et le bloc __main__ tout en bas.         ║
║   3. Construction de la solution initiale remplacée par l'heuristique     ║
║      séquentielle I1 (Solomon, 1987) — cf. _build_initial_i1(), MODULE 2. ║
║      Remplace l'ancienne insertion gloutonne triée par earliest-time      ║
║      (conservée sous _build_initial_greedy_earliest, non appelée) comme   ║
║      point de départ du LNS, pour ENTRAÎNEMENT et TEST. Les meilleurs     ║
║      hyperparamètres Taguchi (LNS_ITER=100, K_DESTROY_RATIO=0.24,         ║
║      GUROBI_TLIMIT=2.0, GUROBI_GAP=0.05, MAX_STEPS=15, EPS_DECAY=0.99,    ║
║      VEHICLE_PENALTY=0.0) sont reportés en dur dans CFG, et le modèle     ║
║      DQN est réentraîné de zéro (REUSE_EXISTING_SYNTH_MODEL=False) car    ║
║      les instances vues à l'entraînement ont désormais une structure de   ║
║      routes de départ différente (voir CFG['I1_MU'/'I1_LAMBDA'/etc.]).    ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import os, sys, json, time, copy, random, csv, glob, itertools
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from contextlib import contextmanager
import torch
import torch.nn as nn
import torch.optim as optim
import gurobipy as gp
from gurobipy import GRB
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG GLOBALE
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    # ── ÉTAPE 1 : ENTRAÎNEMENT sur instances SYNTHÉTIQUES (jamais Solomon) ──
    N_SYNTH_INSTANCES          = 80,     # comme les 80 instances de Code 1
    SYNTH_N_CUSTOMERS          = 100,
    SYNTH_N_VEHICLES           = None,   # None → calculé depuis la demande (cf. generate_instance)
    SYNTH_VEHICLE_SAFETY_FACTOR= 1.5,
    SYNTH_CAPACITY             = 200,
    SYNTH_HORIZON              = 1000,
    EPISODES_PER_INSTANCE      = 20,     # épisodes d'ENTRAÎNEMENT par instance synthétique
    # Optionnel : True → recharge un .pth déjà entraîné sur du synthétique
    # (voir MODEL_FILE) au lieu de régénérer + ré-entraîner. False (par
    # défaut) → tout se fait automatiquement dans CE run, sans intervention
    # manuelle, aucun besoin de relancer "Code 1" séparément.
    # ⚠ Remis à False ici : la construction initiale change (heuristique I1
    # de Solomon 1987, cf. MODULE 2 plus bas, en remplacement de l'ancienne
    # insertion gloutonne triée par earliest-time) → les instances vues à
    # l'entraînement ont une structure de routes de départ différente, donc
    # on RÉENTRAÎNE le DQN de zéro plutôt que de recharger l'ancien .pth
    # (calibré sur l'ancienne construction).
    REUSE_EXISTING_SYNTH_MODEL = False,
    # False ici : les hyperparamètres ci-dessous (LNS_ITER, K_DESTROY_RATIO,
    # GUROBI_TLIMIT, GUROBI_GAP, MAX_STEPS, EPS_DECAY, VEHICLE_PENALTY) sont
    # déjà renseignés EN DUR avec la meilleure combinaison trouvée par le
    # plan Taguchi synthétique lors d'un run précédent — pas besoin de les
    # recharger depuis taguchi_meilleure_combinaison_synthetique.json à
    # l'exécution. Garde le comportement déterministe (n'importe pas le
    # contenu du JSON, qui a été calibré avec l'ANCIENNE heuristique de
    # construction initiale, avant l'ajout d'I1).
    USE_TAGUCHI_BEST            = False,
    # ── ÉTAPE 2 : TEST gelé sur les 56 instances Solomon officielles ───────
    EPISODES_PER_INSTANCE_TEST = 10,     # épisodes d'ÉVALUATION (aucun apprentissage) par instance Solomon
    # VEHICLE_SAFETY_FACTOR = None : on utilise la flotte OFFICIELLE de chaque
    # instance, pas une borne calculée sur la demande. Sur R101/RC101
    # (fenêtres de temps serrées), le BKS utilise 19 et 14 véhicules alors
    # que la demande seule suggérerait ~12-14 : une borne basée uniquement
    # sur la demande sous-estime le besoin réel et peut rendre le MILP
    # infaisable. La flotte officielle est plus grande (MILP plus lourd)
    # mais garantit qu'une solution proche du BKS reste atteignable.
    VEHICLE_SAFETY_FACTOR  = None,
    # ── Hyperparamètres LNS/DQN/Gurobi : meilleure combinaison trouvée par
    # le plan d'expérience Taguchi synthétique (fichier
    # taguchi_meilleure_combinaison_synthetique.json), reportée ici EN DUR
    # (voir USE_TAGUCHI_BEST=False ci-dessus). ────────────────────────────
    LNS_ITER                = 100,
    LNS_K_RATIO            = 0.20,
    K_DESTROY_RATIO        = 0.24,
    MAX_STEPS               = 15,
    # repair par bouts (Situation 1, appelé à CHAQUE step du LNS/DQN — voir
    # check_gurobi_license() plus bas pour l'impact d'une licence à taille
    # limitée). None = aucune limite de temps (supporté par le code).
    GUROBI_TLIMIT            = 2.0,
    GUROBI_GAP               = 0.05,
    HIDDEN                   = 64,
    LR                       = 1e-3,
    GAMMA                    = 0.95,
    EPS_START                = 1.0,
    EPS_END                  = 0.05,
    EPS_DECAY                = 0.99,  # best Taguchi — voir l'écrasement explicite juste après CFG plus bas
    BUFFER_SIZE               = 5000,
    BATCH_SIZE                = 32,
    TARGET_UPDATE              = 10,
    # ── Instances exemples (pour courbes détaillées + routes, TEST/Solomon) ─
    # Noms recherchés en priorité (représentatifs des 6 groupes Solomon) ;
    # si un nom n'est pas présent dans INSTANCE_FILES, il est ignoré et
    # remplacé automatiquement par le 1er type disponible.
    EXAMPLE_NAMES              = ['C101', 'R101', 'RC101'],
    # ── Pénalité par véhicule (repair MILP) ────────────────────────
    # 0.0 = comportement d'origine (aucune pénalité, Gurobi peut ouvrir
    # librement des véhicules supplémentaires à chaque repair). Une valeur
    # > 0 ajoute un coût fixe par véhicule utilisé dans l'objectif du MILP
    # de repair, pour limiter l'inflation du nombre de véhicules observée
    # dans la validation vs BKS. C'est aussi l'un des facteurs calibrés par
    # le plan d'expérience Taguchi (MODULE 10 plus bas) — calibré
    # UNIQUEMENT sur du synthétique (CFG['TAGUCHI_SYNTH']), jamais sur
    # Solomon, pour ne pas faire fuiter d'information du test set vers le
    # choix des hyperparamètres.
    VEHICLE_PENALTY            = 0.0,
    # ── Heuristique de construction initiale : I1 (Solomon, 1987) ──────
    # Remplace l'ancienne construction gloutonne triée par earliest-time
    # (_build_initial, conservée plus bas pour référence mais plus appelée).
    # Paramètres classiques de Solomon (1987) : mu=1 (poids de la distance
    # directe i→j dans le coût d'insertion c11), lambda=2 (poids de
    # l'éloignement au dépôt dans le critère de sélection c2 — favorise les
    # clients "difficiles" excentrés), alpha1=1/alpha2=0 (c1 = c11 pur,
    # sans le terme de décalage temporel c12 — paramétrage le plus courant
    # dans la littérature ; augmenter alpha2 pénalise davantage les
    # insertions qui retardent fortement le client suivant).
    I1_MU                       = 1.0,
    I1_LAMBDA                   = 2.0,
    I1_ALPHA1                   = 1.0,
    I1_ALPHA2                   = 0.0,
    OUTDIR                        = os.path.join(os.getcwd(), 'outputs'),
    MODELDIR                       = os.path.join(os.getcwd(), 'outputs', 'model'),
    MODEL_FILE                      = 'dqn_vrptw_synth_trained.pth',
)
CFG['LNS_K'] = 20   # ~20% de 100 clients, cohérent avec LNS_K_RATIO
CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))  # 24, cohérent avec K_DESTROY_RATIO
# ── Recalibrage automatique de EPS_DECAY : DÉSACTIVÉ ────────────────────
# À l'origine, ce bloc recalculait EPS_DECAY pour que epsilon décroisse de
# EPS_START à EPS_END sur toute la durée de l'entraînement (utile quand
# EPS_DECAY n'était pas calibré). Ici, EPS_DECAY=0.99 vient directement de
# la meilleure combinaison trouvée par le plan Taguchi synthétique (cf.
# CFG plus haut) — un recalcul automatique l'écraserait. On le laisse donc
# tel quel ; ce bloc est conservé en commentaire pour référence/mémoire si
# jamais EPS_DECAY doit être re-dérivé automatiquement dans une variante
# future du script.
# _n_decays_total = max(1, CFG['N_SYNTH_INSTANCES'] * CFG['EPISODES_PER_INSTANCE'])
# CFG['EPS_DECAY'] = (CFG['EPS_END'] / CFG['EPS_START']) ** (1.0 / _n_decays_total)
os.makedirs(CFG['OUTDIR'],   exist_ok=True)
os.makedirs(CFG['MODELDIR'], exist_ok=True)
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG DU PLAN D'EXPÉRIENCE TAGUCHI (calibration des hyperparamètres)
# ══════════════════════════════════════════════════════════════════════════════
# Deux modes de calibration disponibles (voir plus bas pour le mode
# TAGUCHI_SYNTH, recommandé, qui enchaîne désormais automatiquement sur la
# validation finale). Activez explicitement l'UN DES DEUX (pas les deux en
# même temps) :
#
#  • CFG['TAGUCHI_SYNTH']['ENABLE'] = True   (RECOMMANDÉ)
#    Calibre en s'entraînant UNIQUEMENT sur des instances SYNTHÉTIQUES
#    générées à la volée (jamais Solomon). Métrique de classement :
#    amélioration % vs coût LNS initial (pas de BKS requis). La meilleure
#    combinaison trouvée n'a jamais "vu" les 56 instances Solomon → aucune
#    fuite d'hyperparamètres vers le jeu de test, contrairement au mode
#    historique ci-dessous. Une fois la calibration terminée, le script
#    enchaîne AUTOMATIQUEMENT sur la validation finale (entraînement 80
#    instances + test gelé sur les 56 Solomon) avec ces hyperparamètres.
#
#  • CFG['TAGUCHI']['ENABLE'] = True   (⚠ HISTORIQUE — NON RECOMMANDÉ)
#    Calibre en s'entraînant DIRECTEMENT sur les 56 instances Solomon (la
#    matrice externe = le jeu de test lui-même) et classe les combinaisons
#    par leur gap % vs BKS sur ces mêmes instances. C'est exactement le
#    protocole "Run 16" identifié comme fuite d'hyperparamètres (on choisit
#    les hyperparamètres en fonction de la performance sur le test) —
#    conservé uniquement pour comparaison / compatibilité avec d'anciens
#    résultats, PAS pour produire les hyperparamètres d'une validation
#    finale train/test disjointe. Ce mode n'enchaîne PAS automatiquement
#    sur la validation finale (voir le bloc __main__ tout en bas).
CFG['TAGUCHI'] = dict(
    ENABLE                     = False,  # ⚠ historique — fuite vers Solomon (test set), voir ci-dessus
    EPISODES_PER_INSTANCE_DOE  = 5,
    OUTER_INSTANCE_NAMES       = 'ALL',  # 'ALL' = les 56 instances Solomon comme matrice externe
)
CFG['TAGUCHI_SYNTH'] = dict(
    # False ici : la calibration a déjà été faite lors d'un run précédent
    # (le JSON taguchi_meilleure_combinaison_synthetique.json existe déjà).
    # Repasser à True relancerait tout le plan Taguchi (18 runs) pour rien —
    # CFG['USE_TAGUCHI_BEST']=True ci-dessus suffit à réutiliser ses résultats.
    ENABLE                     = False,  # True = lance le plan Taguchi calibré sur synthétique (recommandé)
    # Volume d'instances synthétiques utilisé comme matrice externe pendant
    # CE plan de calibration (indépendant de N_SYNTH_INSTANCES du protocole
    # d'entraînement final — un sous-ensemble suffit ici, l'objectif est de
    # classer les 18 combinaisons, pas d'entraîner le modèle définitif).
    N_INSTANCES_DOE            = 24,   # 8 par type C/R/RC
    EPISODES_PER_INSTANCE_DOE  = 5,
)
# Facteurs de contrôle et leurs niveaux (plan Taguchi L18 : 7 facteurs à 3
# niveaux, 18 combinaisons au lieu des 3^7 = 2187 combinaisons possibles).
TAGUCHI_FACTORS = [
    ('LNS_ITER',         [30, 60, 100]),
    ('K_DESTROY_RATIO',  [0.15, 0.20, 0.24]),
    ('GUROBI_TLIMIT',    [1.0, 2.0, 3.0]),
    ('GUROBI_GAP',       [0.05, 0.03, 0.01]),
    ('MAX_STEPS',        [10, 15, 20]),
    ('EPS_DECAY',        [0.90, 0.95, 0.99]),
    ('VEHICLE_PENALTY',  [0.0, 300.0, 800.0]),
]
GROUP_ORDER = ['C1', 'C2', 'R1', 'R2', 'RC1', 'RC2']
ACTION_NAMES = {
    0: "Destruction spatiale",
    1: "Destruction temporelle",
    2: "Destruction par coût élevé",
    3: "Destruction aléatoire",
}
# ── Dossier contenant les instances Solomon officielles (extraites du zip) ──
# CORRIGÉ : pointait vers '/content/sample_data/Solomon' (chemin Google
# Colab, inexistant sur une machine locale) — repointé vers outputs/Solomon.
SOLOMON_DIR = os.path.join(CFG['OUTDIR'], 'Solomon')
# Toutes les instances *.txt trouvées (décommentez le bloc ci-dessous pour
# revenir à seulement 6 instances représentatives, une par groupe
# C1/C2/R1/R2/RC1/RC2, afin de garder un temps d'exécution raisonnable).
INSTANCE_FILES = sorted(glob.glob(os.path.join(SOLOMON_DIR, '*.txt')))
# INSTANCE_FILES = [
#     os.path.join(SOLOMON_DIR, 'c101.txt'),
#     os.path.join(SOLOMON_DIR, 'c201.txt'),
#     os.path.join(SOLOMON_DIR, 'r101.txt'),
#     os.path.join(SOLOMON_DIR, 'r201.txt'),
#     os.path.join(SOLOMON_DIR, 'rc101.txt'),
#     os.path.join(SOLOMON_DIR, 'rc201.txt'),
# ]
# ── BKS officiel complet, 56 instances (source : SINTEF / Solomon 1987) ──────
BKS_REFERENCE = {
    'C101': {'vehicles': 10, 'distance': 828.94}, 'C102': {'vehicles': 10, 'distance': 828.94},
    'C103': {'vehicles': 10, 'distance': 828.06}, 'C104': {'vehicles': 10, 'distance': 824.78},
    'C105': {'vehicles': 10, 'distance': 828.94}, 'C106': {'vehicles': 10, 'distance': 828.94},
    'C107': {'vehicles': 10, 'distance': 828.94}, 'C108': {'vehicles': 10, 'distance': 828.94},
    'C109': {'vehicles': 10, 'distance': 828.94},
    'C201': {'vehicles': 3, 'distance': 591.56}, 'C202': {'vehicles': 3, 'distance': 591.56},
    'C203': {'vehicles': 3, 'distance': 591.17}, 'C204': {'vehicles': 3, 'distance': 590.60},
    'C205': {'vehicles': 3, 'distance': 588.88}, 'C206': {'vehicles': 3, 'distance': 588.49},
    'C207': {'vehicles': 3, 'distance': 588.29}, 'C208': {'vehicles': 3, 'distance': 588.32},
    'R101': {'vehicles': 19, 'distance': 1650.80}, 'R102': {'vehicles': 17, 'distance': 1486.12},
    'R103': {'vehicles': 13, 'distance': 1292.68}, 'R104': {'vehicles': 9,  'distance': 1007.31},
    'R105': {'vehicles': 14, 'distance': 1377.11}, 'R106': {'vehicles': 12, 'distance': 1252.03},
    'R107': {'vehicles': 10, 'distance': 1104.66}, 'R108': {'vehicles': 9,  'distance': 960.88},
    'R109': {'vehicles': 11, 'distance': 1194.73}, 'R110': {'vehicles': 10, 'distance': 1118.84},
    'R111': {'vehicles': 10, 'distance': 1096.72}, 'R112': {'vehicles': 9,  'distance': 982.14},
    'R201': {'vehicles': 4, 'distance': 1252.37}, 'R202': {'vehicles': 3, 'distance': 1191.70},
    'R203': {'vehicles': 3, 'distance': 939.50},  'R204': {'vehicles': 2, 'distance': 825.52},
    'R205': {'vehicles': 3, 'distance': 994.43},  'R206': {'vehicles': 3, 'distance': 906.14},
    'R207': {'vehicles': 2, 'distance': 890.61},  'R208': {'vehicles': 2, 'distance': 726.82},
    'R209': {'vehicles': 3, 'distance': 909.16},  'R210': {'vehicles': 3, 'distance': 939.37},
    'R211': {'vehicles': 2, 'distance': 885.71},
    'RC101': {'vehicles': 14, 'distance': 1696.95}, 'RC102': {'vehicles': 12, 'distance': 1554.75},
    'RC103': {'vehicles': 11, 'distance': 1261.67}, 'RC104': {'vehicles': 10, 'distance': 1135.48},
    'RC105': {'vehicles': 13, 'distance': 1629.44}, 'RC106': {'vehicles': 11, 'distance': 1424.73},
    'RC107': {'vehicles': 11, 'distance': 1230.48}, 'RC108': {'vehicles': 10, 'distance': 1139.82},
    'RC201': {'vehicles': 4, 'distance': 1406.94}, 'RC202': {'vehicles': 3, 'distance': 1365.65},
    'RC203': {'vehicles': 3, 'distance': 1049.62}, 'RC204': {'vehicles': 3, 'distance': 798.46},
    'RC205': {'vehicles': 4, 'distance': 1297.65}, 'RC206': {'vehicles': 3, 'distance': 1146.32},
    'RC207': {'vehicles': 3, 'distance': 1061.14}, 'RC208': {'vehicles': 3, 'distance': 828.14},
}
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1  —  LECTURE DES INSTANCES SOLOMON OFFICIELLES  (jeu de TEST)
# ══════════════════════════════════════════════════════════════════════════════
def euclidean(a, b):
    return float(np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2))
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1bis  —  GÉNÉRATEUR D'INSTANCES SYNTHÉTIQUES  (jeu d'ENTRAÎNEMENT)
#  Repris tel quel de main_vrptw_comparaison_gurobi.py ("Code 1") : mêmes
#  fonctions _coords_C/_coords_R/_coords_RC et generate_instance(), pour que
#  les instances d'entraînement soient générées exactement comme dans Code 1.
# ══════════════════════════════════════════════════════════════════════════════
def _coords_C(n, rng):
    n_cl = max(3, n // 10)
    centers = rng.uniform(15, 85, (n_cl, 2))
    pts = []
    for i in range(n):
        p = rng.normal(centers[i % n_cl], 8)
        pts.append(np.clip(p, 1, 99))
    return np.array(pts)
def _coords_R(n, rng):
    return rng.uniform(1, 99, (n, 2))
def _coords_RC(n, rng):
    h = n // 2
    return np.vstack([_coords_C(h, rng), _coords_R(n - h, rng)])
def generate_instance(n_customers=50, n_vehicles=None, capacity=200,
                      type_='C', seed=42, horizon=1000,
                      vehicle_safety_factor=1.5):
    rng     = np.random.RandomState(seed)
    depot   = np.array([50.0, 50.0])
    gen_fn  = {'C': _coords_C, 'R': _coords_R, 'RC': _coords_RC}[type_]
    coords  = gen_fn(n_customers, rng)
    max_dem = max(1, capacity // 5)
    demands = rng.randint(1, max_dem + 1, n_customers)
    svc     = rng.randint(10, 31, n_customers)
    if n_vehicles is None:
        min_vehicles = int(np.ceil(demands.sum() / capacity))
        n_vehicles   = max(2, int(np.ceil(min_vehicles * vehicle_safety_factor)))
    customers = []
    for i in range(n_customers):
        d        = euclidean(depot, coords[i])
        earliest = max(0.0, round(d + rng.uniform(-15, 15), 1))
        delta    = rng.uniform(30, 80)
        latest   = min(float(horizon), round(earliest + delta, 1))
        latest   = max(latest, earliest + svc[i] + 5)
        latest   = min(latest, float(horizon))
        if d > latest:
            earliest = round(d, 1)
            latest   = min(float(horizon), round(earliest + delta, 1))
            latest   = max(latest, earliest + svc[i] + 5)
        if d + svc[i] + d > horizon:
            svc[i] = max(5, int(horizon - 2 * d - 5))
        customers.append({
            'id':           i + 1,
            'x':            float(round(coords[i][0], 2)),
            'y':            float(round(coords[i][1], 2)),
            'demand':       int(demands[i]),
            'earliest':     float(earliest),
            'latest':       float(latest),
            'service_time': int(svc[i]),
        })
    return {
        'depot': {
            'id': 0, 'x': 50.0, 'y': 50.0, 'demand': 0,
            'earliest': 0.0, 'latest': float(horizon), 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    capacity,
        'horizon':     horizon,
        'type':        type_,
        'seed':        seed,
        'n_customers': n_customers,
    }
def generate_synthetic_training_set(cfg, verbose=True):
    """
    Génère les instances synthétiques d'ENTRAÎNEMENT en cyclant les 3 types
    Solomon C / R / RC (même logique que train() dans Code 1 : type =
    ['C','R','RC'][i % 3], seed = i*7+1). Ces instances ne recouvrent JAMAIS
    les 56 instances Solomon officielles utilisées plus bas comme jeu de
    TEST : le DQN ne les voit donc jamais avant l'évaluation gelée.
    Chaque instance reçoit en plus une clé 'name' (ex. 'SYN-C001') pour
    rester compatible avec les fonctions d'entraînement/affichage qui
    utilisent inst['name'].
    """
    types = ['C', 'R', 'RC']
    instances = []
    if verbose:
        _sec(f"ÉTAPE 1 — GÉNÉRATION DE {cfg['N_SYNTH_INSTANCES']} INSTANCES "
             f"SYNTHÉTIQUES (ENTRAÎNEMENT)")
    for i in range(cfg['N_SYNTH_INSTANCES']):
        t    = types[i % 3]
        seed = i * 7 + 1
        inst = generate_instance(
            n_customers = cfg['SYNTH_N_CUSTOMERS'],
            n_vehicles  = cfg['SYNTH_N_VEHICLES'],
            capacity    = cfg['SYNTH_CAPACITY'],
            horizon     = cfg['SYNTH_HORIZON'],
            type_       = t,
            seed        = seed,
            vehicle_safety_factor = cfg['SYNTH_VEHICLE_SAFETY_FACTOR'],
        )
        inst['name'] = f"SYN-{t}{i+1:03d}"
        instances.append(inst)
        if verbose:
            print(f"  Instance {i+1:03d}/{cfg['N_SYNTH_INSTANCES']} | {inst['name']:<10} "
                  f"type={t:<3} | seed={seed} | véhicules={inst['n_vehicles']}")
    return instances
def parse_solomon_instance(filepath, vehicle_safety_factor=None):
    """
    Parse un fichier d'instance Solomon au format texte officiel :
        <NOM>
        VEHICLE
        NUMBER  CAPACITY
        <n>     <cap>
        CUSTOMER
        CUST NO.  XCOORD.  YCOORD.  DEMAND  READY TIME  DUE DATE  SERVICE TIME
        0  ...   (dépôt)
        1  ...
        ...
    Retourne un dict compatible avec le reste du pipeline.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = [l.rstrip('\n') for l in f]
    name = lines[0].strip()
    idx_veh = next(i for i, l in enumerate(lines) if l.strip() == 'VEHICLE')
    veh_number, veh_capacity = map(int, lines[idx_veh + 2].split())
    idx_cust = next(i for i, l in enumerate(lines) if l.strip() == 'CUSTOMER')
    data_lines = [l for l in lines[idx_cust + 2:] if l.strip()]
    rows = [list(map(float, l.split())) for l in data_lines]
    depot_row = rows[0]
    customers = []
    for r in rows[1:]:
        cid, x, y, dem, ready, due, svc = r
        customers.append({
            'id':           int(cid),
            'x':            x,
            'y':            y,
            'demand':       int(dem),
            'earliest':     ready,
            'latest':       due,
            'service_time': int(svc),
        })
    name_upper = name.upper()
    type_ = ''.join(ch for ch in name_upper if ch.isalpha())   # 'C101' -> 'C', 'RC101' -> 'RC'
    n_vehicles = veh_number
    if vehicle_safety_factor is not None:
        total_demand = sum(c['demand'] for c in customers)
        min_veh      = int(np.ceil(total_demand / veh_capacity))
        n_vehicles   = min(veh_number, max(2, int(np.ceil(min_veh * vehicle_safety_factor))))
    return {
        'depot': {
            'id': 0, 'x': depot_row[1], 'y': depot_row[2], 'demand': 0,
            'earliest': depot_row[4], 'latest': depot_row[5], 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    veh_capacity,
        'horizon':     depot_row[5],
        'type':        type_,
        'seed':        name_upper,
        'n_customers': len(customers),
        'name':        name_upper,          # nom officiel normalisé (ex. 'C101')
    }
def load_all_instances(files, vehicle_safety_factor=None):
    if not files:
        raise FileNotFoundError(
            f"Aucun fichier d'instance trouvé dans {SOLOMON_DIR}. "
            f"Vérifiez que le dossier existe et contient les .txt Solomon."
        )
    instances = []
    for fp in files:
        inst = parse_solomon_instance(fp, vehicle_safety_factor=vehicle_safety_factor)
        instances.append(inst)
        print(f"  Chargé : {inst['name']:<8} type={inst['type']:<3} "
              f"clients={inst['n_customers']} véhicules={inst['n_vehicles']} "
              f"capacité={inst['capacity']} horizon={inst['horizon']}")
    return instances
def pick_example_indices(instances, preferred_names=None, max_examples=3):
    """
    Sélectionne des instances "exemples" pour les courbes détaillées et les
    tracés de routes. Cherche d'abord les noms préférés (ex. C101/R101/RC101).
    Si absents, retombe sur la 1ère instance de chaque type détecté, puis sur
    les toutes premières instances de la liste.
    """
    preferred_names = preferred_names or []
    by_name = {inst['name']: i for i, inst in enumerate(instances)}
    idxs = [by_name[nm] for nm in preferred_names if nm in by_name]
    if not idxs:
        seen_types = {}
        for i, inst in enumerate(instances):
            if inst['type'] not in seen_types:
                seen_types[inst['type']] = i
            if len(seen_types) >= max_examples:
                break
        idxs = list(seen_types.values())
    if not idxs:
        idxs = list(range(min(max_examples, len(instances))))
    return idxs[:max_examples]
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 2  —  SOLUTION INITIALE PAR LNS  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
def _node_xy(nid, instance):
    if nid == 0:
        return instance['depot']['x'], instance['depot']['y']
    c = instance['customers'][nid - 1]
    return c['x'], c['y']
def _node_info(nid, instance):
    if nid == 0:
        return instance['depot']
    return instance['customers'][nid - 1]
def compute_cost(solution, instance):
    depot = (instance['depot']['x'], instance['depot']['y'])
    total = 0.0
    for route in solution:
        if not route:
            continue
        prev = depot
        for cid in route:
            cur   = _node_xy(cid, instance)
            total += euclidean(prev, cur)
            prev  = cur
        total += euclidean(prev, depot)
    return total
def _route_feasible(route, instance):
    if not route:
        return True
    cap  = instance['capacity']
    load = sum(_node_info(c, instance)['demand'] for c in route)
    if load > cap:
        return False
    depot = instance['depot']
    t     = 0.0
    px, py = depot['x'], depot['y']
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        if t > info['latest']:
            return False
        t      = max(t, info['earliest']) + info['service_time']
        px, py = cx, cy
    t += euclidean((px, py), (depot['x'], depot['y']))
    return t <= depot['latest']
def is_feasible(solution, instance):
    return all(_route_feasible(r, instance) for r in solution)
def solution_stats(solution, instance):
    cap        = instance['capacity']
    total_cost = compute_cost(solution, instance)
    n_vehicles = sum(1 for r in solution if r)
    loads      = []
    total_wait = 0.0
    for route in solution:
        if not route:
            continue
        load = sum(_node_info(c, instance)['demand'] for c in route)
        loads.append(load / cap)
        depot  = instance['depot']
        t      = 0.0
        px, py = depot['x'], depot['y']
        for cid in route:
            cx, cy = _node_xy(cid, instance)
            t     += euclidean((px, py), (cx, cy))
            info   = _node_info(cid, instance)
            total_wait += max(0.0, info['earliest'] - t)
            t      = max(t, info['earliest']) + info['service_time']
            px, py = cx, cy
    return {
        'total_cost':         round(total_cost, 2),
        'n_vehicles':         n_vehicles,
        'avg_load_ratio':     round(np.mean(loads) if loads else 0.0, 3),
        'total_wait_time':    round(total_wait, 2),
        'n_customers_served': sum(len(r) for r in solution),
    }
def _insertion_cost(cid, pos, route, instance):
    depot   = (instance['depot']['x'], instance['depot']['y'])
    nodes   = [0] + route + [0]
    prev_id = nodes[pos]
    next_id = nodes[pos + 1]
    px, py  = _node_xy(prev_id, instance) if prev_id != 0 else depot
    nx, ny  = _node_xy(next_id, instance) if next_id != 0 else depot
    cx, cy  = _node_xy(cid, instance)
    return (euclidean((px, py), (cx, cy)) + euclidean((cx, cy), (nx, ny))
            - euclidean((px, py), (nx, ny)))
def _try_insert(cid, route, instance):
    best_cost, best_pos = float('inf'), -1
    for pos in range(len(route) + 1):
        candidate = route[:pos] + [cid] + route[pos:]
        if _route_feasible(candidate, instance):
            cost = _insertion_cost(cid, pos, route, instance)
            if cost < best_cost:
                best_cost, best_pos = cost, pos
    return best_cost, best_pos
def _repair_greedy(removed, partial, instance):
    routes = [list(r) for r in partial]
    for cid in removed:
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _build_initial_greedy_earliest(instance):
    """
    ⚠ CONSERVÉE POUR RÉFÉRENCE UNIQUEMENT — n'est plus appelée par
    get_initial_solution() (remplacée par _build_initial_i1, l'heuristique
    I1 de Solomon (1987) ci-dessous). Construction gloutonne d'origine :
    trie les clients par earliest-time puis insère chacun au meilleur
    endroit (au sens du coût d'insertion) parmi les routes déjà ouvertes,
    ou ouvre une nouvelle route si aucune insertion n'est faisable.
    """
    customers = sorted(instance['customers'], key=lambda c: c['earliest'])
    routes    = []
    for c in customers:
        cid = c['id']
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _route_service_starts(route, instance):
    """
    Propage les temps le long d'une route (sans le dépôt) et retourne la
    liste des heures de DÉBUT DE SERVICE b_j pour chaque client de la route,
    dans l'ordre. Utilisé par l'heuristique I1 pour calculer le terme de
    décalage temporel c12 (poussée en avant de l'heure de service du client
    suivant, cf. Solomon 1987).
    """
    depot  = instance['depot']
    t      = 0.0
    px, py = depot['x'], depot['y']
    starts = []
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        t      = max(t, info['earliest'])
        starts.append(t)
        t     += info['service_time']
        px, py = cx, cy
    return starts
def _build_initial_i1(instance, mu=1.0, lam=2.0, alpha1=1.0, alpha2=0.0):
    """
    Heuristique de construction séquentielle I1 (Solomon, 1987, "Algorithms
    for the Vehicle Routing and Scheduling Problems with Time Window
    Constraints", Operations Research 35(2)). Remplace l'ancienne
    construction gloutonne triée par earliest-time (_build_initial_greedy_
    earliest ci-dessus) comme point de départ du LNS.
    Principe : construit les routes une par une. Chaque route démarre avec
    un client "graine" (le client non routé le plus éloigné du dépôt), puis
    insère itérativement, parmi tous les clients non routés et toutes les
    positions faisables de la route courante, celui qui maximise le critère
    de sélection c2(u) = lambda*d(0,u) - c1*(u) — favorisant les clients
    excentrés difficiles à insérer plus tard — où c1*(u) est le coût
    d'insertion minimal de u dans la route courante :
        c1(i,u,j) = alpha1*c11(i,u,j) + alpha2*c12(i,u,j)
        c11(i,u,j) = d(i,u) + d(u,j) - mu*d(i,j)                  (détour géométrique)
        c12(i,u,j) = b_ju - b_j                                    (décalage de service induit chez j)
    Quand plus aucun client non routé ne peut être inséré faisablement
    (capacité/fenêtres de temps, vérifiées via _route_feasible) dans la
    route courante, celle-ci est fermée et une nouvelle route est ouverte
    avec un nouveau client graine, jusqu'à ce que tous les clients soient
    routés. Paramètres mu/lambda/alpha1/alpha2 : cf. CFG['I1_MU'] etc.
    """
    def d(a, b):
        return euclidean(_node_xy(a, instance), _node_xy(b, instance))
    unrouted = [c['id'] for c in instance['customers']]
    d0 = {cid: d(0, cid) for cid in unrouted}
    routes = []
    while unrouted:
        seed = max(unrouted, key=lambda c: d0[c])
        route = [seed]
        unrouted.remove(seed)
        while True:
            times_before = _route_service_starts(route, instance)
            nodes = [0] + route + [0]
            best_u, best_pos, best_c1, best_c2 = None, None, None, -float('inf')
            for u in unrouted:
                u_best_cost, u_best_pos = None, None
                cx, cy = _node_xy(u, instance)
                info_u = _node_info(u, instance)
                for pos in range(len(route) + 1):
                    i_id, j_id = nodes[pos], nodes[pos + 1]
                    candidate = route[:pos] + [u] + route[pos:]
                    if not _route_feasible(candidate, instance):
                        continue
                    dij = d(i_id, j_id)
                    c11 = d(i_id, u) + d(u, j_id) - mu * dij
                    c12 = 0.0
                    if alpha2 != 0.0 and j_id != 0:
                        if pos == 0:
                            depart_i, px, py = 0.0, instance['depot']['x'], instance['depot']['y']
                        else:
                            info_i = _node_info(i_id, instance)
                            depart_i = times_before[pos - 1] + info_i['service_time']
                            px, py   = _node_xy(i_id, instance)
                        arrival_u = depart_i + euclidean((px, py), (cx, cy))
                        b_u       = max(arrival_u, info_u['earliest'])
                        depart_u  = b_u + info_u['service_time']
                        jx, jy    = _node_xy(j_id, instance)
                        arrival_j = depart_u + euclidean((cx, cy), (jx, jy))
                        info_j    = _node_info(j_id, instance)
                        b_ju      = max(arrival_j, info_j['earliest'])
                        c12       = b_ju - times_before[pos]
                    c1 = alpha1 * c11 + alpha2 * c12
                    if u_best_cost is None or c1 < u_best_cost:
                        u_best_cost, u_best_pos = c1, pos
                if u_best_cost is None:
                    continue
                c2 = lam * d0[u] - u_best_cost
                if c2 > best_c2:
                    best_c2, best_u, best_pos, best_c1 = c2, u, u_best_pos, u_best_cost
            if best_u is None:
                break
            route.insert(best_pos, best_u)
            unrouted.remove(best_u)
        routes.append(route)
    return routes
def _lns_improve(solution, instance, n_iter=50, k=10, seed=0):
    rng       = np.random.RandomState(seed)
    current   = [list(r) for r in solution]
    best      = copy.deepcopy(current)
    best_cost = compute_cost(best, instance)
    for _ in range(n_iter):
        all_c   = [c for r in current for c in r]
        k_      = min(k, max(1, len(all_c) - 1))
        removed = list(rng.choice(all_c, k_, replace=False))
        removed_set = set(removed)
        partial = [[c for c in r if c not in removed_set] for r in current]
        partial = [r for r in partial if r]
        candidate = _repair_greedy(removed, partial, instance)
        c_cost    = compute_cost(candidate, instance)
        if c_cost < best_cost and is_feasible(candidate, instance):
            best, best_cost = copy.deepcopy(candidate), c_cost
        current = candidate
    return best
def get_initial_solution(instance, lns_iter=50, lns_k=10):
    sol = _build_initial_i1(
        instance,
        mu=CFG.get('I1_MU', 1.0), lam=CFG.get('I1_LAMBDA', 2.0),
        alpha1=CFG.get('I1_ALPHA1', 1.0), alpha2=CFG.get('I1_ALPHA2', 0.0),
    )
    sol = _lns_improve(sol, instance, n_iter=lns_iter, k=lns_k)
    sol = [r for r in sol if r]
    return sol, solution_stats(sol, instance)
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 3  —  ENVIRONNEMENT VRPTW  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class VRPTWEnv:
    def __init__(self, instance, initial_solution,
                 K=15, max_steps=20,
                 gurobi_tlimit=2.0, gurobi_gap=0.05,
                 vehicle_penalty=0.0,
                 verbose=False):
        self.instance    = instance
        self.initial_sol = copy.deepcopy(initial_solution)
        self.K           = K
        self.max_steps   = max_steps
        self.tlimit      = gurobi_tlimit
        self.gap         = gurobi_gap
        self.vehicle_penalty = vehicle_penalty
        self.verbose     = verbose
        self.obs_dim     = 7
        self.n_actions   = 4
        self._dist = {}
        depot = (instance['depot']['x'], instance['depot']['y'])
        for c in instance['customers']:
            d = euclidean(depot, (c['x'], c['y']))
            self._dist[(0, c['id'])] = d
            self._dist[(c['id'], 0)] = d
        for ci in instance['customers']:
            for cj in instance['customers']:
                self._dist[(ci['id'], cj['id'])] = euclidean(
                    (ci['x'], ci['y']), (cj['x'], cj['y']))
        self.reset()
    def _d(self, a, b):
        return self._dist.get((a, b), 0.0)
    def _info(self, nid):
        return _node_info(nid, self.instance)
    def reset(self, start_solution=None):
        if start_solution is not None:
            self.solution = copy.deepcopy(start_solution)
        else:
            self.solution = copy.deepcopy(self.initial_sol)
        self.current_cost     = compute_cost(self.solution, self.instance)
        self.best_cost        = self.current_cost
        self.initial_cost     = compute_cost(self.initial_sol, self.instance)
        self.step_count       = 0
        self.steps_no_improve = 0
        self._action_history  = []
        return self._obs()
    def _obs(self):
        stats = solution_stats(self.solution, self.instance)
        ic    = max(self.initial_cost, 1.0)
        return np.array([
            self.current_cost / ic,
            stats['n_vehicles'] / max(self.instance['n_vehicles'], 1),
            stats['avg_load_ratio'],
            stats['total_wait_time'] / max(ic * 0.5, 1.0),
            self.steps_no_improve / self.max_steps,
            self.best_cost / ic,
            1.0,
        ], dtype=np.float32)
    def _destroy(self, action):
        all_c = [c for r in self.solution for c in r]
        if not all_c:
            return [], copy.deepcopy(self.solution)
        K   = min(self.K, len(all_c))
        rng = np.random.RandomState(self.step_count + action * 100)
        if action == 0:
            pivot   = int(rng.choice(all_c))
            px, py  = self._info(pivot)['x'], self._info(pivot)['y']
            dists   = {c: euclidean((px, py),
                                    (self._info(c)['x'], self._info(c)['y']))
                       for c in all_c}
            destroyed = sorted(all_c, key=lambda c: dists[c])[:K]
        elif action == 1:
            pivot    = int(rng.choice(all_c))
            e_pivot  = self._info(pivot)['earliest']
            tdiffs   = {c: abs(self._info(c)['earliest'] - e_pivot) for c in all_c}
            destroyed = sorted(all_c, key=lambda c: tdiffs[c])[:K]
        elif action == 2:
            savings = {}
            for r in self.solution:
                if not r:
                    continue
                nodes = [0] + r + [0]
                for i, cid in enumerate(r):
                    prev, nxt     = nodes[i], nodes[i + 2]
                    savings[cid]  = (self._d(prev, cid) + self._d(cid, nxt)
                                     - self._d(prev, nxt))
            destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
        else:
            idx       = rng.choice(len(all_c), K, replace=False)
            destroyed = [all_c[i] for i in idx]
        destroyed_set = set(destroyed)
        partial = [[c for c in r if c not in destroyed_set]
                   for r in self.solution]
        partial = [r for r in partial if r]
        return list(destroyed), partial
    def _repair_gurobi(self, destroyed, partial):
        if not destroyed:
            return copy.deepcopy(self.solution)
        inst    = self.instance
        depot   = inst['depot']
        horizon = inst['horizon']
        cap     = inst['capacity']
        nodes   = [0] + destroyed
        N       = len(nodes)
        V       = min(len(destroyed), inst['n_vehicles'])
        env, m = None, None
        try:
            env = gp.Env(empty=True)
            env.setParam('OutputFlag', 0)
            env.start()
            m = gp.Model(env=env)
            if self.tlimit is not None:
                m.setParam('TimeLimit', self.tlimit)
            m.setParam('MIPGap',     self.gap)
            m.setParam('OutputFlag', 0)
            x = m.addVars(N, N, V, vtype=GRB.BINARY, name='x')
            t = m.addVars(N, V, lb=0, ub=horizon,    name='t')
            m.setObjective(
                gp.quicksum(
                    self._d(nodes[i], nodes[j]) * x[i, j, v]
                    for i in range(N) for j in range(N) for v in range(V)
                    if i != j
                )
                # Pénalité par véhicule ouvert (0.0 = comportement d'origine,
                # inchangé). Un coefficient > 0 décourage Gurobi d'ouvrir un
                # véhicule de plus que nécessaire lors du repair, ce qui
                # limite l'inflation du nombre de véhicules par rapport au
                # BKS observée dans la validation.
                + self.vehicle_penalty * gp.quicksum(
                    x[0, j, v] for j in range(1, N) for v in range(V)
                ),
                GRB.MINIMIZE
            )
            for i in range(1, N):
                m.addConstr(
                    gp.quicksum(x[i, j, v]
                                for j in range(N) for v in range(V)
                                if i != j) == 1
                )
            for v in range(V):
                for i in range(1, N):
                    m.addConstr(
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j) ==
                        gp.quicksum(x[j, i, v] for j in range(N) if i != j)
                    )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(x[0, j, v] for j in range(1, N)) <= 1
                )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(
                        inst['customers'][nodes[i] - 1]['demand'] *
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j)
                        for i in range(1, N)
                    ) <= cap
                )
            M = horizon + 1000
            for v in range(V):
                m.addConstr(t[0, v] == 0)
                for i in range(N):
                    ni  = nodes[i]
                    ei  = depot['earliest'] if ni == 0 else inst['customers'][ni-1]['earliest']
                    li  = depot['latest']   if ni == 0 else inst['customers'][ni-1]['latest']
                    sti = depot['service_time'] if ni == 0 else inst['customers'][ni-1]['service_time']
                    m.addConstr(t[i, v] >= ei)
                    m.addConstr(t[i, v] <= li)
                    for j in range(N):
                        if i == j:
                            continue
                        m.addConstr(
                            t[j, v] >= t[i, v] + sti + self._d(ni, nodes[j])
                            - M * (1 - x[i, j, v])
                        )
            m.optimize()
            if m.SolCount > 0:
                eps        = 0.5
                new_routes = []
                for v in range(V):
                    if sum(x[0, j, v].X for j in range(1, N)) < eps:
                        continue
                    route_v, cur = [], 0
                    for _ in range(N):
                        nxt = next((j for j in range(N)
                                    if j != cur and x[cur, j, v].X > eps), None)
                        if nxt is None or nxt == 0:
                            break
                        route_v.append(nodes[nxt])
                        cur = nxt
                    if route_v:
                        new_routes.append(route_v)
                full = [r for r in list(partial) + new_routes if r]
                return full
            else:
                raise ValueError("Gurobi : aucune solution trouvée")
        except Exception as e:
            if self.verbose:
                print(f"    [Gurobi→Greedy fallback] {e}")
            return _repair_greedy(destroyed, partial, self.instance)
        finally:
            if m is not None:
                m.dispose()
            if env is not None:
                env.dispose()
    def step(self, action):
        old_cost            = self.current_cost
        destroyed, partial  = self._destroy(action)
        new_sol             = self._repair_gurobi(destroyed, partial)
        new_cost            = compute_cost(new_sol, self.instance)
        feasible            = is_feasible(new_sol, self.instance)
        if feasible:
            self.solution     = new_sol
            self.current_cost = new_cost
        else:
            new_cost = old_cost * 1.001
        improvement = old_cost - self.current_cost
        if improvement > 0:
            self.best_cost        = min(self.best_cost, self.current_cost)
            self.steps_no_improve = 0
        else:
            self.steps_no_improve += 1
        reward = improvement / max(self.initial_cost, 1.0)
        self.step_count += 1
        self._action_history.append(action)
        done = (self.step_count >= self.max_steps)
        info = {
            'cost':          self.current_cost,
            'best_cost':     self.best_cost,
            'action_taken':  action,
            'action_name':   ACTION_NAMES[action],
            'improvement':   improvement,
            'destroyed_k':   len(destroyed),
            'destroyed_ids': [int(d) for d in destroyed[:5]],
            'feasible':      feasible,
        }
        return self._obs(), float(reward), done, info
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 4  —  AGENT DQN  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((
            np.array(s,  dtype=np.float32), int(a), float(r),
            np.array(s2, dtype=np.float32), float(done),
        ))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.FloatTensor(np.stack(s)), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(np.stack(s2)),
                torch.FloatTensor(d))
    def __len__(self):
        return len(self.buf)
class DQNNet(nn.Module):
    def __init__(self, obs_dim=7, n_actions=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim,  hidden),  nn.ReLU(),
            nn.Linear(hidden,   hidden),  nn.ReLU(),
            nn.Linear(hidden,   n_actions),
        )
    def forward(self, x):
        return self.net(x)
class DQNAgent:
    def __init__(self, obs_dim=7, n_actions=4, hidden=64,
                 lr=1e-3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_size=2000, batch_size=32,
                 target_update_freq=10, seed=0):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        self.n_actions          = n_actions
        self.gamma              = gamma
        self.epsilon            = epsilon_start
        self.epsilon_end        = epsilon_end
        self.epsilon_decay      = epsilon_decay
        self.batch_size         = batch_size
        self.target_update_freq = target_update_freq
        self.step_count         = 0
        self.q_net      = DQNNet(obs_dim, n_actions, hidden)
        self.target_net = DQNNet(obs_dim, n_actions, hidden)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)
        self.loss_fn   = nn.MSELoss()
    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state).unsqueeze(0))
            return int(q.argmax().item())
    def store(self, s, a, r, s2, done):
        self.buffer.push(s, a, r, s2, done)
    def learn(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        s, a, r, s2, done = self.buffer.sample(self.batch_size)
        q_vals = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q  = self.target_net(s2).max(1)[0]
            targets = r + self.gamma * next_q * (1 - done)
        loss = self.loss_fn(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.optimizer.step()
        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.update_target()
        return float(loss.item())
    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    def save(self, path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        torch.save({'q_net': self.q_net.state_dict(),
                    'target_net': self.target_net.state_dict(),
                    'epsilon': self.epsilon, 'step_count': self.step_count}, path)
        print(f"    [DQN] Modèle sauvegardé → {path}")
    def load(self, path):
        """
        Recharge un checkpoint sauvegardé par save() (mêmes clés). Utilisé
        uniquement si CFG['REUSE_EXISTING_SYNTH_MODEL']=True, pour réutiliser
        un modèle déjà entraîné sur du synthétique au lieu de ré-entraîner.
        """
        ckpt = torch.load(path, map_location='cpu')
        self.q_net.load_state_dict(ckpt['q_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.epsilon    = ckpt.get('epsilon', self.epsilon)
        self.step_count = ckpt.get('step_count', self.step_count)
        print(f"    [DQN] Modèle rechargé ← {path}")
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5  —  ENTRAÎNEMENT DU DQN  (ÉTAPE 1, sur les instances SYNTHÉTIQUES)
#  Stocke cost_trace (trajectoire de coût step par step) dans le log, utilisé
#  ensuite pour les courbes de convergence détaillées. Retourne aussi
#  initial_sols pour les tracés de routes avant/après.
# ══════════════════════════════════════════════════════════════════════════════
BANNER = "═" * 72
def _sec(title):
    print(f"\n{BANNER}\n  {title}\n{BANNER}")
def train_on_instances(instances, episodes_per_instance=10, save_model=True, verbose=True):
    """
    ENTRAÎNEMENT (les poids du DQN sont mis à jour ici via store()/learn()/
    decay_epsilon()). Générique : fonctionne sur n'importe quelle liste
    d'instances au format standard — dans ce script, appelé avec les
    instances SYNTHÉTIQUES produites par generate_synthetic_training_set(),
    JAMAIS avec les instances Solomon (réservées à evaluate_frozen_
    on_instances(), plus bas, pour le TEST).
    save_model=False et verbose=False sont utilisés par le plan Taguchi
    (MODULE 10bis, run interne allégé : pas besoin d'écraser le modèle
    final ni d'un log console détaillé à chaque combinaison testée).
    Retourne : agent, training_log, initial_costs, best_costs, best_solutions,
               instance_times, initial_sols
    """
    n = len(instances)
    if verbose:
        _sec(f"ÉTAPE 1 — LNS initial pour les {n} instances d'entraînement (synthétiques)")
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        if verbose:
            print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
                  f"| véhicules={stats['n_vehicles']}")
    if verbose:
        _sec("ÉTAPE 1 — Entraînement DQN sur le synthétique")
    agent = DQNAgent(
        obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
        gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
        epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
        buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
        target_update_freq=CFG['TARGET_UPDATE'],
    )
    training_log   = []
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        if verbose:
            print(f"\n  {'═'*50}\n  [TRAIN] INSTANCE {inst['name']}  type={inst['type']}"
                  f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)
                next_obs, reward, done, info = env.step(action)
                agent.store(obs, action, reward, next_obs, float(done))
                agent.learn()
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            agent.decay_epsilon()
            training_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            if verbose:
                imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
                print(f"  [TRAIN] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                      f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        if verbose:
            print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    if save_model:
        model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
        agent.save(model_path)
    return agent, training_log, initial_costs, best_costs, best_solutions, instance_times, initial_sols
def evaluate_frozen_on_instances(instances, agent, episodes_per_instance=3):
    """
    TEST — ÉVALUATION GELÉE : l'agent DQN déjà entraîné (ÉTAPE 1, sur le
    synthétique) est ici appliqué en pur mode INFÉRENCE sur les instances
    passées en argument (les 56 Solomon officielles). Différences avec
    train_on_instances() :
      - AUCUN appel à agent.store() / agent.learn() / agent.decay_epsilon() :
        les poids du réseau ne sont JAMAIS mis à jour ici.
      - epsilon est temporairement forcé à 0.0 (politique gloutonne, pure
        exploitation du Q-réseau appris), puis restauré à sa valeur
        d'origine à la fin.
      - Le modèle n'est PAS sauvegardé ici (rien n'a changé dans ses poids).
    Le log retourné (test_log) a exactement la même forme que training_log
    (y compris 'cost_trace'), pour pouvoir réutiliser telles quelles les
    fonctions de tracé (plot_instance_convergence_examples, etc.) sur le
    TEST plutôt que sur l'entraînement.
    Retourne : initial_costs, best_costs, best_solutions, instance_times,
               initial_sols, test_log
    """
    n = len(instances)
    _sec(f"ÉTAPE 2 — ÉVALUATION GELÉE (TEST) sur {n} instances Solomon "
         f"— aucun apprentissage, epsilon=0")
    saved_epsilon = agent.epsilon
    agent.epsilon = 0.0
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    test_log       = []
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TEST] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)          # inférence pure (epsilon=0)
                next_obs, reward, done, info = env.step(action)
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            test_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TEST] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    agent.epsilon = saved_epsilon
    return initial_costs, best_costs, best_solutions, instance_times, initial_sols, test_log
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6  —  TABLEAU FINAL : SITUATION 1 vs BKS
#  (la Situation 2 "Gurobi seul sur l'instance complète" a été retirée)
# ══════════════════════════════════════════════════════════════════════════════
def build_bks_comparison_table(instances, initial_costs, best_costs, best_solutions,
                                instance_times, csv_path=None, latex_path=None):
    """
    Tableau final : Situation 1 (LNS+DQN+Gurobi) vs BKS, avec gap % et le
    nombre de véhicules utilisés pour chaque instance.
    """
    _sec("TABLEAU FINAL — SITUATION 1 vs BKS")
    rows = []
    for i, inst in enumerate(instances):
        name     = inst['name']
        bks      = BKS_REFERENCE.get(name)
        bks_dist = bks['distance'] if bks else None
        n_veh_s1 = sum(1 for r in best_solutions[i] if r)
        row = {
            'instance':             name,
            'type':                 inst['type'],
            'bks_vehicules':        bks['vehicles'] if bks else None,
            'bks_distance':         bks_dist,
            'n_vehicules_situation1': n_veh_s1,
            'distance_situation1':  round(best_costs[i], 2),
            'temps_s_situation1':   round(instance_times[i], 2),
            'gap_%_situation1_bks': round((best_costs[i] - bks_dist) / bks_dist * 100, 2) if bks_dist else None,
        }
        rows.append(row)
    print(f"\n  {'Instance':<9} {'BKS Vhc':>8} {'BKS Dist':>10} {'Vhc S1':>7} "
          f"{'Dist. S1':>10} {'Temps S1':>10} {'Gap %':>8}")
    print("  " + "─" * 70)
    for r in rows:
        bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "—"
        bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "—"
        g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {bks_v:>8} {bks_d:>10} "
              f"{r['n_vehicules_situation1']:>7} {r['distance_situation1']:>10.2f} "
              f"{r['temps_s_situation1']:>10.1f} {g1:>8}")
    # ── Gap moyen vs BKS (Situation 1) ──────────────────────────────────
    valid_gaps = [r['gap_%_situation1_bks'] for r in rows if r['gap_%_situation1_bks'] is not None]
    gap_moyen  = float(np.mean(valid_gaps)) if valid_gaps else None
    if gap_moyen is not None:
        print(f"\n  Gap moyen vs BKS (Situation 1) : {gap_moyen:+.2f}%  "
              f"({len(valid_gaps)}/{len(rows)} instances avec BKS connu)")
        print(f"  Gap min : {min(valid_gaps):+.2f}%  |  Gap max : {max(valid_gaps):+.2f}%")
    else:
        print("\n  Gap moyen vs BKS : indisponible (aucun BKS connu pour ces instances)")
    # ── Tableau récapitulatif simple : Instance / Nb véhicules / Distance / Gap % ──
    # (une ligne "Situation 1" + une ligne "BKS" par instance, format demandé)
    print(f"\n  Récapitulatif Instance / Nb véhicules / Distance / Gap % vs BKS")
    print(f"  {'Instance':<9} {'Situation':<26} {'Nb véhicules':>13} {'Distance':>10} {'Gap %':>8}")
    print("  " + "─" * 71)
    recap_rows = []
    for r in rows:
        g1 = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {'Situation 1 (DQN+LNS+Gurobi)':<26} "
              f"{r['n_vehicules_situation1']:>13} {r['distance_situation1']:>10.2f} {g1:>8}")
        recap_rows.append({
            'instance':    r['instance'],
            'situation':   'Situation 1 (DQN+LNS+Gurobi)',
            'n_vehicules': r['n_vehicules_situation1'],
            'distance':    r['distance_situation1'],
            'gap_%_vs_bks': r['gap_%_situation1_bks'],
        })
        if r['bks_distance'] is not None:
            print(f"  {r['instance']:<9} {'BKS (référence)':<26} "
                  f"{(r['bks_vehicules'] if r['bks_vehicules'] is not None else '—'):>13} "
                  f"{r['bks_distance']:>10.2f} {'0.00':>8}")
            recap_rows.append({
                'instance':    r['instance'],
                'situation':   'BKS (référence)',
                'n_vehicules': r['bks_vehicules'],
                'distance':    r['bks_distance'],
                'gap_%_vs_bks': 0.0,
            })
    if gap_moyen is not None:
        print("  " + "─" * 71)
        print(f"  {'':<9} {'MOYENNE (Situation 1)':<26} {'':>13} {'':>10} {gap_moyen:>+8.2f}")
    if csv_path:
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
        print(f"\n  Tableau final exporté (CSV) → {csv_path}")
        recap_csv_path = csv_path.replace('.csv', '_recap_vehicules_distance.csv')
        if recap_csv_path == csv_path:
            recap_csv_path = csv_path + '.recap_vehicules_distance.csv'
        with open(recap_csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=['instance', 'situation', 'n_vehicules', 'distance', 'gap_%_vs_bks'])
            w.writeheader(); w.writerows(recap_rows)
            if gap_moyen is not None:
                w.writerow({'instance': '', 'situation': 'MOYENNE (Situation 1)',
                            'n_vehicules': '', 'distance': '', 'gap_%_vs_bks': round(gap_moyen, 2)})
        print(f"  Récapitulatif Instance/Véhicules/Distance/Gap exporté (CSV) → {recap_csv_path}")
    if latex_path:
        lines = [
            "% Tableau généré automatiquement — Situation 1 vs BKS",
            "\\begin{table}[htbp]", "  \\centering",
            "  \\caption{Comparaison par instance Solomon officielle entre la méthode "
            "hybride (LNS + DQN + Gurobi repair) et le Best Known Solution (BKS) : "
            "nombre de véhicules, distance totale et écart au BKS.}",
            "  \\label{tab:comparaison-bks}",
            "  \\begin{tabular}{lrrrrrr}", "    \\toprule",
            "    Instance & BKS Vhc & BKS Dist. & Vhc. S1 & Dist. S1 & "
            "Temps S1 (s) & Gap S1 (\\%) \\\\",
            "    \\midrule",
        ]
        for r in rows:
            bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "--"
            bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "--"
            g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "--"
            lines.append(f"    {r['instance']} & {bks_v} & {bks_d} & "
                        f"{r['n_vehicules_situation1']} & {r['distance_situation1']:.2f} & "
                        f"{r['temps_s_situation1']:.1f} & {g1} \\\\")
        lines += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
        with open(latex_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        print(f"  Tableau final exporté (LaTeX) → {latex_path}")
    return rows
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6bis  —  DIAGNOSTIC : LICENCE GUROBI À TAILLE LIMITÉE ?
# ══════════════════════════════════════════════════════════════════════════════
def check_gurobi_license(min_vars_needed=2500):
    """
    Test rapide, effectué AVANT tout calcul, pour savoir si la licence
    Gurobi active est une licence "size-limited" (Community/gratuite,
    plafonnée à ~2000 variables ET ~2000 contraintes) plutôt qu'une licence
    complète (académique ou commerciale, sans limite de taille).

    Pourquoi c'est important ici : le repair Gurobi de la Situation 1
    (_repair_gurobi, appelé à CHAQUE step du LNS) crée jusqu'à
    (K_DESTROY+1)² * V variables binaires — selon K_DESTROY (cf. CFG) et le
    nombre de véhicules de l'instance, cela peut dépasser 2000. Un fallback
    vers le repair glouton est déjà en place dans _repair_gurobi (try/
    except), donc l'exécution ne plantera pas, mais peut silencieusement
    retomber sur le glouton plus souvent que prévu si la licence est limitée.

    Ce test ne bloque JAMAIS l'exécution : il informe seulement, une seule
    fois, en tête de run.
    """
    env, m = None, None
    try:
        env = gp.Env(empty=True)
        env.setParam('OutputFlag', 0)
        env.start()
        m = gp.Model(env=env)
        m.setParam('OutputFlag', 0)
        x = m.addVars(min_vars_needed, vtype=GRB.BINARY)
        m.setObjective(gp.quicksum(x[i] for i in range(min_vars_needed)), GRB.MAXIMIZE)
        m.optimize()
        print(f"  ✓ Licence Gurobi : aucune restriction de taille détectée "
              f"(modèle-test à {min_vars_needed} variables résolu sans erreur).")
        return True
    except gp.GurobiError as e:
        errno = getattr(e, 'errno', None)
        is_size_limit = (errno == getattr(GRB.Error, 'SIZE_LIMIT_EXCEEDED', object()))
        if is_size_limit:
            k_repair_vars = (CFG['K_DESTROY'] + 1) ** 2 * CFG['K_DESTROY']
            print(f"\n  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée "
                  f"(erreur {errno} : {e})\n"
                  f"    → Licence Community/gratuite, plafonnée à ~2000 variables "
                  f"et ~2000 contraintes.\n"
                  f"    → Le repair Gurobi de la Situation 1 (jusqu'à "
                  f"~{k_repair_vars} variables selon K_DESTROY={CFG['K_DESTROY']} et "
                  f"le nombre de véhicules) peut dépasser cette limite sur certaines "
                  f"instances — un repli automatique vers le repair glouton est déjà "
                  f"en place, donc pas de plantage, mais un usage de Gurobi moins "
                  f"fréquent que prévu.\n"
                  f"    → Pour lever cette limite : une licence académique Gurobi "
                  f"est gratuite et sans limite de taille pour les universités "
                  f"(https://www.gurobi.com/academia/) — utile si vous êtes "
                  f"affiliée à ENSI-UMA.\n")
        else:
            print(f"  ⚠ Erreur Gurobi lors du test de licence (non liée à la taille) : {e}")
        return False
    except Exception as e:
        print(f"  ⚠ Impossible de tester la licence Gurobi ({type(e).__name__}: {e}) "
              f"— test ignoré, l'exécution continue normalement.")
        return None
    finally:
        if m is not None:
            m.dispose()
        if env is not None:
            env.dispose()
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 7  —  COURBES DE CONVERGENCE
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_mean(vals, window=20):
    out = []
    for i in range(len(vals)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(vals[lo:i + 1])))
    return out
def plot_global_convergence(training_log, instances, initial_costs, best_costs, outdir):
    """
    Figure globale (2 panneaux) :
      (a) reward par épisode global + moyenne mobile → convergence de
          l'apprentissage DQN
      (b) amélioration % par instance (une barre par instance Solomon) vs
          coût LNS initial
    """
    episodes = [e['episode_global'] for e in training_log]
    rewards  = [e['total_reward']   for e in training_log]
    reward_smooth = _rolling_mean(rewards, window=20)
    improvements  = [(initial_costs[i] - best_costs[i]) / initial_costs[i] * 100
                      for i in range(len(initial_costs))]
    names = [inst['name'] for inst in instances]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    axes[0].plot(episodes, rewards, color='lightgray', linewidth=0.6, label='reward brut / épisode')
    axes[0].plot(episodes, reward_smooth, color='crimson', linewidth=2,
                 label='moyenne mobile (20 épisodes)')
    axes[0].set_xlabel('Épisode global')
    axes[0].set_ylabel('Reward de l\'épisode')
    axes[0].set_title("Convergence de l'apprentissage DQN")
    axes[0].legend(loc='best', fontsize=9)
    axes[0].grid(alpha=0.3)
    colors = ['#2ca02c' if v >= 0 else '#d62728' for v in improvements]
    axes[1].bar(range(len(improvements)), improvements, color=colors)
    axes[1].axhline(np.mean(improvements), color='navy', linestyle='--', linewidth=1.5,
                     label=f"Moyenne = {np.mean(improvements):.2f}%")
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names, rotation=90, fontsize=7)
    axes[1].set_xlabel('Instance')
    axes[1].set_ylabel('Amélioration vs solution LNS initiale (%)')
    axes[1].set_title("Amélioration du coût par instance après DQN+Gurobi")
    axes[1].legend(loc='best', fontsize=9)
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_globale.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence globale → {path}")
    return path
def plot_gap_vs_bks(bks_rows, outdir):
    """
    Barres du gap % vs BKS (Situation 1) pour chaque instance, avec la
    moyenne indiquée par une ligne horizontale. N'utilise que les instances
    pour lesquelles un BKS est connu (gap_%_situation1_bks non None).
    `bks_rows` = la liste `rows` retournée par build_bks_comparison_table().
    """
    data = [(r['instance'], r['gap_%_situation1_bks']) for r in bks_rows
            if r['gap_%_situation1_bks'] is not None]
    if not data:
        print("  [plot_gap_vs_bks] Aucune instance avec BKS connu — graphique ignoré.")
        return None
    names = [d[0] for d in data]
    gaps  = [d[1] for d in data]
    gap_moyen = float(np.mean(gaps))
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(names)), 5.5))
    colors = ['#2ca02c' if g <= 0 else '#d62728' for g in gaps]
    ax.bar(range(len(names)), gaps, color=colors)
    ax.axhline(gap_moyen, color='navy', linestyle='--', linewidth=1.5,
               label=f"Gap moyen = {gap_moyen:+.2f}%")
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_xlabel('Instance')
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_title(f"Écart au Best Known Solution par instance "
                 f"({len(names)} instances avec BKS connu)")
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique gap vs BKS → {path}")
    return path
def plot_instance_convergence_examples(training_log, instances, example_idxs, outdir):
    """
    Pour chaque instance exemple :
      - gauche : coût final / meilleur coût cumulé au fil des épisodes locaux
      - droite : trajectoire de coût step par step du dernier épisode local
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(13, 4.2 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        entries = [e for e in training_log if e['instance_id'] == idx]
        entries.sort(key=lambda e: e['ep_local'])
        ep_locals      = [e['ep_local']   for e in entries]
        best_costs_ep  = [e['best_cost']  for e in entries]
        final_costs_ep = [e['final_cost'] for e in entries]
        name = instances[idx]['name']
        axes[row, 0].plot(ep_locals, final_costs_ep, '-o', color='steelblue',
                           label='coût final de l\'épisode')
        axes[row, 0].plot(ep_locals, best_costs_ep, '-s', color='crimson',
                           label='meilleur coût cumulé')
        axes[row, 0].set_title(f"Instance {name} — convergence par épisode local")
        axes[row, 0].set_xlabel('Épisode local')
        axes[row, 0].set_ylabel('Coût de la tournée')
        axes[row, 0].legend(loc='best', fontsize=9)
        axes[row, 0].grid(alpha=0.3)
        last = entries[-1] if entries else None
        trace = last.get('cost_trace', []) if last else []
        if trace:
            axes[row, 1].plot(range(len(trace)), trace, '-o', color='darkorange')
            axes[row, 1].set_title(f"Instance {name} — trajectoire de coût "
                                    f"(dernier épisode local, {len(trace)-1} steps)")
            axes[row, 1].set_xlabel('Step')
            axes[row, 1].set_ylabel('Coût courant')
            axes[row, 1].grid(alpha=0.3)
        else:
            axes[row, 1].axis('off')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_instances_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence instances exemples → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 8  —  TRACÉS DES ROUTES (AVANT LNS / APRÈS DQN)
# ══════════════════════════════════════════════════════════════════════════════
def _plot_routes_on_ax(ax, instance, solution, title):
    depot = instance['depot']
    ax.scatter([depot['x']], [depot['y']], c='black', marker='s', s=110,
               zorder=5, label='Dépôt')
    n_routes = max(len([r for r in solution if r]), 1)
    try:
        cmap = matplotlib.colormaps.get_cmap('tab20').resampled(max(n_routes, 1))
    except AttributeError:
        cmap = plt.cm.get_cmap('tab20', max(n_routes, 1))
    ridx = 0
    for route in solution:
        if not route:
            continue
        xs = [depot['x']] + [instance['customers'][cid - 1]['x'] for cid in route] + [depot['x']]
        ys = [depot['y']] + [instance['customers'][cid - 1]['y'] for cid in route] + [depot['y']]
        ax.plot(xs, ys, '-o', color=cmap(ridx), markersize=4, linewidth=1.3)
        ridx += 1
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.3)
def plot_routes_examples(instances, initial_sols, best_solutions, example_idxs, outdir):
    """
    Pour chaque instance exemple : routes AVANT (solution LNS initiale, avant
    DQN) à gauche, routes APRÈS (meilleure solution trouvée par DQN+Gurobi)
    à droite. Le BKS (véhicules/distance) est indiqué en sous-titre s'il
    est connu pour l'instance.
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.8 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        inst = instances[idx]
        name = inst['name']
        bks  = BKS_REFERENCE.get(name)
        bks_str = (f" | BKS: dist={bks['distance']:.1f}, vhc={bks['vehicles']}"
                   if bks else "")
        cost_before = compute_cost(initial_sols[idx], inst)
        cost_after  = compute_cost(best_solutions[idx], inst)
        nveh_before = sum(1 for r in initial_sols[idx] if r)
        nveh_after  = sum(1 for r in best_solutions[idx] if r)
        _plot_routes_on_ax(
            axes[row, 0], inst, initial_sols[idx],
            f"Instance {name} — AVANT (LNS)\n"
            f"dist={cost_before:.1f}  véhicules={nveh_before}{bks_str}"
        )
        _plot_routes_on_ax(
            axes[row, 1], inst, best_solutions[idx],
            f"Instance {name} — APRÈS (DQN+Gurobi)\n"
            f"dist={cost_after:.1f}  véhicules={nveh_after}{bks_str}"
        )
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_routes_avant_apres_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique routes avant/après → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 9  —  ANALYSES PAR GROUPE D'INSTANCES (C1/C2/R1/R2/RC1/RC2)
#  Appliqué uniquement aux résultats du TEST (gap vs BKS) — ne joue aucun
#  rôle dans le choix des hyperparamètres, purement diagnostique a posteriori.
# ══════════════════════════════════════════════════════════════════════════════
def get_instance_group(inst):
    """
    Déduit le groupe Solomon (C1/C2/R1/R2/RC1/RC2) à partir du type et du nom
    de l'instance. Ex. : type='C', name='C101' -> 'C1' ; type='RC', name=
    'RC201' -> 'RC2'. Le chiffre juste après le préfixe alphabétique indique
    le sous-groupe (1 = fenêtres serrées / faible capacité, 2 = fenêtres
    larges / grande capacité).
    """
    t = inst['type']
    name = inst['name']
    digit = name[len(t)] if len(name) > len(t) else ''
    return f"{t}{digit}"
def plot_gap_boxplot_by_group(bks_rows, instances, outdir):
    """
    Boxplot du gap % vs BKS regroupé par C1/C2/R1/R2/RC1/RC2, pour visualiser
    quels groupes d'instances posent le plus de difficulté à la méthode.
    """
    by_name = {inst['name']: inst for inst in instances}
    groups = defaultdict(list)
    for r in bks_rows:
        if r['gap_%_situation1_bks'] is None:
            continue
        inst = by_name.get(r['instance'])
        if inst is None:
            continue
        groups[get_instance_group(inst)].append(r['gap_%_situation1_bks'])
    labels = [g for g in GROUP_ORDER if g in groups]
    data   = [groups[g] for g in labels]
    if not data:
        print("  [plot_gap_boxplot_by_group] Aucune donnée — graphique ignoré.")
        return None
    fig, ax = plt.subplots(figsize=(8, 5.5))
    try:
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, showmeans=True)
    except TypeError:
        # matplotlib < 3.9 : le paramètre s'appelait encore 'labels'.
        bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#a6c8ff')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_xlabel('Groupe d\'instances Solomon')
    ax.set_title("Distribution du gap par groupe d'instances")
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_boxplot_par_groupe.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique boxplot gap par groupe → {path}")
    return path
def plot_vehicles_vs_bks(bks_rows, instances, outdir):
    """
    Scatter plot : nombre de véhicules utilisés (Situation 1) vs nombre de
    véhicules du BKS, par instance, coloré par groupe. La diagonale y=x
    marque l'égalité ; les points au-dessus utilisent plus de véhicules que
    le BKS — c'est le phénomène d'inflation identifié dans la validation.
    """
    by_name = {inst['name']: inst for inst in instances}
    cmap_colors = plt.cm.tab10.colors
    color_map = {g: cmap_colors[i % len(cmap_colors)] for i, g in enumerate(GROUP_ORDER)}
    fig, ax = plt.subplots(figsize=(7, 7))
    seen_groups = set()
    max_v = 1
    for r in bks_rows:
        if r['bks_vehicules'] is None or r['n_vehicules_situation1'] is None:
            continue
        inst = by_name.get(r['instance'])
        grp  = get_instance_group(inst) if inst else '?'
        c    = color_map.get(grp, 'gray')
        label = grp if grp not in seen_groups else None
        seen_groups.add(grp)
        ax.scatter(r['bks_vehicules'], r['n_vehicules_situation1'], color=c,
                   label=label, s=45, alpha=0.85, edgecolor='black', linewidth=0.4)
        max_v = max(max_v, r['bks_vehicules'], r['n_vehicules_situation1'])
    ax.plot([0, max_v + 1], [0, max_v + 1], '--', color='gray', linewidth=1, label='y = x (égalité)')
    ax.set_xlabel('Véhicules BKS')
    ax.set_ylabel('Véhicules Situation 1 (DQN+LNS+Gurobi)')
    ax.set_title('Nombre de véhicules utilisés vs BKS, par instance')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_vehicules_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique véhicules vs BKS → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 10  —  PLAN D'EXPÉRIENCE TAGUCHI (calibration des hyperparamètres)
#
#  Objectif : identifier quels hyperparamètres (LNS_ITER, K_DESTROY_RATIO,
#  GUROBI_TLIMIT, GUROBI_GAP, MAX_STEPS, EPS_DECAY, VEHICLE_PENALTY) ont le
#  plus d'effet sur la performance, et quelle combinaison la maximise, sans
#  tester les 3^7 = 2187 combinaisons possibles.
#
#  ⚠ POINT CRITIQUE (train/test disjoint) : ce module supporte DEUX modes,
#  sélectionnés par le paramètre `instances_override`/`metric` de
#  run_taguchi_doe() :
#    - mode SYNTHÉTIQUE (recommandé, piloté par CFG['TAGUCHI_SYNTH']) :
#      `instances_override` = des instances synthétiques générées à la
#      volée ; `metric='lns_improvement'` = % d'amélioration vs le coût LNS
#      initial (aucun BKS requis). Les 56 Solomon ne sont JAMAIS chargées
#      dans ce mode → zéro fuite d'hyperparamètres vers le jeu de test.
#    - mode SOLOMON historique (⚠ non recommandé, piloté par CFG['TAGUCHI']) :
#      `instances_override=None` → charge INSTANCE_FILES (les 56 Solomon)
#      comme matrice externe ; `metric='bks_gap'` = gap % vs BKS_REFERENCE.
#      Ce mode entraîne et évalue directement sur le jeu de test : la
#      combinaison qu'il retourne a "vu" Solomon et ne doit pas être
#      utilisée comme hyperparamètres d'une validation train/test disjointe
#      (elle resterait valable seulement comme référence de comparaison).
#
#  Principe commun (matrice orthogonale L9/L18) :
#    - Matrice interne  : la matrice orthogonale L18 ci-dessous (18 lignes,
#      7 facteurs, 3 niveaux) — les combinaisons de facteurs à tester.
#    - Matrice externe (bruit) : les instances sur lesquelles chaque
#      combinaison de la matrice interne est évaluée ; le ratio signal/bruit
#      (S/N) résume à la fois la performance moyenne ET la stabilité face à
#      la variabilité entre instances/groupes.
#    - Après le plan, reportez la meilleure combinaison dans CFG, repassez
#      ENABLE=False, et relancez le script pour la validation ENTRAÎNEMENT/
#      TEST DISJOINT normale (avec ces hyperparamètres).
# ══════════════════════════════════════════════════════════════════════════════
# Matrice orthogonale L9 (3^4) — 9 runs, jusqu'à 4 facteurs à 3 niveaux.
L9_ARRAY = [
    [1, 1, 1, 1], [1, 2, 2, 2], [1, 3, 3, 3],
    [2, 1, 2, 3], [2, 2, 3, 1], [2, 3, 1, 2],
    [3, 1, 3, 2], [3, 2, 1, 3], [3, 3, 2, 1],
]
# Matrice orthogonale L18 (2^1 x 3^7) — 18 runs. La 1ère colonne (2 niveaux)
# n'est pas utilisée ici (tous nos facteurs sont à 3 niveaux) ; seules les
# colonnes 2 à 8 (7 facteurs à 3 niveaux) servent au plan.
L18_ARRAY_FULL = [
    [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 2, 2, 2, 2, 2, 2], [1, 1, 3, 3, 3, 3, 3, 3],
    [1, 2, 1, 1, 2, 2, 3, 3], [1, 2, 2, 2, 3, 3, 1, 1], [1, 2, 3, 3, 1, 1, 2, 2],
    [1, 3, 1, 2, 1, 3, 2, 3], [1, 3, 2, 3, 2, 1, 3, 1], [1, 3, 3, 1, 3, 2, 1, 2],
    [2, 1, 1, 3, 3, 2, 2, 1], [2, 1, 2, 1, 1, 3, 3, 2], [2, 1, 3, 2, 2, 1, 1, 3],
    [2, 2, 1, 2, 3, 1, 3, 2], [2, 2, 2, 3, 1, 2, 1, 3], [2, 2, 3, 1, 2, 3, 2, 1],
    [2, 3, 1, 3, 2, 3, 1, 2], [2, 3, 2, 1, 3, 1, 2, 3], [2, 3, 3, 2, 1, 2, 3, 1],
]
L18_ARRAY = [row[1:] for row in L18_ARRAY_FULL]   # ne garder que les 7 colonnes à 3 niveaux
def check_orthogonal_array(array):
    """
    Vérifie la propriété d'orthogonalité : pour chaque paire de colonnes,
    chaque combinaison de niveaux doit apparaître le même nombre de fois.
    Retourne True/False — sert de garde-fou avant de lancer un plan
    potentiellement long sur une matrice mal formée.
    """
    arr = np.array(array)
    n_runs, n_cols = arr.shape
    ok = True
    for c1, c2 in itertools.combinations(range(n_cols), 2):
        counts = defaultdict(int)
        for row in arr:
            counts[(row[c1], row[c2])] += 1
        if len(set(counts.values())) > 1:
            ok = False
    return ok
def select_orthogonal_array(n_factors):
    if n_factors <= 4:
        return 'L9', [row[:n_factors] for row in L9_ARRAY]
    elif n_factors <= 7:
        return 'L18', [row[:n_factors] for row in L18_ARRAY]
    else:
        raise ValueError(
            f"{n_factors} facteurs demandés, mais seules les matrices L9 (4 "
            f"facteurs max) et L18 (7 facteurs max) sont disponibles. "
            f"Réduisez TAGUCHI_FACTORS ou étendez ce module avec une L27."
        )
@contextmanager
def _cfg_override(**kwargs):
    """Applique temporairement des valeurs de CFG (pour un run du plan
    Taguchi), puis restaure les valeurs d'origine à la sortie du bloc."""
    old = {k: CFG.get(k) for k in kwargs}
    CFG.update(kwargs)
    try:
        yield
    finally:
        CFG.update(old)
def taguchi_sn_ratio_stb(values):
    """
    Ratio signal/bruit 'plus petit est meilleur' (Smaller-The-Better),
    calculé sur une liste de valeurs (le gap %/l'amélioration % peut être
    négatif — la formule standard SN = -10*log10(mean(y_i^2)) reste valide,
    elle pénalise tout écart à zéro, positif ou négatif, ainsi que la
    variance entre les instances de la matrice externe).
    """
    y = np.array(values, dtype=float)
    msd = max(float(np.mean(y ** 2)), 1e-9)
    return -10.0 * np.log10(msd)
def taguchi_main_effects(results, factor_names, response_key):
    """Moyenne de la réponse (gap/amélioration ou S/N) par niveau, pour
    chaque facteur."""
    effects = {}
    for fn in factor_names:
        by_level = defaultdict(list)
        for r in results:
            if r.get(response_key) is not None:
                by_level[r[fn]].append(r[response_key])
        effects[fn] = {lvl: float(np.mean(vals)) for lvl, vals in sorted(by_level.items(), key=lambda kv: kv[0])}
    return effects
def taguchi_anova_contributions(results, factor_names, response_key):
    """
    ANOVA simplifiée (méthode des sommes de carrés à partir des moyennes par
    niveau) : contribution en % de chaque facteur à la variance totale de la
    réponse. Ne remplace pas une ANOVA complète avec terme d'erreur pur,
    mais donne une hiérarchie fiable de l'importance relative des facteurs.
    """
    valid = [r for r in results if r.get(response_key) is not None]
    if not valid:
        return {fn: 0.0 for fn in factor_names}
    y = np.array([r[response_key] for r in valid])
    grand_mean = y.mean()
    ss_total = float(np.sum((y - grand_mean) ** 2))
    contributions = {}
    for fn in factor_names:
        by_level = defaultdict(list)
        for r in valid:
            by_level[r[fn]].append(r[response_key])
        ss_factor = sum(len(vals) * (np.mean(vals) - grand_mean) ** 2 for vals in by_level.values())
        contributions[fn] = (ss_factor / ss_total * 100.0) if ss_total > 0 else 0.0
    return contributions
def plot_taguchi_main_effects(effects, factor_names, response_label, outdir, filename):
    """Graphique classique des effets principaux Taguchi : un panneau par
    facteur, moyenne de la réponse pour chaque niveau, reliée par une ligne."""
    n = len(factor_names)
    ncols = min(4, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, fn in enumerate(factor_names):
        levels = list(effects[fn].keys())
        means  = list(effects[fn].values())
        axes[i].plot(range(len(levels)), means, '-o', color='steelblue')
        axes[i].set_xticks(range(len(levels)))
        axes[i].set_xticklabels([str(l) for l in levels], fontsize=8)
        axes[i].set_title(fn, fontsize=10)
        axes[i].set_ylabel(response_label, fontsize=8)
        axes[i].grid(alpha=0.3)
    for j in range(len(factor_names), len(axes)):
        axes[j].axis('off')
    fig.suptitle(f"Effets principaux — {response_label}", fontsize=13)
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique effets principaux ({response_label}) → {path}")
    return path
def plot_taguchi_sn_bars(results, outdir, filename='taguchi_sn_ratios.png'):
    """Ratio S/N obtenu pour chaque run du plan, run avec le meilleur S/N
    mis en évidence en vert."""
    valid = [r for r in results if r.get('sn_ratio') is not None]
    if not valid:
        print("  [plot_taguchi_sn_bars] Aucun résultat — graphique ignoré.")
        return None
    runs = [r['run'] for r in valid]
    sns_ = [r['sn_ratio'] for r in valid]
    best_idx = int(np.argmax(sns_))
    colors = ['#2ca02c' if i == best_idx else '#4c72b0' for i in range(len(sns_))]
    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(runs)), 5))
    ax.bar([str(r) for r in runs], sns_, color=colors)
    ax.set_xlabel('Run (combinaison de la matrice L9/L18)')
    ax.set_ylabel('Ratio S/N (plus grand = meilleur)')
    ax.set_title('Ratio signal/bruit par combinaison — plan Taguchi')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique S/N par run → {path}")
    return path
def plot_taguchi_pareto(anova_contributions, outdir, filename='taguchi_pareto_effets.png'):
    """Diagramme en barres horizontales de la contribution (%) de chaque
    facteur à la variance de la réponse — équivalent simplifié d'un Pareto ANOVA."""
    items = sorted(anova_contributions.items(), key=lambda kv: -kv[1])
    names = [k for k, v in items][::-1]
    vals  = [v for k, v in items][::-1]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(range(len(names)), vals, color='#4c72b0')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('Contribution à la variance de la réponse (%)')
    ax.set_title("Importance relative des facteurs (ANOVA simplifiée Taguchi)")
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique Pareto des effets → {path}")
    return path
def run_taguchi_doe(factors, outer_instance_names, episodes_per_instance_doe, outdir,
                     instances_override=None, metric='bks_gap', json_suffix=''):
    """
    Exécute le plan d'expérience Taguchi complet :
      1. Construit la matrice orthogonale adaptée au nombre de facteurs
      2. Pour chaque run, applique les niveaux de facteurs (via _cfg_override)
         et entraîne le pipeline complet (LNS+DQN+Gurobi) sur les instances
         de la matrice externe, avec un nombre d'épisodes réduit
      3. Calcule la métrique de classement par instance (voir `metric`), la
         moyenne, et le ratio S/N
      4. Analyse les effets principaux + ANOVA simplifiée, génère les graphes
      5. Rapporte la meilleure combinaison trouvée
    instances_override : si fourni (liste d'instances déjà construites),
        utilisé DIRECTEMENT comme matrice externe, SANS jamais charger les
        fichiers Solomon — c'est ce qui permet de calibrer sur des instances
        SYNTHÉTIQUES (voir CFG['TAGUCHI_SYNTH']) sans aucune fuite vers le
        jeu de test. Si None (mode historique Solomon), charge les instances
        via INSTANCE_FILES et `outer_instance_names`.
    metric : 'bks_gap' (gap % vs BKS_REFERENCE — ne fonctionne que pour des
        instances Solomon officielles connues dans BKS_REFERENCE) ou
        'lns_improvement' (% d'amélioration vs le coût LNS initial — aucun
        BKS requis, utilisable sur des instances synthétiques). Convention
        "plus petit est meilleur" identique dans les deux cas (négatif =
        amélioration), donc toute la mécanique Taguchi (S/N, effets
        principaux, ANOVA, combinaison prédite) fonctionne sans modification.
    json_suffix : suffixe inséré dans les noms des fichiers de sortie (CSV,
        JSON, PNG) — ex. '_synthetique' — pour ne jamais écraser les
        résultats d'un plan calibré sur d'autres instances/métrique.
    Retourne : results (liste de dicts, un par run), effects_gap, effects_sn,
               anova_contributions, best_run
    """
    factor_names  = [f[0] for f in factors]
    factor_levels = [f[1] for f in factors]
    n_factors     = len(factors)
    array_name, array = select_orthogonal_array(n_factors)
    ortho_ok = check_orthogonal_array(array)
    if instances_override is not None:
        outer_instances = list(instances_override)
    else:
        all_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
        if outer_instance_names == 'ALL':
            # Sentinel spécial : utiliser TOUTES les instances Solomon
            # chargées comme matrice externe. ⚠ Mode historique — voir
            # avertissement CFG['TAGUCHI'] plus haut : fuite d'hyperparamètres
            # vers le jeu de test. Multiplie aussi fortement le temps total
            # du plan (voir avertissement de durée ci-dessous).
            outer_instances = list(all_instances)
        else:
            by_name = {inst['name']: inst for inst in all_instances}
            outer_instances = [by_name[nm] for nm in outer_instance_names if nm in by_name]
    if not outer_instances:
        raise ValueError("Aucune instance de la matrice externe trouvée "
                          "(instances_override vide, ou outer_instance_names "
                          "absent de INSTANCE_FILES).")
    gurobi_tlimit_max = max(TAGUCHI_FACTORS[[f[0] for f in TAGUCHI_FACTORS].index('GUROBI_TLIMIT')][1]) \
        if 'GUROBI_TLIMIT' in factor_names else CFG['GUROBI_TLIMIT']
    max_steps_max = max(TAGUCHI_FACTORS[[f[0] for f in TAGUCHI_FACTORS].index('MAX_STEPS')][1]) \
        if 'MAX_STEPS' in factor_names else CFG['MAX_STEPS']
    worst_case_h = (len(array) * len(outer_instances) * gurobi_tlimit_max
                     * max_steps_max * episodes_per_instance_doe) / 3600.0
    _sec(f"PLAN D'EXPÉRIENCE TAGUCHI — {array_name} ({len(array)} runs, "
         f"{n_factors} facteurs, métrique={metric}, orthogonalité="
         f"{'OK' if ortho_ok else 'A VERIFIER !'})")
    if len(outer_instances) <= 12:
        print(f"  Matrice externe (bruit) : {[i['name'] for i in outer_instances]}")
    else:
        print(f"  Matrice externe (bruit) : {len(outer_instances)} instances "
              f"({outer_instances[0]['name']} ... {outer_instances[-1]['name']})")
    print(f"  Épisodes/instance pendant le plan : {episodes_per_instance_doe}")
    print(f"  ⚠ Temps pire cas estimé (si Gurobi atteint systématiquement sa limite de "
          f"temps) : ~{worst_case_h:.1f}h ({worst_case_h/24:.1f} jours). "
          f"Le temps réel est généralement inférieur, mais reste substantiel à cette échelle.")
    results = []
    for run_idx, level_idx_row in enumerate(array, start=1):
        levels = {factor_names[i]: factor_levels[i][level_idx_row[i] - 1] for i in range(n_factors)}
        print(f"\n  Run {run_idx:02d}/{len(array)} : {levels}")
        overrides = dict(levels)
        if 'K_DESTROY_RATIO' in overrides:
            overrides['K_DESTROY'] = max(3, round(100 * overrides['K_DESTROY_RATIO']))
        t_run = time.time()
        with _cfg_override(**overrides):
            _, _, initial_costs_o, best_costs_o, _, _, _ = train_on_instances(
                outer_instances, episodes_per_instance=episodes_per_instance_doe,
                save_model=False, verbose=False,
            )
        values = []
        for i, inst in enumerate(outer_instances):
            if metric == 'bks_gap':
                bks = BKS_REFERENCE.get(inst['name'])
                if bks:
                    values.append((best_costs_o[i] - bks['distance']) / bks['distance'] * 100.0)
            elif metric == 'lns_improvement':
                ic = initial_costs_o[i]
                if ic:
                    values.append((best_costs_o[i] - ic) / ic * 100.0)
            else:
                raise ValueError(f"metric inconnue : {metric!r} (attendu 'bks_gap' ou 'lns_improvement')")
        mean_val = float(np.mean(values)) if values else None
        sn       = taguchi_sn_ratio_stb(values) if values else None
        row_result = dict(levels)
        row_result.update({
            'run': run_idx,
            'mean_gap_%': round(mean_val, 3) if mean_val is not None else None,
            'sn_ratio':   round(sn, 3) if sn is not None else None,
        })
        results.append(row_result)
        elapsed = time.time() - t_run
        if mean_val is not None:
            print(f"    → {'gap' if metric=='bks_gap' else 'amélioration'} moyen(ne) = "
                  f"{mean_val:+.2f}%  |  S/N = {sn:.2f}  |  {elapsed:.1f}s")
        else:
            print(f"    → indisponible  |  {elapsed:.1f}s")
    csv_path = os.path.join(outdir, f'taguchi_resultats{json_suffix}.csv')
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = factor_names + ['run', 'mean_gap_%', 'sn_ratio']
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        w.writeheader()
        for r in results:
            w.writerow(r)
    print(f"\n  Résultats Taguchi exportés (CSV) → {csv_path}")
    effects_gap = taguchi_main_effects(results, factor_names, 'mean_gap_%')
    effects_sn  = taguchi_main_effects(results, factor_names, 'sn_ratio')
    anova       = taguchi_anova_contributions(results, factor_names, 'mean_gap_%')
    response_label = 'Gap moyen vs BKS (%)' if metric == 'bks_gap' else 'Amélioration moyenne vs LNS (%)'
    _sec("GRAPHIQUES TAGUCHI")
    plot_taguchi_main_effects(effects_gap, factor_names, response_label, outdir,
                              f'taguchi_effets_principaux_gap{json_suffix}.png')
    plot_taguchi_main_effects(effects_sn, factor_names, 'Ratio S/N', outdir,
                              f'taguchi_effets_principaux_sn{json_suffix}.png')
    plot_taguchi_sn_bars(results, outdir, filename=f'taguchi_sn_ratios{json_suffix}.png')
    plot_taguchi_pareto(anova, outdir, filename=f'taguchi_pareto_effets{json_suffix}.png')
    valid_runs = [r for r in results if r['mean_gap_%'] is not None]
    best_run = min(valid_runs, key=lambda r: abs(r['mean_gap_%'])) if valid_runs else None
    print(f"\n{'█'*72}")
    print(f"  █  MEILLEURE COMBINAISON TROUVÉE — PLAN TAGUCHI{' (' + json_suffix + ')' if json_suffix else ''}")
    print(f"{'█'*72}")
    if best_run:
        print(f"  Run {best_run['run']} : {'gap' if metric=='bks_gap' else 'amélioration'} moyen(ne) = "
              f"{best_run['mean_gap_%']:+.2f}%  |  S/N = {best_run['sn_ratio']:.2f}")
        print(f"  ── Hyperparamètres recommandés ──")
        for fn in factor_names:
            print(f"    {fn:<18s} = {best_run[fn]}")
        print(f"\n  ── Contribution des facteurs à la variance de la réponse (ANOVA) ──")
        for fn, pct in sorted(anova.items(), key=lambda kv: -kv[1]):
            print(f"    {fn:<18s} : {pct:5.1f}%")
        print(f"\n  → Reportez ces valeurs dans CFG (et TAGUCHI_FACTORS si besoin), "
              f"repassez ENABLE=False, et relancez le script pour la validation "
              f"ENTRAÎNEMENT/TEST DISJOINT sur les 56 instances Solomon.")
        best_json_path = os.path.join(outdir, f'taguchi_meilleure_combinaison{json_suffix}.json')
        with open(best_json_path, 'w', encoding='utf-8') as f:
            json.dump({
                'run':               best_run['run'],
                'metric':            metric,
                'mean_gap_%':        best_run['mean_gap_%'],
                'sn_ratio':          best_run['sn_ratio'],
                'hyperparametres':   {fn: best_run[fn] for fn in factor_names},
                'contribution_anova_%': anova,
                'array':             array_name,
                'n_runs':            len(array),
                'n_outer_instances': len(outer_instances),
                'episodes_per_instance_doe': episodes_per_instance_doe,
            }, f, indent=2, ensure_ascii=False)
        print(f"\n  Meilleure combinaison sauvegardée (JSON) → {best_json_path}")
    print(f"{'█'*72}\n")
    return results, effects_gap, effects_sn, anova, best_run
def load_and_apply_taguchi_best(outdir, json_suffix=''):
    """
    Étape de confirmation : cherche taguchi_meilleure_combinaison<suffix>.json
    (créé par run_taguchi_doe) et applique ces hyperparamètres à CFG, pour
    que la validation ENTRAÎNEMENT/TEST DISJOINT utilise automatiquement la
    meilleure combinaison trouvée — sans avoir à la reporter à la main.
    Passez json_suffix='_synthetique' pour charger la combinaison calibrée
    UNIQUEMENT sur du synthétique (recommandé). Ne modifie rien (et affiche
    un avertissement) si le fichier n'existe pas encore.
    Retourne True si des hyperparamètres ont été appliqués, False sinon.
    """
    path = os.path.join(outdir, f'taguchi_meilleure_combinaison{json_suffix}.json')
    if not os.path.exists(path):
        cfg_key_hint = "CFG['TAGUCHI_SYNTH']" if json_suffix else "CFG['TAGUCHI']"
        print(f"  ⚠ {path} introuvable — lancez d'abord le plan Taguchi correspondant "
              f"({cfg_key_hint}['ENABLE']=True). Hyperparamètres CFG actuels inchangés.")
        return False
    with open(path, 'r', encoding='utf-8') as f:
        best = json.load(f)
    hp = best.get('hyperparametres', {})
    for fn, val in hp.items():
        CFG[fn] = val
    if 'K_DESTROY_RATIO' in hp:
        CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))
    print(f"  ✓ Hyperparamètres Taguchi appliqués depuis {path} :")
    for fn, val in hp.items():
        print(f"    {fn:<18s} = {val}")
    return True
# ══════════════════════════════════════════════════════════════════════════════
#  EXÉCUTION AUTOMATIQUE
# ══════════════════════════════════════════════════════════════════════════════
def run_final_validation():
    """
    MODE VALIDATION : ENTRAÎNEMENT/TEST DISJOINT.
    Entraîne le DQN sur CFG['N_SYNTH_INSTANCES'] instances synthétiques puis
    l'évalue gelé sur les 56 instances Solomon officielles. Si
    CFG['USE_TAGUCHI_BEST']=True, applique d'abord automatiquement la
    meilleure combinaison d'hyperparamètres trouvée par le plan Taguchi
    synthétique (fichier taguchi_meilleure_combinaison_synthetique.json).

    Cette fonction regroupe exactement ce que faisait auparavant la branche
    `else` du bloc __main__ — extraite dans une fonction pour pouvoir être
    appelée aussi bien seule (validation directe, sans calibration) que
    depuis la branche Taguchi synthétique, en enchaînement automatique.
    """
    t0 = time.time()
    print(f"\n{'═'*72}\n  MODE ENTRAÎNEMENT/TEST DISJOINT\n"
          f"  ENTRAÎNEMENT = {CFG['N_SYNTH_INSTANCES']} instances SYNTHÉTIQUES générées "
          f"| TEST = 56 instances Solomon officielles (gelé)\n{'═'*72}")

    # ── DIAGNOSTIC PRÉALABLE : licence Gurobi à taille limitée ? ────────
    # Informatif uniquement — n'interrompt jamais l'exécution.
    _sec("DIAGNOSTIC — LICENCE GUROBI")
    check_gurobi_license()

    # ── Optionnel : appliquer la meilleure combinaison Taguchi calibrée
    # sur synthétique (si le plan CFG['TAGUCHI_SYNTH'] a déjà été lancé
    # au moins une fois et a produit le JSON correspondant) ───────────
    if CFG.get('USE_TAGUCHI_BEST', False):
        _sec("APPLICATION DE LA MEILLEURE COMBINAISON TAGUCHI (SYNTHÉTIQUE)")
        load_and_apply_taguchi_best(CFG['OUTDIR'], json_suffix='_synthetique')

    # ── ÉTAPE 1a : GÉNÉRATION DES INSTANCES SYNTHÉTIQUES D'ENTRAÎNEMENT ─
    # (reprend generate_instance() de Code 1 — cf. MODULE 1bis ci-dessus).
    # Ces instances SYN-* ne recoupent jamais les noms Solomon (C101, R101,
    # ...) chargés à l'ÉTAPE 2 : aucune fuite entre train et test.
    synth_instances = generate_synthetic_training_set(CFG)

    # ── ÉTAPE 1b : ENTRAÎNEMENT DU DQN — uniquement sur le synthétique ──
    # Valeurs par défaut : si le modèle est rechargé (branche ci-dessous),
    # il n'y a pas de training_log/coûts synthétiques dans CE run — les
    # graphiques de convergence d'entraînement seront alors simplement omis.
    training_log, synth_initial_costs, synth_best_costs = [], None, None
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    if CFG.get('REUSE_EXISTING_SYNTH_MODEL', False) and os.path.exists(model_path):
        _sec(f"ÉTAPE 1b — RÉUTILISATION D'UN MODÈLE DÉJÀ ENTRAÎNÉ SUR LE SYNTHÉTIQUE "
             f"({model_path})")
        agent = DQNAgent(
            obs_dim=7, n_actions=4, hidden=CFG['HIDDEN'], lr=CFG['LR'],
            gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
            epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
            buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
            target_update_freq=CFG['TARGET_UPDATE'],
        )
        agent.load(model_path)
    else:
        agent, training_log, synth_initial_costs, synth_best_costs, \
            synth_best_solutions, synth_instance_times, synth_initial_sols = train_on_instances(
                synth_instances, episodes_per_instance=CFG['EPISODES_PER_INSTANCE'])

    # ── ÉTAPE 2a : CHARGEMENT DES 56 INSTANCES SOLOMON (TEST — jamais vues) ─
    _sec("ÉTAPE 2 — CHARGEMENT DES INSTANCES SOLOMON (TEST)")
    test_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
    example_idxs = pick_example_indices(test_instances, preferred_names=CFG['EXAMPLE_NAMES'])

    # ── ÉTAPE 2b : ÉVALUATION GELÉE — le même agent en mémoire, sans réentraîner ──
    initial_costs, best_costs, best_solutions, instance_times, test_initial_sols, test_log = \
        evaluate_frozen_on_instances(
            test_instances, agent, episodes_per_instance=CFG['EPISODES_PER_INSTANCE_TEST'])

    # ── [A] Convergence de l'ENTRAÎNEMENT (synthétique uniquement) ──────
    _sec("GRAPHIQUE — CONVERGENCE DE L'ENTRAÎNEMENT (SYNTHÉTIQUE)")
    if training_log:
        plot_global_convergence(training_log, synth_instances,
                                 synth_initial_costs, synth_best_costs, CFG['OUTDIR'])
    else:
        print("  (pas de courbe de convergence : modèle rechargé, aucun "
              "entraînement effectué dans ce run)")

    # ── [B] Convergence détaillée + [C] Routes avant/après — SUR LE TEST ─
    _sec("GRAPHIQUES — DÉTAIL DU TEST (INSTANCES SOLOMON EXEMPLES)")
    plot_instance_convergence_examples(test_log, test_instances, example_idxs, CFG['OUTDIR'])
    plot_routes_examples(test_instances, test_initial_sols, best_solutions, example_idxs, CFG['OUTDIR'])

    # ── [D] TABLEAU FINAL : SITUATION 1 (test gelé, entraîné sur synthétique) ─
    # vs BKS (gap % par instance + gap moyen) ──────────────────────────
    bks_rows = build_bks_comparison_table(
        test_instances, initial_costs, best_costs, best_solutions, instance_times,
        csv_path=os.path.join(CFG['OUTDIR'], 'solomon_comparaison_bks.csv'),
        latex_path=os.path.join(CFG['OUTDIR'], 'solomon_tableau_bks.tex'),
    )
    # ── [E] Graphique dédié : gap % vs BKS par instance + gap moyen (TEST) ─
    _sec("GRAPHIQUE GAP % vs BKS (TEST)")
    plot_gap_vs_bks(bks_rows, CFG['OUTDIR'])

    # ── [F] Analyses par groupe (C1/C2/R1/R2/RC1/RC2) — véhicules & gap ──
    _sec("GRAPHIQUES — ANALYSES PAR GROUPE D'INSTANCES (TEST)")
    plot_gap_boxplot_by_group(bks_rows, test_instances, CFG['OUTDIR'])
    plot_vehicles_vs_bks(bks_rows, test_instances, CFG['OUTDIR'])

    print(f"\n  ✓ Validation finale terminée en {time.time()-t0:.1f}s "
          f"({(time.time()-t0)/3600:.2f}h)\n")
    return bks_rows


if __name__ == '__main__':
    t_global0 = time.time()

    if CFG['TAGUCHI_SYNTH']['ENABLE'] and CFG['TAGUCHI']['ENABLE']:
        raise ValueError(
            "CFG['TAGUCHI_SYNTH']['ENABLE'] et CFG['TAGUCHI']['ENABLE'] sont tous les "
            "deux True — activez UN SEUL des deux modes de calibration à la fois "
            "(TAGUCHI_SYNTH est recommandé, voir les commentaires plus haut)."
        )

    if CFG['TAGUCHI_SYNTH']['ENABLE']:
        # ── MODE CALIBRATION TAGUCHI — SYNTHÉTIQUE UNIQUEMENT (RECOMMANDÉ) ──
        # Aucune instance Solomon n'est chargée dans cette branche : zéro
        # fuite d'hyperparamètres vers le jeu de test.
        ts = CFG['TAGUCHI_SYNTH']
        print(f"\n{'═'*72}\n  MODE PLAN D'EXPÉRIENCE TAGUCHI — SYNTHÉTIQUE UNIQUEMENT "
              f"({ts['N_INSTANCES_DOE']} instances, métrique = amélioration vs LNS, "
              f"pas de BKS, zéro fuite vers Solomon)\n{'═'*72}")
        synth_doe_instances = generate_synthetic_training_set(
            dict(N_SYNTH_INSTANCES=ts['N_INSTANCES_DOE'],
                 SYNTH_N_CUSTOMERS=CFG['SYNTH_N_CUSTOMERS'],
                 SYNTH_N_VEHICLES=CFG['SYNTH_N_VEHICLES'],
                 SYNTH_VEHICLE_SAFETY_FACTOR=CFG['SYNTH_VEHICLE_SAFETY_FACTOR'],
                 SYNTH_CAPACITY=CFG['SYNTH_CAPACITY'], SYNTH_HORIZON=CFG['SYNTH_HORIZON']),
            verbose=True,
        )
        (ts_results, ts_effects_gap, ts_effects_sn,
         ts_anova, ts_best_run) = run_taguchi_doe(
            factors=TAGUCHI_FACTORS,
            outer_instance_names=None,
            episodes_per_instance_doe=ts['EPISODES_PER_INSTANCE_DOE'],
            outdir=CFG['OUTDIR'],
            instances_override=synth_doe_instances,
            metric='lns_improvement',
            json_suffix='_synthetique',
        )
        _sec("RÉCAPITULATIF FINAL DU PLAN TAGUCHI (SYNTHÉTIQUE)")
        if ts_best_run:
            print("  Meilleurs hyperparamètres trouvés (calibrés sur synthétique) :")
            for fn, _ in TAGUCHI_FACTORS:
                print(f"    {fn:<18s} = {ts_best_run[fn]}")
            print(f"  Amélioration moyenne vs LNS obtenue : {-ts_best_run['mean_gap_%']:.2f}%")
            print(f"  Ratio S/N                           : {ts_best_run['sn_ratio']:.2f}")
        print(f"\n  Meilleure combinaison (JSON) : "
              f"{os.path.join(CFG['OUTDIR'], 'taguchi_meilleure_combinaison_synthetique.json')}")
        print(f"\n  ✓ Plan Taguchi (synthétique) terminé en {time.time()-t_global0:.1f}s "
              f"({(time.time()-t_global0)/3600:.2f}h).")

        # ── ENCHAÎNEMENT AUTOMATIQUE → VALIDATION FINALE AVEC LES BEST HYPERPARAMÈTRES ──
        # Entraînement 80 instances synthétiques + test gelé sur les 56 Solomon,
        # en un seul lancement, sans intervention manuelle.
        if ts_best_run:
            _sec("ENCHAÎNEMENT AUTOMATIQUE → VALIDATION FINALE")
            CFG['USE_TAGUCHI_BEST'] = True
            # Repart d'un modèle neuf avec les NOUVEAUX hyperparamètres —
            # ne jamais réutiliser un .pth calibré avec d'anciens réglages ici.
            CFG['REUSE_EXISTING_SYNTH_MODEL'] = False
            run_final_validation()
        else:
            print("  ⚠ Aucune combinaison valide trouvée par le plan Taguchi — "
                  "validation finale non lancée.")

    elif CFG['TAGUCHI']['ENABLE']:
        # ── MODE CALIBRATION TAGUCHI — SOLOMON (⚠ HISTORIQUE, NON RECOMMANDÉ) ──
        # Volontairement PAS d'enchaînement automatique ici : ce mode calibre
        # directement sur le jeu de test (fuite train/test), la combinaison
        # trouvée ne doit pas servir à une validation train/test disjointe.
        print(f"\n{'═'*72}\n  ⚠ MODE PLAN D'EXPÉRIENCE TAGUCHI — SOLOMON (mode historique)\n"
              f"  Ce mode entraîne ET évalue directement sur les 56 instances Solomon :\n"
              f"  la combinaison retournée aura VU le jeu de test et ne doit PAS être\n"
              f"  utilisée comme hyperparamètres d'une validation train/test disjointe.\n"
              f"  Préférez CFG['TAGUCHI_SYNTH']['ENABLE']=True (calibration synthétique).\n"
              f"{'═'*72}")
        tg = CFG['TAGUCHI']
        (tg_results, tg_effects_gap, tg_effects_sn,
         tg_anova, tg_best_run) = run_taguchi_doe(
            factors=TAGUCHI_FACTORS,
            outer_instance_names=tg['OUTER_INSTANCE_NAMES'],
            episodes_per_instance_doe=tg['EPISODES_PER_INSTANCE_DOE'],
            outdir=CFG['OUTDIR'],
            instances_override=None,
            metric='bks_gap',
            json_suffix='',
        )
        _sec("RÉCAPITULATIF FINAL DU PLAN TAGUCHI (SOLOMON — HISTORIQUE)")
        if tg_best_run:
            print("  Meilleurs hyperparamètres trouvés (⚠ calibrés sur Solomon — fuite) :")
            for fn, _ in TAGUCHI_FACTORS:
                print(f"    {fn:<18s} = {tg_best_run[fn]}")
            print(f"  Gap moyen obtenu : {tg_best_run['mean_gap_%']:+.2f}%")
            print(f"  Ratio S/N        : {tg_best_run['sn_ratio']:.2f}")
        print(f"\n  ✓ Plan Taguchi (Solomon) terminé en {time.time()-t_global0:.1f}s "
              f"({(time.time()-t_global0)/3600:.2f}h). Pas d'enchaînement automatique "
              f"(mode avec fuite) — repassez CFG['TAGUCHI']['ENABLE']=False et relancez "
              f"manuellement si vous souhaitez tout de même l'utiliser, à titre de "
              f"comparaison uniquement.\n")

    else:
        # ── MODE VALIDATION SEULE (comportement normal, sans calibration) ──
        run_final_validation()


════════════════════════════════════════════════════════════════════════
  MODE ENTRAÎNEMENT/TEST DISJOINT
  ENTRAÎNEMENT = 80 instances SYNTHÉTIQUES générées | TEST = 56 instances Solomon officielles (gelé)
════════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════════
  DIAGNOSTIC — LICENCE GUROBI
════════════════════════════════════════════════════════════════════════

  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée (erreur 10010 : Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information)
    → Licence Community/gratuite, plafonnée à ~2000 variables et ~2000 contraintes.
    → Le repair Gurobi de la Situation 1 (jusqu'à ~15000 variables selon K_DESTROY=24 et le nombre de véhicules) peut dépasser cette limite sur certaines instances — un repli automatique vers le repair glouton est déjà en place, donc pas de plantage, mais un usage de Gurobi moins fréquent que

## Ablation des actions

In [3]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON    ║
║                          + GAP PAR RAPPORT AU BKS                           ║
║                                                                              ║
║  Version corrigée : reprend toutes les visualisations de la variante        ║
║  précédente (courbes de convergence, tracés des routes avant/après,        ║
║  graphique dédié gap % vs BKS, tableau récapitulatif véhicules/distance)   ║
║  MAIS applique désormais un vrai protocole ENTRAÎNEMENT / TEST DISJOINT,   ║
║  en un seul lancement, sans aucune intervention manuelle :                 ║
║                                                                              ║
║   ÉTAPE 1 (ENTRAÎNEMENT) — génère des instances SYNTHÉTIQUES (reprises de   ║
║      generate_instance() de main_vrptw_comparaison_gurobi.py / "Code 1")    ║
║      et entraîne le DQN dessus (poids mis à jour via store/learn/decay).    ║
║                                                                              ║
║   ÉTAPE 2 (TEST) — charge les 56 instances Solomon OFFICIELLES et évalue    ║
║      le même agent, GELÉ (aucun store/learn/decay_epsilon, uniquement      ║
║      select_action en pur mode inférence), sur ce jeu jamais vu à          ║
║      l'entraînement. Toutes les courbes/tracés/tableaux ci-dessous portent  ║
║      désormais sur ce TEST gelé, pas sur les résultats d'entraînement.     ║
║                                                                              ║
║  Situation 1 = LNS + DQN(entraîné sur synthétique) + Gurobi, ÉVALUÉ sur     ║
║                Solomon (test)                                → gap vs BKS   ║
║  (La "Situation 2 = Gurobi seul sur l'instance complète" reste retirée,     ║
║   à la demande, car sans limite de temps elle est beaucoup trop longue      ║
║   sur des instances à 100 clients avec fenêtres de temps serrées.)          ║
║                                                                              ║
║  Visualisations produites (toutes sur le TEST gelé, sauf la 1ère) :        ║
║    [A] Convergence de l'ENTRAÎNEMENT (synthétique) : reward global +       ║
║        amélioration % par instance synthétique                            ║
║    [B] Convergence détaillée (instances Solomon exemples, TEST) : coût     ║
║        final/meilleur par épisode local + trajectoire de coût             ║
║    [C] Tracés des routes AVANT (LNS) / APRÈS (DQN gelé) pour des           ║
║        instances Solomon exemples                                         ║
║    [D] Tableau récapitulatif : Instance / Nb véhicules / Distance / Gap %  ║
║        (TEST Solomon vs BKS)                                              ║
║    [E] Graphique dédié gap % vs BKS par instance + gap moyen              ║
║    [F] Diagnostic automatique de licence Gurobi (taille limitée ou non)   ║
║                                                                              ║
║  Aucun rappel manuel de "Code 1" n'est nécessaire : la génération +        ║
║  l'entraînement synthétique sont réintégrés ici (MODULE 1bis) et           ║
║  s'exécutent automatiquement avant le chargement de Solomon. Voir le flag  ║
║  optionnel CFG['REUSE_EXISTING_SYNTH_MODEL'] plus bas si vous préférez     ║
║  recharger un .pth déjà entraîné au lieu de ré-entraîner à chaque run.     ║
║                                                                              ║
║  Par défaut, INSTANCE_FILES contient toutes les instances *.txt trouvées   ║
║  dans outputs/Solomon/. Pour se limiter à 6 instances représentatives      ║
║  (une par groupe C1/C2/R1/R2/RC1/RC2), décommentez le bloc plus bas.       ║
║                                                                              ║
║  Exécution :  python main_vrptw_synth_train_solomon_convergence.py         ║
║                                                                              ║
║  CORRECTIONS APPORTÉES SUR CETTE VERSION :                                 ║
║   1. SOLOMON_DIR pointait vers un chemin Google Colab                      ║
║      ('/content/sample_data/Solomon') — corrigé pour pointer vers          ║
║      outputs/Solomon (dossier local Windows).                              ║
║   2. Le mode CFG['TAGUCHI_SYNTH']['ENABLE']=True s'arrêtait après la       ║
║      calibration et exigeait de relancer le script à la main pour la      ║
║      validation finale. Il enchaîne désormais AUTOMATIQUEMENT, dans le    ║
║      même lancement, sur l'entraînement complet (80 instances               ║
║      synthétiques) + le test gelé sur les 56 Solomon, en appliquant       ║
║      automatiquement la meilleure combinaison d'hyperparamètres trouvée.  ║
║      Voir run_final_validation() et le bloc __main__ tout en bas.         ║
║   3. Construction de la solution initiale remplacée par l'heuristique     ║
║      séquentielle I1 (Solomon, 1987) — cf. _build_initial_i1(), MODULE 2. ║
║      Remplace l'ancienne insertion gloutonne triée par earliest-time      ║
║      (conservée sous _build_initial_greedy_earliest, non appelée) comme   ║
║      point de départ du LNS, pour ENTRAÎNEMENT et TEST. Les meilleurs     ║
║      hyperparamètres Taguchi (LNS_ITER=100, K_DESTROY_RATIO=0.24,         ║
║      GUROBI_TLIMIT=2.0, GUROBI_GAP=0.05, MAX_STEPS=15, EPS_DECAY=0.99,    ║
║      VEHICLE_PENALTY=0.0) sont reportés en dur dans CFG, et le modèle     ║
║      DQN est réentraîné de zéro (REUSE_EXISTING_SYNTH_MODEL=False) car    ║
║      les instances vues à l'entraînement ont désormais une structure de   ║
║      routes de départ différente (voir CFG['I1_MU'/'I1_LAMBDA'/etc.]).    ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import os, sys, json, time, copy, random, csv, glob, itertools
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from contextlib import contextmanager
import torch
import torch.nn as nn
import torch.optim as optim
import gurobipy as gp
from gurobipy import GRB
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG GLOBALE
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    # ── ÉTAPE 1 : ENTRAÎNEMENT sur instances SYNTHÉTIQUES (jamais Solomon) ──
    N_SYNTH_INSTANCES          = 80,     # comme les 80 instances de Code 1
    SYNTH_N_CUSTOMERS          = 100,
    SYNTH_N_VEHICLES           = None,   # None → calculé depuis la demande (cf. generate_instance)
    SYNTH_VEHICLE_SAFETY_FACTOR= 1.5,
    SYNTH_CAPACITY             = 200,
    SYNTH_HORIZON              = 1000,
    EPISODES_PER_INSTANCE      = 20,     # épisodes d'ENTRAÎNEMENT par instance synthétique
    # Optionnel : True → recharge un .pth déjà entraîné sur du synthétique
    # (voir MODEL_FILE) au lieu de régénérer + ré-entraîner. False (par
    # défaut) → tout se fait automatiquement dans CE run, sans intervention
    # manuelle, aucun besoin de relancer "Code 1" séparément.
    # ⚠ Remis à False ici : la construction initiale change (heuristique I1
    # de Solomon 1987, cf. MODULE 2 plus bas, en remplacement de l'ancienne
    # insertion gloutonne triée par earliest-time) → les instances vues à
    # l'entraînement ont une structure de routes de départ différente, donc
    # on RÉENTRAÎNE le DQN de zéro plutôt que de recharger l'ancien .pth
    # (calibré sur l'ancienne construction).
    REUSE_EXISTING_SYNTH_MODEL = False,
    # False ici : les hyperparamètres ci-dessous (LNS_ITER, K_DESTROY_RATIO,
    # GUROBI_TLIMIT, GUROBI_GAP, MAX_STEPS, EPS_DECAY, VEHICLE_PENALTY) sont
    # déjà renseignés EN DUR avec la meilleure combinaison trouvée par le
    # plan Taguchi synthétique lors d'un run précédent — pas besoin de les
    # recharger depuis taguchi_meilleure_combinaison_synthetique.json à
    # l'exécution. Garde le comportement déterministe (n'importe pas le
    # contenu du JSON, qui a été calibré avec l'ANCIENNE heuristique de
    # construction initiale, avant l'ajout d'I1).
    USE_TAGUCHI_BEST            = False,
    # ── ÉTAPE 2 : TEST gelé sur les 56 instances Solomon officielles ───────
    EPISODES_PER_INSTANCE_TEST = 10,     # épisodes d'ÉVALUATION (aucun apprentissage) par instance Solomon
    # VEHICLE_SAFETY_FACTOR = None : on utilise la flotte OFFICIELLE de chaque
    # instance, pas une borne calculée sur la demande. Sur R101/RC101
    # (fenêtres de temps serrées), le BKS utilise 19 et 14 véhicules alors
    # que la demande seule suggérerait ~12-14 : une borne basée uniquement
    # sur la demande sous-estime le besoin réel et peut rendre le MILP
    # infaisable. La flotte officielle est plus grande (MILP plus lourd)
    # mais garantit qu'une solution proche du BKS reste atteignable.
    VEHICLE_SAFETY_FACTOR  = None,
    # ── Hyperparamètres LNS/DQN/Gurobi : meilleure combinaison trouvée par
    # le plan d'expérience Taguchi synthétique (fichier
    # taguchi_meilleure_combinaison_synthetique.json), reportée ici EN DUR
    # (voir USE_TAGUCHI_BEST=False ci-dessus). ────────────────────────────
    LNS_ITER                = 100,
    LNS_K_RATIO            = 0.20,
    K_DESTROY_RATIO        = 0.24,
    MAX_STEPS               = 15,
    # repair par bouts (Situation 1, appelé à CHAQUE step du LNS/DQN — voir
    # check_gurobi_license() plus bas pour l'impact d'une licence à taille
    # limitée). None = aucune limite de temps (supporté par le code).
    GUROBI_TLIMIT            = 2.0,
    GUROBI_GAP               = 0.05,
    HIDDEN                   = 64,
    LR                       = 1e-3,
    GAMMA                    = 0.95,
    EPS_START                = 1.0,
    EPS_END                  = 0.05,
    EPS_DECAY                = 0.99,  # best Taguchi — voir l'écrasement explicite juste après CFG plus bas
    BUFFER_SIZE               = 5000,
    BATCH_SIZE                = 32,
    TARGET_UPDATE              = 10,
    # ── Instances exemples (pour courbes détaillées + routes, TEST/Solomon) ─
    # Noms recherchés en priorité (représentatifs des 6 groupes Solomon) ;
    # si un nom n'est pas présent dans INSTANCE_FILES, il est ignoré et
    # remplacé automatiquement par le 1er type disponible.
    EXAMPLE_NAMES              = ['C101', 'R101', 'RC101'],
    # ── Pénalité par véhicule (repair MILP) ────────────────────────
    # 0.0 = comportement d'origine (aucune pénalité, Gurobi peut ouvrir
    # librement des véhicules supplémentaires à chaque repair). Une valeur
    # > 0 ajoute un coût fixe par véhicule utilisé dans l'objectif du MILP
    # de repair, pour limiter l'inflation du nombre de véhicules observée
    # dans la validation vs BKS. C'est aussi l'un des facteurs calibrés par
    # le plan d'expérience Taguchi (MODULE 10 plus bas) — calibré
    # UNIQUEMENT sur du synthétique (CFG['TAGUCHI_SYNTH']), jamais sur
    # Solomon, pour ne pas faire fuiter d'information du test set vers le
    # choix des hyperparamètres.
    VEHICLE_PENALTY            = 0.0,
    # ── Heuristique de construction initiale : I1 (Solomon, 1987) ──────
    # Remplace l'ancienne construction gloutonne triée par earliest-time
    # (_build_initial, conservée plus bas pour référence mais plus appelée).
    # Paramètres classiques de Solomon (1987) : mu=1 (poids de la distance
    # directe i→j dans le coût d'insertion c11), lambda=2 (poids de
    # l'éloignement au dépôt dans le critère de sélection c2 — favorise les
    # clients "difficiles" excentrés), alpha1=1/alpha2=0 (c1 = c11 pur,
    # sans le terme de décalage temporel c12 — paramétrage le plus courant
    # dans la littérature ; augmenter alpha2 pénalise davantage les
    # insertions qui retardent fortement le client suivant).
    I1_MU                       = 1.0,
    I1_LAMBDA                   = 2.0,
    I1_ALPHA1                   = 1.0,
    I1_ALPHA2                   = 0.0,
    OUTDIR                        = os.path.join(os.getcwd(), 'outputs'),
    MODELDIR                       = os.path.join(os.getcwd(), 'outputs', 'model'),
    MODEL_FILE                      = 'dqn_vrptw_synth_trained.pth',
)
CFG['LNS_K'] = 20   # ~20% de 100 clients, cohérent avec LNS_K_RATIO
CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))  # 24, cohérent avec K_DESTROY_RATIO
# ── Recalibrage automatique de EPS_DECAY : DÉSACTIVÉ ────────────────────
# À l'origine, ce bloc recalculait EPS_DECAY pour que epsilon décroisse de
# EPS_START à EPS_END sur toute la durée de l'entraînement (utile quand
# EPS_DECAY n'était pas calibré). Ici, EPS_DECAY=0.99 vient directement de
# la meilleure combinaison trouvée par le plan Taguchi synthétique (cf.
# CFG plus haut) — un recalcul automatique l'écraserait. On le laisse donc
# tel quel ; ce bloc est conservé en commentaire pour référence/mémoire si
# jamais EPS_DECAY doit être re-dérivé automatiquement dans une variante
# future du script.
# _n_decays_total = max(1, CFG['N_SYNTH_INSTANCES'] * CFG['EPISODES_PER_INSTANCE'])
# CFG['EPS_DECAY'] = (CFG['EPS_END'] / CFG['EPS_START']) ** (1.0 / _n_decays_total)
os.makedirs(CFG['OUTDIR'],   exist_ok=True)
os.makedirs(CFG['MODELDIR'], exist_ok=True)
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG DU PLAN D'EXPÉRIENCE TAGUCHI (calibration des hyperparamètres)
# ══════════════════════════════════════════════════════════════════════════════
# Deux modes de calibration disponibles (voir plus bas pour le mode
# TAGUCHI_SYNTH, recommandé, qui enchaîne désormais automatiquement sur la
# validation finale). Activez explicitement l'UN DES DEUX (pas les deux en
# même temps) :
#
#  • CFG['TAGUCHI_SYNTH']['ENABLE'] = True   (RECOMMANDÉ)
#    Calibre en s'entraînant UNIQUEMENT sur des instances SYNTHÉTIQUES
#    générées à la volée (jamais Solomon). Métrique de classement :
#    amélioration % vs coût LNS initial (pas de BKS requis). La meilleure
#    combinaison trouvée n'a jamais "vu" les 56 instances Solomon → aucune
#    fuite d'hyperparamètres vers le jeu de test, contrairement au mode
#    historique ci-dessous. Une fois la calibration terminée, le script
#    enchaîne AUTOMATIQUEMENT sur la validation finale (entraînement 80
#    instances + test gelé sur les 56 Solomon) avec ces hyperparamètres.
#
#  • CFG['TAGUCHI']['ENABLE'] = True   (⚠ HISTORIQUE — NON RECOMMANDÉ)
#    Calibre en s'entraînant DIRECTEMENT sur les 56 instances Solomon (la
#    matrice externe = le jeu de test lui-même) et classe les combinaisons
#    par leur gap % vs BKS sur ces mêmes instances. C'est exactement le
#    protocole "Run 16" identifié comme fuite d'hyperparamètres (on choisit
#    les hyperparamètres en fonction de la performance sur le test) —
#    conservé uniquement pour comparaison / compatibilité avec d'anciens
#    résultats, PAS pour produire les hyperparamètres d'une validation
#    finale train/test disjointe. Ce mode n'enchaîne PAS automatiquement
#    sur la validation finale (voir le bloc __main__ tout en bas).
CFG['TAGUCHI'] = dict(
    ENABLE                     = False,  # ⚠ historique — fuite vers Solomon (test set), voir ci-dessus
    EPISODES_PER_INSTANCE_DOE  = 5,
    OUTER_INSTANCE_NAMES       = 'ALL',  # 'ALL' = les 56 instances Solomon comme matrice externe
)
CFG['TAGUCHI_SYNTH'] = dict(
    # False ici : la calibration a déjà été faite lors d'un run précédent
    # (le JSON taguchi_meilleure_combinaison_synthetique.json existe déjà).
    # Repasser à True relancerait tout le plan Taguchi (18 runs) pour rien —
    # CFG['USE_TAGUCHI_BEST']=True ci-dessus suffit à réutiliser ses résultats.
    ENABLE                     = False,  # True = lance le plan Taguchi calibré sur synthétique (recommandé)
    # Volume d'instances synthétiques utilisé comme matrice externe pendant
    # CE plan de calibration (indépendant de N_SYNTH_INSTANCES du protocole
    # d'entraînement final — un sous-ensemble suffit ici, l'objectif est de
    # classer les 18 combinaisons, pas d'entraîner le modèle définitif).
    N_INSTANCES_DOE            = 24,   # 8 par type C/R/RC
    EPISODES_PER_INSTANCE_DOE  = 5,
)
# Jeu d'actions actif du DQN pour un run normal (validation / Taguchi) :
# 'full' (4 actions, défaut, comportement inchangé) / 'merged3' (3) /
# 'merged2' (2) — voir ACTION_SETS_N / ACTION_NAMES_BY_SET et MODULE 11
# (ABLATION DES ACTIONS DQN) plus bas. Ne changez ceci à la main que pour
# lancer une validation normale avec un jeu d'actions fusionné précis ; pour
# COMPARER les 3 jeux, utilisez plutôt CFG['ACTION_ABLATION']['ENABLE']=True
# ci-dessous, qui boucle automatiquement sur les 3 sans y toucher ici.
CFG['ACTION_SET'] = 'full'
# ── Ablation de l'espace d'actions du DQN (étude comparative, MODULE 11) ──
# ENABLE=True : au lieu de la validation normale, lance
# run_action_space_ablation(), qui répète EXACTEMENT le protocole
# ENTRAÎNEMENT (synthétique) / TEST (Solomon gelé) de run_final_validation()
# une fois par jeu d'actions dans SETS — mêmes CFG['N_SYNTH_INSTANCES'],
# CFG['EPISODES_PER_INSTANCE'], CFG['EPISODES_PER_INSTANCE_TEST'] et mêmes
# hyperparamètres LNS/Gurobi par ailleurs, SEUL CFG['ACTION_SET'] change
# d'un run à l'autre — pour une comparaison à hyperparamètres égaux.
# ⚠ Ce mode s'exécute 3× le temps d'un run normal (un entraînement+test
# complet par jeu d'actions). Repassez ENABLE=False pour revenir à la
# validation normale.
CFG['ACTION_ABLATION'] = dict(
    ENABLE = True,
    SETS   = ['full', 'merged3', 'merged2'],
)
# Facteurs de contrôle et leurs niveaux (plan Taguchi L18 : 7 facteurs à 3
# niveaux, 18 combinaisons au lieu des 3^7 = 2187 combinaisons possibles).
TAGUCHI_FACTORS = [
    ('LNS_ITER',         [30, 60, 100]),
    ('K_DESTROY_RATIO',  [0.15, 0.20, 0.24]),
    ('GUROBI_TLIMIT',    [1.0, 2.0, 3.0]),
    ('GUROBI_GAP',       [0.05, 0.03, 0.01]),
    ('MAX_STEPS',        [10, 15, 20]),
    ('EPS_DECAY',        [0.90, 0.95, 0.99]),
    ('VEHICLE_PENALTY',  [0.0, 300.0, 800.0]),
]
GROUP_ORDER = ['C1', 'C2', 'R1', 'R2', 'RC1', 'RC2']
ACTION_NAMES = {
    0: "Destruction spatiale",
    1: "Destruction temporelle",
    2: "Destruction par coût élevé",
    3: "Destruction aléatoire",
}
# ── Ablation de l'espace d'actions du DQN (voir MODULE 11) ─────────────────
# 'full'    : 4 actions d'origine (spatial, temporel, coût élevé, aléatoire)
# 'merged3' : 3 actions — spatial+temporel FUSIONNÉS en une seule action
#             "spatio-temporelle" (moyenne 0.5/0.5 des scores normalisés,
#             voir VRPTWEnv._spatiotemporal_destroy), coût élevé et
#             aléatoire inchangés séparément.
# 'merged2' : 2 actions — la même fusion spatio-temporelle, + aléatoire.
#             L'action "coût élevé" est ici purement SUPPRIMÉE (pas
#             fusionnée) : c'est le jeu d'actions le plus réduit.
ACTION_SETS_N = {'full': 4, 'merged3': 3, 'merged2': 2}
ACTION_NAMES_BY_SET = {
    'full':    {0: "Spatiale", 1: "Temporelle", 2: "Coût élevé", 3: "Aléatoire"},
    'merged3': {0: "Spatio-temporelle (fusion)", 1: "Coût élevé", 2: "Aléatoire"},
    'merged2': {0: "Spatio-temporelle (fusion)", 1: "Aléatoire"},
}
# ── Dossier contenant les instances Solomon officielles (extraites du zip) ──
# CORRIGÉ : pointait vers '/content/sample_data/Solomon' (chemin Google
# Colab, inexistant sur une machine locale) — repointé vers outputs/Solomon.
SOLOMON_DIR = os.path.join(CFG['OUTDIR'], 'Solomon')
# Toutes les instances *.txt trouvées (décommentez le bloc ci-dessous pour
# revenir à seulement 6 instances représentatives, une par groupe
# C1/C2/R1/R2/RC1/RC2, afin de garder un temps d'exécution raisonnable).
INSTANCE_FILES = sorted(glob.glob(os.path.join(SOLOMON_DIR, '*.txt')))
# INSTANCE_FILES = [
#     os.path.join(SOLOMON_DIR, 'c101.txt'),
#     os.path.join(SOLOMON_DIR, 'c201.txt'),
#     os.path.join(SOLOMON_DIR, 'r101.txt'),
#     os.path.join(SOLOMON_DIR, 'r201.txt'),
#     os.path.join(SOLOMON_DIR, 'rc101.txt'),
#     os.path.join(SOLOMON_DIR, 'rc201.txt'),
# ]
# ── BKS officiel complet, 56 instances (source : SINTEF / Solomon 1987) ──────
BKS_REFERENCE = {
    'C101': {'vehicles': 10, 'distance': 828.94}, 'C102': {'vehicles': 10, 'distance': 828.94},
    'C103': {'vehicles': 10, 'distance': 828.06}, 'C104': {'vehicles': 10, 'distance': 824.78},
    'C105': {'vehicles': 10, 'distance': 828.94}, 'C106': {'vehicles': 10, 'distance': 828.94},
    'C107': {'vehicles': 10, 'distance': 828.94}, 'C108': {'vehicles': 10, 'distance': 828.94},
    'C109': {'vehicles': 10, 'distance': 828.94},
    'C201': {'vehicles': 3, 'distance': 591.56}, 'C202': {'vehicles': 3, 'distance': 591.56},
    'C203': {'vehicles': 3, 'distance': 591.17}, 'C204': {'vehicles': 3, 'distance': 590.60},
    'C205': {'vehicles': 3, 'distance': 588.88}, 'C206': {'vehicles': 3, 'distance': 588.49},
    'C207': {'vehicles': 3, 'distance': 588.29}, 'C208': {'vehicles': 3, 'distance': 588.32},
    'R101': {'vehicles': 19, 'distance': 1650.80}, 'R102': {'vehicles': 17, 'distance': 1486.12},
    'R103': {'vehicles': 13, 'distance': 1292.68}, 'R104': {'vehicles': 9,  'distance': 1007.31},
    'R105': {'vehicles': 14, 'distance': 1377.11}, 'R106': {'vehicles': 12, 'distance': 1252.03},
    'R107': {'vehicles': 10, 'distance': 1104.66}, 'R108': {'vehicles': 9,  'distance': 960.88},
    'R109': {'vehicles': 11, 'distance': 1194.73}, 'R110': {'vehicles': 10, 'distance': 1118.84},
    'R111': {'vehicles': 10, 'distance': 1096.72}, 'R112': {'vehicles': 9,  'distance': 982.14},
    'R201': {'vehicles': 4, 'distance': 1252.37}, 'R202': {'vehicles': 3, 'distance': 1191.70},
    'R203': {'vehicles': 3, 'distance': 939.50},  'R204': {'vehicles': 2, 'distance': 825.52},
    'R205': {'vehicles': 3, 'distance': 994.43},  'R206': {'vehicles': 3, 'distance': 906.14},
    'R207': {'vehicles': 2, 'distance': 890.61},  'R208': {'vehicles': 2, 'distance': 726.82},
    'R209': {'vehicles': 3, 'distance': 909.16},  'R210': {'vehicles': 3, 'distance': 939.37},
    'R211': {'vehicles': 2, 'distance': 885.71},
    'RC101': {'vehicles': 14, 'distance': 1696.95}, 'RC102': {'vehicles': 12, 'distance': 1554.75},
    'RC103': {'vehicles': 11, 'distance': 1261.67}, 'RC104': {'vehicles': 10, 'distance': 1135.48},
    'RC105': {'vehicles': 13, 'distance': 1629.44}, 'RC106': {'vehicles': 11, 'distance': 1424.73},
    'RC107': {'vehicles': 11, 'distance': 1230.48}, 'RC108': {'vehicles': 10, 'distance': 1139.82},
    'RC201': {'vehicles': 4, 'distance': 1406.94}, 'RC202': {'vehicles': 3, 'distance': 1365.65},
    'RC203': {'vehicles': 3, 'distance': 1049.62}, 'RC204': {'vehicles': 3, 'distance': 798.46},
    'RC205': {'vehicles': 4, 'distance': 1297.65}, 'RC206': {'vehicles': 3, 'distance': 1146.32},
    'RC207': {'vehicles': 3, 'distance': 1061.14}, 'RC208': {'vehicles': 3, 'distance': 828.14},
}
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1  —  LECTURE DES INSTANCES SOLOMON OFFICIELLES  (jeu de TEST)
# ══════════════════════════════════════════════════════════════════════════════
def euclidean(a, b):
    return float(np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2))
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1bis  —  GÉNÉRATEUR D'INSTANCES SYNTHÉTIQUES  (jeu d'ENTRAÎNEMENT)
#  Repris tel quel de main_vrptw_comparaison_gurobi.py ("Code 1") : mêmes
#  fonctions _coords_C/_coords_R/_coords_RC et generate_instance(), pour que
#  les instances d'entraînement soient générées exactement comme dans Code 1.
# ══════════════════════════════════════════════════════════════════════════════
def _coords_C(n, rng):
    n_cl = max(3, n // 10)
    centers = rng.uniform(15, 85, (n_cl, 2))
    pts = []
    for i in range(n):
        p = rng.normal(centers[i % n_cl], 8)
        pts.append(np.clip(p, 1, 99))
    return np.array(pts)
def _coords_R(n, rng):
    return rng.uniform(1, 99, (n, 2))
def _coords_RC(n, rng):
    h = n // 2
    return np.vstack([_coords_C(h, rng), _coords_R(n - h, rng)])
def generate_instance(n_customers=50, n_vehicles=None, capacity=200,
                      type_='C', seed=42, horizon=1000,
                      vehicle_safety_factor=1.5):
    rng     = np.random.RandomState(seed)
    depot   = np.array([50.0, 50.0])
    gen_fn  = {'C': _coords_C, 'R': _coords_R, 'RC': _coords_RC}[type_]
    coords  = gen_fn(n_customers, rng)
    max_dem = max(1, capacity // 5)
    demands = rng.randint(1, max_dem + 1, n_customers)
    svc     = rng.randint(10, 31, n_customers)
    if n_vehicles is None:
        min_vehicles = int(np.ceil(demands.sum() / capacity))
        n_vehicles   = max(2, int(np.ceil(min_vehicles * vehicle_safety_factor)))
    customers = []
    for i in range(n_customers):
        d        = euclidean(depot, coords[i])
        earliest = max(0.0, round(d + rng.uniform(-15, 15), 1))
        delta    = rng.uniform(30, 80)
        latest   = min(float(horizon), round(earliest + delta, 1))
        latest   = max(latest, earliest + svc[i] + 5)
        latest   = min(latest, float(horizon))
        if d > latest:
            earliest = round(d, 1)
            latest   = min(float(horizon), round(earliest + delta, 1))
            latest   = max(latest, earliest + svc[i] + 5)
        if d + svc[i] + d > horizon:
            svc[i] = max(5, int(horizon - 2 * d - 5))
        customers.append({
            'id':           i + 1,
            'x':            float(round(coords[i][0], 2)),
            'y':            float(round(coords[i][1], 2)),
            'demand':       int(demands[i]),
            'earliest':     float(earliest),
            'latest':       float(latest),
            'service_time': int(svc[i]),
        })
    return {
        'depot': {
            'id': 0, 'x': 50.0, 'y': 50.0, 'demand': 0,
            'earliest': 0.0, 'latest': float(horizon), 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    capacity,
        'horizon':     horizon,
        'type':        type_,
        'seed':        seed,
        'n_customers': n_customers,
    }
def generate_synthetic_training_set(cfg, verbose=True):
    """
    Génère les instances synthétiques d'ENTRAÎNEMENT en cyclant les 3 types
    Solomon C / R / RC (même logique que train() dans Code 1 : type =
    ['C','R','RC'][i % 3], seed = i*7+1). Ces instances ne recouvrent JAMAIS
    les 56 instances Solomon officielles utilisées plus bas comme jeu de
    TEST : le DQN ne les voit donc jamais avant l'évaluation gelée.
    Chaque instance reçoit en plus une clé 'name' (ex. 'SYN-C001') pour
    rester compatible avec les fonctions d'entraînement/affichage qui
    utilisent inst['name'].
    """
    types = ['C', 'R', 'RC']
    instances = []
    if verbose:
        _sec(f"ÉTAPE 1 — GÉNÉRATION DE {cfg['N_SYNTH_INSTANCES']} INSTANCES "
             f"SYNTHÉTIQUES (ENTRAÎNEMENT)")
    for i in range(cfg['N_SYNTH_INSTANCES']):
        t    = types[i % 3]
        seed = i * 7 + 1
        inst = generate_instance(
            n_customers = cfg['SYNTH_N_CUSTOMERS'],
            n_vehicles  = cfg['SYNTH_N_VEHICLES'],
            capacity    = cfg['SYNTH_CAPACITY'],
            horizon     = cfg['SYNTH_HORIZON'],
            type_       = t,
            seed        = seed,
            vehicle_safety_factor = cfg['SYNTH_VEHICLE_SAFETY_FACTOR'],
        )
        inst['name'] = f"SYN-{t}{i+1:03d}"
        instances.append(inst)
        if verbose:
            print(f"  Instance {i+1:03d}/{cfg['N_SYNTH_INSTANCES']} | {inst['name']:<10} "
                  f"type={t:<3} | seed={seed} | véhicules={inst['n_vehicles']}")
    return instances
def parse_solomon_instance(filepath, vehicle_safety_factor=None):
    """
    Parse un fichier d'instance Solomon au format texte officiel :
        <NOM>
        VEHICLE
        NUMBER  CAPACITY
        <n>     <cap>
        CUSTOMER
        CUST NO.  XCOORD.  YCOORD.  DEMAND  READY TIME  DUE DATE  SERVICE TIME
        0  ...   (dépôt)
        1  ...
        ...
    Retourne un dict compatible avec le reste du pipeline.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = [l.rstrip('\n') for l in f]
    name = lines[0].strip()
    idx_veh = next(i for i, l in enumerate(lines) if l.strip() == 'VEHICLE')
    veh_number, veh_capacity = map(int, lines[idx_veh + 2].split())
    idx_cust = next(i for i, l in enumerate(lines) if l.strip() == 'CUSTOMER')
    data_lines = [l for l in lines[idx_cust + 2:] if l.strip()]
    rows = [list(map(float, l.split())) for l in data_lines]
    depot_row = rows[0]
    customers = []
    for r in rows[1:]:
        cid, x, y, dem, ready, due, svc = r
        customers.append({
            'id':           int(cid),
            'x':            x,
            'y':            y,
            'demand':       int(dem),
            'earliest':     ready,
            'latest':       due,
            'service_time': int(svc),
        })
    name_upper = name.upper()
    type_ = ''.join(ch for ch in name_upper if ch.isalpha())   # 'C101' -> 'C', 'RC101' -> 'RC'
    n_vehicles = veh_number
    if vehicle_safety_factor is not None:
        total_demand = sum(c['demand'] for c in customers)
        min_veh      = int(np.ceil(total_demand / veh_capacity))
        n_vehicles   = min(veh_number, max(2, int(np.ceil(min_veh * vehicle_safety_factor))))
    return {
        'depot': {
            'id': 0, 'x': depot_row[1], 'y': depot_row[2], 'demand': 0,
            'earliest': depot_row[4], 'latest': depot_row[5], 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    veh_capacity,
        'horizon':     depot_row[5],
        'type':        type_,
        'seed':        name_upper,
        'n_customers': len(customers),
        'name':        name_upper,          # nom officiel normalisé (ex. 'C101')
    }
def load_all_instances(files, vehicle_safety_factor=None):
    if not files:
        raise FileNotFoundError(
            f"Aucun fichier d'instance trouvé dans {SOLOMON_DIR}. "
            f"Vérifiez que le dossier existe et contient les .txt Solomon."
        )
    instances = []
    for fp in files:
        inst = parse_solomon_instance(fp, vehicle_safety_factor=vehicle_safety_factor)
        instances.append(inst)
        print(f"  Chargé : {inst['name']:<8} type={inst['type']:<3} "
              f"clients={inst['n_customers']} véhicules={inst['n_vehicles']} "
              f"capacité={inst['capacity']} horizon={inst['horizon']}")
    return instances
def pick_example_indices(instances, preferred_names=None, max_examples=3):
    """
    Sélectionne des instances "exemples" pour les courbes détaillées et les
    tracés de routes. Cherche d'abord les noms préférés (ex. C101/R101/RC101).
    Si absents, retombe sur la 1ère instance de chaque type détecté, puis sur
    les toutes premières instances de la liste.
    """
    preferred_names = preferred_names or []
    by_name = {inst['name']: i for i, inst in enumerate(instances)}
    idxs = [by_name[nm] for nm in preferred_names if nm in by_name]
    if not idxs:
        seen_types = {}
        for i, inst in enumerate(instances):
            if inst['type'] not in seen_types:
                seen_types[inst['type']] = i
            if len(seen_types) >= max_examples:
                break
        idxs = list(seen_types.values())
    if not idxs:
        idxs = list(range(min(max_examples, len(instances))))
    return idxs[:max_examples]
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 2  —  SOLUTION INITIALE PAR LNS  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
def _node_xy(nid, instance):
    if nid == 0:
        return instance['depot']['x'], instance['depot']['y']
    c = instance['customers'][nid - 1]
    return c['x'], c['y']
def _node_info(nid, instance):
    if nid == 0:
        return instance['depot']
    return instance['customers'][nid - 1]
def compute_cost(solution, instance):
    depot = (instance['depot']['x'], instance['depot']['y'])
    total = 0.0
    for route in solution:
        if not route:
            continue
        prev = depot
        for cid in route:
            cur   = _node_xy(cid, instance)
            total += euclidean(prev, cur)
            prev  = cur
        total += euclidean(prev, depot)
    return total
def _route_feasible(route, instance):
    if not route:
        return True
    cap  = instance['capacity']
    load = sum(_node_info(c, instance)['demand'] for c in route)
    if load > cap:
        return False
    depot = instance['depot']
    t     = 0.0
    px, py = depot['x'], depot['y']
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        if t > info['latest']:
            return False
        t      = max(t, info['earliest']) + info['service_time']
        px, py = cx, cy
    t += euclidean((px, py), (depot['x'], depot['y']))
    return t <= depot['latest']
def is_feasible(solution, instance):
    return all(_route_feasible(r, instance) for r in solution)
def solution_stats(solution, instance):
    cap        = instance['capacity']
    total_cost = compute_cost(solution, instance)
    n_vehicles = sum(1 for r in solution if r)
    loads      = []
    total_wait = 0.0
    for route in solution:
        if not route:
            continue
        load = sum(_node_info(c, instance)['demand'] for c in route)
        loads.append(load / cap)
        depot  = instance['depot']
        t      = 0.0
        px, py = depot['x'], depot['y']
        for cid in route:
            cx, cy = _node_xy(cid, instance)
            t     += euclidean((px, py), (cx, cy))
            info   = _node_info(cid, instance)
            total_wait += max(0.0, info['earliest'] - t)
            t      = max(t, info['earliest']) + info['service_time']
            px, py = cx, cy
    return {
        'total_cost':         round(total_cost, 2),
        'n_vehicles':         n_vehicles,
        'avg_load_ratio':     round(np.mean(loads) if loads else 0.0, 3),
        'total_wait_time':    round(total_wait, 2),
        'n_customers_served': sum(len(r) for r in solution),
    }
def _insertion_cost(cid, pos, route, instance):
    depot   = (instance['depot']['x'], instance['depot']['y'])
    nodes   = [0] + route + [0]
    prev_id = nodes[pos]
    next_id = nodes[pos + 1]
    px, py  = _node_xy(prev_id, instance) if prev_id != 0 else depot
    nx, ny  = _node_xy(next_id, instance) if next_id != 0 else depot
    cx, cy  = _node_xy(cid, instance)
    return (euclidean((px, py), (cx, cy)) + euclidean((cx, cy), (nx, ny))
            - euclidean((px, py), (nx, ny)))
def _try_insert(cid, route, instance):
    best_cost, best_pos = float('inf'), -1
    for pos in range(len(route) + 1):
        candidate = route[:pos] + [cid] + route[pos:]
        if _route_feasible(candidate, instance):
            cost = _insertion_cost(cid, pos, route, instance)
            if cost < best_cost:
                best_cost, best_pos = cost, pos
    return best_cost, best_pos
def _repair_greedy(removed, partial, instance):
    routes = [list(r) for r in partial]
    for cid in removed:
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _build_initial_greedy_earliest(instance):
    """
    ⚠ CONSERVÉE POUR RÉFÉRENCE UNIQUEMENT — n'est plus appelée par
    get_initial_solution() (remplacée par _build_initial_i1, l'heuristique
    I1 de Solomon (1987) ci-dessous). Construction gloutonne d'origine :
    trie les clients par earliest-time puis insère chacun au meilleur
    endroit (au sens du coût d'insertion) parmi les routes déjà ouvertes,
    ou ouvre une nouvelle route si aucune insertion n'est faisable.
    """
    customers = sorted(instance['customers'], key=lambda c: c['earliest'])
    routes    = []
    for c in customers:
        cid = c['id']
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _route_service_starts(route, instance):
    """
    Propage les temps le long d'une route (sans le dépôt) et retourne la
    liste des heures de DÉBUT DE SERVICE b_j pour chaque client de la route,
    dans l'ordre. Utilisé par l'heuristique I1 pour calculer le terme de
    décalage temporel c12 (poussée en avant de l'heure de service du client
    suivant, cf. Solomon 1987).
    """
    depot  = instance['depot']
    t      = 0.0
    px, py = depot['x'], depot['y']
    starts = []
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        t      = max(t, info['earliest'])
        starts.append(t)
        t     += info['service_time']
        px, py = cx, cy
    return starts
def _build_initial_i1(instance, mu=1.0, lam=2.0, alpha1=1.0, alpha2=0.0):
    """
    Heuristique de construction séquentielle I1 (Solomon, 1987, "Algorithms
    for the Vehicle Routing and Scheduling Problems with Time Window
    Constraints", Operations Research 35(2)). Remplace l'ancienne
    construction gloutonne triée par earliest-time (_build_initial_greedy_
    earliest ci-dessus) comme point de départ du LNS.
    Principe : construit les routes une par une. Chaque route démarre avec
    un client "graine" (le client non routé le plus éloigné du dépôt), puis
    insère itérativement, parmi tous les clients non routés et toutes les
    positions faisables de la route courante, celui qui maximise le critère
    de sélection c2(u) = lambda*d(0,u) - c1*(u) — favorisant les clients
    excentrés difficiles à insérer plus tard — où c1*(u) est le coût
    d'insertion minimal de u dans la route courante :
        c1(i,u,j) = alpha1*c11(i,u,j) + alpha2*c12(i,u,j)
        c11(i,u,j) = d(i,u) + d(u,j) - mu*d(i,j)                  (détour géométrique)
        c12(i,u,j) = b_ju - b_j                                    (décalage de service induit chez j)
    Quand plus aucun client non routé ne peut être inséré faisablement
    (capacité/fenêtres de temps, vérifiées via _route_feasible) dans la
    route courante, celle-ci est fermée et une nouvelle route est ouverte
    avec un nouveau client graine, jusqu'à ce que tous les clients soient
    routés. Paramètres mu/lambda/alpha1/alpha2 : cf. CFG['I1_MU'] etc.
    """
    def d(a, b):
        return euclidean(_node_xy(a, instance), _node_xy(b, instance))
    unrouted = [c['id'] for c in instance['customers']]
    d0 = {cid: d(0, cid) for cid in unrouted}
    routes = []
    while unrouted:
        seed = max(unrouted, key=lambda c: d0[c])
        route = [seed]
        unrouted.remove(seed)
        while True:
            times_before = _route_service_starts(route, instance)
            nodes = [0] + route + [0]
            best_u, best_pos, best_c1, best_c2 = None, None, None, -float('inf')
            for u in unrouted:
                u_best_cost, u_best_pos = None, None
                cx, cy = _node_xy(u, instance)
                info_u = _node_info(u, instance)
                for pos in range(len(route) + 1):
                    i_id, j_id = nodes[pos], nodes[pos + 1]
                    candidate = route[:pos] + [u] + route[pos:]
                    if not _route_feasible(candidate, instance):
                        continue
                    dij = d(i_id, j_id)
                    c11 = d(i_id, u) + d(u, j_id) - mu * dij
                    c12 = 0.0
                    if alpha2 != 0.0 and j_id != 0:
                        if pos == 0:
                            depart_i, px, py = 0.0, instance['depot']['x'], instance['depot']['y']
                        else:
                            info_i = _node_info(i_id, instance)
                            depart_i = times_before[pos - 1] + info_i['service_time']
                            px, py   = _node_xy(i_id, instance)
                        arrival_u = depart_i + euclidean((px, py), (cx, cy))
                        b_u       = max(arrival_u, info_u['earliest'])
                        depart_u  = b_u + info_u['service_time']
                        jx, jy    = _node_xy(j_id, instance)
                        arrival_j = depart_u + euclidean((cx, cy), (jx, jy))
                        info_j    = _node_info(j_id, instance)
                        b_ju      = max(arrival_j, info_j['earliest'])
                        c12       = b_ju - times_before[pos]
                    c1 = alpha1 * c11 + alpha2 * c12
                    if u_best_cost is None or c1 < u_best_cost:
                        u_best_cost, u_best_pos = c1, pos
                if u_best_cost is None:
                    continue
                c2 = lam * d0[u] - u_best_cost
                if c2 > best_c2:
                    best_c2, best_u, best_pos, best_c1 = c2, u, u_best_pos, u_best_cost
            if best_u is None:
                break
            route.insert(best_pos, best_u)
            unrouted.remove(best_u)
        routes.append(route)
    return routes
def _lns_improve(solution, instance, n_iter=50, k=10, seed=0):
    rng       = np.random.RandomState(seed)
    current   = [list(r) for r in solution]
    best      = copy.deepcopy(current)
    best_cost = compute_cost(best, instance)
    for _ in range(n_iter):
        all_c   = [c for r in current for c in r]
        k_      = min(k, max(1, len(all_c) - 1))
        removed = list(rng.choice(all_c, k_, replace=False))
        removed_set = set(removed)
        partial = [[c for c in r if c not in removed_set] for r in current]
        partial = [r for r in partial if r]
        candidate = _repair_greedy(removed, partial, instance)
        c_cost    = compute_cost(candidate, instance)
        if c_cost < best_cost and is_feasible(candidate, instance):
            best, best_cost = copy.deepcopy(candidate), c_cost
        current = candidate
    return best
def get_initial_solution(instance, lns_iter=50, lns_k=10):
    sol = _build_initial_i1(
        instance,
        mu=CFG.get('I1_MU', 1.0), lam=CFG.get('I1_LAMBDA', 2.0),
        alpha1=CFG.get('I1_ALPHA1', 1.0), alpha2=CFG.get('I1_ALPHA2', 0.0),
    )
    sol = _lns_improve(sol, instance, n_iter=lns_iter, k=lns_k)
    sol = [r for r in sol if r]
    return sol, solution_stats(sol, instance)
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 3  —  ENVIRONNEMENT VRPTW  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class VRPTWEnv:
    def __init__(self, instance, initial_solution,
                 K=15, max_steps=20,
                 gurobi_tlimit=2.0, gurobi_gap=0.05,
                 vehicle_penalty=0.0,
                 action_set='full',
                 verbose=False):
        self.instance    = instance
        self.initial_sol = copy.deepcopy(initial_solution)
        self.K           = K
        self.max_steps   = max_steps
        self.tlimit      = gurobi_tlimit
        self.gap         = gurobi_gap
        self.vehicle_penalty = vehicle_penalty
        self.verbose     = verbose
        self.obs_dim     = 7
        # Jeu d'actions actif (ablation, voir ACTION_SETS_N / MODULE 11) :
        # 'full' (4 actions, défaut) / 'merged3' (3) / 'merged2' (2). Doit
        # correspondre exactement au n_actions du DQNAgent utilisé avec cet
        # environnement (voir train_on_instances/evaluate_frozen_on_instances).
        self.action_set  = action_set
        self.n_actions   = ACTION_SETS_N.get(action_set, 4)
        self._dist = {}
        depot = (instance['depot']['x'], instance['depot']['y'])
        for c in instance['customers']:
            d = euclidean(depot, (c['x'], c['y']))
            self._dist[(0, c['id'])] = d
            self._dist[(c['id'], 0)] = d
        for ci in instance['customers']:
            for cj in instance['customers']:
                self._dist[(ci['id'], cj['id'])] = euclidean(
                    (ci['x'], ci['y']), (cj['x'], cj['y']))
        self.reset()
    def _d(self, a, b):
        return self._dist.get((a, b), 0.0)
    def _info(self, nid):
        return _node_info(nid, self.instance)
    def reset(self, start_solution=None):
        if start_solution is not None:
            self.solution = copy.deepcopy(start_solution)
        else:
            self.solution = copy.deepcopy(self.initial_sol)
        self.current_cost     = compute_cost(self.solution, self.instance)
        self.best_cost        = self.current_cost
        self.initial_cost     = compute_cost(self.initial_sol, self.instance)
        self.step_count       = 0
        self.steps_no_improve = 0
        self._action_history  = []
        return self._obs()
    def _obs(self):
        stats = solution_stats(self.solution, self.instance)
        ic    = max(self.initial_cost, 1.0)
        return np.array([
            self.current_cost / ic,
            stats['n_vehicles'] / max(self.instance['n_vehicles'], 1),
            stats['avg_load_ratio'],
            stats['total_wait_time'] / max(ic * 0.5, 1.0),
            self.steps_no_improve / self.max_steps,
            self.best_cost / ic,
            1.0,
        ], dtype=np.float32)
    def _spatial_scores(self, pivot, all_c):
        px, py = self._info(pivot)['x'], self._info(pivot)['y']
        return {c: euclidean((px, py), (self._info(c)['x'], self._info(c)['y']))
                for c in all_c}
    def _temporal_scores(self, pivot, all_c):
        e_pivot = self._info(pivot)['earliest']
        return {c: abs(self._info(c)['earliest'] - e_pivot) for c in all_c}
    def _cost_scores(self):
        savings = {}
        for r in self.solution:
            if not r:
                continue
            nodes = [0] + r + [0]
            for i, cid in enumerate(r):
                prev, nxt    = nodes[i], nodes[i + 2]
                savings[cid] = (self._d(prev, cid) + self._d(cid, nxt)
                                 - self._d(prev, nxt))
        return savings
    def _spatiotemporal_destroy(self, rng, all_c, K):
        """
        Action FUSIONNÉE (ablation, jeux 'merged3'/'merged2', voir MODULE 11) :
        combine la proximité spatiale ET temporelle par rapport à un même
        pivot. Chaque critère est normalisé sur [0,1] (divisé par son max sur
        les clients courants) puis moyenné à parts égales (0.5/0.5), pour
        qu'aucune des deux échelles ne domine l'autre.
        """
        pivot = int(rng.choice(all_c))
        sp = self._spatial_scores(pivot, all_c)
        tp = self._temporal_scores(pivot, all_c)
        sp_max = max(sp.values()) or 1.0
        tp_max = max(tp.values()) or 1.0
        combined = {c: 0.5 * (sp[c] / sp_max) + 0.5 * (tp[c] / tp_max) for c in all_c}
        return sorted(all_c, key=lambda c: combined[c])[:K]
    def _destroy(self, action):
        all_c = [c for r in self.solution for c in r]
        if not all_c:
            return [], copy.deepcopy(self.solution)
        K   = min(self.K, len(all_c))
        rng = np.random.RandomState(self.step_count + action * 100)
        if self.action_set == 'full':
            # 4 actions d'origine : spatial, temporel, coût élevé, aléatoire
            if action == 0:
                pivot     = int(rng.choice(all_c))
                scores    = self._spatial_scores(pivot, all_c)
                destroyed = sorted(all_c, key=lambda c: scores[c])[:K]
            elif action == 1:
                pivot     = int(rng.choice(all_c))
                scores    = self._temporal_scores(pivot, all_c)
                destroyed = sorted(all_c, key=lambda c: scores[c])[:K]
            elif action == 2:
                savings   = self._cost_scores()
                destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
            else:
                idx       = rng.choice(len(all_c), K, replace=False)
                destroyed = [all_c[i] for i in idx]
        elif self.action_set == 'merged3':
            # 3 actions : spatio-temporel fusionné, coût élevé, aléatoire
            if action == 0:
                destroyed = self._spatiotemporal_destroy(rng, all_c, K)
            elif action == 1:
                savings   = self._cost_scores()
                destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
            else:
                idx       = rng.choice(len(all_c), K, replace=False)
                destroyed = [all_c[i] for i in idx]
        elif self.action_set == 'merged2':
            # 2 actions : spatio-temporel fusionné, aléatoire
            if action == 0:
                destroyed = self._spatiotemporal_destroy(rng, all_c, K)
            else:
                idx       = rng.choice(len(all_c), K, replace=False)
                destroyed = [all_c[i] for i in idx]
        else:
            raise ValueError(f"CFG['ACTION_SET'] inconnu : {self.action_set!r} "
                              f"(attendu 'full' / 'merged3' / 'merged2')")
        destroyed_set = set(destroyed)
        partial = [[c for c in r if c not in destroyed_set]
                   for r in self.solution]
        partial = [r for r in partial if r]
        return list(destroyed), partial
    def _repair_gurobi(self, destroyed, partial):
        if not destroyed:
            return copy.deepcopy(self.solution)
        inst    = self.instance
        depot   = inst['depot']
        horizon = inst['horizon']
        cap     = inst['capacity']
        nodes   = [0] + destroyed
        N       = len(nodes)
        V       = min(len(destroyed), inst['n_vehicles'])
        env, m = None, None
        try:
            env = gp.Env(empty=True)
            env.setParam('OutputFlag', 0)
            env.start()
            m = gp.Model(env=env)
            if self.tlimit is not None:
                m.setParam('TimeLimit', self.tlimit)
            m.setParam('MIPGap',     self.gap)
            m.setParam('OutputFlag', 0)
            x = m.addVars(N, N, V, vtype=GRB.BINARY, name='x')
            t = m.addVars(N, V, lb=0, ub=horizon,    name='t')
            m.setObjective(
                gp.quicksum(
                    self._d(nodes[i], nodes[j]) * x[i, j, v]
                    for i in range(N) for j in range(N) for v in range(V)
                    if i != j
                )
                # Pénalité par véhicule ouvert (0.0 = comportement d'origine,
                # inchangé). Un coefficient > 0 décourage Gurobi d'ouvrir un
                # véhicule de plus que nécessaire lors du repair, ce qui
                # limite l'inflation du nombre de véhicules par rapport au
                # BKS observée dans la validation.
                + self.vehicle_penalty * gp.quicksum(
                    x[0, j, v] for j in range(1, N) for v in range(V)
                ),
                GRB.MINIMIZE
            )
            for i in range(1, N):
                m.addConstr(
                    gp.quicksum(x[i, j, v]
                                for j in range(N) for v in range(V)
                                if i != j) == 1
                )
            for v in range(V):
                for i in range(1, N):
                    m.addConstr(
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j) ==
                        gp.quicksum(x[j, i, v] for j in range(N) if i != j)
                    )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(x[0, j, v] for j in range(1, N)) <= 1
                )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(
                        inst['customers'][nodes[i] - 1]['demand'] *
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j)
                        for i in range(1, N)
                    ) <= cap
                )
            M = horizon + 1000
            for v in range(V):
                m.addConstr(t[0, v] == 0)
                for i in range(N):
                    ni  = nodes[i]
                    ei  = depot['earliest'] if ni == 0 else inst['customers'][ni-1]['earliest']
                    li  = depot['latest']   if ni == 0 else inst['customers'][ni-1]['latest']
                    sti = depot['service_time'] if ni == 0 else inst['customers'][ni-1]['service_time']
                    m.addConstr(t[i, v] >= ei)
                    m.addConstr(t[i, v] <= li)
                    for j in range(N):
                        if i == j:
                            continue
                        m.addConstr(
                            t[j, v] >= t[i, v] + sti + self._d(ni, nodes[j])
                            - M * (1 - x[i, j, v])
                        )
            m.optimize()
            if m.SolCount > 0:
                eps        = 0.5
                new_routes = []
                for v in range(V):
                    if sum(x[0, j, v].X for j in range(1, N)) < eps:
                        continue
                    route_v, cur = [], 0
                    for _ in range(N):
                        nxt = next((j for j in range(N)
                                    if j != cur and x[cur, j, v].X > eps), None)
                        if nxt is None or nxt == 0:
                            break
                        route_v.append(nodes[nxt])
                        cur = nxt
                    if route_v:
                        new_routes.append(route_v)
                full = [r for r in list(partial) + new_routes if r]
                return full
            else:
                raise ValueError("Gurobi : aucune solution trouvée")
        except Exception as e:
            if self.verbose:
                print(f"    [Gurobi→Greedy fallback] {e}")
            return _repair_greedy(destroyed, partial, self.instance)
        finally:
            if m is not None:
                m.dispose()
            if env is not None:
                env.dispose()
    def step(self, action):
        old_cost            = self.current_cost
        destroyed, partial  = self._destroy(action)
        new_sol             = self._repair_gurobi(destroyed, partial)
        new_cost            = compute_cost(new_sol, self.instance)
        feasible            = is_feasible(new_sol, self.instance)
        if feasible:
            self.solution     = new_sol
            self.current_cost = new_cost
        else:
            new_cost = old_cost * 1.001
        improvement = old_cost - self.current_cost
        if improvement > 0:
            self.best_cost        = min(self.best_cost, self.current_cost)
            self.steps_no_improve = 0
        else:
            self.steps_no_improve += 1
        reward = improvement / max(self.initial_cost, 1.0)
        self.step_count += 1
        self._action_history.append(action)
        done = (self.step_count >= self.max_steps)
        info = {
            'cost':          self.current_cost,
            'best_cost':     self.best_cost,
            'action_taken':  action,
            'action_name':   ACTION_NAMES_BY_SET.get(self.action_set, ACTION_NAMES).get(action, str(action)),
            'improvement':   improvement,
            'destroyed_k':   len(destroyed),
            'destroyed_ids': [int(d) for d in destroyed[:5]],
            'feasible':      feasible,
        }
        return self._obs(), float(reward), done, info
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 4  —  AGENT DQN  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((
            np.array(s,  dtype=np.float32), int(a), float(r),
            np.array(s2, dtype=np.float32), float(done),
        ))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.FloatTensor(np.stack(s)), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(np.stack(s2)),
                torch.FloatTensor(d))
    def __len__(self):
        return len(self.buf)
class DQNNet(nn.Module):
    def __init__(self, obs_dim=7, n_actions=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim,  hidden),  nn.ReLU(),
            nn.Linear(hidden,   hidden),  nn.ReLU(),
            nn.Linear(hidden,   n_actions),
        )
    def forward(self, x):
        return self.net(x)
class DQNAgent:
    def __init__(self, obs_dim=7, n_actions=4, hidden=64,
                 lr=1e-3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_size=2000, batch_size=32,
                 target_update_freq=10, seed=0):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        self.n_actions          = n_actions
        self.gamma              = gamma
        self.epsilon            = epsilon_start
        self.epsilon_end        = epsilon_end
        self.epsilon_decay      = epsilon_decay
        self.batch_size         = batch_size
        self.target_update_freq = target_update_freq
        self.step_count         = 0
        self.q_net      = DQNNet(obs_dim, n_actions, hidden)
        self.target_net = DQNNet(obs_dim, n_actions, hidden)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)
        self.loss_fn   = nn.MSELoss()
    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state).unsqueeze(0))
            return int(q.argmax().item())
    def store(self, s, a, r, s2, done):
        self.buffer.push(s, a, r, s2, done)
    def learn(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        s, a, r, s2, done = self.buffer.sample(self.batch_size)
        q_vals = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q  = self.target_net(s2).max(1)[0]
            targets = r + self.gamma * next_q * (1 - done)
        loss = self.loss_fn(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.optimizer.step()
        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.update_target()
        return float(loss.item())
    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    def save(self, path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        torch.save({'q_net': self.q_net.state_dict(),
                    'target_net': self.target_net.state_dict(),
                    'epsilon': self.epsilon, 'step_count': self.step_count}, path)
        print(f"    [DQN] Modèle sauvegardé → {path}")
    def load(self, path):
        """
        Recharge un checkpoint sauvegardé par save() (mêmes clés). Utilisé
        uniquement si CFG['REUSE_EXISTING_SYNTH_MODEL']=True, pour réutiliser
        un modèle déjà entraîné sur du synthétique au lieu de ré-entraîner.
        """
        ckpt = torch.load(path, map_location='cpu')
        self.q_net.load_state_dict(ckpt['q_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.epsilon    = ckpt.get('epsilon', self.epsilon)
        self.step_count = ckpt.get('step_count', self.step_count)
        print(f"    [DQN] Modèle rechargé ← {path}")
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5  —  ENTRAÎNEMENT DU DQN  (ÉTAPE 1, sur les instances SYNTHÉTIQUES)
#  Stocke cost_trace (trajectoire de coût step par step) dans le log, utilisé
#  ensuite pour les courbes de convergence détaillées. Retourne aussi
#  initial_sols pour les tracés de routes avant/après.
# ══════════════════════════════════════════════════════════════════════════════
BANNER = "═" * 72
def _sec(title):
    print(f"\n{BANNER}\n  {title}\n{BANNER}")
def train_on_instances(instances, episodes_per_instance=10, save_model=True, verbose=True):
    """
    ENTRAÎNEMENT (les poids du DQN sont mis à jour ici via store()/learn()/
    decay_epsilon()). Générique : fonctionne sur n'importe quelle liste
    d'instances au format standard — dans ce script, appelé avec les
    instances SYNTHÉTIQUES produites par generate_synthetic_training_set(),
    JAMAIS avec les instances Solomon (réservées à evaluate_frozen_
    on_instances(), plus bas, pour le TEST).
    save_model=False et verbose=False sont utilisés par le plan Taguchi
    (MODULE 10bis, run interne allégé : pas besoin d'écraser le modèle
    final ni d'un log console détaillé à chaque combinaison testée).
    Retourne : agent, training_log, initial_costs, best_costs, best_solutions,
               instance_times, initial_sols
    """
    n = len(instances)
    if verbose:
        _sec(f"ÉTAPE 1 — LNS initial pour les {n} instances d'entraînement (synthétiques)")
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        if verbose:
            print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
                  f"| véhicules={stats['n_vehicles']}")
    if verbose:
        _sec("ÉTAPE 1 — Entraînement DQN sur le synthétique")
    # Jeu d'actions actif (ablation, voir ACTION_SETS_N / CFG['ACTION_SET']) :
    # 'full' (4, défaut) / 'merged3' (3) / 'merged2' (2). n_actions du DQN et
    # de l'environnement DOIVENT correspondre exactement.
    action_set = CFG.get('ACTION_SET', 'full')
    n_actions_active = ACTION_SETS_N.get(action_set, 4)
    if verbose:
        print(f"  Jeu d'actions actif : '{action_set}' ({n_actions_active} actions)")
    agent = DQNAgent(
        obs_dim=7, n_actions=n_actions_active, hidden=CFG['HIDDEN'], lr=CFG['LR'],
        gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
        epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
        buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
        target_update_freq=CFG['TARGET_UPDATE'],
    )
    training_log   = []
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       action_set=action_set,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        if verbose:
            print(f"\n  {'═'*50}\n  [TRAIN] INSTANCE {inst['name']}  type={inst['type']}"
                  f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)
                next_obs, reward, done, info = env.step(action)
                agent.store(obs, action, reward, next_obs, float(done))
                agent.learn()
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            agent.decay_epsilon()
            training_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            if verbose:
                imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
                print(f"  [TRAIN] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                      f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        if verbose:
            print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    if save_model:
        model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
        agent.save(model_path)
    return agent, training_log, initial_costs, best_costs, best_solutions, instance_times, initial_sols
def evaluate_frozen_on_instances(instances, agent, episodes_per_instance=3):
    """
    TEST — ÉVALUATION GELÉE : l'agent DQN déjà entraîné (ÉTAPE 1, sur le
    synthétique) est ici appliqué en pur mode INFÉRENCE sur les instances
    passées en argument (les 56 Solomon officielles). Différences avec
    train_on_instances() :
      - AUCUN appel à agent.store() / agent.learn() / agent.decay_epsilon() :
        les poids du réseau ne sont JAMAIS mis à jour ici.
      - epsilon est temporairement forcé à 0.0 (politique gloutonne, pure
        exploitation du Q-réseau appris), puis restauré à sa valeur
        d'origine à la fin.
      - Le modèle n'est PAS sauvegardé ici (rien n'a changé dans ses poids).
    Le log retourné (test_log) a exactement la même forme que training_log
    (y compris 'cost_trace'), pour pouvoir réutiliser telles quelles les
    fonctions de tracé (plot_instance_convergence_examples, etc.) sur le
    TEST plutôt que sur l'entraînement.
    Retourne : initial_costs, best_costs, best_solutions, instance_times,
               initial_sols, test_log
    """
    n = len(instances)
    _sec(f"ÉTAPE 2 — ÉVALUATION GELÉE (TEST) sur {n} instances Solomon "
         f"— aucun apprentissage, epsilon=0")
    # Jeu d'actions actif (ablation, voir ACTION_SETS_N / CFG['ACTION_SET']) —
    # DOIT être le même que celui utilisé pour entraîner `agent` (train_
    # on_instances), sinon les indices d'action ne correspondent plus aux
    # mêmes opérateurs de destruction dans l'environnement de test.
    action_set = CFG.get('ACTION_SET', 'full')
    print(f"  Jeu d'actions actif : '{action_set}' ({ACTION_SETS_N.get(action_set, 4)} actions)")
    saved_epsilon = agent.epsilon
    agent.epsilon = 0.0
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    test_log       = []
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       action_set=action_set,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TEST] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)          # inférence pure (epsilon=0)
                next_obs, reward, done, info = env.step(action)
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            test_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TEST] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    agent.epsilon = saved_epsilon
    return initial_costs, best_costs, best_solutions, instance_times, initial_sols, test_log
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6  —  TABLEAU FINAL : SITUATION 1 vs BKS
#  (la Situation 2 "Gurobi seul sur l'instance complète" a été retirée)
# ══════════════════════════════════════════════════════════════════════════════
def build_bks_comparison_table(instances, initial_costs, best_costs, best_solutions,
                                instance_times, csv_path=None, latex_path=None):
    """
    Tableau final : Situation 1 (LNS+DQN+Gurobi) vs BKS, avec gap % et le
    nombre de véhicules utilisés pour chaque instance.
    """
    _sec("TABLEAU FINAL — SITUATION 1 vs BKS")
    rows = []
    for i, inst in enumerate(instances):
        name     = inst['name']
        bks      = BKS_REFERENCE.get(name)
        bks_dist = bks['distance'] if bks else None
        n_veh_s1 = sum(1 for r in best_solutions[i] if r)
        row = {
            'instance':             name,
            'type':                 inst['type'],
            'bks_vehicules':        bks['vehicles'] if bks else None,
            'bks_distance':         bks_dist,
            'n_vehicules_situation1': n_veh_s1,
            'distance_situation1':  round(best_costs[i], 2),
            'temps_s_situation1':   round(instance_times[i], 2),
            'gap_%_situation1_bks': round((best_costs[i] - bks_dist) / bks_dist * 100, 2) if bks_dist else None,
        }
        rows.append(row)
    print(f"\n  {'Instance':<9} {'BKS Vhc':>8} {'BKS Dist':>10} {'Vhc S1':>7} "
          f"{'Dist. S1':>10} {'Temps S1':>10} {'Gap %':>8}")
    print("  " + "─" * 70)
    for r in rows:
        bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "—"
        bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "—"
        g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {bks_v:>8} {bks_d:>10} "
              f"{r['n_vehicules_situation1']:>7} {r['distance_situation1']:>10.2f} "
              f"{r['temps_s_situation1']:>10.1f} {g1:>8}")
    # ── Gap moyen vs BKS (Situation 1) ──────────────────────────────────
    valid_gaps = [r['gap_%_situation1_bks'] for r in rows if r['gap_%_situation1_bks'] is not None]
    gap_moyen  = float(np.mean(valid_gaps)) if valid_gaps else None
    if gap_moyen is not None:
        print(f"\n  Gap moyen vs BKS (Situation 1) : {gap_moyen:+.2f}%  "
              f"({len(valid_gaps)}/{len(rows)} instances avec BKS connu)")
        print(f"  Gap min : {min(valid_gaps):+.2f}%  |  Gap max : {max(valid_gaps):+.2f}%")
    else:
        print("\n  Gap moyen vs BKS : indisponible (aucun BKS connu pour ces instances)")
    # ── Tableau récapitulatif simple : Instance / Nb véhicules / Distance / Gap % ──
    # (une ligne "Situation 1" + une ligne "BKS" par instance, format demandé)
    print(f"\n  Récapitulatif Instance / Nb véhicules / Distance / Gap % vs BKS")
    print(f"  {'Instance':<9} {'Situation':<26} {'Nb véhicules':>13} {'Distance':>10} {'Gap %':>8}")
    print("  " + "─" * 71)
    recap_rows = []
    for r in rows:
        g1 = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {'Situation 1 (DQN+LNS+Gurobi)':<26} "
              f"{r['n_vehicules_situation1']:>13} {r['distance_situation1']:>10.2f} {g1:>8}")
        recap_rows.append({
            'instance':    r['instance'],
            'situation':   'Situation 1 (DQN+LNS+Gurobi)',
            'n_vehicules': r['n_vehicules_situation1'],
            'distance':    r['distance_situation1'],
            'gap_%_vs_bks': r['gap_%_situation1_bks'],
        })
        if r['bks_distance'] is not None:
            print(f"  {r['instance']:<9} {'BKS (référence)':<26} "
                  f"{(r['bks_vehicules'] if r['bks_vehicules'] is not None else '—'):>13} "
                  f"{r['bks_distance']:>10.2f} {'0.00':>8}")
            recap_rows.append({
                'instance':    r['instance'],
                'situation':   'BKS (référence)',
                'n_vehicules': r['bks_vehicules'],
                'distance':    r['bks_distance'],
                'gap_%_vs_bks': 0.0,
            })
    if gap_moyen is not None:
        print("  " + "─" * 71)
        print(f"  {'':<9} {'MOYENNE (Situation 1)':<26} {'':>13} {'':>10} {gap_moyen:>+8.2f}")
    if csv_path:
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
        print(f"\n  Tableau final exporté (CSV) → {csv_path}")
        recap_csv_path = csv_path.replace('.csv', '_recap_vehicules_distance.csv')
        if recap_csv_path == csv_path:
            recap_csv_path = csv_path + '.recap_vehicules_distance.csv'
        with open(recap_csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=['instance', 'situation', 'n_vehicules', 'distance', 'gap_%_vs_bks'])
            w.writeheader(); w.writerows(recap_rows)
            if gap_moyen is not None:
                w.writerow({'instance': '', 'situation': 'MOYENNE (Situation 1)',
                            'n_vehicules': '', 'distance': '', 'gap_%_vs_bks': round(gap_moyen, 2)})
        print(f"  Récapitulatif Instance/Véhicules/Distance/Gap exporté (CSV) → {recap_csv_path}")
    if latex_path:
        lines = [
            "% Tableau généré automatiquement — Situation 1 vs BKS",
            "\\begin{table}[htbp]", "  \\centering",
            "  \\caption{Comparaison par instance Solomon officielle entre la méthode "
            "hybride (LNS + DQN + Gurobi repair) et le Best Known Solution (BKS) : "
            "nombre de véhicules, distance totale et écart au BKS.}",
            "  \\label{tab:comparaison-bks}",
            "  \\begin{tabular}{lrrrrrr}", "    \\toprule",
            "    Instance & BKS Vhc & BKS Dist. & Vhc. S1 & Dist. S1 & "
            "Temps S1 (s) & Gap S1 (\\%) \\\\",
            "    \\midrule",
        ]
        for r in rows:
            bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "--"
            bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "--"
            g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "--"
            lines.append(f"    {r['instance']} & {bks_v} & {bks_d} & "
                        f"{r['n_vehicules_situation1']} & {r['distance_situation1']:.2f} & "
                        f"{r['temps_s_situation1']:.1f} & {g1} \\\\")
        lines += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
        with open(latex_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        print(f"  Tableau final exporté (LaTeX) → {latex_path}")
    return rows
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6bis  —  DIAGNOSTIC : LICENCE GUROBI À TAILLE LIMITÉE ?
# ══════════════════════════════════════════════════════════════════════════════
def check_gurobi_license(min_vars_needed=2500):
    """
    Test rapide, effectué AVANT tout calcul, pour savoir si la licence
    Gurobi active est une licence "size-limited" (Community/gratuite,
    plafonnée à ~2000 variables ET ~2000 contraintes) plutôt qu'une licence
    complète (académique ou commerciale, sans limite de taille).

    Pourquoi c'est important ici : le repair Gurobi de la Situation 1
    (_repair_gurobi, appelé à CHAQUE step du LNS) crée jusqu'à
    (K_DESTROY+1)² * V variables binaires — selon K_DESTROY (cf. CFG) et le
    nombre de véhicules de l'instance, cela peut dépasser 2000. Un fallback
    vers le repair glouton est déjà en place dans _repair_gurobi (try/
    except), donc l'exécution ne plantera pas, mais peut silencieusement
    retomber sur le glouton plus souvent que prévu si la licence est limitée.

    Ce test ne bloque JAMAIS l'exécution : il informe seulement, une seule
    fois, en tête de run.
    """
    env, m = None, None
    try:
        env = gp.Env(empty=True)
        env.setParam('OutputFlag', 0)
        env.start()
        m = gp.Model(env=env)
        m.setParam('OutputFlag', 0)
        x = m.addVars(min_vars_needed, vtype=GRB.BINARY)
        m.setObjective(gp.quicksum(x[i] for i in range(min_vars_needed)), GRB.MAXIMIZE)
        m.optimize()
        print(f"  ✓ Licence Gurobi : aucune restriction de taille détectée "
              f"(modèle-test à {min_vars_needed} variables résolu sans erreur).")
        return True
    except gp.GurobiError as e:
        errno = getattr(e, 'errno', None)
        is_size_limit = (errno == getattr(GRB.Error, 'SIZE_LIMIT_EXCEEDED', object()))
        if is_size_limit:
            k_repair_vars = (CFG['K_DESTROY'] + 1) ** 2 * CFG['K_DESTROY']
            print(f"\n  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée "
                  f"(erreur {errno} : {e})\n"
                  f"    → Licence Community/gratuite, plafonnée à ~2000 variables "
                  f"et ~2000 contraintes.\n"
                  f"    → Le repair Gurobi de la Situation 1 (jusqu'à "
                  f"~{k_repair_vars} variables selon K_DESTROY={CFG['K_DESTROY']} et "
                  f"le nombre de véhicules) peut dépasser cette limite sur certaines "
                  f"instances — un repli automatique vers le repair glouton est déjà "
                  f"en place, donc pas de plantage, mais un usage de Gurobi moins "
                  f"fréquent que prévu.\n"
                  f"    → Pour lever cette limite : une licence académique Gurobi "
                  f"est gratuite et sans limite de taille pour les universités "
                  f"(https://www.gurobi.com/academia/) — utile si vous êtes "
                  f"affiliée à ENSI-UMA.\n")
        else:
            print(f"  ⚠ Erreur Gurobi lors du test de licence (non liée à la taille) : {e}")
        return False
    except Exception as e:
        print(f"  ⚠ Impossible de tester la licence Gurobi ({type(e).__name__}: {e}) "
              f"— test ignoré, l'exécution continue normalement.")
        return None
    finally:
        if m is not None:
            m.dispose()
        if env is not None:
            env.dispose()
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 7  —  COURBES DE CONVERGENCE
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_mean(vals, window=20):
    out = []
    for i in range(len(vals)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(vals[lo:i + 1])))
    return out
def plot_global_convergence(training_log, instances, initial_costs, best_costs, outdir):
    """
    Figure globale (2 panneaux) :
      (a) reward par épisode global + moyenne mobile → convergence de
          l'apprentissage DQN
      (b) amélioration % par instance (une barre par instance Solomon) vs
          coût LNS initial
    """
    episodes = [e['episode_global'] for e in training_log]
    rewards  = [e['total_reward']   for e in training_log]
    reward_smooth = _rolling_mean(rewards, window=20)
    improvements  = [(initial_costs[i] - best_costs[i]) / initial_costs[i] * 100
                      for i in range(len(initial_costs))]
    names = [inst['name'] for inst in instances]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    axes[0].plot(episodes, rewards, color='lightgray', linewidth=0.6, label='reward brut / épisode')
    axes[0].plot(episodes, reward_smooth, color='crimson', linewidth=2,
                 label='moyenne mobile (20 épisodes)')
    axes[0].set_xlabel('Épisode global')
    axes[0].set_ylabel('Reward de l\'épisode')
    axes[0].set_title("Convergence de l'apprentissage DQN")
    axes[0].legend(loc='best', fontsize=9)
    axes[0].grid(alpha=0.3)
    colors = ['#2ca02c' if v >= 0 else '#d62728' for v in improvements]
    axes[1].bar(range(len(improvements)), improvements, color=colors)
    axes[1].axhline(np.mean(improvements), color='navy', linestyle='--', linewidth=1.5,
                     label=f"Moyenne = {np.mean(improvements):.2f}%")
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names, rotation=90, fontsize=7)
    axes[1].set_xlabel('Instance')
    axes[1].set_ylabel('Amélioration vs solution LNS initiale (%)')
    axes[1].set_title("Amélioration du coût par instance après DQN+Gurobi")
    axes[1].legend(loc='best', fontsize=9)
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_globale.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence globale → {path}")
    return path
def plot_gap_vs_bks(bks_rows, outdir):
    """
    Barres du gap % vs BKS (Situation 1) pour chaque instance, avec la
    moyenne indiquée par une ligne horizontale. N'utilise que les instances
    pour lesquelles un BKS est connu (gap_%_situation1_bks non None).
    `bks_rows` = la liste `rows` retournée par build_bks_comparison_table().
    """
    data = [(r['instance'], r['gap_%_situation1_bks']) for r in bks_rows
            if r['gap_%_situation1_bks'] is not None]
    if not data:
        print("  [plot_gap_vs_bks] Aucune instance avec BKS connu — graphique ignoré.")
        return None
    names = [d[0] for d in data]
    gaps  = [d[1] for d in data]
    gap_moyen = float(np.mean(gaps))
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(names)), 5.5))
    colors = ['#2ca02c' if g <= 0 else '#d62728' for g in gaps]
    ax.bar(range(len(names)), gaps, color=colors)
    ax.axhline(gap_moyen, color='navy', linestyle='--', linewidth=1.5,
               label=f"Gap moyen = {gap_moyen:+.2f}%")
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_xlabel('Instance')
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_title(f"Écart au Best Known Solution par instance "
                 f"({len(names)} instances avec BKS connu)")
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique gap vs BKS → {path}")
    return path
def plot_instance_convergence_examples(training_log, instances, example_idxs, outdir):
    """
    Pour chaque instance exemple :
      - gauche : coût final / meilleur coût cumulé au fil des épisodes locaux
      - droite : trajectoire de coût step par step du dernier épisode local
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(13, 4.2 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        entries = [e for e in training_log if e['instance_id'] == idx]
        entries.sort(key=lambda e: e['ep_local'])
        ep_locals      = [e['ep_local']   for e in entries]
        best_costs_ep  = [e['best_cost']  for e in entries]
        final_costs_ep = [e['final_cost'] for e in entries]
        name = instances[idx]['name']
        axes[row, 0].plot(ep_locals, final_costs_ep, '-o', color='steelblue',
                           label='coût final de l\'épisode')
        axes[row, 0].plot(ep_locals, best_costs_ep, '-s', color='crimson',
                           label='meilleur coût cumulé')
        axes[row, 0].set_title(f"Instance {name} — convergence par épisode local")
        axes[row, 0].set_xlabel('Épisode local')
        axes[row, 0].set_ylabel('Coût de la tournée')
        axes[row, 0].legend(loc='best', fontsize=9)
        axes[row, 0].grid(alpha=0.3)
        last = entries[-1] if entries else None
        trace = last.get('cost_trace', []) if last else []
        if trace:
            axes[row, 1].plot(range(len(trace)), trace, '-o', color='darkorange')
            axes[row, 1].set_title(f"Instance {name} — trajectoire de coût "
                                    f"(dernier épisode local, {len(trace)-1} steps)")
            axes[row, 1].set_xlabel('Step')
            axes[row, 1].set_ylabel('Coût courant')
            axes[row, 1].grid(alpha=0.3)
        else:
            axes[row, 1].axis('off')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_instances_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence instances exemples → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 8  —  TRACÉS DES ROUTES (AVANT LNS / APRÈS DQN)
# ══════════════════════════════════════════════════════════════════════════════
def _plot_routes_on_ax(ax, instance, solution, title):
    depot = instance['depot']
    ax.scatter([depot['x']], [depot['y']], c='black', marker='s', s=110,
               zorder=5, label='Dépôt')
    n_routes = max(len([r for r in solution if r]), 1)
    try:
        cmap = matplotlib.colormaps.get_cmap('tab20').resampled(max(n_routes, 1))
    except AttributeError:
        cmap = plt.cm.get_cmap('tab20', max(n_routes, 1))
    ridx = 0
    for route in solution:
        if not route:
            continue
        xs = [depot['x']] + [instance['customers'][cid - 1]['x'] for cid in route] + [depot['x']]
        ys = [depot['y']] + [instance['customers'][cid - 1]['y'] for cid in route] + [depot['y']]
        ax.plot(xs, ys, '-o', color=cmap(ridx), markersize=4, linewidth=1.3)
        ridx += 1
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.3)
def plot_routes_examples(instances, initial_sols, best_solutions, example_idxs, outdir):
    """
    Pour chaque instance exemple : routes AVANT (solution LNS initiale, avant
    DQN) à gauche, routes APRÈS (meilleure solution trouvée par DQN+Gurobi)
    à droite. Le BKS (véhicules/distance) est indiqué en sous-titre s'il
    est connu pour l'instance.
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.8 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        inst = instances[idx]
        name = inst['name']
        bks  = BKS_REFERENCE.get(name)
        bks_str = (f" | BKS: dist={bks['distance']:.1f}, vhc={bks['vehicles']}"
                   if bks else "")
        cost_before = compute_cost(initial_sols[idx], inst)
        cost_after  = compute_cost(best_solutions[idx], inst)
        nveh_before = sum(1 for r in initial_sols[idx] if r)
        nveh_after  = sum(1 for r in best_solutions[idx] if r)
        _plot_routes_on_ax(
            axes[row, 0], inst, initial_sols[idx],
            f"Instance {name} — AVANT (LNS)\n"
            f"dist={cost_before:.1f}  véhicules={nveh_before}{bks_str}"
        )
        _plot_routes_on_ax(
            axes[row, 1], inst, best_solutions[idx],
            f"Instance {name} — APRÈS (DQN+Gurobi)\n"
            f"dist={cost_after:.1f}  véhicules={nveh_after}{bks_str}"
        )
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_routes_avant_apres_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique routes avant/après → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 9  —  ANALYSES PAR GROUPE D'INSTANCES (C1/C2/R1/R2/RC1/RC2)
#  Appliqué uniquement aux résultats du TEST (gap vs BKS) — ne joue aucun
#  rôle dans le choix des hyperparamètres, purement diagnostique a posteriori.
# ══════════════════════════════════════════════════════════════════════════════
def get_instance_group(inst):
    """
    Déduit le groupe Solomon (C1/C2/R1/R2/RC1/RC2) à partir du type et du nom
    de l'instance. Ex. : type='C', name='C101' -> 'C1' ; type='RC', name=
    'RC201' -> 'RC2'. Le chiffre juste après le préfixe alphabétique indique
    le sous-groupe (1 = fenêtres serrées / faible capacité, 2 = fenêtres
    larges / grande capacité).
    """
    t = inst['type']
    name = inst['name']
    digit = name[len(t)] if len(name) > len(t) else ''
    return f"{t}{digit}"
def plot_gap_boxplot_by_group(bks_rows, instances, outdir):
    """
    Boxplot du gap % vs BKS regroupé par C1/C2/R1/R2/RC1/RC2, pour visualiser
    quels groupes d'instances posent le plus de difficulté à la méthode.
    """
    by_name = {inst['name']: inst for inst in instances}
    groups = defaultdict(list)
    for r in bks_rows:
        if r['gap_%_situation1_bks'] is None:
            continue
        inst = by_name.get(r['instance'])
        if inst is None:
            continue
        groups[get_instance_group(inst)].append(r['gap_%_situation1_bks'])
    labels = [g for g in GROUP_ORDER if g in groups]
    data   = [groups[g] for g in labels]
    if not data:
        print("  [plot_gap_boxplot_by_group] Aucune donnée — graphique ignoré.")
        return None
    fig, ax = plt.subplots(figsize=(8, 5.5))
    try:
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, showmeans=True)
    except TypeError:
        # matplotlib < 3.9 : le paramètre s'appelait encore 'labels'.
        bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#a6c8ff')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_xlabel('Groupe d\'instances Solomon')
    ax.set_title("Distribution du gap par groupe d'instances")
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_boxplot_par_groupe.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique boxplot gap par groupe → {path}")
    return path
def plot_vehicles_vs_bks(bks_rows, instances, outdir):
    """
    Scatter plot : nombre de véhicules utilisés (Situation 1) vs nombre de
    véhicules du BKS, par instance, coloré par groupe. La diagonale y=x
    marque l'égalité ; les points au-dessus utilisent plus de véhicules que
    le BKS — c'est le phénomène d'inflation identifié dans la validation.
    """
    by_name = {inst['name']: inst for inst in instances}
    cmap_colors = plt.cm.tab10.colors
    color_map = {g: cmap_colors[i % len(cmap_colors)] for i, g in enumerate(GROUP_ORDER)}
    fig, ax = plt.subplots(figsize=(7, 7))
    seen_groups = set()
    max_v = 1
    for r in bks_rows:
        if r['bks_vehicules'] is None or r['n_vehicules_situation1'] is None:
            continue
        inst = by_name.get(r['instance'])
        grp  = get_instance_group(inst) if inst else '?'
        c    = color_map.get(grp, 'gray')
        label = grp if grp not in seen_groups else None
        seen_groups.add(grp)
        ax.scatter(r['bks_vehicules'], r['n_vehicules_situation1'], color=c,
                   label=label, s=45, alpha=0.85, edgecolor='black', linewidth=0.4)
        max_v = max(max_v, r['bks_vehicules'], r['n_vehicules_situation1'])
    ax.plot([0, max_v + 1], [0, max_v + 1], '--', color='gray', linewidth=1, label='y = x (égalité)')
    ax.set_xlabel('Véhicules BKS')
    ax.set_ylabel('Véhicules Situation 1 (DQN+LNS+Gurobi)')
    ax.set_title('Nombre de véhicules utilisés vs BKS, par instance')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_vehicules_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique véhicules vs BKS → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 10  —  PLAN D'EXPÉRIENCE TAGUCHI (calibration des hyperparamètres)
#
#  Objectif : identifier quels hyperparamètres (LNS_ITER, K_DESTROY_RATIO,
#  GUROBI_TLIMIT, GUROBI_GAP, MAX_STEPS, EPS_DECAY, VEHICLE_PENALTY) ont le
#  plus d'effet sur la performance, et quelle combinaison la maximise, sans
#  tester les 3^7 = 2187 combinaisons possibles.
#
#  ⚠ POINT CRITIQUE (train/test disjoint) : ce module supporte DEUX modes,
#  sélectionnés par le paramètre `instances_override`/`metric` de
#  run_taguchi_doe() :
#    - mode SYNTHÉTIQUE (recommandé, piloté par CFG['TAGUCHI_SYNTH']) :
#      `instances_override` = des instances synthétiques générées à la
#      volée ; `metric='lns_improvement'` = % d'amélioration vs le coût LNS
#      initial (aucun BKS requis). Les 56 Solomon ne sont JAMAIS chargées
#      dans ce mode → zéro fuite d'hyperparamètres vers le jeu de test.
#    - mode SOLOMON historique (⚠ non recommandé, piloté par CFG['TAGUCHI']) :
#      `instances_override=None` → charge INSTANCE_FILES (les 56 Solomon)
#      comme matrice externe ; `metric='bks_gap'` = gap % vs BKS_REFERENCE.
#      Ce mode entraîne et évalue directement sur le jeu de test : la
#      combinaison qu'il retourne a "vu" Solomon et ne doit pas être
#      utilisée comme hyperparamètres d'une validation train/test disjointe
#      (elle resterait valable seulement comme référence de comparaison).
#
#  Principe commun (matrice orthogonale L9/L18) :
#    - Matrice interne  : la matrice orthogonale L18 ci-dessous (18 lignes,
#      7 facteurs, 3 niveaux) — les combinaisons de facteurs à tester.
#    - Matrice externe (bruit) : les instances sur lesquelles chaque
#      combinaison de la matrice interne est évaluée ; le ratio signal/bruit
#      (S/N) résume à la fois la performance moyenne ET la stabilité face à
#      la variabilité entre instances/groupes.
#    - Après le plan, reportez la meilleure combinaison dans CFG, repassez
#      ENABLE=False, et relancez le script pour la validation ENTRAÎNEMENT/
#      TEST DISJOINT normale (avec ces hyperparamètres).
# ══════════════════════════════════════════════════════════════════════════════
# Matrice orthogonale L9 (3^4) — 9 runs, jusqu'à 4 facteurs à 3 niveaux.
L9_ARRAY = [
    [1, 1, 1, 1], [1, 2, 2, 2], [1, 3, 3, 3],
    [2, 1, 2, 3], [2, 2, 3, 1], [2, 3, 1, 2],
    [3, 1, 3, 2], [3, 2, 1, 3], [3, 3, 2, 1],
]
# Matrice orthogonale L18 (2^1 x 3^7) — 18 runs. La 1ère colonne (2 niveaux)
# n'est pas utilisée ici (tous nos facteurs sont à 3 niveaux) ; seules les
# colonnes 2 à 8 (7 facteurs à 3 niveaux) servent au plan.
L18_ARRAY_FULL = [
    [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 2, 2, 2, 2, 2, 2], [1, 1, 3, 3, 3, 3, 3, 3],
    [1, 2, 1, 1, 2, 2, 3, 3], [1, 2, 2, 2, 3, 3, 1, 1], [1, 2, 3, 3, 1, 1, 2, 2],
    [1, 3, 1, 2, 1, 3, 2, 3], [1, 3, 2, 3, 2, 1, 3, 1], [1, 3, 3, 1, 3, 2, 1, 2],
    [2, 1, 1, 3, 3, 2, 2, 1], [2, 1, 2, 1, 1, 3, 3, 2], [2, 1, 3, 2, 2, 1, 1, 3],
    [2, 2, 1, 2, 3, 1, 3, 2], [2, 2, 2, 3, 1, 2, 1, 3], [2, 2, 3, 1, 2, 3, 2, 1],
    [2, 3, 1, 3, 2, 3, 1, 2], [2, 3, 2, 1, 3, 1, 2, 3], [2, 3, 3, 2, 1, 2, 3, 1],
]
L18_ARRAY = [row[1:] for row in L18_ARRAY_FULL]   # ne garder que les 7 colonnes à 3 niveaux
def check_orthogonal_array(array):
    """
    Vérifie la propriété d'orthogonalité : pour chaque paire de colonnes,
    chaque combinaison de niveaux doit apparaître le même nombre de fois.
    Retourne True/False — sert de garde-fou avant de lancer un plan
    potentiellement long sur une matrice mal formée.
    """
    arr = np.array(array)
    n_runs, n_cols = arr.shape
    ok = True
    for c1, c2 in itertools.combinations(range(n_cols), 2):
        counts = defaultdict(int)
        for row in arr:
            counts[(row[c1], row[c2])] += 1
        if len(set(counts.values())) > 1:
            ok = False
    return ok
def select_orthogonal_array(n_factors):
    if n_factors <= 4:
        return 'L9', [row[:n_factors] for row in L9_ARRAY]
    elif n_factors <= 7:
        return 'L18', [row[:n_factors] for row in L18_ARRAY]
    else:
        raise ValueError(
            f"{n_factors} facteurs demandés, mais seules les matrices L9 (4 "
            f"facteurs max) et L18 (7 facteurs max) sont disponibles. "
            f"Réduisez TAGUCHI_FACTORS ou étendez ce module avec une L27."
        )
@contextmanager
def _cfg_override(**kwargs):
    """Applique temporairement des valeurs de CFG (pour un run du plan
    Taguchi), puis restaure les valeurs d'origine à la sortie du bloc."""
    old = {k: CFG.get(k) for k in kwargs}
    CFG.update(kwargs)
    try:
        yield
    finally:
        CFG.update(old)
def taguchi_sn_ratio_stb(values):
    """
    Ratio signal/bruit 'plus petit est meilleur' (Smaller-The-Better),
    calculé sur une liste de valeurs (le gap %/l'amélioration % peut être
    négatif — la formule standard SN = -10*log10(mean(y_i^2)) reste valide,
    elle pénalise tout écart à zéro, positif ou négatif, ainsi que la
    variance entre les instances de la matrice externe).
    """
    y = np.array(values, dtype=float)
    msd = max(float(np.mean(y ** 2)), 1e-9)
    return -10.0 * np.log10(msd)
def taguchi_main_effects(results, factor_names, response_key):
    """Moyenne de la réponse (gap/amélioration ou S/N) par niveau, pour
    chaque facteur."""
    effects = {}
    for fn in factor_names:
        by_level = defaultdict(list)
        for r in results:
            if r.get(response_key) is not None:
                by_level[r[fn]].append(r[response_key])
        effects[fn] = {lvl: float(np.mean(vals)) for lvl, vals in sorted(by_level.items(), key=lambda kv: kv[0])}
    return effects
def taguchi_anova_contributions(results, factor_names, response_key):
    """
    ANOVA simplifiée (méthode des sommes de carrés à partir des moyennes par
    niveau) : contribution en % de chaque facteur à la variance totale de la
    réponse. Ne remplace pas une ANOVA complète avec terme d'erreur pur,
    mais donne une hiérarchie fiable de l'importance relative des facteurs.
    """
    valid = [r for r in results if r.get(response_key) is not None]
    if not valid:
        return {fn: 0.0 for fn in factor_names}
    y = np.array([r[response_key] for r in valid])
    grand_mean = y.mean()
    ss_total = float(np.sum((y - grand_mean) ** 2))
    contributions = {}
    for fn in factor_names:
        by_level = defaultdict(list)
        for r in valid:
            by_level[r[fn]].append(r[response_key])
        ss_factor = sum(len(vals) * (np.mean(vals) - grand_mean) ** 2 for vals in by_level.values())
        contributions[fn] = (ss_factor / ss_total * 100.0) if ss_total > 0 else 0.0
    return contributions
def plot_taguchi_main_effects(effects, factor_names, response_label, outdir, filename):
    """Graphique classique des effets principaux Taguchi : un panneau par
    facteur, moyenne de la réponse pour chaque niveau, reliée par une ligne."""
    n = len(factor_names)
    ncols = min(4, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, fn in enumerate(factor_names):
        levels = list(effects[fn].keys())
        means  = list(effects[fn].values())
        axes[i].plot(range(len(levels)), means, '-o', color='steelblue')
        axes[i].set_xticks(range(len(levels)))
        axes[i].set_xticklabels([str(l) for l in levels], fontsize=8)
        axes[i].set_title(fn, fontsize=10)
        axes[i].set_ylabel(response_label, fontsize=8)
        axes[i].grid(alpha=0.3)
    for j in range(len(factor_names), len(axes)):
        axes[j].axis('off')
    fig.suptitle(f"Effets principaux — {response_label}", fontsize=13)
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique effets principaux ({response_label}) → {path}")
    return path
def plot_taguchi_sn_bars(results, outdir, filename='taguchi_sn_ratios.png'):
    """Ratio S/N obtenu pour chaque run du plan, run avec le meilleur S/N
    mis en évidence en vert."""
    valid = [r for r in results if r.get('sn_ratio') is not None]
    if not valid:
        print("  [plot_taguchi_sn_bars] Aucun résultat — graphique ignoré.")
        return None
    runs = [r['run'] for r in valid]
    sns_ = [r['sn_ratio'] for r in valid]
    best_idx = int(np.argmax(sns_))
    colors = ['#2ca02c' if i == best_idx else '#4c72b0' for i in range(len(sns_))]
    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(runs)), 5))
    ax.bar([str(r) for r in runs], sns_, color=colors)
    ax.set_xlabel('Run (combinaison de la matrice L9/L18)')
    ax.set_ylabel('Ratio S/N (plus grand = meilleur)')
    ax.set_title('Ratio signal/bruit par combinaison — plan Taguchi')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique S/N par run → {path}")
    return path
def plot_taguchi_pareto(anova_contributions, outdir, filename='taguchi_pareto_effets.png'):
    """Diagramme en barres horizontales de la contribution (%) de chaque
    facteur à la variance de la réponse — équivalent simplifié d'un Pareto ANOVA."""
    items = sorted(anova_contributions.items(), key=lambda kv: -kv[1])
    names = [k for k, v in items][::-1]
    vals  = [v for k, v in items][::-1]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(range(len(names)), vals, color='#4c72b0')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('Contribution à la variance de la réponse (%)')
    ax.set_title("Importance relative des facteurs (ANOVA simplifiée Taguchi)")
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique Pareto des effets → {path}")
    return path
def run_taguchi_doe(factors, outer_instance_names, episodes_per_instance_doe, outdir,
                     instances_override=None, metric='bks_gap', json_suffix=''):
    """
    Exécute le plan d'expérience Taguchi complet :
      1. Construit la matrice orthogonale adaptée au nombre de facteurs
      2. Pour chaque run, applique les niveaux de facteurs (via _cfg_override)
         et entraîne le pipeline complet (LNS+DQN+Gurobi) sur les instances
         de la matrice externe, avec un nombre d'épisodes réduit
      3. Calcule la métrique de classement par instance (voir `metric`), la
         moyenne, et le ratio S/N
      4. Analyse les effets principaux + ANOVA simplifiée, génère les graphes
      5. Rapporte la meilleure combinaison trouvée
    instances_override : si fourni (liste d'instances déjà construites),
        utilisé DIRECTEMENT comme matrice externe, SANS jamais charger les
        fichiers Solomon — c'est ce qui permet de calibrer sur des instances
        SYNTHÉTIQUES (voir CFG['TAGUCHI_SYNTH']) sans aucune fuite vers le
        jeu de test. Si None (mode historique Solomon), charge les instances
        via INSTANCE_FILES et `outer_instance_names`.
    metric : 'bks_gap' (gap % vs BKS_REFERENCE — ne fonctionne que pour des
        instances Solomon officielles connues dans BKS_REFERENCE) ou
        'lns_improvement' (% d'amélioration vs le coût LNS initial — aucun
        BKS requis, utilisable sur des instances synthétiques). Convention
        "plus petit est meilleur" identique dans les deux cas (négatif =
        amélioration), donc toute la mécanique Taguchi (S/N, effets
        principaux, ANOVA, combinaison prédite) fonctionne sans modification.
    json_suffix : suffixe inséré dans les noms des fichiers de sortie (CSV,
        JSON, PNG) — ex. '_synthetique' — pour ne jamais écraser les
        résultats d'un plan calibré sur d'autres instances/métrique.
    Retourne : results (liste de dicts, un par run), effects_gap, effects_sn,
               anova_contributions, best_run
    """
    factor_names  = [f[0] for f in factors]
    factor_levels = [f[1] for f in factors]
    n_factors     = len(factors)
    array_name, array = select_orthogonal_array(n_factors)
    ortho_ok = check_orthogonal_array(array)
    if instances_override is not None:
        outer_instances = list(instances_override)
    else:
        all_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
        if outer_instance_names == 'ALL':
            # Sentinel spécial : utiliser TOUTES les instances Solomon
            # chargées comme matrice externe. ⚠ Mode historique — voir
            # avertissement CFG['TAGUCHI'] plus haut : fuite d'hyperparamètres
            # vers le jeu de test. Multiplie aussi fortement le temps total
            # du plan (voir avertissement de durée ci-dessous).
            outer_instances = list(all_instances)
        else:
            by_name = {inst['name']: inst for inst in all_instances}
            outer_instances = [by_name[nm] for nm in outer_instance_names if nm in by_name]
    if not outer_instances:
        raise ValueError("Aucune instance de la matrice externe trouvée "
                          "(instances_override vide, ou outer_instance_names "
                          "absent de INSTANCE_FILES).")
    gurobi_tlimit_max = max(TAGUCHI_FACTORS[[f[0] for f in TAGUCHI_FACTORS].index('GUROBI_TLIMIT')][1]) \
        if 'GUROBI_TLIMIT' in factor_names else CFG['GUROBI_TLIMIT']
    max_steps_max = max(TAGUCHI_FACTORS[[f[0] for f in TAGUCHI_FACTORS].index('MAX_STEPS')][1]) \
        if 'MAX_STEPS' in factor_names else CFG['MAX_STEPS']
    worst_case_h = (len(array) * len(outer_instances) * gurobi_tlimit_max
                     * max_steps_max * episodes_per_instance_doe) / 3600.0
    _sec(f"PLAN D'EXPÉRIENCE TAGUCHI — {array_name} ({len(array)} runs, "
         f"{n_factors} facteurs, métrique={metric}, orthogonalité="
         f"{'OK' if ortho_ok else 'A VERIFIER !'})")
    if len(outer_instances) <= 12:
        print(f"  Matrice externe (bruit) : {[i['name'] for i in outer_instances]}")
    else:
        print(f"  Matrice externe (bruit) : {len(outer_instances)} instances "
              f"({outer_instances[0]['name']} ... {outer_instances[-1]['name']})")
    print(f"  Épisodes/instance pendant le plan : {episodes_per_instance_doe}")
    print(f"  ⚠ Temps pire cas estimé (si Gurobi atteint systématiquement sa limite de "
          f"temps) : ~{worst_case_h:.1f}h ({worst_case_h/24:.1f} jours). "
          f"Le temps réel est généralement inférieur, mais reste substantiel à cette échelle.")
    results = []
    for run_idx, level_idx_row in enumerate(array, start=1):
        levels = {factor_names[i]: factor_levels[i][level_idx_row[i] - 1] for i in range(n_factors)}
        print(f"\n  Run {run_idx:02d}/{len(array)} : {levels}")
        overrides = dict(levels)
        if 'K_DESTROY_RATIO' in overrides:
            overrides['K_DESTROY'] = max(3, round(100 * overrides['K_DESTROY_RATIO']))
        t_run = time.time()
        with _cfg_override(**overrides):
            _, _, initial_costs_o, best_costs_o, _, _, _ = train_on_instances(
                outer_instances, episodes_per_instance=episodes_per_instance_doe,
                save_model=False, verbose=False,
            )
        values = []
        for i, inst in enumerate(outer_instances):
            if metric == 'bks_gap':
                bks = BKS_REFERENCE.get(inst['name'])
                if bks:
                    values.append((best_costs_o[i] - bks['distance']) / bks['distance'] * 100.0)
            elif metric == 'lns_improvement':
                ic = initial_costs_o[i]
                if ic:
                    values.append((best_costs_o[i] - ic) / ic * 100.0)
            else:
                raise ValueError(f"metric inconnue : {metric!r} (attendu 'bks_gap' ou 'lns_improvement')")
        mean_val = float(np.mean(values)) if values else None
        sn       = taguchi_sn_ratio_stb(values) if values else None
        row_result = dict(levels)
        row_result.update({
            'run': run_idx,
            'mean_gap_%': round(mean_val, 3) if mean_val is not None else None,
            'sn_ratio':   round(sn, 3) if sn is not None else None,
        })
        results.append(row_result)
        elapsed = time.time() - t_run
        if mean_val is not None:
            print(f"    → {'gap' if metric=='bks_gap' else 'amélioration'} moyen(ne) = "
                  f"{mean_val:+.2f}%  |  S/N = {sn:.2f}  |  {elapsed:.1f}s")
        else:
            print(f"    → indisponible  |  {elapsed:.1f}s")
    csv_path = os.path.join(outdir, f'taguchi_resultats{json_suffix}.csv')
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = factor_names + ['run', 'mean_gap_%', 'sn_ratio']
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        w.writeheader()
        for r in results:
            w.writerow(r)
    print(f"\n  Résultats Taguchi exportés (CSV) → {csv_path}")
    effects_gap = taguchi_main_effects(results, factor_names, 'mean_gap_%')
    effects_sn  = taguchi_main_effects(results, factor_names, 'sn_ratio')
    anova       = taguchi_anova_contributions(results, factor_names, 'mean_gap_%')
    response_label = 'Gap moyen vs BKS (%)' if metric == 'bks_gap' else 'Amélioration moyenne vs LNS (%)'
    _sec("GRAPHIQUES TAGUCHI")
    plot_taguchi_main_effects(effects_gap, factor_names, response_label, outdir,
                              f'taguchi_effets_principaux_gap{json_suffix}.png')
    plot_taguchi_main_effects(effects_sn, factor_names, 'Ratio S/N', outdir,
                              f'taguchi_effets_principaux_sn{json_suffix}.png')
    plot_taguchi_sn_bars(results, outdir, filename=f'taguchi_sn_ratios{json_suffix}.png')
    plot_taguchi_pareto(anova, outdir, filename=f'taguchi_pareto_effets{json_suffix}.png')
    valid_runs = [r for r in results if r['mean_gap_%'] is not None]
    best_run = min(valid_runs, key=lambda r: abs(r['mean_gap_%'])) if valid_runs else None
    print(f"\n{'█'*72}")
    print(f"  █  MEILLEURE COMBINAISON TROUVÉE — PLAN TAGUCHI{' (' + json_suffix + ')' if json_suffix else ''}")
    print(f"{'█'*72}")
    if best_run:
        print(f"  Run {best_run['run']} : {'gap' if metric=='bks_gap' else 'amélioration'} moyen(ne) = "
              f"{best_run['mean_gap_%']:+.2f}%  |  S/N = {best_run['sn_ratio']:.2f}")
        print(f"  ── Hyperparamètres recommandés ──")
        for fn in factor_names:
            print(f"    {fn:<18s} = {best_run[fn]}")
        print(f"\n  ── Contribution des facteurs à la variance de la réponse (ANOVA) ──")
        for fn, pct in sorted(anova.items(), key=lambda kv: -kv[1]):
            print(f"    {fn:<18s} : {pct:5.1f}%")
        print(f"\n  → Reportez ces valeurs dans CFG (et TAGUCHI_FACTORS si besoin), "
              f"repassez ENABLE=False, et relancez le script pour la validation "
              f"ENTRAÎNEMENT/TEST DISJOINT sur les 56 instances Solomon.")
        best_json_path = os.path.join(outdir, f'taguchi_meilleure_combinaison{json_suffix}.json')
        with open(best_json_path, 'w', encoding='utf-8') as f:
            json.dump({
                'run':               best_run['run'],
                'metric':            metric,
                'mean_gap_%':        best_run['mean_gap_%'],
                'sn_ratio':          best_run['sn_ratio'],
                'hyperparametres':   {fn: best_run[fn] for fn in factor_names},
                'contribution_anova_%': anova,
                'array':             array_name,
                'n_runs':            len(array),
                'n_outer_instances': len(outer_instances),
                'episodes_per_instance_doe': episodes_per_instance_doe,
            }, f, indent=2, ensure_ascii=False)
        print(f"\n  Meilleure combinaison sauvegardée (JSON) → {best_json_path}")
    print(f"{'█'*72}\n")
    return results, effects_gap, effects_sn, anova, best_run
def load_and_apply_taguchi_best(outdir, json_suffix=''):
    """
    Étape de confirmation : cherche taguchi_meilleure_combinaison<suffix>.json
    (créé par run_taguchi_doe) et applique ces hyperparamètres à CFG, pour
    que la validation ENTRAÎNEMENT/TEST DISJOINT utilise automatiquement la
    meilleure combinaison trouvée — sans avoir à la reporter à la main.
    Passez json_suffix='_synthetique' pour charger la combinaison calibrée
    UNIQUEMENT sur du synthétique (recommandé). Ne modifie rien (et affiche
    un avertissement) si le fichier n'existe pas encore.
    Retourne True si des hyperparamètres ont été appliqués, False sinon.
    """
    path = os.path.join(outdir, f'taguchi_meilleure_combinaison{json_suffix}.json')
    if not os.path.exists(path):
        cfg_key_hint = "CFG['TAGUCHI_SYNTH']" if json_suffix else "CFG['TAGUCHI']"
        print(f"  ⚠ {path} introuvable — lancez d'abord le plan Taguchi correspondant "
              f"({cfg_key_hint}['ENABLE']=True). Hyperparamètres CFG actuels inchangés.")
        return False
    with open(path, 'r', encoding='utf-8') as f:
        best = json.load(f)
    hp = best.get('hyperparametres', {})
    for fn, val in hp.items():
        CFG[fn] = val
    if 'K_DESTROY_RATIO' in hp:
        CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))
    print(f"  ✓ Hyperparamètres Taguchi appliqués depuis {path} :")
    for fn, val in hp.items():
        print(f"    {fn:<18s} = {val}")
    return True
# ══════════════════════════════════════════════════════════════════════════════
#  EXÉCUTION AUTOMATIQUE
# ══════════════════════════════════════════════════════════════════════════════
def run_final_validation():
    """
    MODE VALIDATION : ENTRAÎNEMENT/TEST DISJOINT.
    Entraîne le DQN sur CFG['N_SYNTH_INSTANCES'] instances synthétiques puis
    l'évalue gelé sur les 56 instances Solomon officielles. Si
    CFG['USE_TAGUCHI_BEST']=True, applique d'abord automatiquement la
    meilleure combinaison d'hyperparamètres trouvée par le plan Taguchi
    synthétique (fichier taguchi_meilleure_combinaison_synthetique.json).

    Cette fonction regroupe exactement ce que faisait auparavant la branche
    `else` du bloc __main__ — extraite dans une fonction pour pouvoir être
    appelée aussi bien seule (validation directe, sans calibration) que
    depuis la branche Taguchi synthétique, en enchaînement automatique.
    """
    t0 = time.time()
    print(f"\n{'═'*72}\n  MODE ENTRAÎNEMENT/TEST DISJOINT\n"
          f"  ENTRAÎNEMENT = {CFG['N_SYNTH_INSTANCES']} instances SYNTHÉTIQUES générées "
          f"| TEST = 56 instances Solomon officielles (gelé)\n{'═'*72}")

    # ── DIAGNOSTIC PRÉALABLE : licence Gurobi à taille limitée ? ────────
    # Informatif uniquement — n'interrompt jamais l'exécution.
    _sec("DIAGNOSTIC — LICENCE GUROBI")
    check_gurobi_license()

    # ── Optionnel : appliquer la meilleure combinaison Taguchi calibrée
    # sur synthétique (si le plan CFG['TAGUCHI_SYNTH'] a déjà été lancé
    # au moins une fois et a produit le JSON correspondant) ───────────
    if CFG.get('USE_TAGUCHI_BEST', False):
        _sec("APPLICATION DE LA MEILLEURE COMBINAISON TAGUCHI (SYNTHÉTIQUE)")
        load_and_apply_taguchi_best(CFG['OUTDIR'], json_suffix='_synthetique')

    # ── ÉTAPE 1a : GÉNÉRATION DES INSTANCES SYNTHÉTIQUES D'ENTRAÎNEMENT ─
    # (reprend generate_instance() de Code 1 — cf. MODULE 1bis ci-dessus).
    # Ces instances SYN-* ne recoupent jamais les noms Solomon (C101, R101,
    # ...) chargés à l'ÉTAPE 2 : aucune fuite entre train et test.
    synth_instances = generate_synthetic_training_set(CFG)

    # ── ÉTAPE 1b : ENTRAÎNEMENT DU DQN — uniquement sur le synthétique ──
    # Valeurs par défaut : si le modèle est rechargé (branche ci-dessous),
    # il n'y a pas de training_log/coûts synthétiques dans CE run — les
    # graphiques de convergence d'entraînement seront alors simplement omis.
    training_log, synth_initial_costs, synth_best_costs = [], None, None
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    if CFG.get('REUSE_EXISTING_SYNTH_MODEL', False) and os.path.exists(model_path):
        _sec(f"ÉTAPE 1b — RÉUTILISATION D'UN MODÈLE DÉJÀ ENTRAÎNÉ SUR LE SYNTHÉTIQUE "
             f"({model_path})")
        agent = DQNAgent(
            obs_dim=7, n_actions=ACTION_SETS_N.get(CFG.get('ACTION_SET', 'full'), 4),
            hidden=CFG['HIDDEN'], lr=CFG['LR'],
            gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
            epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
            buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
            target_update_freq=CFG['TARGET_UPDATE'],
        )
        agent.load(model_path)
    else:
        agent, training_log, synth_initial_costs, synth_best_costs, \
            synth_best_solutions, synth_instance_times, synth_initial_sols = train_on_instances(
                synth_instances, episodes_per_instance=CFG['EPISODES_PER_INSTANCE'])

    # ── ÉTAPE 2a : CHARGEMENT DES 56 INSTANCES SOLOMON (TEST — jamais vues) ─
    _sec("ÉTAPE 2 — CHARGEMENT DES INSTANCES SOLOMON (TEST)")
    test_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
    example_idxs = pick_example_indices(test_instances, preferred_names=CFG['EXAMPLE_NAMES'])

    # ── ÉTAPE 2b : ÉVALUATION GELÉE — le même agent en mémoire, sans réentraîner ──
    initial_costs, best_costs, best_solutions, instance_times, test_initial_sols, test_log = \
        evaluate_frozen_on_instances(
            test_instances, agent, episodes_per_instance=CFG['EPISODES_PER_INSTANCE_TEST'])

    # ── [A] Convergence de l'ENTRAÎNEMENT (synthétique uniquement) ──────
    _sec("GRAPHIQUE — CONVERGENCE DE L'ENTRAÎNEMENT (SYNTHÉTIQUE)")
    if training_log:
        plot_global_convergence(training_log, synth_instances,
                                 synth_initial_costs, synth_best_costs, CFG['OUTDIR'])
    else:
        print("  (pas de courbe de convergence : modèle rechargé, aucun "
              "entraînement effectué dans ce run)")

    # ── [B] Convergence détaillée + [C] Routes avant/après — SUR LE TEST ─
    _sec("GRAPHIQUES — DÉTAIL DU TEST (INSTANCES SOLOMON EXEMPLES)")
    plot_instance_convergence_examples(test_log, test_instances, example_idxs, CFG['OUTDIR'])
    plot_routes_examples(test_instances, test_initial_sols, best_solutions, example_idxs, CFG['OUTDIR'])

    # ── [D] TABLEAU FINAL : SITUATION 1 (test gelé, entraîné sur synthétique) ─
    # vs BKS (gap % par instance + gap moyen) ──────────────────────────
    bks_rows = build_bks_comparison_table(
        test_instances, initial_costs, best_costs, best_solutions, instance_times,
        csv_path=os.path.join(CFG['OUTDIR'], 'solomon_comparaison_bks.csv'),
        latex_path=os.path.join(CFG['OUTDIR'], 'solomon_tableau_bks.tex'),
    )
    # ── [E] Graphique dédié : gap % vs BKS par instance + gap moyen (TEST) ─
    _sec("GRAPHIQUE GAP % vs BKS (TEST)")
    plot_gap_vs_bks(bks_rows, CFG['OUTDIR'])

    # ── [F] Analyses par groupe (C1/C2/R1/R2/RC1/RC2) — véhicules & gap ──
    _sec("GRAPHIQUES — ANALYSES PAR GROUPE D'INSTANCES (TEST)")
    plot_gap_boxplot_by_group(bks_rows, test_instances, CFG['OUTDIR'])
    plot_vehicles_vs_bks(bks_rows, test_instances, CFG['OUTDIR'])

    print(f"\n  ✓ Validation finale terminée en {time.time()-t0:.1f}s "
          f"({(time.time()-t0)/3600:.2f}h)\n")
    return bks_rows
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 11  —  ABLATION DE L'ESPACE D'ACTIONS DU DQN (étude comparative)
# ══════════════════════════════════════════════════════════════════════════════
# Compare 3 jeux d'actions de destruction (voir ACTION_SETS_N / ACTION_NAMES_
# BY_SET, et VRPTWEnv._destroy) à hyperparamètres CFG identiques par ailleurs
# (seul CFG['ACTION_SET'] change entre les 3 runs) :
#   'full'    (4 actions) : spatial, temporel, coût élevé, aléatoire — jeu
#              d'origine, chaque opérateur de destruction séparé.
#   'merged3' (3 actions) : spatial+temporel FUSIONNÉS en une seule action
#              "spatio-temporelle" (moyenne 0.5/0.5 des scores normalisés),
#              coût élevé et aléatoire inchangés.
#   'merged2' (2 actions) : la même fusion spatio-temporelle + aléatoire —
#              l'action "coût élevé" est ici supprimée, pas fusionnée.
# Chaque jeu répète EXACTEMENT le protocole ENTRAÎNEMENT (synthétique, jamais
# Solomon) / TEST (56 instances Solomon, gelé) de run_final_validation() — un
# agent DQN neuf est entraîné à chaque fois (jamais de réutilisation d'un
# .pth calibré sur un autre jeu d'actions, la taille de la couche de sortie
# diffère de toute façon). Objectif : mesurer si réduire/fusionner les
# actions aide (moins de choix à apprendre, budget d'entraînement fixe) ou
# nuit (moins de diversité dans la recherche de destruction).
def run_action_space_ablation():
    """
    Lance run_final_validation()-comme (entraînement synthétique + test
    Solomon gelé) une fois par jeu d'actions dans CFG['ACTION_ABLATION']
    ['SETS'], calcule le gap % vs BKS moyen/min/max pour chacun, exporte un
    CSV détaillé (instance x jeu) + un CSV résumé + un graphique comparatif,
    affiche un tableau récapitulatif trié par gap moyen croissant, et
    sauvegarde le meilleur jeu trouvé dans un JSON.
    Ne modifie PAS CFG['ACTION_SET'] de façon durable (restauré à la fin,
    voir _cfg_override) : pour lancer ensuite une validation normale avec le
    meilleur jeu identifié, reportez-le vous-même dans CFG['ACTION_SET'] et
    repassez CFG['ACTION_ABLATION']['ENABLE']=False (voir message final).
    Retourne un dict {action_set: {'mean_gap_%':..., ...}}.
    """
    t0 = time.time()
    action_sets = CFG['ACTION_ABLATION']['SETS']
    outdir = CFG['OUTDIR']
    _sec(f"ABLATION DE L'ESPACE D'ACTIONS DU DQN — {len(action_sets)} jeux à "
         f"comparer ({', '.join(action_sets)})")
    print(f"  Protocole IDENTIQUE à run_final_validation() pour chaque jeu : "
          f"entraînement sur {CFG['N_SYNTH_INSTANCES']} instances synthétiques "
          f"({CFG['EPISODES_PER_INSTANCE']} épisodes/instance) puis test gelé "
          f"sur les 56 instances Solomon ({CFG['EPISODES_PER_INSTANCE_TEST']} "
          f"épisodes/instance). ⚠ Temps total ≈ {len(action_sets)}× un run normal.")

    # ── Optionnel : appliquer la meilleure combinaison Taguchi calibrée sur
    # synthétique, UNE SEULE FOIS avant la boucle — comme dans
    # run_final_validation(), pour que les 3 jeux d'actions soient comparés
    # avec exactement les mêmes hyperparamètres LNS/Gurobi par ailleurs ────
    if CFG.get('USE_TAGUCHI_BEST', False):
        _sec("APPLICATION DE LA MEILLEURE COMBINAISON TAGUCHI (SYNTHÉTIQUE)")
        load_and_apply_taguchi_best(outdir, json_suffix='_synthetique')

    # ── Instances de TEST (56 Solomon officielles) — chargées une seule
    # fois, réutilisées à l'identique pour les 3 jeux d'actions ──────────
    _sec("CHARGEMENT DES INSTANCES SOLOMON (TEST)")
    test_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])

    all_results = {}
    detail_rows = []
    for set_name in action_sets:
        n_act = ACTION_SETS_N.get(set_name, '?')
        print(f"\n  {'─'*66}\n  Jeu d'actions : '{set_name}' ({n_act} actions)\n  {'─'*66}")
        t_set = time.time()
        with _cfg_override(ACTION_SET=set_name):
            # ── ENTRAÎNEMENT — instances synthétiques générées à la volée,
            # un agent DQN NEUF à chaque jeu d'actions (n_actions diffère) ─
            synth_instances = generate_synthetic_training_set(CFG, verbose=False)
            agent, _, _, _, _, _, _ = train_on_instances(
                synth_instances, episodes_per_instance=CFG['EPISODES_PER_INSTANCE'],
                save_model=False, verbose=False,
            )
            # ── TEST gelé — mêmes 56 instances Solomon pour les 3 jeux ────
            (_, best_costs, best_solutions, instance_times,
             _, _) = evaluate_frozen_on_instances(
                test_instances, agent, episodes_per_instance=CFG['EPISODES_PER_INSTANCE_TEST'])
        elapsed_set = time.time() - t_set
        gaps = []
        for i, inst in enumerate(test_instances):
            bks   = BKS_REFERENCE.get(inst['name'])
            n_veh = sum(1 for r in best_solutions[i] if r)
            gap = None
            if bks:
                gap = (best_costs[i] - bks['distance']) / bks['distance'] * 100.0
                gaps.append(gap)
            detail_rows.append({
                'action_set':  set_name,
                'n_actions':   n_act,
                'instance':    inst['name'],
                'n_vehicules': n_veh,
                'distance':    round(best_costs[i], 2),
                'gap_%':       round(gap, 2) if gap is not None else None,
                'temps_s':     round(instance_times[i], 2),
            })
        mean_gap = float(np.mean(gaps)) if gaps else None
        all_results[set_name] = {
            'n_actions':     n_act,
            'mean_gap_%':    round(mean_gap, 3) if mean_gap is not None else None,
            'min_gap_%':     round(min(gaps), 3) if gaps else None,
            'max_gap_%':     round(max(gaps), 3) if gaps else None,
            'temps_total_s': round(elapsed_set, 1),
        }
        if mean_gap is not None:
            print(f"\n  → Gap moyen ('{set_name}') = {mean_gap:+.2f}%  "
                  f"(min {min(gaps):+.2f}% / max {max(gaps):+.2f}%)  "
                  f"| {elapsed_set:.1f}s ({elapsed_set/3600:.2f}h)")

    # ── Export CSV détaillé (instance x jeu d'actions) ──────────────────
    csv_path = os.path.join(outdir, 'ablation_actions_detail.csv')
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['action_set', 'n_actions', 'instance', 'n_vehicules',
                      'distance', 'gap_%', 'temps_s']
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in detail_rows:
            w.writerow(r)
    summary_path = os.path.join(outdir, 'ablation_actions_resume.csv')
    with open(summary_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['action_set', 'n_actions', 'mean_gap_%', 'min_gap_%',
                      'max_gap_%', 'temps_total_s']
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for set_name, r in all_results.items():
            w.writerow({'action_set': set_name, **r})

    # ── Tableau récapitulatif (trié par gap moyen croissant) ────────────
    _sec("RÉCAPITULATIF — ABLATION DE L'ESPACE D'ACTIONS DU DQN")
    ranked = sorted(all_results.items(),
                     key=lambda kv: (kv[1]['mean_gap_%'] is None, kv[1]['mean_gap_%']))
    print(f"  {'Jeu':<12} {'#Actions':>9} {'Gap moyen':>11} {'Gap min':>9} {'Gap max':>9} {'Temps':>10}")
    print("  " + "─" * 62)
    for set_name, r in ranked:
        gm   = f"{r['mean_gap_%']:+.2f}%" if r['mean_gap_%'] is not None else "—"
        gmin = f"{r['min_gap_%']:+.2f}%"  if r['min_gap_%']  is not None else "—"
        gmax = f"{r['max_gap_%']:+.2f}%"  if r['max_gap_%']  is not None else "—"
        th   = r['temps_total_s'] / 3600.0
        print(f"  {set_name:<12} {r['n_actions']:>9} {gm:>11} {gmin:>9} {gmax:>9} {th:>9.2f}h")
    best_set = ranked[0][0] if ranked else None
    if best_set:
        print(f"\n  → Meilleur jeu d'actions trouvé : '{best_set}' "
              f"(gap moyen = {all_results[best_set]['mean_gap_%']:+.2f}%)")
        print(f"  Pour lancer une validation normale avec ce jeu : mettez "
              f"CFG['ACTION_SET'] = '{best_set}', CFG['ACTION_ABLATION']['ENABLE'] "
              f"= False, et relancez le script.")
        best_json_path = os.path.join(outdir, 'ablation_actions_meilleur_jeu.json')
        with open(best_json_path, 'w', encoding='utf-8') as f:
            json.dump({
                'best_action_set': best_set,
                'resultats':       all_results,
                'n_synth_instances': CFG['N_SYNTH_INSTANCES'],
                'episodes_per_instance': CFG['EPISODES_PER_INSTANCE'],
                'episodes_per_instance_test': CFG['EPISODES_PER_INSTANCE_TEST'],
            }, f, indent=2, ensure_ascii=False)
        print(f"  Meilleur jeu sauvegardé (JSON) → {best_json_path}")

    # ── Graphique comparatif (barres, gap moyen + min/max en repère) ────
    if ranked:
        fig, ax = plt.subplots(figsize=(7, 5))
        names   = [s for s, _ in ranked]
        means   = [all_results[s]['mean_gap_%'] for s in names]
        mins    = [all_results[s]['min_gap_%'] for s in names]
        maxs    = [all_results[s]['max_gap_%'] for s in names]
        errs_lo = [m - lo for m, lo in zip(means, mins)]
        errs_hi = [hi - m for m, hi in zip(means, maxs)]
        colors  = ['#2ca02c' if s == best_set else '#4C6EF5' for s in names]
        labels  = [f"{s}\n({ACTION_SETS_N.get(s, '?')} actions)" for s in names]
        ax.bar(labels, means, color=colors, yerr=[errs_lo, errs_hi], capsize=6)
        ax.axhline(0, color='black', linewidth=0.8)
        for i, m in enumerate(means):
            ax.text(i, m + (1 if m >= 0 else -1), f"{m:+.2f}%", ha='center',
                    fontsize=10, fontweight='bold')
        ax.set_ylabel('Gap % vs BKS (moyenne, min-max en repère)')
        ax.set_title(f"Comparaison des jeux d'actions DQN "
                     f"(56 instances Solomon, {CFG['EPISODES_PER_INSTANCE_TEST']} "
                     f"épisodes/instance test)")
        ax.grid(alpha=0.3, axis='y')
        plt.tight_layout()
        png_path = os.path.join(outdir, 'ablation_actions_comparaison.png')
        plt.savefig(png_path, dpi=140, bbox_inches='tight')
        plt.close(fig)
        print(f"\n  Graphique comparatif → {png_path}")

    print(f"\n  Détail exporté (CSV) → {csv_path}")
    print(f"  Résumé exporté (CSV) → {summary_path}")
    print(f"\n  ✓ Ablation des actions terminée en {time.time()-t0:.1f}s "
          f"({(time.time()-t0)/3600:.2f}h).\n")
    return all_results


if __name__ == '__main__':
    t_global0 = time.time()

    if CFG['TAGUCHI_SYNTH']['ENABLE'] and CFG['TAGUCHI']['ENABLE']:
        raise ValueError(
            "CFG['TAGUCHI_SYNTH']['ENABLE'] et CFG['TAGUCHI']['ENABLE'] sont tous les "
            "deux True — activez UN SEUL des deux modes de calibration à la fois "
            "(TAGUCHI_SYNTH est recommandé, voir les commentaires plus haut)."
        )

    if CFG['TAGUCHI_SYNTH']['ENABLE']:
        # ── MODE CALIBRATION TAGUCHI — SYNTHÉTIQUE UNIQUEMENT (RECOMMANDÉ) ──
        # Aucune instance Solomon n'est chargée dans cette branche : zéro
        # fuite d'hyperparamètres vers le jeu de test.
        ts = CFG['TAGUCHI_SYNTH']
        print(f"\n{'═'*72}\n  MODE PLAN D'EXPÉRIENCE TAGUCHI — SYNTHÉTIQUE UNIQUEMENT "
              f"({ts['N_INSTANCES_DOE']} instances, métrique = amélioration vs LNS, "
              f"pas de BKS, zéro fuite vers Solomon)\n{'═'*72}")
        synth_doe_instances = generate_synthetic_training_set(
            dict(N_SYNTH_INSTANCES=ts['N_INSTANCES_DOE'],
                 SYNTH_N_CUSTOMERS=CFG['SYNTH_N_CUSTOMERS'],
                 SYNTH_N_VEHICLES=CFG['SYNTH_N_VEHICLES'],
                 SYNTH_VEHICLE_SAFETY_FACTOR=CFG['SYNTH_VEHICLE_SAFETY_FACTOR'],
                 SYNTH_CAPACITY=CFG['SYNTH_CAPACITY'], SYNTH_HORIZON=CFG['SYNTH_HORIZON']),
            verbose=True,
        )
        (ts_results, ts_effects_gap, ts_effects_sn,
         ts_anova, ts_best_run) = run_taguchi_doe(
            factors=TAGUCHI_FACTORS,
            outer_instance_names=None,
            episodes_per_instance_doe=ts['EPISODES_PER_INSTANCE_DOE'],
            outdir=CFG['OUTDIR'],
            instances_override=synth_doe_instances,
            metric='lns_improvement',
            json_suffix='_synthetique',
        )
        _sec("RÉCAPITULATIF FINAL DU PLAN TAGUCHI (SYNTHÉTIQUE)")
        if ts_best_run:
            print("  Meilleurs hyperparamètres trouvés (calibrés sur synthétique) :")
            for fn, _ in TAGUCHI_FACTORS:
                print(f"    {fn:<18s} = {ts_best_run[fn]}")
            print(f"  Amélioration moyenne vs LNS obtenue : {-ts_best_run['mean_gap_%']:.2f}%")
            print(f"  Ratio S/N                           : {ts_best_run['sn_ratio']:.2f}")
        print(f"\n  Meilleure combinaison (JSON) : "
              f"{os.path.join(CFG['OUTDIR'], 'taguchi_meilleure_combinaison_synthetique.json')}")
        print(f"\n  ✓ Plan Taguchi (synthétique) terminé en {time.time()-t_global0:.1f}s "
              f"({(time.time()-t_global0)/3600:.2f}h).")

        # ── ENCHAÎNEMENT AUTOMATIQUE → VALIDATION FINALE AVEC LES BEST HYPERPARAMÈTRES ──
        # Entraînement 80 instances synthétiques + test gelé sur les 56 Solomon,
        # en un seul lancement, sans intervention manuelle.
        if ts_best_run:
            _sec("ENCHAÎNEMENT AUTOMATIQUE → VALIDATION FINALE")
            CFG['USE_TAGUCHI_BEST'] = True
            # Repart d'un modèle neuf avec les NOUVEAUX hyperparamètres —
            # ne jamais réutiliser un .pth calibré avec d'anciens réglages ici.
            CFG['REUSE_EXISTING_SYNTH_MODEL'] = False
            run_final_validation()
        else:
            print("  ⚠ Aucune combinaison valide trouvée par le plan Taguchi — "
                  "validation finale non lancée.")

    elif CFG['TAGUCHI']['ENABLE']:
        # ── MODE CALIBRATION TAGUCHI — SOLOMON (⚠ HISTORIQUE, NON RECOMMANDÉ) ──
        # Volontairement PAS d'enchaînement automatique ici : ce mode calibre
        # directement sur le jeu de test (fuite train/test), la combinaison
        # trouvée ne doit pas servir à une validation train/test disjointe.
        print(f"\n{'═'*72}\n  ⚠ MODE PLAN D'EXPÉRIENCE TAGUCHI — SOLOMON (mode historique)\n"
              f"  Ce mode entraîne ET évalue directement sur les 56 instances Solomon :\n"
              f"  la combinaison retournée aura VU le jeu de test et ne doit PAS être\n"
              f"  utilisée comme hyperparamètres d'une validation train/test disjointe.\n"
              f"  Préférez CFG['TAGUCHI_SYNTH']['ENABLE']=True (calibration synthétique).\n"
              f"{'═'*72}")
        tg = CFG['TAGUCHI']
        (tg_results, tg_effects_gap, tg_effects_sn,
         tg_anova, tg_best_run) = run_taguchi_doe(
            factors=TAGUCHI_FACTORS,
            outer_instance_names=tg['OUTER_INSTANCE_NAMES'],
            episodes_per_instance_doe=tg['EPISODES_PER_INSTANCE_DOE'],
            outdir=CFG['OUTDIR'],
            instances_override=None,
            metric='bks_gap',
            json_suffix='',
        )
        _sec("RÉCAPITULATIF FINAL DU PLAN TAGUCHI (SOLOMON — HISTORIQUE)")
        if tg_best_run:
            print("  Meilleurs hyperparamètres trouvés (⚠ calibrés sur Solomon — fuite) :")
            for fn, _ in TAGUCHI_FACTORS:
                print(f"    {fn:<18s} = {tg_best_run[fn]}")
            print(f"  Gap moyen obtenu : {tg_best_run['mean_gap_%']:+.2f}%")
            print(f"  Ratio S/N        : {tg_best_run['sn_ratio']:.2f}")
        print(f"\n  ✓ Plan Taguchi (Solomon) terminé en {time.time()-t_global0:.1f}s "
              f"({(time.time()-t_global0)/3600:.2f}h). Pas d'enchaînement automatique "
              f"(mode avec fuite) — repassez CFG['TAGUCHI']['ENABLE']=False et relancez "
              f"manuellement si vous souhaitez tout de même l'utiliser, à titre de "
              f"comparaison uniquement.\n")

    elif CFG.get('ACTION_ABLATION', {}).get('ENABLE', False):
        # ── MODE ABLATION DE L'ESPACE D'ACTIONS DU DQN (voir MODULE 11) ────
        # Compare full (4) / merged3 (3) / merged2 (2) au lieu de la
        # validation normale. Repassez CFG['ACTION_ABLATION']['ENABLE']=False
        # pour revenir au mode normal (ou à un autre mode ci-dessus).
        run_action_space_ablation()

    else:
        # ── MODE VALIDATION SEULE (comportement normal, sans calibration) ──
        run_final_validation()


════════════════════════════════════════════════════════════════════════
  ABLATION DE L'ESPACE D'ACTIONS DU DQN — 3 jeux à comparer (full, merged3, merged2)
════════════════════════════════════════════════════════════════════════
  Protocole IDENTIQUE à run_final_validation() pour chaque jeu : entraînement sur 80 instances synthétiques (20 épisodes/instance) puis test gelé sur les 56 instances Solomon (10 épisodes/instance). ⚠ Temps total ≈ 3× un run normal.

════════════════════════════════════════════════════════════════════════
  CHARGEMENT DES INSTANCES SOLOMON (TEST)
════════════════════════════════════════════════════════════════════════
  Chargé : C101     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C102     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C103     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C104     type=C   clients=100 véhicules=25 capacité=200 horizon=1236.0
  Chargé : C105   

## Merged 2 + best hyperparametres+ I1 + Fusion des routes

In [4]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON    ║
║                          + GAP PAR RAPPORT AU BKS                           ║
║                                                                              ║
║  Version corrigée : reprend toutes les visualisations de la variante        ║
║  précédente (courbes de convergence, tracés des routes avant/après,        ║
║  graphique dédié gap % vs BKS, tableau récapitulatif véhicules/distance)   ║
║  MAIS applique désormais un vrai protocole ENTRAÎNEMENT / TEST DISJOINT,   ║
║  en un seul lancement, sans aucune intervention manuelle :                 ║
║                                                                              ║
║   ÉTAPE 1 (ENTRAÎNEMENT) — génère des instances SYNTHÉTIQUES (reprises de   ║
║      generate_instance() de main_vrptw_comparaison_gurobi.py / "Code 1")    ║
║      et entraîne le DQN dessus (poids mis à jour via store/learn/decay).    ║
║                                                                              ║
║   ÉTAPE 2 (TEST) — charge les 56 instances Solomon OFFICIELLES et évalue    ║
║      le même agent, GELÉ (aucun store/learn/decay_epsilon, uniquement      ║
║      select_action en pur mode inférence), sur ce jeu jamais vu à          ║
║      l'entraînement. Toutes les courbes/tracés/tableaux ci-dessous portent  ║
║      désormais sur ce TEST gelé, pas sur les résultats d'entraînement.     ║
║                                                                              ║
║  Situation 1 = LNS + DQN(entraîné sur synthétique) + Gurobi, ÉVALUÉ sur     ║
║                Solomon (test)                                → gap vs BKS   ║
║  (La "Situation 2 = Gurobi seul sur l'instance complète" reste retirée,     ║
║   à la demande, car sans limite de temps elle est beaucoup trop longue      ║
║   sur des instances à 100 clients avec fenêtres de temps serrées.)          ║
║                                                                              ║
║  Visualisations produites (toutes sur le TEST gelé, sauf la 1ère) :        ║
║    [A] Convergence de l'ENTRAÎNEMENT (synthétique) : reward global +       ║
║        amélioration % par instance synthétique                            ║
║    [B] Convergence détaillée (instances Solomon exemples, TEST) : coût     ║
║        final/meilleur par épisode local + trajectoire de coût             ║
║    [C] Tracés des routes AVANT (LNS) / APRÈS (DQN gelé) pour des           ║
║        instances Solomon exemples                                         ║
║    [D] Tableau récapitulatif : Instance / Nb véhicules / Distance / Gap %  ║
║        (TEST Solomon vs BKS)                                              ║
║    [E] Graphique dédié gap % vs BKS par instance + gap moyen              ║
║    [F] Diagnostic automatique de licence Gurobi (taille limitée ou non)   ║
║                                                                              ║
║  Aucun rappel manuel de "Code 1" n'est nécessaire : la génération +        ║
║  l'entraînement synthétique sont réintégrés ici (MODULE 1bis) et           ║
║  s'exécutent automatiquement avant le chargement de Solomon. Voir le flag  ║
║  optionnel CFG['REUSE_EXISTING_SYNTH_MODEL'] plus bas si vous préférez     ║
║  recharger un .pth déjà entraîné au lieu de ré-entraîner à chaque run.     ║
║                                                                              ║
║  Par défaut, INSTANCE_FILES contient toutes les instances *.txt trouvées   ║
║  dans outputs/Solomon/. Pour se limiter à 6 instances représentatives      ║
║  (une par groupe C1/C2/R1/R2/RC1/RC2), décommentez le bloc plus bas.       ║
║                                                                              ║
║  Exécution :  python main_vrptw_synth_train_solomon_convergence.py         ║
║                                                                              ║
║  CORRECTIONS APPORTÉES SUR CETTE VERSION :                                 ║
║   1. SOLOMON_DIR pointait vers un chemin Google Colab                      ║
║      ('/content/sample_data/Solomon') — corrigé pour pointer vers          ║
║      outputs/Solomon (dossier local Windows).                              ║
║   2. Le mode CFG['TAGUCHI_SYNTH']['ENABLE']=True s'arrêtait après la       ║
║      calibration et exigeait de relancer le script à la main pour la      ║
║      validation finale. Il enchaîne désormais AUTOMATIQUEMENT, dans le    ║
║      même lancement, sur l'entraînement complet (80 instances               ║
║      synthétiques) + le test gelé sur les 56 Solomon, en appliquant       ║
║      automatiquement la meilleure combinaison d'hyperparamètres trouvée.  ║
║      Voir run_final_validation() et le bloc __main__ tout en bas.         ║
║   3. Construction de la solution initiale remplacée par l'heuristique     ║
║      séquentielle I1 (Solomon, 1987) — cf. _build_initial_i1(), MODULE 2. ║
║      Remplace l'ancienne insertion gloutonne triée par earliest-time      ║
║      (conservée sous _build_initial_greedy_earliest, non appelée) comme   ║
║      point de départ du LNS, pour ENTRAÎNEMENT et TEST. Les meilleurs     ║
║      hyperparamètres Taguchi (LNS_ITER=100, K_DESTROY_RATIO=0.24,         ║
║      GUROBI_TLIMIT=2.0, GUROBI_GAP=0.05, MAX_STEPS=15, EPS_DECAY=0.99,    ║
║      VEHICLE_PENALTY=0.0) sont reportés en dur dans CFG, et le modèle     ║
║      DQN est réentraîné de zéro (REUSE_EXISTING_SYNTH_MODEL=False) car    ║
║      les instances vues à l'entraînement ont désormais une structure de   ║
║      routes de départ différente (voir CFG['I1_MU'/'I1_LAMBDA'/etc.]).    ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import os, sys, json, time, copy, random, csv, glob, itertools
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from contextlib import contextmanager
import torch
import torch.nn as nn
import torch.optim as optim
import gurobipy as gp
from gurobipy import GRB
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG GLOBALE
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    # ── ÉTAPE 1 : ENTRAÎNEMENT sur instances SYNTHÉTIQUES (jamais Solomon) ──
    N_SYNTH_INSTANCES          = 80,     # comme les 80 instances de Code 1
    SYNTH_N_CUSTOMERS          = 100,
    SYNTH_N_VEHICLES           = None,   # None → calculé depuis la demande (cf. generate_instance)
    SYNTH_VEHICLE_SAFETY_FACTOR= 1.5,
    SYNTH_CAPACITY             = 200,
    SYNTH_HORIZON              = 1000,
    EPISODES_PER_INSTANCE      = 20,     # épisodes d'ENTRAÎNEMENT par instance synthétique
    # Optionnel : True → recharge un .pth déjà entraîné sur du synthétique
    # (voir MODEL_FILE) au lieu de régénérer + ré-entraîner. False (par
    # défaut) → tout se fait automatiquement dans CE run, sans intervention
    # manuelle, aucun besoin de relancer "Code 1" séparément.
    # ⚠ Remis à False ici : la construction initiale change (heuristique I1
    # de Solomon 1987, cf. MODULE 2 plus bas, en remplacement de l'ancienne
    # insertion gloutonne triée par earliest-time) → les instances vues à
    # l'entraînement ont une structure de routes de départ différente, donc
    # on RÉENTRAÎNE le DQN de zéro plutôt que de recharger l'ancien .pth
    # (calibré sur l'ancienne construction).
    REUSE_EXISTING_SYNTH_MODEL = False,
    # False ici : les hyperparamètres ci-dessous (LNS_ITER, K_DESTROY_RATIO,
    # GUROBI_TLIMIT, GUROBI_GAP, MAX_STEPS, EPS_DECAY, VEHICLE_PENALTY) sont
    # déjà renseignés EN DUR avec la meilleure combinaison trouvée par le
    # plan Taguchi synthétique lors d'un run précédent — pas besoin de les
    # recharger depuis taguchi_meilleure_combinaison_synthetique.json à
    # l'exécution. Garde le comportement déterministe (n'importe pas le
    # contenu du JSON, qui a été calibré avec l'ANCIENNE heuristique de
    # construction initiale, avant l'ajout d'I1).
    USE_TAGUCHI_BEST            = False,
    # ── ÉTAPE 2 : TEST gelé sur les 56 instances Solomon officielles ───────
    EPISODES_PER_INSTANCE_TEST = 10,     # épisodes d'ÉVALUATION (aucun apprentissage) par instance Solomon
    # VEHICLE_SAFETY_FACTOR = None : on utilise la flotte OFFICIELLE de chaque
    # instance, pas une borne calculée sur la demande. Sur R101/RC101
    # (fenêtres de temps serrées), le BKS utilise 19 et 14 véhicules alors
    # que la demande seule suggérerait ~12-14 : une borne basée uniquement
    # sur la demande sous-estime le besoin réel et peut rendre le MILP
    # infaisable. La flotte officielle est plus grande (MILP plus lourd)
    # mais garantit qu'une solution proche du BKS reste atteignable.
    VEHICLE_SAFETY_FACTOR  = None,
    # ── Hyperparamètres LNS/DQN/Gurobi : meilleure combinaison trouvée par
    # le plan d'expérience Taguchi synthétique (fichier
    # taguchi_meilleure_combinaison_synthetique.json), reportée ici EN DUR
    # (voir USE_TAGUCHI_BEST=False ci-dessus). ────────────────────────────
    LNS_ITER                = 100,
    LNS_K_RATIO            = 0.20,
    K_DESTROY_RATIO        = 0.24,
    MAX_STEPS               = 15,
    # repair par bouts (Situation 1, appelé à CHAQUE step du LNS/DQN — voir
    # check_gurobi_license() plus bas pour l'impact d'une licence à taille
    # limitée). None = aucune limite de temps (supporté par le code).
    GUROBI_TLIMIT            = 2.0,
    GUROBI_GAP               = 0.05,
    HIDDEN                   = 64,
    LR                       = 1e-3,
    GAMMA                    = 0.95,
    EPS_START                = 1.0,
    EPS_END                  = 0.05,
    EPS_DECAY                = 0.99,  # best Taguchi — voir l'écrasement explicite juste après CFG plus bas
    BUFFER_SIZE               = 5000,
    BATCH_SIZE                = 32,
    TARGET_UPDATE              = 10,
    # ── Instances exemples (pour courbes détaillées + routes, TEST/Solomon) ─
    # Noms recherchés en priorité (représentatifs des 6 groupes Solomon) ;
    # si un nom n'est pas présent dans INSTANCE_FILES, il est ignoré et
    # remplacé automatiquement par le 1er type disponible.
    EXAMPLE_NAMES              = ['C101', 'R101', 'RC101'],
    # ── Pénalité par véhicule (repair MILP) ────────────────────────
    # 0.0 = comportement d'origine (aucune pénalité, Gurobi peut ouvrir
    # librement des véhicules supplémentaires à chaque repair). Une valeur
    # > 0 ajoute un coût fixe par véhicule utilisé dans l'objectif du MILP
    # de repair, pour limiter l'inflation du nombre de véhicules observée
    # dans la validation vs BKS. C'est aussi l'un des facteurs calibrés par
    # le plan d'expérience Taguchi (MODULE 10 plus bas) — calibré
    # UNIQUEMENT sur du synthétique (CFG['TAGUCHI_SYNTH']), jamais sur
    # Solomon, pour ne pas faire fuiter d'information du test set vers le
    # choix des hyperparamètres.
    VEHICLE_PENALTY            = 0.0,
    # ── Heuristique de construction initiale : I1 (Solomon, 1987) ──────
    # Remplace l'ancienne construction gloutonne triée par earliest-time
    # (_build_initial, conservée plus bas pour référence mais plus appelée).
    # Paramètres classiques de Solomon (1987) : mu=1 (poids de la distance
    # directe i→j dans le coût d'insertion c11), lambda=2 (poids de
    # l'éloignement au dépôt dans le critère de sélection c2 — favorise les
    # clients "difficiles" excentrés), alpha1=1/alpha2=0 (c1 = c11 pur,
    # sans le terme de décalage temporel c12 — paramétrage le plus courant
    # dans la littérature ; augmenter alpha2 pénalise davantage les
    # insertions qui retardent fortement le client suivant).
    I1_MU                       = 1.0,
    I1_LAMBDA                   = 2.0,
    I1_ALPHA1                   = 1.0,
    I1_ALPHA2                   = 0.0,
    OUTDIR                        = os.path.join(os.getcwd(), 'outputs'),
    MODELDIR                       = os.path.join(os.getcwd(), 'outputs', 'model'),
    MODEL_FILE                      = 'dqn_vrptw_synth_trained.pth',
)
CFG['LNS_K'] = 20   # ~20% de 100 clients, cohérent avec LNS_K_RATIO
CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))  # 24, cohérent avec K_DESTROY_RATIO
# ── Recalibrage automatique de EPS_DECAY : DÉSACTIVÉ ────────────────────
# À l'origine, ce bloc recalculait EPS_DECAY pour que epsilon décroisse de
# EPS_START à EPS_END sur toute la durée de l'entraînement (utile quand
# EPS_DECAY n'était pas calibré). Ici, EPS_DECAY=0.99 vient directement de
# la meilleure combinaison trouvée par le plan Taguchi synthétique (cf.
# CFG plus haut) — un recalcul automatique l'écraserait. On le laisse donc
# tel quel ; ce bloc est conservé en commentaire pour référence/mémoire si
# jamais EPS_DECAY doit être re-dérivé automatiquement dans une variante
# future du script.
# _n_decays_total = max(1, CFG['N_SYNTH_INSTANCES'] * CFG['EPISODES_PER_INSTANCE'])
# CFG['EPS_DECAY'] = (CFG['EPS_END'] / CFG['EPS_START']) ** (1.0 / _n_decays_total)
os.makedirs(CFG['OUTDIR'],   exist_ok=True)
os.makedirs(CFG['MODELDIR'], exist_ok=True)
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG DU PLAN D'EXPÉRIENCE TAGUCHI (calibration des hyperparamètres)
# ══════════════════════════════════════════════════════════════════════════════
# Deux modes de calibration disponibles (voir plus bas pour le mode
# TAGUCHI_SYNTH, recommandé, qui enchaîne désormais automatiquement sur la
# validation finale). Activez explicitement l'UN DES DEUX (pas les deux en
# même temps) :
#
#  • CFG['TAGUCHI_SYNTH']['ENABLE'] = True   (RECOMMANDÉ)
#    Calibre en s'entraînant UNIQUEMENT sur des instances SYNTHÉTIQUES
#    générées à la volée (jamais Solomon). Métrique de classement :
#    amélioration % vs coût LNS initial (pas de BKS requis). La meilleure
#    combinaison trouvée n'a jamais "vu" les 56 instances Solomon → aucune
#    fuite d'hyperparamètres vers le jeu de test, contrairement au mode
#    historique ci-dessous. Une fois la calibration terminée, le script
#    enchaîne AUTOMATIQUEMENT sur la validation finale (entraînement 80
#    instances + test gelé sur les 56 Solomon) avec ces hyperparamètres.
#
#  • CFG['TAGUCHI']['ENABLE'] = True   (⚠ HISTORIQUE — NON RECOMMANDÉ)
#    Calibre en s'entraînant DIRECTEMENT sur les 56 instances Solomon (la
#    matrice externe = le jeu de test lui-même) et classe les combinaisons
#    par leur gap % vs BKS sur ces mêmes instances. C'est exactement le
#    protocole "Run 16" identifié comme fuite d'hyperparamètres (on choisit
#    les hyperparamètres en fonction de la performance sur le test) —
#    conservé uniquement pour comparaison / compatibilité avec d'anciens
#    résultats, PAS pour produire les hyperparamètres d'une validation
#    finale train/test disjointe. Ce mode n'enchaîne PAS automatiquement
#    sur la validation finale (voir le bloc __main__ tout en bas).
CFG['TAGUCHI'] = dict(
    ENABLE                     = False,  # ⚠ historique — fuite vers Solomon (test set), voir ci-dessus
    EPISODES_PER_INSTANCE_DOE  = 5,
    OUTER_INSTANCE_NAMES       = 'ALL',  # 'ALL' = les 56 instances Solomon comme matrice externe
)
CFG['TAGUCHI_SYNTH'] = dict(
    # False ici : la calibration a déjà été faite lors d'un run précédent
    # (le JSON taguchi_meilleure_combinaison_synthetique.json existe déjà).
    # Repasser à True relancerait tout le plan Taguchi (18 runs) pour rien —
    # CFG['USE_TAGUCHI_BEST']=True ci-dessus suffit à réutiliser ses résultats.
    ENABLE                     = False,  # True = lance le plan Taguchi calibré sur synthétique (recommandé)
    # Volume d'instances synthétiques utilisé comme matrice externe pendant
    # CE plan de calibration (indépendant de N_SYNTH_INSTANCES du protocole
    # d'entraînement final — un sous-ensemble suffit ici, l'objectif est de
    # classer les 18 combinaisons, pas d'entraîner le modèle définitif).
    N_INSTANCES_DOE            = 24,   # 8 par type C/R/RC
    EPISODES_PER_INSTANCE_DOE  = 5,
)
# Jeu d'actions actif du DQN pour un run normal (validation / Taguchi) :
# 'full' (4 actions) / 'merged3' (3) / 'merged2' (2) — voir ACTION_SETS_N /
# ACTION_NAMES_BY_SET et MODULE 11 (ABLATION DES ACTIONS DQN) plus bas.
# ── FIXÉ SUR 'merged2' ────────────────────────────────────────────────
# Résultat de l'ablation des 3 jeux (voir run_action_space_ablation()) :
# merged2 (2 actions : spatio-temporelle fusionnée + aléatoire) a obtenu le
# meilleur gap moyen vs BKS (+7.55%, contre +8.22% pour full et +9.23% pour
# merged3, mêmes hyperparamètres pour les 3). C'est donc le jeu retenu ici
# pour la validation combinée finale (merged2 + best hyperparamètres + I1 +
# fusion de routes). ⚠ Résultat basé sur un seul seed par jeu (voir mise en
# garde donnée en conversation) — à confirmer par une répétition multi-seed
# avant de le considérer comme définitivement supérieur à 'full'.
CFG['ACTION_SET'] = 'merged2'
# ── Ablation de l'espace d'actions du DQN (étude comparative, MODULE 11) ──
# ENABLE=True : au lieu de la validation normale, lance
# run_action_space_ablation(), qui répète EXACTEMENT le protocole
# ENTRAÎNEMENT (synthétique) / TEST (Solomon gelé) de run_final_validation()
# une fois par jeu d'actions dans SETS — mêmes CFG['N_SYNTH_INSTANCES'],
# CFG['EPISODES_PER_INSTANCE'], CFG['EPISODES_PER_INSTANCE_TEST'] et mêmes
# hyperparamètres LNS/Gurobi par ailleurs, SEUL CFG['ACTION_SET'] change
# d'un run à l'autre — pour une comparaison à hyperparamètres égaux.
# ⚠ Ce mode s'exécute 3× le temps d'un run normal (un entraînement+test
# complet par jeu d'actions). Repassez ENABLE=False pour revenir à la
# validation normale.
# ── DÉSACTIVÉ ici : on veut UN SEUL run, avec 'merged2' fixé ci-dessus
# (CFG['ACTION_SET']), pas les 3 jeux à la suite. Repassez ENABLE=True si
# vous voulez relancer la comparaison des 3 jeux (avec fusion de routes
# incluse cette fois, voir CFG['ROUTE_FUSION'] plus bas — la fusion
# s'applique alors identiquement aux 3 jeux, comme les autres hyperparamètres).
CFG['ACTION_ABLATION'] = dict(
    ENABLE = False,
    SETS   = ['full', 'merged3', 'merged2'],
)
# ── Fusion de routes (post-traitement, MODULE 5bis) ─────────────────────
# ENABLE=True : après l'évaluation gelée (TEST, DQN+LNS+Gurobi) sur les 56
# instances Solomon, tente de fusionner deux à deux les routes de chaque
# solution finale quand c'est faisable (capacité + fenêtres de temps) et
# rentable (coût total plus bas) — voir route_merge_postprocess() pour le
# détail. Objectif : réduire l'inflation du nombre de véhicules vs BKS
# observée dans les validations précédentes (ex. C101 : 11 véhicules vs 10
# au BKS), sans toucher au DQN ni à Gurobi. N'affecte QUE le TEST (Solomon) ;
# n'est jamais appliqué pendant l'entraînement synthétique.
CFG['ROUTE_FUSION'] = dict(
    ENABLE = True,
)
# Facteurs de contrôle et leurs niveaux (plan Taguchi L18 : 7 facteurs à 3
# niveaux, 18 combinaisons au lieu des 3^7 = 2187 combinaisons possibles).
TAGUCHI_FACTORS = [
    ('LNS_ITER',         [30, 60, 100]),
    ('K_DESTROY_RATIO',  [0.15, 0.20, 0.24]),
    ('GUROBI_TLIMIT',    [1.0, 2.0, 3.0]),
    ('GUROBI_GAP',       [0.05, 0.03, 0.01]),
    ('MAX_STEPS',        [10, 15, 20]),
    ('EPS_DECAY',        [0.90, 0.95, 0.99]),
    ('VEHICLE_PENALTY',  [0.0, 300.0, 800.0]),
]
GROUP_ORDER = ['C1', 'C2', 'R1', 'R2', 'RC1', 'RC2']
ACTION_NAMES = {
    0: "Destruction spatiale",
    1: "Destruction temporelle",
    2: "Destruction par coût élevé",
    3: "Destruction aléatoire",
}
# ── Ablation de l'espace d'actions du DQN (voir MODULE 11) ─────────────────
# 'full'    : 4 actions d'origine (spatial, temporel, coût élevé, aléatoire)
# 'merged3' : 3 actions — spatial+temporel FUSIONNÉS en une seule action
#             "spatio-temporelle" (moyenne 0.5/0.5 des scores normalisés,
#             voir VRPTWEnv._spatiotemporal_destroy), coût élevé et
#             aléatoire inchangés séparément.
# 'merged2' : 2 actions — la même fusion spatio-temporelle, + aléatoire.
#             L'action "coût élevé" est ici purement SUPPRIMÉE (pas
#             fusionnée) : c'est le jeu d'actions le plus réduit.
ACTION_SETS_N = {'full': 4, 'merged3': 3, 'merged2': 2}
ACTION_NAMES_BY_SET = {
    'full':    {0: "Spatiale", 1: "Temporelle", 2: "Coût élevé", 3: "Aléatoire"},
    'merged3': {0: "Spatio-temporelle (fusion)", 1: "Coût élevé", 2: "Aléatoire"},
    'merged2': {0: "Spatio-temporelle (fusion)", 1: "Aléatoire"},
}
# ── Dossier contenant les instances Solomon officielles (extraites du zip) ──
# CORRIGÉ : pointait vers '/content/sample_data/Solomon' (chemin Google
# Colab, inexistant sur une machine locale) — repointé vers outputs/Solomon.
SOLOMON_DIR = os.path.join(CFG['OUTDIR'], 'Solomon')
# Toutes les instances *.txt trouvées (décommentez le bloc ci-dessous pour
# revenir à seulement 6 instances représentatives, une par groupe
# C1/C2/R1/R2/RC1/RC2, afin de garder un temps d'exécution raisonnable).
INSTANCE_FILES = sorted(glob.glob(os.path.join(SOLOMON_DIR, '*.txt')))
# INSTANCE_FILES = [
#     os.path.join(SOLOMON_DIR, 'c101.txt'),
#     os.path.join(SOLOMON_DIR, 'c201.txt'),
#     os.path.join(SOLOMON_DIR, 'r101.txt'),
#     os.path.join(SOLOMON_DIR, 'r201.txt'),
#     os.path.join(SOLOMON_DIR, 'rc101.txt'),
#     os.path.join(SOLOMON_DIR, 'rc201.txt'),
# ]
# ── BKS officiel complet, 56 instances (source : SINTEF / Solomon 1987) ──────
BKS_REFERENCE = {
    'C101': {'vehicles': 10, 'distance': 828.94}, 'C102': {'vehicles': 10, 'distance': 828.94},
    'C103': {'vehicles': 10, 'distance': 828.06}, 'C104': {'vehicles': 10, 'distance': 824.78},
    'C105': {'vehicles': 10, 'distance': 828.94}, 'C106': {'vehicles': 10, 'distance': 828.94},
    'C107': {'vehicles': 10, 'distance': 828.94}, 'C108': {'vehicles': 10, 'distance': 828.94},
    'C109': {'vehicles': 10, 'distance': 828.94},
    'C201': {'vehicles': 3, 'distance': 591.56}, 'C202': {'vehicles': 3, 'distance': 591.56},
    'C203': {'vehicles': 3, 'distance': 591.17}, 'C204': {'vehicles': 3, 'distance': 590.60},
    'C205': {'vehicles': 3, 'distance': 588.88}, 'C206': {'vehicles': 3, 'distance': 588.49},
    'C207': {'vehicles': 3, 'distance': 588.29}, 'C208': {'vehicles': 3, 'distance': 588.32},
    'R101': {'vehicles': 19, 'distance': 1650.80}, 'R102': {'vehicles': 17, 'distance': 1486.12},
    'R103': {'vehicles': 13, 'distance': 1292.68}, 'R104': {'vehicles': 9,  'distance': 1007.31},
    'R105': {'vehicles': 14, 'distance': 1377.11}, 'R106': {'vehicles': 12, 'distance': 1252.03},
    'R107': {'vehicles': 10, 'distance': 1104.66}, 'R108': {'vehicles': 9,  'distance': 960.88},
    'R109': {'vehicles': 11, 'distance': 1194.73}, 'R110': {'vehicles': 10, 'distance': 1118.84},
    'R111': {'vehicles': 10, 'distance': 1096.72}, 'R112': {'vehicles': 9,  'distance': 982.14},
    'R201': {'vehicles': 4, 'distance': 1252.37}, 'R202': {'vehicles': 3, 'distance': 1191.70},
    'R203': {'vehicles': 3, 'distance': 939.50},  'R204': {'vehicles': 2, 'distance': 825.52},
    'R205': {'vehicles': 3, 'distance': 994.43},  'R206': {'vehicles': 3, 'distance': 906.14},
    'R207': {'vehicles': 2, 'distance': 890.61},  'R208': {'vehicles': 2, 'distance': 726.82},
    'R209': {'vehicles': 3, 'distance': 909.16},  'R210': {'vehicles': 3, 'distance': 939.37},
    'R211': {'vehicles': 2, 'distance': 885.71},
    'RC101': {'vehicles': 14, 'distance': 1696.95}, 'RC102': {'vehicles': 12, 'distance': 1554.75},
    'RC103': {'vehicles': 11, 'distance': 1261.67}, 'RC104': {'vehicles': 10, 'distance': 1135.48},
    'RC105': {'vehicles': 13, 'distance': 1629.44}, 'RC106': {'vehicles': 11, 'distance': 1424.73},
    'RC107': {'vehicles': 11, 'distance': 1230.48}, 'RC108': {'vehicles': 10, 'distance': 1139.82},
    'RC201': {'vehicles': 4, 'distance': 1406.94}, 'RC202': {'vehicles': 3, 'distance': 1365.65},
    'RC203': {'vehicles': 3, 'distance': 1049.62}, 'RC204': {'vehicles': 3, 'distance': 798.46},
    'RC205': {'vehicles': 4, 'distance': 1297.65}, 'RC206': {'vehicles': 3, 'distance': 1146.32},
    'RC207': {'vehicles': 3, 'distance': 1061.14}, 'RC208': {'vehicles': 3, 'distance': 828.14},
}
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1  —  LECTURE DES INSTANCES SOLOMON OFFICIELLES  (jeu de TEST)
# ══════════════════════════════════════════════════════════════════════════════
def euclidean(a, b):
    return float(np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2))
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1bis  —  GÉNÉRATEUR D'INSTANCES SYNTHÉTIQUES  (jeu d'ENTRAÎNEMENT)
#  Repris tel quel de main_vrptw_comparaison_gurobi.py ("Code 1") : mêmes
#  fonctions _coords_C/_coords_R/_coords_RC et generate_instance(), pour que
#  les instances d'entraînement soient générées exactement comme dans Code 1.
# ══════════════════════════════════════════════════════════════════════════════
def _coords_C(n, rng):
    n_cl = max(3, n // 10)
    centers = rng.uniform(15, 85, (n_cl, 2))
    pts = []
    for i in range(n):
        p = rng.normal(centers[i % n_cl], 8)
        pts.append(np.clip(p, 1, 99))
    return np.array(pts)
def _coords_R(n, rng):
    return rng.uniform(1, 99, (n, 2))
def _coords_RC(n, rng):
    h = n // 2
    return np.vstack([_coords_C(h, rng), _coords_R(n - h, rng)])
def generate_instance(n_customers=50, n_vehicles=None, capacity=200,
                      type_='C', seed=42, horizon=1000,
                      vehicle_safety_factor=1.5):
    rng     = np.random.RandomState(seed)
    depot   = np.array([50.0, 50.0])
    gen_fn  = {'C': _coords_C, 'R': _coords_R, 'RC': _coords_RC}[type_]
    coords  = gen_fn(n_customers, rng)
    max_dem = max(1, capacity // 5)
    demands = rng.randint(1, max_dem + 1, n_customers)
    svc     = rng.randint(10, 31, n_customers)
    if n_vehicles is None:
        min_vehicles = int(np.ceil(demands.sum() / capacity))
        n_vehicles   = max(2, int(np.ceil(min_vehicles * vehicle_safety_factor)))
    customers = []
    for i in range(n_customers):
        d        = euclidean(depot, coords[i])
        earliest = max(0.0, round(d + rng.uniform(-15, 15), 1))
        delta    = rng.uniform(30, 80)
        latest   = min(float(horizon), round(earliest + delta, 1))
        latest   = max(latest, earliest + svc[i] + 5)
        latest   = min(latest, float(horizon))
        if d > latest:
            earliest = round(d, 1)
            latest   = min(float(horizon), round(earliest + delta, 1))
            latest   = max(latest, earliest + svc[i] + 5)
        if d + svc[i] + d > horizon:
            svc[i] = max(5, int(horizon - 2 * d - 5))
        customers.append({
            'id':           i + 1,
            'x':            float(round(coords[i][0], 2)),
            'y':            float(round(coords[i][1], 2)),
            'demand':       int(demands[i]),
            'earliest':     float(earliest),
            'latest':       float(latest),
            'service_time': int(svc[i]),
        })
    return {
        'depot': {
            'id': 0, 'x': 50.0, 'y': 50.0, 'demand': 0,
            'earliest': 0.0, 'latest': float(horizon), 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    capacity,
        'horizon':     horizon,
        'type':        type_,
        'seed':        seed,
        'n_customers': n_customers,
    }
def generate_synthetic_training_set(cfg, verbose=True):
    """
    Génère les instances synthétiques d'ENTRAÎNEMENT en cyclant les 3 types
    Solomon C / R / RC (même logique que train() dans Code 1 : type =
    ['C','R','RC'][i % 3], seed = i*7+1). Ces instances ne recouvrent JAMAIS
    les 56 instances Solomon officielles utilisées plus bas comme jeu de
    TEST : le DQN ne les voit donc jamais avant l'évaluation gelée.
    Chaque instance reçoit en plus une clé 'name' (ex. 'SYN-C001') pour
    rester compatible avec les fonctions d'entraînement/affichage qui
    utilisent inst['name'].
    """
    types = ['C', 'R', 'RC']
    instances = []
    if verbose:
        _sec(f"ÉTAPE 1 — GÉNÉRATION DE {cfg['N_SYNTH_INSTANCES']} INSTANCES "
             f"SYNTHÉTIQUES (ENTRAÎNEMENT)")
    for i in range(cfg['N_SYNTH_INSTANCES']):
        t    = types[i % 3]
        seed = i * 7 + 1
        inst = generate_instance(
            n_customers = cfg['SYNTH_N_CUSTOMERS'],
            n_vehicles  = cfg['SYNTH_N_VEHICLES'],
            capacity    = cfg['SYNTH_CAPACITY'],
            horizon     = cfg['SYNTH_HORIZON'],
            type_       = t,
            seed        = seed,
            vehicle_safety_factor = cfg['SYNTH_VEHICLE_SAFETY_FACTOR'],
        )
        inst['name'] = f"SYN-{t}{i+1:03d}"
        instances.append(inst)
        if verbose:
            print(f"  Instance {i+1:03d}/{cfg['N_SYNTH_INSTANCES']} | {inst['name']:<10} "
                  f"type={t:<3} | seed={seed} | véhicules={inst['n_vehicles']}")
    return instances
def parse_solomon_instance(filepath, vehicle_safety_factor=None):
    """
    Parse un fichier d'instance Solomon au format texte officiel :
        <NOM>
        VEHICLE
        NUMBER  CAPACITY
        <n>     <cap>
        CUSTOMER
        CUST NO.  XCOORD.  YCOORD.  DEMAND  READY TIME  DUE DATE  SERVICE TIME
        0  ...   (dépôt)
        1  ...
        ...
    Retourne un dict compatible avec le reste du pipeline.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = [l.rstrip('\n') for l in f]
    name = lines[0].strip()
    idx_veh = next(i for i, l in enumerate(lines) if l.strip() == 'VEHICLE')
    veh_number, veh_capacity = map(int, lines[idx_veh + 2].split())
    idx_cust = next(i for i, l in enumerate(lines) if l.strip() == 'CUSTOMER')
    data_lines = [l for l in lines[idx_cust + 2:] if l.strip()]
    rows = [list(map(float, l.split())) for l in data_lines]
    depot_row = rows[0]
    customers = []
    for r in rows[1:]:
        cid, x, y, dem, ready, due, svc = r
        customers.append({
            'id':           int(cid),
            'x':            x,
            'y':            y,
            'demand':       int(dem),
            'earliest':     ready,
            'latest':       due,
            'service_time': int(svc),
        })
    name_upper = name.upper()
    type_ = ''.join(ch for ch in name_upper if ch.isalpha())   # 'C101' -> 'C', 'RC101' -> 'RC'
    n_vehicles = veh_number
    if vehicle_safety_factor is not None:
        total_demand = sum(c['demand'] for c in customers)
        min_veh      = int(np.ceil(total_demand / veh_capacity))
        n_vehicles   = min(veh_number, max(2, int(np.ceil(min_veh * vehicle_safety_factor))))
    return {
        'depot': {
            'id': 0, 'x': depot_row[1], 'y': depot_row[2], 'demand': 0,
            'earliest': depot_row[4], 'latest': depot_row[5], 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    veh_capacity,
        'horizon':     depot_row[5],
        'type':        type_,
        'seed':        name_upper,
        'n_customers': len(customers),
        'name':        name_upper,          # nom officiel normalisé (ex. 'C101')
    }
def load_all_instances(files, vehicle_safety_factor=None):
    if not files:
        raise FileNotFoundError(
            f"Aucun fichier d'instance trouvé dans {SOLOMON_DIR}. "
            f"Vérifiez que le dossier existe et contient les .txt Solomon."
        )
    instances = []
    for fp in files:
        inst = parse_solomon_instance(fp, vehicle_safety_factor=vehicle_safety_factor)
        instances.append(inst)
        print(f"  Chargé : {inst['name']:<8} type={inst['type']:<3} "
              f"clients={inst['n_customers']} véhicules={inst['n_vehicles']} "
              f"capacité={inst['capacity']} horizon={inst['horizon']}")
    return instances
def pick_example_indices(instances, preferred_names=None, max_examples=3):
    """
    Sélectionne des instances "exemples" pour les courbes détaillées et les
    tracés de routes. Cherche d'abord les noms préférés (ex. C101/R101/RC101).
    Si absents, retombe sur la 1ère instance de chaque type détecté, puis sur
    les toutes premières instances de la liste.
    """
    preferred_names = preferred_names or []
    by_name = {inst['name']: i for i, inst in enumerate(instances)}
    idxs = [by_name[nm] for nm in preferred_names if nm in by_name]
    if not idxs:
        seen_types = {}
        for i, inst in enumerate(instances):
            if inst['type'] not in seen_types:
                seen_types[inst['type']] = i
            if len(seen_types) >= max_examples:
                break
        idxs = list(seen_types.values())
    if not idxs:
        idxs = list(range(min(max_examples, len(instances))))
    return idxs[:max_examples]
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 2  —  SOLUTION INITIALE PAR LNS  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
def _node_xy(nid, instance):
    if nid == 0:
        return instance['depot']['x'], instance['depot']['y']
    c = instance['customers'][nid - 1]
    return c['x'], c['y']
def _node_info(nid, instance):
    if nid == 0:
        return instance['depot']
    return instance['customers'][nid - 1]
def compute_cost(solution, instance):
    depot = (instance['depot']['x'], instance['depot']['y'])
    total = 0.0
    for route in solution:
        if not route:
            continue
        prev = depot
        for cid in route:
            cur   = _node_xy(cid, instance)
            total += euclidean(prev, cur)
            prev  = cur
        total += euclidean(prev, depot)
    return total
def _route_feasible(route, instance):
    if not route:
        return True
    cap  = instance['capacity']
    load = sum(_node_info(c, instance)['demand'] for c in route)
    if load > cap:
        return False
    depot = instance['depot']
    t     = 0.0
    px, py = depot['x'], depot['y']
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        if t > info['latest']:
            return False
        t      = max(t, info['earliest']) + info['service_time']
        px, py = cx, cy
    t += euclidean((px, py), (depot['x'], depot['y']))
    return t <= depot['latest']
def is_feasible(solution, instance):
    return all(_route_feasible(r, instance) for r in solution)
def solution_stats(solution, instance):
    cap        = instance['capacity']
    total_cost = compute_cost(solution, instance)
    n_vehicles = sum(1 for r in solution if r)
    loads      = []
    total_wait = 0.0
    for route in solution:
        if not route:
            continue
        load = sum(_node_info(c, instance)['demand'] for c in route)
        loads.append(load / cap)
        depot  = instance['depot']
        t      = 0.0
        px, py = depot['x'], depot['y']
        for cid in route:
            cx, cy = _node_xy(cid, instance)
            t     += euclidean((px, py), (cx, cy))
            info   = _node_info(cid, instance)
            total_wait += max(0.0, info['earliest'] - t)
            t      = max(t, info['earliest']) + info['service_time']
            px, py = cx, cy
    return {
        'total_cost':         round(total_cost, 2),
        'n_vehicles':         n_vehicles,
        'avg_load_ratio':     round(np.mean(loads) if loads else 0.0, 3),
        'total_wait_time':    round(total_wait, 2),
        'n_customers_served': sum(len(r) for r in solution),
    }
def _insertion_cost(cid, pos, route, instance):
    depot   = (instance['depot']['x'], instance['depot']['y'])
    nodes   = [0] + route + [0]
    prev_id = nodes[pos]
    next_id = nodes[pos + 1]
    px, py  = _node_xy(prev_id, instance) if prev_id != 0 else depot
    nx, ny  = _node_xy(next_id, instance) if next_id != 0 else depot
    cx, cy  = _node_xy(cid, instance)
    return (euclidean((px, py), (cx, cy)) + euclidean((cx, cy), (nx, ny))
            - euclidean((px, py), (nx, ny)))
def _try_insert(cid, route, instance):
    best_cost, best_pos = float('inf'), -1
    for pos in range(len(route) + 1):
        candidate = route[:pos] + [cid] + route[pos:]
        if _route_feasible(candidate, instance):
            cost = _insertion_cost(cid, pos, route, instance)
            if cost < best_cost:
                best_cost, best_pos = cost, pos
    return best_cost, best_pos
def _repair_greedy(removed, partial, instance):
    routes = [list(r) for r in partial]
    for cid in removed:
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _build_initial_greedy_earliest(instance):
    """
    ⚠ CONSERVÉE POUR RÉFÉRENCE UNIQUEMENT — n'est plus appelée par
    get_initial_solution() (remplacée par _build_initial_i1, l'heuristique
    I1 de Solomon (1987) ci-dessous). Construction gloutonne d'origine :
    trie les clients par earliest-time puis insère chacun au meilleur
    endroit (au sens du coût d'insertion) parmi les routes déjà ouvertes,
    ou ouvre une nouvelle route si aucune insertion n'est faisable.
    """
    customers = sorted(instance['customers'], key=lambda c: c['earliest'])
    routes    = []
    for c in customers:
        cid = c['id']
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _route_service_starts(route, instance):
    """
    Propage les temps le long d'une route (sans le dépôt) et retourne la
    liste des heures de DÉBUT DE SERVICE b_j pour chaque client de la route,
    dans l'ordre. Utilisé par l'heuristique I1 pour calculer le terme de
    décalage temporel c12 (poussée en avant de l'heure de service du client
    suivant, cf. Solomon 1987).
    """
    depot  = instance['depot']
    t      = 0.0
    px, py = depot['x'], depot['y']
    starts = []
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        t      = max(t, info['earliest'])
        starts.append(t)
        t     += info['service_time']
        px, py = cx, cy
    return starts
def _build_initial_i1(instance, mu=1.0, lam=2.0, alpha1=1.0, alpha2=0.0):
    """
    Heuristique de construction séquentielle I1 (Solomon, 1987, "Algorithms
    for the Vehicle Routing and Scheduling Problems with Time Window
    Constraints", Operations Research 35(2)). Remplace l'ancienne
    construction gloutonne triée par earliest-time (_build_initial_greedy_
    earliest ci-dessus) comme point de départ du LNS.
    Principe : construit les routes une par une. Chaque route démarre avec
    un client "graine" (le client non routé le plus éloigné du dépôt), puis
    insère itérativement, parmi tous les clients non routés et toutes les
    positions faisables de la route courante, celui qui maximise le critère
    de sélection c2(u) = lambda*d(0,u) - c1*(u) — favorisant les clients
    excentrés difficiles à insérer plus tard — où c1*(u) est le coût
    d'insertion minimal de u dans la route courante :
        c1(i,u,j) = alpha1*c11(i,u,j) + alpha2*c12(i,u,j)
        c11(i,u,j) = d(i,u) + d(u,j) - mu*d(i,j)                  (détour géométrique)
        c12(i,u,j) = b_ju - b_j                                    (décalage de service induit chez j)
    Quand plus aucun client non routé ne peut être inséré faisablement
    (capacité/fenêtres de temps, vérifiées via _route_feasible) dans la
    route courante, celle-ci est fermée et une nouvelle route est ouverte
    avec un nouveau client graine, jusqu'à ce que tous les clients soient
    routés. Paramètres mu/lambda/alpha1/alpha2 : cf. CFG['I1_MU'] etc.
    """
    def d(a, b):
        return euclidean(_node_xy(a, instance), _node_xy(b, instance))
    unrouted = [c['id'] for c in instance['customers']]
    d0 = {cid: d(0, cid) for cid in unrouted}
    routes = []
    while unrouted:
        seed = max(unrouted, key=lambda c: d0[c])
        route = [seed]
        unrouted.remove(seed)
        while True:
            times_before = _route_service_starts(route, instance)
            nodes = [0] + route + [0]
            best_u, best_pos, best_c1, best_c2 = None, None, None, -float('inf')
            for u in unrouted:
                u_best_cost, u_best_pos = None, None
                cx, cy = _node_xy(u, instance)
                info_u = _node_info(u, instance)
                for pos in range(len(route) + 1):
                    i_id, j_id = nodes[pos], nodes[pos + 1]
                    candidate = route[:pos] + [u] + route[pos:]
                    if not _route_feasible(candidate, instance):
                        continue
                    dij = d(i_id, j_id)
                    c11 = d(i_id, u) + d(u, j_id) - mu * dij
                    c12 = 0.0
                    if alpha2 != 0.0 and j_id != 0:
                        if pos == 0:
                            depart_i, px, py = 0.0, instance['depot']['x'], instance['depot']['y']
                        else:
                            info_i = _node_info(i_id, instance)
                            depart_i = times_before[pos - 1] + info_i['service_time']
                            px, py   = _node_xy(i_id, instance)
                        arrival_u = depart_i + euclidean((px, py), (cx, cy))
                        b_u       = max(arrival_u, info_u['earliest'])
                        depart_u  = b_u + info_u['service_time']
                        jx, jy    = _node_xy(j_id, instance)
                        arrival_j = depart_u + euclidean((cx, cy), (jx, jy))
                        info_j    = _node_info(j_id, instance)
                        b_ju      = max(arrival_j, info_j['earliest'])
                        c12       = b_ju - times_before[pos]
                    c1 = alpha1 * c11 + alpha2 * c12
                    if u_best_cost is None or c1 < u_best_cost:
                        u_best_cost, u_best_pos = c1, pos
                if u_best_cost is None:
                    continue
                c2 = lam * d0[u] - u_best_cost
                if c2 > best_c2:
                    best_c2, best_u, best_pos, best_c1 = c2, u, u_best_pos, u_best_cost
            if best_u is None:
                break
            route.insert(best_pos, best_u)
            unrouted.remove(best_u)
        routes.append(route)
    return routes
def _lns_improve(solution, instance, n_iter=50, k=10, seed=0):
    rng       = np.random.RandomState(seed)
    current   = [list(r) for r in solution]
    best      = copy.deepcopy(current)
    best_cost = compute_cost(best, instance)
    for _ in range(n_iter):
        all_c   = [c for r in current for c in r]
        k_      = min(k, max(1, len(all_c) - 1))
        removed = list(rng.choice(all_c, k_, replace=False))
        removed_set = set(removed)
        partial = [[c for c in r if c not in removed_set] for r in current]
        partial = [r for r in partial if r]
        candidate = _repair_greedy(removed, partial, instance)
        c_cost    = compute_cost(candidate, instance)
        if c_cost < best_cost and is_feasible(candidate, instance):
            best, best_cost = copy.deepcopy(candidate), c_cost
        current = candidate
    return best
def get_initial_solution(instance, lns_iter=50, lns_k=10):
    sol = _build_initial_i1(
        instance,
        mu=CFG.get('I1_MU', 1.0), lam=CFG.get('I1_LAMBDA', 2.0),
        alpha1=CFG.get('I1_ALPHA1', 1.0), alpha2=CFG.get('I1_ALPHA2', 0.0),
    )
    sol = _lns_improve(sol, instance, n_iter=lns_iter, k=lns_k)
    sol = [r for r in sol if r]
    return sol, solution_stats(sol, instance)
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 3  —  ENVIRONNEMENT VRPTW  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class VRPTWEnv:
    def __init__(self, instance, initial_solution,
                 K=15, max_steps=20,
                 gurobi_tlimit=2.0, gurobi_gap=0.05,
                 vehicle_penalty=0.0,
                 action_set='full',
                 verbose=False):
        self.instance    = instance
        self.initial_sol = copy.deepcopy(initial_solution)
        self.K           = K
        self.max_steps   = max_steps
        self.tlimit      = gurobi_tlimit
        self.gap         = gurobi_gap
        self.vehicle_penalty = vehicle_penalty
        self.verbose     = verbose
        self.obs_dim     = 7
        # Jeu d'actions actif (ablation, voir ACTION_SETS_N / MODULE 11) :
        # 'full' (4 actions, défaut) / 'merged3' (3) / 'merged2' (2). Doit
        # correspondre exactement au n_actions du DQNAgent utilisé avec cet
        # environnement (voir train_on_instances/evaluate_frozen_on_instances).
        self.action_set  = action_set
        self.n_actions   = ACTION_SETS_N.get(action_set, 4)
        self._dist = {}
        depot = (instance['depot']['x'], instance['depot']['y'])
        for c in instance['customers']:
            d = euclidean(depot, (c['x'], c['y']))
            self._dist[(0, c['id'])] = d
            self._dist[(c['id'], 0)] = d
        for ci in instance['customers']:
            for cj in instance['customers']:
                self._dist[(ci['id'], cj['id'])] = euclidean(
                    (ci['x'], ci['y']), (cj['x'], cj['y']))
        self.reset()
    def _d(self, a, b):
        return self._dist.get((a, b), 0.0)
    def _info(self, nid):
        return _node_info(nid, self.instance)
    def reset(self, start_solution=None):
        if start_solution is not None:
            self.solution = copy.deepcopy(start_solution)
        else:
            self.solution = copy.deepcopy(self.initial_sol)
        self.current_cost     = compute_cost(self.solution, self.instance)
        self.best_cost        = self.current_cost
        self.initial_cost     = compute_cost(self.initial_sol, self.instance)
        self.step_count       = 0
        self.steps_no_improve = 0
        self._action_history  = []
        return self._obs()
    def _obs(self):
        stats = solution_stats(self.solution, self.instance)
        ic    = max(self.initial_cost, 1.0)
        return np.array([
            self.current_cost / ic,
            stats['n_vehicles'] / max(self.instance['n_vehicles'], 1),
            stats['avg_load_ratio'],
            stats['total_wait_time'] / max(ic * 0.5, 1.0),
            self.steps_no_improve / self.max_steps,
            self.best_cost / ic,
            1.0,
        ], dtype=np.float32)
    def _spatial_scores(self, pivot, all_c):
        px, py = self._info(pivot)['x'], self._info(pivot)['y']
        return {c: euclidean((px, py), (self._info(c)['x'], self._info(c)['y']))
                for c in all_c}
    def _temporal_scores(self, pivot, all_c):
        e_pivot = self._info(pivot)['earliest']
        return {c: abs(self._info(c)['earliest'] - e_pivot) for c in all_c}
    def _cost_scores(self):
        savings = {}
        for r in self.solution:
            if not r:
                continue
            nodes = [0] + r + [0]
            for i, cid in enumerate(r):
                prev, nxt    = nodes[i], nodes[i + 2]
                savings[cid] = (self._d(prev, cid) + self._d(cid, nxt)
                                 - self._d(prev, nxt))
        return savings
    def _spatiotemporal_destroy(self, rng, all_c, K):
        """
        Action FUSIONNÉE (ablation, jeux 'merged3'/'merged2', voir MODULE 11) :
        combine la proximité spatiale ET temporelle par rapport à un même
        pivot. Chaque critère est normalisé sur [0,1] (divisé par son max sur
        les clients courants) puis moyenné à parts égales (0.5/0.5), pour
        qu'aucune des deux échelles ne domine l'autre.
        """
        pivot = int(rng.choice(all_c))
        sp = self._spatial_scores(pivot, all_c)
        tp = self._temporal_scores(pivot, all_c)
        sp_max = max(sp.values()) or 1.0
        tp_max = max(tp.values()) or 1.0
        combined = {c: 0.5 * (sp[c] / sp_max) + 0.5 * (tp[c] / tp_max) for c in all_c}
        return sorted(all_c, key=lambda c: combined[c])[:K]
    def _destroy(self, action):
        all_c = [c for r in self.solution for c in r]
        if not all_c:
            return [], copy.deepcopy(self.solution)
        K   = min(self.K, len(all_c))
        rng = np.random.RandomState(self.step_count + action * 100)
        if self.action_set == 'full':
            # 4 actions d'origine : spatial, temporel, coût élevé, aléatoire
            if action == 0:
                pivot     = int(rng.choice(all_c))
                scores    = self._spatial_scores(pivot, all_c)
                destroyed = sorted(all_c, key=lambda c: scores[c])[:K]
            elif action == 1:
                pivot     = int(rng.choice(all_c))
                scores    = self._temporal_scores(pivot, all_c)
                destroyed = sorted(all_c, key=lambda c: scores[c])[:K]
            elif action == 2:
                savings   = self._cost_scores()
                destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
            else:
                idx       = rng.choice(len(all_c), K, replace=False)
                destroyed = [all_c[i] for i in idx]
        elif self.action_set == 'merged3':
            # 3 actions : spatio-temporel fusionné, coût élevé, aléatoire
            if action == 0:
                destroyed = self._spatiotemporal_destroy(rng, all_c, K)
            elif action == 1:
                savings   = self._cost_scores()
                destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
            else:
                idx       = rng.choice(len(all_c), K, replace=False)
                destroyed = [all_c[i] for i in idx]
        elif self.action_set == 'merged2':
            # 2 actions : spatio-temporel fusionné, aléatoire
            if action == 0:
                destroyed = self._spatiotemporal_destroy(rng, all_c, K)
            else:
                idx       = rng.choice(len(all_c), K, replace=False)
                destroyed = [all_c[i] for i in idx]
        else:
            raise ValueError(f"CFG['ACTION_SET'] inconnu : {self.action_set!r} "
                              f"(attendu 'full' / 'merged3' / 'merged2')")
        destroyed_set = set(destroyed)
        partial = [[c for c in r if c not in destroyed_set]
                   for r in self.solution]
        partial = [r for r in partial if r]
        return list(destroyed), partial
    def _repair_gurobi(self, destroyed, partial):
        if not destroyed:
            return copy.deepcopy(self.solution)
        inst    = self.instance
        depot   = inst['depot']
        horizon = inst['horizon']
        cap     = inst['capacity']
        nodes   = [0] + destroyed
        N       = len(nodes)
        V       = min(len(destroyed), inst['n_vehicles'])
        env, m = None, None
        try:
            env = gp.Env(empty=True)
            env.setParam('OutputFlag', 0)
            env.start()
            m = gp.Model(env=env)
            if self.tlimit is not None:
                m.setParam('TimeLimit', self.tlimit)
            m.setParam('MIPGap',     self.gap)
            m.setParam('OutputFlag', 0)
            x = m.addVars(N, N, V, vtype=GRB.BINARY, name='x')
            t = m.addVars(N, V, lb=0, ub=horizon,    name='t')
            m.setObjective(
                gp.quicksum(
                    self._d(nodes[i], nodes[j]) * x[i, j, v]
                    for i in range(N) for j in range(N) for v in range(V)
                    if i != j
                )
                # Pénalité par véhicule ouvert (0.0 = comportement d'origine,
                # inchangé). Un coefficient > 0 décourage Gurobi d'ouvrir un
                # véhicule de plus que nécessaire lors du repair, ce qui
                # limite l'inflation du nombre de véhicules par rapport au
                # BKS observée dans la validation.
                + self.vehicle_penalty * gp.quicksum(
                    x[0, j, v] for j in range(1, N) for v in range(V)
                ),
                GRB.MINIMIZE
            )
            for i in range(1, N):
                m.addConstr(
                    gp.quicksum(x[i, j, v]
                                for j in range(N) for v in range(V)
                                if i != j) == 1
                )
            for v in range(V):
                for i in range(1, N):
                    m.addConstr(
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j) ==
                        gp.quicksum(x[j, i, v] for j in range(N) if i != j)
                    )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(x[0, j, v] for j in range(1, N)) <= 1
                )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(
                        inst['customers'][nodes[i] - 1]['demand'] *
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j)
                        for i in range(1, N)
                    ) <= cap
                )
            M = horizon + 1000
            for v in range(V):
                m.addConstr(t[0, v] == 0)
                for i in range(N):
                    ni  = nodes[i]
                    ei  = depot['earliest'] if ni == 0 else inst['customers'][ni-1]['earliest']
                    li  = depot['latest']   if ni == 0 else inst['customers'][ni-1]['latest']
                    sti = depot['service_time'] if ni == 0 else inst['customers'][ni-1]['service_time']
                    m.addConstr(t[i, v] >= ei)
                    m.addConstr(t[i, v] <= li)
                    for j in range(N):
                        if i == j:
                            continue
                        m.addConstr(
                            t[j, v] >= t[i, v] + sti + self._d(ni, nodes[j])
                            - M * (1 - x[i, j, v])
                        )
            m.optimize()
            if m.SolCount > 0:
                eps        = 0.5
                new_routes = []
                for v in range(V):
                    if sum(x[0, j, v].X for j in range(1, N)) < eps:
                        continue
                    route_v, cur = [], 0
                    for _ in range(N):
                        nxt = next((j for j in range(N)
                                    if j != cur and x[cur, j, v].X > eps), None)
                        if nxt is None or nxt == 0:
                            break
                        route_v.append(nodes[nxt])
                        cur = nxt
                    if route_v:
                        new_routes.append(route_v)
                full = [r for r in list(partial) + new_routes if r]
                return full
            else:
                raise ValueError("Gurobi : aucune solution trouvée")
        except Exception as e:
            if self.verbose:
                print(f"    [Gurobi→Greedy fallback] {e}")
            return _repair_greedy(destroyed, partial, self.instance)
        finally:
            if m is not None:
                m.dispose()
            if env is not None:
                env.dispose()
    def step(self, action):
        old_cost            = self.current_cost
        destroyed, partial  = self._destroy(action)
        new_sol             = self._repair_gurobi(destroyed, partial)
        new_cost            = compute_cost(new_sol, self.instance)
        feasible            = is_feasible(new_sol, self.instance)
        if feasible:
            self.solution     = new_sol
            self.current_cost = new_cost
        else:
            new_cost = old_cost * 1.001
        improvement = old_cost - self.current_cost
        if improvement > 0:
            self.best_cost        = min(self.best_cost, self.current_cost)
            self.steps_no_improve = 0
        else:
            self.steps_no_improve += 1
        reward = improvement / max(self.initial_cost, 1.0)
        self.step_count += 1
        self._action_history.append(action)
        done = (self.step_count >= self.max_steps)
        info = {
            'cost':          self.current_cost,
            'best_cost':     self.best_cost,
            'action_taken':  action,
            'action_name':   ACTION_NAMES_BY_SET.get(self.action_set, ACTION_NAMES).get(action, str(action)),
            'improvement':   improvement,
            'destroyed_k':   len(destroyed),
            'destroyed_ids': [int(d) for d in destroyed[:5]],
            'feasible':      feasible,
        }
        return self._obs(), float(reward), done, info
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 4  —  AGENT DQN  (inchangé)
# ══════════════════════════════════════════════════════════════════════════════
class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((
            np.array(s,  dtype=np.float32), int(a), float(r),
            np.array(s2, dtype=np.float32), float(done),
        ))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.FloatTensor(np.stack(s)), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(np.stack(s2)),
                torch.FloatTensor(d))
    def __len__(self):
        return len(self.buf)
class DQNNet(nn.Module):
    def __init__(self, obs_dim=7, n_actions=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim,  hidden),  nn.ReLU(),
            nn.Linear(hidden,   hidden),  nn.ReLU(),
            nn.Linear(hidden,   n_actions),
        )
    def forward(self, x):
        return self.net(x)
class DQNAgent:
    def __init__(self, obs_dim=7, n_actions=4, hidden=64,
                 lr=1e-3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_size=2000, batch_size=32,
                 target_update_freq=10, seed=0):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        self.n_actions          = n_actions
        self.gamma              = gamma
        self.epsilon            = epsilon_start
        self.epsilon_end        = epsilon_end
        self.epsilon_decay      = epsilon_decay
        self.batch_size         = batch_size
        self.target_update_freq = target_update_freq
        self.step_count         = 0
        self.q_net      = DQNNet(obs_dim, n_actions, hidden)
        self.target_net = DQNNet(obs_dim, n_actions, hidden)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)
        self.loss_fn   = nn.MSELoss()
    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state).unsqueeze(0))
            return int(q.argmax().item())
    def store(self, s, a, r, s2, done):
        self.buffer.push(s, a, r, s2, done)
    def learn(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        s, a, r, s2, done = self.buffer.sample(self.batch_size)
        q_vals = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q  = self.target_net(s2).max(1)[0]
            targets = r + self.gamma * next_q * (1 - done)
        loss = self.loss_fn(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.optimizer.step()
        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.update_target()
        return float(loss.item())
    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    def save(self, path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        torch.save({'q_net': self.q_net.state_dict(),
                    'target_net': self.target_net.state_dict(),
                    'epsilon': self.epsilon, 'step_count': self.step_count}, path)
        print(f"    [DQN] Modèle sauvegardé → {path}")
    def load(self, path):
        """
        Recharge un checkpoint sauvegardé par save() (mêmes clés). Utilisé
        uniquement si CFG['REUSE_EXISTING_SYNTH_MODEL']=True, pour réutiliser
        un modèle déjà entraîné sur du synthétique au lieu de ré-entraîner.
        """
        ckpt = torch.load(path, map_location='cpu')
        self.q_net.load_state_dict(ckpt['q_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.epsilon    = ckpt.get('epsilon', self.epsilon)
        self.step_count = ckpt.get('step_count', self.step_count)
        print(f"    [DQN] Modèle rechargé ← {path}")
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5  —  ENTRAÎNEMENT DU DQN  (ÉTAPE 1, sur les instances SYNTHÉTIQUES)
#  Stocke cost_trace (trajectoire de coût step par step) dans le log, utilisé
#  ensuite pour les courbes de convergence détaillées. Retourne aussi
#  initial_sols pour les tracés de routes avant/après.
# ══════════════════════════════════════════════════════════════════════════════
BANNER = "═" * 72
def _sec(title):
    print(f"\n{BANNER}\n  {title}\n{BANNER}")
def train_on_instances(instances, episodes_per_instance=10, save_model=True, verbose=True):
    """
    ENTRAÎNEMENT (les poids du DQN sont mis à jour ici via store()/learn()/
    decay_epsilon()). Générique : fonctionne sur n'importe quelle liste
    d'instances au format standard — dans ce script, appelé avec les
    instances SYNTHÉTIQUES produites par generate_synthetic_training_set(),
    JAMAIS avec les instances Solomon (réservées à evaluate_frozen_
    on_instances(), plus bas, pour le TEST).
    save_model=False et verbose=False sont utilisés par le plan Taguchi
    (MODULE 10bis, run interne allégé : pas besoin d'écraser le modèle
    final ni d'un log console détaillé à chaque combinaison testée).
    Retourne : agent, training_log, initial_costs, best_costs, best_solutions,
               instance_times, initial_sols
    """
    n = len(instances)
    if verbose:
        _sec(f"ÉTAPE 1 — LNS initial pour les {n} instances d'entraînement (synthétiques)")
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        if verbose:
            print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
                  f"| véhicules={stats['n_vehicles']}")
    if verbose:
        _sec("ÉTAPE 1 — Entraînement DQN sur le synthétique")
    # Jeu d'actions actif (ablation, voir ACTION_SETS_N / CFG['ACTION_SET']) :
    # 'full' (4, défaut) / 'merged3' (3) / 'merged2' (2). n_actions du DQN et
    # de l'environnement DOIVENT correspondre exactement.
    action_set = CFG.get('ACTION_SET', 'full')
    n_actions_active = ACTION_SETS_N.get(action_set, 4)
    if verbose:
        print(f"  Jeu d'actions actif : '{action_set}' ({n_actions_active} actions)")
    agent = DQNAgent(
        obs_dim=7, n_actions=n_actions_active, hidden=CFG['HIDDEN'], lr=CFG['LR'],
        gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
        epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
        buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
        target_update_freq=CFG['TARGET_UPDATE'],
    )
    training_log   = []
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       action_set=action_set,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        if verbose:
            print(f"\n  {'═'*50}\n  [TRAIN] INSTANCE {inst['name']}  type={inst['type']}"
                  f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)
                next_obs, reward, done, info = env.step(action)
                agent.store(obs, action, reward, next_obs, float(done))
                agent.learn()
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            agent.decay_epsilon()
            training_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            if verbose:
                imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
                print(f"  [TRAIN] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                      f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        if verbose:
            print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    if save_model:
        model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
        agent.save(model_path)
    return agent, training_log, initial_costs, best_costs, best_solutions, instance_times, initial_sols
def evaluate_frozen_on_instances(instances, agent, episodes_per_instance=3):
    """
    TEST — ÉVALUATION GELÉE : l'agent DQN déjà entraîné (ÉTAPE 1, sur le
    synthétique) est ici appliqué en pur mode INFÉRENCE sur les instances
    passées en argument (les 56 Solomon officielles). Différences avec
    train_on_instances() :
      - AUCUN appel à agent.store() / agent.learn() / agent.decay_epsilon() :
        les poids du réseau ne sont JAMAIS mis à jour ici.
      - epsilon est temporairement forcé à 0.0 (politique gloutonne, pure
        exploitation du Q-réseau appris), puis restauré à sa valeur
        d'origine à la fin.
      - Le modèle n'est PAS sauvegardé ici (rien n'a changé dans ses poids).
    Le log retourné (test_log) a exactement la même forme que training_log
    (y compris 'cost_trace'), pour pouvoir réutiliser telles quelles les
    fonctions de tracé (plot_instance_convergence_examples, etc.) sur le
    TEST plutôt que sur l'entraînement.
    Retourne : initial_costs, best_costs, best_solutions, instance_times,
               initial_sols, test_log
    """
    n = len(instances)
    _sec(f"ÉTAPE 2 — ÉVALUATION GELÉE (TEST) sur {n} instances Solomon "
         f"— aucun apprentissage, epsilon=0")
    # Jeu d'actions actif (ablation, voir ACTION_SETS_N / CFG['ACTION_SET']) —
    # DOIT être le même que celui utilisé pour entraîner `agent` (train_
    # on_instances), sinon les indices d'action ne correspondent plus aux
    # mêmes opérateurs de destruction dans l'environnement de test.
    action_set = CFG.get('ACTION_SET', 'full')
    print(f"  Jeu d'actions actif : '{action_set}' ({ACTION_SETS_N.get(action_set, 4)} actions)")
    saved_epsilon = agent.epsilon
    agent.epsilon = 0.0
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    test_log       = []
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       action_set=action_set,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TEST] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)          # inférence pure (epsilon=0)
                next_obs, reward, done, info = env.step(action)
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            test_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TEST] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    agent.epsilon = saved_epsilon
    return initial_costs, best_costs, best_solutions, instance_times, initial_sols, test_log
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5bis  —  FUSION DE ROUTES (post-traitement, APRÈS DQN+Gurobi)
# ══════════════════════════════════════════════════════════════════════════════
# Étape de raffinement purement géométrique/temporelle appliquée UNE FOIS,
# après que le DQN+Gurobi (Situation 1) a produit sa meilleure solution sur
# chaque instance de TEST — jamais pendant l'entraînement, jamais vue par le
# DQN. Objectif : réduire l'inflation du nombre de véhicules par rapport au
# BKS observée dans la validation (ex. C101 : 11 véhicules utilisés vs 10 au
# BKS), en cherchant si deux routes existantes peuvent être fusionnées en
# une seule route unique, sans dégrader la solution.
#
# Principe (glouton, "best-improvement" à chaque itération) : pour chaque
# paire de routes (i, j) de la solution courante, on teste si la concaténer
# en une seule route est FAISABLE (capacité totale <= capacité véhicule,
# fenêtres de temps respectées sur toute la route résultante — vérifié via
# _route_feasible, donc EXACTEMENT le même critère de faisabilité que
# partout ailleurs dans le script) et si le coût total du couple de routes
# DIMINUE après fusion (une seule route parcourt souvent moins de distance
# que deux routes séparées repassant chacune par le dépôt). Les deux ordres
# de concaténation (i puis j, ou j puis i) sont testés, car une route VRPTW
# est orientée (ordre de visite des clients) : on garde le meilleur des deux
# s'il est faisable. Parmi toutes les paires testées à une itération, on
# applique la fusion qui réduit le plus le coût total (best-improvement),
# puis on recommence sur la solution mise à jour, jusqu'à ce qu'aucune
# fusion bénéfique ne soit plus trouvée.
# Cette étape ne modifie JAMAIS l'affectation client → route décidée par le
# DQN à l'intérieur de chaque route (aucun client n'est déplacé d'une route
# à l'autre individuellement) : elle se contente de recoller deux routes
# entières bout à bout quand c'est faisable et rentable. C'est donc un
# raffinement local, complémentaire du DQN+LNS+Gurobi, pas un remplacement.
def _route_total_cost(route, instance):
    """Coût d'une seule route (aller-retour dépôt inclus) — réutilise
    compute_cost() en l'appliquant à une "solution" à une seule route."""
    return compute_cost([route], instance) if route else 0.0
def _try_merge_two_routes(route_a, route_b, instance):
    """
    Teste si route_a et route_b peuvent être fusionnées en une seule route
    faisable (capacité + fenêtres de temps), dans l'un ou l'autre ordre de
    concaténation (a puis b, ou b puis a). Retourne (route_fusionnée,
    coût_après_fusion) pour le MEILLEUR ordre faisable trouvé, ou
    (None, None) si aucune fusion n'est faisable (quel que soit l'ordre).
    Ne vérifie PAS ici si la fusion est "rentable" (coût plus bas que les
    deux routes séparées) — c'est à l'appelant (route_merge_postprocess) de
    comparer au coût avant fusion et de décider d'accepter ou non.
    """
    cap = instance['capacity']
    load_a = sum(_node_info(c, instance)['demand'] for c in route_a)
    load_b = sum(_node_info(c, instance)['demand'] for c in route_b)
    if load_a + load_b > cap:
        return None, None
    best_route, best_cost = None, float('inf')
    for candidate in (route_a + route_b, route_b + route_a):
        if _route_feasible(candidate, instance):
            cost = _route_total_cost(candidate, instance)
            if cost < best_cost:
                best_route, best_cost = candidate, cost
    if best_route is None:
        return None, None
    return best_route, best_cost
def route_merge_postprocess(solution, instance, verbose=False):
    """
    Applique la fusion de routes (voir docstring du module ci-dessus) à une
    solution complète, jusqu'à convergence (plus aucune fusion bénéfique
    trouvée). Retourne (nouvelle_solution, stats) où stats contient le
    nombre de fusions appliquées, le nombre de véhicules avant/après, et le
    coût total avant/après.
    """
    routes = [list(r) for r in solution if r]
    cost_before  = compute_cost(routes, instance)
    n_veh_before = len(routes)
    n_merges = 0
    improved = True
    while improved and len(routes) > 1:
        improved = False
        best_pair, best_merged_route, best_delta = None, None, -1e-6
        n = len(routes)
        for i in range(n):
            for j in range(i + 1, n):
                cost_before_pair = (_route_total_cost(routes[i], instance)
                                     + _route_total_cost(routes[j], instance))
                merged_route, cost_after_pair = _try_merge_two_routes(routes[i], routes[j], instance)
                if merged_route is None:
                    continue
                delta = cost_after_pair - cost_before_pair   # négatif = amélioration
                if delta < best_delta:
                    best_delta = delta
                    best_pair  = (i, j)
                    best_merged_route = merged_route
        if best_pair is not None:
            i, j = best_pair
            routes = [r for k, r in enumerate(routes) if k not in (i, j)] + [best_merged_route]
            n_merges += 1
            improved = True
            if verbose:
                print(f"    [Fusion routes] paire ({i},{j}) fusionnée "
                      f"(Δcoût={best_delta:+.2f}) → {len(routes)} véhicules restants")
    cost_after  = compute_cost(routes, instance)
    n_veh_after = len(routes)
    stats = {
        'n_merges':      n_merges,
        'n_veh_before':  n_veh_before,
        'n_veh_after':   n_veh_after,
        'cost_before':   round(cost_before, 2),
        'cost_after':    round(cost_after, 2),
    }
    return routes, stats
def route_merge_postprocess_batch(instances, solutions, verbose=True):
    """
    Applique route_merge_postprocess() à une liste de solutions (une par
    instance — typiquement best_solutions retourné par
    evaluate_frozen_on_instances() sur les 56 Solomon). Affiche un résumé
    par instance + un total agrégé (véhicules économisés, coût total avant/
    après). Retourne (nouvelles_solutions, nouveaux_couts, stats_par_instance).
    """
    _sec("FUSION DE ROUTES — POST-TRAITEMENT (APRÈS DQN+GUROBI, SUR LE TEST)")
    new_solutions, new_costs, all_stats = [], [], []
    total_veh_saved = 0
    for inst, sol in zip(instances, solutions):
        merged_sol, stats = route_merge_postprocess(sol, inst, verbose=False)
        new_solutions.append(merged_sol)
        new_costs.append(stats['cost_after'])
        all_stats.append(stats)
        veh_saved = stats['n_veh_before'] - stats['n_veh_after']
        total_veh_saved += veh_saved
        if verbose:
            tag = f"  {inst['name']:<10} | fusions={stats['n_merges']} | " \
                  f"véhicules {stats['n_veh_before']}→{stats['n_veh_after']} " \
                  f"| coût {stats['cost_before']:.2f}→{stats['cost_after']:.2f}"
            print(tag + ("  ⚠ coût augmenté (ne devrait pas arriver)"
                         if stats['cost_after'] > stats['cost_before'] + 1e-6 else ""))
    if verbose:
        print(f"\n  → Total véhicules économisés par fusion : {total_veh_saved} "
              f"sur {len(instances)} instances")
    return new_solutions, new_costs, all_stats
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6  —  TABLEAU FINAL : SITUATION 1 vs BKS
#  (la Situation 2 "Gurobi seul sur l'instance complète" a été retirée)
# ══════════════════════════════════════════════════════════════════════════════
def build_bks_comparison_table(instances, initial_costs, best_costs, best_solutions,
                                instance_times, csv_path=None, latex_path=None):
    """
    Tableau final : Situation 1 (LNS+DQN+Gurobi) vs BKS, avec gap % et le
    nombre de véhicules utilisés pour chaque instance.
    """
    _sec("TABLEAU FINAL — SITUATION 1 vs BKS")
    rows = []
    for i, inst in enumerate(instances):
        name     = inst['name']
        bks      = BKS_REFERENCE.get(name)
        bks_dist = bks['distance'] if bks else None
        n_veh_s1 = sum(1 for r in best_solutions[i] if r)
        row = {
            'instance':             name,
            'type':                 inst['type'],
            'bks_vehicules':        bks['vehicles'] if bks else None,
            'bks_distance':         bks_dist,
            'n_vehicules_situation1': n_veh_s1,
            'distance_situation1':  round(best_costs[i], 2),
            'temps_s_situation1':   round(instance_times[i], 2),
            'gap_%_situation1_bks': round((best_costs[i] - bks_dist) / bks_dist * 100, 2) if bks_dist else None,
        }
        rows.append(row)
    print(f"\n  {'Instance':<9} {'BKS Vhc':>8} {'BKS Dist':>10} {'Vhc S1':>7} "
          f"{'Dist. S1':>10} {'Temps S1':>10} {'Gap %':>8}")
    print("  " + "─" * 70)
    for r in rows:
        bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "—"
        bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "—"
        g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {bks_v:>8} {bks_d:>10} "
              f"{r['n_vehicules_situation1']:>7} {r['distance_situation1']:>10.2f} "
              f"{r['temps_s_situation1']:>10.1f} {g1:>8}")
    # ── Gap moyen vs BKS (Situation 1) ──────────────────────────────────
    valid_gaps = [r['gap_%_situation1_bks'] for r in rows if r['gap_%_situation1_bks'] is not None]
    gap_moyen  = float(np.mean(valid_gaps)) if valid_gaps else None
    if gap_moyen is not None:
        print(f"\n  Gap moyen vs BKS (Situation 1) : {gap_moyen:+.2f}%  "
              f"({len(valid_gaps)}/{len(rows)} instances avec BKS connu)")
        print(f"  Gap min : {min(valid_gaps):+.2f}%  |  Gap max : {max(valid_gaps):+.2f}%")
    else:
        print("\n  Gap moyen vs BKS : indisponible (aucun BKS connu pour ces instances)")
    # ── Tableau récapitulatif simple : Instance / Nb véhicules / Distance / Gap % ──
    # (une ligne "Situation 1" + une ligne "BKS" par instance, format demandé)
    print(f"\n  Récapitulatif Instance / Nb véhicules / Distance / Gap % vs BKS")
    print(f"  {'Instance':<9} {'Situation':<26} {'Nb véhicules':>13} {'Distance':>10} {'Gap %':>8}")
    print("  " + "─" * 71)
    recap_rows = []
    for r in rows:
        g1 = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {'Situation 1 (DQN+LNS+Gurobi)':<26} "
              f"{r['n_vehicules_situation1']:>13} {r['distance_situation1']:>10.2f} {g1:>8}")
        recap_rows.append({
            'instance':    r['instance'],
            'situation':   'Situation 1 (DQN+LNS+Gurobi)',
            'n_vehicules': r['n_vehicules_situation1'],
            'distance':    r['distance_situation1'],
            'gap_%_vs_bks': r['gap_%_situation1_bks'],
        })
        if r['bks_distance'] is not None:
            print(f"  {r['instance']:<9} {'BKS (référence)':<26} "
                  f"{(r['bks_vehicules'] if r['bks_vehicules'] is not None else '—'):>13} "
                  f"{r['bks_distance']:>10.2f} {'0.00':>8}")
            recap_rows.append({
                'instance':    r['instance'],
                'situation':   'BKS (référence)',
                'n_vehicules': r['bks_vehicules'],
                'distance':    r['bks_distance'],
                'gap_%_vs_bks': 0.0,
            })
    if gap_moyen is not None:
        print("  " + "─" * 71)
        print(f"  {'':<9} {'MOYENNE (Situation 1)':<26} {'':>13} {'':>10} {gap_moyen:>+8.2f}")
    if csv_path:
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
        print(f"\n  Tableau final exporté (CSV) → {csv_path}")
        recap_csv_path = csv_path.replace('.csv', '_recap_vehicules_distance.csv')
        if recap_csv_path == csv_path:
            recap_csv_path = csv_path + '.recap_vehicules_distance.csv'
        with open(recap_csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=['instance', 'situation', 'n_vehicules', 'distance', 'gap_%_vs_bks'])
            w.writeheader(); w.writerows(recap_rows)
            if gap_moyen is not None:
                w.writerow({'instance': '', 'situation': 'MOYENNE (Situation 1)',
                            'n_vehicules': '', 'distance': '', 'gap_%_vs_bks': round(gap_moyen, 2)})
        print(f"  Récapitulatif Instance/Véhicules/Distance/Gap exporté (CSV) → {recap_csv_path}")
    if latex_path:
        lines = [
            "% Tableau généré automatiquement — Situation 1 vs BKS",
            "\\begin{table}[htbp]", "  \\centering",
            "  \\caption{Comparaison par instance Solomon officielle entre la méthode "
            "hybride (LNS + DQN + Gurobi repair) et le Best Known Solution (BKS) : "
            "nombre de véhicules, distance totale et écart au BKS.}",
            "  \\label{tab:comparaison-bks}",
            "  \\begin{tabular}{lrrrrrr}", "    \\toprule",
            "    Instance & BKS Vhc & BKS Dist. & Vhc. S1 & Dist. S1 & "
            "Temps S1 (s) & Gap S1 (\\%) \\\\",
            "    \\midrule",
        ]
        for r in rows:
            bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "--"
            bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "--"
            g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "--"
            lines.append(f"    {r['instance']} & {bks_v} & {bks_d} & "
                        f"{r['n_vehicules_situation1']} & {r['distance_situation1']:.2f} & "
                        f"{r['temps_s_situation1']:.1f} & {g1} \\\\")
        lines += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
        with open(latex_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        print(f"  Tableau final exporté (LaTeX) → {latex_path}")
    return rows
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6bis  —  DIAGNOSTIC : LICENCE GUROBI À TAILLE LIMITÉE ?
# ══════════════════════════════════════════════════════════════════════════════
def check_gurobi_license(min_vars_needed=2500):
    """
    Test rapide, effectué AVANT tout calcul, pour savoir si la licence
    Gurobi active est une licence "size-limited" (Community/gratuite,
    plafonnée à ~2000 variables ET ~2000 contraintes) plutôt qu'une licence
    complète (académique ou commerciale, sans limite de taille).

    Pourquoi c'est important ici : le repair Gurobi de la Situation 1
    (_repair_gurobi, appelé à CHAQUE step du LNS) crée jusqu'à
    (K_DESTROY+1)² * V variables binaires — selon K_DESTROY (cf. CFG) et le
    nombre de véhicules de l'instance, cela peut dépasser 2000. Un fallback
    vers le repair glouton est déjà en place dans _repair_gurobi (try/
    except), donc l'exécution ne plantera pas, mais peut silencieusement
    retomber sur le glouton plus souvent que prévu si la licence est limitée.

    Ce test ne bloque JAMAIS l'exécution : il informe seulement, une seule
    fois, en tête de run.
    """
    env, m = None, None
    try:
        env = gp.Env(empty=True)
        env.setParam('OutputFlag', 0)
        env.start()
        m = gp.Model(env=env)
        m.setParam('OutputFlag', 0)
        x = m.addVars(min_vars_needed, vtype=GRB.BINARY)
        m.setObjective(gp.quicksum(x[i] for i in range(min_vars_needed)), GRB.MAXIMIZE)
        m.optimize()
        print(f"  ✓ Licence Gurobi : aucune restriction de taille détectée "
              f"(modèle-test à {min_vars_needed} variables résolu sans erreur).")
        return True
    except gp.GurobiError as e:
        errno = getattr(e, 'errno', None)
        is_size_limit = (errno == getattr(GRB.Error, 'SIZE_LIMIT_EXCEEDED', object()))
        if is_size_limit:
            k_repair_vars = (CFG['K_DESTROY'] + 1) ** 2 * CFG['K_DESTROY']
            print(f"\n  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée "
                  f"(erreur {errno} : {e})\n"
                  f"    → Licence Community/gratuite, plafonnée à ~2000 variables "
                  f"et ~2000 contraintes.\n"
                  f"    → Le repair Gurobi de la Situation 1 (jusqu'à "
                  f"~{k_repair_vars} variables selon K_DESTROY={CFG['K_DESTROY']} et "
                  f"le nombre de véhicules) peut dépasser cette limite sur certaines "
                  f"instances — un repli automatique vers le repair glouton est déjà "
                  f"en place, donc pas de plantage, mais un usage de Gurobi moins "
                  f"fréquent que prévu.\n"
                  f"    → Pour lever cette limite : une licence académique Gurobi "
                  f"est gratuite et sans limite de taille pour les universités "
                  f"(https://www.gurobi.com/academia/) — utile si vous êtes "
                  f"affiliée à ENSI-UMA.\n")
        else:
            print(f"  ⚠ Erreur Gurobi lors du test de licence (non liée à la taille) : {e}")
        return False
    except Exception as e:
        print(f"  ⚠ Impossible de tester la licence Gurobi ({type(e).__name__}: {e}) "
              f"— test ignoré, l'exécution continue normalement.")
        return None
    finally:
        if m is not None:
            m.dispose()
        if env is not None:
            env.dispose()
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 7  —  COURBES DE CONVERGENCE
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_mean(vals, window=20):
    out = []
    for i in range(len(vals)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(vals[lo:i + 1])))
    return out
def plot_global_convergence(training_log, instances, initial_costs, best_costs, outdir):
    """
    Figure globale (2 panneaux) :
      (a) reward par épisode global + moyenne mobile → convergence de
          l'apprentissage DQN
      (b) amélioration % par instance (une barre par instance Solomon) vs
          coût LNS initial
    """
    episodes = [e['episode_global'] for e in training_log]
    rewards  = [e['total_reward']   for e in training_log]
    reward_smooth = _rolling_mean(rewards, window=20)
    improvements  = [(initial_costs[i] - best_costs[i]) / initial_costs[i] * 100
                      for i in range(len(initial_costs))]
    names = [inst['name'] for inst in instances]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    axes[0].plot(episodes, rewards, color='lightgray', linewidth=0.6, label='reward brut / épisode')
    axes[0].plot(episodes, reward_smooth, color='crimson', linewidth=2,
                 label='moyenne mobile (20 épisodes)')
    axes[0].set_xlabel('Épisode global')
    axes[0].set_ylabel('Reward de l\'épisode')
    axes[0].set_title("Convergence de l'apprentissage DQN")
    axes[0].legend(loc='best', fontsize=9)
    axes[0].grid(alpha=0.3)
    colors = ['#2ca02c' if v >= 0 else '#d62728' for v in improvements]
    axes[1].bar(range(len(improvements)), improvements, color=colors)
    axes[1].axhline(np.mean(improvements), color='navy', linestyle='--', linewidth=1.5,
                     label=f"Moyenne = {np.mean(improvements):.2f}%")
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names, rotation=90, fontsize=7)
    axes[1].set_xlabel('Instance')
    axes[1].set_ylabel('Amélioration vs solution LNS initiale (%)')
    axes[1].set_title("Amélioration du coût par instance après DQN+Gurobi")
    axes[1].legend(loc='best', fontsize=9)
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_globale.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence globale → {path}")
    return path
def plot_gap_vs_bks(bks_rows, outdir):
    """
    Barres du gap % vs BKS (Situation 1) pour chaque instance, avec la
    moyenne indiquée par une ligne horizontale. N'utilise que les instances
    pour lesquelles un BKS est connu (gap_%_situation1_bks non None).
    `bks_rows` = la liste `rows` retournée par build_bks_comparison_table().
    """
    data = [(r['instance'], r['gap_%_situation1_bks']) for r in bks_rows
            if r['gap_%_situation1_bks'] is not None]
    if not data:
        print("  [plot_gap_vs_bks] Aucune instance avec BKS connu — graphique ignoré.")
        return None
    names = [d[0] for d in data]
    gaps  = [d[1] for d in data]
    gap_moyen = float(np.mean(gaps))
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(names)), 5.5))
    colors = ['#2ca02c' if g <= 0 else '#d62728' for g in gaps]
    ax.bar(range(len(names)), gaps, color=colors)
    ax.axhline(gap_moyen, color='navy', linestyle='--', linewidth=1.5,
               label=f"Gap moyen = {gap_moyen:+.2f}%")
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_xlabel('Instance')
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_title(f"Écart au Best Known Solution par instance "
                 f"({len(names)} instances avec BKS connu)")
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique gap vs BKS → {path}")
    return path
def plot_instance_convergence_examples(training_log, instances, example_idxs, outdir):
    """
    Pour chaque instance exemple :
      - gauche : coût final / meilleur coût cumulé au fil des épisodes locaux
      - droite : trajectoire de coût step par step du dernier épisode local
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(13, 4.2 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        entries = [e for e in training_log if e['instance_id'] == idx]
        entries.sort(key=lambda e: e['ep_local'])
        ep_locals      = [e['ep_local']   for e in entries]
        best_costs_ep  = [e['best_cost']  for e in entries]
        final_costs_ep = [e['final_cost'] for e in entries]
        name = instances[idx]['name']
        axes[row, 0].plot(ep_locals, final_costs_ep, '-o', color='steelblue',
                           label='coût final de l\'épisode')
        axes[row, 0].plot(ep_locals, best_costs_ep, '-s', color='crimson',
                           label='meilleur coût cumulé')
        axes[row, 0].set_title(f"Instance {name} — convergence par épisode local")
        axes[row, 0].set_xlabel('Épisode local')
        axes[row, 0].set_ylabel('Coût de la tournée')
        axes[row, 0].legend(loc='best', fontsize=9)
        axes[row, 0].grid(alpha=0.3)
        last = entries[-1] if entries else None
        trace = last.get('cost_trace', []) if last else []
        if trace:
            axes[row, 1].plot(range(len(trace)), trace, '-o', color='darkorange')
            axes[row, 1].set_title(f"Instance {name} — trajectoire de coût "
                                    f"(dernier épisode local, {len(trace)-1} steps)")
            axes[row, 1].set_xlabel('Step')
            axes[row, 1].set_ylabel('Coût courant')
            axes[row, 1].grid(alpha=0.3)
        else:
            axes[row, 1].axis('off')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_instances_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence instances exemples → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 8  —  TRACÉS DES ROUTES (AVANT LNS / APRÈS DQN)
# ══════════════════════════════════════════════════════════════════════════════
def _plot_routes_on_ax(ax, instance, solution, title):
    depot = instance['depot']
    ax.scatter([depot['x']], [depot['y']], c='black', marker='s', s=110,
               zorder=5, label='Dépôt')
    n_routes = max(len([r for r in solution if r]), 1)
    try:
        cmap = matplotlib.colormaps.get_cmap('tab20').resampled(max(n_routes, 1))
    except AttributeError:
        cmap = plt.cm.get_cmap('tab20', max(n_routes, 1))
    ridx = 0
    for route in solution:
        if not route:
            continue
        xs = [depot['x']] + [instance['customers'][cid - 1]['x'] for cid in route] + [depot['x']]
        ys = [depot['y']] + [instance['customers'][cid - 1]['y'] for cid in route] + [depot['y']]
        ax.plot(xs, ys, '-o', color=cmap(ridx), markersize=4, linewidth=1.3)
        ridx += 1
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.3)
def plot_routes_examples(instances, initial_sols, best_solutions, example_idxs, outdir):
    """
    Pour chaque instance exemple : routes AVANT (solution LNS initiale, avant
    DQN) à gauche, routes APRÈS (meilleure solution trouvée par DQN+Gurobi)
    à droite. Le BKS (véhicules/distance) est indiqué en sous-titre s'il
    est connu pour l'instance.
    """
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.8 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        inst = instances[idx]
        name = inst['name']
        bks  = BKS_REFERENCE.get(name)
        bks_str = (f" | BKS: dist={bks['distance']:.1f}, vhc={bks['vehicles']}"
                   if bks else "")
        cost_before = compute_cost(initial_sols[idx], inst)
        cost_after  = compute_cost(best_solutions[idx], inst)
        nveh_before = sum(1 for r in initial_sols[idx] if r)
        nveh_after  = sum(1 for r in best_solutions[idx] if r)
        _plot_routes_on_ax(
            axes[row, 0], inst, initial_sols[idx],
            f"Instance {name} — AVANT (LNS)\n"
            f"dist={cost_before:.1f}  véhicules={nveh_before}{bks_str}"
        )
        _plot_routes_on_ax(
            axes[row, 1], inst, best_solutions[idx],
            f"Instance {name} — APRÈS (DQN+Gurobi)\n"
            f"dist={cost_after:.1f}  véhicules={nveh_after}{bks_str}"
        )
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_routes_avant_apres_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique routes avant/après → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 9  —  ANALYSES PAR GROUPE D'INSTANCES (C1/C2/R1/R2/RC1/RC2)
#  Appliqué uniquement aux résultats du TEST (gap vs BKS) — ne joue aucun
#  rôle dans le choix des hyperparamètres, purement diagnostique a posteriori.
# ══════════════════════════════════════════════════════════════════════════════
def get_instance_group(inst):
    """
    Déduit le groupe Solomon (C1/C2/R1/R2/RC1/RC2) à partir du type et du nom
    de l'instance. Ex. : type='C', name='C101' -> 'C1' ; type='RC', name=
    'RC201' -> 'RC2'. Le chiffre juste après le préfixe alphabétique indique
    le sous-groupe (1 = fenêtres serrées / faible capacité, 2 = fenêtres
    larges / grande capacité).
    """
    t = inst['type']
    name = inst['name']
    digit = name[len(t)] if len(name) > len(t) else ''
    return f"{t}{digit}"
def plot_gap_boxplot_by_group(bks_rows, instances, outdir):
    """
    Boxplot du gap % vs BKS regroupé par C1/C2/R1/R2/RC1/RC2, pour visualiser
    quels groupes d'instances posent le plus de difficulté à la méthode.
    """
    by_name = {inst['name']: inst for inst in instances}
    groups = defaultdict(list)
    for r in bks_rows:
        if r['gap_%_situation1_bks'] is None:
            continue
        inst = by_name.get(r['instance'])
        if inst is None:
            continue
        groups[get_instance_group(inst)].append(r['gap_%_situation1_bks'])
    labels = [g for g in GROUP_ORDER if g in groups]
    data   = [groups[g] for g in labels]
    if not data:
        print("  [plot_gap_boxplot_by_group] Aucune donnée — graphique ignoré.")
        return None
    fig, ax = plt.subplots(figsize=(8, 5.5))
    try:
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, showmeans=True)
    except TypeError:
        # matplotlib < 3.9 : le paramètre s'appelait encore 'labels'.
        bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#a6c8ff')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_xlabel('Groupe d\'instances Solomon')
    ax.set_title("Distribution du gap par groupe d'instances")
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_boxplot_par_groupe.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique boxplot gap par groupe → {path}")
    return path
def plot_vehicles_vs_bks(bks_rows, instances, outdir):
    """
    Scatter plot : nombre de véhicules utilisés (Situation 1) vs nombre de
    véhicules du BKS, par instance, coloré par groupe. La diagonale y=x
    marque l'égalité ; les points au-dessus utilisent plus de véhicules que
    le BKS — c'est le phénomène d'inflation identifié dans la validation.
    """
    by_name = {inst['name']: inst for inst in instances}
    cmap_colors = plt.cm.tab10.colors
    color_map = {g: cmap_colors[i % len(cmap_colors)] for i, g in enumerate(GROUP_ORDER)}
    fig, ax = plt.subplots(figsize=(7, 7))
    seen_groups = set()
    max_v = 1
    for r in bks_rows:
        if r['bks_vehicules'] is None or r['n_vehicules_situation1'] is None:
            continue
        inst = by_name.get(r['instance'])
        grp  = get_instance_group(inst) if inst else '?'
        c    = color_map.get(grp, 'gray')
        label = grp if grp not in seen_groups else None
        seen_groups.add(grp)
        ax.scatter(r['bks_vehicules'], r['n_vehicules_situation1'], color=c,
                   label=label, s=45, alpha=0.85, edgecolor='black', linewidth=0.4)
        max_v = max(max_v, r['bks_vehicules'], r['n_vehicules_situation1'])
    ax.plot([0, max_v + 1], [0, max_v + 1], '--', color='gray', linewidth=1, label='y = x (égalité)')
    ax.set_xlabel('Véhicules BKS')
    ax.set_ylabel('Véhicules Situation 1 (DQN+LNS+Gurobi)')
    ax.set_title('Nombre de véhicules utilisés vs BKS, par instance')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_vehicules_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique véhicules vs BKS → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 10  —  PLAN D'EXPÉRIENCE TAGUCHI (calibration des hyperparamètres)
#
#  Objectif : identifier quels hyperparamètres (LNS_ITER, K_DESTROY_RATIO,
#  GUROBI_TLIMIT, GUROBI_GAP, MAX_STEPS, EPS_DECAY, VEHICLE_PENALTY) ont le
#  plus d'effet sur la performance, et quelle combinaison la maximise, sans
#  tester les 3^7 = 2187 combinaisons possibles.
#
#  ⚠ POINT CRITIQUE (train/test disjoint) : ce module supporte DEUX modes,
#  sélectionnés par le paramètre `instances_override`/`metric` de
#  run_taguchi_doe() :
#    - mode SYNTHÉTIQUE (recommandé, piloté par CFG['TAGUCHI_SYNTH']) :
#      `instances_override` = des instances synthétiques générées à la
#      volée ; `metric='lns_improvement'` = % d'amélioration vs le coût LNS
#      initial (aucun BKS requis). Les 56 Solomon ne sont JAMAIS chargées
#      dans ce mode → zéro fuite d'hyperparamètres vers le jeu de test.
#    - mode SOLOMON historique (⚠ non recommandé, piloté par CFG['TAGUCHI']) :
#      `instances_override=None` → charge INSTANCE_FILES (les 56 Solomon)
#      comme matrice externe ; `metric='bks_gap'` = gap % vs BKS_REFERENCE.
#      Ce mode entraîne et évalue directement sur le jeu de test : la
#      combinaison qu'il retourne a "vu" Solomon et ne doit pas être
#      utilisée comme hyperparamètres d'une validation train/test disjointe
#      (elle resterait valable seulement comme référence de comparaison).
#
#  Principe commun (matrice orthogonale L9/L18) :
#    - Matrice interne  : la matrice orthogonale L18 ci-dessous (18 lignes,
#      7 facteurs, 3 niveaux) — les combinaisons de facteurs à tester.
#    - Matrice externe (bruit) : les instances sur lesquelles chaque
#      combinaison de la matrice interne est évaluée ; le ratio signal/bruit
#      (S/N) résume à la fois la performance moyenne ET la stabilité face à
#      la variabilité entre instances/groupes.
#    - Après le plan, reportez la meilleure combinaison dans CFG, repassez
#      ENABLE=False, et relancez le script pour la validation ENTRAÎNEMENT/
#      TEST DISJOINT normale (avec ces hyperparamètres).
# ══════════════════════════════════════════════════════════════════════════════
# Matrice orthogonale L9 (3^4) — 9 runs, jusqu'à 4 facteurs à 3 niveaux.
L9_ARRAY = [
    [1, 1, 1, 1], [1, 2, 2, 2], [1, 3, 3, 3],
    [2, 1, 2, 3], [2, 2, 3, 1], [2, 3, 1, 2],
    [3, 1, 3, 2], [3, 2, 1, 3], [3, 3, 2, 1],
]
# Matrice orthogonale L18 (2^1 x 3^7) — 18 runs. La 1ère colonne (2 niveaux)
# n'est pas utilisée ici (tous nos facteurs sont à 3 niveaux) ; seules les
# colonnes 2 à 8 (7 facteurs à 3 niveaux) servent au plan.
L18_ARRAY_FULL = [
    [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 2, 2, 2, 2, 2, 2], [1, 1, 3, 3, 3, 3, 3, 3],
    [1, 2, 1, 1, 2, 2, 3, 3], [1, 2, 2, 2, 3, 3, 1, 1], [1, 2, 3, 3, 1, 1, 2, 2],
    [1, 3, 1, 2, 1, 3, 2, 3], [1, 3, 2, 3, 2, 1, 3, 1], [1, 3, 3, 1, 3, 2, 1, 2],
    [2, 1, 1, 3, 3, 2, 2, 1], [2, 1, 2, 1, 1, 3, 3, 2], [2, 1, 3, 2, 2, 1, 1, 3],
    [2, 2, 1, 2, 3, 1, 3, 2], [2, 2, 2, 3, 1, 2, 1, 3], [2, 2, 3, 1, 2, 3, 2, 1],
    [2, 3, 1, 3, 2, 3, 1, 2], [2, 3, 2, 1, 3, 1, 2, 3], [2, 3, 3, 2, 1, 2, 3, 1],
]
L18_ARRAY = [row[1:] for row in L18_ARRAY_FULL]   # ne garder que les 7 colonnes à 3 niveaux
def check_orthogonal_array(array):
    """
    Vérifie la propriété d'orthogonalité : pour chaque paire de colonnes,
    chaque combinaison de niveaux doit apparaître le même nombre de fois.
    Retourne True/False — sert de garde-fou avant de lancer un plan
    potentiellement long sur une matrice mal formée.
    """
    arr = np.array(array)
    n_runs, n_cols = arr.shape
    ok = True
    for c1, c2 in itertools.combinations(range(n_cols), 2):
        counts = defaultdict(int)
        for row in arr:
            counts[(row[c1], row[c2])] += 1
        if len(set(counts.values())) > 1:
            ok = False
    return ok
def select_orthogonal_array(n_factors):
    if n_factors <= 4:
        return 'L9', [row[:n_factors] for row in L9_ARRAY]
    elif n_factors <= 7:
        return 'L18', [row[:n_factors] for row in L18_ARRAY]
    else:
        raise ValueError(
            f"{n_factors} facteurs demandés, mais seules les matrices L9 (4 "
            f"facteurs max) et L18 (7 facteurs max) sont disponibles. "
            f"Réduisez TAGUCHI_FACTORS ou étendez ce module avec une L27."
        )
@contextmanager
def _cfg_override(**kwargs):
    """Applique temporairement des valeurs de CFG (pour un run du plan
    Taguchi), puis restaure les valeurs d'origine à la sortie du bloc."""
    old = {k: CFG.get(k) for k in kwargs}
    CFG.update(kwargs)
    try:
        yield
    finally:
        CFG.update(old)
def taguchi_sn_ratio_stb(values):
    """
    Ratio signal/bruit 'plus petit est meilleur' (Smaller-The-Better),
    calculé sur une liste de valeurs (le gap %/l'amélioration % peut être
    négatif — la formule standard SN = -10*log10(mean(y_i^2)) reste valide,
    elle pénalise tout écart à zéro, positif ou négatif, ainsi que la
    variance entre les instances de la matrice externe).
    """
    y = np.array(values, dtype=float)
    msd = max(float(np.mean(y ** 2)), 1e-9)
    return -10.0 * np.log10(msd)
def taguchi_main_effects(results, factor_names, response_key):
    """Moyenne de la réponse (gap/amélioration ou S/N) par niveau, pour
    chaque facteur."""
    effects = {}
    for fn in factor_names:
        by_level = defaultdict(list)
        for r in results:
            if r.get(response_key) is not None:
                by_level[r[fn]].append(r[response_key])
        effects[fn] = {lvl: float(np.mean(vals)) for lvl, vals in sorted(by_level.items(), key=lambda kv: kv[0])}
    return effects
def taguchi_anova_contributions(results, factor_names, response_key):
    """
    ANOVA simplifiée (méthode des sommes de carrés à partir des moyennes par
    niveau) : contribution en % de chaque facteur à la variance totale de la
    réponse. Ne remplace pas une ANOVA complète avec terme d'erreur pur,
    mais donne une hiérarchie fiable de l'importance relative des facteurs.
    """
    valid = [r for r in results if r.get(response_key) is not None]
    if not valid:
        return {fn: 0.0 for fn in factor_names}
    y = np.array([r[response_key] for r in valid])
    grand_mean = y.mean()
    ss_total = float(np.sum((y - grand_mean) ** 2))
    contributions = {}
    for fn in factor_names:
        by_level = defaultdict(list)
        for r in valid:
            by_level[r[fn]].append(r[response_key])
        ss_factor = sum(len(vals) * (np.mean(vals) - grand_mean) ** 2 for vals in by_level.values())
        contributions[fn] = (ss_factor / ss_total * 100.0) if ss_total > 0 else 0.0
    return contributions
def plot_taguchi_main_effects(effects, factor_names, response_label, outdir, filename):
    """Graphique classique des effets principaux Taguchi : un panneau par
    facteur, moyenne de la réponse pour chaque niveau, reliée par une ligne."""
    n = len(factor_names)
    ncols = min(4, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)
    for i, fn in enumerate(factor_names):
        levels = list(effects[fn].keys())
        means  = list(effects[fn].values())
        axes[i].plot(range(len(levels)), means, '-o', color='steelblue')
        axes[i].set_xticks(range(len(levels)))
        axes[i].set_xticklabels([str(l) for l in levels], fontsize=8)
        axes[i].set_title(fn, fontsize=10)
        axes[i].set_ylabel(response_label, fontsize=8)
        axes[i].grid(alpha=0.3)
    for j in range(len(factor_names), len(axes)):
        axes[j].axis('off')
    fig.suptitle(f"Effets principaux — {response_label}", fontsize=13)
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique effets principaux ({response_label}) → {path}")
    return path
def plot_taguchi_sn_bars(results, outdir, filename='taguchi_sn_ratios.png'):
    """Ratio S/N obtenu pour chaque run du plan, run avec le meilleur S/N
    mis en évidence en vert."""
    valid = [r for r in results if r.get('sn_ratio') is not None]
    if not valid:
        print("  [plot_taguchi_sn_bars] Aucun résultat — graphique ignoré.")
        return None
    runs = [r['run'] for r in valid]
    sns_ = [r['sn_ratio'] for r in valid]
    best_idx = int(np.argmax(sns_))
    colors = ['#2ca02c' if i == best_idx else '#4c72b0' for i in range(len(sns_))]
    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(runs)), 5))
    ax.bar([str(r) for r in runs], sns_, color=colors)
    ax.set_xlabel('Run (combinaison de la matrice L9/L18)')
    ax.set_ylabel('Ratio S/N (plus grand = meilleur)')
    ax.set_title('Ratio signal/bruit par combinaison — plan Taguchi')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique S/N par run → {path}")
    return path
def plot_taguchi_pareto(anova_contributions, outdir, filename='taguchi_pareto_effets.png'):
    """Diagramme en barres horizontales de la contribution (%) de chaque
    facteur à la variance de la réponse — équivalent simplifié d'un Pareto ANOVA."""
    items = sorted(anova_contributions.items(), key=lambda kv: -kv[1])
    names = [k for k, v in items][::-1]
    vals  = [v for k, v in items][::-1]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(range(len(names)), vals, color='#4c72b0')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('Contribution à la variance de la réponse (%)')
    ax.set_title("Importance relative des facteurs (ANOVA simplifiée Taguchi)")
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    path = os.path.join(outdir, filename)
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique Pareto des effets → {path}")
    return path
def run_taguchi_doe(factors, outer_instance_names, episodes_per_instance_doe, outdir,
                     instances_override=None, metric='bks_gap', json_suffix=''):
    """
    Exécute le plan d'expérience Taguchi complet :
      1. Construit la matrice orthogonale adaptée au nombre de facteurs
      2. Pour chaque run, applique les niveaux de facteurs (via _cfg_override)
         et entraîne le pipeline complet (LNS+DQN+Gurobi) sur les instances
         de la matrice externe, avec un nombre d'épisodes réduit
      3. Calcule la métrique de classement par instance (voir `metric`), la
         moyenne, et le ratio S/N
      4. Analyse les effets principaux + ANOVA simplifiée, génère les graphes
      5. Rapporte la meilleure combinaison trouvée
    instances_override : si fourni (liste d'instances déjà construites),
        utilisé DIRECTEMENT comme matrice externe, SANS jamais charger les
        fichiers Solomon — c'est ce qui permet de calibrer sur des instances
        SYNTHÉTIQUES (voir CFG['TAGUCHI_SYNTH']) sans aucune fuite vers le
        jeu de test. Si None (mode historique Solomon), charge les instances
        via INSTANCE_FILES et `outer_instance_names`.
    metric : 'bks_gap' (gap % vs BKS_REFERENCE — ne fonctionne que pour des
        instances Solomon officielles connues dans BKS_REFERENCE) ou
        'lns_improvement' (% d'amélioration vs le coût LNS initial — aucun
        BKS requis, utilisable sur des instances synthétiques). Convention
        "plus petit est meilleur" identique dans les deux cas (négatif =
        amélioration), donc toute la mécanique Taguchi (S/N, effets
        principaux, ANOVA, combinaison prédite) fonctionne sans modification.
    json_suffix : suffixe inséré dans les noms des fichiers de sortie (CSV,
        JSON, PNG) — ex. '_synthetique' — pour ne jamais écraser les
        résultats d'un plan calibré sur d'autres instances/métrique.
    Retourne : results (liste de dicts, un par run), effects_gap, effects_sn,
               anova_contributions, best_run
    """
    factor_names  = [f[0] for f in factors]
    factor_levels = [f[1] for f in factors]
    n_factors     = len(factors)
    array_name, array = select_orthogonal_array(n_factors)
    ortho_ok = check_orthogonal_array(array)
    if instances_override is not None:
        outer_instances = list(instances_override)
    else:
        all_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
        if outer_instance_names == 'ALL':
            # Sentinel spécial : utiliser TOUTES les instances Solomon
            # chargées comme matrice externe. ⚠ Mode historique — voir
            # avertissement CFG['TAGUCHI'] plus haut : fuite d'hyperparamètres
            # vers le jeu de test. Multiplie aussi fortement le temps total
            # du plan (voir avertissement de durée ci-dessous).
            outer_instances = list(all_instances)
        else:
            by_name = {inst['name']: inst for inst in all_instances}
            outer_instances = [by_name[nm] for nm in outer_instance_names if nm in by_name]
    if not outer_instances:
        raise ValueError("Aucune instance de la matrice externe trouvée "
                          "(instances_override vide, ou outer_instance_names "
                          "absent de INSTANCE_FILES).")
    gurobi_tlimit_max = max(TAGUCHI_FACTORS[[f[0] for f in TAGUCHI_FACTORS].index('GUROBI_TLIMIT')][1]) \
        if 'GUROBI_TLIMIT' in factor_names else CFG['GUROBI_TLIMIT']
    max_steps_max = max(TAGUCHI_FACTORS[[f[0] for f in TAGUCHI_FACTORS].index('MAX_STEPS')][1]) \
        if 'MAX_STEPS' in factor_names else CFG['MAX_STEPS']
    worst_case_h = (len(array) * len(outer_instances) * gurobi_tlimit_max
                     * max_steps_max * episodes_per_instance_doe) / 3600.0
    _sec(f"PLAN D'EXPÉRIENCE TAGUCHI — {array_name} ({len(array)} runs, "
         f"{n_factors} facteurs, métrique={metric}, orthogonalité="
         f"{'OK' if ortho_ok else 'A VERIFIER !'})")
    if len(outer_instances) <= 12:
        print(f"  Matrice externe (bruit) : {[i['name'] for i in outer_instances]}")
    else:
        print(f"  Matrice externe (bruit) : {len(outer_instances)} instances "
              f"({outer_instances[0]['name']} ... {outer_instances[-1]['name']})")
    print(f"  Épisodes/instance pendant le plan : {episodes_per_instance_doe}")
    print(f"  ⚠ Temps pire cas estimé (si Gurobi atteint systématiquement sa limite de "
          f"temps) : ~{worst_case_h:.1f}h ({worst_case_h/24:.1f} jours). "
          f"Le temps réel est généralement inférieur, mais reste substantiel à cette échelle.")
    results = []
    for run_idx, level_idx_row in enumerate(array, start=1):
        levels = {factor_names[i]: factor_levels[i][level_idx_row[i] - 1] for i in range(n_factors)}
        print(f"\n  Run {run_idx:02d}/{len(array)} : {levels}")
        overrides = dict(levels)
        if 'K_DESTROY_RATIO' in overrides:
            overrides['K_DESTROY'] = max(3, round(100 * overrides['K_DESTROY_RATIO']))
        t_run = time.time()
        with _cfg_override(**overrides):
            _, _, initial_costs_o, best_costs_o, _, _, _ = train_on_instances(
                outer_instances, episodes_per_instance=episodes_per_instance_doe,
                save_model=False, verbose=False,
            )
        values = []
        for i, inst in enumerate(outer_instances):
            if metric == 'bks_gap':
                bks = BKS_REFERENCE.get(inst['name'])
                if bks:
                    values.append((best_costs_o[i] - bks['distance']) / bks['distance'] * 100.0)
            elif metric == 'lns_improvement':
                ic = initial_costs_o[i]
                if ic:
                    values.append((best_costs_o[i] - ic) / ic * 100.0)
            else:
                raise ValueError(f"metric inconnue : {metric!r} (attendu 'bks_gap' ou 'lns_improvement')")
        mean_val = float(np.mean(values)) if values else None
        sn       = taguchi_sn_ratio_stb(values) if values else None
        row_result = dict(levels)
        row_result.update({
            'run': run_idx,
            'mean_gap_%': round(mean_val, 3) if mean_val is not None else None,
            'sn_ratio':   round(sn, 3) if sn is not None else None,
        })
        results.append(row_result)
        elapsed = time.time() - t_run
        if mean_val is not None:
            print(f"    → {'gap' if metric=='bks_gap' else 'amélioration'} moyen(ne) = "
                  f"{mean_val:+.2f}%  |  S/N = {sn:.2f}  |  {elapsed:.1f}s")
        else:
            print(f"    → indisponible  |  {elapsed:.1f}s")
    csv_path = os.path.join(outdir, f'taguchi_resultats{json_suffix}.csv')
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = factor_names + ['run', 'mean_gap_%', 'sn_ratio']
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        w.writeheader()
        for r in results:
            w.writerow(r)
    print(f"\n  Résultats Taguchi exportés (CSV) → {csv_path}")
    effects_gap = taguchi_main_effects(results, factor_names, 'mean_gap_%')
    effects_sn  = taguchi_main_effects(results, factor_names, 'sn_ratio')
    anova       = taguchi_anova_contributions(results, factor_names, 'mean_gap_%')
    response_label = 'Gap moyen vs BKS (%)' if metric == 'bks_gap' else 'Amélioration moyenne vs LNS (%)'
    _sec("GRAPHIQUES TAGUCHI")
    plot_taguchi_main_effects(effects_gap, factor_names, response_label, outdir,
                              f'taguchi_effets_principaux_gap{json_suffix}.png')
    plot_taguchi_main_effects(effects_sn, factor_names, 'Ratio S/N', outdir,
                              f'taguchi_effets_principaux_sn{json_suffix}.png')
    plot_taguchi_sn_bars(results, outdir, filename=f'taguchi_sn_ratios{json_suffix}.png')
    plot_taguchi_pareto(anova, outdir, filename=f'taguchi_pareto_effets{json_suffix}.png')
    valid_runs = [r for r in results if r['mean_gap_%'] is not None]
    best_run = min(valid_runs, key=lambda r: abs(r['mean_gap_%'])) if valid_runs else None
    print(f"\n{'█'*72}")
    print(f"  █  MEILLEURE COMBINAISON TROUVÉE — PLAN TAGUCHI{' (' + json_suffix + ')' if json_suffix else ''}")
    print(f"{'█'*72}")
    if best_run:
        print(f"  Run {best_run['run']} : {'gap' if metric=='bks_gap' else 'amélioration'} moyen(ne) = "
              f"{best_run['mean_gap_%']:+.2f}%  |  S/N = {best_run['sn_ratio']:.2f}")
        print(f"  ── Hyperparamètres recommandés ──")
        for fn in factor_names:
            print(f"    {fn:<18s} = {best_run[fn]}")
        print(f"\n  ── Contribution des facteurs à la variance de la réponse (ANOVA) ──")
        for fn, pct in sorted(anova.items(), key=lambda kv: -kv[1]):
            print(f"    {fn:<18s} : {pct:5.1f}%")
        print(f"\n  → Reportez ces valeurs dans CFG (et TAGUCHI_FACTORS si besoin), "
              f"repassez ENABLE=False, et relancez le script pour la validation "
              f"ENTRAÎNEMENT/TEST DISJOINT sur les 56 instances Solomon.")
        best_json_path = os.path.join(outdir, f'taguchi_meilleure_combinaison{json_suffix}.json')
        with open(best_json_path, 'w', encoding='utf-8') as f:
            json.dump({
                'run':               best_run['run'],
                'metric':            metric,
                'mean_gap_%':        best_run['mean_gap_%'],
                'sn_ratio':          best_run['sn_ratio'],
                'hyperparametres':   {fn: best_run[fn] for fn in factor_names},
                'contribution_anova_%': anova,
                'array':             array_name,
                'n_runs':            len(array),
                'n_outer_instances': len(outer_instances),
                'episodes_per_instance_doe': episodes_per_instance_doe,
            }, f, indent=2, ensure_ascii=False)
        print(f"\n  Meilleure combinaison sauvegardée (JSON) → {best_json_path}")
    print(f"{'█'*72}\n")
    return results, effects_gap, effects_sn, anova, best_run
def load_and_apply_taguchi_best(outdir, json_suffix=''):
    """
    Étape de confirmation : cherche taguchi_meilleure_combinaison<suffix>.json
    (créé par run_taguchi_doe) et applique ces hyperparamètres à CFG, pour
    que la validation ENTRAÎNEMENT/TEST DISJOINT utilise automatiquement la
    meilleure combinaison trouvée — sans avoir à la reporter à la main.
    Passez json_suffix='_synthetique' pour charger la combinaison calibrée
    UNIQUEMENT sur du synthétique (recommandé). Ne modifie rien (et affiche
    un avertissement) si le fichier n'existe pas encore.
    Retourne True si des hyperparamètres ont été appliqués, False sinon.
    """
    path = os.path.join(outdir, f'taguchi_meilleure_combinaison{json_suffix}.json')
    if not os.path.exists(path):
        cfg_key_hint = "CFG['TAGUCHI_SYNTH']" if json_suffix else "CFG['TAGUCHI']"
        print(f"  ⚠ {path} introuvable — lancez d'abord le plan Taguchi correspondant "
              f"({cfg_key_hint}['ENABLE']=True). Hyperparamètres CFG actuels inchangés.")
        return False
    with open(path, 'r', encoding='utf-8') as f:
        best = json.load(f)
    hp = best.get('hyperparametres', {})
    for fn, val in hp.items():
        CFG[fn] = val
    if 'K_DESTROY_RATIO' in hp:
        CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))
    print(f"  ✓ Hyperparamètres Taguchi appliqués depuis {path} :")
    for fn, val in hp.items():
        print(f"    {fn:<18s} = {val}")
    return True
# ══════════════════════════════════════════════════════════════════════════════
#  EXÉCUTION AUTOMATIQUE
# ══════════════════════════════════════════════════════════════════════════════
def run_final_validation():
    """
    MODE VALIDATION : ENTRAÎNEMENT/TEST DISJOINT.
    Entraîne le DQN sur CFG['N_SYNTH_INSTANCES'] instances synthétiques puis
    l'évalue gelé sur les 56 instances Solomon officielles. Si
    CFG['USE_TAGUCHI_BEST']=True, applique d'abord automatiquement la
    meilleure combinaison d'hyperparamètres trouvée par le plan Taguchi
    synthétique (fichier taguchi_meilleure_combinaison_synthetique.json).

    Cette fonction regroupe exactement ce que faisait auparavant la branche
    `else` du bloc __main__ — extraite dans une fonction pour pouvoir être
    appelée aussi bien seule (validation directe, sans calibration) que
    depuis la branche Taguchi synthétique, en enchaînement automatique.
    """
    t0 = time.time()
    print(f"\n{'═'*72}\n  MODE ENTRAÎNEMENT/TEST DISJOINT\n"
          f"  ENTRAÎNEMENT = {CFG['N_SYNTH_INSTANCES']} instances SYNTHÉTIQUES générées "
          f"| TEST = 56 instances Solomon officielles (gelé)\n{'═'*72}")

    # ── DIAGNOSTIC PRÉALABLE : licence Gurobi à taille limitée ? ────────
    # Informatif uniquement — n'interrompt jamais l'exécution.
    _sec("DIAGNOSTIC — LICENCE GUROBI")
    check_gurobi_license()

    # ── Optionnel : appliquer la meilleure combinaison Taguchi calibrée
    # sur synthétique (si le plan CFG['TAGUCHI_SYNTH'] a déjà été lancé
    # au moins une fois et a produit le JSON correspondant) ───────────
    if CFG.get('USE_TAGUCHI_BEST', False):
        _sec("APPLICATION DE LA MEILLEURE COMBINAISON TAGUCHI (SYNTHÉTIQUE)")
        load_and_apply_taguchi_best(CFG['OUTDIR'], json_suffix='_synthetique')

    # ── ÉTAPE 1a : GÉNÉRATION DES INSTANCES SYNTHÉTIQUES D'ENTRAÎNEMENT ─
    # (reprend generate_instance() de Code 1 — cf. MODULE 1bis ci-dessus).
    # Ces instances SYN-* ne recoupent jamais les noms Solomon (C101, R101,
    # ...) chargés à l'ÉTAPE 2 : aucune fuite entre train et test.
    synth_instances = generate_synthetic_training_set(CFG)

    # ── ÉTAPE 1b : ENTRAÎNEMENT DU DQN — uniquement sur le synthétique ──
    # Valeurs par défaut : si le modèle est rechargé (branche ci-dessous),
    # il n'y a pas de training_log/coûts synthétiques dans CE run — les
    # graphiques de convergence d'entraînement seront alors simplement omis.
    training_log, synth_initial_costs, synth_best_costs = [], None, None
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    if CFG.get('REUSE_EXISTING_SYNTH_MODEL', False) and os.path.exists(model_path):
        _sec(f"ÉTAPE 1b — RÉUTILISATION D'UN MODÈLE DÉJÀ ENTRAÎNÉ SUR LE SYNTHÉTIQUE "
             f"({model_path})")
        agent = DQNAgent(
            obs_dim=7, n_actions=ACTION_SETS_N.get(CFG.get('ACTION_SET', 'full'), 4),
            hidden=CFG['HIDDEN'], lr=CFG['LR'],
            gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
            epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
            buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
            target_update_freq=CFG['TARGET_UPDATE'],
        )
        agent.load(model_path)
    else:
        agent, training_log, synth_initial_costs, synth_best_costs, \
            synth_best_solutions, synth_instance_times, synth_initial_sols = train_on_instances(
                synth_instances, episodes_per_instance=CFG['EPISODES_PER_INSTANCE'])

    # ── ÉTAPE 2a : CHARGEMENT DES 56 INSTANCES SOLOMON (TEST — jamais vues) ─
    _sec("ÉTAPE 2 — CHARGEMENT DES INSTANCES SOLOMON (TEST)")
    test_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
    example_idxs = pick_example_indices(test_instances, preferred_names=CFG['EXAMPLE_NAMES'])

    # ── ÉTAPE 2b : ÉVALUATION GELÉE — le même agent en mémoire, sans réentraîner ──
    initial_costs, best_costs, best_solutions, instance_times, test_initial_sols, test_log = \
        evaluate_frozen_on_instances(
            test_instances, agent, episodes_per_instance=CFG['EPISODES_PER_INSTANCE_TEST'])

    # ── ÉTAPE 2c : FUSION DE ROUTES (post-traitement, MODULE 5bis) ──────
    # Appliquée UNE FOIS sur la solution finale du test gelé (DQN+LNS+Gurobi),
    # AVANT le tableau/les graphiques — donc le tableau final, les graphiques
    # de routes "APRÈS" et le gap % vs BKS reflètent tous la solution
    # post-fusion (best_costs/best_solutions sont écrasés ici). N'affecte que
    # le TEST : la fusion n'intervient jamais pendant l'entraînement
    # synthétique, ni dans la boucle DQN/LNS/Gurobi elle-même.
    if CFG.get('ROUTE_FUSION', {}).get('ENABLE', False):
        best_solutions, best_costs, _route_fusion_stats = route_merge_postprocess_batch(
            test_instances, best_solutions, verbose=True)

    # ── [A] Convergence de l'ENTRAÎNEMENT (synthétique uniquement) ──────
    _sec("GRAPHIQUE — CONVERGENCE DE L'ENTRAÎNEMENT (SYNTHÉTIQUE)")
    if training_log:
        plot_global_convergence(training_log, synth_instances,
                                 synth_initial_costs, synth_best_costs, CFG['OUTDIR'])
    else:
        print("  (pas de courbe de convergence : modèle rechargé, aucun "
              "entraînement effectué dans ce run)")

    # ── [B] Convergence détaillée + [C] Routes avant/après — SUR LE TEST ─
    _sec("GRAPHIQUES — DÉTAIL DU TEST (INSTANCES SOLOMON EXEMPLES)")
    plot_instance_convergence_examples(test_log, test_instances, example_idxs, CFG['OUTDIR'])
    plot_routes_examples(test_instances, test_initial_sols, best_solutions, example_idxs, CFG['OUTDIR'])

    # ── [D] TABLEAU FINAL : SITUATION 1 (test gelé, entraîné sur synthétique) ─
    # vs BKS (gap % par instance + gap moyen) ──────────────────────────
    bks_rows = build_bks_comparison_table(
        test_instances, initial_costs, best_costs, best_solutions, instance_times,
        csv_path=os.path.join(CFG['OUTDIR'], 'solomon_comparaison_bks.csv'),
        latex_path=os.path.join(CFG['OUTDIR'], 'solomon_tableau_bks.tex'),
    )
    # ── [E] Graphique dédié : gap % vs BKS par instance + gap moyen (TEST) ─
    _sec("GRAPHIQUE GAP % vs BKS (TEST)")
    plot_gap_vs_bks(bks_rows, CFG['OUTDIR'])

    # ── [F] Analyses par groupe (C1/C2/R1/R2/RC1/RC2) — véhicules & gap ──
    _sec("GRAPHIQUES — ANALYSES PAR GROUPE D'INSTANCES (TEST)")
    plot_gap_boxplot_by_group(bks_rows, test_instances, CFG['OUTDIR'])
    plot_vehicles_vs_bks(bks_rows, test_instances, CFG['OUTDIR'])

    print(f"\n  ✓ Validation finale terminée en {time.time()-t0:.1f}s "
          f"({(time.time()-t0)/3600:.2f}h)\n")
    return bks_rows
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 11  —  ABLATION DE L'ESPACE D'ACTIONS DU DQN (étude comparative)
# ══════════════════════════════════════════════════════════════════════════════
# Compare 3 jeux d'actions de destruction (voir ACTION_SETS_N / ACTION_NAMES_
# BY_SET, et VRPTWEnv._destroy) à hyperparamètres CFG identiques par ailleurs
# (seul CFG['ACTION_SET'] change entre les 3 runs) :
#   'full'    (4 actions) : spatial, temporel, coût élevé, aléatoire — jeu
#              d'origine, chaque opérateur de destruction séparé.
#   'merged3' (3 actions) : spatial+temporel FUSIONNÉS en une seule action
#              "spatio-temporelle" (moyenne 0.5/0.5 des scores normalisés),
#              coût élevé et aléatoire inchangés.
#   'merged2' (2 actions) : la même fusion spatio-temporelle + aléatoire —
#              l'action "coût élevé" est ici supprimée, pas fusionnée.
# Chaque jeu répète EXACTEMENT le protocole ENTRAÎNEMENT (synthétique, jamais
# Solomon) / TEST (56 instances Solomon, gelé) de run_final_validation() — un
# agent DQN neuf est entraîné à chaque fois (jamais de réutilisation d'un
# .pth calibré sur un autre jeu d'actions, la taille de la couche de sortie
# diffère de toute façon). Objectif : mesurer si réduire/fusionner les
# actions aide (moins de choix à apprendre, budget d'entraînement fixe) ou
# nuit (moins de diversité dans la recherche de destruction).
def run_action_space_ablation():
    """
    Lance run_final_validation()-comme (entraînement synthétique + test
    Solomon gelé) une fois par jeu d'actions dans CFG['ACTION_ABLATION']
    ['SETS'], calcule le gap % vs BKS moyen/min/max pour chacun, exporte un
    CSV détaillé (instance x jeu) + un CSV résumé + un graphique comparatif,
    affiche un tableau récapitulatif trié par gap moyen croissant, et
    sauvegarde le meilleur jeu trouvé dans un JSON.
    Ne modifie PAS CFG['ACTION_SET'] de façon durable (restauré à la fin,
    voir _cfg_override) : pour lancer ensuite une validation normale avec le
    meilleur jeu identifié, reportez-le vous-même dans CFG['ACTION_SET'] et
    repassez CFG['ACTION_ABLATION']['ENABLE']=False (voir message final).
    Retourne un dict {action_set: {'mean_gap_%':..., ...}}.
    """
    t0 = time.time()
    action_sets = CFG['ACTION_ABLATION']['SETS']
    outdir = CFG['OUTDIR']
    _sec(f"ABLATION DE L'ESPACE D'ACTIONS DU DQN — {len(action_sets)} jeux à "
         f"comparer ({', '.join(action_sets)})")
    print(f"  Protocole IDENTIQUE à run_final_validation() pour chaque jeu : "
          f"entraînement sur {CFG['N_SYNTH_INSTANCES']} instances synthétiques "
          f"({CFG['EPISODES_PER_INSTANCE']} épisodes/instance) puis test gelé "
          f"sur les 56 instances Solomon ({CFG['EPISODES_PER_INSTANCE_TEST']} "
          f"épisodes/instance). ⚠ Temps total ≈ {len(action_sets)}× un run normal.")

    # ── Optionnel : appliquer la meilleure combinaison Taguchi calibrée sur
    # synthétique, UNE SEULE FOIS avant la boucle — comme dans
    # run_final_validation(), pour que les 3 jeux d'actions soient comparés
    # avec exactement les mêmes hyperparamètres LNS/Gurobi par ailleurs ────
    if CFG.get('USE_TAGUCHI_BEST', False):
        _sec("APPLICATION DE LA MEILLEURE COMBINAISON TAGUCHI (SYNTHÉTIQUE)")
        load_and_apply_taguchi_best(outdir, json_suffix='_synthetique')

    # ── Instances de TEST (56 Solomon officielles) — chargées une seule
    # fois, réutilisées à l'identique pour les 3 jeux d'actions ──────────
    _sec("CHARGEMENT DES INSTANCES SOLOMON (TEST)")
    test_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])

    all_results = {}
    detail_rows = []
    for set_name in action_sets:
        n_act = ACTION_SETS_N.get(set_name, '?')
        print(f"\n  {'─'*66}\n  Jeu d'actions : '{set_name}' ({n_act} actions)\n  {'─'*66}")
        t_set = time.time()
        with _cfg_override(ACTION_SET=set_name):
            # ── ENTRAÎNEMENT — instances synthétiques générées à la volée,
            # un agent DQN NEUF à chaque jeu d'actions (n_actions diffère) ─
            synth_instances = generate_synthetic_training_set(CFG, verbose=False)
            agent, _, _, _, _, _, _ = train_on_instances(
                synth_instances, episodes_per_instance=CFG['EPISODES_PER_INSTANCE'],
                save_model=False, verbose=False,
            )
            # ── TEST gelé — mêmes 56 instances Solomon pour les 3 jeux ────
            (_, best_costs, best_solutions, instance_times,
             _, _) = evaluate_frozen_on_instances(
                test_instances, agent, episodes_per_instance=CFG['EPISODES_PER_INSTANCE_TEST'])
        elapsed_set = time.time() - t_set
        gaps = []
        for i, inst in enumerate(test_instances):
            bks   = BKS_REFERENCE.get(inst['name'])
            n_veh = sum(1 for r in best_solutions[i] if r)
            gap = None
            if bks:
                gap = (best_costs[i] - bks['distance']) / bks['distance'] * 100.0
                gaps.append(gap)
            detail_rows.append({
                'action_set':  set_name,
                'n_actions':   n_act,
                'instance':    inst['name'],
                'n_vehicules': n_veh,
                'distance':    round(best_costs[i], 2),
                'gap_%':       round(gap, 2) if gap is not None else None,
                'temps_s':     round(instance_times[i], 2),
            })
        mean_gap = float(np.mean(gaps)) if gaps else None
        all_results[set_name] = {
            'n_actions':     n_act,
            'mean_gap_%':    round(mean_gap, 3) if mean_gap is not None else None,
            'min_gap_%':     round(min(gaps), 3) if gaps else None,
            'max_gap_%':     round(max(gaps), 3) if gaps else None,
            'temps_total_s': round(elapsed_set, 1),
        }
        if mean_gap is not None:
            print(f"\n  → Gap moyen ('{set_name}') = {mean_gap:+.2f}%  "
                  f"(min {min(gaps):+.2f}% / max {max(gaps):+.2f}%)  "
                  f"| {elapsed_set:.1f}s ({elapsed_set/3600:.2f}h)")

    # ── Export CSV détaillé (instance x jeu d'actions) ──────────────────
    csv_path = os.path.join(outdir, 'ablation_actions_detail.csv')
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['action_set', 'n_actions', 'instance', 'n_vehicules',
                      'distance', 'gap_%', 'temps_s']
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in detail_rows:
            w.writerow(r)
    summary_path = os.path.join(outdir, 'ablation_actions_resume.csv')
    with open(summary_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['action_set', 'n_actions', 'mean_gap_%', 'min_gap_%',
                      'max_gap_%', 'temps_total_s']
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for set_name, r in all_results.items():
            w.writerow({'action_set': set_name, **r})

    # ── Tableau récapitulatif (trié par gap moyen croissant) ────────────
    _sec("RÉCAPITULATIF — ABLATION DE L'ESPACE D'ACTIONS DU DQN")
    ranked = sorted(all_results.items(),
                     key=lambda kv: (kv[1]['mean_gap_%'] is None, kv[1]['mean_gap_%']))
    print(f"  {'Jeu':<12} {'#Actions':>9} {'Gap moyen':>11} {'Gap min':>9} {'Gap max':>9} {'Temps':>10}")
    print("  " + "─" * 62)
    for set_name, r in ranked:
        gm   = f"{r['mean_gap_%']:+.2f}%" if r['mean_gap_%'] is not None else "—"
        gmin = f"{r['min_gap_%']:+.2f}%"  if r['min_gap_%']  is not None else "—"
        gmax = f"{r['max_gap_%']:+.2f}%"  if r['max_gap_%']  is not None else "—"
        th   = r['temps_total_s'] / 3600.0
        print(f"  {set_name:<12} {r['n_actions']:>9} {gm:>11} {gmin:>9} {gmax:>9} {th:>9.2f}h")
    best_set = ranked[0][0] if ranked else None
    if best_set:
        print(f"\n  → Meilleur jeu d'actions trouvé : '{best_set}' "
              f"(gap moyen = {all_results[best_set]['mean_gap_%']:+.2f}%)")
        print(f"  Pour lancer une validation normale avec ce jeu : mettez "
              f"CFG['ACTION_SET'] = '{best_set}', CFG['ACTION_ABLATION']['ENABLE'] "
              f"= False, et relancez le script.")
        best_json_path = os.path.join(outdir, 'ablation_actions_meilleur_jeu.json')
        with open(best_json_path, 'w', encoding='utf-8') as f:
            json.dump({
                'best_action_set': best_set,
                'resultats':       all_results,
                'n_synth_instances': CFG['N_SYNTH_INSTANCES'],
                'episodes_per_instance': CFG['EPISODES_PER_INSTANCE'],
                'episodes_per_instance_test': CFG['EPISODES_PER_INSTANCE_TEST'],
            }, f, indent=2, ensure_ascii=False)
        print(f"  Meilleur jeu sauvegardé (JSON) → {best_json_path}")

    # ── Graphique comparatif (barres, gap moyen + min/max en repère) ────
    if ranked:
        fig, ax = plt.subplots(figsize=(7, 5))
        names   = [s for s, _ in ranked]
        means   = [all_results[s]['mean_gap_%'] for s in names]
        mins    = [all_results[s]['min_gap_%'] for s in names]
        maxs    = [all_results[s]['max_gap_%'] for s in names]
        errs_lo = [m - lo for m, lo in zip(means, mins)]
        errs_hi = [hi - m for m, hi in zip(means, maxs)]
        colors  = ['#2ca02c' if s == best_set else '#4C6EF5' for s in names]
        labels  = [f"{s}\n({ACTION_SETS_N.get(s, '?')} actions)" for s in names]
        ax.bar(labels, means, color=colors, yerr=[errs_lo, errs_hi], capsize=6)
        ax.axhline(0, color='black', linewidth=0.8)
        for i, m in enumerate(means):
            ax.text(i, m + (1 if m >= 0 else -1), f"{m:+.2f}%", ha='center',
                    fontsize=10, fontweight='bold')
        ax.set_ylabel('Gap % vs BKS (moyenne, min-max en repère)')
        ax.set_title(f"Comparaison des jeux d'actions DQN "
                     f"(56 instances Solomon, {CFG['EPISODES_PER_INSTANCE_TEST']} "
                     f"épisodes/instance test)")
        ax.grid(alpha=0.3, axis='y')
        plt.tight_layout()
        png_path = os.path.join(outdir, 'ablation_actions_comparaison.png')
        plt.savefig(png_path, dpi=140, bbox_inches='tight')
        plt.close(fig)
        print(f"\n  Graphique comparatif → {png_path}")

    print(f"\n  Détail exporté (CSV) → {csv_path}")
    print(f"  Résumé exporté (CSV) → {summary_path}")
    print(f"\n  ✓ Ablation des actions terminée en {time.time()-t0:.1f}s "
          f"({(time.time()-t0)/3600:.2f}h).\n")
    return all_results


if __name__ == '__main__':
    t_global0 = time.time()

    if CFG['TAGUCHI_SYNTH']['ENABLE'] and CFG['TAGUCHI']['ENABLE']:
        raise ValueError(
            "CFG['TAGUCHI_SYNTH']['ENABLE'] et CFG['TAGUCHI']['ENABLE'] sont tous les "
            "deux True — activez UN SEUL des deux modes de calibration à la fois "
            "(TAGUCHI_SYNTH est recommandé, voir les commentaires plus haut)."
        )

    if CFG['TAGUCHI_SYNTH']['ENABLE']:
        # ── MODE CALIBRATION TAGUCHI — SYNTHÉTIQUE UNIQUEMENT (RECOMMANDÉ) ──
        # Aucune instance Solomon n'est chargée dans cette branche : zéro
        # fuite d'hyperparamètres vers le jeu de test.
        ts = CFG['TAGUCHI_SYNTH']
        print(f"\n{'═'*72}\n  MODE PLAN D'EXPÉRIENCE TAGUCHI — SYNTHÉTIQUE UNIQUEMENT "
              f"({ts['N_INSTANCES_DOE']} instances, métrique = amélioration vs LNS, "
              f"pas de BKS, zéro fuite vers Solomon)\n{'═'*72}")
        synth_doe_instances = generate_synthetic_training_set(
            dict(N_SYNTH_INSTANCES=ts['N_INSTANCES_DOE'],
                 SYNTH_N_CUSTOMERS=CFG['SYNTH_N_CUSTOMERS'],
                 SYNTH_N_VEHICLES=CFG['SYNTH_N_VEHICLES'],
                 SYNTH_VEHICLE_SAFETY_FACTOR=CFG['SYNTH_VEHICLE_SAFETY_FACTOR'],
                 SYNTH_CAPACITY=CFG['SYNTH_CAPACITY'], SYNTH_HORIZON=CFG['SYNTH_HORIZON']),
            verbose=True,
        )
        (ts_results, ts_effects_gap, ts_effects_sn,
         ts_anova, ts_best_run) = run_taguchi_doe(
            factors=TAGUCHI_FACTORS,
            outer_instance_names=None,
            episodes_per_instance_doe=ts['EPISODES_PER_INSTANCE_DOE'],
            outdir=CFG['OUTDIR'],
            instances_override=synth_doe_instances,
            metric='lns_improvement',
            json_suffix='_synthetique',
        )
        _sec("RÉCAPITULATIF FINAL DU PLAN TAGUCHI (SYNTHÉTIQUE)")
        if ts_best_run:
            print("  Meilleurs hyperparamètres trouvés (calibrés sur synthétique) :")
            for fn, _ in TAGUCHI_FACTORS:
                print(f"    {fn:<18s} = {ts_best_run[fn]}")
            print(f"  Amélioration moyenne vs LNS obtenue : {-ts_best_run['mean_gap_%']:.2f}%")
            print(f"  Ratio S/N                           : {ts_best_run['sn_ratio']:.2f}")
        print(f"\n  Meilleure combinaison (JSON) : "
              f"{os.path.join(CFG['OUTDIR'], 'taguchi_meilleure_combinaison_synthetique.json')}")
        print(f"\n  ✓ Plan Taguchi (synthétique) terminé en {time.time()-t_global0:.1f}s "
              f"({(time.time()-t_global0)/3600:.2f}h).")

        # ── ENCHAÎNEMENT AUTOMATIQUE → VALIDATION FINALE AVEC LES BEST HYPERPARAMÈTRES ──
        # Entraînement 80 instances synthétiques + test gelé sur les 56 Solomon,
        # en un seul lancement, sans intervention manuelle.
        if ts_best_run:
            _sec("ENCHAÎNEMENT AUTOMATIQUE → VALIDATION FINALE")
            CFG['USE_TAGUCHI_BEST'] = True
            # Repart d'un modèle neuf avec les NOUVEAUX hyperparamètres —
            # ne jamais réutiliser un .pth calibré avec d'anciens réglages ici.
            CFG['REUSE_EXISTING_SYNTH_MODEL'] = False
            run_final_validation()
        else:
            print("  ⚠ Aucune combinaison valide trouvée par le plan Taguchi — "
                  "validation finale non lancée.")

    elif CFG['TAGUCHI']['ENABLE']:
        # ── MODE CALIBRATION TAGUCHI — SOLOMON (⚠ HISTORIQUE, NON RECOMMANDÉ) ──
        # Volontairement PAS d'enchaînement automatique ici : ce mode calibre
        # directement sur le jeu de test (fuite train/test), la combinaison
        # trouvée ne doit pas servir à une validation train/test disjointe.
        print(f"\n{'═'*72}\n  ⚠ MODE PLAN D'EXPÉRIENCE TAGUCHI — SOLOMON (mode historique)\n"
              f"  Ce mode entraîne ET évalue directement sur les 56 instances Solomon :\n"
              f"  la combinaison retournée aura VU le jeu de test et ne doit PAS être\n"
              f"  utilisée comme hyperparamètres d'une validation train/test disjointe.\n"
              f"  Préférez CFG['TAGUCHI_SYNTH']['ENABLE']=True (calibration synthétique).\n"
              f"{'═'*72}")
        tg = CFG['TAGUCHI']
        (tg_results, tg_effects_gap, tg_effects_sn,
         tg_anova, tg_best_run) = run_taguchi_doe(
            factors=TAGUCHI_FACTORS,
            outer_instance_names=tg['OUTER_INSTANCE_NAMES'],
            episodes_per_instance_doe=tg['EPISODES_PER_INSTANCE_DOE'],
            outdir=CFG['OUTDIR'],
            instances_override=None,
            metric='bks_gap',
            json_suffix='',
        )
        _sec("RÉCAPITULATIF FINAL DU PLAN TAGUCHI (SOLOMON — HISTORIQUE)")
        if tg_best_run:
            print("  Meilleurs hyperparamètres trouvés (⚠ calibrés sur Solomon — fuite) :")
            for fn, _ in TAGUCHI_FACTORS:
                print(f"    {fn:<18s} = {tg_best_run[fn]}")
            print(f"  Gap moyen obtenu : {tg_best_run['mean_gap_%']:+.2f}%")
            print(f"  Ratio S/N        : {tg_best_run['sn_ratio']:.2f}")
        print(f"\n  ✓ Plan Taguchi (Solomon) terminé en {time.time()-t_global0:.1f}s "
              f"({(time.time()-t_global0)/3600:.2f}h). Pas d'enchaînement automatique "
              f"(mode avec fuite) — repassez CFG['TAGUCHI']['ENABLE']=False et relancez "
              f"manuellement si vous souhaitez tout de même l'utiliser, à titre de "
              f"comparaison uniquement.\n")

    elif CFG.get('ACTION_ABLATION', {}).get('ENABLE', False):
        # ── MODE ABLATION DE L'ESPACE D'ACTIONS DU DQN (voir MODULE 11) ────
        # Compare full (4) / merged3 (3) / merged2 (2) au lieu de la
        # validation normale. Repassez CFG['ACTION_ABLATION']['ENABLE']=False
        # pour revenir au mode normal (ou à un autre mode ci-dessus).
        run_action_space_ablation()

    else:
        # ── MODE VALIDATION SEULE (comportement normal, sans calibration) ──
        run_final_validation()


════════════════════════════════════════════════════════════════════════
  MODE ENTRAÎNEMENT/TEST DISJOINT
  ENTRAÎNEMENT = 80 instances SYNTHÉTIQUES générées | TEST = 56 instances Solomon officielles (gelé)
════════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════════
  DIAGNOSTIC — LICENCE GUROBI
════════════════════════════════════════════════════════════════════════

  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée (erreur 10010 : Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information)
    → Licence Community/gratuite, plafonnée à ~2000 variables et ~2000 contraintes.
    → Le repair Gurobi de la Situation 1 (jusqu'à ~15000 variables selon K_DESTROY=24 et le nombre de véhicules) peut dépasser cette limite sur certaines instances — un repli automatique vers le repair glouton est déjà en place, donc pas de plantage, mais un usage de Gurobi moins fréquent que

In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   DQN + LNS + GUROBI  —  ENTRAÎNEMENT SUR SYNTHÉTIQUE / TEST SUR SOLOMON    ║
║                          + GAP PAR RAPPORT AU BKS                           ║
║                                                                              ║
║  Version corrigée : reprend toutes les visualisations de la variante        ║
║  précédente (courbes de convergence, tracés des routes avant/après,        ║
║  graphique dédié gap % vs BKS, tableau récapitulatif véhicules/distance)   ║
║  MAIS applique désormais un vrai protocole ENTRAÎNEMENT / TEST DISJOINT,   ║
║  en un seul lancement, sans aucune intervention manuelle :                 ║
║                                                                              ║
║   ÉTAPE 1 (ENTRAÎNEMENT) — génère des instances SYNTHÉTIQUES (reprises de   ║
║      generate_instance() de main_vrptw_comparaison_gurobi.py / "Code 1")    ║
║      et entraîne le DQN dessus (poids mis à jour via store/learn/decay).    ║
║                                                                              ║
║   ÉTAPE 2 (TEST) — charge les 56 instances Solomon OFFICIELLES et évalue    ║
║      le même agent, GELÉ (aucun store/learn/decay_epsilon, uniquement      ║
║      select_action en pur mode inférence), sur ce jeu jamais vu à          ║
║      l'entraînement. Toutes les courbes/tracés/tableaux ci-dessous portent  ║
║      désormais sur ce TEST gelé, pas sur les résultats d'entraînement.     ║
║                                                                              ║
║  Situation 1 = LNS + DQN(entraîné sur synthétique) + Gurobi, ÉVALUÉ sur     ║
║                Solomon (test)                                → gap vs BKS   ║
║  (La "Situation 2 = Gurobi seul sur l'instance complète" reste retirée,     ║
║   à la demande, car sans limite de temps elle est beaucoup trop longue      ║
║   sur des instances à 100 clients avec fenêtres de temps serrées.)          ║
║                                                                              ║
║  Visualisations produites (toutes sur le TEST gelé, sauf la 1ère) :        ║
║    [A] Convergence de l'ENTRAÎNEMENT (synthétique) : reward global +       ║
║        amélioration % par instance synthétique                            ║
║    [B] Convergence détaillée (instances Solomon exemples, TEST) : coût     ║
║        final/meilleur par épisode local + trajectoire de coût             ║
║    [C] Tracés des routes AVANT (LNS) / APRÈS (DQN gelé) pour des           ║
║        instances Solomon exemples                                         ║
║    [D] Tableau récapitulatif : Instance / Nb véhicules / Distance / Gap %  ║
║        (TEST Solomon vs BKS)                                              ║
║    [E] Graphique dédié gap % vs BKS par instance + gap moyen              ║
║    [F] Diagnostic automatique de licence Gurobi (taille limitée ou non)   ║
║                                                                              ║
║  ⚠ MODIFICATION APPORTÉE SUR CETTE COPIE (reproductibilité) :              ║
║   `_repair_gurobi()` fixe désormais `Threads=1` et `Seed=0` sur chaque     ║
║   modèle Gurobi de repair. Dans la version d'origine, ces deux paramètres  ║
║   n'étaient pas fixés : Gurobi pouvait donc retourner des solutions        ║
║   différentes d'un run à l'autre sur un MÊME modèle (recherche             ║
║   branch-and-bound multi-thread non déterministe, surtout en heurtant      ║
║   `TimeLimit` sans avoir fermé le gap d'optimalité) — alors même que tous  ║
║   les seeds Python/NumPy/PyTorch sont fixés (seed=0 par défaut dans        ║
║   DQNAgent). C'est la cause la plus probable de l'écart observé entre deux ║
║   exécutions de CE MÊME script avec un CFG strictement identique (run 1 : ║
║   gap moyen +7,55% pour merged2 dans l'étude d'ablation ; run 2 : +19,77%  ║
║   avec cette configuration merged2+fusion). Fixer Threads=1 et Seed=0      ║
║   rend chaque appel à `_repair_gurobi()` déterministe, donc rend tout le   ║
║   pipeline (entraînement + test) reproductible d'une exécution à l'autre,  ║
║   pour peu que rien d'autre ne change dans CFG.                            ║
║                                                                              ║
║  Aucun rappel manuel de "Code 1" n'est nécessaire : la génération +        ║
║  l'entraînement synthétique sont réintégrés ici (MODULE 1bis) et           ║
║  s'exécutent automatiquement avant le chargement de Solomon. Voir le flag  ║
║  optionnel CFG['REUSE_EXISTING_SYNTH_MODEL'] plus bas si vous préférez     ║
║  recharger un .pth déjà entraîné au lieu de ré-entraîner à chaque run.     ║
║                                                                              ║
║  ⚠ SECONDE MODIFICATION APPORTÉE SUR CETTE COPIE (réduction du gap) :      ║
║   Objectif : faire baisser le gap moyen vs BKS de +7,55% (résultat de      ║
║   l'ablation, sans fusion) vers ~5,1–5,4% (-28 à -33% en relatif), en      ║
║   combinant TROIS leviers appliqués en post-traitement sur le même         ║
║   modèle entraîné une seule fois (pas de retraining supplémentaire) :      ║
║    1) Fusion de routes (MODULE 5bis, déjà présente) — réduit le nombre     ║
║       de véhicules, qui pèse lourd dans le gap BKS.                        ║
║    2) Recherche locale 2-opt + Or-opt (MODULE 5ter, NOUVEAU) — passe de    ║
║       raffinement post-fusion : 2-opt intra-route (réordonne les clients   ║
║       d'une même route) + Or-opt (déplace des segments de 1 à 3 clients    ║
║       vers la meilleure position faisable, même inter-routes), toutes deux ║
║       "best-improvement" et strictement gloutonnes (n'acceptent que les    ║
║       mouvements qui réduisent le coût total, donc ne peuvent jamais       ║
║       dégrader une solution). Une seconde passe de fusion est relancée     ║
║       ensuite car l'Or-opt peut vider une route ou alléger une charge et   ║
║       ouvrir de nouvelles opportunités de fusion.                          ║
║    3) `CFG['GUROBI_TLIMIT']` porté de 2.0s à 4.0s — les hyperparamètres    ║
║       LNS/DQN/Gurobi de CFG restent par ailleurs identiques à la           ║
║       meilleure combinaison déjà trouvée par le plan d'expérience Taguchi  ║
║       (voir commentaire sur CFG plus bas) ; seul TLIMIT est augmenté ici,  ║
║       car les runs précédents montrent Gurobi buter fréquemment sur cette  ║
║       limite sans avoir prouvé l'optimalité, ce qui dégrade la qualité des ║
║       réparations. ⚠ Coût : le temps d'entraînement augmente sensiblement  ║
║       (chaque appel à `_repair_gurobi()` peut désormais durer jusqu'à 2×   ║
║       plus longtemps) — prévoir ~15-20h au lieu de ~10h pour ce run.       ║
║                                                                              ║
║  Par défaut, INSTANCE_FILES contient toutes les instances *.txt trouvées   ║
║  dans outputs/Solomon/. Pour se limiter à 6 instances représentatives      ║
║  (une par groupe C1/C2/R1/R2/RC1/RC2), décommentez le bloc plus bas.       ║
║                                                                              ║
║  Exécution :  python main_vrptw_synth_train_solomon_merged2_fusion_polish.py║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import os, sys, json, time, copy, random, csv, glob, itertools
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from contextlib import contextmanager
import torch
import torch.nn as nn
import torch.optim as optim
import gurobipy as gp
from gurobipy import GRB
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG GLOBALE
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    # ── ÉTAPE 1 : ENTRAÎNEMENT sur instances SYNTHÉTIQUES (jamais Solomon) ──
    N_SYNTH_INSTANCES          = 80,     # comme les 80 instances de Code 1
    SYNTH_N_CUSTOMERS          = 100,
    SYNTH_N_VEHICLES           = None,   # None → calculé depuis la demande (cf. generate_instance)
    SYNTH_VEHICLE_SAFETY_FACTOR= 1.5,
    SYNTH_CAPACITY             = 200,
    SYNTH_HORIZON              = 1000,
    EPISODES_PER_INSTANCE      = 20,     # épisodes d'ENTRAÎNEMENT par instance synthétique
    REUSE_EXISTING_SYNTH_MODEL = False,
    USE_TAGUCHI_BEST            = False,
    # ── ÉTAPE 2 : TEST gelé sur les 56 instances Solomon officielles ───────
    EPISODES_PER_INSTANCE_TEST = 10,     # épisodes d'ÉVALUATION (aucun apprentissage) par instance Solomon
    VEHICLE_SAFETY_FACTOR  = None,
    # ── Hyperparamètres LNS/DQN/Gurobi : meilleure combinaison trouvée par
    # le plan d'expérience Taguchi synthétique — reportée ici EN DUR ───────
    LNS_ITER                = 100,
    LNS_K_RATIO            = 0.20,
    K_DESTROY_RATIO        = 0.24,
    MAX_STEPS               = 15,
    # GUROBI_TLIMIT : 2.0 → 4.0 (levier 3, voir bandeau en tête de fichier).
    # Les autres hyperparamètres LNS/DQN/Gurobi ci-dessus restent la
    # meilleure combinaison déjà trouvée par le plan d'expérience Taguchi
    # synthétique (inchangés) — seul TLIMIT est augmenté ici.
    GUROBI_TLIMIT            = 4.0,
    GUROBI_GAP               = 0.05,
    HIDDEN                   = 64,
    LR                       = 1e-3,
    GAMMA                    = 0.95,
    EPS_START                = 1.0,
    EPS_END                  = 0.05,
    EPS_DECAY                = 0.99,
    BUFFER_SIZE               = 5000,
    BATCH_SIZE                = 32,
    TARGET_UPDATE              = 10,
    EXAMPLE_NAMES              = ['C101', 'R101', 'RC101'],
    VEHICLE_PENALTY            = 0.0,
    I1_MU                       = 1.0,
    I1_LAMBDA                   = 2.0,
    I1_ALPHA1                   = 1.0,
    I1_ALPHA2                   = 0.0,
    OUTDIR                        = os.path.join(os.getcwd(), 'outputs'),
    MODELDIR                       = os.path.join(os.getcwd(), 'outputs', 'model'),
    MODEL_FILE                      = 'dqn_vrptw_synth_trained.pth',
    # ── Reproductibilité du repair Gurobi (voir bandeau en tête de fichier) ─
    GUROBI_SEED               = 0,
    GUROBI_THREADS            = 1,
)
CFG['LNS_K'] = 20
CFG['K_DESTROY'] = max(3, round(100 * CFG['K_DESTROY_RATIO']))
os.makedirs(CFG['OUTDIR'],   exist_ok=True)
os.makedirs(CFG['MODELDIR'], exist_ok=True)
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG DU PLAN D'EXPÉRIENCE TAGUCHI (non utilisé dans ce mode, conservé)
# ══════════════════════════════════════════════════════════════════════════════
CFG['TAGUCHI'] = dict(
    ENABLE                     = False,
    EPISODES_PER_INSTANCE_DOE  = 5,
    OUTER_INSTANCE_NAMES       = 'ALL',
)
CFG['TAGUCHI_SYNTH'] = dict(
    ENABLE                     = False,
    N_INSTANCES_DOE            = 24,
    EPISODES_PER_INSTANCE_DOE  = 5,
)
# ── Jeu d'actions actif : FIXÉ SUR 'merged2' ────────────────────────────
# Résultat de l'ablation des 3 jeux (voir run_action_space_ablation()) :
# merged2 (2 actions : spatio-temporelle fusionnée + aléatoire) a obtenu le
# meilleur gap moyen vs BKS (+7.55%, contre +8.22% pour full et +9.23% pour
# merged3, mêmes hyperparamètres pour les 3). C'est donc le jeu retenu ici
# pour la validation combinée finale (merged2 + best hyperparamètres + I1 +
# fusion de routes).
CFG['ACTION_SET'] = 'merged2'
CFG['ACTION_ABLATION'] = dict(
    ENABLE = False,
    SETS   = ['full', 'merged3', 'merged2'],
)
# ── Fusion de routes (post-traitement, MODULE 5bis) ─────────────────────
CFG['ROUTE_FUSION'] = dict(
    ENABLE = True,
)
# ── Recherche locale 2-opt + Or-opt (post-traitement, MODULE 5ter) ──────
# Appliquée APRÈS la première fusion, suivie d'une seconde passe de
# fusion (voir run_final_validation()). Voir bandeau en tête de fichier.
CFG['LOCAL_SEARCH_POLISH'] = dict(
    ENABLE       = True,
    MAX_ROUNDS   = 5,
    SEG_LEN_MAX  = 3,
)
TAGUCHI_FACTORS = [
    ('LNS_ITER',         [30, 60, 100]),
    ('K_DESTROY_RATIO',  [0.15, 0.20, 0.24]),
    ('GUROBI_TLIMIT',    [1.0, 2.0, 3.0]),
    ('GUROBI_GAP',       [0.05, 0.03, 0.01]),
    ('MAX_STEPS',        [10, 15, 20]),
    ('EPS_DECAY',        [0.90, 0.95, 0.99]),
    ('VEHICLE_PENALTY',  [0.0, 300.0, 800.0]),
]
GROUP_ORDER = ['C1', 'C2', 'R1', 'R2', 'RC1', 'RC2']
ACTION_NAMES = {
    0: "Destruction spatiale",
    1: "Destruction temporelle",
    2: "Destruction par coût élevé",
    3: "Destruction aléatoire",
}
ACTION_SETS_N = {'full': 4, 'merged3': 3, 'merged2': 2}
ACTION_NAMES_BY_SET = {
    'full':    {0: "Spatiale", 1: "Temporelle", 2: "Coût élevé", 3: "Aléatoire"},
    'merged3': {0: "Spatio-temporelle (fusion)", 1: "Coût élevé", 2: "Aléatoire"},
    'merged2': {0: "Spatio-temporelle (fusion)", 1: "Aléatoire"},
}
SOLOMON_DIR = os.path.join(CFG['OUTDIR'], 'Solomon')
INSTANCE_FILES = sorted(glob.glob(os.path.join(SOLOMON_DIR, '*.txt')))
# ── BKS officiel complet, 56 instances (source : SINTEF / Solomon 1987) ──────
BKS_REFERENCE = {
    'C101': {'vehicles': 10, 'distance': 828.94}, 'C102': {'vehicles': 10, 'distance': 828.94},
    'C103': {'vehicles': 10, 'distance': 828.06}, 'C104': {'vehicles': 10, 'distance': 824.78},
    'C105': {'vehicles': 10, 'distance': 828.94}, 'C106': {'vehicles': 10, 'distance': 828.94},
    'C107': {'vehicles': 10, 'distance': 828.94}, 'C108': {'vehicles': 10, 'distance': 828.94},
    'C109': {'vehicles': 10, 'distance': 828.94},
    'C201': {'vehicles': 3, 'distance': 591.56}, 'C202': {'vehicles': 3, 'distance': 591.56},
    'C203': {'vehicles': 3, 'distance': 591.17}, 'C204': {'vehicles': 3, 'distance': 590.60},
    'C205': {'vehicles': 3, 'distance': 588.88}, 'C206': {'vehicles': 3, 'distance': 588.49},
    'C207': {'vehicles': 3, 'distance': 588.29}, 'C208': {'vehicles': 3, 'distance': 588.32},
    'R101': {'vehicles': 19, 'distance': 1650.80}, 'R102': {'vehicles': 17, 'distance': 1486.12},
    'R103': {'vehicles': 13, 'distance': 1292.68}, 'R104': {'vehicles': 9,  'distance': 1007.31},
    'R105': {'vehicles': 14, 'distance': 1377.11}, 'R106': {'vehicles': 12, 'distance': 1252.03},
    'R107': {'vehicles': 10, 'distance': 1104.66}, 'R108': {'vehicles': 9,  'distance': 960.88},
    'R109': {'vehicles': 11, 'distance': 1194.73}, 'R110': {'vehicles': 10, 'distance': 1118.84},
    'R111': {'vehicles': 10, 'distance': 1096.72}, 'R112': {'vehicles': 9,  'distance': 982.14},
    'R201': {'vehicles': 4, 'distance': 1252.37}, 'R202': {'vehicles': 3, 'distance': 1191.70},
    'R203': {'vehicles': 3, 'distance': 939.50},  'R204': {'vehicles': 2, 'distance': 825.52},
    'R205': {'vehicles': 3, 'distance': 994.43},  'R206': {'vehicles': 3, 'distance': 906.14},
    'R207': {'vehicles': 2, 'distance': 890.61},  'R208': {'vehicles': 2, 'distance': 726.82},
    'R209': {'vehicles': 3, 'distance': 909.16},  'R210': {'vehicles': 3, 'distance': 939.37},
    'R211': {'vehicles': 2, 'distance': 885.71},
    'RC101': {'vehicles': 14, 'distance': 1696.95}, 'RC102': {'vehicles': 12, 'distance': 1554.75},
    'RC103': {'vehicles': 11, 'distance': 1261.67}, 'RC104': {'vehicles': 10, 'distance': 1135.48},
    'RC105': {'vehicles': 13, 'distance': 1629.44}, 'RC106': {'vehicles': 11, 'distance': 1424.73},
    'RC107': {'vehicles': 11, 'distance': 1230.48}, 'RC108': {'vehicles': 10, 'distance': 1139.82},
    'RC201': {'vehicles': 4, 'distance': 1406.94}, 'RC202': {'vehicles': 3, 'distance': 1365.65},
    'RC203': {'vehicles': 3, 'distance': 1049.62}, 'RC204': {'vehicles': 3, 'distance': 798.46},
    'RC205': {'vehicles': 4, 'distance': 1297.65}, 'RC206': {'vehicles': 3, 'distance': 1146.32},
    'RC207': {'vehicles': 3, 'distance': 1061.14}, 'RC208': {'vehicles': 3, 'distance': 828.14},
}
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1  —  LECTURE DES INSTANCES SOLOMON OFFICIELLES  (jeu de TEST)
# ══════════════════════════════════════════════════════════════════════════════
def euclidean(a, b):
    return float(np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2))
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 1bis  —  GÉNÉRATEUR D'INSTANCES SYNTHÉTIQUES  (jeu d'ENTRAÎNEMENT)
# ══════════════════════════════════════════════════════════════════════════════
def _coords_C(n, rng):
    n_cl = max(3, n // 10)
    centers = rng.uniform(15, 85, (n_cl, 2))
    pts = []
    for i in range(n):
        p = rng.normal(centers[i % n_cl], 8)
        pts.append(np.clip(p, 1, 99))
    return np.array(pts)
def _coords_R(n, rng):
    return rng.uniform(1, 99, (n, 2))
def _coords_RC(n, rng):
    h = n // 2
    return np.vstack([_coords_C(h, rng), _coords_R(n - h, rng)])
def generate_instance(n_customers=50, n_vehicles=None, capacity=200,
                      type_='C', seed=42, horizon=1000,
                      vehicle_safety_factor=1.5):
    rng     = np.random.RandomState(seed)
    depot   = np.array([50.0, 50.0])
    gen_fn  = {'C': _coords_C, 'R': _coords_R, 'RC': _coords_RC}[type_]
    coords  = gen_fn(n_customers, rng)
    max_dem = max(1, capacity // 5)
    demands = rng.randint(1, max_dem + 1, n_customers)
    svc     = rng.randint(10, 31, n_customers)
    if n_vehicles is None:
        min_vehicles = int(np.ceil(demands.sum() / capacity))
        n_vehicles   = max(2, int(np.ceil(min_vehicles * vehicle_safety_factor)))
    customers = []
    for i in range(n_customers):
        d        = euclidean(depot, coords[i])
        earliest = max(0.0, round(d + rng.uniform(-15, 15), 1))
        delta    = rng.uniform(30, 80)
        latest   = min(float(horizon), round(earliest + delta, 1))
        latest   = max(latest, earliest + svc[i] + 5)
        latest   = min(latest, float(horizon))
        if d > latest:
            earliest = round(d, 1)
            latest   = min(float(horizon), round(earliest + delta, 1))
            latest   = max(latest, earliest + svc[i] + 5)
        if d + svc[i] + d > horizon:
            svc[i] = max(5, int(horizon - 2 * d - 5))
        customers.append({
            'id':           i + 1,
            'x':            float(round(coords[i][0], 2)),
            'y':            float(round(coords[i][1], 2)),
            'demand':       int(demands[i]),
            'earliest':     float(earliest),
            'latest':       float(latest),
            'service_time': int(svc[i]),
        })
    return {
        'depot': {
            'id': 0, 'x': 50.0, 'y': 50.0, 'demand': 0,
            'earliest': 0.0, 'latest': float(horizon), 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    capacity,
        'horizon':     horizon,
        'type':        type_,
        'seed':        seed,
        'n_customers': n_customers,
    }
def generate_synthetic_training_set(cfg, verbose=True):
    types = ['C', 'R', 'RC']
    instances = []
    if verbose:
        _sec(f"ÉTAPE 1 — GÉNÉRATION DE {cfg['N_SYNTH_INSTANCES']} INSTANCES "
             f"SYNTHÉTIQUES (ENTRAÎNEMENT)")
    for i in range(cfg['N_SYNTH_INSTANCES']):
        t    = types[i % 3]
        seed = i * 7 + 1
        inst = generate_instance(
            n_customers = cfg['SYNTH_N_CUSTOMERS'],
            n_vehicles  = cfg['SYNTH_N_VEHICLES'],
            capacity    = cfg['SYNTH_CAPACITY'],
            horizon     = cfg['SYNTH_HORIZON'],
            type_       = t,
            seed        = seed,
            vehicle_safety_factor = cfg['SYNTH_VEHICLE_SAFETY_FACTOR'],
        )
        inst['name'] = f"SYN-{t}{i+1:03d}"
        instances.append(inst)
        if verbose:
            print(f"  Instance {i+1:03d}/{cfg['N_SYNTH_INSTANCES']} | {inst['name']:<10} "
                  f"type={t:<3} | seed={seed} | véhicules={inst['n_vehicles']}")
    return instances
def parse_solomon_instance(filepath, vehicle_safety_factor=None):
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = [l.rstrip('\n') for l in f]
    name = lines[0].strip()
    idx_veh = next(i for i, l in enumerate(lines) if l.strip() == 'VEHICLE')
    veh_number, veh_capacity = map(int, lines[idx_veh + 2].split())
    idx_cust = next(i for i, l in enumerate(lines) if l.strip() == 'CUSTOMER')
    data_lines = [l for l in lines[idx_cust + 2:] if l.strip()]
    rows = [list(map(float, l.split())) for l in data_lines]
    depot_row = rows[0]
    customers = []
    for r in rows[1:]:
        cid, x, y, dem, ready, due, svc = r
        customers.append({
            'id':           int(cid),
            'x':            x,
            'y':            y,
            'demand':       int(dem),
            'earliest':     ready,
            'latest':       due,
            'service_time': int(svc),
        })
    name_upper = name.upper()
    type_ = ''.join(ch for ch in name_upper if ch.isalpha())
    n_vehicles = veh_number
    if vehicle_safety_factor is not None:
        total_demand = sum(c['demand'] for c in customers)
        min_veh      = int(np.ceil(total_demand / veh_capacity))
        n_vehicles   = min(veh_number, max(2, int(np.ceil(min_veh * vehicle_safety_factor))))
    return {
        'depot': {
            'id': 0, 'x': depot_row[1], 'y': depot_row[2], 'demand': 0,
            'earliest': depot_row[4], 'latest': depot_row[5], 'service_time': 0,
        },
        'customers':   customers,
        'n_vehicles':  n_vehicles,
        'capacity':    veh_capacity,
        'horizon':     depot_row[5],
        'type':        type_,
        'seed':        name_upper,
        'n_customers': len(customers),
        'name':        name_upper,
    }
def load_all_instances(files, vehicle_safety_factor=None):
    if not files:
        raise FileNotFoundError(
            f"Aucun fichier d'instance trouvé dans {SOLOMON_DIR}. "
            f"Vérifiez que le dossier existe et contient les .txt Solomon."
        )
    instances = []
    for fp in files:
        inst = parse_solomon_instance(fp, vehicle_safety_factor=vehicle_safety_factor)
        instances.append(inst)
        print(f"  Chargé : {inst['name']:<8} type={inst['type']:<3} "
              f"clients={inst['n_customers']} véhicules={inst['n_vehicles']} "
              f"capacité={inst['capacity']} horizon={inst['horizon']}")
    return instances
def pick_example_indices(instances, preferred_names=None, max_examples=3):
    preferred_names = preferred_names or []
    by_name = {inst['name']: i for i, inst in enumerate(instances)}
    idxs = [by_name[nm] for nm in preferred_names if nm in by_name]
    if not idxs:
        seen_types = {}
        for i, inst in enumerate(instances):
            if inst['type'] not in seen_types:
                seen_types[inst['type']] = i
            if len(seen_types) >= max_examples:
                break
        idxs = list(seen_types.values())
    if not idxs:
        idxs = list(range(min(max_examples, len(instances))))
    return idxs[:max_examples]
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 2  —  SOLUTION INITIALE PAR LNS
# ══════════════════════════════════════════════════════════════════════════════
def _node_xy(nid, instance):
    if nid == 0:
        return instance['depot']['x'], instance['depot']['y']
    c = instance['customers'][nid - 1]
    return c['x'], c['y']
def _node_info(nid, instance):
    if nid == 0:
        return instance['depot']
    return instance['customers'][nid - 1]
def compute_cost(solution, instance):
    depot = (instance['depot']['x'], instance['depot']['y'])
    total = 0.0
    for route in solution:
        if not route:
            continue
        prev = depot
        for cid in route:
            cur   = _node_xy(cid, instance)
            total += euclidean(prev, cur)
            prev  = cur
        total += euclidean(prev, depot)
    return total
def _route_feasible(route, instance):
    if not route:
        return True
    cap  = instance['capacity']
    load = sum(_node_info(c, instance)['demand'] for c in route)
    if load > cap:
        return False
    depot = instance['depot']
    t     = 0.0
    px, py = depot['x'], depot['y']
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        if t > info['latest']:
            return False
        t      = max(t, info['earliest']) + info['service_time']
        px, py = cx, cy
    t += euclidean((px, py), (depot['x'], depot['y']))
    return t <= depot['latest']
def is_feasible(solution, instance):
    return all(_route_feasible(r, instance) for r in solution)
def solution_stats(solution, instance):
    cap        = instance['capacity']
    total_cost = compute_cost(solution, instance)
    n_vehicles = sum(1 for r in solution if r)
    loads      = []
    total_wait = 0.0
    for route in solution:
        if not route:
            continue
        load = sum(_node_info(c, instance)['demand'] for c in route)
        loads.append(load / cap)
        depot  = instance['depot']
        t      = 0.0
        px, py = depot['x'], depot['y']
        for cid in route:
            cx, cy = _node_xy(cid, instance)
            t     += euclidean((px, py), (cx, cy))
            info   = _node_info(cid, instance)
            total_wait += max(0.0, info['earliest'] - t)
            t      = max(t, info['earliest']) + info['service_time']
            px, py = cx, cy
    return {
        'total_cost':         round(total_cost, 2),
        'n_vehicles':         n_vehicles,
        'avg_load_ratio':     round(np.mean(loads) if loads else 0.0, 3),
        'total_wait_time':    round(total_wait, 2),
        'n_customers_served': sum(len(r) for r in solution),
    }
def _insertion_cost(cid, pos, route, instance):
    depot   = (instance['depot']['x'], instance['depot']['y'])
    nodes   = [0] + route + [0]
    prev_id = nodes[pos]
    next_id = nodes[pos + 1]
    px, py  = _node_xy(prev_id, instance) if prev_id != 0 else depot
    nx, ny  = _node_xy(next_id, instance) if next_id != 0 else depot
    cx, cy  = _node_xy(cid, instance)
    return (euclidean((px, py), (cx, cy)) + euclidean((cx, cy), (nx, ny))
            - euclidean((px, py), (nx, ny)))
def _try_insert(cid, route, instance):
    best_cost, best_pos = float('inf'), -1
    for pos in range(len(route) + 1):
        candidate = route[:pos] + [cid] + route[pos:]
        if _route_feasible(candidate, instance):
            cost = _insertion_cost(cid, pos, route, instance)
            if cost < best_cost:
                best_cost, best_pos = cost, pos
    return best_cost, best_pos
def _repair_greedy(removed, partial, instance):
    routes = [list(r) for r in partial]
    for cid in removed:
        best_cost, best_r, best_pos = float('inf'), -1, -1
        for ri, route in enumerate(routes):
            cost, pos = _try_insert(cid, route, instance)
            if pos >= 0 and cost < best_cost:
                best_cost, best_r, best_pos = cost, ri, pos
        if best_r >= 0:
            routes[best_r].insert(best_pos, cid)
        else:
            routes.append([cid])
    return routes
def _route_service_starts(route, instance):
    depot  = instance['depot']
    t      = 0.0
    px, py = depot['x'], depot['y']
    starts = []
    for cid in route:
        cx, cy = _node_xy(cid, instance)
        t     += euclidean((px, py), (cx, cy))
        info   = _node_info(cid, instance)
        t      = max(t, info['earliest'])
        starts.append(t)
        t     += info['service_time']
        px, py = cx, cy
    return starts
def _build_initial_i1(instance, mu=1.0, lam=2.0, alpha1=1.0, alpha2=0.0):
    def d(a, b):
        return euclidean(_node_xy(a, instance), _node_xy(b, instance))
    unrouted = [c['id'] for c in instance['customers']]
    d0 = {cid: d(0, cid) for cid in unrouted}
    routes = []
    while unrouted:
        seed = max(unrouted, key=lambda c: d0[c])
        route = [seed]
        unrouted.remove(seed)
        while True:
            times_before = _route_service_starts(route, instance)
            nodes = [0] + route + [0]
            best_u, best_pos, best_c1, best_c2 = None, None, None, -float('inf')
            for u in unrouted:
                u_best_cost, u_best_pos = None, None
                cx, cy = _node_xy(u, instance)
                info_u = _node_info(u, instance)
                for pos in range(len(route) + 1):
                    i_id, j_id = nodes[pos], nodes[pos + 1]
                    candidate = route[:pos] + [u] + route[pos:]
                    if not _route_feasible(candidate, instance):
                        continue
                    dij = d(i_id, j_id)
                    c11 = d(i_id, u) + d(u, j_id) - mu * dij
                    c12 = 0.0
                    if alpha2 != 0.0 and j_id != 0:
                        if pos == 0:
                            depart_i, px, py = 0.0, instance['depot']['x'], instance['depot']['y']
                        else:
                            info_i = _node_info(i_id, instance)
                            depart_i = times_before[pos - 1] + info_i['service_time']
                            px, py   = _node_xy(i_id, instance)
                        arrival_u = depart_i + euclidean((px, py), (cx, cy))
                        b_u       = max(arrival_u, info_u['earliest'])
                        depart_u  = b_u + info_u['service_time']
                        jx, jy    = _node_xy(j_id, instance)
                        arrival_j = depart_u + euclidean((cx, cy), (jx, jy))
                        info_j    = _node_info(j_id, instance)
                        b_ju      = max(arrival_j, info_j['earliest'])
                        c12       = b_ju - times_before[pos]
                    c1 = alpha1 * c11 + alpha2 * c12
                    if u_best_cost is None or c1 < u_best_cost:
                        u_best_cost, u_best_pos = c1, pos
                if u_best_cost is None:
                    continue
                c2 = lam * d0[u] - u_best_cost
                if c2 > best_c2:
                    best_c2, best_u, best_pos, best_c1 = c2, u, u_best_pos, u_best_cost
            if best_u is None:
                break
            route.insert(best_pos, best_u)
            unrouted.remove(best_u)
        routes.append(route)
    return routes
def _lns_improve(solution, instance, n_iter=50, k=10, seed=0):
    rng       = np.random.RandomState(seed)
    current   = [list(r) for r in solution]
    best      = copy.deepcopy(current)
    best_cost = compute_cost(best, instance)
    for _ in range(n_iter):
        all_c   = [c for r in current for c in r]
        k_      = min(k, max(1, len(all_c) - 1))
        removed = list(rng.choice(all_c, k_, replace=False))
        removed_set = set(removed)
        partial = [[c for c in r if c not in removed_set] for r in current]
        partial = [r for r in partial if r]
        candidate = _repair_greedy(removed, partial, instance)
        c_cost    = compute_cost(candidate, instance)
        if c_cost < best_cost and is_feasible(candidate, instance):
            best, best_cost = copy.deepcopy(candidate), c_cost
        current = candidate
    return best
def get_initial_solution(instance, lns_iter=50, lns_k=10):
    sol = _build_initial_i1(
        instance,
        mu=CFG.get('I1_MU', 1.0), lam=CFG.get('I1_LAMBDA', 2.0),
        alpha1=CFG.get('I1_ALPHA1', 1.0), alpha2=CFG.get('I1_ALPHA2', 0.0),
    )
    sol = _lns_improve(sol, instance, n_iter=lns_iter, k=lns_k)
    sol = [r for r in sol if r]
    return sol, solution_stats(sol, instance)
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 3  —  ENVIRONNEMENT VRPTW
# ══════════════════════════════════════════════════════════════════════════════
class VRPTWEnv:
    def __init__(self, instance, initial_solution,
                 K=15, max_steps=20,
                 gurobi_tlimit=2.0, gurobi_gap=0.05,
                 vehicle_penalty=0.0,
                 action_set='full',
                 verbose=False):
        self.instance    = instance
        self.initial_sol = copy.deepcopy(initial_solution)
        self.K           = K
        self.max_steps   = max_steps
        self.tlimit      = gurobi_tlimit
        self.gap         = gurobi_gap
        self.vehicle_penalty = vehicle_penalty
        self.verbose     = verbose
        self.obs_dim     = 7
        self.action_set  = action_set
        self.n_actions   = ACTION_SETS_N.get(action_set, 4)
        self._dist = {}
        depot = (instance['depot']['x'], instance['depot']['y'])
        for c in instance['customers']:
            d = euclidean(depot, (c['x'], c['y']))
            self._dist[(0, c['id'])] = d
            self._dist[(c['id'], 0)] = d
        for ci in instance['customers']:
            for cj in instance['customers']:
                self._dist[(ci['id'], cj['id'])] = euclidean(
                    (ci['x'], ci['y']), (cj['x'], cj['y']))
        self.reset()
    def _d(self, a, b):
        return self._dist.get((a, b), 0.0)
    def _info(self, nid):
        return _node_info(nid, self.instance)
    def reset(self, start_solution=None):
        if start_solution is not None:
            self.solution = copy.deepcopy(start_solution)
        else:
            self.solution = copy.deepcopy(self.initial_sol)
        self.current_cost     = compute_cost(self.solution, self.instance)
        self.best_cost        = self.current_cost
        self.initial_cost     = compute_cost(self.initial_sol, self.instance)
        self.step_count       = 0
        self.steps_no_improve = 0
        self._action_history  = []
        return self._obs()
    def _obs(self):
        stats = solution_stats(self.solution, self.instance)
        ic    = max(self.initial_cost, 1.0)
        return np.array([
            self.current_cost / ic,
            stats['n_vehicles'] / max(self.instance['n_vehicles'], 1),
            stats['avg_load_ratio'],
            stats['total_wait_time'] / max(ic * 0.5, 1.0),
            self.steps_no_improve / self.max_steps,
            self.best_cost / ic,
            1.0,
        ], dtype=np.float32)
    def _spatial_scores(self, pivot, all_c):
        px, py = self._info(pivot)['x'], self._info(pivot)['y']
        return {c: euclidean((px, py), (self._info(c)['x'], self._info(c)['y']))
                for c in all_c}
    def _temporal_scores(self, pivot, all_c):
        e_pivot = self._info(pivot)['earliest']
        return {c: abs(self._info(c)['earliest'] - e_pivot) for c in all_c}
    def _cost_scores(self):
        savings = {}
        for r in self.solution:
            if not r:
                continue
            nodes = [0] + r + [0]
            for i, cid in enumerate(r):
                prev, nxt    = nodes[i], nodes[i + 2]
                savings[cid] = (self._d(prev, cid) + self._d(cid, nxt)
                                 - self._d(prev, nxt))
        return savings
    def _spatiotemporal_destroy(self, rng, all_c, K):
        pivot = int(rng.choice(all_c))
        sp = self._spatial_scores(pivot, all_c)
        tp = self._temporal_scores(pivot, all_c)
        sp_max = max(sp.values()) or 1.0
        tp_max = max(tp.values()) or 1.0
        combined = {c: 0.5 * (sp[c] / sp_max) + 0.5 * (tp[c] / tp_max) for c in all_c}
        return sorted(all_c, key=lambda c: combined[c])[:K]
    def _destroy(self, action):
        all_c = [c for r in self.solution for c in r]
        if not all_c:
            return [], copy.deepcopy(self.solution)
        K   = min(self.K, len(all_c))
        rng = np.random.RandomState(self.step_count + action * 100)
        if self.action_set == 'full':
            if action == 0:
                pivot     = int(rng.choice(all_c))
                scores    = self._spatial_scores(pivot, all_c)
                destroyed = sorted(all_c, key=lambda c: scores[c])[:K]
            elif action == 1:
                pivot     = int(rng.choice(all_c))
                scores    = self._temporal_scores(pivot, all_c)
                destroyed = sorted(all_c, key=lambda c: scores[c])[:K]
            elif action == 2:
                savings   = self._cost_scores()
                destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
            else:
                idx       = rng.choice(len(all_c), K, replace=False)
                destroyed = [all_c[i] for i in idx]
        elif self.action_set == 'merged3':
            if action == 0:
                destroyed = self._spatiotemporal_destroy(rng, all_c, K)
            elif action == 1:
                savings   = self._cost_scores()
                destroyed = sorted(all_c, key=lambda c: -savings.get(c, 0))[:K]
            else:
                idx       = rng.choice(len(all_c), K, replace=False)
                destroyed = [all_c[i] for i in idx]
        elif self.action_set == 'merged2':
            if action == 0:
                destroyed = self._spatiotemporal_destroy(rng, all_c, K)
            else:
                idx       = rng.choice(len(all_c), K, replace=False)
                destroyed = [all_c[i] for i in idx]
        else:
            raise ValueError(f"CFG['ACTION_SET'] inconnu : {self.action_set!r} "
                              f"(attendu 'full' / 'merged3' / 'merged2')")
        destroyed_set = set(destroyed)
        partial = [[c for c in r if c not in destroyed_set]
                   for r in self.solution]
        partial = [r for r in partial if r]
        return list(destroyed), partial
    def _repair_gurobi(self, destroyed, partial):
        if not destroyed:
            return copy.deepcopy(self.solution)
        inst    = self.instance
        depot   = inst['depot']
        horizon = inst['horizon']
        cap     = inst['capacity']
        nodes   = [0] + destroyed
        N       = len(nodes)
        V       = min(len(destroyed), inst['n_vehicles'])
        env, m = None, None
        try:
            env = gp.Env(empty=True)
            env.setParam('OutputFlag', 0)
            env.start()
            m = gp.Model(env=env)
            if self.tlimit is not None:
                m.setParam('TimeLimit', self.tlimit)
            m.setParam('MIPGap',     self.gap)
            m.setParam('OutputFlag', 0)
            # ── Reproductibilité (voir bandeau en tête de fichier) ──────
            # Sans ces deux lignes, Gurobi peut retourner une solution
            # différente d'un run à l'autre sur un MÊME modèle (recherche
            # branch-and-bound multi-thread non déterministe, surtout en
            # heurtant TimeLimit sans avoir prouvé l'optimalité) — même
            # avec des seeds Python/NumPy/PyTorch identiques par ailleurs.
            m.setParam('Threads', CFG.get('GUROBI_THREADS', 1))
            m.setParam('Seed',    CFG.get('GUROBI_SEED', 0))
            x = m.addVars(N, N, V, vtype=GRB.BINARY, name='x')
            t = m.addVars(N, V, lb=0, ub=horizon,    name='t')
            m.setObjective(
                gp.quicksum(
                    self._d(nodes[i], nodes[j]) * x[i, j, v]
                    for i in range(N) for j in range(N) for v in range(V)
                    if i != j
                )
                + self.vehicle_penalty * gp.quicksum(
                    x[0, j, v] for j in range(1, N) for v in range(V)
                ),
                GRB.MINIMIZE
            )
            for i in range(1, N):
                m.addConstr(
                    gp.quicksum(x[i, j, v]
                                for j in range(N) for v in range(V)
                                if i != j) == 1
                )
            for v in range(V):
                for i in range(1, N):
                    m.addConstr(
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j) ==
                        gp.quicksum(x[j, i, v] for j in range(N) if i != j)
                    )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(x[0, j, v] for j in range(1, N)) <= 1
                )
            for v in range(V):
                m.addConstr(
                    gp.quicksum(
                        inst['customers'][nodes[i] - 1]['demand'] *
                        gp.quicksum(x[i, j, v] for j in range(N) if i != j)
                        for i in range(1, N)
                    ) <= cap
                )
            M = horizon + 1000
            for v in range(V):
                m.addConstr(t[0, v] == 0)
                for i in range(N):
                    ni  = nodes[i]
                    ei  = depot['earliest'] if ni == 0 else inst['customers'][ni-1]['earliest']
                    li  = depot['latest']   if ni == 0 else inst['customers'][ni-1]['latest']
                    sti = depot['service_time'] if ni == 0 else inst['customers'][ni-1]['service_time']
                    m.addConstr(t[i, v] >= ei)
                    m.addConstr(t[i, v] <= li)
                    for j in range(N):
                        if i == j:
                            continue
                        m.addConstr(
                            t[j, v] >= t[i, v] + sti + self._d(ni, nodes[j])
                            - M * (1 - x[i, j, v])
                        )
            m.optimize()
            if m.SolCount > 0:
                eps        = 0.5
                new_routes = []
                for v in range(V):
                    if sum(x[0, j, v].X for j in range(1, N)) < eps:
                        continue
                    route_v, cur = [], 0
                    for _ in range(N):
                        nxt = next((j for j in range(N)
                                    if j != cur and x[cur, j, v].X > eps), None)
                        if nxt is None or nxt == 0:
                            break
                        route_v.append(nodes[nxt])
                        cur = nxt
                    if route_v:
                        new_routes.append(route_v)
                full = [r for r in list(partial) + new_routes if r]
                return full
            else:
                raise ValueError("Gurobi : aucune solution trouvée")
        except Exception as e:
            if self.verbose:
                print(f"    [Gurobi→Greedy fallback] {e}")
            return _repair_greedy(destroyed, partial, self.instance)
        finally:
            if m is not None:
                m.dispose()
            if env is not None:
                env.dispose()
    def step(self, action):
        old_cost            = self.current_cost
        destroyed, partial  = self._destroy(action)
        new_sol             = self._repair_gurobi(destroyed, partial)
        new_cost            = compute_cost(new_sol, self.instance)
        feasible            = is_feasible(new_sol, self.instance)
        if feasible:
            self.solution     = new_sol
            self.current_cost = new_cost
        else:
            new_cost = old_cost * 1.001
        improvement = old_cost - self.current_cost
        if improvement > 0:
            self.best_cost        = min(self.best_cost, self.current_cost)
            self.steps_no_improve = 0
        else:
            self.steps_no_improve += 1
        reward = improvement / max(self.initial_cost, 1.0)
        self.step_count += 1
        self._action_history.append(action)
        done = (self.step_count >= self.max_steps)
        info = {
            'cost':          self.current_cost,
            'best_cost':     self.best_cost,
            'action_taken':  action,
            'action_name':   ACTION_NAMES_BY_SET.get(self.action_set, ACTION_NAMES).get(action, str(action)),
            'improvement':   improvement,
            'destroyed_k':   len(destroyed),
            'destroyed_ids': [int(d) for d in destroyed[:5]],
            'feasible':      feasible,
        }
        return self._obs(), float(reward), done, info
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 4  —  AGENT DQN
# ══════════════════════════════════════════════════════════════════════════════
class ReplayBuffer:
    def __init__(self, capacity=2000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((
            np.array(s,  dtype=np.float32), int(a), float(r),
            np.array(s2, dtype=np.float32), float(done),
        ))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.FloatTensor(np.stack(s)), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(np.stack(s2)),
                torch.FloatTensor(d))
    def __len__(self):
        return len(self.buf)
class DQNNet(nn.Module):
    def __init__(self, obs_dim=7, n_actions=4, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim,  hidden),  nn.ReLU(),
            nn.Linear(hidden,   hidden),  nn.ReLU(),
            nn.Linear(hidden,   n_actions),
        )
    def forward(self, x):
        return self.net(x)
class DQNAgent:
    def __init__(self, obs_dim=7, n_actions=4, hidden=64,
                 lr=1e-3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_size=2000, batch_size=32,
                 target_update_freq=10, seed=0):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        self.n_actions          = n_actions
        self.gamma              = gamma
        self.epsilon            = epsilon_start
        self.epsilon_end        = epsilon_end
        self.epsilon_decay      = epsilon_decay
        self.batch_size         = batch_size
        self.target_update_freq = target_update_freq
        self.step_count         = 0
        self.q_net      = DQNNet(obs_dim, n_actions, hidden)
        self.target_net = DQNNet(obs_dim, n_actions, hidden)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)
        self.loss_fn   = nn.MSELoss()
    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state).unsqueeze(0))
            return int(q.argmax().item())
    def store(self, s, a, r, s2, done):
        self.buffer.push(s, a, r, s2, done)
    def learn(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        s, a, r, s2, done = self.buffer.sample(self.batch_size)
        q_vals = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q  = self.target_net(s2).max(1)[0]
            targets = r + self.gamma * next_q * (1 - done)
        loss = self.loss_fn(q_vals, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.optimizer.step()
        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.update_target()
        return float(loss.item())
    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    def save(self, path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        torch.save({'q_net': self.q_net.state_dict(),
                    'target_net': self.target_net.state_dict(),
                    'epsilon': self.epsilon, 'step_count': self.step_count}, path)
        print(f"    [DQN] Modèle sauvegardé → {path}")
    def load(self, path):
        ckpt = torch.load(path, map_location='cpu')
        self.q_net.load_state_dict(ckpt['q_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.epsilon    = ckpt.get('epsilon', self.epsilon)
        self.step_count = ckpt.get('step_count', self.step_count)
        print(f"    [DQN] Modèle rechargé ← {path}")
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5  —  ENTRAÎNEMENT DU DQN  (ÉTAPE 1, sur les instances SYNTHÉTIQUES)
# ══════════════════════════════════════════════════════════════════════════════
BANNER = "═" * 72
def _sec(title):
    print(f"\n{BANNER}\n  {title}\n{BANNER}")
def train_on_instances(instances, episodes_per_instance=10, save_model=True, verbose=True):
    n = len(instances)
    if verbose:
        _sec(f"ÉTAPE 1 — LNS initial pour les {n} instances d'entraînement (synthétiques)")
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        if verbose:
            print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
                  f"| véhicules={stats['n_vehicles']}")
    if verbose:
        _sec("ÉTAPE 1 — Entraînement DQN sur le synthétique")
    action_set = CFG.get('ACTION_SET', 'full')
    n_actions_active = ACTION_SETS_N.get(action_set, 4)
    if verbose:
        print(f"  Jeu d'actions actif : '{action_set}' ({n_actions_active} actions)")
    agent = DQNAgent(
        obs_dim=7, n_actions=n_actions_active, hidden=CFG['HIDDEN'], lr=CFG['LR'],
        gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
        epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
        buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
        target_update_freq=CFG['TARGET_UPDATE'],
    )
    training_log   = []
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       action_set=action_set,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        if verbose:
            print(f"\n  {'═'*50}\n  [TRAIN] INSTANCE {inst['name']}  type={inst['type']}"
                  f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)
                next_obs, reward, done, info = env.step(action)
                agent.store(obs, action, reward, next_obs, float(done))
                agent.learn()
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            agent.decay_epsilon()
            training_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            if verbose:
                imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
                print(f"  [TRAIN] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                      f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        if verbose:
            print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    if save_model:
        model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
        agent.save(model_path)
    return agent, training_log, initial_costs, best_costs, best_solutions, instance_times, initial_sols
def evaluate_frozen_on_instances(instances, agent, episodes_per_instance=3):
    n = len(instances)
    _sec(f"ÉTAPE 2 — ÉVALUATION GELÉE (TEST) sur {n} instances Solomon "
         f"— aucun apprentissage, epsilon=0")
    action_set = CFG.get('ACTION_SET', 'full')
    print(f"  Jeu d'actions actif : '{action_set}' ({ACTION_SETS_N.get(action_set, 4)} actions)")
    saved_epsilon = agent.epsilon
    agent.epsilon = 0.0
    initial_sols, initial_stats = [], []
    for i, inst in enumerate(instances):
        sol, stats = get_initial_solution(inst, lns_iter=CFG['LNS_ITER'], lns_k=CFG['LNS_K'])
        initial_sols.append(sol)
        initial_stats.append(stats)
        print(f"  {inst['name']:<10} | coût LNS={stats['total_cost']:.2f} "
              f"| véhicules={stats['n_vehicles']}")
    initial_costs  = [initial_stats[i]['total_cost'] for i in range(n)]
    best_costs     = list(initial_costs)
    best_solutions = list(initial_sols)
    instance_times = [0.0] * n
    test_log       = []
    episode_global = 0
    vehicle_penalty = CFG.get('VEHICLE_PENALTY', 0.0)
    for inst_idx, inst in enumerate(instances):
        env = VRPTWEnv(inst, initial_sols[inst_idx], K=CFG['K_DESTROY'],
                       max_steps=CFG['MAX_STEPS'], gurobi_tlimit=CFG['GUROBI_TLIMIT'],
                       gurobi_gap=CFG['GUROBI_GAP'], vehicle_penalty=vehicle_penalty,
                       action_set=action_set,
                       verbose=False)
        current_start = None
        t_inst_start  = time.time()
        print(f"\n  {'═'*50}\n  [TEST] INSTANCE {inst['name']}  type={inst['type']}"
              f"  coût LNS={initial_costs[inst_idx]:.2f}\n  {'═'*50}")
        for ep_local in range(episodes_per_instance):
            episode_global += 1
            obs  = env.reset(start_solution=current_start)
            done = False
            ep_reward = 0.0
            ep_costs  = [env.current_cost]
            while not done:
                action = agent.select_action(obs)
                next_obs, reward, done, info = env.step(action)
                ep_reward += reward
                ep_costs.append(env.current_cost)
                obs = next_obs
            current_start = copy.deepcopy(env.solution)
            if env.best_cost < best_costs[inst_idx]:
                best_costs[inst_idx]     = env.best_cost
                best_solutions[inst_idx] = copy.deepcopy(env.solution)
            test_log.append({
                'episode_global': episode_global, 'instance_id': inst_idx,
                'instance_name': inst['name'],
                'ep_local': ep_local + 1, 'total_reward': ep_reward,
                'start_cost': ep_costs[0], 'final_cost': env.current_cost,
                'best_cost': env.best_cost, 'epsilon': agent.epsilon,
                'cost_trace': [float(c) for c in ep_costs],
            })
            imp = (initial_costs[inst_idx] - env.best_cost) / initial_costs[inst_idx] * 100
            print(f"  [TEST] Ep {ep_local+1:02d}/{episodes_per_instance} | reward={ep_reward:+.4f} "
                  f"| meilleur={env.best_cost:.2f} | amélioration={imp:.2f}%")
        instance_times[inst_idx] = time.time() - t_inst_start
        print(f"  ⏱ Temps {inst['name']} : {instance_times[inst_idx]:.1f}s")
    agent.epsilon = saved_epsilon
    return initial_costs, best_costs, best_solutions, instance_times, initial_sols, test_log
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5bis  —  FUSION DE ROUTES (post-traitement, APRÈS DQN+Gurobi)
# ══════════════════════════════════════════════════════════════════════════════
def _route_total_cost(route, instance):
    return compute_cost([route], instance) if route else 0.0
def _try_merge_two_routes(route_a, route_b, instance):
    cap = instance['capacity']
    load_a = sum(_node_info(c, instance)['demand'] for c in route_a)
    load_b = sum(_node_info(c, instance)['demand'] for c in route_b)
    if load_a + load_b > cap:
        return None, None
    best_route, best_cost = None, float('inf')
    for candidate in (route_a + route_b, route_b + route_a):
        if _route_feasible(candidate, instance):
            cost = _route_total_cost(candidate, instance)
            if cost < best_cost:
                best_route, best_cost = candidate, cost
    if best_route is None:
        return None, None
    return best_route, best_cost
def route_merge_postprocess(solution, instance, verbose=False):
    routes = [list(r) for r in solution if r]
    cost_before  = compute_cost(routes, instance)
    n_veh_before = len(routes)
    n_merges = 0
    improved = True
    while improved and len(routes) > 1:
        improved = False
        best_pair, best_merged_route, best_delta = None, None, -1e-6
        n = len(routes)
        for i in range(n):
            for j in range(i + 1, n):
                cost_before_pair = (_route_total_cost(routes[i], instance)
                                     + _route_total_cost(routes[j], instance))
                merged_route, cost_after_pair = _try_merge_two_routes(routes[i], routes[j], instance)
                if merged_route is None:
                    continue
                delta = cost_after_pair - cost_before_pair
                if delta < best_delta:
                    best_delta = delta
                    best_pair  = (i, j)
                    best_merged_route = merged_route
        if best_pair is not None:
            i, j = best_pair
            routes = [r for k, r in enumerate(routes) if k not in (i, j)] + [best_merged_route]
            n_merges += 1
            improved = True
            if verbose:
                print(f"    [Fusion routes] paire ({i},{j}) fusionnée "
                      f"(Δcoût={best_delta:+.2f}) → {len(routes)} véhicules restants")
    cost_after  = compute_cost(routes, instance)
    n_veh_after = len(routes)
    stats = {
        'n_merges':      n_merges,
        'n_veh_before':  n_veh_before,
        'n_veh_after':   n_veh_after,
        'cost_before':   round(cost_before, 2),
        'cost_after':    round(cost_after, 2),
    }
    return routes, stats
def route_merge_postprocess_batch(instances, solutions, verbose=True):
    _sec("FUSION DE ROUTES — POST-TRAITEMENT (APRÈS DQN+GUROBI, SUR LE TEST)")
    new_solutions, new_costs, all_stats = [], [], []
    total_veh_saved = 0
    for inst, sol in zip(instances, solutions):
        merged_sol, stats = route_merge_postprocess(sol, inst, verbose=False)
        new_solutions.append(merged_sol)
        new_costs.append(stats['cost_after'])
        all_stats.append(stats)
        veh_saved = stats['n_veh_before'] - stats['n_veh_after']
        total_veh_saved += veh_saved
        if verbose:
            tag = f"  {inst['name']:<10} | fusions={stats['n_merges']} | " \
                  f"véhicules {stats['n_veh_before']}→{stats['n_veh_after']} " \
                  f"| coût {stats['cost_before']:.2f}→{stats['cost_after']:.2f}"
            print(tag + ("  ⚠ coût augmenté (ne devrait pas arriver)"
                         if stats['cost_after'] > stats['cost_before'] + 1e-6 else ""))
    if verbose:
        print(f"\n  → Total véhicules économisés par fusion : {total_veh_saved} "
              f"sur {len(instances)} instances")
    return new_solutions, new_costs, all_stats
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 5ter  —  RECHERCHE LOCALE 2-OPT + OR-OPT (post-traitement, TEST)
# ══════════════════════════════════════════════════════════════════════════════
def _two_opt_route(route, instance, max_sweeps=20):
    """
    2-opt intra-route, best-improvement : à chaque balayage, cherche le
    renversement de segment [i:j+1] qui réduit le plus le coût de LA route,
    tout en restant faisable (capacité + fenêtres de temps). Répété jusqu'à
    convergence ou max_sweeps. Ne dégrade jamais le coût de la route.
    """
    if len(route) < 3:
        return list(route)
    cur      = list(route)
    cur_cost = _route_total_cost(cur, instance)
    for _ in range(max_sweeps):
        n = len(cur)
        best_delta, best_i, best_j = -1e-6, -1, -1
        for i in range(n - 1):
            for j in range(i + 1, n):
                cand = cur[:i] + cur[i:j + 1][::-1] + cur[j + 1:]
                if not _route_feasible(cand, instance):
                    continue
                delta = _route_total_cost(cand, instance) - cur_cost
                if delta < best_delta:
                    best_delta, best_i, best_j = delta, i, j
        if best_i < 0:
            break
        cur       = cur[:best_i] + cur[best_i:best_j + 1][::-1] + cur[best_j + 1:]
        cur_cost += best_delta
    return cur
def _or_opt_pass(solution, instance, seg_len_max=3, max_sweeps=20):
    """
    Or-opt best-improvement : déplace un segment de 1 à seg_len_max
    clients consécutifs (au sein d'une route ou vers une autre route) à la
    meilleure position faisable trouvée, uniquement si cela réduit le coût
    TOTAL de la solution. Répété jusqu'à convergence ou max_sweeps. Comme
    pour le 2-opt, aucun mouvement dégradant n'est jamais accepté.
    """
    routes = [list(r) for r in solution if r]
    for _ in range(max_sweeps):
        n_routes = len(routes)
        if n_routes == 0:
            break
        best_gain, best_move = 1e-6, None
        for src in range(n_routes):
            r_src = routes[src]
            for seg_len in range(1, min(seg_len_max, len(r_src)) + 1):
                for start in range(len(r_src) - seg_len + 1):
                    segment      = r_src[start:start + seg_len]
                    remainder    = r_src[:start] + r_src[start + seg_len:]
                    removed_cost = (_route_total_cost(r_src, instance)
                                     - _route_total_cost(remainder, instance))
                    for dst in range(n_routes):
                        r_dst = remainder if dst == src else routes[dst]
                        for pos in range(len(r_dst) + 1):
                            if dst == src and start <= pos <= start + seg_len:
                                continue  # réinsertion à la même place : pas un mouvement
                            candidate = r_dst[:pos] + segment + r_dst[pos:]
                            if not _route_feasible(candidate, instance):
                                continue
                            added_cost = (_route_total_cost(candidate, instance)
                                           - _route_total_cost(r_dst, instance))
                            gain = removed_cost - added_cost
                            if gain > best_gain:
                                best_gain = gain
                                best_move = (src, dst, remainder, candidate)
        if best_move is None:
            break
        src, dst, remainder, candidate = best_move
        if dst == src:
            routes[src] = candidate
        else:
            routes[src] = remainder
            routes[dst] = candidate
        routes = [r for r in routes if r]
    return routes
def local_search_polish(solution, instance, max_rounds=5, seg_len_max=3):
    """
    Alterne 2-opt intra-route et Or-opt inter/intra-route, `max_rounds`
    fois ou jusqu'à convergence (coût inchangé d'un round à l'autre).
    Appliquée en post-traitement APRÈS la fusion de routes, sur les
    solutions gelées du TEST — jamais pendant l'entraînement.
    """
    routes       = [list(r) for r in solution if r]
    cost_before  = compute_cost(routes, instance)
    n_veh_before = len(routes)
    prev_cost = None
    for _ in range(max_rounds):
        routes     = [_two_opt_route(r, instance) for r in routes]
        routes     = _or_opt_pass(routes, instance, seg_len_max=seg_len_max)
        routes     = [r for r in routes if r]
        cur_cost   = compute_cost(routes, instance)
        if prev_cost is not None and cur_cost >= prev_cost - 1e-6:
            break
        prev_cost = cur_cost
    cost_after = compute_cost(routes, instance)
    stats = {
        'cost_before':  round(cost_before, 2),
        'cost_after':   round(cost_after, 2),
        'n_veh_before': n_veh_before,
        'n_veh_after':  len(routes),
    }
    return routes, stats
def local_search_polish_batch(instances, solutions, verbose=True,
                               max_rounds=5, seg_len_max=3):
    _sec("RECHERCHE LOCALE 2-OPT + OR-OPT — POST-TRAITEMENT (APRÈS FUSION, SUR LE TEST)")
    new_solutions, new_costs, all_stats = [], [], []
    total_saved = 0.0
    for inst, sol in zip(instances, solutions):
        polished_sol, stats = local_search_polish(
            sol, inst, max_rounds=max_rounds, seg_len_max=seg_len_max)
        new_solutions.append(polished_sol)
        new_costs.append(stats['cost_after'])
        all_stats.append(stats)
        saved = stats['cost_before'] - stats['cost_after']
        total_saved += saved
        if verbose:
            print(f"  {inst['name']:<10} | coût {stats['cost_before']:.2f}→{stats['cost_after']:.2f} "
                  f"(Δ={-saved:+.2f}) | véhicules {stats['n_veh_before']}→{stats['n_veh_after']}")
    if verbose:
        print(f"\n  → Distance totale économisée par recherche locale : {total_saved:.2f} "
              f"sur {len(instances)} instances")
    return new_solutions, new_costs, all_stats
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6  —  TABLEAU FINAL : SITUATION 1 vs BKS
# ══════════════════════════════════════════════════════════════════════════════
def build_bks_comparison_table(instances, initial_costs, best_costs, best_solutions,
                                instance_times, csv_path=None, latex_path=None):
    _sec("TABLEAU FINAL — SITUATION 1 vs BKS")
    rows = []
    for i, inst in enumerate(instances):
        name     = inst['name']
        bks      = BKS_REFERENCE.get(name)
        bks_dist = bks['distance'] if bks else None
        n_veh_s1 = sum(1 for r in best_solutions[i] if r)
        row = {
            'instance':             name,
            'type':                 inst['type'],
            'bks_vehicules':        bks['vehicles'] if bks else None,
            'bks_distance':         bks_dist,
            'n_vehicules_situation1': n_veh_s1,
            'distance_situation1':  round(best_costs[i], 2),
            'temps_s_situation1':   round(instance_times[i], 2),
            'gap_%_situation1_bks': round((best_costs[i] - bks_dist) / bks_dist * 100, 2) if bks_dist else None,
        }
        rows.append(row)
    print(f"\n  {'Instance':<9} {'BKS Vhc':>8} {'BKS Dist':>10} {'Vhc S1':>7} "
          f"{'Dist. S1':>10} {'Temps S1':>10} {'Gap %':>8}")
    print("  " + "─" * 70)
    for r in rows:
        bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "—"
        bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "—"
        g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "—"
        print(f"  {r['instance']:<9} {bks_v:>8} {bks_d:>10} "
              f"{r['n_vehicules_situation1']:>7} {r['distance_situation1']:>10.2f} "
              f"{r['temps_s_situation1']:>10.1f} {g1:>8}")
    valid_gaps = [r['gap_%_situation1_bks'] for r in rows if r['gap_%_situation1_bks'] is not None]
    gap_moyen  = float(np.mean(valid_gaps)) if valid_gaps else None
    if gap_moyen is not None:
        print(f"\n  Gap moyen vs BKS (Situation 1) : {gap_moyen:+.2f}%  "
              f"({len(valid_gaps)}/{len(rows)} instances avec BKS connu)")
        print(f"  Gap min : {min(valid_gaps):+.2f}%  |  Gap max : {max(valid_gaps):+.2f}%")
    else:
        print("\n  Gap moyen vs BKS : indisponible (aucun BKS connu pour ces instances)")
    if csv_path:
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
        print(f"\n  Tableau final exporté (CSV) → {csv_path}")
    if latex_path:
        lines = [
            "% Tableau généré automatiquement — Situation 1 vs BKS",
            "\\begin{table}[htbp]", "  \\centering",
            "  \\caption{Comparaison par instance Solomon officielle entre la méthode "
            "hybride (LNS + DQN + Gurobi repair) et le Best Known Solution (BKS).}",
            "  \\label{tab:comparaison-bks}",
            "  \\begin{tabular}{lrrrrrr}", "    \\toprule",
            "    Instance & BKS Vhc & BKS Dist. & Vhc. S1 & Dist. S1 & "
            "Temps S1 (s) & Gap S1 (\\%) \\\\",
            "    \\midrule",
        ]
        for r in rows:
            bks_v = r['bks_vehicules'] if r['bks_vehicules'] is not None else "--"
            bks_d = f"{r['bks_distance']:.2f}" if r['bks_distance'] else "--"
            g1    = f"{r['gap_%_situation1_bks']:+.2f}" if r['gap_%_situation1_bks'] is not None else "--"
            lines.append(f"    {r['instance']} & {bks_v} & {bks_d} & "
                        f"{r['n_vehicules_situation1']} & {r['distance_situation1']:.2f} & "
                        f"{r['temps_s_situation1']:.1f} & {g1} \\\\")
        lines += ["    \\bottomrule", "  \\end{tabular}", "\\end{table}"]
        with open(latex_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))
        print(f"  Tableau final exporté (LaTeX) → {latex_path}")
    return rows
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 6bis  —  DIAGNOSTIC : LICENCE GUROBI À TAILLE LIMITÉE ?
# ══════════════════════════════════════════════════════════════════════════════
def check_gurobi_license(min_vars_needed=2500):
    env, m = None, None
    try:
        env = gp.Env(empty=True)
        env.setParam('OutputFlag', 0)
        env.start()
        m = gp.Model(env=env)
        m.setParam('OutputFlag', 0)
        x = m.addVars(min_vars_needed, vtype=GRB.BINARY)
        m.setObjective(gp.quicksum(x[i] for i in range(min_vars_needed)), GRB.MAXIMIZE)
        m.optimize()
        print(f"  ✓ Licence Gurobi : aucune restriction de taille détectée "
              f"(modèle-test à {min_vars_needed} variables résolu sans erreur).")
        return True
    except gp.GurobiError as e:
        errno = getattr(e, 'errno', None)
        is_size_limit = (errno == getattr(GRB.Error, 'SIZE_LIMIT_EXCEEDED', object()))
        if is_size_limit:
            k_repair_vars = (CFG['K_DESTROY'] + 1) ** 2 * CFG['K_DESTROY']
            print(f"\n  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée "
                  f"(erreur {errno} : {e})\n"
                  f"    → Licence Community/gratuite, plafonnée à ~2000 variables "
                  f"et ~2000 contraintes.\n"
                  f"    → Le repair Gurobi de la Situation 1 (jusqu'à "
                  f"~{k_repair_vars} variables selon K_DESTROY={CFG['K_DESTROY']} et "
                  f"le nombre de véhicules) peut dépasser cette limite — repli "
                  f"automatique déjà en place vers le repair glouton.\n"
                  f"    → Une licence académique Gurobi est gratuite et sans limite "
                  f"de taille pour les universités (https://www.gurobi.com/academia/).\n")
        else:
            print(f"  ⚠ Erreur Gurobi lors du test de licence (non liée à la taille) : {e}")
        return False
    except Exception as e:
        print(f"  ⚠ Impossible de tester la licence Gurobi ({type(e).__name__}: {e}) "
              f"— test ignoré, l'exécution continue normalement.")
        return None
    finally:
        if m is not None:
            m.dispose()
        if env is not None:
            env.dispose()
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 7  —  COURBES DE CONVERGENCE
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_mean(vals, window=20):
    out = []
    for i in range(len(vals)):
        lo = max(0, i - window + 1)
        out.append(float(np.mean(vals[lo:i + 1])))
    return out
def plot_global_convergence(training_log, instances, initial_costs, best_costs, outdir):
    episodes = [e['episode_global'] for e in training_log]
    rewards  = [e['total_reward']   for e in training_log]
    reward_smooth = _rolling_mean(rewards, window=20)
    improvements  = [(initial_costs[i] - best_costs[i]) / initial_costs[i] * 100
                      for i in range(len(initial_costs))]
    names = [inst['name'] for inst in instances]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    axes[0].plot(episodes, rewards, color='lightgray', linewidth=0.6, label='reward brut / épisode')
    axes[0].plot(episodes, reward_smooth, color='crimson', linewidth=2,
                 label='moyenne mobile (20 épisodes)')
    axes[0].set_xlabel('Épisode global')
    axes[0].set_ylabel('Reward de l\'épisode')
    axes[0].set_title("Convergence de l'apprentissage DQN")
    axes[0].legend(loc='best', fontsize=9)
    axes[0].grid(alpha=0.3)
    colors = ['#2ca02c' if v >= 0 else '#d62728' for v in improvements]
    axes[1].bar(range(len(improvements)), improvements, color=colors)
    axes[1].axhline(np.mean(improvements), color='navy', linestyle='--', linewidth=1.5,
                     label=f"Moyenne = {np.mean(improvements):.2f}%")
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names, rotation=90, fontsize=7)
    axes[1].set_xlabel('Instance')
    axes[1].set_ylabel('Amélioration vs solution LNS initiale (%)')
    axes[1].set_title("Amélioration du coût par instance après DQN+Gurobi")
    axes[1].legend(loc='best', fontsize=9)
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_globale.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence globale → {path}")
    return path
def plot_gap_vs_bks(bks_rows, outdir):
    data = [(r['instance'], r['gap_%_situation1_bks']) for r in bks_rows
            if r['gap_%_situation1_bks'] is not None]
    if not data:
        print("  [plot_gap_vs_bks] Aucune instance avec BKS connu — graphique ignoré.")
        return None
    names = [d[0] for d in data]
    gaps  = [d[1] for d in data]
    gap_moyen = float(np.mean(gaps))
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(names)), 5.5))
    colors = ['#2ca02c' if g <= 0 else '#d62728' for g in gaps]
    ax.bar(range(len(names)), gaps, color=colors)
    ax.axhline(gap_moyen, color='navy', linestyle='--', linewidth=1.5,
               label=f"Gap moyen = {gap_moyen:+.2f}%")
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_xlabel('Instance')
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_title(f"Écart au Best Known Solution par instance "
                 f"({len(names)} instances avec BKS connu)")
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique gap vs BKS → {path}")
    return path
def plot_instance_convergence_examples(training_log, instances, example_idxs, outdir):
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(13, 4.2 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        entries = [e for e in training_log if e['instance_id'] == idx]
        entries.sort(key=lambda e: e['ep_local'])
        ep_locals      = [e['ep_local']   for e in entries]
        best_costs_ep  = [e['best_cost']  for e in entries]
        final_costs_ep = [e['final_cost'] for e in entries]
        name = instances[idx]['name']
        axes[row, 0].plot(ep_locals, final_costs_ep, '-o', color='steelblue',
                           label='coût final de l\'épisode')
        axes[row, 0].plot(ep_locals, best_costs_ep, '-s', color='crimson',
                           label='meilleur coût cumulé')
        axes[row, 0].set_title(f"Instance {name} — convergence par épisode local")
        axes[row, 0].set_xlabel('Épisode local')
        axes[row, 0].set_ylabel('Coût de la tournée')
        axes[row, 0].legend(loc='best', fontsize=9)
        axes[row, 0].grid(alpha=0.3)
        last = entries[-1] if entries else None
        trace = last.get('cost_trace', []) if last else []
        if trace:
            axes[row, 1].plot(range(len(trace)), trace, '-o', color='darkorange')
            axes[row, 1].set_title(f"Instance {name} — trajectoire de coût "
                                    f"(dernier épisode local, {len(trace)-1} steps)")
            axes[row, 1].set_xlabel('Step')
            axes[row, 1].set_ylabel('Coût courant')
            axes[row, 1].grid(alpha=0.3)
        else:
            axes[row, 1].axis('off')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_convergence_instances_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique convergence instances exemples → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 8  —  TRACÉS DES ROUTES (AVANT LNS / APRÈS DQN)
# ══════════════════════════════════════════════════════════════════════════════
def _plot_routes_on_ax(ax, instance, solution, title):
    depot = instance['depot']
    ax.scatter([depot['x']], [depot['y']], c='black', marker='s', s=110,
               zorder=5, label='Dépôt')
    n_routes = max(len([r for r in solution if r]), 1)
    try:
        cmap = matplotlib.colormaps.get_cmap('tab20').resampled(max(n_routes, 1))
    except AttributeError:
        cmap = plt.cm.get_cmap('tab20', max(n_routes, 1))
    ridx = 0
    for route in solution:
        if not route:
            continue
        xs = [depot['x']] + [instance['customers'][cid - 1]['x'] for cid in route] + [depot['x']]
        ys = [depot['y']] + [instance['customers'][cid - 1]['y'] for cid in route] + [depot['y']]
        ax.plot(xs, ys, '-o', color=cmap(ridx), markersize=4, linewidth=1.3)
        ridx += 1
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.3)
def plot_routes_examples(instances, initial_sols, best_solutions, example_idxs, outdir):
    n = len(example_idxs)
    fig, axes = plt.subplots(n, 2, figsize=(11, 4.8 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, idx in enumerate(example_idxs):
        inst = instances[idx]
        name = inst['name']
        bks  = BKS_REFERENCE.get(name)
        bks_str = (f" | BKS: dist={bks['distance']:.1f}, vhc={bks['vehicles']}"
                   if bks else "")
        cost_before = compute_cost(initial_sols[idx], inst)
        cost_after  = compute_cost(best_solutions[idx], inst)
        nveh_before = sum(1 for r in initial_sols[idx] if r)
        nveh_after  = sum(1 for r in best_solutions[idx] if r)
        _plot_routes_on_ax(
            axes[row, 0], inst, initial_sols[idx],
            f"Instance {name} — AVANT (LNS)\n"
            f"dist={cost_before:.1f}  véhicules={nveh_before}{bks_str}"
        )
        _plot_routes_on_ax(
            axes[row, 1], inst, best_solutions[idx],
            f"Instance {name} — APRÈS (DQN+Gurobi)\n"
            f"dist={cost_after:.1f}  véhicules={nveh_after}{bks_str}"
        )
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_routes_avant_apres_exemples.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique routes avant/après → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  MODULE 9  —  ANALYSES PAR GROUPE D'INSTANCES (C1/C2/R1/R2/RC1/RC2)
# ══════════════════════════════════════════════════════════════════════════════
def get_instance_group(inst):
    t = inst['type']
    name = inst['name']
    digit = name[len(t)] if len(name) > len(t) else ''
    return f"{t}{digit}"
def plot_gap_boxplot_by_group(bks_rows, instances, outdir):
    by_name = {inst['name']: inst for inst in instances}
    groups = defaultdict(list)
    for r in bks_rows:
        if r['gap_%_situation1_bks'] is None:
            continue
        inst = by_name.get(r['instance'])
        if inst is None:
            continue
        groups[get_instance_group(inst)].append(r['gap_%_situation1_bks'])
    labels = [g for g in GROUP_ORDER if g in groups]
    data   = [groups[g] for g in labels]
    if not data:
        print("  [plot_gap_boxplot_by_group] Aucune donnée — graphique ignoré.")
        return None
    fig, ax = plt.subplots(figsize=(8, 5.5))
    try:
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, showmeans=True)
    except TypeError:
        bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#a6c8ff')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Gap % vs BKS (Situation 1)')
    ax.set_xlabel('Groupe d\'instances Solomon')
    ax.set_title("Distribution du gap par groupe d'instances")
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_gap_boxplot_par_groupe.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique boxplot gap par groupe → {path}")
    return path
def plot_vehicles_vs_bks(bks_rows, instances, outdir):
    by_name = {inst['name']: inst for inst in instances}
    cmap_colors = plt.cm.tab10.colors
    color_map = {g: cmap_colors[i % len(cmap_colors)] for i, g in enumerate(GROUP_ORDER)}
    fig, ax = plt.subplots(figsize=(7, 7))
    seen_groups = set()
    max_v = 1
    for r in bks_rows:
        if r['bks_vehicules'] is None or r['n_vehicules_situation1'] is None:
            continue
        inst = by_name.get(r['instance'])
        grp  = get_instance_group(inst) if inst else '?'
        c    = color_map.get(grp, 'gray')
        label = grp if grp not in seen_groups else None
        seen_groups.add(grp)
        ax.scatter(r['bks_vehicules'], r['n_vehicules_situation1'], color=c,
                   label=label, s=45, alpha=0.85, edgecolor='black', linewidth=0.4)
        max_v = max(max_v, r['bks_vehicules'], r['n_vehicules_situation1'])
    ax.plot([0, max_v + 1], [0, max_v + 1], '--', color='gray', linewidth=1, label='y = x (égalité)')
    ax.set_xlabel('Véhicules BKS')
    ax.set_ylabel('Véhicules Situation 1 (DQN+LNS+Gurobi)')
    ax.set_title('Nombre de véhicules utilisés vs BKS, par instance')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(outdir, 'solomon_vehicules_vs_bks.png')
    plt.savefig(path, dpi=140)
    plt.close(fig)
    print(f"  Graphique véhicules vs BKS → {path}")
    return path
# ══════════════════════════════════════════════════════════════════════════════
#  EXÉCUTION AUTOMATIQUE  —  run_final_validation()
# ══════════════════════════════════════════════════════════════════════════════
def run_final_validation():
    """
    MODE VALIDATION : ENTRAÎNEMENT/TEST DISJOINT, jeu d'actions 'merged2' fixé,
    fusion de routes + recherche locale activées (CFG['ROUTE_FUSION'] et
    CFG['LOCAL_SEARCH_POLISH']).
    Entraîne le DQN sur CFG['N_SYNTH_INSTANCES'] instances synthétiques puis
    l'évalue gelé sur les 56 instances Solomon officielles, applique en
    post-traitement : (1) fusion de routes, (2) recherche locale 2-opt +
    Or-opt, (3) une seconde fusion de routes — puis produit le tableau
    final et tous les graphiques.
    """
    t0 = time.time()
    print(f"\n{'═'*72}\n  MODE ENTRAÎNEMENT/TEST DISJOINT — ACTION_SET={CFG['ACTION_SET']!r}, "
          f"ROUTE_FUSION={CFG['ROUTE_FUSION']['ENABLE']}\n"
          f"  ENTRAÎNEMENT = {CFG['N_SYNTH_INSTANCES']} instances SYNTHÉTIQUES générées "
          f"| TEST = 56 instances Solomon officielles (gelé)\n{'═'*72}")
    _sec("DIAGNOSTIC — LICENCE GUROBI")
    check_gurobi_license()
    if CFG.get('USE_TAGUCHI_BEST', False):
        _sec("APPLICATION DE LA MEILLEURE COMBINAISON TAGUCHI (SYNTHÉTIQUE)")
        load_and_apply_taguchi_best(CFG['OUTDIR'], json_suffix='_synthetique')
    synth_instances = generate_synthetic_training_set(CFG)
    training_log, synth_initial_costs, synth_best_costs = [], None, None
    model_path = os.path.join(CFG['MODELDIR'], CFG['MODEL_FILE'])
    if CFG.get('REUSE_EXISTING_SYNTH_MODEL', False) and os.path.exists(model_path):
        _sec(f"ÉTAPE 1b — RÉUTILISATION D'UN MODÈLE DÉJÀ ENTRAÎNÉ SUR LE SYNTHÉTIQUE "
             f"({model_path})")
        agent = DQNAgent(
            obs_dim=7, n_actions=ACTION_SETS_N.get(CFG.get('ACTION_SET', 'full'), 4),
            hidden=CFG['HIDDEN'], lr=CFG['LR'],
            gamma=CFG['GAMMA'], epsilon_start=CFG['EPS_START'],
            epsilon_end=CFG['EPS_END'], epsilon_decay=CFG['EPS_DECAY'],
            buffer_size=CFG['BUFFER_SIZE'], batch_size=CFG['BATCH_SIZE'],
            target_update_freq=CFG['TARGET_UPDATE'],
        )
        agent.load(model_path)
    else:
        agent, training_log, synth_initial_costs, synth_best_costs, \
            synth_best_solutions, synth_instance_times, synth_initial_sols = train_on_instances(
                synth_instances, episodes_per_instance=CFG['EPISODES_PER_INSTANCE'])
    _sec("ÉTAPE 2 — CHARGEMENT DES INSTANCES SOLOMON (TEST)")
    test_instances = load_all_instances(INSTANCE_FILES, vehicle_safety_factor=CFG['VEHICLE_SAFETY_FACTOR'])
    example_idxs = pick_example_indices(test_instances, preferred_names=CFG['EXAMPLE_NAMES'])
    initial_costs, best_costs, best_solutions, instance_times, test_initial_sols, test_log = \
        evaluate_frozen_on_instances(
            test_instances, agent, episodes_per_instance=CFG['EPISODES_PER_INSTANCE_TEST'])
    # ── FUSION DE ROUTES (MODULE 5bis) — post-traitement sur le TEST gelé ──
    if CFG.get('ROUTE_FUSION', {}).get('ENABLE', False):
        best_solutions, best_costs, _route_fusion_stats = route_merge_postprocess_batch(
            test_instances, best_solutions, verbose=True)
    # ── RECHERCHE LOCALE 2-OPT + OR-OPT (MODULE 5ter) — post-traitement,   ──
    # APRÈS la première fusion, sur les mêmes solutions gelées du TEST.    ──
    if CFG.get('LOCAL_SEARCH_POLISH', {}).get('ENABLE', False):
        ls_cfg = CFG['LOCAL_SEARCH_POLISH']
        best_solutions, best_costs, _ls_stats = local_search_polish_batch(
            test_instances, best_solutions, verbose=True,
            max_rounds=ls_cfg.get('MAX_ROUNDS', 5),
            seg_len_max=ls_cfg.get('SEG_LEN_MAX', 3))
        # ── Seconde passe de fusion : l'Or-opt peut avoir vidé une route ──
        # ou allégé une charge, ouvrant de nouvelles opportunités de       ──
        # fusion qui n'existaient pas avant la recherche locale.          ──
        if CFG.get('ROUTE_FUSION', {}).get('ENABLE', False):
            best_solutions, best_costs, _route_fusion_stats2 = route_merge_postprocess_batch(
                test_instances, best_solutions, verbose=True)
    _sec("GRAPHIQUE — CONVERGENCE DE L'ENTRAÎNEMENT (SYNTHÉTIQUE)")
    if training_log:
        plot_global_convergence(training_log, synth_instances,
                                 synth_initial_costs, synth_best_costs, CFG['OUTDIR'])
    else:
        print("  (pas de courbe de convergence : modèle rechargé, aucun "
              "entraînement effectué dans ce run)")
    _sec("GRAPHIQUES — DÉTAIL DU TEST (INSTANCES SOLOMON EXEMPLES)")
    plot_instance_convergence_examples(test_log, test_instances, example_idxs, CFG['OUTDIR'])
    plot_routes_examples(test_instances, test_initial_sols, best_solutions, example_idxs, CFG['OUTDIR'])
    bks_rows = build_bks_comparison_table(
        test_instances, initial_costs, best_costs, best_solutions, instance_times,
        csv_path=os.path.join(CFG['OUTDIR'], 'solomon_comparaison_bks.csv'),
        latex_path=os.path.join(CFG['OUTDIR'], 'solomon_tableau_bks.tex'),
    )
    _sec("GRAPHIQUE GAP % vs BKS (TEST)")
    plot_gap_vs_bks(bks_rows, CFG['OUTDIR'])
    _sec("GRAPHIQUES — ANALYSES PAR GROUPE D'INSTANCES (TEST)")
    plot_gap_boxplot_by_group(bks_rows, test_instances, CFG['OUTDIR'])
    plot_vehicles_vs_bks(bks_rows, test_instances, CFG['OUTDIR'])
    print(f"\n  ✓ Validation finale terminée en {time.time()-t0:.1f}s "
          f"({(time.time()-t0)/3600:.2f}h)\n")
    return bks_rows
if __name__ == '__main__':
    run_final_validation()


════════════════════════════════════════════════════════════════════════
  MODE ENTRAÎNEMENT/TEST DISJOINT — ACTION_SET='merged2', ROUTE_FUSION=True
  ENTRAÎNEMENT = 80 instances SYNTHÉTIQUES générées | TEST = 56 instances Solomon officielles (gelé)
════════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════════
  DIAGNOSTIC — LICENCE GUROBI
════════════════════════════════════════════════════════════════════════

  ⚠ LICENCE GUROBI À TAILLE LIMITÉE détectée (erreur 10010 : Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information)
    → Licence Community/gratuite, plafonnée à ~2000 variables et ~2000 contraintes.
    → Le repair Gurobi de la Situation 1 (jusqu'à ~15000 variables selon K_DESTROY=24 et le nombre de véhicules) peut dépasser cette limite — repli automatique déjà en place vers le repair glouton.
    → Une licence académique Gurobi est gratuite et s